In [ ]:
# Import libraries
## Basic libs
import pandas as pd
import numpy as np
import warnings
## Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Data Pre-Processing

In [ ]:
import pandas as pd
# Load dataset
df = pd.read_csv('2567-2Q9Q8Q.csv')

# print(df_depression.info())
print('Shape of dataframe:', df.shape)
df.head()

In [ ]:
import pandas as pd
# Load dataset
dfvalid = pd.read_csv('2568-2Q9Q8Q.csv')

# print(df_depression.info())
print('Shape of dataframe:', df.shape)
dfvalid.head()

## Normalization MinMaxScaler

In [ ]:
# from sklearn.preprocessing import MinMaxScaler

# Copying original dataframe
# df_total = df.copy()

# scaler = MinMaxScaler()
# num_cols = ['grade_entry']
# df[num_cols] = scaler.fit_transform(df[num_cols])

df.head()

## PART 0: Define Feature Sets & Screening Strategies (7 Models)

In [ ]:
# ============================================================
# MDD DX ENGINE
# PART 0: DEFINE FEATURE SETS, SCREENING STRATEGIES,
#         AND RESAMPLING METADATA
# ============================================================
#
# Objective:
# 1. Define the primary clinically diagnosed MDD outcome.
# 2. Define the seven predefined screening strategies.
# 3. Classify predictors as continuous or categorical.
# 4. Assign a categorical-aware sampler to each strategy.
#
# Resampling rules:
# - Mixed continuous and categorical predictors: SMOTENC
# - Categorical predictors only: SMOTEN
# - Continuous predictors only: SMOTE
#
# Questionnaire items are treated as categorical/ordinal
# response categories during synthetic resampling to avoid
# generating fractional response values.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. PRIMARY OUTCOME
# ============================================================

OUTCOME = "mdddx"


# ============================================================
# 2. FEATURE GROUPS
# ============================================================

# Administrative variables:
# 8 features and 0 additional respondent questions

student_features = [
    "institute",
    "GPAX",
    "dadstatus",
    "momstatus",
    "dadeducation",
    "momeducation",
    "parent",
    "gender"
]


# Health-behavior variables:
# 7 additional respondent questions

behavior_features = [
    "disease",
    "smoking",
    "elecsmoking",
    "drinking",
    "kratom",
    "cannabis",
    "addictive"
]


# Thai 2Q depression-screening items:
# 2 additional respondent questions

phq2_features = [
    "2Q1",
    "2Q2"
]


# Selected 9Q items:
# 3 additional respondent questions

phq9_features = [
    "9Q2",
    "9Q4",
    "9Q6"
]


# Selected 8Q items:
# 4 additional respondent questions

q8_features = [
    "8Q1",
    "8Q2",
    "8Q3_1",
    "8Q8"
]


# Conventional questionnaire strategy:
# 2Q + selected 9Q + selected 8Q = 9 questions

conventional_features = (
    phq2_features
    + phq9_features
    + q8_features
)


# ============================================================
# 3. PREDICTOR-TYPE DEFINITIONS
# ============================================================

# GPAX is treated as a continuous numerical predictor.

continuous_features = [
    "GPAX"
]


# Administrative categorical predictors

administrative_categorical_features = [
    "institute",
    "dadstatus",
    "momstatus",
    "dadeducation",
    "momeducation",
    "parent",
    "gender"
]


# Binary or categorical health-behavior predictors

behavior_categorical_features = [
    "disease",
    "smoking",
    "elecsmoking",
    "drinking",
    "kratom",
    "cannabis",
    "addictive"
]


# Individual questionnaire responses are treated as
# categorical/ordinal categories during resampling.

questionnaire_categorical_features = list(
    dict.fromkeys(
        phq2_features
        + phq9_features
        + q8_features
    )
)


# Complete categorical predictor list

categorical_features = list(
    dict.fromkeys(
        administrative_categorical_features
        + behavior_categorical_features
        + questionnaire_categorical_features
    )
)


# ============================================================
# 4. SCREENING STRATEGIES
# ============================================================

screening_strategies = {

    "M0_Admin": {
        "type": "Administrative",
        "feature_group": "Administrative",
        "description": "Administrative data only",
        "features": student_features.copy(),
        "additional_questions": 0
    },

    "M1_Admin_Behavior": {
        "type": "Administrative + Behavior",
        "feature_group": "Administrative + Behavior",
        "description": (
            "Administrative data plus health-behavior questions"
        ),
        "features": (
            student_features
            + behavior_features
        ),
        "additional_questions": 7
    },

    "M2_Admin_2Q": {
        "type": "Administrative + 2Q",
        "feature_group": "Administrative + 2Q",
        "description": (
            "Administrative data plus the two 2Q items"
        ),
        "features": (
            student_features
            + phq2_features
        ),
        "additional_questions": 2
    },

    "M3_Admin_2Q_Behavior": {
        "type": "Administrative + Behavior + 2Q",
        "feature_group": "Administrative + Behavior + 2Q",
        "description": (
            "Administrative data, health-behavior questions, "
            "and the two 2Q items"
        ),
        "features": (
            student_features
            + behavior_features
            + phq2_features
        ),
        "additional_questions": 9
    },

    "M4_Conventional": {
        "type": "Conventional",
        "feature_group": (
            "2Q + selected 9Q + selected 8Q"
        ),
        "description": (
            "Conventional symptom-screening strategy"
        ),
        "features": conventional_features.copy(),
        "additional_questions": 9
    },

    "M5_Conventional_Admin": {
        "type": "Administrative + Conventional",
        "feature_group": (
            "Administrative + conventional screening"
        ),
        "description": (
            "Administrative data plus conventional "
            "symptom screening"
        ),
        "features": (
            student_features
            + conventional_features
        ),
        "additional_questions": 9
    },

    "M6_Full": {
        "type": "Full Assessment",
        "feature_group": (
            "Administrative + Behavior + Conventional"
        ),
        "description": (
            "Administrative data, health-behavior questions, "
            "and conventional symptom screening"
        ),
        "features": (
            student_features
            + behavior_features
            + conventional_features
        ),
        "additional_questions": 16
    }
}


# ============================================================
# 5. EXPECTED FEATURE COUNTS
# ============================================================

expected_feature_counts = {
    "M0_Admin": 8,
    "M1_Admin_Behavior": 15,
    "M2_Admin_2Q": 10,
    "M3_Admin_2Q_Behavior": 17,
    "M4_Conventional": 9,
    "M5_Conventional_Admin": 17,
    "M6_Full": 24
}


for model_name, expected_count in (
    expected_feature_counts.items()
):

    actual_count = len(
        screening_strategies[
            model_name
        ]["features"]
    )

    if actual_count != expected_count:

        raise ValueError(
            f"{model_name}: expected {expected_count} "
            f"features but found {actual_count}."
        )


# ============================================================
# 6. VALIDATE REQUIRED DATAFRAME COLUMNS
# ============================================================

if "df" not in globals():

    raise NameError(
        "The source dataframe 'df' has not been created."
    )


all_predictor_features = list(
    dict.fromkeys(
        feature
        for strategy_info in screening_strategies.values()
        for feature in strategy_info["features"]
    )
)


all_required_columns = (
    all_predictor_features
    + [OUTCOME]
)


missing_columns = [
    column
    for column in all_required_columns
    if column not in df.columns
]


if missing_columns:

    raise KeyError(
        "The source dataframe is missing required columns: "
        f"{missing_columns}"
    )


# ============================================================
# 7. VALIDATE PREDICTOR-TYPE ASSIGNMENTS
# ============================================================

unclassified_features = sorted(
    set(all_predictor_features)
    - set(continuous_features)
    - set(categorical_features)
)


duplicated_type_assignments = sorted(
    set(continuous_features)
    & set(categorical_features)
)


if unclassified_features:

    raise ValueError(
        "The following predictors have not been classified "
        "as continuous or categorical: "
        f"{unclassified_features}"
    )


if duplicated_type_assignments:

    raise ValueError(
        "The following predictors were assigned to both "
        "continuous and categorical groups: "
        f"{duplicated_type_assignments}"
    )


# ============================================================
# 8. ASSIGN STRATEGY-SPECIFIC RESAMPLING METADATA
# ============================================================

for model_name, strategy_info in (
    screening_strategies.items()
):

    strategy_features = list(
        strategy_info["features"]
    )

    strategy_continuous_features = [
        feature
        for feature in strategy_features
        if feature in continuous_features
    ]

    strategy_categorical_features = [
        feature
        for feature in strategy_features
        if feature in categorical_features
    ]

    strategy_categorical_indices = [
        strategy_features.index(feature)
        for feature in strategy_categorical_features
    ]

    if (
        strategy_continuous_features
        and strategy_categorical_features
    ):

        sampler_type = "SMOTENC"

    elif (
        not strategy_continuous_features
        and strategy_categorical_features
    ):

        sampler_type = "SMOTEN"

    elif (
        strategy_continuous_features
        and not strategy_categorical_features
    ):

        sampler_type = "SMOTE"

    else:

        raise ValueError(
            f"{model_name}: no valid predictors were found."
        )

    strategy_info[
        "continuous_features"
    ] = strategy_continuous_features

    strategy_info[
        "categorical_features"
    ] = strategy_categorical_features

    strategy_info[
        "categorical_indices"
    ] = strategy_categorical_indices

    strategy_info[
        "sampler_type"
    ] = sampler_type


# ============================================================
# 9. VALIDATE OBSERVED CATEGORY VALUES
# ============================================================

category_validation_rows = []


for feature in categorical_features:

    observed_values = (
        df[feature]
        .dropna()
        .unique()
        .tolist()
    )

    try:

        observed_values = sorted(
            observed_values
        )

    except TypeError:

        observed_values = sorted(
            [
                str(value)
                for value in observed_values
            ]
        )

    category_validation_rows.append({
        "Feature": feature,
        "Observed_Category_Count": len(
            observed_values
        ),
        "Observed_Values": observed_values
    })


category_validation_table = pd.DataFrame(
    category_validation_rows
)


# A categorical predictor must contain at least one
# nonmissing observed value.

empty_categorical_features = (
    category_validation_table.loc[
        category_validation_table[
            "Observed_Category_Count"
        ] == 0,
        "Feature"
    ]
    .tolist()
)


if empty_categorical_features:

    raise ValueError(
        "The following categorical predictors contain no "
        "observed nonmissing values: "
        f"{empty_categorical_features}"
    )


# ============================================================
# 10. CREATE STRATEGY-SPECIFIC DATASETS
# ============================================================

mdd_datasets = {}


for model_name, strategy_info in (
    screening_strategies.items()
):

    selected_features = list(
        strategy_info["features"]
    )

    mdd_datasets[model_name] = (
        df[
            selected_features
            + [OUTCOME]
        ]
        .copy()
    )


# ============================================================
# 11. CREATE SCREENING-STRATEGY SUMMARY
# ============================================================

strategy_summary_rows = []


for model_name, strategy_info in (
    screening_strategies.items()
):

    strategy_summary_rows.append({

        "Model": model_name,

        "Type": strategy_info["type"],

        "Description":
            strategy_info["description"],

        "Number_of_Features":
            len(strategy_info["features"]),

        "Additional_Questions":
            strategy_info["additional_questions"],

        "Continuous_Features":
            ", ".join(
                strategy_info[
                    "continuous_features"
                ]
            ),

        "Categorical_Features":
            ", ".join(
                strategy_info[
                    "categorical_features"
                ]
            ),

        "Categorical_Indices":
            strategy_info[
                "categorical_indices"
            ],

        "Sampler_Type":
            strategy_info[
                "sampler_type"
            ],

        "Features":
            ", ".join(
                strategy_info["features"]
            )
    })


screening_summary = pd.DataFrame(
    strategy_summary_rows
)


strategy_order = [
    "M0_Admin",
    "M1_Admin_Behavior",
    "M2_Admin_2Q",
    "M3_Admin_2Q_Behavior",
    "M4_Conventional",
    "M5_Conventional_Admin",
    "M6_Full"
]


screening_summary["Model"] = pd.Categorical(
    screening_summary["Model"],
    categories=strategy_order,
    ordered=True
)


screening_summary = (
    screening_summary
    .sort_values("Model")
    .reset_index(drop=True)
)


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

expected_sampler_types = {
    "M0_Admin": "SMOTENC",
    "M1_Admin_Behavior": "SMOTENC",
    "M2_Admin_2Q": "SMOTENC",
    "M3_Admin_2Q_Behavior": "SMOTENC",
    "M4_Conventional": "SMOTEN",
    "M5_Conventional_Admin": "SMOTENC",
    "M6_Full": "SMOTENC"
}


for model_name, expected_sampler in (
    expected_sampler_types.items()
):

    observed_sampler = (
        screening_strategies[
            model_name
        ]["sampler_type"]
    )

    if observed_sampler != expected_sampler:

        raise ValueError(
            f"{model_name}: expected sampler "
            f"{expected_sampler}, but found "
            f"{observed_sampler}."
        )


for model_name, strategy_info in (
    screening_strategies.items()
):

    expected_features = list(
        strategy_info["features"]
    )

    observed_features = (
        mdd_datasets[model_name]
        .drop(columns=[OUTCOME])
        .columns
        .tolist()
    )

    if observed_features != expected_features:

        raise ValueError(
            f"{model_name}: feature order changed during "
            "dataset construction."
        )


# ============================================================
# 13. EXPORT AND DISPLAY
# ============================================================

screening_summary.to_csv(
    "part0_screening_strategy_summary.csv",
    index=False
)


category_validation_table.to_csv(
    "part0_categorical_value_validation.csv",
    index=False
)


print("=" * 90)
print("PART 0 COMPLETED")
print("=" * 90)

print(
    f"\nOutcome: {OUTCOME}"
)

print(
    f"Number of screening strategies: "
    f"{len(screening_strategies)}"
)

print(
    f"Continuous predictors: "
    f"{continuous_features}"
)

print(
    f"Number of categorical predictors: "
    f"{len(categorical_features)}"
)

print(
    "\nStrategy-specific resampling configuration:"
)


display(
    screening_summary[
        [
            "Model",
            "Number_of_Features",
            "Additional_Questions",
            "Continuous_Features",
            "Categorical_Indices",
            "Sampler_Type"
        ]
    ]
)


print(
    "\nObserved values of categorical predictors:"
)


display(
    category_validation_table
)


print(
    "\nExported files:"
)

print(
    "- part0_screening_strategy_summary.csv"
)

print(
    "- part0_categorical_value_validation.csv"
)

## PART 1: DATA PREPARATION (Mild Balancing Version)

In [ ]:
# ============================================================
# MDD DX ENGINE
# PART 1: SHARED DEVELOPMENT–VALIDATION SPLIT
# ============================================================
#
# Objectives:
# 1. Create one shared stratified 70:30 split for all strategies.
# 2. Preserve participant alignment across M0–M6.
# 3. Retain the original development data without resampling.
# 4. Keep the held-out 2024 validation cohort untouched.
# 5. Store strategy-specific categorical metadata for later
#    fold-specific SMOTENC/SMOTEN resampling.
#
# IMPORTANT:
# - No synthetic resampling is performed in PART 1.
# - SMOTENC/SMOTEN will be fitted only within training folds.
# - Held-out observations are not used to fit a sampler,
#   preprocessing procedure, model, or threshold.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

RANDOM_STATE = 42
TEST_SIZE = 0.30

N_FOLDS = 5

SMOTE_SAMPLING_STRATEGY = 0.20
UNDER_SAMPLING_STRATEGY = 0.50
SMOTE_K_NEIGHBORS = 5


# ============================================================
# 2. REQUIRED-OBJECT VALIDATION
# ============================================================

required_objects = [
    "mdd_datasets",
    "screening_strategies",
    "OUTCOME"
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


if not isinstance(mdd_datasets, dict):

    raise TypeError(
        "mdd_datasets must be a dictionary."
    )


if not isinstance(screening_strategies, dict):

    raise TypeError(
        "screening_strategies must be a dictionary."
    )


if not mdd_datasets:

    raise ValueError(
        "mdd_datasets is empty."
    )


if set(mdd_datasets) != set(screening_strategies):

    raise ValueError(
        "The model names in mdd_datasets and "
        "screening_strategies do not match."
    )


# ============================================================
# 3. RESULT CONTAINERS
# ============================================================

prepared_data = {}

part1_summary_rows = []

part1_alignment_rows = []


# ============================================================
# 4. REFERENCE DATASET
# ============================================================
#
# All screening strategies must contain the same participants
# in the same row order before the shared split is created.
# ============================================================

reference_model_name = next(
    iter(mdd_datasets)
)


reference_dataset = (
    mdd_datasets[
        reference_model_name
    ]
    .copy()
)


if OUTCOME not in reference_dataset.columns:

    raise KeyError(
        f"Outcome '{OUTCOME}' was not found in "
        f"{reference_model_name}."
    )


reference_y = (
    reference_dataset[OUTCOME]
    .copy()
)


if reference_y.isna().any():

    raise ValueError(
        f"Outcome '{OUTCOME}' contains missing values."
    )


try:

    reference_y = reference_y.astype(int)

except Exception as error:

    raise TypeError(
        f"Outcome '{OUTCOME}' could not be converted to integer."
    ) from error


observed_outcome_values = sorted(
    reference_y
    .unique()
    .tolist()
)


if observed_outcome_values != [0, 1]:

    raise ValueError(
        f"Outcome '{OUTCOME}' must be coded as 0/1. "
        f"Observed values: {observed_outcome_values}"
    )


reference_index = (
    reference_dataset
    .index
    .to_numpy()
)


# ============================================================
# 5. VALIDATE ALIGNMENT ACROSS STRATEGIES
# ============================================================

for model_name, dataset in mdd_datasets.items():

    if OUTCOME not in dataset.columns:

        raise KeyError(
            f"{model_name}: outcome '{OUTCOME}' is missing."
        )

    strategy_features = list(
        screening_strategies[
            model_name
        ]["features"]
    )

    missing_features = [
        feature
        for feature in strategy_features
        if feature not in dataset.columns
    ]

    same_number_of_rows = (
        len(dataset)
        == len(reference_dataset)
    )

    same_index_order = (
        same_number_of_rows
        and np.array_equal(
            dataset.index.to_numpy(),
            reference_index
        )
    )

    candidate_y = (
        dataset[OUTCOME]
        .astype(int)
    )

    same_outcome_order = (
        len(candidate_y) == len(reference_y)
        and np.array_equal(
            candidate_y.to_numpy(),
            reference_y.to_numpy()
        )
    )

    observed_predictors = (
        dataset
        .drop(columns=[OUTCOME])
        .columns
        .tolist()
    )

    expected_feature_order = (
        observed_predictors
        == strategy_features
    )

    part1_alignment_rows.append({

        "Screening_Model":
            model_name,

        "N":
            int(len(dataset)),

        "Same_Number_of_Rows":
            bool(same_number_of_rows),

        "Same_Index_Order":
            bool(same_index_order),

        "Same_Outcome_Order":
            bool(same_outcome_order),

        "Correct_Feature_Order":
            bool(expected_feature_order),

        "Expected_Feature_Count":
            int(len(strategy_features)),

        "Observed_Predictor_Count":
            int(len(observed_predictors)),

        "Missing_Features":
            ", ".join(missing_features)
    })

    if not same_number_of_rows:

        raise ValueError(
            f"{model_name}: participant count differs from "
            f"{reference_model_name}."
        )

    if not same_index_order:

        raise ValueError(
            f"{model_name}: participant index order differs "
            "from the reference dataset."
        )

    if not same_outcome_order:

        raise ValueError(
            f"{model_name}: outcome order differs from "
            "the reference dataset."
        )

    if missing_features:

        raise KeyError(
            f"{model_name}: missing expected features: "
            + ", ".join(missing_features)
        )

    if not expected_feature_order:

        raise ValueError(
            f"{model_name}: predictor order does not match "
            "the predefined screening strategy."
        )


part1_alignment_check = pd.DataFrame(
    part1_alignment_rows
)


print("=" * 90)
print("DATASET ALIGNMENT CHECK")
print("=" * 90)

display(
    part1_alignment_check
)


# ============================================================
# 6. CREATE ONE SHARED STRATIFIED SPLIT
# ============================================================
#
# Row positions are split once and applied unchanged to every
# screening strategy.
# ============================================================

all_row_positions = np.arange(
    len(reference_dataset)
)


train_positions, test_positions = train_test_split(

    all_row_positions,

    test_size=TEST_SIZE,

    stratify=reference_y.to_numpy(),

    shuffle=True,

    random_state=RANDOM_STATE
)


# Sort positions to preserve original participant order within
# the development and held-out partitions.

train_positions = np.sort(
    train_positions
)

test_positions = np.sort(
    test_positions
)


shared_y_train = (
    reference_y
    .iloc[train_positions]
    .copy()
    .reset_index(drop=True)
)


shared_y_test = (
    reference_y
    .iloc[test_positions]
    .copy()
    .reset_index(drop=True)
)


shared_train_original_index = (
    reference_dataset
    .index[train_positions]
    .to_numpy()
)


shared_test_original_index = (
    reference_dataset
    .index[test_positions]
    .to_numpy()
)


# ============================================================
# 7. SHARED-SPLIT VERIFICATION
# ============================================================

if set(train_positions).intersection(
    set(test_positions)
):

    raise RuntimeError(
        "Participant overlap was detected between the "
        "development and held-out partitions."
    )


if (
    len(train_positions)
    + len(test_positions)
    != len(reference_dataset)
):

    raise RuntimeError(
        "The development and held-out partitions do not "
        "cover the complete 2024 analytic cohort."
    )


if len(np.unique(train_positions)) != len(
    train_positions
):

    raise RuntimeError(
        "Duplicate participant positions were detected "
        "in the development partition."
    )


if len(np.unique(test_positions)) != len(
    test_positions
):

    raise RuntimeError(
        "Duplicate participant positions were detected "
        "in the held-out partition."
    )


shared_train_counts = (
    shared_y_train
    .value_counts()
    .sort_index()
)


shared_test_counts = (
    shared_y_test
    .value_counts()
    .sort_index()
)


shared_train_events = int(
    shared_y_train.sum()
)


shared_test_events = int(
    shared_y_test.sum()
)


shared_train_nonevents = int(
    len(shared_y_train)
    - shared_train_events
)


shared_test_nonevents = int(
    len(shared_y_test)
    - shared_test_events
)


print("\n" + "=" * 90)
print("SHARED 2024 DEVELOPMENT–VALIDATION SPLIT")
print("=" * 90)


print("\nComplete 2024 analytic cohort:")

print(
    reference_y
    .value_counts()
    .sort_index()
)

print(
    f"N={len(reference_y):,}; "
    f"MDD events={int(reference_y.sum()):,}; "
    f"prevalence={reference_y.mean() * 100:.3f}%"
)


print("\n2024 development cohort:")

print(
    shared_train_counts
)

print(
    f"N={len(shared_y_train):,}; "
    f"MDD events={shared_train_events:,}; "
    f"prevalence={shared_y_train.mean() * 100:.3f}%"
)


print("\nHeld-out 2024 validation cohort:")

print(
    shared_test_counts
)

print(
    f"N={len(shared_y_test):,}; "
    f"MDD events={shared_test_events:,}; "
    f"prevalence={shared_y_test.mean() * 100:.3f}%"
)


# ============================================================
# 8. CHECK FOLD-SPECIFIC RESAMPLING FEASIBILITY
# ============================================================
#
# The check is conservative because every OOF training fold
# must contain more than k_neighbors minority observations.
# The actual fold-level check will also be performed during OOF
# generation.
# ============================================================

minimum_expected_fold_training_events = int(
    np.floor(
        shared_train_events
        * (N_FOLDS - 1)
        / N_FOLDS
    )
)


if (
    minimum_expected_fold_training_events
    <= SMOTE_K_NEIGHBORS
):

    raise ValueError(
        "The expected number of minority observations in an "
        "OOF training fold is insufficient for SMOTENC/SMOTEN: "
        f"expected training-fold events="
        f"{minimum_expected_fold_training_events}, "
        f"k_neighbors={SMOTE_K_NEIGHBORS}."
    )


print(
    "\nFold-specific resampling feasibility:"
)

print(
    f"- Total development events: "
    f"{shared_train_events}"
)

print(
    f"- Expected minimum training-fold events: "
    f"{minimum_expected_fold_training_events}"
)

print(
    f"- k_neighbors: "
    f"{SMOTE_K_NEIGHBORS}"
)


# ============================================================
# 9. CREATE STRATEGY-SPECIFIC DEVELOPMENT AND HELD-OUT DATA
# ============================================================

for model_name, dataset in mdd_datasets.items():

    print("\n" + "=" * 90)
    print(f"{model_name}: DATA PREPARATION")
    print("=" * 90)

    strategy_info = (
        screening_strategies[
            model_name
        ]
    )

    expected_features = list(
        strategy_info["features"]
    )

    categorical_feature_names = list(
        strategy_info[
            "categorical_features"
        ]
    )

    categorical_feature_indices = list(
        strategy_info[
            "categorical_indices"
        ]
    )

    continuous_feature_names = list(
        strategy_info[
            "continuous_features"
        ]
    )

    sampler_type = (
        strategy_info[
            "sampler_type"
        ]
    )

    X = (
        dataset[
            expected_features
        ]
        .copy()
    )

    y = (
        dataset[OUTCOME]
        .astype(int)
        .copy()
    )

    X_train = (
        X
        .iloc[train_positions]
        .copy()
        .reset_index(drop=True)
    )

    X_test = (
        X
        .iloc[test_positions]
        .copy()
        .reset_index(drop=True)
    )

    y_train = (
        y
        .iloc[train_positions]
        .copy()
        .reset_index(drop=True)
    )

    y_test = (
        y
        .iloc[test_positions]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Validate feature order
    # --------------------------------------------------------

    if X_train.columns.tolist() != expected_features:

        raise ValueError(
            f"{model_name}: development feature order changed."
        )

    if X_test.columns.tolist() != expected_features:

        raise ValueError(
            f"{model_name}: held-out feature order changed."
        )

    # --------------------------------------------------------
    # Validate participant/outcome alignment
    # --------------------------------------------------------

    if not np.array_equal(
        y_train.to_numpy(),
        shared_y_train.to_numpy()
    ):

        raise ValueError(
            f"{model_name}: development outcome alignment failed."
        )

    if not np.array_equal(
        y_test.to_numpy(),
        shared_y_test.to_numpy()
    ):

        raise ValueError(
            f"{model_name}: held-out outcome alignment failed."
        )

    # --------------------------------------------------------
    # Missing values are not permitted in the current
    # complete-case analysis.
    # --------------------------------------------------------

    if X_train.isna().any().any():

        missing_counts = (
            X_train
            .isna()
            .sum()
        )

        missing_counts = missing_counts[
            missing_counts > 0
        ]

        raise ValueError(
            f"{model_name}: development predictors contain "
            f"missing values:\n{missing_counts}"
        )

    if X_test.isna().any().any():

        missing_counts = (
            X_test
            .isna()
            .sum()
        )

        missing_counts = missing_counts[
            missing_counts > 0
        ]

        raise ValueError(
            f"{model_name}: held-out predictors contain "
            f"missing values:\n{missing_counts}"
        )

    if y_train.isna().any() or y_test.isna().any():

        raise ValueError(
            f"{model_name}: outcome contains missing values."
        )

    # --------------------------------------------------------
    # Validate categorical indices
    # --------------------------------------------------------

    observed_categorical_names = [
        expected_features[index]
        for index in categorical_feature_indices
    ]

    if (
        observed_categorical_names
        != categorical_feature_names
    ):

        raise ValueError(
            f"{model_name}: categorical feature indices do not "
            "match the categorical feature names."
        )

    # --------------------------------------------------------
    # Original, nonresampled modeling datasets
    # --------------------------------------------------------

    development_data = pd.concat(
        [
            X_train,
            y_train.rename(OUTCOME)
        ],
        axis=1
    )

    heldout_data = pd.concat(
        [
            X_test,
            y_test.rename(OUTCOME)
        ],
        axis=1
    )

    # --------------------------------------------------------
    # Store data and metadata
    # --------------------------------------------------------

    prepared_data[model_name] = {

        "model_name":
            model_name,

        "features":
            expected_features.copy(),

        "continuous_features":
            continuous_feature_names.copy(),

        "categorical_features":
            categorical_feature_names.copy(),

        "categorical_indices":
            categorical_feature_indices.copy(),

        "sampler_type":
            sampler_type,

        "additional_questions":
            int(
                strategy_info[
                    "additional_questions"
                ]
            ),

        "train_positions":
            train_positions.copy(),

        "test_positions":
            test_positions.copy(),

        "train_original_index":
            shared_train_original_index.copy(),

        "test_original_index":
            shared_test_original_index.copy(),

        "X_train":
            X_train,

        "y_train":
            y_train,

        "X_test":
            X_test,

        "y_test":
            y_test,

        # These objects contain original observations only.
        # No resampling has occurred in PART 1.

        "development_data":
            development_data,

        "heldout_data":
            heldout_data,

        # Retained for compatibility with downstream code.
        # Both aliases refer to original, nonresampled data.

        "train_data":
            development_data,

        "train_data_primary":
            development_data,

        "test_data":
            heldout_data,

        "resampling_applied_part1":
            False
    }

    part1_summary_rows.append({

        "Screening_Model":
            model_name,

        "Features":
            int(len(expected_features)),

        "Additional_Questions":
            int(
                strategy_info[
                    "additional_questions"
                ]
            ),

        "Sampler_Type":
            sampler_type,

        "Categorical_Features":
            int(len(categorical_feature_names)),

        "Continuous_Features":
            int(len(continuous_feature_names)),

        "Development_N":
            int(len(y_train)),

        "Development_Events":
            int(y_train.sum()),

        "Development_NonEvents":
            int(len(y_train) - y_train.sum()),

        "Development_Prevalence":
            float(y_train.mean()),

        "Heldout_N":
            int(len(y_test)),

        "Heldout_Events":
            int(y_test.sum()),

        "Heldout_NonEvents":
            int(len(y_test) - y_test.sum()),

        "Heldout_Prevalence":
            float(y_test.mean()),

        "Resampling_Applied_in_PART1":
            False
    })

    print(
        f"Development: N={len(y_train):,}; "
        f"events={int(y_train.sum()):,}"
    )

    print(
        f"Held-out: N={len(y_test):,}; "
        f"events={int(y_test.sum()):,}"
    )

    print(
        f"Sampler reserved for fold-specific training: "
        f"{sampler_type}"
    )

    print(
        "Resampling applied in PART 1: No"
    )


# ============================================================
# 10. CREATE SUMMARY TABLE
# ============================================================

part1_summary = pd.DataFrame(
    part1_summary_rows
)


strategy_order = [
    "M0_Admin",
    "M1_Admin_Behavior",
    "M2_Admin_2Q",
    "M3_Admin_2Q_Behavior",
    "M4_Conventional",
    "M5_Conventional_Admin",
    "M6_Full"
]


part1_summary[
    "Screening_Model"
] = pd.Categorical(

    part1_summary[
        "Screening_Model"
    ],

    categories=strategy_order,

    ordered=True
)


part1_summary = (
    part1_summary
    .sort_values("Screening_Model")
    .reset_index(drop=True)
)


# ============================================================
# 11. VERIFY ALIGNMENT OF PREPARED DATA
# ============================================================

reference_train_outcome = (
    prepared_data[
        reference_model_name
    ]["y_train"]
    .to_numpy()
)


reference_test_outcome = (
    prepared_data[
        reference_model_name
    ]["y_test"]
    .to_numpy()
)


for model_name, model_data in prepared_data.items():

    if not np.array_equal(
        model_data["y_train"].to_numpy(),
        reference_train_outcome
    ):

        raise RuntimeError(
            f"{model_name}: development outcome alignment "
            "failed after data preparation."
        )

    if not np.array_equal(
        model_data["y_test"].to_numpy(),
        reference_test_outcome
    ):

        raise RuntimeError(
            f"{model_name}: held-out outcome alignment "
            "failed after data preparation."
        )

    if not np.array_equal(
        model_data["train_original_index"],
        shared_train_original_index
    ):

        raise RuntimeError(
            f"{model_name}: development participant-index "
            "alignment failed."
        )

    if not np.array_equal(
        model_data["test_original_index"],
        shared_test_original_index
    ):

        raise RuntimeError(
            f"{model_name}: held-out participant-index "
            "alignment failed."
        )

    if model_data["resampling_applied_part1"]:

        raise RuntimeError(
            f"{model_name}: unexpected resampling was "
            "detected in PART 1."
        )


# ============================================================
# 12. SHARED-SPLIT VERIFICATION TABLE
# ============================================================

part1_shared_split_verification = pd.DataFrame({

    "Partition": [
        "Complete 2024 analytic cohort",
        "2024 development cohort",
        "Held-out 2024 validation cohort"
    ],

    "N": [
        int(len(reference_y)),
        int(len(shared_y_train)),
        int(len(shared_y_test))
    ],

    "MDD_Events": [
        int(reference_y.sum()),
        shared_train_events,
        shared_test_events
    ],

    "MDD_NonEvents": [
        int(len(reference_y) - reference_y.sum()),
        shared_train_nonevents,
        shared_test_nonevents
    ],

    "Prevalence": [
        float(reference_y.mean()),
        float(shared_y_train.mean()),
        float(shared_y_test.mean())
    ],

    "Used_for_Resampling": [
        False,
        False,
        False
    ]
})


# ============================================================
# 13. SHARED-SPLIT MANIFEST
# ============================================================

shared_split_manifest = pd.DataFrame({

    "Original_Index": np.concatenate([
        shared_train_original_index,
        shared_test_original_index
    ]),

    "Row_Position": np.concatenate([
        train_positions,
        test_positions
    ]),

    "Partition": (
        ["Development_2024"]
        * len(train_positions)
        + ["Heldout_2024"]
        * len(test_positions)
    ),

    OUTCOME: np.concatenate([
        shared_y_train.to_numpy(),
        shared_y_test.to_numpy()
    ])
})


shared_split_manifest = (
    shared_split_manifest
    .sort_values("Row_Position")
    .reset_index(drop=True)
)


# ============================================================
# 14. EXPORT
# ============================================================

part1_summary.to_csv(
    "part1_data_preparation_summary.csv",
    index=False
)


part1_alignment_check.to_csv(
    "part1_alignment_check.csv",
    index=False
)


part1_shared_split_verification.to_csv(
    "part1_shared_split_verification.csv",
    index=False
)


shared_split_manifest.to_csv(
    "part1_shared_split_manifest.csv",
    index=False
)


# ============================================================
# 15. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 90)
print("PART 1 COMPLETED")
print("=" * 90)


print(
    "\nNo SMOTE, SMOTENC, SMOTEN, or random "
    "undersampling was performed in PART 1."
)


print(
    "\nDevelopment and held-out summary:"
)

display(
    part1_summary
)


print(
    "\nShared-split verification:"
)

display(
    part1_shared_split_verification
)


print(
    "\nPrepared objects:"
)

print(
    "- prepared_data"
)

print(
    "- part1_summary"
)

print(
    "- part1_alignment_check"
)

print(
    "- part1_shared_split_verification"
)

print(
    "- shared_split_manifest"
)


print(
    "\nExported files:"
)

print(
    "- part1_data_preparation_summary.csv"
)

print(
    "- part1_alignment_check.csv"
)

print(
    "- part1_shared_split_verification.csv"
)

print(
    "- part1_shared_split_manifest.csv"
)

## PART 2: PyCaret Model Development

In [ ]:
# ============================================================
# MDD DX ENGINE
# PART 2: PYCARET MODEL COMPARISON AND TUNING
# ============================================================
#
# Objectives:
# 1. Use only the original 2024 development cohort.
# 2. Keep the held-out 2024 cohort completely unseen.
# 3. Apply categorical-aware resampling inside CV training folds.
# 4. Use SMOTENC for mixed numerical/categorical strategies.
# 5. Use SMOTEN for categorical-only strategies.
# 6. Apply RandomUnderSampler after over-sampling.
# 7. Compare seven candidate algorithms using ROC AUC.
# 8. Retain and tune the five highest-performing algorithms.
# 9. Refit tuned models using the complete development cohort.
#
# IMPORTANT:
# - preprocessing=False prevents PyCaret from one-hot encoding
#   predictors before SMOTENC/SMOTEN.
# - All predictors are already represented numerically.
# - Resampling is performed by the PyCaret/imblearn pipeline
#   within model-training data only.
# - The study held-out 2024 cohort is not supplied to setup().
# ============================================================

import copy
import inspect
import warnings

import numpy as np
import pandas as pd

from pycaret.classification import (
    setup,
    compare_models,
    tune_model,
    finalize_model,
    pull
)

from imblearn.over_sampling import (
    SMOTE,
    SMOTENC,
    SMOTEN
)

from imblearn.under_sampling import (
    RandomUnderSampler
)

from imblearn.pipeline import (
    Pipeline as ImbPipeline
)


# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

RANDOM_STATE = 42

N_FOLDS = 5

PYCARET_TRAIN_SIZE = 0.90

N_TOP_MODELS = 5

N_TUNING_ITERATIONS = 30

SELECTION_METRIC = "AUC"

SMOTE_SAMPLING_STRATEGY = 0.20

UNDER_SAMPLING_STRATEGY = 0.50

SMOTE_K_NEIGHBORS = 5


# ============================================================
# 2. REQUIRED-OBJECT VALIDATION
# ============================================================

required_objects = [
    "prepared_data",
    "screening_strategies",
    "OUTCOME"
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


if not isinstance(prepared_data, dict):

    raise TypeError(
        "prepared_data must be a dictionary."
    )


if not prepared_data:

    raise ValueError(
        "prepared_data is empty."
    )


if set(prepared_data) != set(screening_strategies):

    raise ValueError(
        "prepared_data and screening_strategies contain "
        "different strategy names."
    )


# ============================================================
# 3. CANDIDATE ALGORITHMS
# ============================================================

candidate_models = [
    "et",
    "rf",
    "lightgbm",
    "catboost",
    "xgboost",
    "ada",
    "gbc"
]


# ============================================================
# 4. RESULT CONTAINERS
# ============================================================

top5_models = {}

tuned_models = {}

evaluation_models = {}

final_models = {}

pycaret_experiments = {}

hybrid_resamplers = {}

cv_results_rows = []

part2_summary_rows = []

part2_status_rows = []

part2_hyperparameter_rows = []


# ============================================================
# 5. HELPER: CREATE STRATEGY-SPECIFIC OVER-SAMPLER
# ============================================================

def create_strategy_oversampler(
    sampler_type,
    categorical_indices,
    random_state
):
    """
    Create a fresh, unfitted strategy-specific over-sampler.

    Parameters
    ----------
    sampler_type : str
        SMOTENC, SMOTEN, or SMOTE.

    categorical_indices : list of int
        Positions of categorical predictors in the original
        strategy-specific feature matrix.

    random_state : int
        Reproducibility seed.
    """

    sampler_type = str(
        sampler_type
    ).upper()

    categorical_indices = list(
        categorical_indices
    )

    if sampler_type == "SMOTENC":

        if not categorical_indices:

            raise ValueError(
                "SMOTENC requires at least one categorical "
                "predictor index."
            )

        return SMOTENC(

            categorical_features=
                categorical_indices,

            sampling_strategy=
                SMOTE_SAMPLING_STRATEGY,

            random_state=
                random_state,

            k_neighbors=
                SMOTE_K_NEIGHBORS
        )

    if sampler_type == "SMOTEN":

        return SMOTEN(

            sampling_strategy=
                SMOTE_SAMPLING_STRATEGY,

            random_state=
                random_state,

            k_neighbors=
                SMOTE_K_NEIGHBORS
        )

    if sampler_type == "SMOTE":

        return SMOTE(

            sampling_strategy=
                SMOTE_SAMPLING_STRATEGY,

            random_state=
                random_state,

            k_neighbors=
                SMOTE_K_NEIGHBORS
        )

    raise ValueError(
        f"Unsupported sampler type: {sampler_type}"
    )


# ============================================================
# 6. HELPER: CREATE HYBRID RESAMPLER
# ============================================================

def create_hybrid_resampler(
    sampler_type,
    categorical_indices,
    random_state
):
    """
    Create a fresh over-sampling plus random-under-sampling
    pipeline.

    Processing order:
        SMOTENC/SMOTEN/SMOTE
        -> RandomUnderSampler

    This object contains samplers only. The fitted classifier
    is managed by PyCaret.
    """

    over_sampler = create_strategy_oversampler(

        sampler_type=sampler_type,

        categorical_indices=categorical_indices,

        random_state=random_state
    )

    hybrid_resampler = ImbPipeline([

        (
            "over_sampler",
            over_sampler
        ),

        (
            "under_sampler",

            RandomUnderSampler(

                sampling_strategy=
                    UNDER_SAMPLING_STRATEGY,

                random_state=
                    random_state
            )
        )
    ])

    return hybrid_resampler


# ============================================================
# 7. HELPER: MODEL NAME
# ============================================================

def get_algorithm_name(model):
    """
    Return a stable algorithm name for dictionary storage.
    """

    if model is None:

        return "UnknownModel"

    return type(model).__name__


# ============================================================
# 8. HELPER: EXTRACT MODEL PARAMETERS
# ============================================================

def extract_model_parameters(
    model,
    model_name,
    algorithm_name
):
    """
    Extract final model parameters for reproducibility.
    """

    parameter_rows = []

    if model is None:

        return parameter_rows

    candidate_objects = [
        ("Finalized_Object", model)
    ]

    # PyCaret finalized objects may contain an estimator as
    # the final pipeline step.

    if hasattr(model, "steps"):

        try:

            final_step_name, final_estimator = (
                model.steps[-1]
            )

            candidate_objects.append(
                (
                    f"Final_Step:{final_step_name}",
                    final_estimator
                )
            )

        except Exception:

            pass

    for object_location, candidate_object in (
        candidate_objects
    ):

        if not hasattr(
            candidate_object,
            "get_params"
        ):

            continue

        try:

            parameters = (
                candidate_object
                .get_params(
                    deep=False
                )
            )

        except Exception:

            continue

        for parameter_name, parameter_value in (
            parameters.items()
        ):

            parameter_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Object_Location":
                    object_location,

                "Parameter":
                    parameter_name,

                "Value":
                    repr(parameter_value)
            })

    return parameter_rows


# ============================================================
# 9. HELPER: VALIDATE RAW NUMERICAL REPRESENTATION
# ============================================================

def validate_numeric_representation(
    X,
    model_name
):
    """
    Validate that predictors can be supplied to SMOTENC,
    SMOTEN, and candidate classifiers without PyCaret
    preprocessing.
    """

    nonnumeric_columns = [
        column
        for column in X.columns
        if not pd.api.types.is_numeric_dtype(
            X[column]
        )
    ]

    if nonnumeric_columns:

        raise TypeError(
            f"{model_name}: predictors must be represented "
            "numerically when preprocess=False. "
            f"Nonnumeric columns: {nonnumeric_columns}"
        )

    if X.isna().any().any():

        missing_counts = (
            X
            .isna()
            .sum()
        )

        missing_counts = missing_counts[
            missing_counts > 0
        ]

        raise ValueError(
            f"{model_name}: predictors contain missing "
            f"values:\n{missing_counts}"
        )

    values = X.to_numpy(
        dtype=float
    )

    if not np.isfinite(values).all():

        raise ValueError(
            f"{model_name}: predictors contain infinite "
            "or nonfinite values."
        )


# ============================================================
# 10. MODEL-DEVELOPMENT LOOP
# ============================================================

for model_index, (
    model_name,
    info
) in enumerate(
    prepared_data.items()
):

    print("\n" + "=" * 100)
    print(f"{model_name}: PYCARET MODEL DEVELOPMENT")
    print("=" * 100)

    strategy_info = (
        screening_strategies[
            model_name
        ]
    )

    expected_features = list(
        strategy_info[
            "features"
        ]
    )

    categorical_features = list(
        strategy_info[
            "categorical_features"
        ]
    )

    categorical_indices = list(
        strategy_info[
            "categorical_indices"
        ]
    )

    continuous_features = list(
        strategy_info[
            "continuous_features"
        ]
    )

    sampler_type = str(
        strategy_info[
            "sampler_type"
        ]
    ).upper()

    # --------------------------------------------------------
    # 10.1 Retrieve original, nonresampled development data
    # --------------------------------------------------------

    X_train_original = (
        info["X_train"]
        .copy()
        .reset_index(drop=True)
    )

    y_train_original = (
        info["y_train"]
        .copy()
        .astype(int)
        .reset_index(drop=True)
    )

    if (
        X_train_original
        .columns
        .tolist()
        != expected_features
    ):

        raise ValueError(
            f"{model_name}: development feature order "
            "does not match PART 0."
        )

    validate_numeric_representation(
        X=X_train_original,
        model_name=model_name
    )

    observed_classes = sorted(
        y_train_original
        .unique()
        .tolist()
    )

    if observed_classes != [0, 1]:

        raise ValueError(
            f"{model_name}: expected outcome classes [0, 1], "
            f"observed {observed_classes}."
        )

    if (
        int(y_train_original.sum())
        <= SMOTE_K_NEIGHBORS
    ):

        raise ValueError(
            f"{model_name}: insufficient development events "
            "for the selected k_neighbors value."
        )

    # --------------------------------------------------------
    # 10.2 Validate categorical positions
    # --------------------------------------------------------

    observed_categorical_features = [
        expected_features[index]
        for index in categorical_indices
    ]

    if (
        observed_categorical_features
        != categorical_features
    ):

        raise ValueError(
            f"{model_name}: categorical indices do not "
            "match categorical feature names."
        )

    if sampler_type == "SMOTENC":

        if not categorical_features:

            raise ValueError(
                f"{model_name}: SMOTENC was selected but "
                "no categorical predictors were identified."
            )

        if not continuous_features:

            raise ValueError(
                f"{model_name}: SMOTENC was selected but "
                "no continuous predictors were identified."
            )

    elif sampler_type == "SMOTEN":

        if continuous_features:

            raise ValueError(
                f"{model_name}: SMOTEN was selected despite "
                "the presence of continuous predictors."
            )

    elif sampler_type == "SMOTE":

        if categorical_features:

            raise ValueError(
                f"{model_name}: standard SMOTE was selected "
                "despite categorical predictors."
            )

    else:

        raise ValueError(
            f"{model_name}: unsupported sampler type "
            f"{sampler_type}."
        )

    # --------------------------------------------------------
    # 10.3 Create original modeling data
    # --------------------------------------------------------

    modeling_data = pd.concat(
        [
            X_train_original,
            y_train_original.rename(
                OUTCOME
            )
        ],
        axis=1
    )

    original_training_n = int(
        len(modeling_data)
    )

    original_training_events = int(
        modeling_data[
            OUTCOME
        ].sum()
    )

    original_training_prevalence = float(
        modeling_data[
            OUTCOME
        ].mean()
    )

    print(
        f"\nOriginal development cohort: "
        f"N={original_training_n:,}; "
        f"events={original_training_events:,}; "
        f"prevalence="
        f"{original_training_prevalence * 100:.3f}%"
    )

    print(
        f"Sampler: {sampler_type}"
    )

    print(
        f"Categorical indices: "
        f"{categorical_indices}"
    )

    # --------------------------------------------------------
    # 10.4 Create fresh strategy-specific resampler
    # --------------------------------------------------------

    model_random_state = (
        RANDOM_STATE
        + model_index
    )

    hybrid_resampler = create_hybrid_resampler(

        sampler_type=sampler_type,

        categorical_indices=
            categorical_indices,

        random_state=
            model_random_state
    )

    hybrid_resamplers[
        model_name
    ] = hybrid_resampler

    # --------------------------------------------------------
    # 10.5 PyCaret setup
    # --------------------------------------------------------
    #
    # preprocess=False is intentional:
    # - predictors are already represented numerically;
    # - SMOTENC categorical indices refer to the original
    #   strategy-specific feature matrix;
    # - no one-hot encoding is permitted before SMOTENC.
    #
    # The held-out 2024 cohort is not supplied as test_data.
    # --------------------------------------------------------

    print(
        "\nInitializing the PyCaret experiment..."
    )

    try:

        experiment = setup(

            data=modeling_data,

            target=OUTCOME,

            train_size=
                PYCARET_TRAIN_SIZE,

            session_id=
                model_random_state,

            index=False,

            preprocess=False,

            profile=False,

            html=False,

            verbose=False,

            fold_strategy=
                "stratifiedkfold",

            fold=N_FOLDS,

            fold_shuffle=True,

            data_split_shuffle=True,

            data_split_stratify=True,

            fix_imbalance=True,

            fix_imbalance_method=
                hybrid_resampler,

            n_jobs=-1,

            use_gpu=False
        )

    except Exception as error:

        part2_status_rows.append({

            "Screening_Model":
                model_name,

            "Stage":
                "Setup",

            "Algorithm":
                "",

            "Status":
                "Failed",

            "Sampler_Type":
                sampler_type,

            "Error":
                str(error)
        })

        raise RuntimeError(
            f"{model_name}: PyCaret setup failed."
        ) from error

    pycaret_experiments[
        model_name
    ] = experiment

    # --------------------------------------------------------
    # 10.6 Compare candidate algorithms
    # --------------------------------------------------------

    print(
        f"\nComparing candidate algorithms using "
        f"{N_FOLDS}-fold cross-validation..."
    )

    try:

        best_models = compare_models(

            include=candidate_models,

            sort=SELECTION_METRIC,

            n_select=N_TOP_MODELS,

            fold=N_FOLDS,

            cross_validation=True,

            errors="ignore",

            verbose=False
        )

    except Exception as error:

        part2_status_rows.append({

            "Screening_Model":
                model_name,

            "Stage":
                "Compare",

            "Algorithm":
                "",

            "Status":
                "Failed",

            "Sampler_Type":
                sampler_type,

            "Error":
                str(error)
        })

        raise RuntimeError(
            f"{model_name}: compare_models failed."
        ) from error

    if not isinstance(
        best_models,
        list
    ):

        best_models = [
            best_models
        ]

    if not best_models:

        raise RuntimeError(
            f"{model_name}: compare_models returned "
            "no successful candidate models."
        )

    top5_models[
        model_name
    ] = best_models

    # Store comparison table immediately after compare_models.

    try:

        compare_results = (
            pull()
            .copy()
        )

        compare_results[
            "Screening_Model"
        ] = model_name

        compare_results[
            "Stage"
        ] = "Compare"

        compare_results[
            "Sampler_Type"
        ] = sampler_type

        compare_results[
            "Selection_Metric"
        ] = SELECTION_METRIC

        cv_results_rows.append(
            compare_results
        )

    except Exception:

        pass

    print("\nRetained candidate algorithms:")

    for rank, candidate_model in enumerate(
        best_models,
        start=1
    ):

        print(
            f"{rank}. "
            f"{get_algorithm_name(candidate_model)}"
        )

    tuned_models[
        model_name
    ] = {}

    evaluation_models[
        model_name
    ] = {}

    # --------------------------------------------------------
    # 10.7 Tune and finalize retained algorithms
    # --------------------------------------------------------

    for rank, candidate_model in enumerate(
        best_models,
        start=1
    ):

        algorithm_name = (
            get_algorithm_name(
                candidate_model
            )
        )

        print(
            f"\n[{rank}/{len(best_models)}] "
            f"{algorithm_name}"
        )

        tuning_error = ""

        try:

            tuned_model = tune_model(

                candidate_model,

                optimize=
                    SELECTION_METRIC,

                choose_better=True,

                n_iter=
                    N_TUNING_ITERATIONS,

                fold=N_FOLDS,

                search_library=
                    "scikit-learn",

                search_algorithm=
                    "random",

                verbose=False,

                tuner_verbose=False,

                return_train_score=False
            )

            tuning_status = "Tuned"

            print(
                f"{algorithm_name}: tuning successful."
            )

        except Exception as error:

            tuned_model = candidate_model

            tuning_status = "Untuned_Fallback"

            tuning_error = str(error)

            warnings.warn(
                f"{model_name} | {algorithm_name}: "
                "tuning failed; the untuned candidate "
                f"was retained. Reason: {error}"
            )

        tuned_models[
            model_name
        ][
            algorithm_name
        ] = tuned_model

        # Store tuning table before finalize_model.

        try:

            tuning_results = (
                pull()
                .copy()
            )

            tuning_results[
                "Screening_Model"
            ] = model_name

            tuning_results[
                "Algorithm"
            ] = algorithm_name

            tuning_results[
                "Stage"
            ] = tuning_status

            tuning_results[
                "Sampler_Type"
            ] = sampler_type

            tuning_results[
                "Selection_Metric"
            ] = SELECTION_METRIC

            cv_results_rows.append(
                tuning_results
            )

        except Exception:

            pass

        # Refit the PyCaret model using all data supplied to
        # the development experiment.

        try:

            evaluation_model = finalize_model(
                tuned_model
            )

            refit_status = "Success"

            refit_error = ""

            evaluation_models[
                model_name
            ][
                algorithm_name
            ] = evaluation_model

            part2_hyperparameter_rows.extend(

                extract_model_parameters(

                    model=evaluation_model,

                    model_name=model_name,

                    algorithm_name=
                        algorithm_name
                )
            )

            print(
                f"{algorithm_name}: finalized using the "
                "complete development data supplied to "
                "the PyCaret experiment."
            )

        except Exception as error:

            evaluation_model = None

            refit_status = "Failed"

            refit_error = str(error)

            warnings.warn(
                f"{model_name} | {algorithm_name}: "
                f"finalize_model failed. Reason: {error}"
            )

        part2_summary_rows.append({

            "Screening_Model":
                model_name,

            "Type":
                strategy_info["type"],

            "Self_Report_Items":
                int(
                    strategy_info[
                        "additional_questions"
                    ]
                ),

            "Feature_Count":
                int(len(expected_features)),

            "Continuous_Feature_Count":
                int(len(continuous_features)),

            "Categorical_Feature_Count":
                int(len(categorical_features)),

            "Categorical_Indices":
                categorical_indices.copy(),

            "Sampler_Type":
                sampler_type,

            "Over_Sampling_Strategy":
                SMOTE_SAMPLING_STRATEGY,

            "Under_Sampling_Strategy":
                UNDER_SAMPLING_STRATEGY,

            "K_Neighbors":
                SMOTE_K_NEIGHBORS,

            "Algorithm":
                algorithm_name,

            "Initial_Rank":
                rank,

            "Tuning_Status":
                tuning_status,

            "Refit_Status":
                refit_status,

            "Development_N":
                original_training_n,

            "Development_Events":
                original_training_events,

            "Development_Prevalence":
                original_training_prevalence,

            "Heldout_Used":
                False
        })

        part2_status_rows.append({

            "Screening_Model":
                model_name,

            "Stage":
                "Tune_and_Finalize",

            "Algorithm":
                algorithm_name,

            "Status":
                (
                    "Success"
                    if refit_status == "Success"
                    else "Failed"
                ),

            "Sampler_Type":
                sampler_type,

            "Tuning_Status":
                tuning_status,

            "Tuning_Error":
                tuning_error,

            "Refit_Error":
                refit_error
        })


# ============================================================
# 11. BACKWARD-COMPATIBILITY ALIAS
# ============================================================

final_models = evaluation_models


# ============================================================
# 12. CREATE SUMMARY TABLES
# ============================================================

part2_summary = pd.DataFrame(
    part2_summary_rows
)

part2_status = pd.DataFrame(
    part2_status_rows
)

part2_hyperparameters = pd.DataFrame(
    part2_hyperparameter_rows
)


if cv_results_rows:

    part2_cv_results = pd.concat(
        cv_results_rows,
        ignore_index=True,
        sort=False
    )

else:

    part2_cv_results = pd.DataFrame()


# ============================================================
# 13. FINAL VERIFICATION
# ============================================================

expected_model_names = set(
    prepared_data.keys()
)


if set(top5_models.keys()) != expected_model_names:

    raise RuntimeError(
        "Not all screening strategies produced retained "
        "candidate algorithms."
    )


if set(tuned_models.keys()) != expected_model_names:

    raise RuntimeError(
        "Not all screening strategies produced tuned-model "
        "containers."
    )


if set(evaluation_models.keys()) != expected_model_names:

    raise RuntimeError(
        "Not all screening strategies produced finalized-model "
        "containers."
    )


verification_rows = []


for model_name in prepared_data:

    expected_sampler = (
        screening_strategies[
            model_name
        ]["sampler_type"]
    )

    retained_count = len(
        top5_models.get(
            model_name,
            []
        )
    )

    tuned_count = len(
        tuned_models.get(
            model_name,
            {}
        )
    )

    finalized_count = len(
        evaluation_models.get(
            model_name,
            {}
        )
    )

    sampler_object = (
        hybrid_resamplers[
            model_name
        ]
    )

    sampler_steps = [
        type(step_object).__name__
        for _, step_object in sampler_object.steps
    ]

    observed_over_sampler = (
        sampler_steps[0]
        if sampler_steps
        else ""
    )

    expected_over_sampler = (
        expected_sampler
    )

    sampler_correct = (
        observed_over_sampler
        == expected_over_sampler
    )

    verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Sampler":
            expected_over_sampler,

        "Observed_Sampler":
            observed_over_sampler,

        "Sampler_Correct":
            sampler_correct,

        "Top_Models":
            retained_count,

        "Tuned_Models":
            tuned_count,

        "Finalized_Models":
            finalized_count,

        "Expected_Top_Models":
            N_TOP_MODELS,

        "Complete":
            bool(
                sampler_correct
                and retained_count
                    == N_TOP_MODELS
                and tuned_count
                    == N_TOP_MODELS
                and finalized_count
                    == N_TOP_MODELS
            )
    })


part2_verification = pd.DataFrame(
    verification_rows
)


if not part2_verification[
    "Sampler_Correct"
].all():

    raise RuntimeError(
        "One or more screening strategies used an "
        "incorrect over-sampler."
    )


if not part2_verification[
    "Complete"
].all():

    incomplete_models = (
        part2_verification.loc[
            ~part2_verification["Complete"],
            "Screening_Model"
        ]
        .tolist()
    )

    raise RuntimeError(
        "One or more screening strategies did not produce "
        "five complete finalized candidate models: "
        f"{incomplete_models}"
    )


# ============================================================
# 14. EXPORT
# ============================================================

part2_summary.to_csv(
    "part2_model_development_summary.csv",
    index=False
)


part2_status.to_csv(
    "part2_model_development_status.csv",
    index=False
)


part2_hyperparameters.to_csv(
    "part2_final_model_hyperparameters.csv",
    index=False
)


part2_cv_results.to_csv(
    "part2_cross_validation_results.csv",
    index=False
)


part2_verification.to_csv(
    "part2_model_development_verification.csv",
    index=False
)


# ============================================================
# 15. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 100)
print("PART 2 COMPLETED")
print("=" * 100)


print(
    "\nResampling configuration:"
)

display(
    part2_verification[
        [
            "Screening_Model",
            "Expected_Sampler",
            "Observed_Sampler",
            "Sampler_Correct",
            "Top_Models",
            "Tuned_Models",
            "Finalized_Models",
            "Complete"
        ]
    ]
)


print(
    "\nModel-development summary:"
)

display(
    part2_summary
)


print(
    "\nMethodological safeguards:"
)

print(
    "- Only the original 2024 development cohort was "
    "supplied to PyCaret."
)

print(
    "- preprocess=False was used so that categorical "
    "indices remained aligned before resampling."
)

print(
    "- SMOTENC was used for mixed-type strategies."
)

print(
    "- SMOTEN was used for the categorical-only M4 strategy."
)

print(
    "- Random undersampling followed over-sampling."
)

print(
    "- Held-out 2024 observations were not supplied to setup()."
)

print(
    "- Temporal 2025 observations were not used."
)


print(
    "\nPrepared objects:"
)

print(
    "- top5_models"
)

print(
    "- tuned_models"
)

print(
    "- evaluation_models"
)

print(
    "- final_models"
)

print(
    "- hybrid_resamplers"
)

print(
    "- pycaret_experiments"
)

print(
    "- part2_summary"
)

print(
    "- part2_status"
)

print(
    "- part2_cv_results"
)

print(
    "- part2_hyperparameters"
)

print(
    "- part2_verification"
)

### PART 2B: Build Full Hybrid Pipelines

In [ ]:
# ============================================================
# MDD DX ENGINE
# PART 2B: BUILD CATEGORICAL-AWARE HYBRID PIPELINES
# ============================================================

import copy
import numpy as np
import pandas as pd

from sklearn.base import clone

from imblearn.over_sampling import (
    SMOTE,
    SMOTENC,
    SMOTEN
)

from imblearn.under_sampling import (
    RandomUnderSampler
)

from imblearn.pipeline import (
    Pipeline as ImbPipeline
)


# ============================================================
# 1. SETTINGS
# ============================================================

RANDOM_STATE = 42

SMOTE_SAMPLING_STRATEGY = 0.20

UNDER_SAMPLING_STRATEGY = 0.50

SMOTE_K_NEIGHBORS = 5


# ============================================================
# 2. REQUIRED-OBJECT VALIDATION
# ============================================================

required_objects = [
    "tuned_models",
    "prepared_data",
    "screening_strategies"
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


if not isinstance(
    tuned_models,
    dict
):

    raise TypeError(
        "tuned_models must be a dictionary."
    )


if not isinstance(
    prepared_data,
    dict
):

    raise TypeError(
        "prepared_data must be a dictionary."
    )


if not isinstance(
    screening_strategies,
    dict
):

    raise TypeError(
        "screening_strategies must be a dictionary."
    )


if set(prepared_data) != set(tuned_models):

    raise ValueError(
        "prepared_data and tuned_models contain "
        "different strategies."
    )


# ============================================================
# 3. RESULT CONTAINERS
# ============================================================

hybrid_pipeline_templates = {}

evaluation_models = {}

final_models = {}

part2b_pipeline_status_rows = []

part2b_pipeline_manifest_rows = []

part2b_category_validation_rows = []


# ============================================================
# 4. HELPER: CLONE ESTIMATOR SAFELY
# ============================================================

def safe_clone_estimator(
    estimator
):
    """
    Create an unfitted copy of an estimator.
    """

    try:

        cloned_estimator = clone(
            estimator
        )

        clone_method = "sklearn.clone"

    except Exception:

        cloned_estimator = copy.deepcopy(
            estimator
        )

        fitted_attributes = [
            "classes_",
            "feature_names_in_",
            "n_features_in_",
            "feature_importances_",
            "estimators_",
            "best_score_",
            "best_iteration_"
        ]

        for attribute_name in fitted_attributes:

            if hasattr(
                cloned_estimator,
                attribute_name
            ):

                try:

                    delattr(
                        cloned_estimator,
                        attribute_name
                    )

                except Exception:

                    pass

        clone_method = "copy.deepcopy"

    return (
        cloned_estimator,
        clone_method
    )


# ============================================================
# 5. HELPER: CREATE STRATEGY-SPECIFIC OVER-SAMPLER
# ============================================================

def create_strategy_oversampler(
    sampler_type,
    categorical_indices,
    random_state
):
    """
    Create a fresh, unfitted strategy-specific over-sampler.
    """

    sampler_type = str(
        sampler_type
    ).upper()

    categorical_indices = list(
        categorical_indices
    )

    if sampler_type == "SMOTENC":

        if not categorical_indices:

            raise ValueError(
                "SMOTENC requires categorical "
                "feature indices."
            )

        return SMOTENC(

            categorical_features=
                categorical_indices,

            sampling_strategy=
                SMOTE_SAMPLING_STRATEGY,

            random_state=
                random_state,

            k_neighbors=
                SMOTE_K_NEIGHBORS
        )

    if sampler_type == "SMOTEN":

        return SMOTEN(

            sampling_strategy=
                SMOTE_SAMPLING_STRATEGY,

            random_state=
                random_state,

            k_neighbors=
                SMOTE_K_NEIGHBORS
        )

    if sampler_type == "SMOTE":

        return SMOTE(

            sampling_strategy=
                SMOTE_SAMPLING_STRATEGY,

            random_state=
                random_state,

            k_neighbors=
                SMOTE_K_NEIGHBORS
        )

    raise ValueError(
        f"Unsupported sampler type: "
        f"{sampler_type}"
    )


# ============================================================
# 6. HELPER: BUILD FRESH HYBRID PIPELINE
# ============================================================

def build_hybrid_pipeline(
    estimator,
    sampler_type,
    categorical_indices,
    random_state
):
    """
    Construct a complete unfitted pipeline:

    over_sampler
        -> RandomUnderSampler
        -> estimator
    """

    (
        estimator_clone,
        clone_method
    ) = safe_clone_estimator(
        estimator
    )

    over_sampler = create_strategy_oversampler(

        sampler_type=
            sampler_type,

        categorical_indices=
            categorical_indices,

        random_state=
            random_state
    )

    pipeline = ImbPipeline([

        (
            "over_sampler",
            over_sampler
        ),

        (
            "under_sampler",

            RandomUnderSampler(

                sampling_strategy=
                    UNDER_SAMPLING_STRATEGY,

                random_state=
                    random_state
            )
        ),

        (
            "estimator",
            estimator_clone
        )
    ])

    return (
        pipeline,
        clone_method
    )


# ============================================================
# 7. HELPER: DESCRIBE PIPELINE
# ============================================================

def describe_hybrid_pipeline(
    pipeline
):
    """
    Return the ordered pipeline-step description.
    """

    return " | ".join(

        f"{step_name}:"
        f"{type(step_object).__name__}"

        for step_name, step_object
        in pipeline.steps
    )


# ============================================================
# 8. HELPER: VALIDATE CATEGORICAL VALUES
# ============================================================

def validate_resampled_categories(
    X,
    y,
    sampler_type,
    categorical_indices,
    random_state,
    feature_names
):
    """
    Verify that categorical-aware resampling does not create
    previously unobserved category values.
    """

    sampler = create_strategy_oversampler(

        sampler_type=
            sampler_type,

        categorical_indices=
            categorical_indices,

        random_state=
            random_state
    )

    X_resampled, y_resampled = (
        sampler.fit_resample(
            X,
            y
        )
    )

    X_resampled = pd.DataFrame(

        X_resampled,

        columns=
            feature_names
    )

    if sampler_type == "SMOTEN":

        indices_to_check = list(
            range(
                len(feature_names)
            )
        )

    elif sampler_type == "SMOTENC":

        indices_to_check = list(
            categorical_indices
        )

    else:

        indices_to_check = []

    validation_rows = []

    all_categories_valid = True

    for feature_index in indices_to_check:

        feature_name = (
            feature_names[
                feature_index
            ]
        )

        original_values = set(

            X.iloc[
                :,
                feature_index
            ]
            .dropna()
            .tolist()
        )

        resampled_values = set(

            X_resampled.iloc[
                :,
                feature_index
            ]
            .dropna()
            .tolist()
        )

        unexpected_values = sorted(
            resampled_values
            - original_values
        )

        category_valid = (
            len(unexpected_values) == 0
        )

        all_categories_valid = (
            all_categories_valid
            and category_valid
        )

        validation_rows.append({

            "Feature":
                feature_name,

            "Feature_Index":
                feature_index,

            "Original_Category_Count":
                len(original_values),

            "Resampled_Category_Count":
                len(resampled_values),

            "Unexpected_Values":
                unexpected_values,

            "Category_Valid":
                category_valid
        })

    return (
        X_resampled,
        pd.Series(y_resampled),
        validation_rows,
        all_categories_valid
    )


# ============================================================
# 9. BUILD AND FIT PIPELINES
# ============================================================

print("=" * 100)

print(
    "PART 2B: BUILD CATEGORICAL-AWARE "
    "HYBRID PIPELINES"
)

print("=" * 100)


for model_index, model_name in enumerate(
    prepared_data
):

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{model_name}: "
        "HYBRID PIPELINE CONSTRUCTION"
    )

    print("=" * 100)

    strategy_info = (
        screening_strategies[
            model_name
        ]
    )

    feature_names = list(
        strategy_info[
            "features"
        ]
    )

    categorical_names = list(
        strategy_info[
            "categorical_features"
        ]
    )

    categorical_indices = list(
        strategy_info[
            "categorical_indices"
        ]
    )

    continuous_names = list(
        strategy_info[
            "continuous_features"
        ]
    )

    sampler_type = str(
        strategy_info[
            "sampler_type"
        ]
    ).upper()

    X_train = (
        prepared_data[
            model_name
        ]["X_train"]
        .copy()
        .reset_index(
            drop=True
        )
    )

    y_train = (
        prepared_data[
            model_name
        ]["y_train"]
        .copy()
        .astype(int)
        .reset_index(
            drop=True
        )
    )

    # --------------------------------------------------------
    # Validate training data
    # --------------------------------------------------------

    if (
        X_train.columns.tolist()
        != feature_names
    ):

        raise ValueError(
            f"{model_name}: feature order does not "
            "match PART 0."
        )

    if len(X_train) != len(y_train):

        raise ValueError(
            f"{model_name}: X_train and y_train "
            "have different lengths."
        )

    if X_train.isna().any().any():

        raise ValueError(
            f"{model_name}: X_train contains "
            "missing values."
        )

    if y_train.isna().any():

        raise ValueError(
            f"{model_name}: y_train contains "
            "missing values."
        )

    if (
        int(y_train.sum())
        <= SMOTE_K_NEIGHBORS
    ):

        raise ValueError(
            f"{model_name}: insufficient minority "
            "events for the selected k_neighbors."
        )

    observed_categorical_names = [

        feature_names[index]

        for index in categorical_indices
    ]

    if (
        observed_categorical_names
        != categorical_names
    ):

        raise ValueError(
            f"{model_name}: categorical indices do "
            "not match categorical feature names."
        )

    # --------------------------------------------------------
    # Validate sampler configuration
    # --------------------------------------------------------

    if sampler_type == "SMOTENC":

        if (
            not categorical_names
            or not continuous_names
        ):

            raise ValueError(
                f"{model_name}: invalid mixed-feature "
                "SMOTENC configuration."
            )

    elif sampler_type == "SMOTEN":

        if continuous_names:

            raise ValueError(
                f"{model_name}: SMOTEN cannot include "
                "continuous predictors."
            )

    elif sampler_type == "SMOTE":

        if categorical_names:

            raise ValueError(
                f"{model_name}: standard SMOTE cannot "
                "include categorical predictors."
            )

    else:

        raise ValueError(
            f"{model_name}: unsupported sampler type "
            f"{sampler_type}."
        )

    hybrid_pipeline_templates[
        model_name
    ] = {}

    evaluation_models[
        model_name
    ] = {}

    # --------------------------------------------------------
    # Validate category preservation once per strategy
    # --------------------------------------------------------

    validation_random_state = (
        RANDOM_STATE
        + model_index * 1000
    )

    (
        X_over_sampled,
        y_over_sampled,
        category_validation_rows,
        categories_valid
    ) = validate_resampled_categories(

        X=X_train,

        y=y_train,

        sampler_type=
            sampler_type,

        categorical_indices=
            categorical_indices,

        random_state=
            validation_random_state,

        feature_names=
            feature_names
    )

    for validation_row in (
        category_validation_rows
    ):

        validation_row[
            "Screening_Model"
        ] = model_name

        validation_row[
            "Sampler_Type"
        ] = sampler_type

        part2b_category_validation_rows.append(
            validation_row
        )

    if not categories_valid:

        raise RuntimeError(
            f"{model_name}: invalid synthetic "
            "categories were detected."
        )

    original_class_counts = (
        y_train
        .value_counts()
        .sort_index()
        .to_dict()
    )

    resampled_class_counts = (
        y_over_sampled
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print(
        f"Sampler: {sampler_type}"
    )

    print(
        "Category preservation check: Passed"
    )

    print(
        "Original class counts: "
        f"{original_class_counts}"
    )

    print(
        "After over-sampling only: "
        f"{resampled_class_counts}"
    )

    # --------------------------------------------------------
    # Build every retained algorithm
    # --------------------------------------------------------

    for algorithm_index, (
        algorithm_name,
        tuned_estimator
    ) in enumerate(
        tuned_models[
            model_name
        ].items()
    ):

        model_random_state = (
            RANDOM_STATE
            + model_index * 100
            + algorithm_index
        )

        print(
            f"\nBuilding {algorithm_name}..."
        )

        try:

            # Unfitted template for genuine OOF

            (
                pipeline_template,
                clone_method
            ) = build_hybrid_pipeline(

                estimator=
                    tuned_estimator,

                sampler_type=
                    sampler_type,

                categorical_indices=
                    categorical_indices,

                random_state=
                    model_random_state
            )

            hybrid_pipeline_templates[
                model_name
            ][
                algorithm_name
            ] = pipeline_template

            # Independent pipeline fitted using the complete
            # development cohort

            fitted_pipeline = clone(
                pipeline_template
            )

            fitted_pipeline.fit(
                X_train,
                y_train
            )

            evaluation_models[
                model_name
            ][
                algorithm_name
            ] = fitted_pipeline

            # ------------------------------------------------
            # Verify pipeline
            # ------------------------------------------------

            expected_step_order = [
                "over_sampler",
                "under_sampler",
                "estimator"
            ]

            observed_step_order = list(
                pipeline_template
                .named_steps
                .keys()
            )

            correct_step_order = (
                observed_step_order
                == expected_step_order
            )

            observed_sampler_class = type(
                pipeline_template
                .named_steps[
                    "over_sampler"
                ]
            ).__name__

            correct_sampler_class = (
                observed_sampler_class
                == sampler_type
            )

            if not hasattr(
                fitted_pipeline,
                "predict_proba"
            ):

                raise TypeError(
                    f"{algorithm_name} does not support "
                    "predict_proba."
                )

            test_probabilities = np.asarray(

                fitted_pipeline.predict_proba(
                    X_train.iloc[:10]
                )
            )

            probabilities_valid = bool(

                test_probabilities.shape
                == (10, 2)

                and np.isfinite(
                    test_probabilities
                ).all()

                and (
                    test_probabilities >= 0
                ).all()

                and (
                    test_probabilities <= 1
                ).all()
            )

            if not correct_step_order:

                raise RuntimeError(
                    "Incorrect pipeline-step order."
                )

            if not correct_sampler_class:

                raise RuntimeError(
                    f"Expected {sampler_type}, "
                    f"observed {observed_sampler_class}."
                )

            if not probabilities_valid:

                raise RuntimeError(
                    "Invalid probability output."
                )

            pipeline_description = (
                describe_hybrid_pipeline(
                    pipeline_template
                )
            )

            part2b_pipeline_status_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Success",

                "Sampler_Type":
                    sampler_type,

                "Categorical_Feature_Count":
                    len(categorical_names),

                "Continuous_Feature_Count":
                    len(continuous_names),

                "Categorical_Indices":
                    categorical_indices.copy(),

                "Training_N":
                    int(len(y_train)),

                "Training_Events":
                    int(y_train.sum()),

                "Random_State":
                    model_random_state,

                "Clone_Method":
                    clone_method,

                "Correct_Step_Order":
                    correct_step_order,

                "Correct_Sampler_Class":
                    correct_sampler_class,

                "Probability_Check_Passed":
                    probabilities_valid,

                "Pipeline_Description":
                    pipeline_description,

                "Error":
                    ""
            })

            for step_order, (
                step_name,
                step_object
            ) in enumerate(
                pipeline_template.steps,
                start=1
            ):

                try:

                    step_parameters = repr(

                        step_object.get_params(
                            deep=False
                        )
                    )

                except Exception:

                    step_parameters = ""

                part2b_pipeline_manifest_rows.append({

                    "Screening_Model":
                        model_name,

                    "Algorithm":
                        algorithm_name,

                    "Sampler_Type":
                        sampler_type,

                    "Step_Order":
                        step_order,

                    "Step_Name":
                        step_name,

                    "Step_Class":
                        type(
                            step_object
                        ).__name__,

                    "Parameters":
                        step_parameters
                })

            print(
                "Success | "
                f"{pipeline_description}"
            )

        except Exception as error:

            part2b_pipeline_status_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Failed",

                "Sampler_Type":
                    sampler_type,

                "Categorical_Feature_Count":
                    len(categorical_names),

                "Continuous_Feature_Count":
                    len(continuous_names),

                "Categorical_Indices":
                    categorical_indices.copy(),

                "Training_N":
                    int(len(y_train)),

                "Training_Events":
                    int(y_train.sum()),

                "Random_State":
                    model_random_state,

                "Clone_Method":
                    "",

                "Correct_Step_Order":
                    False,

                "Correct_Sampler_Class":
                    False,

                "Probability_Check_Passed":
                    False,

                "Pipeline_Description":
                    "",

                "Error":
                    str(error)
            })

            print(
                f"Failed | Reason: {error}"
            )


# ============================================================
# 10. BACKWARD-COMPATIBILITY ALIAS
# ============================================================

final_models = evaluation_models


# ============================================================
# 11. CREATE RESULT TABLES
# ============================================================

part2b_pipeline_status = pd.DataFrame(
    part2b_pipeline_status_rows
)


part2b_pipeline_manifest = pd.DataFrame(
    part2b_pipeline_manifest_rows
)


part2b_category_validation = pd.DataFrame(
    part2b_category_validation_rows
)


# ============================================================
# 12. VERIFY ALL PIPELINES
# ============================================================

verification_rows = []


for model_name in prepared_data:

    expected_algorithms = set(
        tuned_models[
            model_name
        ].keys()
    )

    template_algorithms = set(
        hybrid_pipeline_templates.get(
            model_name,
            {}
        ).keys()
    )

    fitted_algorithms = set(
        evaluation_models.get(
            model_name,
            {}
        ).keys()
    )

    model_status = (

        part2b_pipeline_status.loc[

            part2b_pipeline_status[
                "Screening_Model"
            ] == model_name
        ]
    )

    pipeline_ready = bool(

        template_algorithms
        == expected_algorithms

        and fitted_algorithms
        == expected_algorithms

        and len(model_status)
        == len(expected_algorithms)

        and (
            model_status[
                "Status"
            ] == "Success"
        ).all()

        and model_status[
            "Correct_Step_Order"
        ].all()

        and model_status[
            "Correct_Sampler_Class"
        ].all()

        and model_status[
            "Probability_Check_Passed"
        ].all()
    )

    verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Sampler":
            screening_strategies[
                model_name
            ]["sampler_type"],

        "Expected_Algorithms":
            len(expected_algorithms),

        "Pipeline_Templates":
            len(template_algorithms),

        "Fitted_Evaluation_Models":
            len(fitted_algorithms),

        "Pipeline_Ready":
            pipeline_ready
    })


part2b_verification = pd.DataFrame(
    verification_rows
)


successful_pipelines = int(

    (
        part2b_pipeline_status[
            "Status"
        ] == "Success"
    ).sum()
)


failed_pipelines = int(

    (
        part2b_pipeline_status[
            "Status"
        ] == "Failed"
    ).sum()
)


if failed_pipelines > 0:

    display(

        part2b_pipeline_status.loc[

            part2b_pipeline_status[
                "Status"
            ] == "Failed",

            [
                "Screening_Model",
                "Algorithm",
                "Error"
            ]
        ]
    )

    raise RuntimeError(
        f"{failed_pipelines} hybrid pipelines failed."
    )


if not part2b_verification[
    "Pipeline_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies are "
        "not pipeline-ready."
    )


if (
    not part2b_category_validation.empty
    and not part2b_category_validation[
        "Category_Valid"
    ].all()
):

    raise RuntimeError(
        "Invalid synthetic categorical values detected."
    )


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "PART 2B PIPELINE STATUS"
)

print("=" * 100)


display(
    part2b_pipeline_status
)


print(
    "\nPART 2B STRATEGY VERIFICATION"
)


display(
    part2b_verification
)


print(
    "\nCATEGORICAL-VALUE PRESERVATION CHECK"
)


display(
    part2b_category_validation
)


# ============================================================
# 14. EXPORT
# ============================================================

part2b_pipeline_status.to_csv(

    "part2b_hybrid_pipeline_status.csv",

    index=False,

    encoding="utf-8-sig"
)


part2b_pipeline_manifest.to_csv(

    "part2b_hybrid_pipeline_manifest.csv",

    index=False,

    encoding="utf-8-sig"
)


part2b_category_validation.to_csv(

    "part2b_categorical_value_validation.csv",

    index=False,

    encoding="utf-8-sig"
)


part2b_verification.to_csv(

    "part2b_hybrid_pipeline_verification.csv",

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 15. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "PART 2B COMPLETED"
)

print("=" * 100)


print(
    f"Successful pipelines: "
    f"{successful_pipelines}"
)


print(
    f"Failed pipelines: "
    f"{failed_pipelines}"
)


print(
    "\nArtifacts created:"
)


print(
    "- hybrid_pipeline_templates"
)

print(
    "- evaluation_models"
)

print(
    "- final_models"
)

print(
    "- part2b_pipeline_status"
)

print(
    "- part2b_pipeline_manifest"
)

print(
    "- part2b_category_validation"
)

print(
    "- part2b_verification"
)


print(
    "\nRequired downstream usage:"
)


print(
    "- OOF generation must use "
    "hybrid_pipeline_templates."
)


print(
    "- Held-out and temporal validation must use "
    "evaluation_models."
)


print(
    "- Neither held-out nor temporal observations "
    "may be resampled."
)

## PART 3: Held-out Test Evaluation

### PART 3A-1A: GENERATE OOF PROBABILITIES

In [ ]:
# ============================================================
# PART 3A-1A: GENUINE 5-FOLD OOF PROBABILITY GENERATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)


# ============================================================
# 1. SETTINGS
# ============================================================

RANDOM_STATE = 42
N_FOLDS = 5


# ============================================================
# 2. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "hybrid_pipeline_templates",
    "prepared_data",
    "screening_strategies"
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 3. RESULT CONTAINERS
# ============================================================

oof_predictions = {}

oof_prediction_rows = []

oof_status_rows = []

oof_fold_manifest_rows = []


# ============================================================
# 4. POSITIVE-CLASS PROBABILITY
# ============================================================

def positive_class_probability(
    estimator,
    X
):
    """
    Return the probability corresponding to outcome class 1.
    """

    probabilities = np.asarray(
        estimator.predict_proba(X)
    )

    if probabilities.ndim != 2:

        raise ValueError(
            "predict_proba must return a "
            "two-dimensional array."
        )

    classes = np.asarray(
        estimator.classes_
    )

    positive_positions = np.flatnonzero(
        classes == 1
    )

    if len(positive_positions) != 1:

        raise ValueError(
            "Could not identify class 1 in estimator "
            f"classes: {classes}"
        )

    positive_probability = probabilities[
        :,
        int(positive_positions[0])
    ].astype(float)

    if not np.isfinite(
        positive_probability
    ).all():

        raise ValueError(
            "Positive-class probabilities contain "
            "nonfinite values."
        )

    if (
        (positive_probability < 0).any()
        or (positive_probability > 1).any()
    ):

        raise ValueError(
            "Positive-class probabilities are "
            "outside [0,1]."
        )

    return positive_probability


# ============================================================
# 5. DESCRIBE PIPELINE
# ============================================================

def describe_pipeline(
    pipeline
):
    """
    Return the ordered pipeline-step description.
    """

    if not hasattr(
        pipeline,
        "steps"
    ):

        return type(
            pipeline
        ).__name__

    return " | ".join(

        f"{step_name}:"
        f"{type(step_object).__name__}"

        for step_name, step_object
        in pipeline.steps
    )


# ============================================================
# 6. VALIDATE PIPELINE TEMPLATE
# ============================================================

def validate_pipeline_template(
    pipeline,
    model_name,
    algorithm_name
):
    """
    Verify the sampler and pipeline-step order before OOF.
    """

    if not hasattr(
        pipeline,
        "named_steps"
    ):

        raise TypeError(
            f"{model_name} | {algorithm_name}: "
            "object is not an imbalanced-learn pipeline."
        )

    expected_step_order = [
        "over_sampler",
        "under_sampler",
        "estimator"
    ]

    observed_step_order = list(
        pipeline
        .named_steps
        .keys()
    )

    if (
        observed_step_order
        != expected_step_order
    ):

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            f"expected steps {expected_step_order}, "
            f"observed {observed_step_order}."
        )

    expected_sampler = str(

        screening_strategies[
            model_name
        ]["sampler_type"]

    ).upper()

    observed_sampler = type(

        pipeline
        .named_steps[
            "over_sampler"
        ]

    ).__name__

    if observed_sampler != expected_sampler:

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            f"expected {expected_sampler}, "
            f"observed {observed_sampler}."
        )

    observed_under_sampler = type(

        pipeline
        .named_steps[
            "under_sampler"
        ]

    ).__name__

    if (
        observed_under_sampler
        != "RandomUnderSampler"
    ):

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            "invalid under-sampler."
        )

    return (
        True,
        expected_sampler
    )


# ============================================================
# 7. GENERATE GENUINE OOF PREDICTIONS
# ============================================================

def generate_oof_predictions(
    template,
    X,
    y,
    original_indices,
    cv,
    model_name,
    algorithm_name
):
    """
    Generate genuine OOF predictions.

    The complete pipeline is cloned and fitted separately
    within each training fold. Only the corresponding validation
    fold is predicted.
    """

    number_of_observations = len(y)

    oof_probabilities = np.full(
        number_of_observations,
        np.nan,
        dtype=float
    )

    oof_fold_ids = np.full(
        number_of_observations,
        -1,
        dtype=int
    )

    validation_counter = np.zeros(
        number_of_observations,
        dtype=int
    )

    fold_rows = []

    for fold_number, (
        training_indices,
        validation_indices
    ) in enumerate(
        cv.split(
            X,
            y
        ),
        start=1
    ):

        X_fold_train = (
            X.iloc[
                training_indices
            ]
            .copy()
        )

        y_fold_train = (
            y.iloc[
                training_indices
            ]
            .copy()
        )

        X_fold_validation = (
            X.iloc[
                validation_indices
            ]
            .copy()
        )

        y_fold_validation = (
            y.iloc[
                validation_indices
            ]
            .copy()
        )

        # Clone the entire unfitted pipeline.
        # The sampler is fitted only on X_fold_train.

        fold_pipeline = clone(
            template
        )

        fold_pipeline.fit(
            X_fold_train,
            y_fold_train
        )

        fold_probabilities = (
            positive_class_probability(
                fold_pipeline,
                X_fold_validation
            )
        )

        oof_probabilities[
            validation_indices
        ] = fold_probabilities

        oof_fold_ids[
            validation_indices
        ] = fold_number

        validation_counter[
            validation_indices
        ] += 1

        participant_overlap = bool(

            set(
                training_indices
            ).intersection(
                validation_indices
            )
        )

        fold_rows.append({

            "Screening_Model":
                model_name,

            "Algorithm":
                algorithm_name,

            "Fold":
                fold_number,

            "Training_N":
                int(
                    len(training_indices)
                ),

            "Training_Events":
                int(
                    y_fold_train.sum()
                ),

            "Validation_N":
                int(
                    len(validation_indices)
                ),

            "Validation_Events":
                int(
                    y_fold_validation.sum()
                ),

            "Validation_Probability_Min":
                float(
                    fold_probabilities.min()
                ),

            "Validation_Probability_Max":
                float(
                    fold_probabilities.max()
                ),

            "Training_Original_Indices":
                original_indices[
                    training_indices
                ].tolist(),

            "Validation_Original_Indices":
                original_indices[
                    validation_indices
                ].tolist(),

            "Training_Validation_Overlap":
                participant_overlap
        })

    if np.isnan(
        oof_probabilities
    ).any():

        raise RuntimeError(
            f"{model_name} | {algorithm_name}: "
            "missing OOF probabilities."
        )

    if not np.all(
        validation_counter == 1
    ):

        raise RuntimeError(
            f"{model_name} | {algorithm_name}: "
            "each participant was not predicted "
            "exactly once."
        )

    if (
        oof_fold_ids < 1
    ).any():

        raise RuntimeError(
            f"{model_name} | {algorithm_name}: "
            "missing fold assignments."
        )

    return {

        "y_true":
            y.to_numpy(
                dtype=int
            ),

        "y_prob":
            oof_probabilities,

        "fold_id":
            oof_fold_ids,

        "row_position":
            np.arange(
                number_of_observations,
                dtype=int
            ),

        "original_index":
            np.asarray(
                original_indices
            ),

        "validation_counter":
            validation_counter,

        "fold_results":
            pd.DataFrame(
                fold_rows
            )
    }


# ============================================================
# 8. SHARED CROSS-VALIDATION DEFINITION
# ============================================================

oof_cv = StratifiedKFold(

    n_splits=N_FOLDS,

    shuffle=True,

    random_state=RANDOM_STATE
)


# ============================================================
# 9. RUN OOF GENERATION
# ============================================================

print("=" * 100)

print(
    "PART 3A-1A: GENUINE OOF "
    "PROBABILITY GENERATION"
)

print("=" * 100)


for model_name in prepared_data:

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{model_name}: OOF GENERATION"
    )

    print("=" * 100)

    X_train = (
        prepared_data[
            model_name
        ]["X_train"]
        .copy()
        .reset_index(
            drop=True
        )
    )

    y_train = (
        prepared_data[
            model_name
        ]["y_train"]
        .copy()
        .astype(int)
        .reset_index(
            drop=True
        )
    )

    original_indices = np.asarray(

        prepared_data[
            model_name
        ]["train_original_index"]
    )

    expected_features = list(

        screening_strategies[
            model_name
        ]["features"]
    )

    if (
        X_train.columns.tolist()
        != expected_features
    ):

        raise ValueError(
            f"{model_name}: feature order does not "
            "match PART 0."
        )

    if (
        len(X_train) != len(y_train)
        or len(original_indices) != len(y_train)
    ):

        raise ValueError(
            f"{model_name}: development data "
            "are not aligned."
        )

    if (
        X_train.isna().any().any()
        or y_train.isna().any()
    ):

        raise ValueError(
            f"{model_name}: development data "
            "contain missing values."
        )

    print(
        f"Development N={len(y_train):,}; "
        f"events={int(y_train.sum()):,}; "
        f"prevalence="
        f"{y_train.mean() * 100:.3f}%"
    )

    oof_predictions[
        model_name
    ] = {}

    model_pipelines = (
        hybrid_pipeline_templates[
            model_name
        ]
    )

    if not model_pipelines:

        raise ValueError(
            f"{model_name}: no pipeline templates "
            "were found."
        )

    for algorithm_index, (
        algorithm_name,
        pipeline_template
    ) in enumerate(
        model_pipelines.items(),
        start=1
    ):

        print(
            f"\n[{algorithm_index}/"
            f"{len(model_pipelines)}] "
            f"{algorithm_name}"
        )

        pipeline_description = (
            describe_pipeline(
                pipeline_template
            )
        )

        try:

            (
                pipeline_validated,
                sampler_type
            ) = validate_pipeline_template(

                pipeline=
                    pipeline_template,

                model_name=
                    model_name,

                algorithm_name=
                    algorithm_name
            )

            prediction_result = (
                generate_oof_predictions(

                    template=
                        pipeline_template,

                    X=
                        X_train,

                    y=
                        y_train,

                    original_indices=
                        original_indices,

                    cv=
                        oof_cv,

                    model_name=
                        model_name,

                    algorithm_name=
                        algorithm_name
                )
            )

            y_true_oof = (
                prediction_result[
                    "y_true"
                ]
            )

            y_prob_oof = (
                prediction_result[
                    "y_prob"
                ]
            )

            overall_auc = float(

                roc_auc_score(
                    y_true_oof,
                    y_prob_oof
                )
            )

            overall_average_precision = float(

                average_precision_score(
                    y_true_oof,
                    y_prob_oof
                )
            )

            overall_brier_score = float(

                brier_score_loss(
                    y_true_oof,
                    y_prob_oof
                )
            )

            # -----------------------------------------------
            # Nested OOF record used by threshold selection
            # -----------------------------------------------

            oof_predictions[
                model_name
            ][
                algorithm_name
            ] = {

                "y_true":
                    y_true_oof,

                "y_prob":
                    y_prob_oof,

                "fold_id":
                    prediction_result[
                        "fold_id"
                    ],

                "row_position":
                    prediction_result[
                        "row_position"
                    ],

                "original_index":
                    prediction_result[
                        "original_index"
                    ],

                "n":
                    int(
                        len(y_true_oof)
                    ),

                "events":
                    int(
                        y_true_oof.sum()
                    ),

                "prevalence":
                    float(
                        y_true_oof.mean()
                    ),

                "oof_auc":
                    overall_auc,

                "oof_average_precision":
                    overall_average_precision,

                "oof_brier_score":
                    overall_brier_score,

                "cv_folds":
                    N_FOLDS,

                "cv_random_state":
                    RANDOM_STATE,

                "sampler_type":
                    sampler_type,

                "pipeline_description":
                    pipeline_description
            }

            # -----------------------------------------------
            # Participant-level prediction table
            # -----------------------------------------------

            oof_prediction_rows.append(

                pd.DataFrame({

                    "Screening_Model":
                        model_name,

                    "Algorithm":
                        algorithm_name,

                    "Row_Position":
                        prediction_result[
                            "row_position"
                        ],

                    "Original_Index":
                        prediction_result[
                            "original_index"
                        ],

                    "Fold":
                        prediction_result[
                            "fold_id"
                        ],

                    "y_true":
                        y_true_oof,

                    "y_prob":
                        y_prob_oof
                })
            )

            oof_fold_manifest_rows.append(

                prediction_result[
                    "fold_results"
                ]
            )

            oof_status_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Success",

                "Sampler_Type":
                    sampler_type,

                "N":
                    int(
                        len(y_true_oof)
                    ),

                "Events":
                    int(
                        y_true_oof.sum()
                    ),

                "Prevalence":
                    float(
                        y_true_oof.mean()
                    ),

                "OOF_AUC":
                    overall_auc,

                "OOF_Average_Precision":
                    overall_average_precision,

                "OOF_Brier_Score":
                    overall_brier_score,

                "Probability_Min":
                    float(
                        y_prob_oof.min()
                    ),

                "Probability_Median":
                    float(
                        np.median(
                            y_prob_oof
                        )
                    ),

                "Probability_Max":
                    float(
                        y_prob_oof.max()
                    ),

                "Predicted_Exactly_Once":
                    bool(
                        np.all(
                            prediction_result[
                                "validation_counter"
                            ] == 1
                        )
                    ),

                "Pipeline_Validated":
                    bool(
                        pipeline_validated
                    ),

                "Pipeline_Description":
                    pipeline_description,

                "Error":
                    ""
            })

            print(
                f"Success | "
                f"AUC={overall_auc:.4f} | "
                f"AP={overall_average_precision:.4f} | "
                f"Brier={overall_brier_score:.4f}"
            )

        except Exception as error:

            oof_status_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Failed",

                "Sampler_Type":
                    screening_strategies[
                        model_name
                    ]["sampler_type"],

                "N":
                    int(
                        len(y_train)
                    ),

                "Events":
                    int(
                        y_train.sum()
                    ),

                "Prevalence":
                    float(
                        y_train.mean()
                    ),

                "OOF_AUC":
                    np.nan,

                "OOF_Average_Precision":
                    np.nan,

                "OOF_Brier_Score":
                    np.nan,

                "Probability_Min":
                    np.nan,

                "Probability_Median":
                    np.nan,

                "Probability_Max":
                    np.nan,

                "Predicted_Exactly_Once":
                    False,

                "Pipeline_Validated":
                    False,

                "Pipeline_Description":
                    pipeline_description,

                "Error":
                    str(error)
            })

            print(
                f"Failed | Reason: {error}"
            )


# ============================================================
# 10. CREATE RESULT TABLES
# ============================================================

oof_status = pd.DataFrame(
    oof_status_rows
)


if oof_prediction_rows:

    oof_prediction_table = pd.concat(
        oof_prediction_rows,
        ignore_index=True
    )

else:

    oof_prediction_table = pd.DataFrame()


if oof_fold_manifest_rows:

    oof_fold_manifest = pd.concat(
        oof_fold_manifest_rows,
        ignore_index=True
    )

else:

    oof_fold_manifest = pd.DataFrame()


# ============================================================
# 11. VERIFY OOF COMPLETENESS
# ============================================================

verification_rows = []


for model_name in prepared_data:

    expected_algorithms = set(

        hybrid_pipeline_templates[
            model_name
        ].keys()
    )

    successful_algorithms = set(

        oof_predictions.get(
            model_name,
            {}
        ).keys()
    )

    expected_n = len(

        prepared_data[
            model_name
        ]["y_train"]
    )

    expected_events = int(

        prepared_data[
            model_name
        ]["y_train"]
        .sum()
    )

    all_n_correct = all(

        oof_predictions[
            model_name
        ][
            algorithm_name
        ]["n"] == expected_n

        for algorithm_name
        in successful_algorithms
    )

    all_event_counts_correct = all(

        oof_predictions[
            model_name
        ][
            algorithm_name
        ]["events"] == expected_events

        for algorithm_name
        in successful_algorithms
    )

    all_alignment_correct = all(

        np.array_equal(

            oof_predictions[
                model_name
            ][
                algorithm_name
            ]["y_true"],

            prepared_data[
                model_name
            ]["y_train"]
            .to_numpy(
                dtype=int
            )
        )

        for algorithm_name
        in successful_algorithms
    )

    all_pipelines_validated = True

    for algorithm_name in successful_algorithms:

        status_match = oof_status.loc[

            (
                oof_status[
                    "Screening_Model"
                ] == model_name
            )

            &

            (
                oof_status[
                    "Algorithm"
                ] == algorithm_name
            ),

            "Pipeline_Validated"
        ]

        if (
            status_match.empty
            or not bool(
                status_match.iloc[0]
            )
        ):

            all_pipelines_validated = False

    oof_ready = bool(

        successful_algorithms
        == expected_algorithms

        and all_n_correct

        and all_event_counts_correct

        and all_alignment_correct

        and all_pipelines_validated
    )

    verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            len(expected_algorithms),

        "Successful_Algorithms":
            len(successful_algorithms),

        "Missing_Algorithms":
            ", ".join(
                sorted(
                    expected_algorithms
                    - successful_algorithms
                )
            ),

        "All_N_Correct":
            all_n_correct,

        "All_Event_Counts_Correct":
            all_event_counts_correct,

        "All_Outcome_Alignment_Correct":
            all_alignment_correct,

        "All_Pipelines_Validated":
            all_pipelines_validated,

        "OOF_Ready":
            oof_ready
    })


part3a_oof_verification = pd.DataFrame(
    verification_rows
)


successful_algorithm_count = int(

    (
        oof_status[
            "Status"
        ] == "Success"
    ).sum()
)


failed_algorithm_count = int(

    (
        oof_status[
            "Status"
        ] == "Failed"
    ).sum()
)


# ============================================================
# 12. HARD SAFETY CHECKS
# ============================================================

if failed_algorithm_count > 0:

    display(

        oof_status.loc[

            oof_status[
                "Status"
            ] == "Failed",

            [
                "Screening_Model",
                "Algorithm",
                "Error"
            ]
        ]
    )

    raise RuntimeError(
        f"{failed_algorithm_count} OOF runs failed."
    )


if not part3a_oof_verification[
    "OOF_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies are "
        "not OOF-ready."
    )


if (
    not oof_fold_manifest.empty
    and oof_fold_manifest[
        "Training_Validation_Overlap"
    ].any()
):

    raise RuntimeError(
        "Training-validation participant overlap "
        "was detected in an OOF fold."
    )


# ============================================================
# 13. EXPORT
# ============================================================

oof_status.to_csv(

    "part3a_oof_status.csv",

    index=False,

    encoding="utf-8-sig"
)


oof_prediction_table.to_csv(

    "part3a_oof_predictions_long.csv",

    index=False,

    encoding="utf-8-sig"
)


oof_fold_manifest.to_csv(

    "part3a_oof_fold_manifest.csv",

    index=False,

    encoding="utf-8-sig"
)


part3a_oof_verification.to_csv(

    "part3a_oof_verification.csv",

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 14. FINAL OUTPUT
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "PART 3A-1A COMPLETED"
)

print("=" * 100)


print(
    f"Successful OOF runs: "
    f"{successful_algorithm_count}"
)


print(
    f"Failed OOF runs: "
    f"{failed_algorithm_count}"
)


print(
    "\nOOF performance:"
)


display(
    oof_status
)


print(
    "\nOOF verification:"
)


display(
    part3a_oof_verification
)


print(
    "\nArtifacts created:"
)


print(
    "- oof_predictions"
)

print(
    "- oof_prediction_table"
)

print(
    "- oof_status"
)

print(
    "- oof_fold_manifest"
)

print(
    "- part3a_oof_verification"
)

### Part 3A-2: Recall-constrained Threshold Locking

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 3A-2: RECALL-CONSTRAINED OOF THRESHOLD LOCKING
# ============================================================
#
# Objectives:
# 1. Use genuine OOF probabilities generated in PART 3A-1A
# 2. Select one decision threshold per model-algorithm pair
# 3. Require OOF sensitivity >= 0.50 whenever feasible
# 4. Maximize OOF F1 among thresholds meeting the constraint
# 5. Lock thresholds before held-out and temporal validation
# 6. Store OOF binary predictions and confusion matrices
#
# Prespecified threshold-selection rule:
#
#   Candidate thresholds
#           ↓
#   Sensitivity >= 0.50?
#           ↓
#   Yes: highest F1
#           ↓
#   Tie 1: higher precision
#           ↓
#   Tie 2: higher specificity
#           ↓
#   Tie 3: higher threshold
#
# If no candidate satisfies sensitivity >= 0.50:
#
#   Highest F1 overall
#           ↓
#   Higher sensitivity
#           ↓
#   Higher precision
#           ↓
#   Higher specificity
#           ↓
#   Higher threshold
#
# Inputs:
# - oof_predictions
# - oof_status
# - part3a_oof_verification
#
# Outputs:
# - locked_thresholds
# - threshold_summary
# - threshold_search_table
# - oof_predictions updated with threshold and y_pred
#
# IMPORTANT:
# - Held-out 2024 data are not used.
# - Temporal 2025 data are not used.
# - Thresholds must not be changed after this step.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)


# ============================================================
# 2. SETTINGS
# ============================================================

TARGET_RECALL = 0.50

PROBABILITY_UNIQUE_WARNING = 20

PROBABILITY_RANGE_WARNING = 1e-4

THRESHOLD_EPSILON = np.finfo(float).eps


# ============================================================
# 3. REQUIRED OBJECT VALIDATION
# ============================================================

required_objects = [
    "oof_predictions",
    "oof_status",
    "part3a_oof_verification"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )

if not isinstance(oof_predictions, dict):
    raise TypeError(
        "oof_predictions must be a dictionary."
    )

if len(oof_predictions) == 0:
    raise ValueError(
        "oof_predictions is empty. Run PART 3A-1A first."
    )


# ============================================================
# 4. VERIFY OOF READINESS
# ============================================================

if not part3a_oof_verification[
    "OOF_Ready"
].all():

    failed_strategies = (
        part3a_oof_verification.loc[
            ~part3a_oof_verification["OOF_Ready"],
            "Screening_Model"
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Threshold locking cannot proceed because the following "
        "strategies are not OOF-ready: "
        + ", ".join(failed_strategies)
    )


successful_status = oof_status[
    oof_status["Status"] == "Success"
].copy()

if successful_status.empty:
    raise RuntimeError(
        "No successful OOF algorithms are available."
    )


# ============================================================
# 5. RESULT CONTAINERS
# ============================================================

locked_thresholds = {}

threshold_summary_rows = []

threshold_search_rows = []

threshold_status_rows = []


# ============================================================
# 6. HELPER: CALCULATE THRESHOLD-BASED METRICS
# ============================================================

def calculate_threshold_metrics(
    y_true,
    y_prob,
    threshold
):
    """
    Calculate threshold-dependent classification metrics.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    negative_predictive_value = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )

    predicted_positive_n = int(
        np.sum(y_pred == 1)
    )

    predicted_positive_rate = float(
        np.mean(y_pred == 1)
    )

    return {
        "Threshold":
            float(threshold),

        "Sensitivity":
            float(sensitivity),

        "Recall":
            float(sensitivity),

        "Specificity":
            float(specificity),

        "Precision":
            float(precision),

        "Negative_Predictive_Value":
            float(negative_predictive_value),

        "F1":
            float(f1),

        "Accuracy":
            float(accuracy),

        "Balanced_Accuracy":
            float(balanced_accuracy),

        "Predicted_Positive_N":
            predicted_positive_n,

        "Predicted_Positive_Rate":
            predicted_positive_rate,

        "TP":
            int(tp),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TN":
            int(tn),

        "y_pred":
            y_pred
    }


# ============================================================
# 7. HELPER: CREATE EMPIRICAL THRESHOLD CANDIDATES
# ============================================================

def create_empirical_threshold_candidates(
    y_prob
):
    """
    Create candidate thresholds from all unique OOF probabilities.

    No probability rounding is applied.

    Two boundary thresholds are included:
    - slightly below the minimum: predicts all observations positive
    - slightly above the maximum: predicts all observations negative
    """

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    if not np.all(
        np.isfinite(y_prob)
    ):
        raise ValueError(
            "OOF probabilities contain nonfinite values."
        )

    if np.any(y_prob < 0) or np.any(y_prob > 1):
        raise ValueError(
            "OOF probabilities must lie within [0, 1]."
        )

    unique_probabilities = np.unique(
        y_prob
    )

    minimum_probability = float(
        np.min(unique_probabilities)
    )

    maximum_probability = float(
        np.max(unique_probabilities)
    )

    lower_boundary = np.nextafter(
        minimum_probability,
        -np.inf
    )

    upper_boundary = np.nextafter(
        maximum_probability,
        np.inf
    )

    lower_boundary = max(
        0.0,
        lower_boundary
    )

    upper_boundary = min(
        1.0 + THRESHOLD_EPSILON,
        upper_boundary
    )

    threshold_candidates = np.unique(
        np.concatenate([
            np.array(
                [lower_boundary],
                dtype=float
            ),
            unique_probabilities.astype(float),
            np.array(
                [upper_boundary],
                dtype=float
            )
        ])
    )

    threshold_candidates = np.sort(
        threshold_candidates
    )

    return threshold_candidates


# ============================================================
# 8. HELPER: SELECT THRESHOLD
# ============================================================

def select_locked_threshold(
    search_table,
    target_recall=0.50
):
    """
    Select a threshold using a prespecified sensitivity constraint
    and deterministic tie-breaking rules.
    """

    eligible = search_table[
        search_table["Sensitivity"] >= target_recall
    ].copy()

    if not eligible.empty:

        selected = (
            eligible
            .sort_values(
                by=[
                    "F1",
                    "Precision",
                    "Specificity",
                    "Threshold"
                ],
                ascending=[
                    False,
                    False,
                    False,
                    False
                ],
                kind="mergesort"
            )
            .iloc[0]
        )

        selection_rule = (
            f"Sensitivity>={target_recall:.2f}; "
            "max F1; tie: precision, specificity, threshold"
        )

        target_recall_achieved = True

    else:

        selected = (
            search_table
            .sort_values(
                by=[
                    "F1",
                    "Sensitivity",
                    "Precision",
                    "Specificity",
                    "Threshold"
                ],
                ascending=[
                    False,
                    False,
                    False,
                    False,
                    False
                ],
                kind="mergesort"
            )
            .iloc[0]
        )

        selection_rule = (
            "Fallback: max F1 overall; "
            "tie: sensitivity, precision, specificity, threshold"
        )

        target_recall_achieved = False

    return (
        selected,
        selection_rule,
        target_recall_achieved,
        int(len(eligible))
    )


# ============================================================
# 9. THRESHOLD LOCKING LOOP
# ============================================================

print("=" * 95)
print("PART 3A-2: OOF THRESHOLD LOCKING")
print("=" * 95)

print(
    f"Target OOF sensitivity: "
    f"{TARGET_RECALL:.2f}"
)


for model_name, algorithms in (
    oof_predictions.items()
):

    print("\n" + "=" * 95)
    print(f"{model_name}: THRESHOLD LOCKING")
    print("=" * 95)

    locked_thresholds[
        model_name
    ] = {}

    if len(algorithms) == 0:
        raise ValueError(
            f"{model_name} contains no OOF prediction records."
        )


    for algorithm_name, result in (
        algorithms.items()
    ):

        print(
            f"\nOptimizing: {algorithm_name}"
        )

        try:

            # ------------------------------------------------
            # Extract and validate OOF data
            # ------------------------------------------------

            y_true = np.asarray(
                result["y_true"],
                dtype=int
            )

            y_prob = np.asarray(
                result["y_prob"],
                dtype=float
            )

            if len(y_true) != len(y_prob):
                raise ValueError(
                    "y_true and y_prob have different lengths."
                )

            if len(np.unique(y_true)) != 2:
                raise ValueError(
                    "OOF outcome does not contain both classes."
                )

            if not np.all(
                np.isfinite(y_prob)
            ):
                raise ValueError(
                    "OOF probabilities contain nonfinite values."
                )

            if np.any(y_prob < 0) or np.any(y_prob > 1):
                raise ValueError(
                    "OOF probabilities are outside [0, 1]."
                )


            # ------------------------------------------------
            # Probability-distribution diagnostics
            # ------------------------------------------------

            unique_probability_count = int(
                len(np.unique(y_prob))
            )

            probability_minimum = float(
                np.min(y_prob)
            )

            probability_maximum = float(
                np.max(y_prob)
            )

            probability_range = float(
                probability_maximum
                -
                probability_minimum
            )

            probability_degenerate_flag = bool(
                unique_probability_count
                < PROBABILITY_UNIQUE_WARNING
                or probability_range
                < PROBABILITY_RANGE_WARNING
            )


            # ------------------------------------------------
            # Threshold-independent metrics
            # ------------------------------------------------

            auc = roc_auc_score(
                y_true,
                y_prob
            )

            average_precision = (
                average_precision_score(
                    y_true,
                    y_prob
                )
            )

            brier_score = (
                brier_score_loss(
                    y_true,
                    y_prob
                )
            )

            prevalence = float(
                np.mean(y_true)
            )


            # ------------------------------------------------
            # Create empirical threshold candidates
            # ------------------------------------------------

            threshold_candidates = (
                create_empirical_threshold_candidates(
                    y_prob
                )
            )

            if len(threshold_candidates) == 0:
                raise RuntimeError(
                    "No threshold candidates were generated."
                )


            # ------------------------------------------------
            # Evaluate all candidate thresholds
            # ------------------------------------------------

            candidate_rows = []

            for threshold_rank, threshold in enumerate(
                threshold_candidates,
                start=1
            ):

                metrics = (
                    calculate_threshold_metrics(
                        y_true=y_true,
                        y_prob=y_prob,
                        threshold=threshold
                    )
                )

                candidate_rows.append({

                    "Screening_Model":
                        model_name,

                    "Algorithm":
                        algorithm_name,

                    "Candidate_Number":
                        threshold_rank,

                    "Threshold":
                        metrics["Threshold"],

                    "Sensitivity":
                        metrics["Sensitivity"],

                    "Recall":
                        metrics["Recall"],

                    "Specificity":
                        metrics["Specificity"],

                    "Precision":
                        metrics["Precision"],

                    "Negative_Predictive_Value":
                        metrics[
                            "Negative_Predictive_Value"
                        ],

                    "F1":
                        metrics["F1"],

                    "Accuracy":
                        metrics["Accuracy"],

                    "Balanced_Accuracy":
                        metrics["Balanced_Accuracy"],

                    "Predicted_Positive_N":
                        metrics[
                            "Predicted_Positive_N"
                        ],

                    "Predicted_Positive_Rate":
                        metrics[
                            "Predicted_Positive_Rate"
                        ],

                    "TP":
                        metrics["TP"],

                    "FP":
                        metrics["FP"],

                    "FN":
                        metrics["FN"],

                    "TN":
                        metrics["TN"],

                    "Meets_Target_Recall":
                        bool(
                            metrics["Sensitivity"]
                            >= TARGET_RECALL
                        )
                })


            model_search_table = pd.DataFrame(
                candidate_rows
            )


            # ------------------------------------------------
            # Select and lock threshold
            # ------------------------------------------------

            (
                selected,
                selection_rule,
                target_recall_achieved,
                eligible_candidate_count
            ) = select_locked_threshold(
                search_table=model_search_table,
                target_recall=TARGET_RECALL
            )


            best_threshold = float(
                selected["Threshold"]
            )

            selected_metrics = (
                calculate_threshold_metrics(
                    y_true=y_true,
                    y_prob=y_prob,
                    threshold=best_threshold
                )
            )

            y_pred_locked = (
                selected_metrics["y_pred"]
            )


            # ------------------------------------------------
            # Confirm selected metrics match table
            # ------------------------------------------------

            if not np.isclose(
                selected_metrics["F1"],
                selected["F1"]
            ):
                raise RuntimeError(
                    "Selected F1 does not match recomputed F1."
                )

            if not np.isclose(
                selected_metrics["Sensitivity"],
                selected["Sensitivity"]
            ):
                raise RuntimeError(
                    "Selected sensitivity does not match "
                    "recomputed sensitivity."
                )


            # ------------------------------------------------
            # Store locked threshold
            # ------------------------------------------------

            locked_thresholds[
                model_name
            ][
                algorithm_name
            ] = best_threshold


            # ------------------------------------------------
            # Update OOF prediction record
            # ------------------------------------------------

            result[
                "threshold"
            ] = best_threshold

            result[
                "locked_threshold"
            ] = best_threshold

            result[
                "y_pred"
            ] = y_pred_locked

            result[
                "threshold_rule"
            ] = selection_rule

            result[
                "target_recall"
            ] = TARGET_RECALL

            result[
                "target_recall_achieved"
            ] = target_recall_achieved

            result[
                "oof_sensitivity"
            ] = selected_metrics["Sensitivity"]

            result[
                "oof_specificity"
            ] = selected_metrics["Specificity"]

            result[
                "oof_precision"
            ] = selected_metrics["Precision"]

            result[
                "oof_f1"
            ] = selected_metrics["F1"]

            result[
                "oof_accuracy"
            ] = selected_metrics["Accuracy"]

            result[
                "oof_balanced_accuracy"
            ] = selected_metrics[
                "Balanced_Accuracy"
            ]

            result[
                "oof_tp"
            ] = selected_metrics["TP"]

            result[
                "oof_fp"
            ] = selected_metrics["FP"]

            result[
                "oof_fn"
            ] = selected_metrics["FN"]

            result[
                "oof_tn"
            ] = selected_metrics["TN"]

            result[
                "probability_unique_count"
            ] = unique_probability_count

            result[
                "probability_range"
            ] = probability_range

            result[
                "probability_degenerate_flag"
            ] = probability_degenerate_flag


            # ------------------------------------------------
            # Mark selected candidate in search table
            # ------------------------------------------------

            model_search_table[
                "Selected"
            ] = np.isclose(
                model_search_table["Threshold"],
                best_threshold,
                rtol=0,
                atol=1e-15
            )

            threshold_search_rows.append(
                model_search_table
            )


            # ------------------------------------------------
            # Store summary row
            # ------------------------------------------------

            threshold_summary_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Training_N":
                    int(len(y_true)),

                "Training_Events":
                    int(np.sum(y_true == 1)),

                "Training_Prevalence":
                    prevalence,

                "Threshold":
                    best_threshold,

                "Locked_Threshold":
                    best_threshold,

                "OOF_Sensitivity":
                    selected_metrics["Sensitivity"],

                "OOF_Recall":
                    selected_metrics["Recall"],

                "OOF_Specificity":
                    selected_metrics["Specificity"],

                "OOF_Precision":
                    selected_metrics["Precision"],

                "OOF_NPV":
                    selected_metrics[
                        "Negative_Predictive_Value"
                    ],

                "OOF_F1":
                    selected_metrics["F1"],

                "OOF_Accuracy":
                    selected_metrics["Accuracy"],

                "OOF_Balanced_Accuracy":
                    selected_metrics[
                        "Balanced_Accuracy"
                    ],

                "OOF_AUC":
                    float(auc),

                "OOF_Average_Precision":
                    float(average_precision),

                "OOF_Brier_Score":
                    float(brier_score),

                "TP":
                    selected_metrics["TP"],

                "FP":
                    selected_metrics["FP"],

                "FN":
                    selected_metrics["FN"],

                "TN":
                    selected_metrics["TN"],

                "Predicted_Positive_N":
                    selected_metrics[
                        "Predicted_Positive_N"
                    ],

                "Predicted_Positive_Rate":
                    selected_metrics[
                        "Predicted_Positive_Rate"
                    ],

                "Target_Recall":
                    TARGET_RECALL,

                "Target_Recall_Achieved":
                    target_recall_achieved,

                "Eligible_Threshold_Count":
                    eligible_candidate_count,

                "Total_Threshold_Candidates":
                    int(
                        len(threshold_candidates)
                    ),

                "Unique_Probability_Count":
                    unique_probability_count,

                "Probability_Min":
                    probability_minimum,

                "Probability_Max":
                    probability_maximum,

                "Probability_Range":
                    probability_range,

                "Probability_Degenerate_Flag":
                    probability_degenerate_flag,

                "Threshold_Rule":
                    selection_rule,

                "Rule":
                    (
                        "Recall-constrained"
                        if target_recall_achieved
                        else "Fallback"
                    )
            })


            threshold_status_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Success",

                "Threshold":
                    best_threshold,

                "Target_Recall_Achieved":
                    target_recall_achieved,

                "Probability_Degenerate_Flag":
                    probability_degenerate_flag,

                "Error":
                    ""
            })


            print(
                f"Threshold={best_threshold:.8f} | "
                f"Sensitivity="
                f"{selected_metrics['Sensitivity']:.3f} | "
                f"Precision="
                f"{selected_metrics['Precision']:.3f} | "
                f"Specificity="
                f"{selected_metrics['Specificity']:.3f} | "
                f"F1={selected_metrics['F1']:.3f} | "
                f"TP={selected_metrics['TP']} | "
                f"FP={selected_metrics['FP']} | "
                f"{'Target achieved' if target_recall_achieved else 'Fallback'}"
            )


        except Exception as error:

            threshold_status_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Failed",

                "Threshold":
                    np.nan,

                "Target_Recall_Achieved":
                    False,

                "Probability_Degenerate_Flag":
                    np.nan,

                "Error":
                    str(error)
            })

            print("Failed")
            print(f"Reason: {error}")


# ============================================================
# 10. CREATE RESULT TABLES
# ============================================================

threshold_summary = pd.DataFrame(
    threshold_summary_rows
)

threshold_status = pd.DataFrame(
    threshold_status_rows
)


if len(threshold_search_rows) > 0:

    threshold_search_table = pd.concat(
        threshold_search_rows,
        ignore_index=True
    )

else:

    threshold_search_table = pd.DataFrame()


# ============================================================
# 11. FORMAT THRESHOLD FOR DISPLAY ONLY
# ============================================================

threshold_summary[
    "Threshold_Display"
] = threshold_summary[
    "Threshold"
].map(
    lambda value: f"{value:.8f}"
)


# ============================================================
# 12. VERIFICATION
# ============================================================

expected_algorithm_count = sum(
    len(algorithms)
    for algorithms in oof_predictions.values()
)

successful_threshold_count = int(
    (
        threshold_status["Status"]
        == "Success"
    ).sum()
)

failed_threshold_count = int(
    (
        threshold_status["Status"]
        == "Failed"
    ).sum()
)


verification_rows = []

for model_name, algorithms in (
    oof_predictions.items()
):

    expected_algorithms = set(
        algorithms.keys()
    )

    locked_algorithms = set(
        locked_thresholds.get(
            model_name,
            {}
        ).keys()
    )

    missing_algorithms = sorted(
        expected_algorithms
        -
        locked_algorithms
    )

    all_prediction_lengths_correct = True
    all_thresholds_finite = True
    all_predictions_binary = True
    all_confusion_counts_correct = True

    for algorithm_name in locked_algorithms:

        record = oof_predictions[
            model_name
        ][
            algorithm_name
        ]

        y_true = np.asarray(
            record["y_true"],
            dtype=int
        )

        y_pred = np.asarray(
            record["y_pred"],
            dtype=int
        )

        threshold = record[
            "locked_threshold"
        ]

        if len(y_true) != len(y_pred):
            all_prediction_lengths_correct = False

        if not np.isfinite(threshold):
            all_thresholds_finite = False

        if not set(
            np.unique(y_pred)
        ).issubset({0, 1}):
            all_predictions_binary = False

        confusion_total = (
            record["oof_tp"]
            + record["oof_fp"]
            + record["oof_fn"]
            + record["oof_tn"]
        )

        if confusion_total != len(y_true):
            all_confusion_counts_correct = False


    verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            int(len(expected_algorithms)),

        "Locked_Algorithms":
            int(len(locked_algorithms)),

        "Missing_Algorithms":
            ", ".join(
                missing_algorithms
            ),

        "All_Prediction_Lengths_Correct":
            all_prediction_lengths_correct,

        "All_Thresholds_Finite":
            all_thresholds_finite,

        "All_Predictions_Binary":
            all_predictions_binary,

        "All_Confusion_Counts_Correct":
            all_confusion_counts_correct,

        "Thresholds_Ready":
            (
                len(missing_algorithms) == 0
                and all_prediction_lengths_correct
                and all_thresholds_finite
                and all_predictions_binary
                and all_confusion_counts_correct
            )
    })


part3a_threshold_verification = pd.DataFrame(
    verification_rows
)


# ============================================================
# 13. DISPLAY SUMMARY
# ============================================================

print("\n" + "=" * 95)
print("THRESHOLD LOCKING SUMMARY")
print("=" * 95)

print("\nExecution status:")

print(
    threshold_status[
        "Status"
    ].value_counts(
        dropna=False
    )
)


print("\nSelection-rule counts:")

print(
    threshold_summary[
        "Rule"
    ].value_counts(
        dropna=False
    )
)


print("\nTarget-recall achievement:")

print(
    threshold_summary[
        "Target_Recall_Achieved"
    ].value_counts(
        dropna=False
    )
)


print("\nProbability-degeneracy flags:")

print(
    threshold_summary[
        "Probability_Degenerate_Flag"
    ].value_counts(
        dropna=False
    )
)


# ============================================================
# 14. DISPLAY THRESHOLD TABLE
# ============================================================

display_columns = [
    "Screening_Model",
    "Algorithm",
    "Threshold_Display",
    "OOF_Sensitivity",
    "OOF_Specificity",
    "OOF_Precision",
    "OOF_F1",
    "OOF_AUC",
    "OOF_Average_Precision",
    "Predicted_Positive_N",
    "TP",
    "FP",
    "FN",
    "TN",
    "Target_Recall_Achieved",
    "Probability_Degenerate_Flag",
    "Rule"
]


print("\n" + "=" * 95)
print("LOCKED THRESHOLD PERFORMANCE")
print("=" * 95)

display(
    threshold_summary[
        display_columns
    ]
    .sort_values(
        by=[
            "Screening_Model",
            "OOF_F1",
            "OOF_AUC"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .round(4)
    .reset_index(drop=True)
)


# ============================================================
# 15. DISPLAY VERIFICATION
# ============================================================

print("\n" + "=" * 95)
print("THRESHOLD VERIFICATION")
print("=" * 95)

display(
    part3a_threshold_verification
)


# ============================================================
# 16. HARD SAFETY CHECKS
# ============================================================

if failed_threshold_count > 0:

    failed_rows = threshold_status[
        threshold_status["Status"] == "Failed"
    ]

    display(
        failed_rows
    )

    raise RuntimeError(
        f"{failed_threshold_count} threshold-locking procedures "
        "failed. Do not proceed to held-out validation."
    )


if not part3a_threshold_verification[
    "Thresholds_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies do not have complete "
        "locked thresholds."
    )


if successful_threshold_count != expected_algorithm_count:

    raise RuntimeError(
        "The number of successfully locked thresholds does not "
        "match the number of successful OOF algorithms."
    )


# ============================================================
# 17. EXPORT RESULTS
# ============================================================

threshold_summary.to_csv(
    "part3a_threshold_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

threshold_status.to_csv(
    "part3a_threshold_status.csv",
    index=False,
    encoding="utf-8-sig"
)

threshold_search_table.to_csv(
    "part3a_threshold_search_all_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

part3a_threshold_verification.to_csv(
    "part3a_threshold_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 18. EXPORT UPDATED OOF PREDICTIONS
# ============================================================

updated_oof_rows = []

for model_name, algorithms in (
    oof_predictions.items()
):

    for algorithm_name, record in (
        algorithms.items()
    ):

        updated_oof_rows.append(
            pd.DataFrame({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Row_Position":
                    record["row_position"],

                "Original_Index":
                    record["original_index"],

                "Fold":
                    record["fold_id"],

                "y_true":
                    record["y_true"],

                "y_prob":
                    record["y_prob"],

                "Locked_Threshold":
                    record["locked_threshold"],

                "y_pred":
                    record["y_pred"]
            })
        )


oof_predictions_with_thresholds = pd.concat(
    updated_oof_rows,
    ignore_index=True
)


oof_predictions_with_thresholds.to_csv(
    "part3a_oof_predictions_with_locked_thresholds.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. FINAL REPORT
# ============================================================

print("\n" + "=" * 95)
print("PART 3A-2 COMPLETED")
print("=" * 95)

print(
    f"Expected algorithms       : "
    f"{expected_algorithm_count}"
)

print(
    f"Locked thresholds         : "
    f"{successful_threshold_count}"
)

print(
    f"Failed threshold searches : "
    f"{failed_threshold_count}"
)

print(
    f"Target recall achieved    : "
    f"{int(threshold_summary['Target_Recall_Achieved'].sum())}"
)

print(
    f"Fallback thresholds       : "
    f"{int((~threshold_summary['Target_Recall_Achieved']).sum())}"
)

print(
    f"Degenerate probability flags: "
    f"{int(threshold_summary['Probability_Degenerate_Flag'].sum())}"
)

print("\nArtifacts created:")

print("- locked_thresholds")
print("- threshold_summary")
print("- threshold_status")
print("- threshold_search_table")
print("- part3a_threshold_verification")
print("- oof_predictions_with_thresholds")

print("\nExported files:")

print("- part3a_threshold_summary.csv")
print("- part3a_threshold_status.csv")
print("- part3a_threshold_search_all_candidates.csv")
print("- part3a_threshold_verification.csv")
print("- part3a_oof_predictions_with_locked_thresholds.csv")

print(
    "\nAll thresholds were derived exclusively from genuine "
    "OOF predictions in the 2024 training partition."
)

print(
    "These thresholds are now locked and must be reused "
    "without modification in held-out 2024 and temporal 2025 "
    "validation."
)

### Part 3A Supplement — Primary MDD AdaBoost Settings for Appendix 7

In [ ]:
mdd_pipeline = hybrid_pipeline_templates["M2_Admin_2Q"]["AdaBoostClassifier"]
mdd_adaboost = mdd_pipeline.steps[-1][1]

assert type(mdd_adaboost).__name__ == "AdaBoostClassifier"
mdd_params = mdd_adaboost.get_params(deep=False)

print("n_estimators:", mdd_params["n_estimators"])
print("learning_rate:", mdd_params["learning_rate"])

### PART 3B: Held-out Test Evaluation

In [ ]:
# ============================================================
# PART 3B: GENUINE HELD-OUT 2024 EVALUATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef
)


# ============================================================
# 1. SETTINGS
# ============================================================

VALIDATION_SET_NAME = "Held-out 2024"

THRESHOLD_ATOL = 1e-12


# ============================================================
# 2. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "evaluation_models",
    "prepared_data",
    "screening_strategies",
    "locked_thresholds",
    "threshold_summary"
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 3. RESULT CONTAINERS
# ============================================================

heldout_predictions = {}

heldout_result_rows = []

heldout_prediction_rows = []

heldout_status_rows = []


# ============================================================
# 4. VALIDATE FITTED PIPELINE
# ============================================================

def validate_evaluation_pipeline(
    pipeline,
    model_name,
    algorithm_name
):
    """
    Validate the fitted categorical-aware pipeline.
    """

    if not hasattr(
        pipeline,
        "named_steps"
    ):

        raise TypeError(
            f"{model_name} | {algorithm_name}: "
            "invalid evaluation pipeline."
        )

    expected_step_order = [
        "over_sampler",
        "under_sampler",
        "estimator"
    ]

    observed_step_order = list(
        pipeline
        .named_steps
        .keys()
    )

    if observed_step_order != expected_step_order:

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            "incorrect pipeline-step order."
        )

    expected_sampler = str(
        screening_strategies[
            model_name
        ]["sampler_type"]
    ).upper()

    observed_sampler = type(
        pipeline
        .named_steps[
            "over_sampler"
        ]
    ).__name__

    if observed_sampler != expected_sampler:

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            f"expected {expected_sampler}, "
            f"observed {observed_sampler}."
        )

    observed_under_sampler = type(
        pipeline
        .named_steps[
            "under_sampler"
        ]
    ).__name__

    if observed_under_sampler != "RandomUnderSampler":

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            "invalid under-sampler."
        )

    pipeline_description = " | ".join(
        f"{step_name}:{type(step_object).__name__}"
        for step_name, step_object
        in pipeline.steps
    )

    fitted_state_note = (
        "Fitted on complete 2024 development cohort only"
    )

    return (
        pipeline_description,
        fitted_state_note
    )


# ============================================================
# 5. POSITIVE-CLASS PROBABILITY
# ============================================================

def positive_probability(
    estimator,
    X
):
    """
    Return the probability corresponding to class 1.
    """

    probabilities = np.asarray(
        estimator.predict_proba(X)
    )

    classes = np.asarray(
        estimator.classes_
    )

    positive_positions = np.flatnonzero(
        classes == 1
    )

    if probabilities.ndim != 2:

        raise ValueError(
            "predict_proba must return a "
            "two-dimensional array."
        )

    if len(positive_positions) != 1:

        raise ValueError(
            "Could not identify class 1 in "
            f"estimator classes: {classes}"
        )

    positive_probability = probabilities[
        :,
        int(positive_positions[0])
    ].astype(float)

    if not np.isfinite(
        positive_probability
    ).all():

        raise ValueError(
            "Probability output contains "
            "nonfinite values."
        )

    if (
        (positive_probability < 0).any()
        or (positive_probability > 1).any()
    ):

        raise ValueError(
            "Probability output is outside [0,1]."
        )

    return positive_probability


# ============================================================
# 6. VALIDATION METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_prob,
    threshold
):
    """
    Calculate threshold-independent and threshold-dependent
    performance metrics.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    if len(y_true) != len(y_prob):

        raise ValueError(
            "Outcome and probability lengths differ."
        )

    if len(np.unique(y_true)) != 2:

        raise ValueError(
            "The held-out outcome must contain both classes."
        )

    if not np.isfinite(
        y_prob
    ).all():

        raise ValueError(
            "Probabilities contain nonfinite values."
        )

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    negative_predictive_value = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    return {
        "y_pred":
            y_pred,

        "AUC":
            float(
                roc_auc_score(
                    y_true,
                    y_prob
                )
            ),

        "Average_Precision":
            float(
                average_precision_score(
                    y_true,
                    y_prob
                )
            ),

        "Brier_Score":
            float(
                brier_score_loss(
                    y_true,
                    y_prob
                )
            ),

        "Sensitivity":
            float(sensitivity),

        "Recall":
            float(sensitivity),

        "Specificity":
            float(specificity),

        "Precision":
            float(precision),

        "Negative_Predictive_Value":
            float(
                negative_predictive_value
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    zero_division=0
                )
            ),

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "Matthews_Correlation_Coefficient":
            float(
                matthews_corrcoef(
                    y_true,
                    y_pred
                )
            ),

        "Predicted_Positive_N":
            int(
                y_pred.sum()
            ),

        "Predicted_Positive_Rate":
            float(
                y_pred.mean()
            ),

        "TP":
            int(tp),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TN":
            int(tn)
    }


# ============================================================
# 7. RETRIEVE OOF THRESHOLD METADATA
# ============================================================

def get_threshold_metadata(
    model_name,
    algorithm_name
):
    """
    Retrieve exactly one development OOF threshold record.
    """

    matching_rows = threshold_summary.loc[
        (
            threshold_summary[
                "Screening_Model"
            ] == model_name
        )
        &
        (
            threshold_summary[
                "Algorithm"
            ] == algorithm_name
        )
    ]

    if len(matching_rows) != 1:

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            "expected exactly one threshold row, "
            f"found {len(matching_rows)}."
        )

    row = matching_rows.iloc[0]

    return {
        "Locked_Threshold":
            float(
                row[
                    "Locked_Threshold"
                ]
            ),

        "OOF_AUC":
            float(
                row[
                    "OOF_AUC"
                ]
            ),

        "OOF_Average_Precision":
            float(
                row[
                    "OOF_Average_Precision"
                ]
            ),

        "OOF_Sensitivity":
            float(
                row[
                    "OOF_Sensitivity"
                ]
            ),

        "OOF_Specificity":
            float(
                row[
                    "OOF_Specificity"
                ]
            ),

        "OOF_Precision":
            float(
                row[
                    "OOF_Precision"
                ]
            ),

        "OOF_F1":
            float(
                row[
                    "OOF_F1"
                ]
            ),

        "Threshold_Rule":
            str(
                row[
                    "Threshold_Rule"
                ]
            ),

        "Target_Recall_Achieved":
            bool(
                row[
                    "Target_Recall_Achieved"
                ]
            ),

        "Probability_Degenerate_Flag":
            bool(
                row[
                    "Probability_Degenerate_Flag"
                ]
            )
    }


# ============================================================
# 8. HELD-OUT EVALUATION LOOP
# ============================================================

print("=" * 100)

print(
    "PART 3B: GENUINE HELD-OUT "
    "2024 EVALUATION"
)

print("=" * 100)


for model_name in prepared_data:

    X_test = (
        prepared_data[
            model_name
        ]["X_test"]
        .copy()
        .reset_index(
            drop=True
        )
    )

    y_test = (
        prepared_data[
            model_name
        ]["y_test"]
        .copy()
        .astype(int)
        .reset_index(
            drop=True
        )
    )

    test_original_index = np.asarray(
        prepared_data[
            model_name
        ]["test_original_index"]
    )

    expected_features = list(
        screening_strategies[
            model_name
        ]["features"]
    )

    if X_test.columns.tolist() != expected_features:

        raise ValueError(
            f"{model_name}: held-out feature "
            "order mismatch."
        )

    if len(X_test) != len(y_test):

        raise ValueError(
            f"{model_name}: X_test and y_test "
            "lengths differ."
        )

    if len(test_original_index) != len(y_test):

        raise ValueError(
            f"{model_name}: held-out participant "
            "indices are not aligned."
        )

    if X_test.isna().any().any():

        raise ValueError(
            f"{model_name}: held-out predictors "
            "contain missing values."
        )

    if y_test.isna().any():

        raise ValueError(
            f"{model_name}: held-out outcome "
            "contains missing values."
        )

    fitted_algorithms = set(
        evaluation_models[
            model_name
        ].keys()
    )

    threshold_algorithms = set(
        locked_thresholds[
            model_name
        ].keys()
    )

    if fitted_algorithms != threshold_algorithms:

        raise ValueError(
            f"{model_name}: fitted algorithms and "
            "locked thresholds differ."
        )

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{model_name}: "
        f"N={len(y_test):,}; "
        f"events={int(y_test.sum()):,}; "
        f"prevalence="
        f"{y_test.mean() * 100:.3f}%"
    )

    print("=" * 100)

    heldout_predictions[
        model_name
    ] = {}

    for algorithm_name, fitted_pipeline in (
        evaluation_models[
            model_name
        ].items()
    ):

        try:

            (
                pipeline_description,
                fitted_state_note
            ) = validate_evaluation_pipeline(
                pipeline=fitted_pipeline,
                model_name=model_name,
                algorithm_name=algorithm_name
            )

            locked_threshold = float(
                locked_thresholds[
                    model_name
                ][
                    algorithm_name
                ]
            )

            threshold_metadata = (
                get_threshold_metadata(
                    model_name=model_name,
                    algorithm_name=algorithm_name
                )
            )

            if not np.isclose(
                locked_threshold,
                threshold_metadata[
                    "Locked_Threshold"
                ],
                rtol=0,
                atol=THRESHOLD_ATOL
            ):

                raise RuntimeError(
                    "locked_thresholds and "
                    "threshold_summary do not agree."
                )

            # Prediction only.
            # No fit, resampling, or threshold optimization.

            y_prob = positive_probability(
                fitted_pipeline,
                X_test
            )

            metrics = calculate_metrics(
                y_test,
                y_prob,
                locked_threshold
            )

            y_pred = metrics[
                "y_pred"
            ]

            heldout_predictions[
                model_name
            ][
                algorithm_name
            ] = {
                "validation_set":
                    VALIDATION_SET_NAME,

                "y_true":
                    y_test.to_numpy(
                        dtype=int
                    ),

                "y_prob":
                    y_prob,

                "y_pred":
                    y_pred,

                "threshold":
                    locked_threshold,

                "locked_threshold":
                    locked_threshold,

                "row_position":
                    np.arange(
                        len(y_test)
                    ),

                "original_index":
                    test_original_index.copy(),

                "n":
                    int(
                        len(y_test)
                    ),

                "events":
                    int(
                        y_test.sum()
                    ),

                "prevalence":
                    float(
                        y_test.mean()
                    ),

                "auc":
                    metrics["AUC"],

                "average_precision":
                    metrics[
                        "Average_Precision"
                    ],

                "brier_score":
                    metrics[
                        "Brier_Score"
                    ],

                "sensitivity":
                    metrics[
                        "Sensitivity"
                    ],

                "specificity":
                    metrics[
                        "Specificity"
                    ],

                "precision":
                    metrics[
                        "Precision"
                    ],

                "negative_predictive_value":
                    metrics[
                        "Negative_Predictive_Value"
                    ],

                "f1":
                    metrics["F1"],

                "accuracy":
                    metrics[
                        "Accuracy"
                    ],

                "balanced_accuracy":
                    metrics[
                        "Balanced_Accuracy"
                    ],

                "mcc":
                    metrics[
                        "Matthews_Correlation_Coefficient"
                    ],

                "predicted_positive_n":
                    metrics[
                        "Predicted_Positive_N"
                    ],

                "predicted_positive_rate":
                    metrics[
                        "Predicted_Positive_Rate"
                    ],

                "tp":
                    metrics["TP"],

                "fp":
                    metrics["FP"],

                "fn":
                    metrics["FN"],

                "tn":
                    metrics["TN"],

                "pipeline_description":
                    pipeline_description,

                "fitted_state_note":
                    fitted_state_note,

                "threshold_source":
                    (
                        "Genuine 5-fold OOF predictions "
                        "from the 2024 development cohort"
                    ),

                "model_training_source":
                    (
                        "Original 2024 development "
                        "cohort only"
                    )
            }

            heldout_prediction_rows.append(
                pd.DataFrame({
                    "Validation_Set":
                        VALIDATION_SET_NAME,

                    "Screening_Model":
                        model_name,

                    "Algorithm":
                        algorithm_name,

                    "Row_Position":
                        np.arange(
                            len(y_test)
                        ),

                    "Original_Index":
                        test_original_index,

                    "y_true":
                        y_test.to_numpy(
                            dtype=int
                        ),

                    "y_prob":
                        y_prob,

                    "Locked_Threshold":
                        locked_threshold,

                    "y_pred":
                        y_pred
                })
            )

            result_row = {
                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Sampler_Type":
                    screening_strategies[
                        model_name
                    ]["sampler_type"],

                "N":
                    int(
                        len(y_test)
                    ),

                "Events":
                    int(
                        y_test.sum()
                    ),

                "Prevalence":
                    float(
                        y_test.mean()
                    ),

                "Locked_Threshold":
                    locked_threshold
            }

            result_row.update({
                key: value
                for key, value
                in metrics.items()
                if key != "y_pred"
            })

            result_row.update({
                "OOF_AUC":
                    threshold_metadata[
                        "OOF_AUC"
                    ],

                "OOF_Average_Precision":
                    threshold_metadata[
                        "OOF_Average_Precision"
                    ],

                "OOF_Sensitivity":
                    threshold_metadata[
                        "OOF_Sensitivity"
                    ],

                "OOF_Specificity":
                    threshold_metadata[
                        "OOF_Specificity"
                    ],

                "OOF_Precision":
                    threshold_metadata[
                        "OOF_Precision"
                    ],

                "OOF_F1":
                    threshold_metadata[
                        "OOF_F1"
                    ],

                "AUC_Change_Heldout_minus_OOF":
                    (
                        metrics["AUC"]
                        - threshold_metadata[
                            "OOF_AUC"
                        ]
                    ),

                "Sensitivity_Change_Heldout_minus_OOF":
                    (
                        metrics["Sensitivity"]
                        - threshold_metadata[
                            "OOF_Sensitivity"
                        ]
                    ),

                "F1_Change_Heldout_minus_OOF":
                    (
                        metrics["F1"]
                        - threshold_metadata[
                            "OOF_F1"
                        ]
                    ),

                "Threshold_Rule":
                    threshold_metadata[
                        "Threshold_Rule"
                    ],

                "Target_Recall_Achieved_OOF":
                    threshold_metadata[
                        "Target_Recall_Achieved"
                    ],

                "Probability_Degenerate_Flag_OOF":
                    threshold_metadata[
                        "Probability_Degenerate_Flag"
                    ],

                "Pipeline_Description":
                    pipeline_description
            })

            heldout_result_rows.append(
                result_row
            )

            heldout_status_rows.append({
                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Success",

                "N":
                    int(
                        len(y_test)
                    ),

                "Events":
                    int(
                        y_test.sum()
                    ),

                "Locked_Threshold":
                    locked_threshold,

                "Pipeline_Validated":
                    True,

                "Prediction_Length_Correct":
                    bool(
                        len(y_prob)
                        == len(y_test)
                    ),

                "Outcome_Alignment_Correct":
                    True,

                "Error":
                    ""
            })

            print(
                f"{algorithm_name} | "
                f"AUC={metrics['AUC']:.4f} | "
                f"AP={metrics['Average_Precision']:.4f} | "
                f"Sensitivity={metrics['Sensitivity']:.3f} | "
                f"Specificity={metrics['Specificity']:.3f} | "
                f"Precision={metrics['Precision']:.3f} | "
                f"F1={metrics['F1']:.3f}"
            )

        except Exception as error:

            heldout_status_rows.append({
                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Failed",

                "N":
                    int(
                        len(y_test)
                    ),

                "Events":
                    int(
                        y_test.sum()
                    ),

                "Locked_Threshold":
                    locked_thresholds.get(
                        model_name,
                        {}
                    ).get(
                        algorithm_name,
                        np.nan
                    ),

                "Pipeline_Validated":
                    False,

                "Prediction_Length_Correct":
                    False,

                "Outcome_Alignment_Correct":
                    False,

                "Error":
                    str(error)
            })

            print(
                f"{algorithm_name} | "
                f"Failed: {error}"
            )


# ============================================================
# 9. CREATE RESULT TABLES
# ============================================================

heldout_results = pd.DataFrame(
    heldout_result_rows
)


heldout_status = pd.DataFrame(
    heldout_status_rows
)


if heldout_prediction_rows:

    heldout_prediction_table = pd.concat(
        heldout_prediction_rows,
        ignore_index=True
    )

else:

    heldout_prediction_table = pd.DataFrame()


# Backward-compatible alias

primary_heldout_predictions = (
    heldout_predictions
)


# ============================================================
# 10. SHARED ALIGNMENT VERIFICATION
# ============================================================

reference_model_name = next(
    iter(
        prepared_data
    )
)


reference_y_test = (
    prepared_data[
        reference_model_name
    ]["y_test"]
    .to_numpy(
        dtype=int
    )
)


reference_test_index = np.asarray(
    prepared_data[
        reference_model_name
    ]["test_original_index"]
)


verification_rows = []


for model_name in prepared_data:

    expected_algorithms = set(
        evaluation_models[
            model_name
        ].keys()
    )

    successful_algorithms = set(
        heldout_predictions.get(
            model_name,
            {}
        ).keys()
    )

    model_y_test = (
        prepared_data[
            model_name
        ]["y_test"]
        .to_numpy(
            dtype=int
        )
    )

    model_test_index = np.asarray(
        prepared_data[
            model_name
        ]["test_original_index"]
    )

    shared_alignment_correct = bool(
        np.array_equal(
            model_y_test,
            reference_y_test
        )
        and
        np.array_equal(
            model_test_index,
            reference_test_index
        )
    )

    all_prediction_lengths_correct = all(
        heldout_predictions[
            model_name
        ][
            algorithm_name
        ]["n"] == len(
            reference_y_test
        )
        for algorithm_name
        in successful_algorithms
    )

    all_thresholds_matched = all(
        np.isclose(
            heldout_predictions[
                model_name
            ][
                algorithm_name
            ]["locked_threshold"],
            locked_thresholds[
                model_name
            ][
                algorithm_name
            ],
            rtol=0,
            atol=THRESHOLD_ATOL
        )
        for algorithm_name
        in successful_algorithms
    )

    heldout_ready = bool(
        successful_algorithms
        == expected_algorithms
        and shared_alignment_correct
        and all_prediction_lengths_correct
        and all_thresholds_matched
    )

    verification_rows.append({
        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            len(
                expected_algorithms
            ),

        "Successful_Algorithms":
            len(
                successful_algorithms
            ),

        "Shared_Outcome_and_Index_Alignment":
            shared_alignment_correct,

        "All_Prediction_Lengths_Correct":
            all_prediction_lengths_correct,

        "All_Locked_Thresholds_Matched":
            all_thresholds_matched,

        "Heldout_Ready":
            heldout_ready
    })


part3b_heldout_verification = pd.DataFrame(
    verification_rows
)


# ============================================================
# 11. HARD SAFETY CHECKS
# ============================================================

successful_evaluation_count = int(
    (
        heldout_status[
            "Status"
        ] == "Success"
    ).sum()
)


failed_evaluation_count = int(
    (
        heldout_status[
            "Status"
        ] == "Failed"
    ).sum()
)


if failed_evaluation_count > 0:

    display(
        heldout_status.loc[
            heldout_status[
                "Status"
            ] == "Failed",
            [
                "Screening_Model",
                "Algorithm",
                "Error"
            ]
        ]
    )

    raise RuntimeError(
        f"{failed_evaluation_count} "
        "held-out evaluations failed."
    )


if not part3b_heldout_verification[
    "Heldout_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies failed "
        "held-out verification."
    )


# ============================================================
# 12. PUBLICATION TABLE
# ============================================================

publication_columns = [
    "Screening_Model",
    "Algorithm",
    "N",
    "Events",
    "Locked_Threshold",
    "AUC",
    "Average_Precision",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "TP",
    "FP",
    "FN",
    "TN"
]


heldout_publication_table = (
    heldout_results[
        publication_columns
    ]
    .copy()
)


# ============================================================
# 13. EXPORT
# ============================================================

heldout_results.to_csv(
    "part3b_heldout_results.csv",
    index=False,
    encoding="utf-8-sig"
)


heldout_status.to_csv(
    "part3b_heldout_status.csv",
    index=False,
    encoding="utf-8-sig"
)


heldout_prediction_table.to_csv(
    "part3b_heldout_predictions_long.csv",
    index=False,
    encoding="utf-8-sig"
)


part3b_heldout_verification.to_csv(
    "part3b_heldout_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


heldout_publication_table.to_csv(
    "part3b_heldout_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. FINAL OUTPUT
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PART 3B COMPLETED"
)


print("=" * 100)


print(
    "Successful evaluations: "
    f"{successful_evaluation_count}"
)


print(
    "Failed evaluations: "
    f"{failed_evaluation_count}"
)


print(
    "\nHeld-out performance:"
)


display(
    heldout_publication_table
)


print(
    "\nHeld-out verification:"
)


display(
    part3b_heldout_verification
)


print(
    "\nAll models were trained only on the "
    "2024 development cohort."
)


print(
    "All thresholds were locked using genuine "
    "development OOF predictions."
)


print(
    "The held-out cohort was neither fitted "
    "nor resampled."
)

### PART 3C: Bootstrap 95% CI สำหรับ Held-out metrics

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.3
# PART 3C: STRATIFIED BOOTSTRAP 95% CONFIDENCE INTERVALS
# FOR GENUINE HELD-OUT 2024 METRICS
# ============================================================
#
# Objectives:
# 1. Estimate percentile bootstrap 95% confidence intervals for
#    held-out performance metrics.
# 2. Use participant-level predictions generated in PART 3B.
# 3. Preserve the observed numbers of events and non-events in
#    every bootstrap replicate through stratified resampling.
# 4. Reuse the locked OOF threshold without modification.
# 5. Store bootstrap distributions, summary statistics,
#    execution status, verification outputs, and a compact
#    publication-ready table.
#
# Inputs:
# - heldout_predictions
# - heldout_results
# - part3b_heldout_verification
#
# Outputs:
# - part3c_bootstrap_results
# - part3c_bootstrap_distribution
# - part3c_bootstrap_status
# - part3c_bootstrap_verification
# - part3c_bootstrap_publication_table
#
# IMPORTANT:
# - No model fitting occurs in this part.
# - No new predictions are generated in this part.
# - No threshold optimization or recovery occurs in this part.
# - Confidence intervals quantify uncertainty conditional on
#   the fixed held-out predictions and locked threshold.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score
)

warnings.filterwarnings("ignore")


# ============================================================
# 2. SETTINGS
# ============================================================

N_BOOTSTRAP = 2000
RANDOM_STATE = 42

CI_LEVEL = 0.95
CI_LOWER_PERCENTILE = 2.5
CI_UPPER_PERCENTILE = 97.5

MINIMUM_VALID_BOOTSTRAPS = 1800

EXPECTED_HELDOUT_N = 1227
EXPECTED_HELDOUT_EVENTS = 14

THRESHOLD_ATOL = 1e-15

# Metrics required for the primary held-out CI analysis.
# These metrics remain estimable even when a classifier predicts
# every participant positive or every participant negative.
REQUIRED_CI_METRICS = [
    "AUC",
    "Average_Precision",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "Balanced_Accuracy",
    "Matthews_Correlation_Coefficient",
    "Brier_Score",
    "Youden_Index"
]

# Metrics that may be structurally undefined in some bootstrap
# samples (for example, NPV when there are no predicted negatives,
# or LR+/DOR when specificity equals 1). They are retained for
# reporting but do not determine whether the primary bootstrap
# analysis is ready.
OPTIONAL_CI_METRICS = [
    "Negative_Predictive_Value",
    "Positive_Likelihood_Ratio",
    "Negative_Likelihood_Ratio",
    "Diagnostic_Odds_Ratio"
]

BOOTSTRAP_METRICS = (
    REQUIRED_CI_METRICS
    +
    OPTIONAL_CI_METRICS
)


# ============================================================
# 3. REQUIRED OBJECT VALIDATION
# ============================================================

required_objects = [
    "heldout_predictions",
    "heldout_results",
    "part3b_heldout_verification"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )

if not isinstance(heldout_predictions, dict):
    raise TypeError(
        "heldout_predictions must be a dictionary."
    )

if not isinstance(heldout_results, pd.DataFrame):
    raise TypeError(
        "heldout_results must be a pandas DataFrame."
    )

if not isinstance(
    part3b_heldout_verification,
    pd.DataFrame
):
    raise TypeError(
        "part3b_heldout_verification must be a pandas DataFrame."
    )

if len(heldout_predictions) == 0:
    raise ValueError(
        "heldout_predictions is empty. Run PART 3B first."
    )

if "Heldout_Ready" not in (
    part3b_heldout_verification.columns
):
    raise KeyError(
        "part3b_heldout_verification does not contain "
        "'Heldout_Ready'."
    )

if not part3b_heldout_verification[
    "Heldout_Ready"
].all():
    failed_strategies = (
        part3b_heldout_verification.loc[
            ~part3b_heldout_verification[
                "Heldout_Ready"
            ],
            "Screening_Model"
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Bootstrap analysis cannot proceed because the "
        "following strategies are not held-out ready: "
        + ", ".join(failed_strategies)
    )


# ============================================================
# 4. RESULT CONTAINERS
# ============================================================

bootstrap_result_rows = []

bootstrap_distribution_rows = []

bootstrap_status_rows = []

bootstrap_verification_rows = []


# ============================================================
# 5. HELPER: SAFE RATIO
# ============================================================

def safe_ratio(
    numerator,
    denominator
):
    """
    Return numerator / denominator when the denominator is
    strictly positive; otherwise return NaN.
    """

    if denominator is None:
        return np.nan

    if not np.isfinite(denominator):
        return np.nan

    if denominator <= 0:
        return np.nan

    return float(
        numerator / denominator
    )


# ============================================================
# 6. HELPER: CALCULATE FIXED-THRESHOLD METRICS
# ============================================================

def calculate_fixed_threshold_metrics(
    y_true,
    y_prob,
    threshold
):
    """
    Calculate threshold-independent and threshold-dependent
    metrics using an already locked threshold.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    if len(y_true) != len(y_prob):
        raise ValueError(
            "y_true and y_prob have different lengths."
        )

    if len(np.unique(y_true)) != 2:
        raise ValueError(
            "Both outcome classes are required."
        )

    if not np.all(
        np.isfinite(y_prob)
    ):
        raise ValueError(
            "y_prob contains nonfinite values."
        )

    if np.any(y_prob < 0) or np.any(y_prob > 1):
        raise ValueError(
            "y_prob contains values outside [0, 1]."
        )

    if not np.isfinite(threshold):
        raise ValueError(
            "The locked threshold is not finite."
        )

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    specificity = safe_ratio(
        tn,
        tn + fp
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    negative_predictive_value = safe_ratio(
        tn,
        tn + fn
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )

    matthews_correlation_coefficient = (
        matthews_corrcoef(
            y_true,
            y_pred
        )
    )

    auc = roc_auc_score(
        y_true,
        y_prob
    )

    average_precision = average_precision_score(
        y_true,
        y_prob
    )

    brier_score = brier_score_loss(
        y_true,
        y_prob
    )

    youden_index = (
        sensitivity + specificity - 1
        if np.isfinite(specificity)
        else np.nan
    )

    positive_likelihood_ratio = safe_ratio(
        sensitivity,
        1 - specificity
        if np.isfinite(specificity)
        else np.nan
    )

    negative_likelihood_ratio = safe_ratio(
        1 - sensitivity,
        specificity
    )

    diagnostic_odds_ratio = safe_ratio(
        positive_likelihood_ratio,
        negative_likelihood_ratio
    )

    return {
        "AUC":
            float(auc),

        "Average_Precision":
            float(average_precision),

        "Sensitivity":
            float(sensitivity),

        "Specificity":
            float(specificity),

        "Precision":
            float(precision),

        "Negative_Predictive_Value":
            float(negative_predictive_value),

        "F1":
            float(f1),

        "Balanced_Accuracy":
            float(balanced_accuracy),

        "Matthews_Correlation_Coefficient":
            float(
                matthews_correlation_coefficient
            ),

        "Brier_Score":
            float(brier_score),

        "Youden_Index":
            float(youden_index),

        "Positive_Likelihood_Ratio":
            float(positive_likelihood_ratio)
            if np.isfinite(
                positive_likelihood_ratio
            )
            else np.nan,

        "Negative_Likelihood_Ratio":
            float(negative_likelihood_ratio)
            if np.isfinite(
                negative_likelihood_ratio
            )
            else np.nan,

        "Diagnostic_Odds_Ratio":
            float(diagnostic_odds_ratio)
            if np.isfinite(
                diagnostic_odds_ratio
            )
            else np.nan,

        "TP":
            int(tp),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TN":
            int(tn),

        "Predicted_Positive_N":
            int(np.sum(y_pred == 1)),

        "Predicted_Positive_Rate":
            float(np.mean(y_pred == 1))
    }


# ============================================================
# 7. HELPER: STRATIFIED BOOTSTRAP INDICES
# ============================================================

def generate_stratified_bootstrap_indices(
    y_true,
    rng
):
    """
    Sample events and non-events independently with replacement,
    preserving the original class counts in every replicate.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    negative_indices = np.where(
        y_true == 0
    )[0]

    positive_indices = np.where(
        y_true == 1
    )[0]

    if len(negative_indices) == 0:
        raise ValueError(
            "No negative observations are available."
        )

    if len(positive_indices) == 0:
        raise ValueError(
            "No positive observations are available."
        )

    sampled_negative_indices = rng.choice(
        negative_indices,
        size=len(negative_indices),
        replace=True
    )

    sampled_positive_indices = rng.choice(
        positive_indices,
        size=len(positive_indices),
        replace=True
    )

    sampled_indices = np.concatenate(
        [
            sampled_negative_indices,
            sampled_positive_indices
        ]
    )

    # Shuffle the combined sample so class ordering does not
    # influence downstream functions.
    rng.shuffle(
        sampled_indices
    )

    return sampled_indices


# ============================================================
# 8. HELPER: SUMMARIZE BOOTSTRAP METRIC
# ============================================================

def summarize_bootstrap_metric(
    values,
    point_estimate,
    metric_name
):
    """
    Summarize a bootstrap distribution using percentile CIs.
    """

    values = np.asarray(
        values,
        dtype=float
    )

    finite_values = values[
        np.isfinite(values)
    ]

    valid_n = int(
        len(finite_values)
    )

    invalid_n = int(
        len(values) - valid_n
    )

    if valid_n == 0:
        return {
            "Metric":
                metric_name,

            "Point_Estimate":
                float(point_estimate)
                if np.isfinite(point_estimate)
                else np.nan,

            "Bootstrap_Mean":
                np.nan,

            "Bootstrap_Median":
                np.nan,

            "Bootstrap_Standard_Error":
                np.nan,

            "CI_Lower":
                np.nan,

            "CI_Upper":
                np.nan,

            "Valid_Bootstrap_Replicates":
                0,

            "Invalid_Bootstrap_Replicates":
                invalid_n,

            "CI_Computed":
                False
        }

    bootstrap_mean = float(
        np.mean(finite_values)
    )

    bootstrap_median = float(
        np.median(finite_values)
    )

    bootstrap_standard_error = float(
        np.std(
            finite_values,
            ddof=1
        )
    ) if valid_n > 1 else np.nan

    ci_lower = float(
        np.percentile(
            finite_values,
            CI_LOWER_PERCENTILE
        )
    )

    ci_upper = float(
        np.percentile(
            finite_values,
            CI_UPPER_PERCENTILE
        )
    )

    return {
        "Metric":
            metric_name,

        "Point_Estimate":
            float(point_estimate)
            if np.isfinite(point_estimate)
            else np.nan,

        "Bootstrap_Mean":
            bootstrap_mean,

        "Bootstrap_Median":
            bootstrap_median,

        "Bootstrap_Standard_Error":
            bootstrap_standard_error,

        "CI_Lower":
            ci_lower,

        "CI_Upper":
            ci_upper,

        "Valid_Bootstrap_Replicates":
            valid_n,

        "Invalid_Bootstrap_Replicates":
            invalid_n,

        "CI_Computed":
            bool(
                valid_n >= MINIMUM_VALID_BOOTSTRAPS
            )
    }


# ============================================================
# 9. HELPER: PUBLICATION FORMAT
# ============================================================

def format_estimate_with_ci(
    estimate,
    lower,
    upper,
    digits=3
):
    """
    Format a point estimate and 95% CI for publication tables.
    """

    if (
        not np.isfinite(estimate)
        or
        not np.isfinite(lower)
        or
        not np.isfinite(upper)
    ):
        return "Not estimable"

    return (
        f"{estimate:.{digits}f} "
        f"({lower:.{digits}f}-{upper:.{digits}f})"
    )


# ============================================================
# 10. BOOTSTRAP LOOP
# ============================================================

print("=" * 100)
print(
    "PART 3C VERSION 3.3: STRATIFIED BOOTSTRAP "
    "95% CONFIDENCE INTERVALS"
)
print("=" * 100)

print(
    f"Requested bootstrap replicates: "
    f"{N_BOOTSTRAP:,}"
)

print(
    f"Confidence interval: "
    f"{CI_LEVEL * 100:.1f}% percentile interval"
)


global_algorithm_counter = 0


for model_index, (
    model_name,
    algorithms
) in enumerate(
    heldout_predictions.items()
):

    print("\n" + "=" * 100)
    print(f"{model_name}: HELD-OUT BOOTSTRAP")
    print("=" * 100)

    if len(algorithms) == 0:
        raise ValueError(
            f"{model_name} contains no held-out predictions."
        )

    for algorithm_index, (
        algorithm_name,
        record
    ) in enumerate(
        algorithms.items()
    ):

        global_algorithm_counter += 1

        print(
            f"\n[{algorithm_index + 1}/"
            f"{len(algorithms)}] "
            f"{algorithm_name}"
        )

        algorithm_start_time = time.time()

        try:

            # ------------------------------------------------
            # Extract and validate fixed predictions
            # ------------------------------------------------

            required_record_keys = [
                "y_true",
                "y_prob",
                "y_pred",
                "locked_threshold",
                "original_index"
            ]

            missing_record_keys = [
                key
                for key in required_record_keys
                if key not in record
            ]

            if missing_record_keys:
                raise KeyError(
                    "Missing held-out prediction fields: "
                    + ", ".join(missing_record_keys)
                )

            y_true = np.asarray(
                record["y_true"],
                dtype=int
            )

            y_prob = np.asarray(
                record["y_prob"],
                dtype=float
            )

            stored_y_pred = np.asarray(
                record["y_pred"],
                dtype=int
            )

            original_index = np.asarray(
                record["original_index"]
            )

            locked_threshold = float(
                record["locked_threshold"]
            )

            if not (
                len(y_true)
                == len(y_prob)
                == len(stored_y_pred)
                == len(original_index)
            ):
                raise ValueError(
                    "Held-out prediction arrays have "
                    "different lengths."
                )

            if len(np.unique(y_true)) != 2:
                raise ValueError(
                    "Both outcome classes are required."
                )

            if not np.all(
                np.isfinite(y_prob)
            ):
                raise ValueError(
                    "Held-out probabilities contain "
                    "nonfinite values."
                )

            if not np.isfinite(
                locked_threshold
            ):
                raise ValueError(
                    "The locked threshold is not finite."
                )

            recomputed_y_pred = (
                y_prob >= locked_threshold
            ).astype(int)

            if not np.array_equal(
                stored_y_pred,
                recomputed_y_pred
            ):
                raise RuntimeError(
                    "Stored held-out predictions do not match "
                    "the locked threshold."
                )

            heldout_n = int(
                len(y_true)
            )

            heldout_events = int(
                np.sum(y_true == 1)
            )

            heldout_nonevents = int(
                np.sum(y_true == 0)
            )

            heldout_prevalence = float(
                np.mean(y_true)
            )

            # ------------------------------------------------
            # Calculate observed point estimates
            # ------------------------------------------------

            observed_metrics = (
                calculate_fixed_threshold_metrics(
                    y_true=y_true,
                    y_prob=y_prob,
                    threshold=locked_threshold
                )
            )

            # ------------------------------------------------
            # Model-specific deterministic random generator
            # ------------------------------------------------

            algorithm_seed = (
                RANDOM_STATE
                + model_index * 10000
                + algorithm_index * 100
            )

            rng = np.random.default_rng(
                algorithm_seed
            )

            bootstrap_metric_values = {
                metric_name: []
                for metric_name in BOOTSTRAP_METRICS
            }

            valid_replicates = 0
            skipped_replicates = 0
            failed_replicates = 0

            skipped_reasons = {}

            # ------------------------------------------------
            # Generate bootstrap replicates
            # ------------------------------------------------

            for bootstrap_id in range(
                1,
                N_BOOTSTRAP + 1
            ):

                try:

                    sampled_indices = (
                        generate_stratified_bootstrap_indices(
                            y_true=y_true,
                            rng=rng
                        )
                    )

                    bootstrap_y_true = y_true[
                        sampled_indices
                    ]

                    bootstrap_y_prob = y_prob[
                        sampled_indices
                    ]

                    bootstrap_metrics = (
                        calculate_fixed_threshold_metrics(
                            y_true=bootstrap_y_true,
                            y_prob=bootstrap_y_prob,
                            threshold=locked_threshold
                        )
                    )

                    for metric_name in BOOTSTRAP_METRICS:

                        metric_value = bootstrap_metrics[
                            metric_name
                        ]

                        bootstrap_metric_values[
                            metric_name
                        ].append(
                            metric_value
                        )

                        bootstrap_distribution_rows.append({

                            "Validation_Set":
                                "Held-out 2024",

                            "Screening_Model":
                                model_name,

                            "Algorithm":
                                algorithm_name,

                            "Bootstrap_ID":
                                bootstrap_id,

                            "Random_Seed":
                                algorithm_seed,

                            "Metric":
                                metric_name,

                            "Value":
                                metric_value,

                            "Bootstrap_N":
                                int(
                                    len(
                                        bootstrap_y_true
                                    )
                                ),

                            "Bootstrap_Events":
                                int(
                                    np.sum(
                                        bootstrap_y_true == 1
                                    )
                                ),

                            "Bootstrap_Nonevents":
                                int(
                                    np.sum(
                                        bootstrap_y_true == 0
                                    )
                                )
                        })

                    valid_replicates += 1

                except ValueError as error:

                    skipped_replicates += 1

                    reason = str(error)

                    skipped_reasons[
                        reason
                    ] = (
                        skipped_reasons.get(
                            reason,
                            0
                        )
                        + 1
                    )

                    continue

                except Exception as error:

                    failed_replicates += 1

                    reason = (
                        "Unexpected bootstrap error: "
                        + str(error)
                    )

                    skipped_reasons[
                        reason
                    ] = (
                        skipped_reasons.get(
                            reason,
                            0
                        )
                        + 1
                    )

                    continue

            # ------------------------------------------------
            # Summarize each metric
            # ------------------------------------------------

            metric_summary_records = []

            for metric_name in BOOTSTRAP_METRICS:

                metric_summary = summarize_bootstrap_metric(
                    values=bootstrap_metric_values[
                        metric_name
                    ],
                    point_estimate=observed_metrics[
                        metric_name
                    ],
                    metric_name=metric_name
                )

                metric_summary.update({

                    "Validation_Set":
                        "Held-out 2024",

                    "Screening_Model":
                        model_name,

                    "Algorithm":
                        algorithm_name,

                    "N":
                        heldout_n,

                    "Events":
                        heldout_events,

                    "Nonevents":
                        heldout_nonevents,

                    "Prevalence":
                        heldout_prevalence,

                    "Locked_Threshold":
                        locked_threshold,

                    "CI_Level":
                        CI_LEVEL,

                    "CI_Method":
                        "Stratified percentile bootstrap",

                    "Bootstrap_Requested":
                        N_BOOTSTRAP,

                    "Bootstrap_Completed":
                        valid_replicates,

                    "Bootstrap_Skipped":
                        skipped_replicates,

                    "Bootstrap_Failed":
                        failed_replicates,

                    "Random_Seed":
                        algorithm_seed
                })

                metric_summary_records.append(
                    metric_summary
                )

                bootstrap_result_rows.append(
                    metric_summary
                )

            # ------------------------------------------------
            # Verify metric-level CI completeness
            # ------------------------------------------------

            metric_summary_table = pd.DataFrame(
                metric_summary_records
            )

            required_metric_summary = (
                metric_summary_table[
                    metric_summary_table[
                        "Metric"
                    ].isin(
                        REQUIRED_CI_METRICS
                    )
                ]
                .copy()
            )

            optional_metric_summary = (
                metric_summary_table[
                    metric_summary_table[
                        "Metric"
                    ].isin(
                        OPTIONAL_CI_METRICS
                    )
                ]
                .copy()
            )

            all_required_metrics_ci_computed = bool(
                required_metric_summary[
                    "CI_Computed"
                ].all()
            )

            minimum_required_metric_valid_replicates = int(
                required_metric_summary[
                    "Valid_Bootstrap_Replicates"
                ].min()
            )

            optional_metrics_not_estimable = (
                optional_metric_summary.loc[
                    ~optional_metric_summary[
                        "CI_Computed"
                    ],
                    "Metric"
                ]
                .astype(str)
                .tolist()
            )

            # ------------------------------------------------
            # Status record
            # ------------------------------------------------

            elapsed_seconds = float(
                time.time()
                -
                algorithm_start_time
            )

            skipped_reason_text = " | ".join(
                [
                    f"{reason}: {count}"
                    for reason, count
                    in skipped_reasons.items()
                ]
            )

            bootstrap_status_rows.append({

                "Validation_Set":
                    "Held-out 2024",

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    (
                        "Success"
                        if all_required_metrics_ci_computed
                        else "Insufficient valid replicates"
                    ),

                "N":
                    heldout_n,

                "Events":
                    heldout_events,

                "Nonevents":
                    heldout_nonevents,

                "Locked_Threshold":
                    locked_threshold,

                "Bootstrap_Requested":
                    N_BOOTSTRAP,

                "Bootstrap_Completed":
                    valid_replicates,

                "Bootstrap_Skipped":
                    skipped_replicates,

                "Bootstrap_Failed":
                    failed_replicates,

                "Minimum_Required_Metric_Valid_Replicates":
                    minimum_required_metric_valid_replicates,

                "All_Required_Metrics_CI_Computed":
                    all_required_metrics_ci_computed,

                "Optional_Metrics_Not_Estimable":
                    ", ".join(
                        optional_metrics_not_estimable
                    ),

                "Random_Seed":
                    algorithm_seed,

                "Elapsed_Seconds":
                    elapsed_seconds,

                "Skipped_Reasons":
                    skipped_reason_text,

                "Error":
                    ""
            })

            print(
                f"Completed={valid_replicates:,}/"
                f"{N_BOOTSTRAP:,} | "
                f"Skipped={skipped_replicates:,} | "
                f"Failed={failed_replicates:,} | "
                f"Required CIs="
                f"{all_required_metrics_ci_computed} | "
                f"Optional not estimable="
                f"{len(optional_metrics_not_estimable)}"
            )

        except Exception as error:

            elapsed_seconds = float(
                time.time()
                -
                algorithm_start_time
            )

            bootstrap_status_rows.append({

                "Validation_Set":
                    "Held-out 2024",

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Failed",

                "N":
                    (
                        len(record.get("y_true", []))
                        if isinstance(record, dict)
                        else np.nan
                    ),

                "Events":
                    (
                        int(
                            np.sum(
                                np.asarray(
                                    record.get(
                                        "y_true",
                                        []
                                    )
                                )
                                == 1
                            )
                        )
                        if isinstance(record, dict)
                        else np.nan
                    ),

                "Nonevents":
                    np.nan,

                "Locked_Threshold":
                    (
                        record.get(
                            "locked_threshold",
                            np.nan
                        )
                        if isinstance(record, dict)
                        else np.nan
                    ),

                "Bootstrap_Requested":
                    N_BOOTSTRAP,

                "Bootstrap_Completed":
                    0,

                "Bootstrap_Skipped":
                    0,

                "Bootstrap_Failed":
                    N_BOOTSTRAP,

                "Minimum_Required_Metric_Valid_Replicates":
                    0,

                "All_Required_Metrics_CI_Computed":
                    False,

                "Optional_Metrics_Not_Estimable":
                    "",

                "Random_Seed":
                    np.nan,

                "Elapsed_Seconds":
                    elapsed_seconds,

                "Skipped_Reasons":
                    "",

                "Error":
                    str(error)
            })

            print("Failed")
            print(f"Reason: {error}")


# ============================================================
# 11. CREATE RESULT TABLES
# ============================================================

part3c_bootstrap_results = pd.DataFrame(
    bootstrap_result_rows
)

part3c_bootstrap_distribution = pd.DataFrame(
    bootstrap_distribution_rows
)

part3c_bootstrap_status = pd.DataFrame(
    bootstrap_status_rows
)


# ============================================================
# 12. BOOTSTRAP VERIFICATION
# ============================================================

expected_algorithm_count = sum(
    len(algorithms)
    for algorithms in heldout_predictions.values()
)

successful_algorithm_count = int(
    (
        part3c_bootstrap_status[
            "Status"
        ] == "Success"
    ).sum()
)

failed_algorithm_count = int(
    (
        part3c_bootstrap_status[
            "Status"
        ] == "Failed"
    ).sum()
)

insufficient_algorithm_count = int(
    (
        part3c_bootstrap_status[
            "Status"
        ] == "Insufficient valid replicates"
    ).sum()
)


for model_name, algorithms in (
    heldout_predictions.items()
):

    expected_algorithms = set(
        algorithms.keys()
    )

    successful_algorithms = set(
        part3c_bootstrap_status.loc[
            (
                part3c_bootstrap_status[
                    "Screening_Model"
                ] == model_name
            )
            &
            (
                part3c_bootstrap_status[
                    "Status"
                ] == "Success"
            ),
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )

    missing_algorithms = sorted(
        expected_algorithms
        -
        successful_algorithms
    )

    model_result_table = (
        part3c_bootstrap_results[
            part3c_bootstrap_results[
                "Screening_Model"
            ] == model_name
        ]
    )

    expected_metric_rows = (
        len(expected_algorithms)
        *
        len(BOOTSTRAP_METRICS)
    )

    observed_metric_rows = int(
        len(model_result_table)
    )

    all_metrics_present = True

    for algorithm_name in expected_algorithms:

        algorithm_metric_set = set(
            model_result_table.loc[
                model_result_table[
                    "Algorithm"
                ] == algorithm_name,
                "Metric"
            ]
            .astype(str)
            .tolist()
        )

        if algorithm_metric_set != set(
            BOOTSTRAP_METRICS
        ):
            all_metrics_present = False

    required_model_result_table = (
        model_result_table[
            model_result_table[
                "Metric"
            ].isin(
                REQUIRED_CI_METRICS
            )
        ]
        .copy()
    )

    all_required_ci_computed = bool(
        required_model_result_table[
            "CI_Computed"
        ].all()
    ) if not required_model_result_table.empty else False

    all_required_valid_replicates_sufficient = bool(
        (
            required_model_result_table[
                "Valid_Bootstrap_Replicates"
            ]
            >= MINIMUM_VALID_BOOTSTRAPS
        ).all()
    ) if not required_model_result_table.empty else False

    optional_not_estimable_table = (
        model_result_table.loc[
            (
                model_result_table[
                    "Metric"
                ].isin(
                    OPTIONAL_CI_METRICS
                )
            )
            &
            (
                ~model_result_table[
                    "CI_Computed"
                ]
            ),
            [
                "Algorithm",
                "Metric"
            ]
        ]
        .copy()
    )

    if optional_not_estimable_table.empty:

        optional_not_estimable = []

    else:

        optional_not_estimable = (
            optional_not_estimable_table
            .astype(str)
            .apply(
                lambda row:
                    f"{row['Algorithm']}: {row['Metric']}",
                axis=1
            )
            .tolist()
        )

    point_estimates_match_part3b = True

    for algorithm_name in expected_algorithms:

        result_row = heldout_results[
            (
                heldout_results[
                    "Screening_Model"
                ] == model_name
            )
            &
            (
                heldout_results[
                    "Algorithm"
                ] == algorithm_name
            )
        ]

        if len(result_row) != 1:
            point_estimates_match_part3b = False
            continue

        result_row = result_row.iloc[0]

        metric_column_map = {
            "AUC":
                "AUC",

            "Average_Precision":
                "Average_Precision",

            "Sensitivity":
                "Sensitivity",

            "Specificity":
                "Specificity",

            "Precision":
                "Precision",

            "Negative_Predictive_Value":
                "Negative_Predictive_Value",

            "F1":
                "F1",

            "Balanced_Accuracy":
                "Balanced_Accuracy",

            "Matthews_Correlation_Coefficient":
                "Matthews_Correlation_Coefficient",

            "Brier_Score":
                "Brier_Score"
        }

        for metric_name, result_column in (
            metric_column_map.items()
        ):

            bootstrap_point_row = (
                model_result_table[
                    (
                        model_result_table[
                            "Algorithm"
                        ] == algorithm_name
                    )
                    &
                    (
                        model_result_table[
                            "Metric"
                        ] == metric_name
                    )
                ]
            )

            if len(bootstrap_point_row) != 1:
                point_estimates_match_part3b = False
                continue

            bootstrap_point = float(
                bootstrap_point_row.iloc[0][
                    "Point_Estimate"
                ]
            )

            part3b_point = float(
                result_row[
                    result_column
                ]
            )

            both_nan = (
                np.isnan(bootstrap_point)
                and
                np.isnan(part3b_point)
            )

            values_close = (
                np.isclose(
                    bootstrap_point,
                    part3b_point,
                    rtol=0,
                    atol=1e-12
                )
                if not both_nan
                else True
            )

            if not values_close:
                point_estimates_match_part3b = False

    bootstrap_verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            int(len(expected_algorithms)),

        "Successful_Algorithms":
            int(len(successful_algorithms)),

        "Missing_Algorithms":
            ", ".join(
                missing_algorithms
            ),

        "Expected_Metric_Rows":
            int(expected_metric_rows),

        "Observed_Metric_Rows":
            int(observed_metric_rows),

        "All_Metrics_Present":
            all_metrics_present,

        "All_Required_CI_Computed":
            all_required_ci_computed,

        "All_Required_Valid_Replicates_Sufficient":
            all_required_valid_replicates_sufficient,

        "Optional_Metrics_Not_Estimable":
            " | ".join(
                optional_not_estimable
            ),

        "Point_Estimates_Match_PART3B":
            point_estimates_match_part3b,

        "Heldout_Bootstrap_Ready":
            (
                len(missing_algorithms) == 0
                and observed_metric_rows
                == expected_metric_rows
                and all_metrics_present
                and all_required_ci_computed
                and all_required_valid_replicates_sufficient
                and point_estimates_match_part3b
            )
    })


part3c_bootstrap_verification = pd.DataFrame(
    bootstrap_verification_rows
)


# ============================================================
# 13. CREATE PUBLICATION TABLE
# ============================================================

publication_metric_order = [
    "AUC",
    "Average_Precision",
    "Sensitivity",
    "Specificity",
    "Precision",
    "Negative_Predictive_Value",
    "F1",
    "Balanced_Accuracy",
    "Matthews_Correlation_Coefficient",
    "Brier_Score",
    "Positive_Likelihood_Ratio",
    "Negative_Likelihood_Ratio",
    "Diagnostic_Odds_Ratio"
]

publication_rows = []


for model_name, algorithms in (
    heldout_predictions.items()
):

    for algorithm_name, record in (
        algorithms.items()
    ):

        row = {

            "Validation_Set":
                "Held-out 2024",

            "Screening_Model":
                model_name,

            "Algorithm":
                algorithm_name,

            "N":
                int(len(record["y_true"])),

            "Events":
                int(
                    np.sum(
                        np.asarray(
                            record["y_true"]
                        )
                        == 1
                    )
                ),

            "Prevalence":
                float(
                    np.mean(
                        np.asarray(
                            record["y_true"]
                        )
                    )
                ),

            "Locked_Threshold":
                float(
                    record[
                        "locked_threshold"
                    ]
                )
        }

        for metric_name in publication_metric_order:

            metric_row = (
                part3c_bootstrap_results[
                    (
                        part3c_bootstrap_results[
                            "Screening_Model"
                        ] == model_name
                    )
                    &
                    (
                        part3c_bootstrap_results[
                            "Algorithm"
                        ] == algorithm_name
                    )
                    &
                    (
                        part3c_bootstrap_results[
                            "Metric"
                        ] == metric_name
                    )
                ]
            )

            if len(metric_row) != 1:

                row[
                    metric_name
                ] = "Not estimable"

                row[
                    f"{metric_name}_Point"
                ] = np.nan

                row[
                    f"{metric_name}_CI_Lower"
                ] = np.nan

                row[
                    f"{metric_name}_CI_Upper"
                ] = np.nan

                continue

            metric_row = metric_row.iloc[0]

            point_estimate = float(
                metric_row[
                    "Point_Estimate"
                ]
            )

            ci_lower = float(
                metric_row[
                    "CI_Lower"
                ]
            )

            ci_upper = float(
                metric_row[
                    "CI_Upper"
                ]
            )

            digits = (
                2
                if metric_name
                in [
                    "Positive_Likelihood_Ratio",
                    "Negative_Likelihood_Ratio",
                    "Diagnostic_Odds_Ratio"
                ]
                else 3
            )

            row[
                metric_name
            ] = format_estimate_with_ci(
                estimate=point_estimate,
                lower=ci_lower,
                upper=ci_upper,
                digits=digits
            )

            row[
                f"{metric_name}_Point"
            ] = point_estimate

            row[
                f"{metric_name}_CI_Lower"
            ] = ci_lower

            row[
                f"{metric_name}_CI_Upper"
            ] = ci_upper

        publication_rows.append(
            row
        )


part3c_bootstrap_publication_table = (
    pd.DataFrame(
        publication_rows
    )
)


# ============================================================
# 14. DISPLAY STATUS
# ============================================================

print("\n" + "=" * 100)
print("PART 3C BOOTSTRAP STATUS")
print("=" * 100)

print(
    part3c_bootstrap_status[
        "Status"
    ].value_counts(
        dropna=False
    )
)


# ============================================================
# 15. DISPLAY CORE METRIC SUMMARY
# ============================================================

core_display_metrics = [
    "AUC",
    "Average_Precision",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "Brier_Score"
]

core_metric_summary = (
    part3c_bootstrap_results[
        part3c_bootstrap_results[
            "Metric"
        ].isin(
            core_display_metrics
        )
    ]
    [
        [
            "Screening_Model",
            "Algorithm",
            "Metric",
            "Point_Estimate",
            "CI_Lower",
            "CI_Upper",
            "Bootstrap_Mean",
            "Bootstrap_Median",
            "Bootstrap_Standard_Error",
            "Valid_Bootstrap_Replicates"
        ]
    ]
    .copy()
)

print("\n" + "=" * 100)
print("HELD-OUT BOOTSTRAP 95% CI SUMMARY")
print("=" * 100)

display(
    core_metric_summary
    .sort_values(
        by=[
            "Screening_Model",
            "Algorithm",
            "Metric"
        ]
    )
    .round(4)
    .reset_index(drop=True)
)


# ============================================================
# 16. DISPLAY VERIFICATION
# ============================================================

print("\n" + "=" * 100)
print("PART 3C BOOTSTRAP VERIFICATION")
print("=" * 100)

display(
    part3c_bootstrap_verification
)


# ============================================================
# 17. HARD SAFETY CHECKS
# ============================================================

if failed_algorithm_count > 0:

    failed_rows = (
        part3c_bootstrap_status[
            part3c_bootstrap_status[
                "Status"
            ] == "Failed"
        ]
    )

    display(
        failed_rows
    )

    raise RuntimeError(
        f"{failed_algorithm_count} held-out bootstrap "
        "analyses failed."
    )


if insufficient_algorithm_count > 0:

    insufficient_rows = (
        part3c_bootstrap_status[
            part3c_bootstrap_status[
                "Status"
            ]
            == "Insufficient valid replicates"
        ]
    )

    display(
        insufficient_rows
    )

    raise RuntimeError(
        f"{insufficient_algorithm_count} held-out bootstrap "
        "analyses had fewer than the required number of valid "
        "replicates."
    )


if successful_algorithm_count != (
    expected_algorithm_count
):

    raise RuntimeError(
        "The number of successful bootstrap analyses does not "
        "match the number of held-out model-algorithm pairs."
    )


if not part3c_bootstrap_verification[
    "Heldout_Bootstrap_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies failed bootstrap "
        "verification."
    )


# ============================================================
# 18. COHORT WARNINGS
# ============================================================

reference_model_name = next(
    iter(
        heldout_predictions
    )
)

reference_algorithm_name = next(
    iter(
        heldout_predictions[
            reference_model_name
        ]
    )
)

reference_record = (
    heldout_predictions[
        reference_model_name
    ][
        reference_algorithm_name
    ]
)

reference_y_true = np.asarray(
    reference_record["y_true"],
    dtype=int
)


if len(reference_y_true) != EXPECTED_HELDOUT_N:

    print(
        f"\nWarning: expected held-out N="
        f"{EXPECTED_HELDOUT_N:,}, observed "
        f"N={len(reference_y_true):,}."
    )


if int(
    np.sum(
        reference_y_true == 1
    )
) != EXPECTED_HELDOUT_EVENTS:

    print(
        f"\nWarning: expected held-out events="
        f"{EXPECTED_HELDOUT_EVENTS}, observed events="
        f"{int(np.sum(reference_y_true == 1))}."
    )


# ============================================================
# 19. EXPORT RESULTS
# ============================================================

part3c_bootstrap_results.to_csv(
    "part3c_bootstrap_results.csv",
    index=False,
    encoding="utf-8-sig"
)

part3c_bootstrap_distribution.to_csv(
    "part3c_bootstrap_distribution.csv",
    index=False,
    encoding="utf-8-sig"
)

part3c_bootstrap_status.to_csv(
    "part3c_bootstrap_status.csv",
    index=False,
    encoding="utf-8-sig"
)

part3c_bootstrap_verification.to_csv(
    "part3c_bootstrap_verification.csv",
    index=False,
    encoding="utf-8-sig"
)

part3c_bootstrap_publication_table.to_csv(
    "part3c_bootstrap_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 20. EXPORT COMPACT CORE PUBLICATION TABLE
# ============================================================

compact_columns = [
    "Validation_Set",
    "Screening_Model",
    "Algorithm",
    "N",
    "Events",
    "Prevalence",
    "Locked_Threshold",
    "AUC",
    "Average_Precision",
    "Sensitivity",
    "Specificity",
    "Precision",
    "Negative_Predictive_Value",
    "F1",
    "Balanced_Accuracy",
    "Matthews_Correlation_Coefficient",
    "Brier_Score"
]

part3c_bootstrap_publication_table_compact = (
    part3c_bootstrap_publication_table[
        compact_columns
    ]
    .copy()
)

part3c_bootstrap_publication_table_compact.to_csv(
    "part3c_bootstrap_publication_table_compact.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 21. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 3C VERSION 3.3 COMPLETED")
print("=" * 100)

print(
    f"Expected model-algorithm pairs : "
    f"{expected_algorithm_count}"
)

print(
    f"Successful bootstrap analyses  : "
    f"{successful_algorithm_count}"
)

print(
    f"Failed bootstrap analyses      : "
    f"{failed_algorithm_count}"
)

print(
    f"Insufficient analyses          : "
    f"{insufficient_algorithm_count}"
)

print(
    f"Bootstrap replicates requested : "
    f"{N_BOOTSTRAP:,} per model"
)

print(
    f"Minimum valid replicates       : "
    f"{MINIMUM_VALID_BOOTSTRAPS:,}"
)

print(
    f"Confidence interval method     : "
    f"Stratified percentile bootstrap "
    f"({CI_LOWER_PERCENTILE:.1f}th-"
    f"{CI_UPPER_PERCENTILE:.1f}th percentiles)"
)

print("\nArtifacts created:")

print("- part3c_bootstrap_results")
print("- part3c_bootstrap_distribution")
print("- part3c_bootstrap_status")
print("- part3c_bootstrap_verification")
print("- part3c_bootstrap_publication_table")
print("- part3c_bootstrap_publication_table_compact")

print("\nExported files:")

print("- part3c_bootstrap_results.csv")
print("- part3c_bootstrap_distribution.csv")
print("- part3c_bootstrap_status.csv")
print("- part3c_bootstrap_verification.csv")
print("- part3c_bootstrap_publication_table.csv")
print("- part3c_bootstrap_publication_table_compact.csv")

print(
    "\nNo model was refitted, no prediction was regenerated, "
    "and no threshold was modified during bootstrap analysis."
)

print(
    "Confidence intervals reflect uncertainty in the fixed "
    "held-out 2024 predictions under stratified participant-"
    "level resampling."
)

### PART 3D: Held-out Calibration Analysis

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.1
# PART 3D: HELD-OUT CALIBRATION ANALYSIS WITH LOWESS
# ============================================================
#
# Objectives:
# 1. Assess calibration on the untouched held-out 2024 cohort.
# 2. Use fixed participant-level probabilities from PART 3B.
# 3. Estimate calibration intercept and calibration slope.
# 4. Calculate calibration-in-the-large, Brier score,
#    Brier skill score, ECE, MCE, and ICI.
# 5. Generate adaptive quantile-binned calibration tables.
# 6. Produce publication-ready calibration plots.
# 7. Preserve all model-algorithm pairs for supplementary use.
#
# Inputs:
# - heldout_predictions
# - heldout_results
# - part3b_heldout_verification
#
# Outputs:
# - part3d_calibration_results
# - part3d_calibration_bins
# - part3d_calibration_status
# - part3d_calibration_verification
# - part3d_calibration_publication_table
# - part3d_calibration_publication_table_compact
# - calibration plot PNG/PDF files
#
# IMPORTANT:
# - No model fitting for prediction occurs in this part.
# - No probability recalibration is applied to validation results.
# - No decision threshold is changed.
# - Calibration models are diagnostic summaries only.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm

from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.metrics import brier_score_loss

warnings.filterwarnings("ignore")


# ============================================================
# 2. SETTINGS
# ============================================================

VALIDATION_SET_NAME = "Held-out 2024"

EXPECTED_HELDOUT_N = 1227
EXPECTED_HELDOUT_EVENTS = 14

N_CALIBRATION_BINS = 10

# Probabilities are clipped only for logit transformation.
# Original probabilities remain unchanged for all other metrics.
PROBABILITY_CLIP_EPSILON = 1e-6

# ICI is estimated from a LOWESS calibration curve fitted only
# as a descriptive smoother of the fixed held-out probabilities.
#
# With only 14 held-out events, LOWESS-based calibration measures
# should be interpreted as exploratory and visually descriptive.
LOWESS_FRACTION = 0.40
LOWESS_ITERATIONS = 0

ICI_METHOD = (
    "LOWESS absolute calibration error "
    f"(frac={LOWESS_FRACTION:.2f}, it={LOWESS_ITERATIONS})"
)

PLOT_OUTPUT_DIRECTORY = Path(
    "part3d_calibration_plots"
)

PLOT_DPI = 300

FIGURE_WIDTH = 6.5
FIGURE_HEIGHT = 5.5

THRESHOLD_ATOL = 1e-15


# ============================================================
# 3. REQUIRED OBJECT VALIDATION
# ============================================================

required_objects = [
    "heldout_predictions",
    "heldout_results",
    "part3b_heldout_verification"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )

if not isinstance(
    heldout_predictions,
    dict
):
    raise TypeError(
        "heldout_predictions must be a dictionary."
    )

if not isinstance(
    heldout_results,
    pd.DataFrame
):
    raise TypeError(
        "heldout_results must be a pandas DataFrame."
    )

if not isinstance(
    part3b_heldout_verification,
    pd.DataFrame
):
    raise TypeError(
        "part3b_heldout_verification must be a DataFrame."
    )

if "Heldout_Ready" not in (
    part3b_heldout_verification.columns
):
    raise KeyError(
        "part3b_heldout_verification does not contain "
        "'Heldout_Ready'."
    )

if not part3b_heldout_verification[
    "Heldout_Ready"
].all():

    failed_strategies = (
        part3b_heldout_verification.loc[
            ~part3b_heldout_verification[
                "Heldout_Ready"
            ],
            "Screening_Model"
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Calibration analysis cannot proceed because these "
        "strategies are not held-out ready: "
        + ", ".join(failed_strategies)
    )

if len(heldout_predictions) == 0:
    raise ValueError(
        "heldout_predictions is empty. Run PART 3B first."
    )


# ============================================================
# 4. CREATE OUTPUT DIRECTORY
# ============================================================

PLOT_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5. RESULT CONTAINERS
# ============================================================

calibration_result_rows = []

calibration_bin_rows = []

calibration_status_rows = []

calibration_verification_rows = []

calibration_plot_manifest_rows = []


# ============================================================
# 6. HELPER: LOGIT TRANSFORMATION
# ============================================================

def probability_to_logit(
    probabilities,
    epsilon=1e-6
):
    """
    Convert probabilities to logits after clipping exclusively
    for numerical stability.
    """

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )

    clipped_probabilities = np.clip(
        probabilities,
        epsilon,
        1 - epsilon
    )

    logits = np.log(
        clipped_probabilities
        /
        (
            1 - clipped_probabilities
        )
    )

    return (
        logits,
        clipped_probabilities
    )


# ============================================================
# 7. HELPER: CALIBRATION INTERCEPT AND SLOPE
# ============================================================

def estimate_calibration_intercept_and_slope(
    y_true,
    y_prob
):
    """
    Estimate:
    - Calibration intercept with the prediction logit used as
      an offset and coefficient fixed at 1.
    - Calibration slope from a logistic calibration model with
      a freely estimated intercept and prediction-logit slope.

    Ideal calibration:
    - Intercept = 0
    - Slope = 1
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    logits, clipped_probabilities = (
        probability_to_logit(
            y_prob,
            epsilon=PROBABILITY_CLIP_EPSILON
        )
    )

    intercept_design = np.ones(
        (
            len(y_true),
            1
        ),
        dtype=float
    )

    intercept_model = sm.GLM(
        y_true,
        intercept_design,
        family=sm.families.Binomial(),
        offset=logits
    )

    intercept_fit = intercept_model.fit(
        maxiter=200,
        disp=0
    )

    calibration_intercept = float(
        intercept_fit.params[0]
    )

    calibration_intercept_se = float(
        intercept_fit.bse[0]
    )

    calibration_intercept_ci_lower = float(
        calibration_intercept
        -
        1.96 * calibration_intercept_se
    )

    calibration_intercept_ci_upper = float(
        calibration_intercept
        +
        1.96 * calibration_intercept_se
    )

    slope_design = sm.add_constant(
        logits,
        has_constant="add"
    )

    slope_model = sm.GLM(
        y_true,
        slope_design,
        family=sm.families.Binomial()
    )

    slope_fit = slope_model.fit(
        maxiter=200,
        disp=0
    )

    calibration_model_intercept = float(
        slope_fit.params[0]
    )

    calibration_slope = float(
        slope_fit.params[1]
    )

    calibration_slope_se = float(
        slope_fit.bse[1]
    )

    calibration_slope_ci_lower = float(
        calibration_slope
        -
        1.96 * calibration_slope_se
    )

    calibration_slope_ci_upper = float(
        calibration_slope
        +
        1.96 * calibration_slope_se
    )

    fitted_calibration_probabilities = (
        slope_fit.predict(
            slope_design
        )
    )

    return {
        "Calibration_Intercept":
            calibration_intercept,

        "Calibration_Intercept_SE":
            calibration_intercept_se,

        "Calibration_Intercept_CI_Lower":
            calibration_intercept_ci_lower,

        "Calibration_Intercept_CI_Upper":
            calibration_intercept_ci_upper,

        "Calibration_Model_Intercept":
            calibration_model_intercept,

        "Calibration_Slope":
            calibration_slope,

        "Calibration_Slope_SE":
            calibration_slope_se,

        "Calibration_Slope_CI_Lower":
            calibration_slope_ci_lower,

        "Calibration_Slope_CI_Upper":
            calibration_slope_ci_upper,

        "Fitted_Calibration_Probability":
            np.asarray(
                fitted_calibration_probabilities,
                dtype=float
            ),

        "Clipped_Probability_Count":
            int(
                np.sum(
                    y_prob
                    !=
                    clipped_probabilities
                )
            )
    }


# ============================================================
# 8. HELPER: ADAPTIVE QUANTILE CALIBRATION BINS
# ============================================================

def create_adaptive_calibration_bins(
    y_true,
    y_prob,
    requested_bins=10
):
    """
    Create quantile-based calibration groups.

    Duplicate probability boundaries are automatically handled.
    The actual number of bins may be smaller than requested.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    working_table = pd.DataFrame({
        "y_true":
            y_true,

        "y_prob":
            y_prob
    })

    unique_probability_count = int(
        working_table[
            "y_prob"
        ].nunique()
    )

    actual_requested_bins = int(
        min(
            requested_bins,
            unique_probability_count,
            len(working_table)
        )
    )

    if actual_requested_bins < 2:

        working_table[
            "Calibration_Bin"
        ] = 1

    else:

        try:

            working_table[
                "Calibration_Bin"
            ] = (
                pd.qcut(
                    working_table[
                        "y_prob"
                    ],
                    q=actual_requested_bins,
                    labels=False,
                    duplicates="drop"
                )
                + 1
            )

        except Exception:

            working_table[
                "Probability_Rank"
            ] = (
                working_table[
                    "y_prob"
                ]
                .rank(
                    method="first"
                )
            )

            working_table[
                "Calibration_Bin"
            ] = (
                pd.qcut(
                    working_table[
                        "Probability_Rank"
                    ],
                    q=actual_requested_bins,
                    labels=False,
                    duplicates="drop"
                )
                + 1
            )

    bin_summary = (
        working_table
        .groupby(
            "Calibration_Bin",
            observed=True
        )
        .agg(
            Bin_N=(
                "y_true",
                "size"
            ),
            Bin_Events=(
                "y_true",
                "sum"
            ),
            Mean_Predicted_Probability=(
                "y_prob",
                "mean"
            ),
            Median_Predicted_Probability=(
                "y_prob",
                "median"
            ),
            Minimum_Predicted_Probability=(
                "y_prob",
                "min"
            ),
            Maximum_Predicted_Probability=(
                "y_prob",
                "max"
            ),
            Observed_Event_Rate=(
                "y_true",
                "mean"
            )
        )
        .reset_index()
    )

    bin_summary[
        "Absolute_Calibration_Error"
    ] = np.abs(
        bin_summary[
            "Observed_Event_Rate"
        ]
        -
        bin_summary[
            "Mean_Predicted_Probability"
        ]
    )

    bin_summary[
        "Squared_Calibration_Error"
    ] = (
        bin_summary[
            "Observed_Event_Rate"
        ]
        -
        bin_summary[
            "Mean_Predicted_Probability"
        ]
    ) ** 2

    return (
        working_table,
        bin_summary
    )


# ============================================================
# 9. HELPER: ECE AND MCE
# ============================================================

def calculate_binned_calibration_errors(
    bin_summary,
    total_n
):
    """
    Calculate:
    - Expected Calibration Error (ECE)
    - Maximum Calibration Error (MCE)

    ECE is the bin-size-weighted absolute difference between
    observed and mean predicted probabilities.
    """

    if len(bin_summary) == 0:
        return np.nan, np.nan

    weights = (
        bin_summary[
            "Bin_N"
        ]
        /
        total_n
    )

    ece = float(
        np.sum(
            weights
            *
            bin_summary[
                "Absolute_Calibration_Error"
            ]
        )
    )

    mce = float(
        np.max(
            bin_summary[
                "Absolute_Calibration_Error"
            ]
        )
    )

    return ece, mce


# ============================================================
# 10. HELPER: LOWESS CALIBRATION CURVE AND ICI
# ============================================================

def calculate_lowess_calibration(
    y_true,
    y_prob,
    fraction=0.40,
    iterations=0
):
    """
    Estimate a LOWESS calibration curve and calculate:

    - ICI: mean absolute difference between the LOWESS-smoothed
      observed risk and the original predicted probability
    - E50: median absolute calibration error
    - E90: 90th percentile absolute calibration error

    Duplicate predicted probabilities are aggregated before
    LOWESS fitting. The fitted curve is interpolated back to each
    participant's original predicted probability.

    This is a diagnostic smoother only. It does not recalibrate
    probabilities used in validation.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    unique_probability_count = int(
        len(
            np.unique(
                y_prob
            )
        )
    )

    if unique_probability_count < 3:

        return {
            "ICI":
                np.nan,

            "E50":
                np.nan,

            "E90":
                np.nan,

            "LOWESS_Probability":
                np.full(
                    len(
                        y_prob
                    ),
                    np.nan
                ),

            "LOWESS_X":
                np.array(
                    [],
                    dtype=float
                ),

            "LOWESS_Y":
                np.array(
                    [],
                    dtype=float
                ),

            "ICI_Estimable":
                False,

            "LOWESS_Fraction":
                fraction,

            "LOWESS_Iterations":
                iterations
        }

    aggregated_table = (
        pd.DataFrame({
            "y_prob":
                y_prob,

            "y_true":
                y_true
        })
        .groupby(
            "y_prob",
            as_index=False
        )
        .agg(
            Observed_Rate=(
                "y_true",
                "mean"
            ),

            Weight=(
                "y_true",
                "size"
            )
        )
        .sort_values(
            "y_prob"
        )
        .reset_index(
            drop=True
        )
    )

    lowess_result = lowess(
        endog=aggregated_table[
            "Observed_Rate"
        ].to_numpy(
            dtype=float
        ),
        exog=aggregated_table[
            "y_prob"
        ].to_numpy(
            dtype=float
        ),
        frac=fraction,
        it=iterations,
        delta=0.0,
        is_sorted=True,
        return_sorted=True
    )

    lowess_x = np.asarray(
        lowess_result[
            :,
            0
        ],
        dtype=float
    )

    lowess_y = np.asarray(
        lowess_result[
            :,
            1
        ],
        dtype=float
    )

    lowess_y = np.clip(
        lowess_y,
        0.0,
        1.0
    )

    # LOWESS can occasionally return repeated x values.
    lowess_curve_table = (
        pd.DataFrame({
            "x":
                lowess_x,

            "y":
                lowess_y
        })
        .groupby(
            "x",
            as_index=False
        )[
            "y"
        ]
        .mean()
        .sort_values(
            "x"
        )
    )

    lowess_x = lowess_curve_table[
        "x"
    ].to_numpy(
        dtype=float
    )

    lowess_y = lowess_curve_table[
        "y"
    ].to_numpy(
        dtype=float
    )

    if len(
        lowess_x
    ) < 2:

        return {
            "ICI":
                np.nan,

            "E50":
                np.nan,

            "E90":
                np.nan,

            "LOWESS_Probability":
                np.full(
                    len(
                        y_prob
                    ),
                    np.nan
                ),

            "LOWESS_X":
                lowess_x,

            "LOWESS_Y":
                lowess_y,

            "ICI_Estimable":
                False,

            "LOWESS_Fraction":
                fraction,

            "LOWESS_Iterations":
                iterations
        }

    lowess_probability = np.interp(
        y_prob,
        lowess_x,
        lowess_y,
        left=lowess_y[
            0
        ],
        right=lowess_y[
            -1
        ]
    )

    lowess_probability = np.clip(
        lowess_probability,
        0.0,
        1.0
    )

    absolute_errors = np.abs(
        lowess_probability
        -
        y_prob
    )

    return {
        "ICI":
            float(
                np.mean(
                    absolute_errors
                )
            ),

        "E50":
            float(
                np.percentile(
                    absolute_errors,
                    50
                )
            ),

        "E90":
            float(
                np.percentile(
                    absolute_errors,
                    90
                )
            ),

        "LOWESS_Probability":
            np.asarray(
                lowess_probability,
                dtype=float
            ),

        "LOWESS_X":
            lowess_x,

        "LOWESS_Y":
            lowess_y,

        "ICI_Estimable":
            True,

        "LOWESS_Fraction":
            fraction,

        "LOWESS_Iterations":
            iterations
    }


# ============================================================
# 11. HELPER: BRIER DECOMPOSITION SUMMARY
# ============================================================

def calculate_brier_summary(
    y_true,
    y_prob
):
    """
    Calculate Brier score and Brier skill score relative to a
    prevalence-only null model.

    Brier Skill Score:
        1 - model Brier / null Brier

    Positive values indicate improvement over the null model.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    prevalence = float(
        np.mean(
            y_true
        )
    )

    brier_score = float(
        brier_score_loss(
            y_true,
            y_prob
        )
    )

    null_probabilities = np.full(
        len(y_true),
        prevalence,
        dtype=float
    )

    null_brier_score = float(
        brier_score_loss(
            y_true,
            null_probabilities
        )
    )

    if null_brier_score > 0:

        brier_skill_score = float(
            1
            -
            brier_score
            /
            null_brier_score
        )

    else:

        brier_skill_score = np.nan

    mean_predicted_probability = float(
        np.mean(
            y_prob
        )
    )

    calibration_in_the_large_difference = float(
        mean_predicted_probability
        -
        prevalence
    )

    observed_events = float(
        np.sum(
            y_true
        )
    )

    expected_events = float(
        np.sum(
            y_prob
        )
    )

    observed_to_expected_ratio = (
        observed_events
        /
        expected_events
        if expected_events > 0
        else np.nan
    )

    expected_to_observed_ratio = (
        expected_events
        /
        observed_events
        if observed_events > 0
        else np.nan
    )

    return {
        "Brier_Score":
            brier_score,

        "Null_Brier_Score":
            null_brier_score,

        "Brier_Skill_Score":
            brier_skill_score,

        "Mean_Predicted_Probability":
            mean_predicted_probability,

        "Observed_Prevalence":
            prevalence,

        "Calibration_In_The_Large_Difference":
            calibration_in_the_large_difference,

        "Observed_Events":
            observed_events,

        "Expected_Events":
            expected_events,

        "Observed_to_Expected_Ratio":
            float(
                observed_to_expected_ratio
            )
            if np.isfinite(
                observed_to_expected_ratio
            )
            else np.nan,

        "Expected_to_Observed_Ratio":
            float(
                expected_to_observed_ratio
            )
            if np.isfinite(
                expected_to_observed_ratio
            )
            else np.nan
    }


# ============================================================
# 12. HELPER: CREATE CALIBRATION PLOT
# ============================================================

def create_calibration_plot(
    model_name,
    algorithm_name,
    y_true,
    y_prob,
    bin_summary,
    lowess_x,
    lowess_y,
    calibration_intercept,
    calibration_slope
):
    """
    Create one publication-ready calibration plot per
    model-algorithm pair.
    """

    figure, axis = plt.subplots(
        figsize=(
            FIGURE_WIDTH,
            FIGURE_HEIGHT
        )
    )

    axis.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Perfect calibration"
    )

    if len(bin_summary) > 0:

        axis.plot(
            bin_summary[
                "Mean_Predicted_Probability"
            ],
            bin_summary[
                "Observed_Event_Rate"
            ],
            marker="o",
            linewidth=1.5,
            label="Quantile-binned observed"
        )

    if (
        len(
            lowess_x
        )
        >= 2
        and
        np.all(
            np.isfinite(
                lowess_x
            )
        )
        and
        np.all(
            np.isfinite(
                lowess_y
            )
        )
    ):

        axis.plot(
            lowess_x,
            lowess_y,
            linewidth=1.75,
            label=(
                f"LOWESS smoother "
                f"(frac={LOWESS_FRACTION:.2f})"
            )
        )

    # Rug plot showing the distribution of predicted risks.
    rug_y = np.full(
        len(
            y_prob
        ),
        -0.0125
    )

    axis.plot(
        y_prob,
        rug_y,
        "|",
        markersize=5,
        alpha=0.20,
        clip_on=False,
        label="Predicted-risk distribution"
    )

    max_display_probability = float(
        max(
            0.05,
            np.percentile(
                y_prob,
                99
            ),
            np.max(
                bin_summary[
                    "Mean_Predicted_Probability"
                ]
            )
            if len(bin_summary) > 0
            else 0.05,
            np.max(
                bin_summary[
                    "Observed_Event_Rate"
                ]
            )
            if len(bin_summary) > 0
            else 0.05
        )
    )

    max_display_probability = min(
        1.0,
        max_display_probability * 1.10
    )

    axis.set_xlim(
        0,
        max_display_probability
    )

    axis.set_ylim(
        0,
        max_display_probability
    )

    axis.set_xlabel(
        "Predicted probability"
    )

    axis.set_ylabel(
        "Observed event rate"
    )

    axis.set_title(
        f"{model_name} — {algorithm_name}\n"
        f"Held-out 2024 calibration"
    )

    annotation_text = (
        f"Intercept = "
        f"{calibration_intercept:.3f}\n"
        f"Slope = "
        f"{calibration_slope:.3f}\n"
        f"Ideal: intercept 0; slope 1\n"
        f"N = {len(y_true):,}; "
        f"events = {int(np.sum(y_true))}"
    )

    axis.text(
        0.98,
        0.02,
        annotation_text,
        transform=axis.transAxes,
        horizontalalignment="right",
        verticalalignment="bottom",
        fontsize=9,
        bbox={
            "boxstyle":
                "round",

            "alpha":
                0.15
        }
    )

    axis.legend(
        loc="upper left",
        fontsize=8
    )

    axis.grid(
        alpha=0.25
    )

    figure.tight_layout()

    safe_model_name = (
        str(model_name)
        .replace(
            " ",
            "_"
        )
        .replace(
            "/",
            "_"
        )
    )

    safe_algorithm_name = (
        str(algorithm_name)
        .replace(
            " ",
            "_"
        )
        .replace(
            "/",
            "_"
        )
    )

    png_path = (
        PLOT_OUTPUT_DIRECTORY
        /
        (
            f"part3d_calibration_"
            f"{safe_model_name}_"
            f"{safe_algorithm_name}.png"
        )
    )

    pdf_path = (
        PLOT_OUTPUT_DIRECTORY
        /
        (
            f"part3d_calibration_"
            f"{safe_model_name}_"
            f"{safe_algorithm_name}.pdf"
        )
    )

    figure.savefig(
        png_path,
        dpi=PLOT_DPI,
        bbox_inches="tight"
    )

    figure.savefig(
        pdf_path,
        bbox_inches="tight"
    )

    plt.close(
        figure
    )

    return (
        str(png_path),
        str(pdf_path)
    )


# ============================================================
# 13. CALIBRATION ANALYSIS LOOP
# ============================================================

print("=" * 100)
print("PART 3D VERSION 3.1: HELD-OUT 2024 CALIBRATION ANALYSIS")
print("=" * 100)

print(
    f"Requested calibration bins: "
    f"{N_CALIBRATION_BINS}"
)

print(
    f"ICI method: {ICI_METHOD}"
)


for model_name, algorithms in (
    heldout_predictions.items()
):

    print("\n" + "=" * 100)
    print(f"{model_name}: HELD-OUT CALIBRATION")
    print("=" * 100)

    if len(algorithms) == 0:
        raise ValueError(
            f"{model_name} contains no held-out predictions."
        )

    for algorithm_index, (
        algorithm_name,
        record
    ) in enumerate(
        algorithms.items(),
        start=1
    ):

        print(
            f"\n[{algorithm_index}/"
            f"{len(algorithms)}] "
            f"{algorithm_name}"
        )

        try:

            required_record_keys = [
                "y_true",
                "y_prob",
                "y_pred",
                "locked_threshold",
                "original_index"
            ]

            missing_record_keys = [
                key
                for key in required_record_keys
                if key not in record
            ]

            if missing_record_keys:
                raise KeyError(
                    "Missing held-out prediction fields: "
                    + ", ".join(
                        missing_record_keys
                    )
                )

            y_true = np.asarray(
                record[
                    "y_true"
                ],
                dtype=int
            )

            y_prob = np.asarray(
                record[
                    "y_prob"
                ],
                dtype=float
            )

            stored_y_pred = np.asarray(
                record[
                    "y_pred"
                ],
                dtype=int
            )

            original_index = np.asarray(
                record[
                    "original_index"
                ]
            )

            locked_threshold = float(
                record[
                    "locked_threshold"
                ]
            )

            if not (
                len(y_true)
                ==
                len(y_prob)
                ==
                len(stored_y_pred)
                ==
                len(original_index)
            ):
                raise ValueError(
                    "Held-out arrays have different lengths."
                )

            if len(np.unique(y_true)) != 2:
                raise ValueError(
                    "Both outcome classes are required."
                )

            if not np.all(
                np.isfinite(
                    y_prob
                )
            ):
                raise ValueError(
                    "Probabilities contain nonfinite values."
                )

            if np.any(
                y_prob < 0
            ) or np.any(
                y_prob > 1
            ):
                raise ValueError(
                    "Probabilities are outside [0, 1]."
                )

            recomputed_y_pred = (
                y_prob
                >=
                locked_threshold
            ).astype(int)

            if not np.array_equal(
                stored_y_pred,
                recomputed_y_pred
            ):
                raise RuntimeError(
                    "Stored y_pred does not match the "
                    "locked threshold."
                )

            heldout_n = int(
                len(y_true)
            )

            heldout_events = int(
                np.sum(
                    y_true == 1
                )
            )

            heldout_nonevents = int(
                np.sum(
                    y_true == 0
                )
            )

            prevalence = float(
                np.mean(
                    y_true
                )
            )

            # ------------------------------------------------
            # Calibration intercept and slope
            # ------------------------------------------------

            regression_calibration = (
                estimate_calibration_intercept_and_slope(
                    y_true=y_true,
                    y_prob=y_prob
                )
            )

            # ------------------------------------------------
            # Adaptive calibration bins
            # ------------------------------------------------

            (
                participant_bin_table,
                bin_summary
            ) = create_adaptive_calibration_bins(
                y_true=y_true,
                y_prob=y_prob,
                requested_bins=N_CALIBRATION_BINS
            )

            actual_bin_count = int(
                bin_summary[
                    "Calibration_Bin"
                ].nunique()
            )

            ece, mce = (
                calculate_binned_calibration_errors(
                    bin_summary=bin_summary,
                    total_n=heldout_n
                )
            )

            # ------------------------------------------------
            # ICI and percentile calibration errors
            # ------------------------------------------------

            ici_result = calculate_lowess_calibration(
                y_true=y_true,
                y_prob=y_prob,
                fraction=LOWESS_FRACTION,
                iterations=LOWESS_ITERATIONS
            )

            # ------------------------------------------------
            # Brier summaries
            # ------------------------------------------------

            brier_summary = (
                calculate_brier_summary(
                    y_true=y_true,
                    y_prob=y_prob
                )
            )

            # ------------------------------------------------
            # Store bin-level outputs
            # ------------------------------------------------

            model_bin_summary = (
                bin_summary.copy()
            )

            model_bin_summary[
                "Validation_Set"
            ] = VALIDATION_SET_NAME

            model_bin_summary[
                "Screening_Model"
            ] = model_name

            model_bin_summary[
                "Algorithm"
            ] = algorithm_name

            model_bin_summary[
                "Requested_Bins"
            ] = N_CALIBRATION_BINS

            model_bin_summary[
                "Actual_Bins"
            ] = actual_bin_count

            model_bin_summary[
                "Locked_Threshold"
            ] = locked_threshold

            calibration_bin_rows.append(
                model_bin_summary
            )

            # ------------------------------------------------
            # Create calibration figure
            # ------------------------------------------------

            png_path, pdf_path = (
                create_calibration_plot(
                    model_name=model_name,
                    algorithm_name=algorithm_name,
                    y_true=y_true,
                    y_prob=y_prob,
                    bin_summary=bin_summary,
                    lowess_x=ici_result[
                        "LOWESS_X"
                    ],
                    lowess_y=ici_result[
                        "LOWESS_Y"
                    ],
                    calibration_intercept=(
                        regression_calibration[
                            "Calibration_Intercept"
                        ]
                    ),
                    calibration_slope=(
                        regression_calibration[
                            "Calibration_Slope"
                        ]
                    )
                )
            )

            calibration_plot_manifest_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "PNG_File":
                    png_path,

                "PDF_File":
                    pdf_path,

                "Plot_Created":
                    True
            })

            # ------------------------------------------------
            # Summary row
            # ------------------------------------------------

            calibration_result_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "N":
                    heldout_n,

                "Events":
                    heldout_events,

                "Nonevents":
                    heldout_nonevents,

                "Prevalence":
                    prevalence,

                "Locked_Threshold":
                    locked_threshold,

                "Mean_Predicted_Probability":
                    brier_summary[
                        "Mean_Predicted_Probability"
                    ],

                "Calibration_In_The_Large_Difference":
                    brier_summary[
                        "Calibration_In_The_Large_Difference"
                    ],

                "Observed_Events":
                    brier_summary[
                        "Observed_Events"
                    ],

                "Expected_Events":
                    brier_summary[
                        "Expected_Events"
                    ],

                "Observed_to_Expected_Ratio":
                    brier_summary[
                        "Observed_to_Expected_Ratio"
                    ],

                "Expected_to_Observed_Ratio":
                    brier_summary[
                        "Expected_to_Observed_Ratio"
                    ],

                "Calibration_Intercept":
                    regression_calibration[
                        "Calibration_Intercept"
                    ],

                "Calibration_Intercept_SE":
                    regression_calibration[
                        "Calibration_Intercept_SE"
                    ],

                "Calibration_Intercept_CI_Lower":
                    regression_calibration[
                        "Calibration_Intercept_CI_Lower"
                    ],

                "Calibration_Intercept_CI_Upper":
                    regression_calibration[
                        "Calibration_Intercept_CI_Upper"
                    ],

                "Calibration_Model_Intercept":
                    regression_calibration[
                        "Calibration_Model_Intercept"
                    ],

                "Calibration_Slope":
                    regression_calibration[
                        "Calibration_Slope"
                    ],

                "Calibration_Slope_SE":
                    regression_calibration[
                        "Calibration_Slope_SE"
                    ],

                "Calibration_Slope_CI_Lower":
                    regression_calibration[
                        "Calibration_Slope_CI_Lower"
                    ],

                "Calibration_Slope_CI_Upper":
                    regression_calibration[
                        "Calibration_Slope_CI_Upper"
                    ],

                "Brier_Score":
                    brier_summary[
                        "Brier_Score"
                    ],

                "Null_Brier_Score":
                    brier_summary[
                        "Null_Brier_Score"
                    ],

                "Brier_Skill_Score":
                    brier_summary[
                        "Brier_Skill_Score"
                    ],

                "ECE":
                    ece,

                "MCE":
                    mce,

                "ICI":
                    ici_result[
                        "ICI"
                    ],

                "E50":
                    ici_result[
                        "E50"
                    ],

                "E90":
                    ici_result[
                        "E90"
                    ],

                "ICI_Estimable":
                    ici_result[
                        "ICI_Estimable"
                    ],

                "Requested_Calibration_Bins":
                    N_CALIBRATION_BINS,

                "Actual_Calibration_Bins":
                    actual_bin_count,

                "Unique_Probability_Count":
                    int(
                        len(
                            np.unique(
                                y_prob
                            )
                        )
                    ),

                "Probability_Min":
                    float(
                        np.min(
                            y_prob
                        )
                    ),

                "Probability_Median":
                    float(
                        np.median(
                            y_prob
                        )
                    ),

                "Probability_Max":
                    float(
                        np.max(
                            y_prob
                        )
                    ),

                "Clipped_Probability_Count_For_Logit":
                    regression_calibration[
                        "Clipped_Probability_Count"
                    ],

                "LOWESS_Fraction":
                    ici_result[
                        "LOWESS_Fraction"
                    ],

                "LOWESS_Iterations":
                    ici_result[
                        "LOWESS_Iterations"
                    ],

                "ICI_Method":
                    ICI_METHOD,

                "Calibration_Assessment_Note":
                    (
                        "Diagnostic assessment only; "
                        "probabilities were not recalibrated."
                    )
            })

            calibration_status_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Success",

                "N":
                    heldout_n,

                "Events":
                    heldout_events,

                "Actual_Calibration_Bins":
                    actual_bin_count,

                "Calibration_Regression_Estimable":
                    True,

                "ICI_Estimable":
                    ici_result[
                        "ICI_Estimable"
                    ],

                "Plot_Created":
                    True,

                "Error":
                    ""
            })

            print(
                f"Intercept="
                f"{regression_calibration['Calibration_Intercept']:.3f} | "
                f"Slope="
                f"{regression_calibration['Calibration_Slope']:.3f} | "
                f"Brier="
                f"{brier_summary['Brier_Score']:.4f} | "
                f"ECE={ece:.4f} | "
                f"ICI={ici_result['ICI']:.4f} | "
                f"Bins={actual_bin_count}"
            )

        except Exception as error:

            calibration_status_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Failed",

                "N":
                    (
                        len(
                            record.get(
                                "y_true",
                                []
                            )
                        )
                        if isinstance(
                            record,
                            dict
                        )
                        else np.nan
                    ),

                "Events":
                    (
                        int(
                            np.sum(
                                np.asarray(
                                    record.get(
                                        "y_true",
                                        []
                                    )
                                )
                                == 1
                            )
                        )
                        if isinstance(
                            record,
                            dict
                        )
                        else np.nan
                    ),

                "Actual_Calibration_Bins":
                    np.nan,

                "Calibration_Regression_Estimable":
                    False,

                "ICI_Estimable":
                    False,

                "Plot_Created":
                    False,

                "Error":
                    str(
                        error
                    )
            })

            calibration_plot_manifest_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "PNG_File":
                    "",

                "PDF_File":
                    "",

                "Plot_Created":
                    False
            })

            print("Failed")
            print(
                f"Reason: {error}"
            )


# ============================================================
# 14. CREATE RESULT TABLES
# ============================================================

part3d_calibration_results = pd.DataFrame(
    calibration_result_rows
)

if calibration_bin_rows:

    part3d_calibration_bins = pd.concat(
        calibration_bin_rows,
        ignore_index=True
    )

else:

    part3d_calibration_bins = pd.DataFrame()

part3d_calibration_status = pd.DataFrame(
    calibration_status_rows
)

part3d_calibration_plot_manifest = pd.DataFrame(
    calibration_plot_manifest_rows
)


# ============================================================
# 15. VERIFICATION
# ============================================================

expected_algorithm_count = sum(
    len(
        algorithms
    )
    for algorithms in heldout_predictions.values()
)

successful_algorithm_count = int(
    (
        part3d_calibration_status[
            "Status"
        ]
        ==
        "Success"
    ).sum()
)

failed_algorithm_count = int(
    (
        part3d_calibration_status[
            "Status"
        ]
        ==
        "Failed"
    ).sum()
)


for model_name, algorithms in (
    heldout_predictions.items()
):

    expected_algorithms = set(
        algorithms.keys()
    )

    successful_algorithms = set(
        part3d_calibration_status.loc[
            (
                part3d_calibration_status[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                part3d_calibration_status[
                    "Status"
                ]
                ==
                "Success"
            ),
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )

    missing_algorithms = sorted(
        expected_algorithms
        -
        successful_algorithms
    )

    model_results = (
        part3d_calibration_results[
            part3d_calibration_results[
                "Screening_Model"
            ]
            ==
            model_name
        ]
    )

    expected_result_rows = int(
        len(
            expected_algorithms
        )
    )

    observed_result_rows = int(
        len(
            model_results
        )
    )

    all_core_metrics_finite = bool(
        np.isfinite(
            model_results[
                [
                    "Calibration_Intercept",
                    "Calibration_Slope",
                    "Brier_Score",
                    "Null_Brier_Score",
                    "ECE",
                    "MCE",
                    "ICI"
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )
        .all()
    ) if not model_results.empty else False

    all_bin_counts_valid = bool(
        (
            model_results[
                "Actual_Calibration_Bins"
            ]
            >= 1
        )
        .all()
    ) if not model_results.empty else False

    all_plots_created = bool(
        part3d_calibration_plot_manifest.loc[
            part3d_calibration_plot_manifest[
                "Screening_Model"
            ]
            ==
            model_name,
            "Plot_Created"
        ]
        .all()
    )

    point_estimates_match_part3b = True

    for algorithm_name in expected_algorithms:

        calibration_row = (
            model_results[
                model_results[
                    "Algorithm"
                ]
                ==
                algorithm_name
            ]
        )

        heldout_row = (
            heldout_results[
                (
                    heldout_results[
                        "Screening_Model"
                    ]
                    ==
                    model_name
                )
                &
                (
                    heldout_results[
                        "Algorithm"
                    ]
                    ==
                    algorithm_name
                )
            ]
        )

        if (
            len(
                calibration_row
            )
            != 1
            or
            len(
                heldout_row
            )
            != 1
        ):

            point_estimates_match_part3b = False
            continue

        calibration_brier = float(
            calibration_row.iloc[0][
                "Brier_Score"
            ]
        )

        heldout_brier = float(
            heldout_row.iloc[0][
                "Brier_Score"
            ]
        )

        if not np.isclose(
            calibration_brier,
            heldout_brier,
            rtol=0,
            atol=1e-12
        ):

            point_estimates_match_part3b = False

    calibration_verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            expected_result_rows,

        "Successful_Algorithms":
            int(
                len(
                    successful_algorithms
                )
            ),

        "Missing_Algorithms":
            ", ".join(
                missing_algorithms
            ),

        "Expected_Result_Rows":
            expected_result_rows,

        "Observed_Result_Rows":
            observed_result_rows,

        "All_Core_Metrics_Finite":
            all_core_metrics_finite,

        "All_Bin_Counts_Valid":
            all_bin_counts_valid,

        "All_Plots_Created":
            all_plots_created,

        "Brier_Point_Estimates_Match_PART3B":
            point_estimates_match_part3b,

        "Heldout_Calibration_Ready":
            (
                len(
                    missing_algorithms
                )
                ==
                0
                and
                observed_result_rows
                ==
                expected_result_rows
                and
                all_core_metrics_finite
                and
                all_bin_counts_valid
                and
                all_plots_created
                and
                point_estimates_match_part3b
            )
    })


part3d_calibration_verification = pd.DataFrame(
    calibration_verification_rows
)


# ============================================================
# 16. PUBLICATION TABLE
# ============================================================

part3d_calibration_publication_table = (
    part3d_calibration_results[
        [
            "Validation_Set",
            "Screening_Model",
            "Algorithm",
            "N",
            "Events",
            "Prevalence",
            "Mean_Predicted_Probability",
            "Calibration_In_The_Large_Difference",
            "Observed_Events",
            "Expected_Events",
            "Observed_to_Expected_Ratio",
            "Expected_to_Observed_Ratio",
            "Calibration_Intercept",
            "Calibration_Intercept_CI_Lower",
            "Calibration_Intercept_CI_Upper",
            "Calibration_Slope",
            "Calibration_Slope_CI_Lower",
            "Calibration_Slope_CI_Upper",
            "Brier_Score",
            "Null_Brier_Score",
            "Brier_Skill_Score",
            "ECE",
            "MCE",
            "ICI",
            "E50",
            "E90",
            "Actual_Calibration_Bins"
        ]
    ]
    .copy()
)




# ============================================================
# 17. COMPACT PUBLICATION TABLE
# ============================================================

part3d_calibration_publication_table_compact = (
    part3d_calibration_results[
        [
            "Validation_Set",
            "Screening_Model",
            "Algorithm",
            "N",
            "Events",
            "Prevalence",
            "Mean_Predicted_Probability",
            "Observed_to_Expected_Ratio",
            "Calibration_Intercept",
            "Calibration_Intercept_CI_Lower",
            "Calibration_Intercept_CI_Upper",
            "Calibration_Slope",
            "Calibration_Slope_CI_Lower",
            "Calibration_Slope_CI_Upper",
            "Brier_Score",
            "Brier_Skill_Score",
            "ECE",
            "ICI",
            "E50",
            "E90"
        ]
    ]
    .copy()
)


def format_estimate_with_ci(
    estimate,
    lower,
    upper,
    digits=3
):
    """
    Format an estimate and confidence interval for manuscript
    or supplementary tables.
    """

    if (
        not np.isfinite(
            estimate
        )
        or
        not np.isfinite(
            lower
        )
        or
        not np.isfinite(
            upper
        )
    ):

        return "Not estimable"

    return (
        f"{estimate:.{digits}f} "
        f"({lower:.{digits}f}-{upper:.{digits}f})"
    )


part3d_calibration_publication_table_compact[
    "Calibration_Intercept_95CI"
] = part3d_calibration_publication_table_compact.apply(
    lambda row:
        format_estimate_with_ci(
            estimate=row[
                "Calibration_Intercept"
            ],
            lower=row[
                "Calibration_Intercept_CI_Lower"
            ],
            upper=row[
                "Calibration_Intercept_CI_Upper"
            ],
            digits=3
        ),
    axis=1
)


part3d_calibration_publication_table_compact[
    "Calibration_Slope_95CI"
] = part3d_calibration_publication_table_compact.apply(
    lambda row:
        format_estimate_with_ci(
            estimate=row[
                "Calibration_Slope"
            ],
            lower=row[
                "Calibration_Slope_CI_Lower"
            ],
            upper=row[
                "Calibration_Slope_CI_Upper"
            ],
            digits=3
        ),
    axis=1
)


part3d_calibration_publication_table_compact = (
    part3d_calibration_publication_table_compact[
        [
            "Validation_Set",
            "Screening_Model",
            "Algorithm",
            "N",
            "Events",
            "Prevalence",
            "Mean_Predicted_Probability",
            "Observed_to_Expected_Ratio",
            "Calibration_Intercept_95CI",
            "Calibration_Slope_95CI",
            "Brier_Score",
            "Brier_Skill_Score",
            "ECE",
            "ICI",
            "E50",
            "E90"
        ]
    ]
    .copy()
)


# ============================================================
# 18. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 100)
print("PART 3D CALIBRATION STATUS")
print("=" * 100)

print(
    part3d_calibration_status[
        "Status"
    ]
    .value_counts(
        dropna=False
    )
)


print("\n" + "=" * 100)
print("HELD-OUT CALIBRATION SUMMARY")
print("=" * 100)

display(
    part3d_calibration_publication_table
    .sort_values(
        by=[
            "Screening_Model",
            "Algorithm"
        ]
    )
    .round(4)
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 100)
print("PART 3D CALIBRATION VERIFICATION")
print("=" * 100)

display(
    part3d_calibration_verification
)


# ============================================================
# 19. HARD SAFETY CHECKS
# ============================================================

if failed_algorithm_count > 0:

    failed_rows = (
        part3d_calibration_status[
            part3d_calibration_status[
                "Status"
            ]
            ==
            "Failed"
        ]
    )

    display(
        failed_rows
    )

    raise RuntimeError(
        f"{failed_algorithm_count} held-out calibration "
        "analyses failed."
    )


if successful_algorithm_count != (
    expected_algorithm_count
):

    raise RuntimeError(
        "The number of successful calibration analyses does "
        "not match the number of held-out model-algorithm pairs."
    )


if not part3d_calibration_verification[
    "Heldout_Calibration_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies failed held-out "
        "calibration verification."
    )


# ============================================================
# 20. COHORT WARNINGS
# ============================================================

reference_model_name = next(
    iter(
        heldout_predictions
    )
)

reference_algorithm_name = next(
    iter(
        heldout_predictions[
            reference_model_name
        ]
    )
)

reference_y_true = np.asarray(
    heldout_predictions[
        reference_model_name
    ][
        reference_algorithm_name
    ][
        "y_true"
    ],
    dtype=int
)


if len(
    reference_y_true
) != EXPECTED_HELDOUT_N:

    print(
        f"\nWarning: expected held-out N="
        f"{EXPECTED_HELDOUT_N:,}, observed "
        f"N={len(reference_y_true):,}."
    )


if int(
    np.sum(
        reference_y_true
        ==
        1
    )
) != EXPECTED_HELDOUT_EVENTS:

    print(
        f"\nWarning: expected held-out events="
        f"{EXPECTED_HELDOUT_EVENTS}, observed events="
        f"{int(np.sum(reference_y_true == 1))}."
    )


# ============================================================
# 21. EXPORT RESULTS
# ============================================================

part3d_calibration_results.to_csv(
    "part3d_calibration_results.csv",
    index=False,
    encoding="utf-8-sig"
)

part3d_calibration_bins.to_csv(
    "part3d_calibration_bins.csv",
    index=False,
    encoding="utf-8-sig"
)

part3d_calibration_status.to_csv(
    "part3d_calibration_status.csv",
    index=False,
    encoding="utf-8-sig"
)

part3d_calibration_verification.to_csv(
    "part3d_calibration_verification.csv",
    index=False,
    encoding="utf-8-sig"
)

part3d_calibration_publication_table.to_csv(
    "part3d_calibration_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


part3d_calibration_publication_table_compact.to_csv(
    "part3d_calibration_publication_table_compact.csv",
    index=False,
    encoding="utf-8-sig"
)

part3d_calibration_plot_manifest.to_csv(
    "part3d_calibration_plot_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 22. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 3D VERSION 3.1 COMPLETED")
print("=" * 100)

print(
    f"Expected model-algorithm pairs : "
    f"{expected_algorithm_count}"
)

print(
    f"Successful calibration analyses: "
    f"{successful_algorithm_count}"
)

print(
    f"Failed calibration analyses    : "
    f"{failed_algorithm_count}"
)

print(
    f"Requested calibration bins     : "
    f"{N_CALIBRATION_BINS}"
)

print(
    f"Calibration plot directory     : "
    f"{PLOT_OUTPUT_DIRECTORY}"
)

print("\nArtifacts created:")

print("- part3d_calibration_results")
print("- part3d_calibration_bins")
print("- part3d_calibration_status")
print("- part3d_calibration_verification")
print("- part3d_calibration_publication_table")
print("- part3d_calibration_publication_table_compact")
print("- part3d_calibration_plot_manifest")

print("\nExported files:")

print("- part3d_calibration_results.csv")
print("- part3d_calibration_bins.csv")
print("- part3d_calibration_status.csv")
print("- part3d_calibration_verification.csv")
print("- part3d_calibration_publication_table.csv")
print("- part3d_calibration_publication_table_compact.csv")
print("- part3d_calibration_plot_manifest.csv")
print(
    f"- PNG/PDF calibration plots in "
    f"{PLOT_OUTPUT_DIRECTORY}/"
)

print(
    "\nNo model was refitted for prediction, no probability was "
    "recalibrated, and no locked threshold was modified."
)

print(
    "Calibration regression and LOWESS smoothing were used "
    "only as diagnostic summaries of the fixed held-out "
    "probabilities."
)

### PART 3D-2: OOF-Based Probability Recalibration

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.2
# PART 3D-2: OOF-BASED PROBABILITY RECALIBRATION
# ============================================================
#
# Objectives:
# 1. Fit a probability-recalibration mapping using genuine OOF
#    predictions from the 2024 training partition only.
# 2. Apply the locked recalibration mapping to the untouched
#    held-out 2024 probabilities.
# 3. Compare raw versus recalibrated probability performance.
# 4. Preserve the original locked classification threshold and
#    raw held-out predictions without modification.
# 5. Produce recalibrated probabilities suitable for exploratory
#    Decision Curve Analysis.
#
# Primary recalibration method:
#
#     Logistic recalibration
#
#     logit(P(Y=1)) = intercept + slope * logit(raw probability)
#
# Rationale:
# - It adjusts both calibration-in-the-large and probability
#   extremeness.
# - It is more data-efficient than flexible nonparametric
#   calibration when only 32 training events are available.
#
# Inputs:
# - oof_predictions
# - heldout_predictions
# - threshold_summary
# - part3a_oof_verification
# - part3b_heldout_verification
#
# Outputs:
# - oof_recalibration_models
# - oof_recalibration_results
# - recalibrated_heldout_predictions
# - recalibrated_heldout_results
# - recalibrated_heldout_prediction_table
# - part3d2_recalibration_status
# - part3d2_recalibration_verification
# - part3d2_recalibration_publication_table
#
# IMPORTANT:
# - Held-out outcomes are NEVER used to fit recalibration models.
# - Temporal outcomes are NEVER used in this part.
# - Raw probabilities and locked thresholds remain unchanged.
# - Recalibrated probabilities are not used to revise the
#   prespecified classification threshold.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import warnings

import numpy as np
import pandas as pd

import statsmodels.api as sm

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)

warnings.filterwarnings("ignore")


# ============================================================
# 2. SETTINGS
# ============================================================

RECALIBRATION_METHOD = "Logistic recalibration"

PROBABILITY_CLIP_EPSILON = 1e-6

MINIMUM_TRAINING_EVENTS = 10

EXPECTED_TRAINING_N = 2863
EXPECTED_TRAINING_EVENTS = 32

EXPECTED_HELDOUT_N = 1227
EXPECTED_HELDOUT_EVENTS = 14

COEFFICIENT_ABSOLUTE_LIMIT = 100.0

PREDICTION_TOLERANCE = 1e-12


# ============================================================
# 3. REQUIRED OBJECT VALIDATION
# ============================================================

required_objects = [
    "oof_predictions",
    "heldout_predictions",
    "threshold_summary",
    "part3a_oof_verification",
    "part3b_heldout_verification"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(
            missing_objects
        )
    )


if not isinstance(
    oof_predictions,
    dict
):

    raise TypeError(
        "oof_predictions must be a dictionary."
    )


if not isinstance(
    heldout_predictions,
    dict
):

    raise TypeError(
        "heldout_predictions must be a dictionary."
    )


if not isinstance(
    threshold_summary,
    pd.DataFrame
):

    raise TypeError(
        "threshold_summary must be a pandas DataFrame."
    )


if not part3a_oof_verification[
    "OOF_Ready"
].all():

    raise RuntimeError(
        "OOF verification failed. Recalibration cannot proceed."
    )


if not part3b_heldout_verification[
    "Heldout_Ready"
].all():

    raise RuntimeError(
        "Held-out verification failed. Recalibration cannot proceed."
    )


# ============================================================
# 4. RESULT CONTAINERS
# ============================================================

oof_recalibration_models = {}

oof_recalibration_result_rows = []

recalibrated_heldout_predictions = {}

recalibrated_heldout_result_rows = []

recalibrated_heldout_prediction_rows = []

recalibration_status_rows = []

recalibration_verification_rows = []


# ============================================================
# 5. HELPER: CLIP AND LOGIT
# ============================================================

def clip_probabilities(
    probabilities,
    epsilon=1e-6
):
    """
    Clip probabilities only for logit transformation and
    numerical stability.
    """

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )

    if not np.all(
        np.isfinite(
            probabilities
        )
    ):

        raise ValueError(
            "Probabilities contain nonfinite values."
        )

    if np.any(
        probabilities < 0
    ) or np.any(
        probabilities > 1
    ):

        raise ValueError(
            "Probabilities must lie within [0, 1]."
        )

    clipped = np.clip(
        probabilities,
        epsilon,
        1 - epsilon
    )

    return clipped


def probability_to_logit(
    probabilities,
    epsilon=1e-6
):
    """
    Convert probabilities to logits after numerical clipping.
    """

    clipped = clip_probabilities(
        probabilities,
        epsilon=epsilon
    )

    logits = np.log(
        clipped
        /
        (
            1 - clipped
        )
    )

    return logits, clipped


def inverse_logit(
    linear_predictor
):
    """
    Numerically stable inverse-logit transformation.
    """

    linear_predictor = np.asarray(
        linear_predictor,
        dtype=float
    )

    clipped_linear_predictor = np.clip(
        linear_predictor,
        -700,
        700
    )

    return (
        1
        /
        (
            1
            +
            np.exp(
                -clipped_linear_predictor
            )
        )
    )


# ============================================================
# 6. HELPER: FIT LOGISTIC RECALIBRATION
# ============================================================

def fit_logistic_recalibration(
    y_true,
    y_prob
):
    """
    Fit logistic recalibration using genuine OOF probabilities.

    Model:
        logit(outcome probability)
        =
        intercept
        +
        slope * logit(raw probability)

    The returned mapping is later applied unchanged to held-out
    and temporal probabilities.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    if len(
        y_true
    ) != len(
        y_prob
    ):

        raise ValueError(
            "y_true and y_prob lengths differ."
        )


    observed_classes = set(
        np.unique(
            y_true
        )
    )

    if observed_classes != {
        0,
        1
    }:

        raise ValueError(
            f"Expected classes {{0, 1}}, observed "
            f"{sorted(observed_classes)}."
        )


    event_count = int(
        np.sum(
            y_true == 1
        )
    )

    if event_count < MINIMUM_TRAINING_EVENTS:

        raise ValueError(
            f"Only {event_count} training events are available; "
            f"at least {MINIMUM_TRAINING_EVENTS} are required."
        )


    logits, clipped_probabilities = (
        probability_to_logit(
            y_prob,
            epsilon=PROBABILITY_CLIP_EPSILON
        )
    )


    design_matrix = sm.add_constant(
        logits,
        has_constant="add"
    )


    calibration_model = sm.GLM(
        y_true,
        design_matrix,
        family=sm.families.Binomial()
    )


    calibration_fit = calibration_model.fit(
        maxiter=500,
        disp=0
    )


    intercept = float(
        calibration_fit.params[
            0
        ]
    )

    slope = float(
        calibration_fit.params[
            1
        ]
    )


    intercept_se = float(
        calibration_fit.bse[
            0
        ]
    )

    slope_se = float(
        calibration_fit.bse[
            1
        ]
    )


    converged = bool(
        getattr(
            calibration_fit,
            "converged",
            True
        )
    )


    coefficients_finite = bool(
        np.isfinite(
            intercept
        )
        and
        np.isfinite(
            slope
        )
    )


    coefficients_within_limit = bool(
        abs(
            intercept
        )
        <= COEFFICIENT_ABSOLUTE_LIMIT
        and
        abs(
            slope
        )
        <= COEFFICIENT_ABSOLUTE_LIMIT
    )


    if not converged:

        raise RuntimeError(
            "Logistic recalibration model did not converge."
        )


    if not coefficients_finite:

        raise RuntimeError(
            "Logistic recalibration coefficients are nonfinite."
        )


    if not coefficients_within_limit:

        raise RuntimeError(
            "Logistic recalibration coefficients are numerically "
            "unstable."
        )


    recalibrated_oof_probability = inverse_logit(
        intercept
        +
        slope
        *
        logits
    )


    raw_brier = float(
        brier_score_loss(
            y_true,
            y_prob
        )
    )


    recalibrated_brier = float(
        brier_score_loss(
            y_true,
            recalibrated_oof_probability
        )
    )


    raw_auc = float(
        roc_auc_score(
            y_true,
            y_prob
        )
    )


    recalibrated_auc = float(
        roc_auc_score(
            y_true,
            recalibrated_oof_probability
        )
    )


    raw_average_precision = float(
        average_precision_score(
            y_true,
            y_prob
        )
    )


    recalibrated_average_precision = float(
        average_precision_score(
            y_true,
            recalibrated_oof_probability
        )
    )


    return {
        "Method":
            RECALIBRATION_METHOD,

        "Intercept":
            intercept,

        "Slope":
            slope,

        "Intercept_SE":
            intercept_se,

        "Slope_SE":
            slope_se,

        "Intercept_CI_Lower":
            float(
                intercept
                -
                1.96
                *
                intercept_se
            ),

        "Intercept_CI_Upper":
            float(
                intercept
                +
                1.96
                *
                intercept_se
            ),

        "Slope_CI_Lower":
            float(
                slope
                -
                1.96
                *
                slope_se
            ),

        "Slope_CI_Upper":
            float(
                slope
                +
                1.96
                *
                slope_se
            ),

        "Converged":
            converged,

        "Coefficients_Finite":
            coefficients_finite,

        "Coefficients_Within_Limit":
            coefficients_within_limit,

        "Training_N":
            int(
                len(
                    y_true
                )
            ),

        "Training_Events":
            event_count,

        "Training_Prevalence":
            float(
                np.mean(
                    y_true
                )
            ),

        "Clipped_Probability_Count":
            int(
                np.sum(
                    clipped_probabilities
                    !=
                    y_prob
                )
            ),

        "Raw_OOF_Mean_Probability":
            float(
                np.mean(
                    y_prob
                )
            ),

        "Recalibrated_OOF_Mean_Probability":
            float(
                np.mean(
                    recalibrated_oof_probability
                )
            ),

        "Raw_OOF_AUC":
            raw_auc,

        "Recalibrated_OOF_AUC":
            recalibrated_auc,

        "Raw_OOF_Average_Precision":
            raw_average_precision,

        "Recalibrated_OOF_Average_Precision":
            recalibrated_average_precision,

        "Raw_OOF_Brier":
            raw_brier,

        "Recalibrated_OOF_Brier":
            recalibrated_brier,

        "OOF_Brier_Improvement":
            float(
                raw_brier
                -
                recalibrated_brier
            ),

        "OOF_Recalibrated_Probability":
            recalibrated_oof_probability
    }


# ============================================================
# 7. HELPER: APPLY LOCKED RECALIBRATION MAPPING
# ============================================================

def apply_logistic_recalibration(
    raw_probability,
    intercept,
    slope
):
    """
    Apply a previously fitted logistic recalibration mapping.
    """

    logits, clipped_probability = (
        probability_to_logit(
            raw_probability,
            epsilon=PROBABILITY_CLIP_EPSILON
        )
    )


    recalibrated_probability = inverse_logit(
        intercept
        +
        slope
        *
        logits
    )


    return (
        recalibrated_probability,
        clipped_probability
    )


# ============================================================
# 8. HELPER: CALCULATE PROBABILITY METRICS
# ============================================================

def calculate_probability_metrics(
    y_true,
    y_prob
):
    """
    Calculate discrimination and probability-accuracy metrics.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )


    prevalence = float(
        np.mean(
            y_true
        )
    )


    null_probability = np.full(
        len(
            y_true
        ),
        prevalence,
        dtype=float
    )


    brier_score = float(
        brier_score_loss(
            y_true,
            y_prob
        )
    )


    null_brier_score = float(
        brier_score_loss(
            y_true,
            null_probability
        )
    )


    brier_skill_score = (
        float(
            1
            -
            brier_score
            /
            null_brier_score
        )
        if null_brier_score > 0
        else np.nan
    )


    expected_events = float(
        np.sum(
            y_prob
        )
    )


    observed_events = float(
        np.sum(
            y_true
        )
    )


    observed_to_expected_ratio = (
        float(
            observed_events
            /
            expected_events
        )
        if expected_events > 0
        else np.nan
    )


    return {
        "AUC":
            float(
                roc_auc_score(
                    y_true,
                    y_prob
                )
            ),

        "Average_Precision":
            float(
                average_precision_score(
                    y_true,
                    y_prob
                )
            ),

        "Brier_Score":
            brier_score,

        "Null_Brier_Score":
            null_brier_score,

        "Brier_Skill_Score":
            brier_skill_score,

        "Mean_Predicted_Probability":
            float(
                np.mean(
                    y_prob
                )
            ),

        "Observed_Prevalence":
            prevalence,

        "Observed_Events":
            observed_events,

        "Expected_Events":
            expected_events,

        "Observed_to_Expected_Ratio":
            observed_to_expected_ratio
    }


# ============================================================
# 9. FIT OOF RECALIBRATION AND APPLY TO HELD-OUT
# ============================================================

print("=" * 100)
print("PART 3D-2 VERSION 3.2: OOF-BASED PROBABILITY RECALIBRATION")
print("=" * 100)

print(
    f"Primary method: "
    f"{RECALIBRATION_METHOD}"
)


for model_name, algorithms in (
    oof_predictions.items()
):

    print("\n" + "=" * 100)
    print(f"{model_name}: OOF RECALIBRATION")
    print("=" * 100)


    if model_name not in (
        heldout_predictions
    ):

        raise KeyError(
            f"{model_name} is missing from heldout_predictions."
        )


    oof_recalibration_models[
        model_name
    ] = {}


    recalibrated_heldout_predictions[
        model_name
    ] = {}


    for algorithm_index, (
        algorithm_name,
        oof_record
    ) in enumerate(
        algorithms.items(),
        start=1
    ):

        print(
            f"\n[{algorithm_index}/"
            f"{len(algorithms)}] "
            f"{algorithm_name}"
        )


        try:

            if algorithm_name not in (
                heldout_predictions[
                    model_name
                ]
            ):

                raise KeyError(
                    "Matching held-out prediction record is missing."
                )


            heldout_record = (
                heldout_predictions[
                    model_name
                ][
                    algorithm_name
                ]
            )


            y_oof = np.asarray(
                oof_record[
                    "y_true"
                ],
                dtype=int
            )


            raw_oof_probability = np.asarray(
                oof_record[
                    "y_prob"
                ],
                dtype=float
            )


            y_heldout = np.asarray(
                heldout_record[
                    "y_true"
                ],
                dtype=int
            )


            raw_heldout_probability = np.asarray(
                heldout_record[
                    "y_prob"
                ],
                dtype=float
            )


            original_index = np.asarray(
                heldout_record[
                    "original_index"
                ]
            )


            if len(
                y_oof
            ) != len(
                raw_oof_probability
            ):

                raise ValueError(
                    "OOF outcome and probability lengths differ."
                )


            if len(
                y_heldout
            ) != len(
                raw_heldout_probability
            ):

                raise ValueError(
                    "Held-out outcome and probability lengths differ."
                )


            if len(
                original_index
            ) != len(
                y_heldout
            ):

                raise ValueError(
                    "Held-out original index length is incorrect."
                )


            # ------------------------------------------------
            # Fit mapping exclusively on genuine OOF predictions
            # ------------------------------------------------

            recalibration_fit = (
                fit_logistic_recalibration(
                    y_true=y_oof,
                    y_prob=raw_oof_probability
                )
            )


            intercept = float(
                recalibration_fit[
                    "Intercept"
                ]
            )


            slope = float(
                recalibration_fit[
                    "Slope"
                ]
            )


            oof_recalibration_models[
                model_name
            ][
                algorithm_name
            ] = {
                "Method":
                    RECALIBRATION_METHOD,

                "Intercept":
                    intercept,

                "Slope":
                    slope,

                "Intercept_SE":
                    recalibration_fit[
                        "Intercept_SE"
                    ],

                "Slope_SE":
                    recalibration_fit[
                        "Slope_SE"
                    ],

                "Training_N":
                    recalibration_fit[
                        "Training_N"
                    ],

                "Training_Events":
                    recalibration_fit[
                        "Training_Events"
                    ],

                "Training_Prevalence":
                    recalibration_fit[
                        "Training_Prevalence"
                    ],

                "Probability_Clip_Epsilon":
                    PROBABILITY_CLIP_EPSILON,

                "Source":
                    (
                        "Genuine 5-fold OOF probabilities from "
                        "2024 training partition"
                    )
            }


            # ------------------------------------------------
            # Apply locked mapping to held-out probabilities
            # ------------------------------------------------

            (
                recalibrated_heldout_probability,
                clipped_heldout_probability
            ) = apply_logistic_recalibration(
                raw_probability=raw_heldout_probability,
                intercept=intercept,
                slope=slope
            )


            raw_heldout_metrics = (
                calculate_probability_metrics(
                    y_true=y_heldout,
                    y_prob=raw_heldout_probability
                )
            )


            recalibrated_heldout_metrics = (
                calculate_probability_metrics(
                    y_true=y_heldout,
                    y_prob=recalibrated_heldout_probability
                )
            )


            # A monotonic positive-slope transformation should
            # preserve ranking and therefore preserve AUC/AP.
            monotonic_increasing = bool(
                slope > 0
            )


            if monotonic_increasing:

                auc_preserved = bool(
                    np.isclose(
                        raw_heldout_metrics[
                            "AUC"
                        ],
                        recalibrated_heldout_metrics[
                            "AUC"
                        ],
                        rtol=0,
                        atol=PREDICTION_TOLERANCE
                    )
                )

            else:

                auc_preserved = False


            # ------------------------------------------------
            # Store recalibrated held-out predictions
            # ------------------------------------------------

            recalibrated_heldout_predictions[
                model_name
            ][
                algorithm_name
            ] = {
                "validation_set":
                    "Held-out 2024",

                "y_true":
                    y_heldout.copy(),

                "raw_y_prob":
                    raw_heldout_probability.copy(),

                "recalibrated_y_prob":
                    recalibrated_heldout_probability.copy(),

                "original_index":
                    original_index.copy(),

                "row_position":
                    np.arange(
                        len(
                            y_heldout
                        )
                    ),

                "recalibration_method":
                    RECALIBRATION_METHOD,

                "recalibration_intercept":
                    intercept,

                "recalibration_slope":
                    slope,

                "recalibration_source":
                    (
                        "Genuine 5-fold OOF probabilities from "
                        "2024 training partition"
                    ),

                "raw_locked_threshold":
                    float(
                        heldout_record[
                            "locked_threshold"
                        ]
                    ),

                "classification_threshold_modified":
                    False,

                "n":
                    int(
                        len(
                            y_heldout
                        )
                    ),

                "events":
                    int(
                        np.sum(
                            y_heldout
                        )
                    ),

                "raw_auc":
                    raw_heldout_metrics[
                        "AUC"
                    ],

                "recalibrated_auc":
                    recalibrated_heldout_metrics[
                        "AUC"
                    ],

                "raw_average_precision":
                    raw_heldout_metrics[
                        "Average_Precision"
                    ],

                "recalibrated_average_precision":
                    recalibrated_heldout_metrics[
                        "Average_Precision"
                    ],

                "raw_brier_score":
                    raw_heldout_metrics[
                        "Brier_Score"
                    ],

                "recalibrated_brier_score":
                    recalibrated_heldout_metrics[
                        "Brier_Score"
                    ],

                "raw_brier_skill_score":
                    raw_heldout_metrics[
                        "Brier_Skill_Score"
                    ],

                "recalibrated_brier_skill_score":
                    recalibrated_heldout_metrics[
                        "Brier_Skill_Score"
                    ]
            }


            # ------------------------------------------------
            # Long participant-level table
            # ------------------------------------------------

            recalibrated_heldout_prediction_rows.append(
                pd.DataFrame({

                    "Validation_Set":
                        "Held-out 2024",

                    "Screening_Model":
                        model_name,

                    "Algorithm":
                        algorithm_name,

                    "Row_Position":
                        np.arange(
                            len(
                                y_heldout
                            )
                        ),

                    "Original_Index":
                        original_index,

                    "y_true":
                        y_heldout,

                    "Raw_Probability":
                        raw_heldout_probability,

                    "Recalibrated_Probability":
                        recalibrated_heldout_probability,

                    "Recalibration_Intercept":
                        intercept,

                    "Recalibration_Slope":
                        slope,

                    "Raw_Locked_Threshold":
                        float(
                            heldout_record[
                                "locked_threshold"
                            ]
                        )
                })
            )


            # ------------------------------------------------
            # OOF recalibration summary
            # ------------------------------------------------

            oof_recalibration_result_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Method":
                    RECALIBRATION_METHOD,

                "Training_N":
                    recalibration_fit[
                        "Training_N"
                    ],

                "Training_Events":
                    recalibration_fit[
                        "Training_Events"
                    ],

                "Training_Prevalence":
                    recalibration_fit[
                        "Training_Prevalence"
                    ],

                "Intercept":
                    intercept,

                "Intercept_SE":
                    recalibration_fit[
                        "Intercept_SE"
                    ],

                "Intercept_CI_Lower":
                    recalibration_fit[
                        "Intercept_CI_Lower"
                    ],

                "Intercept_CI_Upper":
                    recalibration_fit[
                        "Intercept_CI_Upper"
                    ],

                "Slope":
                    slope,

                "Slope_SE":
                    recalibration_fit[
                        "Slope_SE"
                    ],

                "Slope_CI_Lower":
                    recalibration_fit[
                        "Slope_CI_Lower"
                    ],

                "Slope_CI_Upper":
                    recalibration_fit[
                        "Slope_CI_Upper"
                    ],

                "Converged":
                    recalibration_fit[
                        "Converged"
                    ],

                "Raw_OOF_Mean_Probability":
                    recalibration_fit[
                        "Raw_OOF_Mean_Probability"
                    ],

                "Recalibrated_OOF_Mean_Probability":
                    recalibration_fit[
                        "Recalibrated_OOF_Mean_Probability"
                    ],

                "Raw_OOF_AUC":
                    recalibration_fit[
                        "Raw_OOF_AUC"
                    ],

                "Recalibrated_OOF_AUC":
                    recalibration_fit[
                        "Recalibrated_OOF_AUC"
                    ],

                "Raw_OOF_Average_Precision":
                    recalibration_fit[
                        "Raw_OOF_Average_Precision"
                    ],

                "Recalibrated_OOF_Average_Precision":
                    recalibration_fit[
                        "Recalibrated_OOF_Average_Precision"
                    ],

                "Raw_OOF_Brier":
                    recalibration_fit[
                        "Raw_OOF_Brier"
                    ],

                "Recalibrated_OOF_Brier":
                    recalibration_fit[
                        "Recalibrated_OOF_Brier"
                    ],

                "OOF_Brier_Improvement":
                    recalibration_fit[
                        "OOF_Brier_Improvement"
                    ]
            })


            # ------------------------------------------------
            # Held-out comparison summary
            # ------------------------------------------------

            recalibrated_heldout_result_rows.append({

                "Validation_Set":
                    "Held-out 2024",

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "N":
                    int(
                        len(
                            y_heldout
                        )
                    ),

                "Events":
                    int(
                        np.sum(
                            y_heldout
                        )
                    ),

                "Prevalence":
                    float(
                        np.mean(
                            y_heldout
                        )
                    ),

                "Recalibration_Method":
                    RECALIBRATION_METHOD,

                "Recalibration_Intercept":
                    intercept,

                "Recalibration_Slope":
                    slope,

                "Monotonic_Increasing":
                    monotonic_increasing,

                "AUC_Preserved":
                    auc_preserved,

                "Eligible_For_DCA":
                    bool(
                        monotonic_increasing
                        and
                        auc_preserved
                    ),

                "Raw_AUC":
                    raw_heldout_metrics[
                        "AUC"
                    ],

                "Recalibrated_AUC":
                    recalibrated_heldout_metrics[
                        "AUC"
                    ],

                "Raw_Average_Precision":
                    raw_heldout_metrics[
                        "Average_Precision"
                    ],

                "Recalibrated_Average_Precision":
                    recalibrated_heldout_metrics[
                        "Average_Precision"
                    ],

                "Raw_Brier_Score":
                    raw_heldout_metrics[
                        "Brier_Score"
                    ],

                "Recalibrated_Brier_Score":
                    recalibrated_heldout_metrics[
                        "Brier_Score"
                    ],

                "Brier_Improvement":
                    float(
                        raw_heldout_metrics[
                            "Brier_Score"
                        ]
                        -
                        recalibrated_heldout_metrics[
                            "Brier_Score"
                        ]
                    ),

                "Raw_Brier_Skill_Score":
                    raw_heldout_metrics[
                        "Brier_Skill_Score"
                    ],

                "Recalibrated_Brier_Skill_Score":
                    recalibrated_heldout_metrics[
                        "Brier_Skill_Score"
                    ],

                "Raw_Mean_Predicted_Probability":
                    raw_heldout_metrics[
                        "Mean_Predicted_Probability"
                    ],

                "Recalibrated_Mean_Predicted_Probability":
                    recalibrated_heldout_metrics[
                        "Mean_Predicted_Probability"
                    ],

                "Observed_Prevalence":
                    recalibrated_heldout_metrics[
                        "Observed_Prevalence"
                    ],

                "Raw_Expected_Events":
                    raw_heldout_metrics[
                        "Expected_Events"
                    ],

                "Recalibrated_Expected_Events":
                    recalibrated_heldout_metrics[
                        "Expected_Events"
                    ],

                "Observed_Events":
                    recalibrated_heldout_metrics[
                        "Observed_Events"
                    ],

                "Raw_Observed_to_Expected_Ratio":
                    raw_heldout_metrics[
                        "Observed_to_Expected_Ratio"
                    ],

                "Recalibrated_Observed_to_Expected_Ratio":
                    recalibrated_heldout_metrics[
                        "Observed_to_Expected_Ratio"
                    ],

                "Heldout_Probability_Clipped_Count":
                    int(
                        np.sum(
                            clipped_heldout_probability
                            !=
                            raw_heldout_probability
                        )
                    ),

                "Raw_Classification_Threshold_Modified":
                    False
            })


            recalibration_status_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Success",

                "Training_N":
                    int(
                        len(
                            y_oof
                        )
                    ),

                "Training_Events":
                    int(
                        np.sum(
                            y_oof
                        )
                    ),

                "Heldout_N":
                    int(
                        len(
                            y_heldout
                        )
                    ),

                "Heldout_Events":
                    int(
                        np.sum(
                            y_heldout
                        )
                    ),

                "Converged":
                    recalibration_fit[
                        "Converged"
                    ],

                "Monotonic_Increasing":
                    monotonic_increasing,

                "AUC_Preserved":
                    auc_preserved,

                "Brier_Improved_Heldout":
                    bool(
                        recalibrated_heldout_metrics[
                            "Brier_Score"
                        ]
                        <
                        raw_heldout_metrics[
                            "Brier_Score"
                        ]
                    ),

                "Eligible_For_DCA":
                    bool(
                        monotonic_increasing
                        and
                        auc_preserved
                    ),

                "Error":
                    ""
            })


            print(
                f"Intercept={intercept:.3f} | "
                f"Slope={slope:.3f} | "
                f"Raw Brier="
                f"{raw_heldout_metrics['Brier_Score']:.4f} | "
                f"Recalibrated Brier="
                f"{recalibrated_heldout_metrics['Brier_Score']:.4f} | "
                f"Mean p: "
                f"{raw_heldout_metrics['Mean_Predicted_Probability']:.4f}"
                f" -> "
                f"{recalibrated_heldout_metrics['Mean_Predicted_Probability']:.4f}"
            )


        except Exception as error:

            recalibration_status_rows.append({

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Not estimable",

                "Training_N":
                    (
                        len(
                            oof_record.get(
                                "y_true",
                                []
                            )
                        )
                        if isinstance(
                            oof_record,
                            dict
                        )
                        else np.nan
                    ),

                "Training_Events":
                    (
                        int(
                            np.sum(
                                np.asarray(
                                    oof_record.get(
                                        "y_true",
                                        []
                                    )
                                )
                                ==
                                1
                            )
                        )
                        if isinstance(
                            oof_record,
                            dict
                        )
                        else np.nan
                    ),

                "Heldout_N":
                    np.nan,

                "Heldout_Events":
                    np.nan,

                "Converged":
                    False,

                "Monotonic_Increasing":
                    False,

                "AUC_Preserved":
                    False,

                "Brier_Improved_Heldout":
                    False,

                "Eligible_For_DCA":
                    False,

                "Error":
                    str(
                        error
                    )
            })


            print("Not estimable")
            print(
                f"Reason: {error}"
            )


# ============================================================
# 10. CREATE RESULT TABLES
# ============================================================

oof_recalibration_results = pd.DataFrame(
    oof_recalibration_result_rows
)


recalibrated_heldout_results = pd.DataFrame(
    recalibrated_heldout_result_rows
)


if recalibrated_heldout_prediction_rows:

    recalibrated_heldout_prediction_table = pd.concat(
        recalibrated_heldout_prediction_rows,
        ignore_index=True
    )

else:

    recalibrated_heldout_prediction_table = pd.DataFrame()


part3d2_recalibration_status = pd.DataFrame(
    recalibration_status_rows
)


# ============================================================
# 11. VERIFICATION
# ============================================================

expected_algorithm_count = sum(
    len(
        algorithms
    )
    for algorithms in oof_predictions.values()
)


successful_algorithm_count = int(
    (
        part3d2_recalibration_status[
            "Status"
        ]
        ==
        "Success"
    ).sum()
)


not_estimable_algorithm_count = int(
    (
        part3d2_recalibration_status[
            "Status"
        ]
        ==
        "Not estimable"
    ).sum()
)


for model_name, algorithms in (
    oof_predictions.items()
):

    expected_algorithms = set(
        algorithms.keys()
    )


    successful_algorithms = set(
        part3d2_recalibration_status.loc[
            (
                part3d2_recalibration_status[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                part3d2_recalibration_status[
                    "Status"
                ]
                ==
                "Success"
            ),
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )


    not_estimable_algorithms = set(
        part3d2_recalibration_status.loc[
            (
                part3d2_recalibration_status[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                part3d2_recalibration_status[
                    "Status"
                ]
                ==
                "Not estimable"
            ),
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )

    accounted_algorithms = (
        successful_algorithms
        |
        not_estimable_algorithms
    )

    missing_algorithms = sorted(
        expected_algorithms
        -
        accounted_algorithms
    )


    model_results = (
        recalibrated_heldout_results[
            recalibrated_heldout_results[
                "Screening_Model"
            ]
            ==
            model_name
        ]
    )


    all_coefficients_finite = bool(
        np.isfinite(
            model_results[
                [
                    "Recalibration_Intercept",
                    "Recalibration_Slope"
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )
        .all()
    ) if not model_results.empty else False


    all_probabilities_valid = True

    all_alignment_correct = True

    all_raw_probabilities_unchanged = True

    all_thresholds_unchanged = True


    for algorithm_name in successful_algorithms:

        recalibrated_record = (
            recalibrated_heldout_predictions[
                model_name
            ][
                algorithm_name
            ]
        )


        original_record = (
            heldout_predictions[
                model_name
            ][
                algorithm_name
            ]
        )


        recalibrated_probability = np.asarray(
            recalibrated_record[
                "recalibrated_y_prob"
            ],
            dtype=float
        )


        if not (
            np.all(
                np.isfinite(
                    recalibrated_probability
                )
            )
            and
            np.all(
                recalibrated_probability
                >=
                0
            )
            and
            np.all(
                recalibrated_probability
                <=
                1
            )
        ):

            all_probabilities_valid = False


        if not np.array_equal(
            recalibrated_record[
                "y_true"
            ],
            original_record[
                "y_true"
            ]
        ):

            all_alignment_correct = False


        if not np.array_equal(
            recalibrated_record[
                "original_index"
            ],
            original_record[
                "original_index"
            ]
        ):

            all_alignment_correct = False


        if not np.array_equal(
            recalibrated_record[
                "raw_y_prob"
            ],
            original_record[
                "y_prob"
            ]
        ):

            all_raw_probabilities_unchanged = False


        if not np.isclose(
            recalibrated_record[
                "raw_locked_threshold"
            ],
            original_record[
                "locked_threshold"
            ],
            rtol=0,
            atol=PREDICTION_TOLERANCE
        ):

            all_thresholds_unchanged = False


        if recalibrated_record[
            "classification_threshold_modified"
        ]:

            all_thresholds_unchanged = False


    all_models_converged = bool(
        part3d2_recalibration_status.loc[
            (
                part3d2_recalibration_status[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                part3d2_recalibration_status[
                    "Status"
                ]
                ==
                "Success"
            ),
            "Converged"
        ]
        .all()
    )


    all_slopes_positive = bool(
        (
            model_results[
                "Recalibration_Slope"
            ]
            >
            0
        )
        .all()
    ) if not model_results.empty else False


    dca_eligible_algorithms = set(
        model_results.loc[
            model_results[
                "Eligible_For_DCA"
            ],
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )


    nonpositive_slope_algorithms = set(
        model_results.loc[
            model_results[
                "Recalibration_Slope"
            ]
            <=
            0,
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )


    at_least_one_dca_eligible = bool(
        len(
            dca_eligible_algorithms
        )
        >=
        1
    )


    recalibration_verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            int(
                len(
                    expected_algorithms
                )
            ),

        "Successful_Algorithms":
            int(
                len(
                    successful_algorithms
                )
            ),

        "Not_Estimable_Algorithms":
            int(
                len(
                    not_estimable_algorithms
                )
            ),

        "Not_Estimable_Algorithm_Names":
            ", ".join(
                sorted(
                    not_estimable_algorithms
                )
            ),

        "Missing_Algorithms":
            ", ".join(
                missing_algorithms
            ),

        "All_Models_Converged":
            all_models_converged,

        "All_Coefficients_Finite":
            all_coefficients_finite,

        "All_Slopes_Positive":
            all_slopes_positive,

        "DCA_Eligible_Algorithms":
            int(
                len(
                    dca_eligible_algorithms
                )
            ),

        "DCA_Eligible_Algorithm_Names":
            ", ".join(
                sorted(
                    dca_eligible_algorithms
                )
            ),

        "Nonpositive_Slope_Algorithms":
            ", ".join(
                sorted(
                    nonpositive_slope_algorithms
                )
            ),

        "At_Least_One_DCA_Eligible":
            at_least_one_dca_eligible,

        "All_Recalibrated_Probabilities_Valid":
            all_probabilities_valid,

        "All_Outcome_Index_Alignment_Correct":
            all_alignment_correct,

        "All_Raw_Probabilities_Unchanged":
            all_raw_probabilities_unchanged,

        "All_Locked_Thresholds_Unchanged":
            all_thresholds_unchanged,

        "OOF_Recalibration_Ready":
            (
                len(
                    missing_algorithms
                )
                ==
                0
                and
                len(
                    successful_algorithms
                )
                >=
                1
                and
                all_models_converged
                and
                all_coefficients_finite
                and
                at_least_one_dca_eligible
                and
                all_probabilities_valid
                and
                all_alignment_correct
                and
                all_raw_probabilities_unchanged
                and
                all_thresholds_unchanged
            )
    })


part3d2_recalibration_verification = pd.DataFrame(
    recalibration_verification_rows
)


# ============================================================
# 12. PUBLICATION TABLE
# ============================================================

part3d2_recalibration_publication_table = (
    recalibrated_heldout_results[
        [
            "Validation_Set",
            "Screening_Model",
            "Algorithm",
            "N",
            "Events",
            "Prevalence",
            "Recalibration_Method",
            "Recalibration_Intercept",
            "Recalibration_Slope",
            "Raw_AUC",
            "Recalibrated_AUC",
            "Raw_Average_Precision",
            "Recalibrated_Average_Precision",
            "Raw_Brier_Score",
            "Recalibrated_Brier_Score",
            "Brier_Improvement",
            "Raw_Brier_Skill_Score",
            "Recalibrated_Brier_Skill_Score",
            "Raw_Mean_Predicted_Probability",
            "Recalibrated_Mean_Predicted_Probability",
            "Observed_Prevalence",
            "Raw_Expected_Events",
            "Recalibrated_Expected_Events",
            "Observed_Events",
            "Raw_Observed_to_Expected_Ratio",
            "Recalibrated_Observed_to_Expected_Ratio",
            "Monotonic_Increasing",
            "AUC_Preserved",
            "Eligible_For_DCA"
        ]
    ]
    .copy()
)


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 100)
print("PART 3D-2 RECALIBRATION STATUS")
print("=" * 100)

print(
    part3d2_recalibration_status[
        "Status"
    ]
    .value_counts(
        dropna=False
    )
)


print("\n" + "=" * 100)
print("HELD-OUT RAW VERSUS RECALIBRATED PROBABILITIES")
print("=" * 100)

display(
    part3d2_recalibration_publication_table
    .sort_values(
        by=[
            "Screening_Model",
            "Algorithm"
        ]
    )
    .round(
        4
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 100)
print("PART 3D-2 RECALIBRATION VERIFICATION")
print("=" * 100)

display(
    part3d2_recalibration_verification
)


# ============================================================
# 14. HARD SAFETY CHECKS
# ============================================================

unaccounted_algorithm_count = (
    expected_algorithm_count
    -
    successful_algorithm_count
    -
    not_estimable_algorithm_count
)


if unaccounted_algorithm_count > 0:

    raise RuntimeError(
        f"{unaccounted_algorithm_count} recalibration analyses "
        "were neither successful nor explicitly classified as "
        "not estimable."
    )


if not part3d2_recalibration_verification[
    "OOF_Recalibration_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies failed OOF-based "
        "recalibration verification."
    )


if not_estimable_algorithm_count > 0:

    print(
        "\nWarning: "
        f"{not_estimable_algorithm_count} recalibration "
        "mapping(s) were not estimable and are excluded from "
        "recalibrated-probability DCA."
    )

    display(
        part3d2_recalibration_status[
            part3d2_recalibration_status[
                "Status"
            ]
            ==
            "Not estimable"
        ][
            [
                "Screening_Model",
                "Algorithm",
                "Training_N",
                "Training_Events",
                "Error"
            ]
        ]
    )


# ============================================================
# 15. COHORT WARNINGS
# ============================================================

reference_model_name = next(
    iter(
        oof_predictions
    )
)


reference_algorithm_name = next(
    iter(
        oof_predictions[
            reference_model_name
        ]
    )
)


reference_oof_y = np.asarray(
    oof_predictions[
        reference_model_name
    ][
        reference_algorithm_name
    ][
        "y_true"
    ],
    dtype=int
)


reference_heldout_y = np.asarray(
    heldout_predictions[
        reference_model_name
    ][
        reference_algorithm_name
    ][
        "y_true"
    ],
    dtype=int
)


if len(
    reference_oof_y
) != EXPECTED_TRAINING_N:

    print(
        f"\nWarning: expected OOF training N="
        f"{EXPECTED_TRAINING_N:,}, observed "
        f"N={len(reference_oof_y):,}."
    )


if int(
    np.sum(
        reference_oof_y
    )
) != EXPECTED_TRAINING_EVENTS:

    print(
        f"\nWarning: expected OOF training events="
        f"{EXPECTED_TRAINING_EVENTS}, observed "
        f"events={int(np.sum(reference_oof_y))}."
    )


if len(
    reference_heldout_y
) != EXPECTED_HELDOUT_N:

    print(
        f"\nWarning: expected held-out N="
        f"{EXPECTED_HELDOUT_N:,}, observed "
        f"N={len(reference_heldout_y):,}."
    )


if int(
    np.sum(
        reference_heldout_y
    )
) != EXPECTED_HELDOUT_EVENTS:

    print(
        f"\nWarning: expected held-out events="
        f"{EXPECTED_HELDOUT_EVENTS}, observed "
        f"events={int(np.sum(reference_heldout_y))}."
    )


# ============================================================
# 16. EXPORT RESULTS
# ============================================================

oof_recalibration_results.to_csv(
    "part3d2_oof_recalibration_results.csv",
    index=False,
    encoding="utf-8-sig"
)


recalibrated_heldout_results.to_csv(
    "part3d2_recalibrated_heldout_results.csv",
    index=False,
    encoding="utf-8-sig"
)


recalibrated_heldout_prediction_table.to_csv(
    "part3d2_recalibrated_heldout_predictions_long.csv",
    index=False,
    encoding="utf-8-sig"
)


part3d2_recalibration_status.to_csv(
    "part3d2_recalibration_status.csv",
    index=False,
    encoding="utf-8-sig"
)


part3d2_recalibration_verification.to_csv(
    "part3d2_recalibration_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


part3d2_recalibration_publication_table.to_csv(
    "part3d2_recalibration_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 17. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 3D-2 VERSION 3.2 COMPLETED")
print("=" * 100)

print(
    f"Expected model-algorithm pairs : "
    f"{expected_algorithm_count}"
)


print(
    f"Successful recalibration models: "
    f"{successful_algorithm_count}"
)


print(
    f"Not-estimable mappings         : "
    f"{not_estimable_algorithm_count}"
)


print(
    f"Recalibration method           : "
    f"{RECALIBRATION_METHOD}"
)


print("\nArtifacts created:")

print("- oof_recalibration_models")
print("- oof_recalibration_results")
print("- recalibrated_heldout_predictions")
print("- recalibrated_heldout_results")
print("- recalibrated_heldout_prediction_table")
print("- part3d2_recalibration_status")
print("- part3d2_recalibration_verification")
print("- part3d2_recalibration_publication_table")


print("\nExported files:")

print("- part3d2_oof_recalibration_results.csv")
print("- part3d2_recalibrated_heldout_results.csv")
print("- part3d2_recalibrated_heldout_predictions_long.csv")
print("- part3d2_recalibration_status.csv")
print("- part3d2_recalibration_verification.csv")
print("- part3d2_recalibration_publication_table.csv")


print(
    "\nRecalibration models were fitted exclusively from genuine "
    "OOF predictions in the 2024 training partition."
)


print(
    "Held-out outcomes were used only for independent evaluation "
    "of the locked recalibration mapping."
)


print(
    "Raw probabilities and prespecified classification thresholds "
    "were retained unchanged."
)


print(
    "\nOnly mappings with Status='Success' and "
    "Eligible_For_DCA=True may be used for exploratory "
    "Decision Curve Analysis."
)

In [ ]:
display(
    recalibrated_heldout_results.loc[
        recalibrated_heldout_results[
            "Eligible_For_DCA"
        ],
        [
            "Screening_Model",
            "Algorithm",
            "Recalibration_Intercept",
            "Recalibration_Slope",
            "Raw_AUC",
            "Recalibrated_AUC",
            "Raw_Brier_Score",
            "Recalibrated_Brier_Score",
            "Raw_Mean_Predicted_Probability",
            "Recalibrated_Mean_Predicted_Probability",
            "Observed_Prevalence",
            "Recalibrated_Observed_to_Expected_Ratio",
            "Eligible_For_DCA"
        ]
    ]
    .sort_values(
        [
            "Screening_Model",
            "Recalibrated_Brier_Score"
        ]
    )
)

## PART 4: Temporal Validation (2025 cohort)

### Part 4A: Temporal Validation

In [ ]:
# ============================================================
# PART 4A: TEMPORAL VALIDATION IN THE 2025 COHORT
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef
)


# ============================================================
# 1. SETTINGS
# ============================================================

TEMPORAL_PATH = "2568-2Q9Q8Q.csv"

VALIDATION_SET_NAME = "Temporal 2025"

EXPECTED_TEMPORAL_N = 815

EXPECTED_TEMPORAL_EVENTS = 16

THRESHOLD_ATOL = 1e-12


# ============================================================
# 2. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "evaluation_models",
    "locked_thresholds",
    "threshold_summary",
    "screening_strategies",
    "OUTCOME"
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


if (
    "part3b_heldout_verification"
    in globals()
):

    if not part3b_heldout_verification[
        "Heldout_Ready"
    ].all():

        raise RuntimeError(
            "PART 3B is not fully ready."
        )


# ============================================================
# 3. LOAD TEMPORAL DATA
# ============================================================

df_temporal = pd.read_csv(
    TEMPORAL_PATH
)


if OUTCOME not in df_temporal.columns:

    raise KeyError(
        f"Outcome '{OUTCOME}' was not found "
        f"in {TEMPORAL_PATH}."
    )


if df_temporal[
    OUTCOME
].isna().any():

    raise ValueError(
        "Temporal outcome contains missing values."
    )


df_temporal[
    OUTCOME
] = (
    df_temporal[
        OUTCOME
    ]
    .astype(int)
)


observed_classes = set(
    df_temporal[
        OUTCOME
    ].unique()
)


if observed_classes != {0, 1}:

    raise ValueError(
        "Temporal outcome must contain "
        "classes 0 and 1."
    )


print("=" * 100)

print(
    "2025 TEMPORAL DATA"
)

print("=" * 100)

print(
    f"Source: {TEMPORAL_PATH}"
)

print(
    f"Original shape: {df_temporal.shape}"
)

print(
    "\nOutcome distribution:"
)

print(
    df_temporal[
        OUTCOME
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 4. BUILD TEMPORAL DATASETS
# ============================================================

temporal_datasets = {}

alignment_rows = []


reference_index = np.asarray(
    df_temporal.index
)


reference_y = (
    df_temporal[
        OUTCOME
    ]
    .to_numpy(
        dtype=int
    )
)


for model_name in evaluation_models:

    features = list(
        screening_strategies[
            model_name
        ]["features"]
    )

    missing_features = [
        feature
        for feature in features
        if feature not in df_temporal.columns
    ]

    if missing_features:

        raise KeyError(
            f"{model_name}: missing temporal "
            f"features: {missing_features}"
        )

    dataset = (
        df_temporal[
            features
            + [OUTCOME]
        ]
        .copy()
    )

    temporal_datasets[
        model_name
    ] = dataset

    same_index_order = np.array_equal(
        np.asarray(
            dataset.index
        ),
        reference_index
    )

    same_outcome_order = np.array_equal(
        dataset[
            OUTCOME
        ].to_numpy(
            dtype=int
        ),
        reference_y
    )

    feature_order_correct = (
        dataset[
            features
        ].columns.tolist()
        == features
    )

    temporal_dataset_ready = bool(
        len(dataset) == len(df_temporal)
        and same_index_order
        and same_outcome_order
        and feature_order_correct
        and len(missing_features) == 0
    )

    alignment_rows.append({
        "Screening_Model":
            model_name,

        "N":
            int(
                len(dataset)
            ),

        "Same_Index_Order":
            bool(
                same_index_order
            ),

        "Same_Outcome_Order":
            bool(
                same_outcome_order
            ),

        "Feature_Order_Correct":
            bool(
                feature_order_correct
            ),

        "Expected_Feature_Count":
            int(
                len(features)
            ),

        "Observed_Predictor_Count":
            int(
                len(dataset.columns)
                - 1
            ),

        "Missing_Features":
            ", ".join(
                missing_features
            ),

        "Temporal_Dataset_Ready":
            temporal_dataset_ready
    })


part4a_temporal_alignment_check = pd.DataFrame(
    alignment_rows
)


if not part4a_temporal_alignment_check[
    "Temporal_Dataset_Ready"
].all():

    raise RuntimeError(
        "Temporal dataset alignment failed."
    )


# ============================================================
# 5. TEMPORAL COHORT SUMMARY
# ============================================================

temporal_n = int(
    len(reference_y)
)


temporal_events = int(
    reference_y.sum()
)


temporal_nonevents = int(
    temporal_n
    - temporal_events
)


temporal_prevalence = float(
    reference_y.mean()
)


if (
    EXPECTED_TEMPORAL_N is not None
    and temporal_n != EXPECTED_TEMPORAL_N
):

    raise ValueError(
        f"Expected temporal N="
        f"{EXPECTED_TEMPORAL_N}, "
        f"observed {temporal_n}."
    )


if (
    EXPECTED_TEMPORAL_EVENTS is not None
    and temporal_events
    != EXPECTED_TEMPORAL_EVENTS
):

    raise ValueError(
        f"Expected temporal events="
        f"{EXPECTED_TEMPORAL_EVENTS}, "
        f"observed {temporal_events}."
    )


print(
    "\n"
    + "=" * 100
)

print(
    "2025 TEMPORAL COHORT"
)

print("=" * 100)

print(
    f"N={temporal_n:,}"
)

print(
    f"Events={temporal_events:,}"
)

print(
    f"Non-events={temporal_nonevents:,}"
)

print(
    f"Prevalence="
    f"{temporal_prevalence * 100:.3f}%"
)


# ============================================================
# 6. VALIDATE FROZEN PIPELINE
# ============================================================

def validate_temporal_pipeline(
    pipeline,
    model_name,
    algorithm_name
):
    """
    Validate the frozen categorical-aware pipeline.

    This function does not fit or resample temporal data.
    """

    if not hasattr(
        pipeline,
        "named_steps"
    ):

        raise TypeError(
            f"{model_name} | {algorithm_name}: "
            "invalid evaluation pipeline."
        )

    expected_step_order = [
        "over_sampler",
        "under_sampler",
        "estimator"
    ]

    observed_step_order = list(
        pipeline
        .named_steps
        .keys()
    )

    if observed_step_order != expected_step_order:

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            f"expected {expected_step_order}, "
            f"observed {observed_step_order}."
        )

    expected_sampler = str(
        screening_strategies[
            model_name
        ]["sampler_type"]
    ).upper()

    observed_sampler = type(
        pipeline
        .named_steps[
            "over_sampler"
        ]
    ).__name__

    if observed_sampler != expected_sampler:

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            f"expected {expected_sampler}, "
            f"observed {observed_sampler}."
        )

    observed_under_sampler = type(
        pipeline
        .named_steps[
            "under_sampler"
        ]
    ).__name__

    if observed_under_sampler != "RandomUnderSampler":

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            "invalid under-sampler."
        )

    if not hasattr(
        pipeline,
        "predict_proba"
    ):

        raise AttributeError(
            f"{model_name} | {algorithm_name}: "
            "predict_proba is unavailable."
        )

    pipeline_description = " | ".join(
        f"{step_name}:{type(step_object).__name__}"
        for step_name, step_object
        in pipeline.steps
    )

    return pipeline_description


# ============================================================
# 7. POSITIVE-CLASS PROBABILITY
# ============================================================

def positive_probability(
    estimator,
    X
):
    """
    Return probability corresponding to class 1.
    """

    probability_matrix = np.asarray(
        estimator.predict_proba(X),
        dtype=float
    )

    classes = np.asarray(
        estimator.classes_
    )

    positive_positions = np.flatnonzero(
        classes == 1
    )

    if (
        probability_matrix.ndim != 2
        or len(positive_positions) != 1
    ):

        raise ValueError(
            "Unable to identify class-1 probability."
        )

    probabilities = probability_matrix[
        :,
        int(
            positive_positions[0]
        )
    ]

    if not np.isfinite(
        probabilities
    ).all():

        raise ValueError(
            "Temporal probabilities contain "
            "nonfinite values."
        )

    if (
        (probabilities < 0).any()
        or (probabilities > 1).any()
    ):

        raise ValueError(
            "Temporal probabilities are "
            "outside [0,1]."
        )

    return probabilities


# ============================================================
# 8. TEMPORAL PERFORMANCE METRICS
# ============================================================

def calculate_temporal_metrics(
    y_true,
    y_prob,
    threshold
):
    """
    Calculate temporal performance using a fixed threshold.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float
    )

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    negative_predictive_value = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    return {
        "y_pred":
            y_pred,

        "AUC":
            float(
                roc_auc_score(
                    y_true,
                    y_prob
                )
            ),

        "Average_Precision":
            float(
                average_precision_score(
                    y_true,
                    y_prob
                )
            ),

        "Brier_Score":
            float(
                brier_score_loss(
                    y_true,
                    y_prob
                )
            ),

        "Sensitivity":
            float(sensitivity),

        "Recall":
            float(sensitivity),

        "Specificity":
            float(specificity),

        "Precision":
            float(precision),

        "Negative_Predictive_Value":
            float(
                negative_predictive_value
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    zero_division=0
                )
            ),

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "Matthews_Correlation_Coefficient":
            float(
                matthews_corrcoef(
                    y_true,
                    y_pred
                )
            ),

        "Predicted_Positive_N":
            int(
                y_pred.sum()
            ),

        "Predicted_Positive_Rate":
            float(
                y_pred.mean()
            ),

        "TP":
            int(tp),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TN":
            int(tn)
    }


# ============================================================
# 9. THRESHOLD METADATA
# ============================================================

def get_threshold_metadata(
    model_name,
    algorithm_name
):
    """
    Retrieve one development OOF threshold record.
    """

    matching_rows = threshold_summary.loc[
        (
            threshold_summary[
                "Screening_Model"
            ] == model_name
        )
        &
        (
            threshold_summary[
                "Algorithm"
            ] == algorithm_name
        )
    ]

    if len(matching_rows) != 1:

        raise ValueError(
            f"{model_name} | {algorithm_name}: "
            "invalid threshold record count."
        )

    row = matching_rows.iloc[0]

    required_fields = [
        "Locked_Threshold",
        "OOF_AUC",
        "OOF_Average_Precision",
        "OOF_Sensitivity",
        "OOF_Specificity",
        "OOF_Precision",
        "OOF_F1",
        "Threshold_Rule",
        "Target_Recall_Achieved",
        "Probability_Degenerate_Flag"
    ]

    return {
        field: row[field]
        for field in required_fields
    }


# ============================================================
# 10. RESULT CONTAINERS
# ============================================================

temporal_predictions = {}

temporal_result_rows = []

temporal_prediction_rows = []

temporal_status_rows = []


# ============================================================
# 11. TEMPORAL EVALUATION
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "PART 4A: TEMPORAL VALIDATION "
    "IN THE 2025 COHORT"
)

print("=" * 100)


for model_name in evaluation_models:

    features = list(
        screening_strategies[
            model_name
        ]["features"]
    )

    dataset = (
        temporal_datasets[
            model_name
        ]
        .copy()
    )

    X_temporal = (
        dataset[
            features
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    y_temporal = (
        dataset[
            OUTCOME
        ]
        .copy()
        .astype(int)
        .reset_index(
            drop=True
        )
    )

    temporal_original_index = np.asarray(
        dataset.index
    )

    if X_temporal.columns.tolist() != features:

        raise ValueError(
            f"{model_name}: temporal feature "
            "order mismatch."
        )

    if X_temporal.isna().any().any():

        raise ValueError(
            f"{model_name}: temporal predictors "
            "contain missing values."
        )

    fitted_algorithms = set(
        evaluation_models[
            model_name
        ].keys()
    )

    threshold_algorithms = set(
        locked_thresholds[
            model_name
        ].keys()
    )

    if fitted_algorithms != threshold_algorithms:

        raise ValueError(
            f"{model_name}: fitted models and "
            "thresholds differ."
        )

    temporal_predictions[
        model_name
    ] = {}

    for algorithm_name, fitted_pipeline in (
        evaluation_models[
            model_name
        ].items()
    ):

        try:

            pipeline_description = (
                validate_temporal_pipeline(
                    fitted_pipeline,
                    model_name,
                    algorithm_name
                )
            )

            locked_threshold = float(
                locked_thresholds[
                    model_name
                ][
                    algorithm_name
                ]
            )

            threshold_metadata = (
                get_threshold_metadata(
                    model_name,
                    algorithm_name
                )
            )

            if not np.isclose(
                locked_threshold,
                float(
                    threshold_metadata[
                        "Locked_Threshold"
                    ]
                ),
                rtol=0,
                atol=THRESHOLD_ATOL
            ):

                raise RuntimeError(
                    "Locked threshold mismatch."
                )

            # Prediction only: no fit or resampling.

            y_prob = positive_probability(
                fitted_pipeline,
                X_temporal
            )

            metrics = (
                calculate_temporal_metrics(
                    y_temporal,
                    y_prob,
                    locked_threshold
                )
            )

            y_pred = metrics[
                "y_pred"
            ]

            temporal_predictions[
                model_name
            ][
                algorithm_name
            ] = {
                "validation_set":
                    VALIDATION_SET_NAME,

                "features":
                    features.copy(),

                "y_true":
                    y_temporal.to_numpy(
                        dtype=int
                    ),

                "y_prob":
                    y_prob,

                "y_pred":
                    y_pred,

                "threshold":
                    locked_threshold,

                "locked_threshold":
                    locked_threshold,

                "row_position":
                    np.arange(
                        temporal_n
                    ),

                "original_index":
                    temporal_original_index.copy(),

                "n":
                    temporal_n,

                "events":
                    temporal_events,

                "prevalence":
                    temporal_prevalence,

                "auc":
                    metrics["AUC"],

                "average_precision":
                    metrics[
                        "Average_Precision"
                    ],

                "brier_score":
                    metrics[
                        "Brier_Score"
                    ],

                "sensitivity":
                    metrics[
                        "Sensitivity"
                    ],

                "specificity":
                    metrics[
                        "Specificity"
                    ],

                "precision":
                    metrics[
                        "Precision"
                    ],

                "negative_predictive_value":
                    metrics[
                        "Negative_Predictive_Value"
                    ],

                "f1":
                    metrics["F1"],

                "accuracy":
                    metrics[
                        "Accuracy"
                    ],

                "balanced_accuracy":
                    metrics[
                        "Balanced_Accuracy"
                    ],

                "mcc":
                    metrics[
                        "Matthews_Correlation_Coefficient"
                    ],

                "predicted_positive_n":
                    metrics[
                        "Predicted_Positive_N"
                    ],

                "predicted_positive_rate":
                    metrics[
                        "Predicted_Positive_Rate"
                    ],

                "tp":
                    metrics["TP"],

                "fp":
                    metrics["FP"],

                "fn":
                    metrics["FN"],

                "tn":
                    metrics["TN"],

                "pipeline_description":
                    pipeline_description,

                "probability_recalibrated":
                    False,

                "model_refitted":
                    False,

                "threshold_modified":
                    False
            }

            temporal_prediction_rows.append(
                pd.DataFrame({
                    "Validation_Set":
                        VALIDATION_SET_NAME,

                    "Screening_Model":
                        model_name,

                    "Algorithm":
                        algorithm_name,

                    "Row_Position":
                        np.arange(
                            temporal_n
                        ),

                    "Original_Index":
                        temporal_original_index,

                    "y_true":
                        y_temporal.to_numpy(
                            dtype=int
                        ),

                    "y_prob":
                        y_prob,

                    "Locked_Threshold":
                        locked_threshold,

                    "y_pred":
                        y_pred
                })
            )

            result_row = {
                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Sampler_Type":
                    screening_strategies[
                        model_name
                    ]["sampler_type"],

                "N":
                    temporal_n,

                "Events":
                    temporal_events,

                "Prevalence":
                    temporal_prevalence,

                "Locked_Threshold":
                    locked_threshold
            }

            result_row.update({
                key: value
                for key, value
                in metrics.items()
                if key != "y_pred"
            })

            result_row.update({
                "OOF_AUC":
                    float(
                        threshold_metadata[
                            "OOF_AUC"
                        ]
                    ),

                "OOF_Average_Precision":
                    float(
                        threshold_metadata[
                            "OOF_Average_Precision"
                        ]
                    ),

                "OOF_Sensitivity":
                    float(
                        threshold_metadata[
                            "OOF_Sensitivity"
                        ]
                    ),

                "OOF_Specificity":
                    float(
                        threshold_metadata[
                            "OOF_Specificity"
                        ]
                    ),

                "OOF_Precision":
                    float(
                        threshold_metadata[
                            "OOF_Precision"
                        ]
                    ),

                "OOF_F1":
                    float(
                        threshold_metadata[
                            "OOF_F1"
                        ]
                    ),

                "AUC_Change_Temporal_minus_OOF":
                    (
                        metrics["AUC"]
                        - float(
                            threshold_metadata[
                                "OOF_AUC"
                            ]
                        )
                    ),

                "Sensitivity_Change_Temporal_minus_OOF":
                    (
                        metrics["Sensitivity"]
                        - float(
                            threshold_metadata[
                                "OOF_Sensitivity"
                            ]
                        )
                    ),

                "F1_Change_Temporal_minus_OOF":
                    (
                        metrics["F1"]
                        - float(
                            threshold_metadata[
                                "OOF_F1"
                            ]
                        )
                    ),

                "Threshold_Rule":
                    str(
                        threshold_metadata[
                            "Threshold_Rule"
                        ]
                    ),

                "Target_Recall_Achieved_OOF":
                    bool(
                        threshold_metadata[
                            "Target_Recall_Achieved"
                        ]
                    ),

                "Probability_Degenerate_Flag_OOF":
                    bool(
                        threshold_metadata[
                            "Probability_Degenerate_Flag"
                        ]
                    ),

                "Pipeline_Description":
                    pipeline_description
            })

            temporal_result_rows.append(
                result_row
            )

            temporal_status_rows.append({
                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Success",

                "N":
                    temporal_n,

                "Events":
                    temporal_events,

                "Locked_Threshold":
                    locked_threshold,

                "Pipeline_Validated":
                    True,

                "Prediction_Length_Correct":
                    bool(
                        len(y_prob)
                        == temporal_n
                    ),

                "No_Model_Refit":
                    True,

                "No_Threshold_Modification":
                    True,

                "No_Probability_Recalibration":
                    True,

                "Error":
                    ""
            })

            print(
                f"{model_name} | "
                f"{algorithm_name} | "
                f"AUC={metrics['AUC']:.4f} | "
                f"AP={metrics['Average_Precision']:.4f} | "
                f"Sensitivity={metrics['Sensitivity']:.3f} | "
                f"Specificity={metrics['Specificity']:.3f} | "
                f"F1={metrics['F1']:.3f}"
            )

        except Exception as error:

            temporal_status_rows.append({
                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Failed",

                "N":
                    temporal_n,

                "Events":
                    temporal_events,

                "Locked_Threshold":
                    locked_thresholds.get(
                        model_name,
                        {}
                    ).get(
                        algorithm_name,
                        np.nan
                    ),

                "Pipeline_Validated":
                    False,

                "Prediction_Length_Correct":
                    False,

                "No_Model_Refit":
                    True,

                "No_Threshold_Modification":
                    True,

                "No_Probability_Recalibration":
                    True,

                "Error":
                    str(error)
            })

            print(
                f"{model_name} | "
                f"{algorithm_name} | "
                f"Failed: {error}"
            )


# ============================================================
# 12. CREATE RESULT TABLES
# ============================================================

temporal_results = pd.DataFrame(
    temporal_result_rows
)


temporal_status = pd.DataFrame(
    temporal_status_rows
)


if temporal_prediction_rows:

    temporal_prediction_table = pd.concat(
        temporal_prediction_rows,
        ignore_index=True
    )

else:

    temporal_prediction_table = pd.DataFrame()


primary_temporal_predictions = (
    temporal_predictions
)


# ============================================================
# 13. TEMPORAL VERIFICATION
# ============================================================

verification_rows = []


for model_name in evaluation_models:

    expected_algorithms = set(
        evaluation_models[
            model_name
        ].keys()
    )

    successful_algorithms = set(
        temporal_predictions.get(
            model_name,
            {}
        ).keys()
    )

    prediction_records = temporal_predictions.get(
        model_name,
        {}
    )

    all_n_correct = all(
        prediction_records[
            algorithm_name
        ]["n"] == temporal_n
        for algorithm_name
        in successful_algorithms
    )

    all_event_counts_correct = all(
        prediction_records[
            algorithm_name
        ]["events"] == temporal_events
        for algorithm_name
        in successful_algorithms
    )

    all_thresholds_matched = all(
        np.isclose(
            prediction_records[
                algorithm_name
            ]["locked_threshold"],
            locked_thresholds[
                model_name
            ][
                algorithm_name
            ],
            rtol=0,
            atol=THRESHOLD_ATOL
        )
        for algorithm_name
        in successful_algorithms
    )

    expected_sampler = str(
        screening_strategies[
            model_name
        ]["sampler_type"]
    ).upper()

    all_pipelines_validated = all(
        (
            f"over_sampler:{expected_sampler}"
            in prediction_records[
                algorithm_name
            ]["pipeline_description"]
        )
        and
        (
            "under_sampler:RandomUnderSampler"
            in prediction_records[
                algorithm_name
            ]["pipeline_description"]
        )
        for algorithm_name
        in successful_algorithms
    )

    no_model_refit = all(
        not prediction_records[
            algorithm_name
        ]["model_refitted"]
        for algorithm_name
        in successful_algorithms
    )

    no_threshold_modification = all(
        not prediction_records[
            algorithm_name
        ]["threshold_modified"]
        for algorithm_name
        in successful_algorithms
    )

    no_probability_recalibration = all(
        not prediction_records[
            algorithm_name
        ]["probability_recalibrated"]
        for algorithm_name
        in successful_algorithms
    )

    temporal_ready = bool(
        successful_algorithms
        == expected_algorithms
        and all_n_correct
        and all_event_counts_correct
        and all_thresholds_matched
        and all_pipelines_validated
        and no_model_refit
        and no_threshold_modification
        and no_probability_recalibration
    )

    verification_rows.append({
        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            len(
                expected_algorithms
            ),

        "Successful_Algorithms":
            len(
                successful_algorithms
            ),

        "All_N_Correct":
            all_n_correct,

        "All_Event_Counts_Correct":
            all_event_counts_correct,

        "All_Locked_Thresholds_Matched":
            all_thresholds_matched,

        "All_Pipelines_Validated":
            all_pipelines_validated,

        "No_Model_Refit":
            no_model_refit,

        "No_Threshold_Modification":
            no_threshold_modification,

        "No_Probability_Recalibration":
            no_probability_recalibration,

        "Temporal_Ready":
            temporal_ready
    })


part4a_temporal_verification = pd.DataFrame(
    verification_rows
)


# ============================================================
# 14. HARD SAFETY CHECKS
# ============================================================

successful_temporal_count = int(
    (
        temporal_status[
            "Status"
        ] == "Success"
    ).sum()
)


failed_temporal_count = int(
    (
        temporal_status[
            "Status"
        ] == "Failed"
    ).sum()
)


if failed_temporal_count > 0:

    display(
        temporal_status.loc[
            temporal_status[
                "Status"
            ] == "Failed",
            [
                "Screening_Model",
                "Algorithm",
                "Error"
            ]
        ]
    )

    raise RuntimeError(
        f"{failed_temporal_count} "
        "temporal evaluations failed."
    )


if not part4a_temporal_verification[
    "Temporal_Ready"
].all():

    raise RuntimeError(
        "One or more strategies failed "
        "temporal verification."
    )


# ============================================================
# 15. PUBLICATION TABLE
# ============================================================

publication_columns = [
    "Screening_Model",
    "Algorithm",
    "N",
    "Events",
    "Locked_Threshold",
    "AUC",
    "Average_Precision",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "TP",
    "FP",
    "FN",
    "TN"
]


temporal_publication_table = (
    temporal_results[
        publication_columns
    ]
    .copy()
)


# ============================================================
# 16. HELD-OUT VERSUS TEMPORAL COMPARISON
# ============================================================

if "heldout_results" in globals():

    heldout_compare = (
        heldout_results[
            [
                "Screening_Model",
                "Algorithm",
                "AUC",
                "Average_Precision",
                "Sensitivity",
                "Specificity",
                "Precision",
                "F1"
            ]
        ]
        .rename(columns={
            "AUC":
                "Heldout_AUC",

            "Average_Precision":
                "Heldout_AP",

            "Sensitivity":
                "Heldout_Sensitivity",

            "Specificity":
                "Heldout_Specificity",

            "Precision":
                "Heldout_Precision",

            "F1":
                "Heldout_F1"
        })
    )

    temporal_compare = (
        temporal_results[
            [
                "Screening_Model",
                "Algorithm",
                "AUC",
                "Average_Precision",
                "Sensitivity",
                "Specificity",
                "Precision",
                "F1"
            ]
        ]
        .rename(columns={
            "AUC":
                "Temporal_AUC",

            "Average_Precision":
                "Temporal_AP",

            "Sensitivity":
                "Temporal_Sensitivity",

            "Specificity":
                "Temporal_Specificity",

            "Precision":
                "Temporal_Precision",

            "F1":
                "Temporal_F1"
        })
    )

    temporal_vs_heldout_table = (
        heldout_compare
        .merge(
            temporal_compare,
            on=[
                "Screening_Model",
                "Algorithm"
            ],
            how="inner"
        )
    )

else:

    temporal_vs_heldout_table = pd.DataFrame()


# ============================================================
# 17. EXPORT
# ============================================================

part4a_temporal_alignment_check.to_csv(
    "part4a_temporal_dataset_alignment_check.csv",
    index=False,
    encoding="utf-8-sig"
)


temporal_results.to_csv(
    "part4a_temporal_results.csv",
    index=False,
    encoding="utf-8-sig"
)


temporal_status.to_csv(
    "part4a_temporal_status.csv",
    index=False,
    encoding="utf-8-sig"
)


temporal_prediction_table.to_csv(
    "part4a_temporal_predictions_long.csv",
    index=False,
    encoding="utf-8-sig"
)


part4a_temporal_verification.to_csv(
    "part4a_temporal_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


temporal_publication_table.to_csv(
    "part4a_temporal_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


temporal_vs_heldout_table.to_csv(
    "part4a_temporal_vs_heldout_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 18. FINAL OUTPUT
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "PART 4A COMPLETED"
)

print("=" * 100)


print(
    "Successful temporal evaluations: "
    f"{successful_temporal_count}"
)


print(
    "Failed temporal evaluations: "
    f"{failed_temporal_count}"
)


print(
    "\nTemporal performance:"
)


display(
    temporal_publication_table
)


print(
    "\nTemporal verification:"
)


display(
    part4a_temporal_verification
)


print(
    "\nNo temporal observation was used "
    "for model fitting or resampling."
)


print(
    "No classification threshold was modified."
)

### PART 4B: Bootstrap 95% CI for Temporal Validation 2025

In [ ]:
# ============================================================
# PART 4B: STRATIFIED BOOTSTRAP 95% CONFIDENCE INTERVALS
# FOR TEMPORAL 2025 VALIDATION
# ============================================================
# Uses only the fixed participant-level predictions from PART 4A.
# No model fitting, resampling, prediction regeneration,
# recalibration, or threshold optimization is performed here.
# ============================================================

import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. SETTINGS
# ============================================================

N_BOOTSTRAP = 2000
RANDOM_STATE = 42
CI_LOWER_PERCENTILE = 2.5
CI_UPPER_PERCENTILE = 97.5
MINIMUM_VALID_BOOTSTRAPS = 1800

EXPECTED_TEMPORAL_N = 815
EXPECTED_TEMPORAL_EVENTS = 16
VALIDATION_SET_NAME = "Temporal 2025"

REQUIRED_CI_METRICS = [
    "AUC",
    "Average_Precision",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "Balanced_Accuracy",
    "Matthews_Correlation_Coefficient",
    "Brier_Score",
    "Youden_Index",
]

OPTIONAL_CI_METRICS = [
    "Negative_Predictive_Value",
    "Positive_Likelihood_Ratio",
    "Negative_Likelihood_Ratio",
    "Diagnostic_Odds_Ratio",
]

BOOTSTRAP_METRICS = REQUIRED_CI_METRICS + OPTIONAL_CI_METRICS


# ============================================================
# 2. REQUIRED OBJECTS FROM PART 4A
# ============================================================

required_objects = [
    "temporal_predictions",
    "temporal_results",
    "part4a_temporal_verification",
]

missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(
        "Missing required PART 4A objects: " + ", ".join(missing_objects)
    )

if not isinstance(temporal_predictions, dict) or not temporal_predictions:
    raise TypeError("temporal_predictions must be a nonempty dictionary.")
if not isinstance(temporal_results, pd.DataFrame):
    raise TypeError("temporal_results must be a pandas DataFrame.")
if not isinstance(part4a_temporal_verification, pd.DataFrame):
    raise TypeError("part4a_temporal_verification must be a DataFrame.")
if "Temporal_Ready" not in part4a_temporal_verification.columns:
    raise KeyError("PART 4A verification does not contain Temporal_Ready.")
if not part4a_temporal_verification["Temporal_Ready"].astype(bool).all():
    failed = part4a_temporal_verification.loc[
        ~part4a_temporal_verification["Temporal_Ready"].astype(bool),
        "Screening_Model",
    ].astype(str).tolist()
    raise RuntimeError(
        "PART 4B cannot proceed because these strategies are not ready: "
        + ", ".join(failed)
    )


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator != 0 else np.nan


def calculate_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= float(threshold)).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    specificity = safe_divide(tn, tn + fp)
    precision = precision_score(y_true, y_pred, zero_division=0)
    npv = safe_divide(tn, tn + fn)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    lr_positive = safe_divide(sensitivity, 1.0 - specificity)
    lr_negative = safe_divide(1.0 - sensitivity, specificity)
    diagnostic_odds_ratio = safe_divide(lr_positive, lr_negative)

    return {
        "AUC": roc_auc_score(y_true, y_prob),
        "Average_Precision": average_precision_score(y_true, y_prob),
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "Precision": float(precision),
        "Negative_Predictive_Value": float(npv),
        "F1": float(f1),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Matthews_Correlation_Coefficient": matthews_corrcoef(y_true, y_pred),
        "Brier_Score": brier_score_loss(y_true, y_prob),
        "Youden_Index": float(sensitivity + specificity - 1.0),
        "Positive_Likelihood_Ratio": float(lr_positive),
        "Negative_Likelihood_Ratio": float(lr_negative),
        "Diagnostic_Odds_Ratio": float(diagnostic_odds_ratio),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
    }


def stratified_bootstrap_indices(y_true, rng):
    y_true = np.asarray(y_true, dtype=int)
    negative_indices = np.flatnonzero(y_true == 0)
    positive_indices = np.flatnonzero(y_true == 1)

    if len(negative_indices) == 0 or len(positive_indices) == 0:
        raise ValueError("Both outcome classes are required for bootstrap analysis.")

    sampled = np.concatenate([
        rng.choice(negative_indices, size=len(negative_indices), replace=True),
        rng.choice(positive_indices, size=len(positive_indices), replace=True),
    ])
    rng.shuffle(sampled)
    return sampled


def summarize_metric(values, point_estimate):
    values = np.asarray(values, dtype=float)
    finite_values = values[np.isfinite(values)]
    valid_n = int(len(finite_values))
    invalid_n = int(len(values) - valid_n)

    if valid_n == 0:
        return {
            "Point_Estimate": float(point_estimate),
            "Bootstrap_Mean": np.nan,
            "Bootstrap_Median": np.nan,
            "Bootstrap_Standard_Error": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "Valid_Replicates": 0,
            "Invalid_Replicates": invalid_n,
            "CI_Computed": False,
        }

    return {
        "Point_Estimate": float(point_estimate),
        "Bootstrap_Mean": float(np.mean(finite_values)),
        "Bootstrap_Median": float(np.median(finite_values)),
        "Bootstrap_Standard_Error": float(np.std(finite_values, ddof=1)),
        "CI_Lower": float(np.percentile(finite_values, CI_LOWER_PERCENTILE)),
        "CI_Upper": float(np.percentile(finite_values, CI_UPPER_PERCENTILE)),
        "Valid_Replicates": valid_n,
        "Invalid_Replicates": invalid_n,
        "CI_Computed": valid_n >= MINIMUM_VALID_BOOTSTRAPS,
    }


def format_estimate_ci(estimate, lower, upper, digits=3):
    if not np.all(np.isfinite([estimate, lower, upper])):
        return "Not estimable"
    return f"{estimate:.{digits}f} ({lower:.{digits}f}-{upper:.{digits}f})"


# ============================================================
# 4. VERIFY THAT ALL RECORDS USE THE SAME TEMPORAL COHORT
# ============================================================

all_records = [
    (model_name, algorithm_name, record)
    for model_name, algorithms in temporal_predictions.items()
    for algorithm_name, record in algorithms.items()
]

if not all_records:
    raise ValueError("No temporal prediction records were found.")

reference_y_true = np.asarray(all_records[0][2]["y_true"], dtype=int)
reference_n = len(reference_y_true)
reference_events = int(reference_y_true.sum())

if reference_n != EXPECTED_TEMPORAL_N:
    raise ValueError(
        f"Temporal N mismatch: expected {EXPECTED_TEMPORAL_N}, observed {reference_n}."
    )
if reference_events != EXPECTED_TEMPORAL_EVENTS:
    raise ValueError(
        "Temporal event-count mismatch: expected "
        f"{EXPECTED_TEMPORAL_EVENTS}, observed {reference_events}."
    )

for model_name, algorithm_name, record in all_records:
    required_keys = ["y_true", "y_prob", "locked_threshold"]
    missing_keys = [key for key in required_keys if key not in record]
    if missing_keys:
        raise KeyError(
            f"{model_name} | {algorithm_name}: missing " + ", ".join(missing_keys)
        )

    y_true_check = np.asarray(record["y_true"], dtype=int)
    y_prob_check = np.asarray(record["y_prob"], dtype=float)
    if not np.array_equal(y_true_check, reference_y_true):
        raise RuntimeError(
            f"{model_name} | {algorithm_name}: outcome order differs from PART 4A."
        )
    if len(y_prob_check) != reference_n or not np.isfinite(y_prob_check).all():
        raise RuntimeError(
            f"{model_name} | {algorithm_name}: invalid probability vector."
        )
    if not np.isfinite(float(record["locked_threshold"])):
        raise RuntimeError(
            f"{model_name} | {algorithm_name}: invalid locked threshold."
        )


# ============================================================
# 5. STRATIFIED PARTICIPANT-LEVEL BOOTSTRAP
# ============================================================

bootstrap_result_rows = []
bootstrap_distribution_rows = []
bootstrap_status_rows = []

print("=" * 100)
print("PART 4B: TEMPORAL 2025 STRATIFIED BOOTSTRAP 95% CI")
print("=" * 100)
print(f"Temporal cohort: N={reference_n:,}; events={reference_events:,}")
print(f"Bootstrap replicates requested: {N_BOOTSTRAP:,} per model")
print("CI method: stratified percentile bootstrap (2.5th-97.5th percentiles)")

pair_number = 0
for model_name, algorithms in temporal_predictions.items():
    for algorithm_name, record in algorithms.items():
        pair_number += 1
        start_time = time.time()
        algorithm_seed = RANDOM_STATE + pair_number - 1
        rng = np.random.default_rng(algorithm_seed)

        print(f"\n[{pair_number:02d}/{len(all_records):02d}] {model_name} | {algorithm_name}")

        try:
            y_true = np.asarray(record["y_true"], dtype=int)
            y_prob = np.asarray(record["y_prob"], dtype=float)
            threshold = float(record["locked_threshold"])
            point_metrics = calculate_metrics(y_true, y_prob, threshold)
            metric_values = {metric: [] for metric in BOOTSTRAP_METRICS}
            failed_replicates = 0

            for bootstrap_id in range(1, N_BOOTSTRAP + 1):
                try:
                    sampled_indices = stratified_bootstrap_indices(y_true, rng)
                    sampled_metrics = calculate_metrics(
                        y_true[sampled_indices],
                        y_prob[sampled_indices],
                        threshold,
                    )

                    for metric_name in BOOTSTRAP_METRICS:
                        value = float(sampled_metrics[metric_name])
                        metric_values[metric_name].append(value)
                        bootstrap_distribution_rows.append({
                            "Validation_Set": VALIDATION_SET_NAME,
                            "Screening_Model": model_name,
                            "Algorithm": algorithm_name,
                            "Bootstrap_ID": bootstrap_id,
                            "Random_Seed": algorithm_seed,
                            "Metric": metric_name,
                            "Value": value,
                            "Bootstrap_N": len(sampled_indices),
                            "Bootstrap_Events": int(y_true[sampled_indices].sum()),
                        })
                except Exception:
                    failed_replicates += 1
                    for metric_name in BOOTSTRAP_METRICS:
                        metric_values[metric_name].append(np.nan)

            required_ci_flags = []
            optional_not_estimable = []

            for metric_name in BOOTSTRAP_METRICS:
                summary = summarize_metric(
                    metric_values[metric_name], point_metrics[metric_name]
                )
                bootstrap_result_rows.append({
                    "Validation_Set": VALIDATION_SET_NAME,
                    "Screening_Model": model_name,
                    "Algorithm": algorithm_name,
                    "N": len(y_true),
                    "Events": int(y_true.sum()),
                    "Nonevents": int((y_true == 0).sum()),
                    "Prevalence": float(y_true.mean()),
                    "Locked_Threshold": threshold,
                    "Metric": metric_name,
                    **summary,
                    "Bootstrap_Requested": N_BOOTSTRAP,
                    "Random_Seed": algorithm_seed,
                })

                if metric_name in REQUIRED_CI_METRICS:
                    required_ci_flags.append(bool(summary["CI_Computed"]))
                elif not bool(summary["CI_Computed"]):
                    optional_not_estimable.append(metric_name)

            all_required_ci_computed = all(required_ci_flags)
            status = "Success" if all_required_ci_computed else "Insufficient valid replicates"
            elapsed_seconds = time.time() - start_time

            bootstrap_status_rows.append({
                "Validation_Set": VALIDATION_SET_NAME,
                "Screening_Model": model_name,
                "Algorithm": algorithm_name,
                "Status": status,
                "N": len(y_true),
                "Events": int(y_true.sum()),
                "Nonevents": int((y_true == 0).sum()),
                "Locked_Threshold": threshold,
                "Bootstrap_Requested": N_BOOTSTRAP,
                "Bootstrap_Completed": N_BOOTSTRAP - failed_replicates,
                "Bootstrap_Skipped": 0,
                "Bootstrap_Failed": failed_replicates,
                "Minimum_Required_Metric_Valid_Replicates": min(
                    sum(np.isfinite(metric_values[m])) for m in REQUIRED_CI_METRICS
                ),
                "All_Required_Metrics_CI_Computed": all_required_ci_computed,
                "Optional_Metrics_Not_Estimable": ", ".join(optional_not_estimable),
                "Random_Seed": algorithm_seed,
                "Elapsed_Seconds": elapsed_seconds,
                "Error": "",
            })
            print(
                f"Status={status} | completed={N_BOOTSTRAP - failed_replicates:,}/"
                f"{N_BOOTSTRAP:,} | failed={failed_replicates:,}"
            )

        except Exception as error:
            bootstrap_status_rows.append({
                "Validation_Set": VALIDATION_SET_NAME,
                "Screening_Model": model_name,
                "Algorithm": algorithm_name,
                "Status": "Failed",
                "N": len(record.get("y_true", [])),
                "Events": int(np.sum(record.get("y_true", []))),
                "Nonevents": np.nan,
                "Locked_Threshold": record.get("locked_threshold", np.nan),
                "Bootstrap_Requested": N_BOOTSTRAP,
                "Bootstrap_Completed": 0,
                "Bootstrap_Skipped": 0,
                "Bootstrap_Failed": N_BOOTSTRAP,
                "Minimum_Required_Metric_Valid_Replicates": 0,
                "All_Required_Metrics_CI_Computed": False,
                "Optional_Metrics_Not_Estimable": "",
                "Random_Seed": algorithm_seed,
                "Elapsed_Seconds": time.time() - start_time,
                "Error": str(error),
            })
            print(f"Status=Failed | Error={error}")


# ============================================================
# 6. RESULT AND VERIFICATION TABLES
# ============================================================

part4b_bootstrap_results = pd.DataFrame(bootstrap_result_rows)
part4b_bootstrap_distribution = pd.DataFrame(bootstrap_distribution_rows)
part4b_bootstrap_status = pd.DataFrame(bootstrap_status_rows)

verification_rows = []
for model_name, algorithms in temporal_predictions.items():
    expected_algorithms = set(algorithms)
    model_status = part4b_bootstrap_status.loc[
        part4b_bootstrap_status["Screening_Model"] == model_name
    ]
    successful_algorithms = set(
        model_status.loc[model_status["Status"] == "Success", "Algorithm"].astype(str)
    )
    model_results = part4b_bootstrap_results.loc[
        part4b_bootstrap_results["Screening_Model"] == model_name
    ]

    metric_sets_correct = all(
        set(model_results.loc[
            model_results["Algorithm"] == algorithm_name, "Metric"
        ].astype(str)) == set(BOOTSTRAP_METRICS)
        for algorithm_name in expected_algorithms
    )

    point_estimates_match = True
    for algorithm_name in expected_algorithms:
        record = algorithms[algorithm_name]
        expected_points = calculate_metrics(
            record["y_true"], record["y_prob"], record["locked_threshold"]
        )
        for metric_name in BOOTSTRAP_METRICS:
            rows = model_results.loc[
                (model_results["Algorithm"] == algorithm_name)
                & (model_results["Metric"] == metric_name),
                "Point_Estimate",
            ]
            if len(rows) != 1 or not np.isclose(
                float(rows.iloc[0]), float(expected_points[metric_name]),
                rtol=0, atol=1e-12, equal_nan=True,
            ):
                point_estimates_match = False

    verification_rows.append({
        "Screening_Model": model_name,
        "Expected_Algorithms": len(expected_algorithms),
        "Successful_Algorithms": len(successful_algorithms),
        "Missing_Algorithms": ", ".join(sorted(expected_algorithms - successful_algorithms)),
        "Expected_Metric_Rows": len(expected_algorithms) * len(BOOTSTRAP_METRICS),
        "Observed_Metric_Rows": len(model_results),
        "All_Metrics_Present": metric_sets_correct,
        "All_Required_CIs_Computed": bool(
            model_status["All_Required_Metrics_CI_Computed"].all()
        ),
        "Point_Estimates_Match_PART4A": point_estimates_match,
        "All_N_Correct": bool((model_status["N"] == EXPECTED_TEMPORAL_N).all()),
        "All_Event_Counts_Correct": bool(
            (model_status["Events"] == EXPECTED_TEMPORAL_EVENTS).all()
        ),
        "Temporal_Bootstrap_Ready": (
            successful_algorithms == expected_algorithms
            and metric_sets_correct
            and point_estimates_match
        ),
    })

part4b_bootstrap_verification = pd.DataFrame(verification_rows)


# ============================================================
# 7. PUBLICATION TABLES
# ============================================================

publication_metric_order = [
    "AUC",
    "Average_Precision",
    "Sensitivity",
    "Specificity",
    "Precision",
    "Negative_Predictive_Value",
    "F1",
    "Balanced_Accuracy",
    "Matthews_Correlation_Coefficient",
    "Brier_Score",
    "Positive_Likelihood_Ratio",
    "Negative_Likelihood_Ratio",
    "Diagnostic_Odds_Ratio",
]

publication_rows = []
for model_name, algorithms in temporal_predictions.items():
    for algorithm_name, record in algorithms.items():
        row = {
            "Validation_Set": VALIDATION_SET_NAME,
            "Screening_Model": model_name,
            "Algorithm": algorithm_name,
            "N": len(record["y_true"]),
            "Events": int(np.sum(record["y_true"])),
            "Prevalence": float(np.mean(record["y_true"])),
            "Locked_Threshold": float(record["locked_threshold"]),
        }

        for metric_name in publication_metric_order:
            metric_rows = part4b_bootstrap_results.loc[
                (part4b_bootstrap_results["Screening_Model"] == model_name)
                & (part4b_bootstrap_results["Algorithm"] == algorithm_name)
                & (part4b_bootstrap_results["Metric"] == metric_name)
            ]
            if len(metric_rows) != 1:
                point = lower = upper = np.nan
            else:
                metric_row = metric_rows.iloc[0]
                point = float(metric_row["Point_Estimate"])
                lower = float(metric_row["CI_Lower"])
                upper = float(metric_row["CI_Upper"])

            digits = 2 if metric_name in [
                "Positive_Likelihood_Ratio",
                "Negative_Likelihood_Ratio",
                "Diagnostic_Odds_Ratio",
            ] else 3
            row[metric_name] = format_estimate_ci(point, lower, upper, digits)
            row[f"{metric_name}_Point"] = point
            row[f"{metric_name}_CI_Lower"] = lower
            row[f"{metric_name}_CI_Upper"] = upper

        publication_rows.append(row)

part4b_bootstrap_publication_table = pd.DataFrame(publication_rows)

compact_columns = [
    "Validation_Set",
    "Screening_Model",
    "Algorithm",
    "N",
    "Events",
    "Prevalence",
    "Locked_Threshold",
    "AUC",
    "Average_Precision",
    "Sensitivity",
    "Specificity",
    "Precision",
    "Negative_Predictive_Value",
    "F1",
    "Balanced_Accuracy",
    "Matthews_Correlation_Coefficient",
    "Brier_Score",
]
part4b_bootstrap_publication_table_compact = (
    part4b_bootstrap_publication_table[compact_columns].copy()
)


# ============================================================
# 8. FINAL SAFETY CHECKS
# ============================================================

expected_pair_count = len(all_records)
successful_pair_count = int(
    (part4b_bootstrap_status["Status"] == "Success").sum()
)
failed_pair_count = int(
    (part4b_bootstrap_status["Status"] == "Failed").sum()
)
insufficient_pair_count = int(
    (part4b_bootstrap_status["Status"] == "Insufficient valid replicates").sum()
)

if failed_pair_count > 0:
    failed_table = part4b_bootstrap_status.loc[
        part4b_bootstrap_status["Status"] == "Failed",
        ["Screening_Model", "Algorithm", "Error"],
    ]
    display(failed_table)
    raise RuntimeError(f"PART 4B failed for {failed_pair_count} model(s).")

if insufficient_pair_count > 0:
    insufficient_table = part4b_bootstrap_status.loc[
        part4b_bootstrap_status["Status"] == "Insufficient valid replicates",
        ["Screening_Model", "Algorithm", "Minimum_Required_Metric_Valid_Replicates"],
    ]
    display(insufficient_table)
    raise RuntimeError(
        f"PART 4B had insufficient valid replicates for {insufficient_pair_count} model(s)."
    )

if successful_pair_count != expected_pair_count:
    raise RuntimeError(
        f"Expected {expected_pair_count} successful analyses, observed {successful_pair_count}."
    )

if not part4b_bootstrap_verification["Temporal_Bootstrap_Ready"].all():
    raise RuntimeError("PART 4B verification failed for one or more strategies.")


# ============================================================
# 9. EXPORT
# ============================================================

part4b_bootstrap_results.to_csv(
    "part4b_bootstrap_results.csv", index=False, encoding="utf-8-sig"
)
part4b_bootstrap_distribution.to_csv(
    "part4b_bootstrap_distribution.csv", index=False, encoding="utf-8-sig"
)
part4b_bootstrap_status.to_csv(
    "part4b_bootstrap_status.csv", index=False, encoding="utf-8-sig"
)
part4b_bootstrap_verification.to_csv(
    "part4b_bootstrap_verification.csv", index=False, encoding="utf-8-sig"
)
part4b_bootstrap_publication_table.to_csv(
    "part4b_bootstrap_publication_table.csv", index=False, encoding="utf-8-sig"
)
part4b_bootstrap_publication_table_compact.to_csv(
    "part4b_bootstrap_publication_table_compact.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 10. DISPLAY AND COMPLETION REPORT
# ============================================================

print("\nTemporal bootstrap verification:")
display(part4b_bootstrap_verification)

print("\nCompact publication table:")
display(part4b_bootstrap_publication_table_compact)

print("\n" + "=" * 100)
print("PART 4B VERSION 3.1 COMPLETED")
print("=" * 100)
print(f"Expected model-algorithm pairs : {expected_pair_count}")
print(f"Successful bootstrap analyses  : {successful_pair_count}")
print(f"Failed bootstrap analyses      : {failed_pair_count}")
print(f"Insufficient analyses          : {insufficient_pair_count}")
print(f"Bootstrap replicates requested : {N_BOOTSTRAP:,} per model")
print(f"Minimum valid replicates       : {MINIMUM_VALID_BOOTSTRAPS:,}")
print("Confidence interval method     : Stratified percentile bootstrap (2.5th-97.5th percentiles)")
print("\nArtifacts created:")
print("- part4b_bootstrap_results")
print("- part4b_bootstrap_distribution")
print("- part4b_bootstrap_status")
print("- part4b_bootstrap_verification")
print("- part4b_bootstrap_publication_table")
print("- part4b_bootstrap_publication_table_compact")
print("\nNo model was refitted, no observation was resampled for model training,")
print("no prediction was regenerated, and no locked threshold was modified.")
print("Confidence intervals quantify uncertainty in the fixed temporal predictions")
print("using stratified participant-level resampling of the temporal cohort.")

### PART 4C: Temporal Calibration Analysis 2025

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 4C-1: RAW TEMPORAL CALIBRATION ANALYSIS (2025)
# ============================================================
#
# Objectives:
# 1. Assess calibration of RAW model probabilities in the
#    independent 2025 temporal cohort.
# 2. Use fixed temporal probabilities generated in PART 4A.
# 3. Estimate calibration intercept and calibration slope.
# 4. Calculate Brier score, Brier skill score, ECE, MCE,
#    LOWESS-based ICI, E50, E90, expected events, and O:E ratio.
# 5. Generate adaptive quantile-binned calibration tables and
#    publication-ready calibration plots.
# 6. Preserve all raw probabilities and locked thresholds
#    without modification.
#
# Inputs:
# - temporal_predictions
# - temporal_results
# - part4a_temporal_verification
#
# Outputs:
# - part4c1_calibration_results
# - part4c1_calibration_bins
# - part4c1_calibration_status
# - part4c1_calibration_verification
# - part4c1_calibration_publication_table
# - part4c1_calibration_publication_table_compact
# - part4c1_calibration_plot_manifest
#
# IMPORTANT:
# - No prediction model is refitted.
# - No probability is recalibrated.
# - No locked threshold is changed.
# - Calibration regression and LOWESS are diagnostic only.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm

from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.metrics import brier_score_loss

warnings.filterwarnings("ignore")


# ============================================================
# 2. SETTINGS
# ============================================================

VALIDATION_SET_NAME = "Temporal 2025"

EXPECTED_TEMPORAL_N = 815
EXPECTED_TEMPORAL_EVENTS = 16

N_CALIBRATION_BINS = 10

PROBABILITY_CLIP_EPSILON = 1e-6

LOWESS_FRACTION = 0.40
LOWESS_ITERATIONS = 0

ICI_METHOD = (
    "LOWESS absolute calibration error "
    f"(frac={LOWESS_FRACTION:.2f}, it={LOWESS_ITERATIONS})"
)

PLOT_OUTPUT_DIRECTORY = Path(
    "part4c1_temporal_calibration_plots"
)

PLOT_DPI = 300

FIGURE_WIDTH = 6.5
FIGURE_HEIGHT = 5.5

THRESHOLD_ATOL = 1e-15


# ============================================================
# 3. REQUIRED OBJECT VALIDATION
# ============================================================

required_objects = [
    "temporal_predictions",
    "temporal_results",
    "part4a_temporal_verification"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(
            missing_objects
        )
    )


if not isinstance(
    temporal_predictions,
    dict
):

    raise TypeError(
        "temporal_predictions must be a dictionary."
    )


if not isinstance(
    temporal_results,
    pd.DataFrame
):

    raise TypeError(
        "temporal_results must be a pandas DataFrame."
    )


if not isinstance(
    part4a_temporal_verification,
    pd.DataFrame
):

    raise TypeError(
        "part4a_temporal_verification must be a DataFrame."
    )


if "Temporal_Ready" not in (
    part4a_temporal_verification.columns
):

    raise KeyError(
        "part4a_temporal_verification does not contain "
        "'Temporal_Ready'."
    )


if not part4a_temporal_verification[
    "Temporal_Ready"
].all():

    failed_strategies = (
        part4a_temporal_verification.loc[
            ~part4a_temporal_verification[
                "Temporal_Ready"
            ],
            "Screening_Model"
        ]
        .astype(str)
        .tolist()
    )

    raise RuntimeError(
        "Temporal calibration cannot proceed because these "
        "strategies are not Temporal_Ready: "
        + ", ".join(
            failed_strategies
        )
    )


if len(
    temporal_predictions
) == 0:

    raise ValueError(
        "temporal_predictions is empty. Run PART 4A first."
    )


# ============================================================
# 4. CREATE OUTPUT DIRECTORY
# ============================================================

PLOT_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5. RESULT CONTAINERS
# ============================================================

calibration_result_rows = []

calibration_bin_rows = []

calibration_status_rows = []

calibration_verification_rows = []

calibration_plot_manifest_rows = []


# ============================================================
# 6. HELPER: LOGIT TRANSFORMATION
# ============================================================

def probability_to_logit(
    probabilities,
    epsilon=1e-6
):
    """
    Convert probabilities to logits after clipping only for
    numerical stability.
    """

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )


    clipped_probabilities = np.clip(
        probabilities,
        epsilon,
        1 - epsilon
    )


    logits = np.log(
        clipped_probabilities
        /
        (
            1
            -
            clipped_probabilities
        )
    )


    return (
        logits,
        clipped_probabilities
    )


# ============================================================
# 7. HELPER: CALIBRATION INTERCEPT AND SLOPE
# ============================================================

def estimate_calibration_intercept_and_slope(
    y_true,
    y_prob
):
    """
    Estimate:
    - Calibration intercept using the prediction logit as an
      offset with coefficient fixed at 1.
    - Calibration slope using a logistic calibration model with
      a freely estimated intercept and slope.

    Ideal values:
    - Calibration intercept = 0
    - Calibration slope = 1

    These regressions are diagnostic summaries only.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    y_prob = np.asarray(
        y_prob,
        dtype=float
    )


    logits, clipped_probabilities = (
        probability_to_logit(
            y_prob,
            epsilon=PROBABILITY_CLIP_EPSILON
        )
    )


    intercept_design = np.ones(
        (
            len(
                y_true
            ),
            1
        ),
        dtype=float
    )


    intercept_model = sm.GLM(
        y_true,
        intercept_design,
        family=sm.families.Binomial(),
        offset=logits
    )


    intercept_fit = intercept_model.fit(
        maxiter=500,
        disp=0
    )


    calibration_intercept = float(
        intercept_fit.params[
            0
        ]
    )


    calibration_intercept_se = float(
        intercept_fit.bse[
            0
        ]
    )


    calibration_intercept_ci_lower = float(
        calibration_intercept
        -
        1.96
        *
        calibration_intercept_se
    )


    calibration_intercept_ci_upper = float(
        calibration_intercept
        +
        1.96
        *
        calibration_intercept_se
    )


    slope_design = sm.add_constant(
        logits,
        has_constant="add"
    )


    slope_model = sm.GLM(
        y_true,
        slope_design,
        family=sm.families.Binomial()
    )


    slope_fit = slope_model.fit(
        maxiter=500,
        disp=0
    )


    calibration_model_intercept = float(
        slope_fit.params[
            0
        ]
    )


    calibration_slope = float(
        slope_fit.params[
            1
        ]
    )


    calibration_slope_se = float(
        slope_fit.bse[
            1
        ]
    )


    calibration_slope_ci_lower = float(
        calibration_slope
        -
        1.96
        *
        calibration_slope_se
    )


    calibration_slope_ci_upper = float(
        calibration_slope
        +
        1.96
        *
        calibration_slope_se
    )


    return {

        "Calibration_Intercept":
            calibration_intercept,

        "Calibration_Intercept_SE":
            calibration_intercept_se,

        "Calibration_Intercept_CI_Lower":
            calibration_intercept_ci_lower,

        "Calibration_Intercept_CI_Upper":
            calibration_intercept_ci_upper,

        "Calibration_Model_Intercept":
            calibration_model_intercept,

        "Calibration_Slope":
            calibration_slope,

        "Calibration_Slope_SE":
            calibration_slope_se,

        "Calibration_Slope_CI_Lower":
            calibration_slope_ci_lower,

        "Calibration_Slope_CI_Upper":
            calibration_slope_ci_upper,

        "Clipped_Probability_Count":
            int(
                np.sum(
                    clipped_probabilities
                    !=
                    y_prob
                )
            )
    }


# ============================================================
# 8. HELPER: ADAPTIVE QUANTILE CALIBRATION BINS
# ============================================================

def create_adaptive_calibration_bins(
    y_true,
    y_prob,
    requested_bins=10
):
    """
    Create quantile-based calibration groups.

    Duplicate probability boundaries are handled automatically.
    The actual number of bins may be smaller than requested.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    y_prob = np.asarray(
        y_prob,
        dtype=float
    )


    working_table = pd.DataFrame({

        "y_true":
            y_true,

        "y_prob":
            y_prob
    })


    unique_probability_count = int(
        working_table[
            "y_prob"
        ].nunique()
    )


    actual_requested_bins = int(
        min(
            requested_bins,
            unique_probability_count,
            len(
                working_table
            )
        )
    )


    if actual_requested_bins < 2:

        working_table[
            "Calibration_Bin"
        ] = 1


    else:

        try:

            working_table[
                "Calibration_Bin"
            ] = (
                pd.qcut(
                    working_table[
                        "y_prob"
                    ],
                    q=actual_requested_bins,
                    labels=False,
                    duplicates="drop"
                )
                +
                1
            )


        except Exception:

            working_table[
                "Probability_Rank"
            ] = (
                working_table[
                    "y_prob"
                ]
                .rank(
                    method="first"
                )
            )


            working_table[
                "Calibration_Bin"
            ] = (
                pd.qcut(
                    working_table[
                        "Probability_Rank"
                    ],
                    q=actual_requested_bins,
                    labels=False,
                    duplicates="drop"
                )
                +
                1
            )


    bin_summary = (
        working_table
        .groupby(
            "Calibration_Bin",
            observed=True
        )
        .agg(

            Bin_N=(
                "y_true",
                "size"
            ),

            Bin_Events=(
                "y_true",
                "sum"
            ),

            Mean_Predicted_Probability=(
                "y_prob",
                "mean"
            ),

            Median_Predicted_Probability=(
                "y_prob",
                "median"
            ),

            Minimum_Predicted_Probability=(
                "y_prob",
                "min"
            ),

            Maximum_Predicted_Probability=(
                "y_prob",
                "max"
            ),

            Observed_Event_Rate=(
                "y_true",
                "mean"
            )
        )
        .reset_index()
    )


    bin_summary[
        "Absolute_Calibration_Error"
    ] = np.abs(
        bin_summary[
            "Observed_Event_Rate"
        ]
        -
        bin_summary[
            "Mean_Predicted_Probability"
        ]
    )


    bin_summary[
        "Squared_Calibration_Error"
    ] = (
        bin_summary[
            "Observed_Event_Rate"
        ]
        -
        bin_summary[
            "Mean_Predicted_Probability"
        ]
    ) ** 2


    return (
        working_table,
        bin_summary
    )


# ============================================================
# 9. HELPER: ECE AND MCE
# ============================================================

def calculate_binned_calibration_errors(
    bin_summary,
    total_n
):
    """
    Calculate:
    - Expected Calibration Error (ECE)
    - Maximum Calibration Error (MCE)
    """

    if len(
        bin_summary
    ) == 0:

        return (
            np.nan,
            np.nan
        )


    weights = (
        bin_summary[
            "Bin_N"
        ]
        /
        total_n
    )


    ece = float(
        np.sum(
            weights
            *
            bin_summary[
                "Absolute_Calibration_Error"
            ]
        )
    )


    mce = float(
        np.max(
            bin_summary[
                "Absolute_Calibration_Error"
            ]
        )
    )


    return (
        ece,
        mce
    )


# ============================================================
# 10. HELPER: LOWESS CALIBRATION CURVE AND ICI
# ============================================================

def calculate_lowess_calibration(
    y_true,
    y_prob,
    fraction=0.40,
    iterations=0
):
    """
    Estimate a LOWESS calibration curve and calculate:

    - ICI: mean absolute difference between LOWESS-smoothed
      observed risk and original predicted probability
    - E50: median absolute calibration error
    - E90: 90th percentile absolute calibration error

    LOWESS is descriptive only and does not modify validation
    probabilities.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    y_prob = np.asarray(
        y_prob,
        dtype=float
    )


    unique_probability_count = int(
        len(
            np.unique(
                y_prob
            )
        )
    )


    if unique_probability_count < 3:

        return {

            "ICI":
                np.nan,

            "E50":
                np.nan,

            "E90":
                np.nan,

            "LOWESS_Probability":
                np.full(
                    len(
                        y_prob
                    ),
                    np.nan
                ),

            "LOWESS_X":
                np.array(
                    [],
                    dtype=float
                ),

            "LOWESS_Y":
                np.array(
                    [],
                    dtype=float
                ),

            "ICI_Estimable":
                False,

            "LOWESS_Fraction":
                fraction,

            "LOWESS_Iterations":
                iterations
        }


    aggregated_table = (
        pd.DataFrame({

            "y_prob":
                y_prob,

            "y_true":
                y_true
        })
        .groupby(
            "y_prob",
            as_index=False
        )
        .agg(

            Observed_Rate=(
                "y_true",
                "mean"
            ),

            Weight=(
                "y_true",
                "size"
            )
        )
        .sort_values(
            "y_prob"
        )
        .reset_index(
            drop=True
        )
    )


    lowess_result = lowess(

        endog=aggregated_table[
            "Observed_Rate"
        ].to_numpy(
            dtype=float
        ),

        exog=aggregated_table[
            "y_prob"
        ].to_numpy(
            dtype=float
        ),

        frac=fraction,

        it=iterations,

        delta=0.0,

        is_sorted=True,

        return_sorted=True
    )


    lowess_x = np.asarray(
        lowess_result[
            :,
            0
        ],
        dtype=float
    )


    lowess_y = np.asarray(
        lowess_result[
            :,
            1
        ],
        dtype=float
    )


    lowess_y = np.clip(
        lowess_y,
        0.0,
        1.0
    )


    lowess_curve_table = (
        pd.DataFrame({

            "x":
                lowess_x,

            "y":
                lowess_y
        })
        .groupby(
            "x",
            as_index=False
        )[
            "y"
        ]
        .mean()
        .sort_values(
            "x"
        )
    )


    lowess_x = lowess_curve_table[
        "x"
    ].to_numpy(
        dtype=float
    )


    lowess_y = lowess_curve_table[
        "y"
    ].to_numpy(
        dtype=float
    )


    if len(
        lowess_x
    ) < 2:

        return {

            "ICI":
                np.nan,

            "E50":
                np.nan,

            "E90":
                np.nan,

            "LOWESS_Probability":
                np.full(
                    len(
                        y_prob
                    ),
                    np.nan
                ),

            "LOWESS_X":
                lowess_x,

            "LOWESS_Y":
                lowess_y,

            "ICI_Estimable":
                False,

            "LOWESS_Fraction":
                fraction,

            "LOWESS_Iterations":
                iterations
        }


    lowess_probability = np.interp(
        y_prob,
        lowess_x,
        lowess_y,
        left=lowess_y[
            0
        ],
        right=lowess_y[
            -1
        ]
    )


    lowess_probability = np.clip(
        lowess_probability,
        0.0,
        1.0
    )


    absolute_errors = np.abs(
        lowess_probability
        -
        y_prob
    )


    return {

        "ICI":
            float(
                np.mean(
                    absolute_errors
                )
            ),

        "E50":
            float(
                np.percentile(
                    absolute_errors,
                    50
                )
            ),

        "E90":
            float(
                np.percentile(
                    absolute_errors,
                    90
                )
            ),

        "LOWESS_Probability":
            np.asarray(
                lowess_probability,
                dtype=float
            ),

        "LOWESS_X":
            lowess_x,

        "LOWESS_Y":
            lowess_y,

        "ICI_Estimable":
            True,

        "LOWESS_Fraction":
            fraction,

        "LOWESS_Iterations":
            iterations
    }


# ============================================================
# 11. HELPER: BRIER / O:E SUMMARY
# ============================================================

def calculate_probability_summary(
    y_true,
    y_prob
):
    """
    Calculate probability-scale calibration summaries.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    y_prob = np.asarray(
        y_prob,
        dtype=float
    )


    prevalence = float(
        np.mean(
            y_true
        )
    )


    brier_score = float(
        brier_score_loss(
            y_true,
            y_prob
        )
    )


    null_probabilities = np.full(
        len(
            y_true
        ),
        prevalence,
        dtype=float
    )


    null_brier_score = float(
        brier_score_loss(
            y_true,
            null_probabilities
        )
    )


    brier_skill_score = (
        float(
            1
            -
            brier_score
            /
            null_brier_score
        )
        if null_brier_score > 0
        else np.nan
    )


    mean_predicted_probability = float(
        np.mean(
            y_prob
        )
    )


    calibration_in_the_large_difference = float(
        mean_predicted_probability
        -
        prevalence
    )


    observed_events = float(
        np.sum(
            y_true
        )
    )


    expected_events = float(
        np.sum(
            y_prob
        )
    )


    observed_to_expected_ratio = (
        float(
            observed_events
            /
            expected_events
        )
        if expected_events > 0
        else np.nan
    )


    expected_to_observed_ratio = (
        float(
            expected_events
            /
            observed_events
        )
        if observed_events > 0
        else np.nan
    )


    return {

        "Brier_Score":
            brier_score,

        "Null_Brier_Score":
            null_brier_score,

        "Brier_Skill_Score":
            brier_skill_score,

        "Mean_Predicted_Probability":
            mean_predicted_probability,

        "Observed_Prevalence":
            prevalence,

        "Calibration_In_The_Large_Difference":
            calibration_in_the_large_difference,

        "Observed_Events":
            observed_events,

        "Expected_Events":
            expected_events,

        "Observed_to_Expected_Ratio":
            observed_to_expected_ratio,

        "Expected_to_Observed_Ratio":
            expected_to_observed_ratio
    }


# ============================================================
# 12. HELPER: CREATE CALIBRATION PLOT
# ============================================================

def create_calibration_plot(
    model_name,
    algorithm_name,
    y_true,
    y_prob,
    bin_summary,
    lowess_x,
    lowess_y,
    calibration_intercept,
    calibration_slope
):
    """
    Create one publication-ready temporal calibration plot per
    model-algorithm pair.
    """

    figure, axis = plt.subplots(
        figsize=(
            FIGURE_WIDTH,
            FIGURE_HEIGHT
        )
    )


    axis.plot(
        [
            0,
            1
        ],
        [
            0,
            1
        ],
        linestyle="--",
        linewidth=1.5,
        label="Perfect calibration"
    )


    if len(
        bin_summary
    ) > 0:

        axis.plot(
            bin_summary[
                "Mean_Predicted_Probability"
            ],
            bin_summary[
                "Observed_Event_Rate"
            ],
            marker="o",
            linewidth=1.5,
            label="Quantile-binned observed"
        )


    if (
        len(
            lowess_x
        )
        >=
        2
        and
        np.all(
            np.isfinite(
                lowess_x
            )
        )
        and
        np.all(
            np.isfinite(
                lowess_y
            )
        )
    ):

        axis.plot(
            lowess_x,
            lowess_y,
            linewidth=1.75,
            label=(
                f"LOWESS smoother "
                f"(frac={LOWESS_FRACTION:.2f})"
            )
        )


    rug_y = np.full(
        len(
            y_prob
        ),
        -0.0125
    )


    axis.plot(
        y_prob,
        rug_y,
        "|",
        markersize=5,
        alpha=0.20,
        clip_on=False,
        label="Predicted-risk distribution"
    )


    max_display_probability = float(
        max(
            0.05,
            np.percentile(
                y_prob,
                99
            ),
            np.max(
                bin_summary[
                    "Mean_Predicted_Probability"
                ]
            )
            if len(
                bin_summary
            ) > 0
            else 0.05,
            np.max(
                bin_summary[
                    "Observed_Event_Rate"
                ]
            )
            if len(
                bin_summary
            ) > 0
            else 0.05
        )
    )


    max_display_probability = min(
        1.0,
        max_display_probability
        *
        1.10
    )


    axis.set_xlim(
        0,
        max_display_probability
    )


    axis.set_ylim(
        0,
        max_display_probability
    )


    axis.set_xlabel(
        "Predicted probability"
    )


    axis.set_ylabel(
        "Observed event rate"
    )


    axis.set_title(
        f"{model_name} — {algorithm_name}\n"
        f"Temporal 2025 calibration"
    )


    annotation_text = (
        f"Intercept = "
        f"{calibration_intercept:.3f}\n"
        f"Slope = "
        f"{calibration_slope:.3f}\n"
        f"Ideal: intercept 0; slope 1\n"
        f"N = {len(y_true):,}; "
        f"events = {int(np.sum(y_true))}"
    )


    axis.text(
        0.98,
        0.02,
        annotation_text,
        transform=axis.transAxes,
        horizontalalignment="right",
        verticalalignment="bottom",
        fontsize=9,
        bbox={
            "boxstyle":
                "round",

            "alpha":
                0.15
        }
    )


    axis.legend(
        loc="upper left",
        fontsize=8
    )


    axis.grid(
        alpha=0.25
    )


    figure.tight_layout()


    safe_model_name = (
        str(
            model_name
        )
        .replace(
            " ",
            "_"
        )
        .replace(
            "/",
            "_"
        )
    )


    safe_algorithm_name = (
        str(
            algorithm_name
        )
        .replace(
            " ",
            "_"
        )
        .replace(
            "/",
            "_"
        )
    )


    png_path = (
        PLOT_OUTPUT_DIRECTORY
        /
        (
            f"part4c1_calibration_"
            f"{safe_model_name}_"
            f"{safe_algorithm_name}.png"
        )
    )


    pdf_path = (
        PLOT_OUTPUT_DIRECTORY
        /
        (
            f"part4c1_calibration_"
            f"{safe_model_name}_"
            f"{safe_algorithm_name}.pdf"
        )
    )


    figure.savefig(
        png_path,
        dpi=PLOT_DPI,
        bbox_inches="tight"
    )


    figure.savefig(
        pdf_path,
        bbox_inches="tight"
    )


    plt.close(
        figure
    )


    return (
        str(
            png_path
        ),
        str(
            pdf_path
        )
    )


# ============================================================
# 13. CALIBRATION ANALYSIS LOOP
# ============================================================

print("=" * 100)
print(
    "PART 4C-1 VERSION 3.0: RAW TEMPORAL CALIBRATION "
    "ANALYSIS (2025)"
)
print("=" * 100)


print(
    f"Requested calibration bins: "
    f"{N_CALIBRATION_BINS}"
)


print(
    f"ICI method: "
    f"{ICI_METHOD}"
)


for model_name, algorithms in (
    temporal_predictions.items()
):

    print("\n" + "=" * 100)
    print(
        f"{model_name}: RAW TEMPORAL CALIBRATION"
    )
    print("=" * 100)


    if len(
        algorithms
    ) == 0:

        raise ValueError(
            f"{model_name} contains no temporal predictions."
        )


    for algorithm_index, (
        algorithm_name,
        record
    ) in enumerate(
        algorithms.items(),
        start=1
    ):

        print(
            f"\n[{algorithm_index}/"
            f"{len(algorithms)}] "
            f"{algorithm_name}"
        )


        try:

            required_record_keys = [
                "y_true",
                "y_prob",
                "y_pred",
                "locked_threshold",
                "original_index"
            ]


            missing_record_keys = [
                key
                for key in required_record_keys
                if key not in record
            ]


            if missing_record_keys:

                raise KeyError(
                    "Missing temporal prediction fields: "
                    + ", ".join(
                        missing_record_keys
                    )
                )


            y_true = np.asarray(
                record[
                    "y_true"
                ],
                dtype=int
            )


            y_prob = np.asarray(
                record[
                    "y_prob"
                ],
                dtype=float
            )


            stored_y_pred = np.asarray(
                record[
                    "y_pred"
                ],
                dtype=int
            )


            original_index = np.asarray(
                record[
                    "original_index"
                ]
            )


            locked_threshold = float(
                record[
                    "locked_threshold"
                ]
            )


            if not (
                len(
                    y_true
                )
                ==
                len(
                    y_prob
                )
                ==
                len(
                    stored_y_pred
                )
                ==
                len(
                    original_index
                )
            ):

                raise ValueError(
                    "Temporal arrays have different lengths."
                )


            if set(
                np.unique(
                    y_true
                )
            ) != {
                0,
                1
            }:

                raise ValueError(
                    "Both temporal outcome classes are required."
                )


            if not np.all(
                np.isfinite(
                    y_prob
                )
            ):

                raise ValueError(
                    "Temporal probabilities contain "
                    "nonfinite values."
                )


            if (
                np.any(
                    y_prob < 0
                )
                or
                np.any(
                    y_prob > 1
                )
            ):

                raise ValueError(
                    "Temporal probabilities are outside [0, 1]."
                )


            recomputed_y_pred = (
                y_prob
                >=
                locked_threshold
            ).astype(int)


            if not np.array_equal(
                stored_y_pred,
                recomputed_y_pred
            ):

                raise RuntimeError(
                    "Stored temporal y_pred does not match "
                    "the locked threshold."
                )


            temporal_n = int(
                len(
                    y_true
                )
            )


            temporal_events = int(
                np.sum(
                    y_true
                    ==
                    1
                )
            )


            temporal_nonevents = int(
                np.sum(
                    y_true
                    ==
                    0
                )
            )


            temporal_prevalence = float(
                np.mean(
                    y_true
                )
            )


            # ------------------------------------------------
            # Calibration intercept and slope
            # ------------------------------------------------

            regression_calibration = (
                estimate_calibration_intercept_and_slope(
                    y_true=y_true,
                    y_prob=y_prob
                )
            )


            # ------------------------------------------------
            # Adaptive calibration bins
            # ------------------------------------------------

            (
                participant_bin_table,
                bin_summary
            ) = create_adaptive_calibration_bins(
                y_true=y_true,
                y_prob=y_prob,
                requested_bins=N_CALIBRATION_BINS
            )


            actual_bin_count = int(
                bin_summary[
                    "Calibration_Bin"
                ].nunique()
            )


            ece, mce = (
                calculate_binned_calibration_errors(
                    bin_summary=bin_summary,
                    total_n=temporal_n
                )
            )


            # ------------------------------------------------
            # LOWESS ICI
            # ------------------------------------------------

            lowess_result = (
                calculate_lowess_calibration(
                    y_true=y_true,
                    y_prob=y_prob,
                    fraction=LOWESS_FRACTION,
                    iterations=LOWESS_ITERATIONS
                )
            )


            # ------------------------------------------------
            # Probability summary
            # ------------------------------------------------

            probability_summary = (
                calculate_probability_summary(
                    y_true=y_true,
                    y_prob=y_prob
                )
            )


            # ------------------------------------------------
            # Store bin-level outputs
            # ------------------------------------------------

            model_bin_summary = (
                bin_summary.copy()
            )


            model_bin_summary[
                "Validation_Set"
            ] = VALIDATION_SET_NAME


            model_bin_summary[
                "Screening_Model"
            ] = model_name


            model_bin_summary[
                "Algorithm"
            ] = algorithm_name


            model_bin_summary[
                "Requested_Bins"
            ] = N_CALIBRATION_BINS


            model_bin_summary[
                "Actual_Bins"
            ] = actual_bin_count


            model_bin_summary[
                "Locked_Threshold"
            ] = locked_threshold


            calibration_bin_rows.append(
                model_bin_summary
            )


            # ------------------------------------------------
            # Create calibration figure
            # ------------------------------------------------

            png_path, pdf_path = (
                create_calibration_plot(
                    model_name=model_name,
                    algorithm_name=algorithm_name,
                    y_true=y_true,
                    y_prob=y_prob,
                    bin_summary=bin_summary,
                    lowess_x=lowess_result[
                        "LOWESS_X"
                    ],
                    lowess_y=lowess_result[
                        "LOWESS_Y"
                    ],
                    calibration_intercept=(
                        regression_calibration[
                            "Calibration_Intercept"
                        ]
                    ),
                    calibration_slope=(
                        regression_calibration[
                            "Calibration_Slope"
                        ]
                    )
                )
            )


            calibration_plot_manifest_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "PNG_File":
                    png_path,

                "PDF_File":
                    pdf_path,

                "Plot_Created":
                    True
            })


            # ------------------------------------------------
            # Main summary row
            # ------------------------------------------------

            calibration_result_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "N":
                    temporal_n,

                "Events":
                    temporal_events,

                "Nonevents":
                    temporal_nonevents,

                "Prevalence":
                    temporal_prevalence,

                "Locked_Threshold":
                    locked_threshold,

                "Mean_Predicted_Probability":
                    probability_summary[
                        "Mean_Predicted_Probability"
                    ],

                "Calibration_In_The_Large_Difference":
                    probability_summary[
                        "Calibration_In_The_Large_Difference"
                    ],

                "Observed_Events":
                    probability_summary[
                        "Observed_Events"
                    ],

                "Expected_Events":
                    probability_summary[
                        "Expected_Events"
                    ],

                "Observed_to_Expected_Ratio":
                    probability_summary[
                        "Observed_to_Expected_Ratio"
                    ],

                "Expected_to_Observed_Ratio":
                    probability_summary[
                        "Expected_to_Observed_Ratio"
                    ],

                "Calibration_Intercept":
                    regression_calibration[
                        "Calibration_Intercept"
                    ],

                "Calibration_Intercept_SE":
                    regression_calibration[
                        "Calibration_Intercept_SE"
                    ],

                "Calibration_Intercept_CI_Lower":
                    regression_calibration[
                        "Calibration_Intercept_CI_Lower"
                    ],

                "Calibration_Intercept_CI_Upper":
                    regression_calibration[
                        "Calibration_Intercept_CI_Upper"
                    ],

                "Calibration_Model_Intercept":
                    regression_calibration[
                        "Calibration_Model_Intercept"
                    ],

                "Calibration_Slope":
                    regression_calibration[
                        "Calibration_Slope"
                    ],

                "Calibration_Slope_SE":
                    regression_calibration[
                        "Calibration_Slope_SE"
                    ],

                "Calibration_Slope_CI_Lower":
                    regression_calibration[
                        "Calibration_Slope_CI_Lower"
                    ],

                "Calibration_Slope_CI_Upper":
                    regression_calibration[
                        "Calibration_Slope_CI_Upper"
                    ],

                "Brier_Score":
                    probability_summary[
                        "Brier_Score"
                    ],

                "Null_Brier_Score":
                    probability_summary[
                        "Null_Brier_Score"
                    ],

                "Brier_Skill_Score":
                    probability_summary[
                        "Brier_Skill_Score"
                    ],

                "ECE":
                    ece,

                "MCE":
                    mce,

                "ICI":
                    lowess_result[
                        "ICI"
                    ],

                "E50":
                    lowess_result[
                        "E50"
                    ],

                "E90":
                    lowess_result[
                        "E90"
                    ],

                "ICI_Estimable":
                    lowess_result[
                        "ICI_Estimable"
                    ],

                "LOWESS_Fraction":
                    lowess_result[
                        "LOWESS_Fraction"
                    ],

                "LOWESS_Iterations":
                    lowess_result[
                        "LOWESS_Iterations"
                    ],

                "ICI_Method":
                    ICI_METHOD,

                "Requested_Calibration_Bins":
                    N_CALIBRATION_BINS,

                "Actual_Calibration_Bins":
                    actual_bin_count,

                "Unique_Probability_Count":
                    int(
                        len(
                            np.unique(
                                y_prob
                            )
                        )
                    ),

                "Probability_Min":
                    float(
                        np.min(
                            y_prob
                        )
                    ),

                "Probability_Median":
                    float(
                        np.median(
                            y_prob
                        )
                    ),

                "Probability_Max":
                    float(
                        np.max(
                            y_prob
                        )
                    ),

                "Clipped_Probability_Count_For_Logit":
                    regression_calibration[
                        "Clipped_Probability_Count"
                    ],

                "Probability_Scale":
                    "Raw model probability",

                "Probability_Recalibrated":
                    False,

                "Calibration_Assessment_Note":
                    (
                        "Diagnostic assessment only; "
                        "temporal probabilities were not "
                        "recalibrated."
                    )
            })


            calibration_status_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Success",

                "N":
                    temporal_n,

                "Events":
                    temporal_events,

                "Actual_Calibration_Bins":
                    actual_bin_count,

                "Calibration_Regression_Estimable":
                    True,

                "ICI_Estimable":
                    lowess_result[
                        "ICI_Estimable"
                    ],

                "Plot_Created":
                    True,

                "Probability_Recalibrated":
                    False,

                "Threshold_Modified":
                    False,

                "Error":
                    ""
            })


            print(
                f"Intercept="
                f"{regression_calibration['Calibration_Intercept']:.3f} | "
                f"Slope="
                f"{regression_calibration['Calibration_Slope']:.3f} | "
                f"Brier="
                f"{probability_summary['Brier_Score']:.4f} | "
                f"ECE={ece:.4f} | "
                f"ICI={lowess_result['ICI']:.4f} | "
                f"O:E="
                f"{probability_summary['Observed_to_Expected_Ratio']:.3f} | "
                f"Bins={actual_bin_count}"
            )


        except Exception as error:

            calibration_status_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Failed",

                "N":
                    (
                        len(
                            record.get(
                                "y_true",
                                []
                            )
                        )
                        if isinstance(
                            record,
                            dict
                        )
                        else np.nan
                    ),

                "Events":
                    (
                        int(
                            np.sum(
                                np.asarray(
                                    record.get(
                                        "y_true",
                                        []
                                    )
                                )
                                ==
                                1
                            )
                        )
                        if isinstance(
                            record,
                            dict
                        )
                        else np.nan
                    ),

                "Actual_Calibration_Bins":
                    np.nan,

                "Calibration_Regression_Estimable":
                    False,

                "ICI_Estimable":
                    False,

                "Plot_Created":
                    False,

                "Probability_Recalibrated":
                    False,

                "Threshold_Modified":
                    False,

                "Error":
                    str(
                        error
                    )
            })


            calibration_plot_manifest_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "PNG_File":
                    "",

                "PDF_File":
                    "",

                "Plot_Created":
                    False
            })


            print(
                "Failed"
            )


            print(
                f"Reason: {error}"
            )


# ============================================================
# 14. CREATE RESULT TABLES
# ============================================================

part4c1_calibration_results = pd.DataFrame(
    calibration_result_rows
)


if calibration_bin_rows:

    part4c1_calibration_bins = pd.concat(
        calibration_bin_rows,
        ignore_index=True
    )


else:

    part4c1_calibration_bins = pd.DataFrame()


part4c1_calibration_status = pd.DataFrame(
    calibration_status_rows
)


part4c1_calibration_plot_manifest = pd.DataFrame(
    calibration_plot_manifest_rows
)


# ============================================================
# 15. VERIFICATION
# ============================================================

expected_algorithm_count = sum(
    len(
        algorithms
    )
    for algorithms in temporal_predictions.values()
)


successful_algorithm_count = int(
    (
        part4c1_calibration_status[
            "Status"
        ]
        ==
        "Success"
    ).sum()
)


failed_algorithm_count = int(
    (
        part4c1_calibration_status[
            "Status"
        ]
        ==
        "Failed"
    ).sum()
)


for model_name, algorithms in (
    temporal_predictions.items()
):

    expected_algorithms = set(
        algorithms.keys()
    )


    successful_algorithms = set(
        part4c1_calibration_status.loc[
            (
                part4c1_calibration_status[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                part4c1_calibration_status[
                    "Status"
                ]
                ==
                "Success"
            ),
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )


    missing_algorithms = sorted(
        expected_algorithms
        -
        successful_algorithms
    )


    model_results = (
        part4c1_calibration_results[
            part4c1_calibration_results[
                "Screening_Model"
            ]
            ==
            model_name
        ]
    )


    expected_result_rows = int(
        len(
            expected_algorithms
        )
    )


    observed_result_rows = int(
        len(
            model_results
        )
    )


    core_metric_columns = [
        "Calibration_Intercept",
        "Calibration_Slope",
        "Brier_Score",
        "Null_Brier_Score",
        "ECE",
        "MCE",
        "ICI",
        "Observed_to_Expected_Ratio"
    ]


    all_core_metrics_finite = bool(
        np.isfinite(
            model_results[
                core_metric_columns
            ]
            .to_numpy(
                dtype=float
            )
        )
        .all()
    ) if not model_results.empty else False


    all_bin_counts_valid = bool(
        (
            model_results[
                "Actual_Calibration_Bins"
            ]
            >=
            1
        )
        .all()
    ) if not model_results.empty else False


    all_plots_created = bool(
        part4c1_calibration_plot_manifest.loc[
            part4c1_calibration_plot_manifest[
                "Screening_Model"
            ]
            ==
            model_name,
            "Plot_Created"
        ]
        .all()
    )


    brier_point_estimates_match_part4a = True

    all_raw_probabilities_unchanged = True

    all_thresholds_unchanged = True

    all_outcome_alignment_correct = True

    all_index_alignment_correct = True


    for algorithm_name in (
        expected_algorithms
    ):

        calibration_row = (
            model_results[
                model_results[
                    "Algorithm"
                ]
                ==
                algorithm_name
            ]
        )


        temporal_result_row = (
            temporal_results[
                (
                    temporal_results[
                        "Screening_Model"
                    ]
                    ==
                    model_name
                )
                &
                (
                    temporal_results[
                        "Algorithm"
                    ]
                    ==
                    algorithm_name
                )
            ]
        )


        if (
            len(
                calibration_row
            )
            !=
            1
            or
            len(
                temporal_result_row
            )
            !=
            1
        ):

            brier_point_estimates_match_part4a = False

            continue


        calibration_brier = float(
            calibration_row.iloc[
                0
            ][
                "Brier_Score"
            ]
        )


        temporal_brier = float(
            temporal_result_row.iloc[
                0
            ][
                "Brier_Score"
            ]
        )


        if not np.isclose(
            calibration_brier,
            temporal_brier,
            rtol=0,
            atol=1e-12
        ):

            brier_point_estimates_match_part4a = False


        record = (
            temporal_predictions[
                model_name
            ][
                algorithm_name
            ]
        )


        if record[
            "probability_recalibrated"
        ]:

            all_raw_probabilities_unchanged = False


        if not np.isclose(
            calibration_row.iloc[
                0
            ][
                "Locked_Threshold"
            ],
            record[
                "locked_threshold"
            ],
            rtol=0,
            atol=THRESHOLD_ATOL
        ):

            all_thresholds_unchanged = False


        reference_record = next(
            iter(
                temporal_predictions[
                    model_name
                ].values()
            )
        )


        if not np.array_equal(
            record[
                "y_true"
            ],
            reference_record[
                "y_true"
            ]
        ):

            all_outcome_alignment_correct = False


        if not np.array_equal(
            record[
                "original_index"
            ],
            reference_record[
                "original_index"
            ]
        ):

            all_index_alignment_correct = False


    calibration_verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            expected_result_rows,

        "Successful_Algorithms":
            int(
                len(
                    successful_algorithms
                )
            ),

        "Missing_Algorithms":
            ", ".join(
                missing_algorithms
            ),

        "Expected_Result_Rows":
            expected_result_rows,

        "Observed_Result_Rows":
            observed_result_rows,

        "All_Core_Metrics_Finite":
            all_core_metrics_finite,

        "All_Bin_Counts_Valid":
            all_bin_counts_valid,

        "All_Plots_Created":
            all_plots_created,

        "Brier_Point_Estimates_Match_PART4A":
            brier_point_estimates_match_part4a,

        "All_Raw_Probabilities_Unchanged":
            all_raw_probabilities_unchanged,

        "All_Locked_Thresholds_Unchanged":
            all_thresholds_unchanged,

        "All_Outcome_Alignment_Correct":
            all_outcome_alignment_correct,

        "All_Index_Alignment_Correct":
            all_index_alignment_correct,

        "Temporal_Raw_Calibration_Ready":
            (
                len(
                    missing_algorithms
                )
                ==
                0
                and
                observed_result_rows
                ==
                expected_result_rows
                and
                all_core_metrics_finite
                and
                all_bin_counts_valid
                and
                all_plots_created
                and
                brier_point_estimates_match_part4a
                and
                all_raw_probabilities_unchanged
                and
                all_thresholds_unchanged
                and
                all_outcome_alignment_correct
                and
                all_index_alignment_correct
            )
    })


part4c1_calibration_verification = pd.DataFrame(
    calibration_verification_rows
)


# ============================================================
# 16. PUBLICATION TABLE
# ============================================================

part4c1_calibration_publication_table = (
    part4c1_calibration_results[
        [
            "Validation_Set",
            "Screening_Model",
            "Algorithm",
            "N",
            "Events",
            "Prevalence",
            "Mean_Predicted_Probability",
            "Calibration_In_The_Large_Difference",
            "Observed_Events",
            "Expected_Events",
            "Observed_to_Expected_Ratio",
            "Expected_to_Observed_Ratio",
            "Calibration_Intercept",
            "Calibration_Intercept_CI_Lower",
            "Calibration_Intercept_CI_Upper",
            "Calibration_Slope",
            "Calibration_Slope_CI_Lower",
            "Calibration_Slope_CI_Upper",
            "Brier_Score",
            "Null_Brier_Score",
            "Brier_Skill_Score",
            "ECE",
            "MCE",
            "ICI",
            "E50",
            "E90",
            "Actual_Calibration_Bins"
        ]
    ]
    .copy()
)


# ============================================================
# 17. COMPACT PUBLICATION TABLE
# ============================================================

def format_estimate_with_ci(
    estimate,
    lower,
    upper,
    digits=3
):
    """
    Format estimate and 95% CI for manuscript tables.
    """

    if (
        not np.isfinite(
            estimate
        )
        or
        not np.isfinite(
            lower
        )
        or
        not np.isfinite(
            upper
        )
    ):

        return "Not estimable"


    return (
        f"{estimate:.{digits}f} "
        f"({lower:.{digits}f}-{upper:.{digits}f})"
    )


part4c1_calibration_publication_table_compact = (
    part4c1_calibration_results[
        [
            "Validation_Set",
            "Screening_Model",
            "Algorithm",
            "N",
            "Events",
            "Prevalence",
            "Mean_Predicted_Probability",
            "Observed_to_Expected_Ratio",
            "Calibration_Intercept",
            "Calibration_Intercept_CI_Lower",
            "Calibration_Intercept_CI_Upper",
            "Calibration_Slope",
            "Calibration_Slope_CI_Lower",
            "Calibration_Slope_CI_Upper",
            "Brier_Score",
            "Brier_Skill_Score",
            "ECE",
            "ICI",
            "E50",
            "E90"
        ]
    ]
    .copy()
)


part4c1_calibration_publication_table_compact[
    "Calibration_Intercept_95CI"
] = part4c1_calibration_publication_table_compact.apply(

    lambda row:
        format_estimate_with_ci(

            estimate=row[
                "Calibration_Intercept"
            ],

            lower=row[
                "Calibration_Intercept_CI_Lower"
            ],

            upper=row[
                "Calibration_Intercept_CI_Upper"
            ],

            digits=3
        ),

    axis=1
)


part4c1_calibration_publication_table_compact[
    "Calibration_Slope_95CI"
] = part4c1_calibration_publication_table_compact.apply(

    lambda row:
        format_estimate_with_ci(

            estimate=row[
                "Calibration_Slope"
            ],

            lower=row[
                "Calibration_Slope_CI_Lower"
            ],

            upper=row[
                "Calibration_Slope_CI_Upper"
            ],

            digits=3
        ),

    axis=1
)


part4c1_calibration_publication_table_compact = (
    part4c1_calibration_publication_table_compact[
        [
            "Validation_Set",
            "Screening_Model",
            "Algorithm",
            "N",
            "Events",
            "Prevalence",
            "Mean_Predicted_Probability",
            "Observed_to_Expected_Ratio",
            "Calibration_Intercept_95CI",
            "Calibration_Slope_95CI",
            "Brier_Score",
            "Brier_Skill_Score",
            "ECE",
            "ICI",
            "E50",
            "E90"
        ]
    ]
    .copy()
)


# ============================================================
# 18. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 100)
print("PART 4C-1 CALIBRATION STATUS")
print("=" * 100)


print(
    part4c1_calibration_status[
        "Status"
    ]
    .value_counts(
        dropna=False
    )
)


print("\n" + "=" * 100)
print("RAW TEMPORAL CALIBRATION SUMMARY")
print("=" * 100)


display(
    part4c1_calibration_publication_table
    .sort_values(
        by=[
            "Screening_Model",
            "Algorithm"
        ]
    )
    .round(
        4
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 100)
print("PART 4C-1 CALIBRATION VERIFICATION")
print("=" * 100)


display(
    part4c1_calibration_verification
)


# ============================================================
# 19. HARD SAFETY CHECKS
# ============================================================

if failed_algorithm_count > 0:

    failed_rows = (
        part4c1_calibration_status[
            part4c1_calibration_status[
                "Status"
            ]
            ==
            "Failed"
        ]
    )


    display(
        failed_rows
    )


    raise RuntimeError(
        f"{failed_algorithm_count} raw temporal calibration "
        "analyses failed."
    )


if successful_algorithm_count != (
    expected_algorithm_count
):

    raise RuntimeError(
        "The number of successful raw temporal calibration "
        "analyses does not match the number of temporal "
        "model-algorithm pairs."
    )


if not part4c1_calibration_verification[
    "Temporal_Raw_Calibration_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies failed raw temporal "
        "calibration verification."
    )


# ============================================================
# 20. COHORT CHECK
# ============================================================

reference_model_name = next(
    iter(
        temporal_predictions
    )
)


reference_algorithm_name = next(
    iter(
        temporal_predictions[
            reference_model_name
        ]
    )
)


reference_y_true = np.asarray(
    temporal_predictions[
        reference_model_name
    ][
        reference_algorithm_name
    ][
        "y_true"
    ],
    dtype=int
)


if len(
    reference_y_true
) != EXPECTED_TEMPORAL_N:

    print(
        f"\nWarning: expected temporal N="
        f"{EXPECTED_TEMPORAL_N:,}, observed "
        f"N={len(reference_y_true):,}."
    )


if int(
    np.sum(
        reference_y_true
        ==
        1
    )
) != EXPECTED_TEMPORAL_EVENTS:

    print(
        f"\nWarning: expected temporal events="
        f"{EXPECTED_TEMPORAL_EVENTS}, observed "
        f"events="
        f"{int(np.sum(reference_y_true == 1))}."
    )


# ============================================================
# 21. EXPORT RESULTS
# ============================================================

part4c1_calibration_results.to_csv(
    "part4c1_raw_temporal_calibration_results.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c1_calibration_bins.to_csv(
    "part4c1_raw_temporal_calibration_bins.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c1_calibration_status.to_csv(
    "part4c1_raw_temporal_calibration_status.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c1_calibration_verification.to_csv(
    "part4c1_raw_temporal_calibration_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c1_calibration_publication_table.to_csv(
    "part4c1_raw_temporal_calibration_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c1_calibration_publication_table_compact.to_csv(
    "part4c1_raw_temporal_calibration_publication_table_compact.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c1_calibration_plot_manifest.to_csv(
    "part4c1_raw_temporal_calibration_plot_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 22. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 4C-1 VERSION 3.0 COMPLETED")
print("=" * 100)


print(
    f"Expected model-algorithm pairs : "
    f"{expected_algorithm_count}"
)


print(
    f"Successful calibration analyses: "
    f"{successful_algorithm_count}"
)


print(
    f"Failed calibration analyses    : "
    f"{failed_algorithm_count}"
)


print(
    f"Requested calibration bins     : "
    f"{N_CALIBRATION_BINS}"
)


print(
    f"ICI method                     : "
    f"{ICI_METHOD}"
)


print(
    f"Calibration plot directory     : "
    f"{PLOT_OUTPUT_DIRECTORY}"
)


print("\nArtifacts created:")

print("- part4c1_calibration_results")
print("- part4c1_calibration_bins")
print("- part4c1_calibration_status")
print("- part4c1_calibration_verification")
print("- part4c1_calibration_publication_table")
print("- part4c1_calibration_publication_table_compact")
print("- part4c1_calibration_plot_manifest")


print("\nExported files:")

print("- part4c1_raw_temporal_calibration_results.csv")
print("- part4c1_raw_temporal_calibration_bins.csv")
print("- part4c1_raw_temporal_calibration_status.csv")
print("- part4c1_raw_temporal_calibration_verification.csv")
print("- part4c1_raw_temporal_calibration_publication_table.csv")
print("- part4c1_raw_temporal_calibration_publication_table_compact.csv")
print("- part4c1_raw_temporal_calibration_plot_manifest.csv")
print(
    f"- PNG/PDF calibration plots in "
    f"{PLOT_OUTPUT_DIRECTORY}/"
)


print(
    "\nNo model was refitted, no temporal probability was "
    "recalibrated, and no locked threshold was modified."
)


print(
    "Calibration regression and LOWESS smoothing were used "
    "only as diagnostic summaries of the fixed raw temporal "
    "2025 probabilities."
)


print(
    "\nProceed to PART 4C-2 only after "
    "Temporal_Raw_Calibration_Ready=True for all strategies."
)

### PART 4C-2: Temporal Evaluation Using Development-Derived Recalibration Mappings

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 4C-2: OOF-RECALIBRATED TEMPORAL CALIBRATION (2025)
# ============================================================
#
# Objectives:
# 1. Apply probability-recalibration mappings learned ONLY from
#    genuine 2024 OOF predictions to the independent 2025
#    temporal probabilities.
# 2. Compare RAW versus OOF-recalibrated temporal probability
#    performance without fitting any recalibration model on 2025.
# 3. Evaluate calibration intercept/slope, Brier score,
#    Brier skill score, ECE, MCE, LOWESS-based ICI, E50/E90,
#    expected events, and O:E ratio after recalibration.
# 4. Preserve the original raw temporal probabilities and the
#    original locked classification thresholds unchanged.
# 5. Explicitly retain mappings with negative slopes as
#    diagnostically valid transported mappings, while flagging
#    their reversal of risk ranking.
# 6. Explicitly exclude OOF recalibration mappings previously
#    classified as "Not estimable".
#
# Inputs:
# - temporal_predictions
# - temporal_results
# - part4a_temporal_verification
# - oof_recalibration_models
# - part3d2_recalibration_status
#
# Recommended:
# - part4c1_calibration_results
# - part4c1_calibration_verification
#
# Outputs:
# - temporal_recalibrated_predictions
# - part4c2_recalibration_results
# - part4c2_recalibration_bins
# - part4c2_recalibration_status
# - part4c2_recalibration_verification
# - part4c2_recalibration_publication_table
# - part4c2_recalibration_publication_table_compact
# - part4c2_recalibration_plot_manifest
#
# IMPORTANT:
# - NO recalibration model is fit using 2025 outcomes.
# - NO prediction model is refitted.
# - NO locked threshold is modified.
# - Recalibrated probabilities are evaluated only as probability
#   estimates; PART 4A raw-threshold classification results
#   remain the primary temporal classification analysis.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm

from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score
)

warnings.filterwarnings("ignore")


# ============================================================
# 2. SETTINGS
# ============================================================

VALIDATION_SET_NAME = "Temporal 2025"

EXPECTED_TEMPORAL_N = 815
EXPECTED_TEMPORAL_EVENTS = 16

PROBABILITY_CLIP_EPSILON = 1e-6

N_CALIBRATION_BINS = 10

LOWESS_FRACTION = 0.40
LOWESS_ITERATIONS = 0

ICI_METHOD = (
    "LOWESS absolute calibration error "
    f"(frac={LOWESS_FRACTION:.2f}, it={LOWESS_ITERATIONS})"
)

PLOT_OUTPUT_DIRECTORY = Path(
    "part4c2_recalibrated_temporal_calibration_plots"
)

PLOT_DPI = 300

FIGURE_WIDTH = 6.5
FIGURE_HEIGHT = 5.5

NUMERICAL_ATOL = 1e-12
THRESHOLD_ATOL = 1e-15


# ============================================================
# 3. REQUIRED OBJECT VALIDATION
# ============================================================

required_objects = [
    "temporal_predictions",
    "temporal_results",
    "part4a_temporal_verification",
    "oof_recalibration_models",
    "part3d2_recalibration_status"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(
            missing_objects
        )
    )


if not isinstance(
    temporal_predictions,
    dict
):

    raise TypeError(
        "temporal_predictions must be a dictionary."
    )


if not isinstance(
    temporal_results,
    pd.DataFrame
):

    raise TypeError(
        "temporal_results must be a pandas DataFrame."
    )


if not isinstance(
    oof_recalibration_models,
    dict
):

    raise TypeError(
        "oof_recalibration_models must be a dictionary."
    )


if not isinstance(
    part3d2_recalibration_status,
    pd.DataFrame
):

    raise TypeError(
        "part3d2_recalibration_status must be a DataFrame."
    )


if not part4a_temporal_verification[
    "Temporal_Ready"
].all():

    raise RuntimeError(
        "PART 4A temporal validation is not fully ready."
    )


if (
    "part4c1_calibration_verification"
    in globals()
):

    if not part4c1_calibration_verification[
        "Temporal_Raw_Calibration_Ready"
    ].all():

        raise RuntimeError(
            "PART 4C-1 raw temporal calibration is not fully ready."
        )


# ============================================================
# 4. OUTPUT DIRECTORY
# ============================================================

PLOT_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5. RESULT CONTAINERS
# ============================================================

temporal_recalibrated_predictions = {}

recalibration_result_rows = []

recalibration_bin_rows = []

recalibration_status_rows = []

recalibration_verification_rows = []

recalibration_plot_manifest_rows = []


# ============================================================
# 6. HELPERS: LOGIT / INVERSE LOGIT
# ============================================================

def probability_to_logit(
    probabilities,
    epsilon=1e-6
):
    """
    Convert probabilities to logits after clipping only for
    numerical stability.
    """

    probabilities = np.asarray(
        probabilities,
        dtype=float
    )


    if not np.all(
        np.isfinite(
            probabilities
        )
    ):

        raise ValueError(
            "Probabilities contain nonfinite values."
        )


    if (
        np.any(
            probabilities < 0
        )
        or
        np.any(
            probabilities > 1
        )
    ):

        raise ValueError(
            "Probabilities are outside [0, 1]."
        )


    clipped = np.clip(
        probabilities,
        epsilon,
        1 - epsilon
    )


    logits = np.log(
        clipped
        /
        (
            1
            -
            clipped
        )
    )


    return (
        logits,
        clipped
    )


def inverse_logit(
    linear_predictor
):
    """
    Numerically stable inverse-logit transformation.
    """

    linear_predictor = np.asarray(
        linear_predictor,
        dtype=float
    )


    clipped_linear_predictor = np.clip(
        linear_predictor,
        -700,
        700
    )


    return (
        1
        /
        (
            1
            +
            np.exp(
                -clipped_linear_predictor
            )
        )
    )


def apply_locked_logistic_recalibration(
    raw_probability,
    intercept,
    slope
):
    """
    Apply a previously fitted logistic recalibration mapping:

        logit(p_recal) = intercept + slope * logit(p_raw)

    No fitting occurs here.
    """

    raw_logit, clipped_raw_probability = (
        probability_to_logit(
            raw_probability,
            epsilon=PROBABILITY_CLIP_EPSILON
        )
    )


    recalibrated_probability = inverse_logit(
        intercept
        +
        slope
        *
        raw_logit
    )


    return (
        recalibrated_probability,
        clipped_raw_probability
    )


# ============================================================
# 7. HELPER: CALIBRATION INTERCEPT AND SLOPE
# ============================================================

def estimate_calibration_intercept_and_slope(
    y_true,
    y_prob
):
    """
    Diagnostic temporal calibration regression.

    Ideal:
    - intercept = 0
    - slope = 1

    This function evaluates transported probabilities only.
    It does not create a recalibration mapping.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    y_prob = np.asarray(
        y_prob,
        dtype=float
    )


    logits, clipped_probabilities = (
        probability_to_logit(
            y_prob,
            epsilon=PROBABILITY_CLIP_EPSILON
        )
    )


    intercept_design = np.ones(
        (
            len(
                y_true
            ),
            1
        ),
        dtype=float
    )


    intercept_model = sm.GLM(
        y_true,
        intercept_design,
        family=sm.families.Binomial(),
        offset=logits
    )


    intercept_fit = intercept_model.fit(
        maxiter=500,
        disp=0
    )


    calibration_intercept = float(
        intercept_fit.params[
            0
        ]
    )


    calibration_intercept_se = float(
        intercept_fit.bse[
            0
        ]
    )


    slope_design = sm.add_constant(
        logits,
        has_constant="add"
    )


    slope_model = sm.GLM(
        y_true,
        slope_design,
        family=sm.families.Binomial()
    )


    slope_fit = slope_model.fit(
        maxiter=500,
        disp=0
    )


    calibration_model_intercept = float(
        slope_fit.params[
            0
        ]
    )


    calibration_slope = float(
        slope_fit.params[
            1
        ]
    )


    calibration_slope_se = float(
        slope_fit.bse[
            1
        ]
    )


    return {

        "Calibration_Intercept":
            calibration_intercept,

        "Calibration_Intercept_SE":
            calibration_intercept_se,

        "Calibration_Intercept_CI_Lower":
            float(
                calibration_intercept
                -
                1.96
                *
                calibration_intercept_se
            ),

        "Calibration_Intercept_CI_Upper":
            float(
                calibration_intercept
                +
                1.96
                *
                calibration_intercept_se
            ),

        "Calibration_Model_Intercept":
            calibration_model_intercept,

        "Calibration_Slope":
            calibration_slope,

        "Calibration_Slope_SE":
            calibration_slope_se,

        "Calibration_Slope_CI_Lower":
            float(
                calibration_slope
                -
                1.96
                *
                calibration_slope_se
            ),

        "Calibration_Slope_CI_Upper":
            float(
                calibration_slope
                +
                1.96
                *
                calibration_slope_se
            ),

        "Clipped_Probability_Count":
            int(
                np.sum(
                    clipped_probabilities
                    !=
                    y_prob
                )
            )
    }


# ============================================================
# 8. HELPER: ADAPTIVE QUANTILE BINS
# ============================================================

def create_adaptive_calibration_bins(
    y_true,
    y_prob,
    requested_bins=10
):
    """
    Create quantile-based calibration groups.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    y_prob = np.asarray(
        y_prob,
        dtype=float
    )


    working_table = pd.DataFrame({

        "y_true":
            y_true,

        "y_prob":
            y_prob
    })


    unique_probability_count = int(
        working_table[
            "y_prob"
        ].nunique()
    )


    actual_requested_bins = int(
        min(
            requested_bins,
            unique_probability_count,
            len(
                working_table
            )
        )
    )


    if actual_requested_bins < 2:

        working_table[
            "Calibration_Bin"
        ] = 1


    else:

        try:

            working_table[
                "Calibration_Bin"
            ] = (
                pd.qcut(
                    working_table[
                        "y_prob"
                    ],
                    q=actual_requested_bins,
                    labels=False,
                    duplicates="drop"
                )
                +
                1
            )


        except Exception:

            working_table[
                "Probability_Rank"
            ] = (
                working_table[
                    "y_prob"
                ]
                .rank(
                    method="first"
                )
            )


            working_table[
                "Calibration_Bin"
            ] = (
                pd.qcut(
                    working_table[
                        "Probability_Rank"
                    ],
                    q=actual_requested_bins,
                    labels=False,
                    duplicates="drop"
                )
                +
                1
            )


    bin_summary = (
        working_table
        .groupby(
            "Calibration_Bin",
            observed=True
        )
        .agg(

            Bin_N=(
                "y_true",
                "size"
            ),

            Bin_Events=(
                "y_true",
                "sum"
            ),

            Mean_Predicted_Probability=(
                "y_prob",
                "mean"
            ),

            Median_Predicted_Probability=(
                "y_prob",
                "median"
            ),

            Minimum_Predicted_Probability=(
                "y_prob",
                "min"
            ),

            Maximum_Predicted_Probability=(
                "y_prob",
                "max"
            ),

            Observed_Event_Rate=(
                "y_true",
                "mean"
            )
        )
        .reset_index()
    )


    bin_summary[
        "Absolute_Calibration_Error"
    ] = np.abs(
        bin_summary[
            "Observed_Event_Rate"
        ]
        -
        bin_summary[
            "Mean_Predicted_Probability"
        ]
    )


    return (
        working_table,
        bin_summary
    )


# ============================================================
# 9. HELPER: ECE / MCE
# ============================================================

def calculate_binned_calibration_errors(
    bin_summary,
    total_n
):
    """
    Calculate ECE and MCE.
    """

    if len(
        bin_summary
    ) == 0:

        return (
            np.nan,
            np.nan
        )


    weights = (
        bin_summary[
            "Bin_N"
        ]
        /
        total_n
    )


    ece = float(
        np.sum(
            weights
            *
            bin_summary[
                "Absolute_Calibration_Error"
            ]
        )
    )


    mce = float(
        np.max(
            bin_summary[
                "Absolute_Calibration_Error"
            ]
        )
    )


    return (
        ece,
        mce
    )


# ============================================================
# 10. HELPER: LOWESS ICI
# ============================================================

def calculate_lowess_calibration(
    y_true,
    y_prob,
    fraction=0.40,
    iterations=0
):
    """
    Calculate LOWESS-based ICI, E50, and E90.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    y_prob = np.asarray(
        y_prob,
        dtype=float
    )


    unique_probability_count = int(
        len(
            np.unique(
                y_prob
            )
        )
    )


    if unique_probability_count < 3:

        return {

            "ICI":
                np.nan,

            "E50":
                np.nan,

            "E90":
                np.nan,

            "LOWESS_X":
                np.array(
                    [],
                    dtype=float
                ),

            "LOWESS_Y":
                np.array(
                    [],
                    dtype=float
                ),

            "ICI_Estimable":
                False
        }


    aggregated_table = (
        pd.DataFrame({

            "y_prob":
                y_prob,

            "y_true":
                y_true
        })
        .groupby(
            "y_prob",
            as_index=False
        )
        .agg(

            Observed_Rate=(
                "y_true",
                "mean"
            ),

            Weight=(
                "y_true",
                "size"
            )
        )
        .sort_values(
            "y_prob"
        )
        .reset_index(
            drop=True
        )
    )


    lowess_result = lowess(

        endog=aggregated_table[
            "Observed_Rate"
        ].to_numpy(
            dtype=float
        ),

        exog=aggregated_table[
            "y_prob"
        ].to_numpy(
            dtype=float
        ),

        frac=fraction,

        it=iterations,

        delta=0.0,

        is_sorted=True,

        return_sorted=True
    )


    lowess_x = np.asarray(
        lowess_result[
            :,
            0
        ],
        dtype=float
    )


    lowess_y = np.asarray(
        lowess_result[
            :,
            1
        ],
        dtype=float
    )


    lowess_y = np.clip(
        lowess_y,
        0.0,
        1.0
    )


    curve_table = (
        pd.DataFrame({

            "x":
                lowess_x,

            "y":
                lowess_y
        })
        .groupby(
            "x",
            as_index=False
        )[
            "y"
        ]
        .mean()
        .sort_values(
            "x"
        )
    )


    lowess_x = curve_table[
        "x"
    ].to_numpy(
        dtype=float
    )


    lowess_y = curve_table[
        "y"
    ].to_numpy(
        dtype=float
    )


    if len(
        lowess_x
    ) < 2:

        return {

            "ICI":
                np.nan,

            "E50":
                np.nan,

            "E90":
                np.nan,

            "LOWESS_X":
                lowess_x,

            "LOWESS_Y":
                lowess_y,

            "ICI_Estimable":
                False
        }


    lowess_probability = np.interp(
        y_prob,
        lowess_x,
        lowess_y,
        left=lowess_y[
            0
        ],
        right=lowess_y[
            -1
        ]
    )


    lowess_probability = np.clip(
        lowess_probability,
        0.0,
        1.0
    )


    absolute_error = np.abs(
        lowess_probability
        -
        y_prob
    )


    return {

        "ICI":
            float(
                np.mean(
                    absolute_error
                )
            ),

        "E50":
            float(
                np.percentile(
                    absolute_error,
                    50
                )
            ),

        "E90":
            float(
                np.percentile(
                    absolute_error,
                    90
                )
            ),

        "LOWESS_X":
            lowess_x,

        "LOWESS_Y":
            lowess_y,

        "ICI_Estimable":
            True
    }


# ============================================================
# 11. HELPER: PROBABILITY PERFORMANCE SUMMARY
# ============================================================

def calculate_probability_summary(
    y_true,
    y_prob
):
    """
    Calculate discrimination and probability-scale metrics.
    """

    y_true = np.asarray(
        y_true,
        dtype=int
    )


    y_prob = np.asarray(
        y_prob,
        dtype=float
    )


    prevalence = float(
        np.mean(
            y_true
        )
    )


    brier_score = float(
        brier_score_loss(
            y_true,
            y_prob
        )
    )


    null_probability = np.full(
        len(
            y_true
        ),
        prevalence,
        dtype=float
    )


    null_brier_score = float(
        brier_score_loss(
            y_true,
            null_probability
        )
    )


    brier_skill_score = (
        float(
            1
            -
            brier_score
            /
            null_brier_score
        )
        if null_brier_score > 0
        else np.nan
    )


    observed_events = float(
        np.sum(
            y_true
        )
    )


    expected_events = float(
        np.sum(
            y_prob
        )
    )


    observed_to_expected_ratio = (
        float(
            observed_events
            /
            expected_events
        )
        if expected_events > 0
        else np.nan
    )


    return {

        "AUC":
            float(
                roc_auc_score(
                    y_true,
                    y_prob
                )
            ),

        "Average_Precision":
            float(
                average_precision_score(
                    y_true,
                    y_prob
                )
            ),

        "Brier_Score":
            brier_score,

        "Null_Brier_Score":
            null_brier_score,

        "Brier_Skill_Score":
            brier_skill_score,

        "Observed_Prevalence":
            prevalence,

        "Mean_Predicted_Probability":
            float(
                np.mean(
                    y_prob
                )
            ),

        "Observed_Events":
            observed_events,

        "Expected_Events":
            expected_events,

        "Observed_to_Expected_Ratio":
            observed_to_expected_ratio
    }


# ============================================================
# 12. HELPER: CREATE RAW VS RECALIBRATED PLOT
# ============================================================

def create_comparison_calibration_plot(
    model_name,
    algorithm_name,
    y_true,
    raw_probability,
    recalibrated_probability,
    recalibrated_bin_summary,
    recalibrated_lowess_x,
    recalibrated_lowess_y
):
    """
    Create a compact comparison plot for transported
    recalibration.
    """

    figure, axis = plt.subplots(
        figsize=(
            FIGURE_WIDTH,
            FIGURE_HEIGHT
        )
    )


    axis.plot(
        [
            0,
            1
        ],
        [
            0,
            1
        ],
        linestyle="--",
        linewidth=1.5,
        label="Perfect calibration"
    )


    if len(
        recalibrated_bin_summary
    ) > 0:

        axis.plot(
            recalibrated_bin_summary[
                "Mean_Predicted_Probability"
            ],
            recalibrated_bin_summary[
                "Observed_Event_Rate"
            ],
            marker="o",
            linewidth=1.5,
            label="Recalibrated quantile bins"
        )


    if (
        len(
            recalibrated_lowess_x
        )
        >=
        2
    ):

        axis.plot(
            recalibrated_lowess_x,
            recalibrated_lowess_y,
            linewidth=1.75,
            label="Recalibrated LOWESS"
        )


    max_display_probability = float(
        max(
            0.05,
            np.percentile(
                raw_probability,
                99
            ),
            np.percentile(
                recalibrated_probability,
                99
            ),
            np.max(
                recalibrated_bin_summary[
                    "Observed_Event_Rate"
                ]
            )
            if len(
                recalibrated_bin_summary
            ) > 0
            else 0.05
        )
    )


    max_display_probability = min(
        1.0,
        max_display_probability
        *
        1.10
    )


    axis.set_xlim(
        0,
        max_display_probability
    )


    axis.set_ylim(
        0,
        max_display_probability
    )


    axis.set_xlabel(
        "Predicted probability"
    )


    axis.set_ylabel(
        "Observed event rate"
    )


    axis.set_title(
        f"{model_name} — {algorithm_name}\n"
        "OOF-recalibrated temporal calibration (2025)"
    )


    axis.legend(
        loc="upper left",
        fontsize=8
    )


    axis.grid(
        alpha=0.25
    )


    figure.tight_layout()


    safe_model_name = (
        str(
            model_name
        )
        .replace(
            " ",
            "_"
        )
        .replace(
            "/",
            "_"
        )
    )


    safe_algorithm_name = (
        str(
            algorithm_name
        )
        .replace(
            " ",
            "_"
        )
        .replace(
            "/",
            "_"
        )
    )


    png_path = (
        PLOT_OUTPUT_DIRECTORY
        /
        (
            f"part4c2_recalibrated_"
            f"{safe_model_name}_"
            f"{safe_algorithm_name}.png"
        )
    )


    pdf_path = (
        PLOT_OUTPUT_DIRECTORY
        /
        (
            f"part4c2_recalibrated_"
            f"{safe_model_name}_"
            f"{safe_algorithm_name}.pdf"
        )
    )


    figure.savefig(
        png_path,
        dpi=PLOT_DPI,
        bbox_inches="tight"
    )


    figure.savefig(
        pdf_path,
        bbox_inches="tight"
    )


    plt.close(
        figure
    )


    return (
        str(
            png_path
        ),
        str(
            pdf_path
        )
    )


# ============================================================
# 13. DETERMINE EXPECTED MAPPING STATUS
# ============================================================

expected_total_pairs = sum(
    len(
        algorithms
    )
    for algorithms in temporal_predictions.values()
)


successful_mapping_rows = (
    part3d2_recalibration_status[
        part3d2_recalibration_status[
            "Status"
        ]
        ==
        "Success"
    ]
    .copy()
)


not_estimable_mapping_rows = (
    part3d2_recalibration_status[
        part3d2_recalibration_status[
            "Status"
        ]
        ==
        "Not estimable"
    ]
    .copy()
)


expected_successful_mappings = int(
    len(
        successful_mapping_rows
    )
)


expected_not_estimable_mappings = int(
    len(
        not_estimable_mapping_rows
    )
)


if (
    expected_successful_mappings
    +
    expected_not_estimable_mappings
) != expected_total_pairs:

    raise RuntimeError(
        "PART 3D-2 mapping status does not account for every "
        "temporal model-algorithm pair."
    )


# ============================================================
# 14. APPLY LOCKED OOF RECALIBRATION TO TEMPORAL 2025
# ============================================================

print("=" * 100)
print(
    "PART 4C-2 VERSION 3.0: OOF-RECALIBRATED TEMPORAL "
    "CALIBRATION (2025)"
)
print("=" * 100)


print(
    f"Total temporal model-algorithm pairs : "
    f"{expected_total_pairs}"
)


print(
    f"OOF recalibration mappings available: "
    f"{expected_successful_mappings}"
)


print(
    f"OOF mappings not estimable           : "
    f"{expected_not_estimable_mappings}"
)


for model_name, algorithms in (
    temporal_predictions.items()
):

    print("\n" + "=" * 100)
    print(
        f"{model_name}: TRANSPORTED OOF RECALIBRATION"
    )
    print("=" * 100)


    temporal_recalibrated_predictions[
        model_name
    ] = {}


    for algorithm_index, (
        algorithm_name,
        temporal_record
    ) in enumerate(
        algorithms.items(),
        start=1
    ):

        print(
            f"\n[{algorithm_index}/"
            f"{len(algorithms)}] "
            f"{algorithm_name}"
        )


        status_rows = (
            part3d2_recalibration_status[
                (
                    part3d2_recalibration_status[
                        "Screening_Model"
                    ]
                    ==
                    model_name
                )
                &
                (
                    part3d2_recalibration_status[
                        "Algorithm"
                    ]
                    ==
                    algorithm_name
                )
            ]
        )


        if len(
            status_rows
        ) != 1:

            raise RuntimeError(
                f"{model_name} | {algorithm_name}: expected "
                "exactly one PART 3D-2 status row."
            )


        mapping_status = str(
            status_rows.iloc[
                0
            ][
                "Status"
            ]
        )


        if mapping_status == "Not estimable":

            recalibration_status_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Not estimable",

                "Mapping_Status_PART3D2":
                    mapping_status,

                "Mapping_Intercept":
                    np.nan,

                "Mapping_Slope":
                    np.nan,

                "Monotonic_Increasing":
                    False,

                "AUC_Preserved":
                    False,

                "N":
                    int(
                        len(
                            temporal_record[
                                "y_true"
                            ]
                        )
                    ),

                "Events":
                    int(
                        np.sum(
                            temporal_record[
                                "y_true"
                            ]
                        )
                    ),

                "Error":
                    str(
                        status_rows.iloc[
                            0
                        ].get(
                            "Error",
                            ""
                        )
                    )
            })


            recalibration_plot_manifest_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "PNG_File":
                    "",

                "PDF_File":
                    "",

                "Plot_Created":
                    False,

                "Reason":
                    "OOF recalibration mapping not estimable"
            })


            print(
                "Not estimable — excluded from transported "
                "recalibration analysis."
            )


            continue


        if mapping_status != "Success":

            raise RuntimeError(
                f"{model_name} | {algorithm_name}: unexpected "
                f"PART 3D-2 mapping status '{mapping_status}'."
            )


        try:

            if (
                model_name
                not in
                oof_recalibration_models
                or
                algorithm_name
                not in
                oof_recalibration_models[
                    model_name
                ]
            ):

                raise KeyError(
                    "Successful OOF recalibration mapping is "
                    "missing from oof_recalibration_models."
                )


            mapping = (
                oof_recalibration_models[
                    model_name
                ][
                    algorithm_name
                ]
            )


            mapping_intercept = float(
                mapping[
                    "Intercept"
                ]
            )


            mapping_slope = float(
                mapping[
                    "Slope"
                ]
            )


            if not (
                np.isfinite(
                    mapping_intercept
                )
                and
                np.isfinite(
                    mapping_slope
                )
            ):

                raise ValueError(
                    "Recalibration mapping coefficients are "
                    "nonfinite."
                )


            y_true = np.asarray(
                temporal_record[
                    "y_true"
                ],
                dtype=int
            )


            raw_probability = np.asarray(
                temporal_record[
                    "y_prob"
                ],
                dtype=float
            )


            raw_locked_threshold = float(
                temporal_record[
                    "locked_threshold"
                ]
            )


            original_index = np.asarray(
                temporal_record[
                    "original_index"
                ]
            )


            (
                recalibrated_probability,
                clipped_raw_probability
            ) = apply_locked_logistic_recalibration(
                raw_probability=raw_probability,
                intercept=mapping_intercept,
                slope=mapping_slope
            )


            if not (
                np.all(
                    np.isfinite(
                        recalibrated_probability
                    )
                )
                and
                np.all(
                    recalibrated_probability
                    >=
                    0
                )
                and
                np.all(
                    recalibrated_probability
                    <=
                    1
                )
            ):

                raise ValueError(
                    "Recalibrated temporal probabilities are "
                    "invalid."
                )


            # ------------------------------------------------
            # Raw probability summary
            # ------------------------------------------------

            raw_summary = (
                calculate_probability_summary(
                    y_true=y_true,
                    y_prob=raw_probability
                )
            )


            # ------------------------------------------------
            # Recalibrated probability summary
            # ------------------------------------------------

            recalibrated_summary = (
                calculate_probability_summary(
                    y_true=y_true,
                    y_prob=recalibrated_probability
                )
            )


            # ------------------------------------------------
            # Recalibrated diagnostic calibration regression
            # ------------------------------------------------

            recalibrated_calibration_regression = (
                estimate_calibration_intercept_and_slope(
                    y_true=y_true,
                    y_prob=recalibrated_probability
                )
            )


            # ------------------------------------------------
            # Recalibrated calibration bins
            # ------------------------------------------------

            (
                recalibrated_participant_bins,
                recalibrated_bin_summary
            ) = create_adaptive_calibration_bins(
                y_true=y_true,
                y_prob=recalibrated_probability,
                requested_bins=N_CALIBRATION_BINS
            )


            actual_bin_count = int(
                recalibrated_bin_summary[
                    "Calibration_Bin"
                ].nunique()
            )


            recalibrated_ece, recalibrated_mce = (
                calculate_binned_calibration_errors(
                    bin_summary=recalibrated_bin_summary,
                    total_n=len(
                        y_true
                    )
                )
            )


            recalibrated_lowess = (
                calculate_lowess_calibration(
                    y_true=y_true,
                    y_prob=recalibrated_probability,
                    fraction=LOWESS_FRACTION,
                    iterations=LOWESS_ITERATIONS
                )
            )


            # ------------------------------------------------
            # Raw calibration metrics from PART 4C-1 if present
            # ------------------------------------------------

            raw_calibration_metrics = {

                "Raw_Calibration_Intercept":
                    np.nan,

                "Raw_Calibration_Slope":
                    np.nan,

                "Raw_ECE":
                    np.nan,

                "Raw_MCE":
                    np.nan,

                "Raw_ICI":
                    np.nan,

                "Raw_E50":
                    np.nan,

                "Raw_E90":
                    np.nan
            }


            if (
                "part4c1_calibration_results"
                in globals()
                and
                isinstance(
                    part4c1_calibration_results,
                    pd.DataFrame
                )
            ):

                raw_calibration_row = (
                    part4c1_calibration_results[
                        (
                            part4c1_calibration_results[
                                "Screening_Model"
                            ]
                            ==
                            model_name
                        )
                        &
                        (
                            part4c1_calibration_results[
                                "Algorithm"
                            ]
                            ==
                            algorithm_name
                        )
                    ]
                )


                if len(
                    raw_calibration_row
                ) != 1:

                    raise RuntimeError(
                        "Expected exactly one PART 4C-1 "
                        "calibration result row."
                    )


                raw_calibration_row = (
                    raw_calibration_row.iloc[
                        0
                    ]
                )


                raw_calibration_metrics = {

                    "Raw_Calibration_Intercept":
                        float(
                            raw_calibration_row[
                                "Calibration_Intercept"
                            ]
                        ),

                    "Raw_Calibration_Slope":
                        float(
                            raw_calibration_row[
                                "Calibration_Slope"
                            ]
                        ),

                    "Raw_ECE":
                        float(
                            raw_calibration_row[
                                "ECE"
                            ]
                        ),

                    "Raw_MCE":
                        float(
                            raw_calibration_row[
                                "MCE"
                            ]
                        ),

                    "Raw_ICI":
                        float(
                            raw_calibration_row[
                                "ICI"
                            ]
                        ),

                    "Raw_E50":
                        float(
                            raw_calibration_row[
                                "E50"
                            ]
                        ),

                    "Raw_E90":
                        float(
                            raw_calibration_row[
                                "E90"
                            ]
                        )
                }


            # ------------------------------------------------
            # Monotonicity / discrimination preservation
            # ------------------------------------------------

            monotonic_increasing = bool(
                mapping_slope
                >
                0
            )


            if monotonic_increasing:

                auc_preserved = bool(
                    np.isclose(
                        raw_summary[
                            "AUC"
                        ],
                        recalibrated_summary[
                            "AUC"
                        ],
                        rtol=0,
                        atol=NUMERICAL_ATOL
                    )
                )


                ap_preserved = bool(
                    np.isclose(
                        raw_summary[
                            "Average_Precision"
                        ],
                        recalibrated_summary[
                            "Average_Precision"
                        ],
                        rtol=0,
                        atol=NUMERICAL_ATOL
                    )
                )


            else:

                auc_preserved = False

                ap_preserved = False


            # ------------------------------------------------
            # Save bin table
            # ------------------------------------------------

            model_bin_summary = (
                recalibrated_bin_summary.copy()
            )


            model_bin_summary[
                "Validation_Set"
            ] = VALIDATION_SET_NAME


            model_bin_summary[
                "Screening_Model"
            ] = model_name


            model_bin_summary[
                "Algorithm"
            ] = algorithm_name


            model_bin_summary[
                "Probability_Scale"
            ] = "OOF-recalibrated probability"


            model_bin_summary[
                "Mapping_Intercept"
            ] = mapping_intercept


            model_bin_summary[
                "Mapping_Slope"
            ] = mapping_slope


            recalibration_bin_rows.append(
                model_bin_summary
            )


            # ------------------------------------------------
            # Plot
            # ------------------------------------------------

            png_path, pdf_path = (
                create_comparison_calibration_plot(
                    model_name=model_name,
                    algorithm_name=algorithm_name,
                    y_true=y_true,
                    raw_probability=raw_probability,
                    recalibrated_probability=recalibrated_probability,
                    recalibrated_bin_summary=recalibrated_bin_summary,
                    recalibrated_lowess_x=recalibrated_lowess[
                        "LOWESS_X"
                    ],
                    recalibrated_lowess_y=recalibrated_lowess[
                        "LOWESS_Y"
                    ]
                )
            )


            recalibration_plot_manifest_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "PNG_File":
                    png_path,

                "PDF_File":
                    pdf_path,

                "Plot_Created":
                    True,

                "Reason":
                    ""
            })


            # ------------------------------------------------
            # Participant-level nested object
            # ------------------------------------------------

            temporal_recalibrated_predictions[
                model_name
            ][
                algorithm_name
            ] = {

                "validation_set":
                    VALIDATION_SET_NAME,

                "y_true":
                    y_true.copy(),

                "raw_y_prob":
                    raw_probability.copy(),

                "recalibrated_y_prob":
                    recalibrated_probability.copy(),

                "raw_locked_threshold":
                    raw_locked_threshold,

                "original_index":
                    original_index.copy(),

                "recalibration_method":
                    "Logistic recalibration",

                "recalibration_intercept":
                    mapping_intercept,

                "recalibration_slope":
                    mapping_slope,

                "recalibration_source":
                    (
                        "Genuine 2024 OOF predictions only"
                    ),

                "classification_threshold_modified":
                    False,

                "recalibration_fitted_on_temporal_data":
                    False,

                "monotonic_increasing":
                    monotonic_increasing,

                "auc_preserved":
                    auc_preserved,

                "average_precision_preserved":
                    ap_preserved
            }


            # ------------------------------------------------
            # Main result row
            # ------------------------------------------------

            result_row = {

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "N":
                    int(
                        len(
                            y_true
                        )
                    ),

                "Events":
                    int(
                        np.sum(
                            y_true
                        )
                    ),

                "Prevalence":
                    float(
                        np.mean(
                            y_true
                        )
                    ),

                "Mapping_Method":
                    "Logistic recalibration",

                "Mapping_Source":
                    "Genuine 2024 OOF predictions only",

                "Mapping_Intercept":
                    mapping_intercept,

                "Mapping_Slope":
                    mapping_slope,

                "Monotonic_Increasing":
                    monotonic_increasing,

                "AUC_Preserved":
                    auc_preserved,

                "Average_Precision_Preserved":
                    ap_preserved,

                "Raw_AUC":
                    raw_summary[
                        "AUC"
                    ],

                "Recalibrated_AUC":
                    recalibrated_summary[
                        "AUC"
                    ],

                "Raw_Average_Precision":
                    raw_summary[
                        "Average_Precision"
                    ],

                "Recalibrated_Average_Precision":
                    recalibrated_summary[
                        "Average_Precision"
                    ],

                "Raw_Brier_Score":
                    raw_summary[
                        "Brier_Score"
                    ],

                "Recalibrated_Brier_Score":
                    recalibrated_summary[
                        "Brier_Score"
                    ],

                "Brier_Improvement":
                    float(
                        raw_summary[
                            "Brier_Score"
                        ]
                        -
                        recalibrated_summary[
                            "Brier_Score"
                        ]
                    ),

                "Raw_Brier_Skill_Score":
                    raw_summary[
                        "Brier_Skill_Score"
                    ],

                "Recalibrated_Brier_Skill_Score":
                    recalibrated_summary[
                        "Brier_Skill_Score"
                    ],

                "Raw_Mean_Predicted_Probability":
                    raw_summary[
                        "Mean_Predicted_Probability"
                    ],

                "Recalibrated_Mean_Predicted_Probability":
                    recalibrated_summary[
                        "Mean_Predicted_Probability"
                    ],

                "Observed_Prevalence":
                    recalibrated_summary[
                        "Observed_Prevalence"
                    ],

                "Raw_Expected_Events":
                    raw_summary[
                        "Expected_Events"
                    ],

                "Recalibrated_Expected_Events":
                    recalibrated_summary[
                        "Expected_Events"
                    ],

                "Observed_Events":
                    recalibrated_summary[
                        "Observed_Events"
                    ],

                "Raw_Observed_to_Expected_Ratio":
                    raw_summary[
                        "Observed_to_Expected_Ratio"
                    ],

                "Recalibrated_Observed_to_Expected_Ratio":
                    recalibrated_summary[
                        "Observed_to_Expected_Ratio"
                    ],

                **raw_calibration_metrics,

                "Recalibrated_Calibration_Intercept":
                    recalibrated_calibration_regression[
                        "Calibration_Intercept"
                    ],

                "Recalibrated_Calibration_Intercept_CI_Lower":
                    recalibrated_calibration_regression[
                        "Calibration_Intercept_CI_Lower"
                    ],

                "Recalibrated_Calibration_Intercept_CI_Upper":
                    recalibrated_calibration_regression[
                        "Calibration_Intercept_CI_Upper"
                    ],

                "Recalibrated_Calibration_Slope":
                    recalibrated_calibration_regression[
                        "Calibration_Slope"
                    ],

                "Recalibrated_Calibration_Slope_CI_Lower":
                    recalibrated_calibration_regression[
                        "Calibration_Slope_CI_Lower"
                    ],

                "Recalibrated_Calibration_Slope_CI_Upper":
                    recalibrated_calibration_regression[
                        "Calibration_Slope_CI_Upper"
                    ],

                "Recalibrated_ECE":
                    recalibrated_ece,

                "Recalibrated_MCE":
                    recalibrated_mce,

                "Recalibrated_ICI":
                    recalibrated_lowess[
                        "ICI"
                    ],

                "Recalibrated_E50":
                    recalibrated_lowess[
                        "E50"
                    ],

                "Recalibrated_E90":
                    recalibrated_lowess[
                        "E90"
                    ],

                "Recalibrated_ICI_Estimable":
                    recalibrated_lowess[
                        "ICI_Estimable"
                    ],

                "Brier_Improved":
                    bool(
                        recalibrated_summary[
                            "Brier_Score"
                        ]
                        <
                        raw_summary[
                            "Brier_Score"
                        ]
                    ),

                "OE_Closer_To_One":
                    bool(
                        abs(
                            recalibrated_summary[
                                "Observed_to_Expected_Ratio"
                            ]
                            -
                            1
                        )
                        <
                        abs(
                            raw_summary[
                                "Observed_to_Expected_Ratio"
                            ]
                            -
                            1
                        )
                    ),

                "Raw_Probability_Clipped_Count_For_Logit":
                    int(
                        np.sum(
                            clipped_raw_probability
                            !=
                            raw_probability
                        )
                    ),

                "Locked_Threshold":
                    raw_locked_threshold,

                "Threshold_Modified":
                    False,

                "Recalibration_Fitted_On_Temporal_Data":
                    False
            }


            recalibration_result_rows.append(
                result_row
            )


            recalibration_status_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Success",

                "Mapping_Status_PART3D2":
                    mapping_status,

                "Mapping_Intercept":
                    mapping_intercept,

                "Mapping_Slope":
                    mapping_slope,

                "Monotonic_Increasing":
                    monotonic_increasing,

                "AUC_Preserved":
                    auc_preserved,

                "N":
                    int(
                        len(
                            y_true
                        )
                    ),

                "Events":
                    int(
                        np.sum(
                            y_true
                        )
                    ),

                "Brier_Improved":
                    result_row[
                        "Brier_Improved"
                    ],

                "OE_Closer_To_One":
                    result_row[
                        "OE_Closer_To_One"
                    ],

                "Threshold_Modified":
                    False,

                "Recalibration_Fitted_On_Temporal_Data":
                    False,

                "Error":
                    ""
            })


            print(
                f"Map intercept="
                f"{mapping_intercept:.3f} | "
                f"Map slope="
                f"{mapping_slope:.3f} | "
                f"Raw Brier="
                f"{raw_summary['Brier_Score']:.4f} -> "
                f"Recalibrated="
                f"{recalibrated_summary['Brier_Score']:.4f} | "
                f"O:E="
                f"{raw_summary['Observed_to_Expected_Ratio']:.3f} -> "
                f"{recalibrated_summary['Observed_to_Expected_Ratio']:.3f} | "
                f"AUC="
                f"{raw_summary['AUC']:.4f} -> "
                f"{recalibrated_summary['AUC']:.4f}"
            )


        except Exception as error:

            recalibration_status_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "Status":
                    "Failed",

                "Mapping_Status_PART3D2":
                    mapping_status,

                "Mapping_Intercept":
                    np.nan,

                "Mapping_Slope":
                    np.nan,

                "Monotonic_Increasing":
                    False,

                "AUC_Preserved":
                    False,

                "N":
                    int(
                        len(
                            temporal_record[
                                "y_true"
                            ]
                        )
                    ),

                "Events":
                    int(
                        np.sum(
                            temporal_record[
                                "y_true"
                            ]
                        )
                    ),

                "Brier_Improved":
                    False,

                "OE_Closer_To_One":
                    False,

                "Threshold_Modified":
                    False,

                "Recalibration_Fitted_On_Temporal_Data":
                    False,

                "Error":
                    str(
                        error
                    )
            })


            recalibration_plot_manifest_rows.append({

                "Validation_Set":
                    VALIDATION_SET_NAME,

                "Screening_Model":
                    model_name,

                "Algorithm":
                    algorithm_name,

                "PNG_File":
                    "",

                "PDF_File":
                    "",

                "Plot_Created":
                    False,

                "Reason":
                    str(
                        error
                    )
            })


            print(
                "Failed"
            )


            print(
                f"Reason: {error}"
            )


# ============================================================
# 15. CREATE RESULT TABLES
# ============================================================

part4c2_recalibration_results = pd.DataFrame(
    recalibration_result_rows
)


if recalibration_bin_rows:

    part4c2_recalibration_bins = pd.concat(
        recalibration_bin_rows,
        ignore_index=True
    )


else:

    part4c2_recalibration_bins = pd.DataFrame()


part4c2_recalibration_status = pd.DataFrame(
    recalibration_status_rows
)


part4c2_recalibration_plot_manifest = pd.DataFrame(
    recalibration_plot_manifest_rows
)


# ============================================================
# 16. VERIFICATION
# ============================================================

successful_transport_count = int(
    (
        part4c2_recalibration_status[
            "Status"
        ]
        ==
        "Success"
    ).sum()
)


not_estimable_transport_count = int(
    (
        part4c2_recalibration_status[
            "Status"
        ]
        ==
        "Not estimable"
    ).sum()
)


failed_transport_count = int(
    (
        part4c2_recalibration_status[
            "Status"
        ]
        ==
        "Failed"
    ).sum()
)


for model_name, algorithms in (
    temporal_predictions.items()
):

    expected_algorithms = set(
        algorithms.keys()
    )


    success_algorithms = set(
        part4c2_recalibration_status.loc[
            (
                part4c2_recalibration_status[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                part4c2_recalibration_status[
                    "Status"
                ]
                ==
                "Success"
            ),
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )


    not_estimable_algorithms = set(
        part4c2_recalibration_status.loc[
            (
                part4c2_recalibration_status[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                part4c2_recalibration_status[
                    "Status"
                ]
                ==
                "Not estimable"
            ),
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )


    accounted_algorithms = (
        success_algorithms
        |
        not_estimable_algorithms
    )


    missing_algorithms = sorted(
        expected_algorithms
        -
        accounted_algorithms
    )


    model_results = (
        part4c2_recalibration_results[
            part4c2_recalibration_results[
                "Screening_Model"
            ]
            ==
            model_name
        ]
    )


    all_recalibrated_probabilities_valid = True

    all_raw_probabilities_unchanged = True

    all_outcome_alignment_correct = True

    all_index_alignment_correct = True

    all_thresholds_unchanged = True

    no_temporal_recalibration_fit = True

    all_mapping_parameters_match = True


    for algorithm_name in (
        success_algorithms
    ):

        transported_record = (
            temporal_recalibrated_predictions[
                model_name
            ][
                algorithm_name
            ]
        )


        raw_record = (
            temporal_predictions[
                model_name
            ][
                algorithm_name
            ]
        )


        mapping = (
            oof_recalibration_models[
                model_name
            ][
                algorithm_name
            ]
        )


        recalibrated_probability = np.asarray(
            transported_record[
                "recalibrated_y_prob"
            ],
            dtype=float
        )


        if not (
            np.all(
                np.isfinite(
                    recalibrated_probability
                )
            )
            and
            np.all(
                recalibrated_probability
                >=
                0
            )
            and
            np.all(
                recalibrated_probability
                <=
                1
            )
        ):

            all_recalibrated_probabilities_valid = False


        if not np.array_equal(
            transported_record[
                "raw_y_prob"
            ],
            raw_record[
                "y_prob"
            ]
        ):

            all_raw_probabilities_unchanged = False


        if not np.array_equal(
            transported_record[
                "y_true"
            ],
            raw_record[
                "y_true"
            ]
        ):

            all_outcome_alignment_correct = False


        if not np.array_equal(
            transported_record[
                "original_index"
            ],
            raw_record[
                "original_index"
            ]
        ):

            all_index_alignment_correct = False


        if not np.isclose(
            transported_record[
                "raw_locked_threshold"
            ],
            raw_record[
                "locked_threshold"
            ],
            rtol=0,
            atol=THRESHOLD_ATOL
        ):

            all_thresholds_unchanged = False


        if transported_record[
            "classification_threshold_modified"
        ]:

            all_thresholds_unchanged = False


        if transported_record[
            "recalibration_fitted_on_temporal_data"
        ]:

            no_temporal_recalibration_fit = False


        if not np.isclose(
            transported_record[
                "recalibration_intercept"
            ],
            mapping[
                "Intercept"
            ],
            rtol=0,
            atol=NUMERICAL_ATOL
        ):

            all_mapping_parameters_match = False


        if not np.isclose(
            transported_record[
                "recalibration_slope"
            ],
            mapping[
                "Slope"
            ],
            rtol=0,
            atol=NUMERICAL_ATOL
        ):

            all_mapping_parameters_match = False


    at_least_one_successful_mapping = bool(
        len(
            success_algorithms
        )
        >=
        1
    )


    all_result_metrics_finite = bool(
        np.isfinite(
            model_results[
                [
                    "Recalibrated_Brier_Score",
                    "Recalibrated_Mean_Predicted_Probability",
                    "Recalibrated_Observed_to_Expected_Ratio",
                    "Recalibrated_Calibration_Intercept",
                    "Recalibrated_Calibration_Slope",
                    "Recalibrated_ECE",
                    "Recalibrated_ICI"
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )
        .all()
    ) if not model_results.empty else False


    recalibration_verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            int(
                len(
                    expected_algorithms
                )
            ),

        "Successful_Transported_Mappings":
            int(
                len(
                    success_algorithms
                )
            ),

        "Not_Estimable_Mappings":
            int(
                len(
                    not_estimable_algorithms
                )
            ),

        "Not_Estimable_Algorithm_Names":
            ", ".join(
                sorted(
                    not_estimable_algorithms
                )
            ),

        "Missing_Algorithms":
            ", ".join(
                missing_algorithms
            ),

        "At_Least_One_Successful_Mapping":
            at_least_one_successful_mapping,

        "All_Recalibrated_Probabilities_Valid":
            all_recalibrated_probabilities_valid,

        "All_Raw_Probabilities_Unchanged":
            all_raw_probabilities_unchanged,

        "All_Outcome_Alignment_Correct":
            all_outcome_alignment_correct,

        "All_Index_Alignment_Correct":
            all_index_alignment_correct,

        "All_Locked_Thresholds_Unchanged":
            all_thresholds_unchanged,

        "No_Recalibration_Fit_On_Temporal_Data":
            no_temporal_recalibration_fit,

        "All_Mapping_Parameters_Match_PART3D2":
            all_mapping_parameters_match,

        "All_Result_Metrics_Finite":
            all_result_metrics_finite,

        "Temporal_OOF_Recalibration_Ready":
            (
                len(
                    missing_algorithms
                )
                ==
                0
                and
                at_least_one_successful_mapping
                and
                all_recalibrated_probabilities_valid
                and
                all_raw_probabilities_unchanged
                and
                all_outcome_alignment_correct
                and
                all_index_alignment_correct
                and
                all_thresholds_unchanged
                and
                no_temporal_recalibration_fit
                and
                all_mapping_parameters_match
                and
                all_result_metrics_finite
            )
    })


part4c2_recalibration_verification = pd.DataFrame(
    recalibration_verification_rows
)


# ============================================================
# 17. PUBLICATION TABLE
# ============================================================

part4c2_recalibration_publication_table = (
    part4c2_recalibration_results[
        [
            "Validation_Set",
            "Screening_Model",
            "Algorithm",
            "N",
            "Events",
            "Prevalence",
            "Mapping_Intercept",
            "Mapping_Slope",
            "Monotonic_Increasing",
            "Raw_AUC",
            "Recalibrated_AUC",
            "Raw_Average_Precision",
            "Recalibrated_Average_Precision",
            "Raw_Brier_Score",
            "Recalibrated_Brier_Score",
            "Brier_Improvement",
            "Raw_Brier_Skill_Score",
            "Recalibrated_Brier_Skill_Score",
            "Raw_Mean_Predicted_Probability",
            "Recalibrated_Mean_Predicted_Probability",
            "Observed_Prevalence",
            "Raw_Observed_to_Expected_Ratio",
            "Recalibrated_Observed_to_Expected_Ratio",
            "Raw_Calibration_Intercept",
            "Recalibrated_Calibration_Intercept",
            "Raw_Calibration_Slope",
            "Recalibrated_Calibration_Slope",
            "Raw_ECE",
            "Recalibrated_ECE",
            "Raw_ICI",
            "Recalibrated_ICI",
            "Brier_Improved",
            "OE_Closer_To_One",
            "AUC_Preserved",
            "Threshold_Modified",
            "Recalibration_Fitted_On_Temporal_Data"
        ]
    ]
    .copy()
)


# ============================================================
# 18. COMPACT PUBLICATION TABLE
# ============================================================

part4c2_recalibration_publication_table_compact = (
    part4c2_recalibration_results[
        [
            "Validation_Set",
            "Screening_Model",
            "Algorithm",
            "Mapping_Slope",
            "Raw_AUC",
            "Recalibrated_AUC",
            "Raw_Brier_Score",
            "Recalibrated_Brier_Score",
            "Raw_Mean_Predicted_Probability",
            "Recalibrated_Mean_Predicted_Probability",
            "Observed_Prevalence",
            "Raw_Observed_to_Expected_Ratio",
            "Recalibrated_Observed_to_Expected_Ratio",
            "Raw_Calibration_Intercept",
            "Recalibrated_Calibration_Intercept",
            "Raw_Calibration_Slope",
            "Recalibrated_Calibration_Slope",
            "Raw_ECE",
            "Recalibrated_ECE",
            "Raw_ICI",
            "Recalibrated_ICI",
            "Brier_Improved",
            "OE_Closer_To_One"
        ]
    ]
    .copy()
)


# ============================================================
# 19. DISPLAY
# ============================================================

print("\n" + "=" * 100)
print("PART 4C-2 TRANSPORTED RECALIBRATION STATUS")
print("=" * 100)


print(
    part4c2_recalibration_status[
        "Status"
    ]
    .value_counts(
        dropna=False
    )
)


print("\n" + "=" * 100)
print("RAW VS OOF-RECALIBRATED TEMPORAL CALIBRATION")
print("=" * 100)


display(
    part4c2_recalibration_publication_table_compact
    .sort_values(
        by=[
            "Screening_Model",
            "Algorithm"
        ]
    )
    .round(
        4
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 100)
print("PART 4C-2 RECALIBRATION VERIFICATION")
print("=" * 100)


display(
    part4c2_recalibration_verification
)


# ============================================================
# 20. HARD SAFETY CHECKS
# ============================================================

if failed_transport_count > 0:

    failed_rows = (
        part4c2_recalibration_status[
            part4c2_recalibration_status[
                "Status"
            ]
            ==
            "Failed"
        ]
    )


    display(
        failed_rows
    )


    raise RuntimeError(
        f"{failed_transport_count} transported OOF "
        "recalibration analyses failed."
    )


if (
    successful_transport_count
    !=
    expected_successful_mappings
):

    raise RuntimeError(
        "Successful transported recalibration count does not "
        "match the number of successful PART 3D-2 mappings."
    )


if (
    not_estimable_transport_count
    !=
    expected_not_estimable_mappings
):

    raise RuntimeError(
        "Not-estimable transported mapping count does not match "
        "PART 3D-2."
    )


if not part4c2_recalibration_verification[
    "Temporal_OOF_Recalibration_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies failed transported "
        "OOF recalibration verification."
    )


# ============================================================
# 21. COHORT CHECK
# ============================================================

reference_model_name = next(
    iter(
        temporal_predictions
    )
)


reference_algorithm_name = next(
    iter(
        temporal_predictions[
            reference_model_name
        ]
    )
)


reference_y_true = np.asarray(
    temporal_predictions[
        reference_model_name
    ][
        reference_algorithm_name
    ][
        "y_true"
    ],
    dtype=int
)


if len(
    reference_y_true
) != EXPECTED_TEMPORAL_N:

    print(
        f"\nWarning: expected temporal N="
        f"{EXPECTED_TEMPORAL_N:,}, observed "
        f"N={len(reference_y_true):,}."
    )


if int(
    np.sum(
        reference_y_true
        ==
        1
    )
) != EXPECTED_TEMPORAL_EVENTS:

    print(
        f"\nWarning: expected temporal events="
        f"{EXPECTED_TEMPORAL_EVENTS}, observed "
        f"events="
        f"{int(np.sum(reference_y_true == 1))}."
    )


# ============================================================
# 22. EXPORT
# ============================================================

part4c2_recalibration_results.to_csv(
    "part4c2_oof_recalibrated_temporal_results.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c2_recalibration_bins.to_csv(
    "part4c2_oof_recalibrated_temporal_bins.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c2_recalibration_status.to_csv(
    "part4c2_oof_recalibrated_temporal_status.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c2_recalibration_verification.to_csv(
    "part4c2_oof_recalibrated_temporal_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c2_recalibration_publication_table.to_csv(
    "part4c2_oof_recalibrated_temporal_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c2_recalibration_publication_table_compact.to_csv(
    "part4c2_oof_recalibrated_temporal_publication_table_compact.csv",
    index=False,
    encoding="utf-8-sig"
)


part4c2_recalibration_plot_manifest.to_csv(
    "part4c2_oof_recalibrated_temporal_plot_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 23. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 4C-2 VERSION 3.0 COMPLETED")
print("=" * 100)


print(
    f"Total temporal model-algorithm pairs : "
    f"{expected_total_pairs}"
)


print(
    f"Successful transported mappings      : "
    f"{successful_transport_count}"
)


print(
    f"Not-estimable mappings               : "
    f"{not_estimable_transport_count}"
)


print(
    f"Failed transported mappings          : "
    f"{failed_transport_count}"
)


print(
    f"Recalibration source                 : "
    f"Genuine 2024 OOF predictions only"
)


print("\nArtifacts created:")

print("- temporal_recalibrated_predictions")
print("- part4c2_recalibration_results")
print("- part4c2_recalibration_bins")
print("- part4c2_recalibration_status")
print("- part4c2_recalibration_verification")
print("- part4c2_recalibration_publication_table")
print("- part4c2_recalibration_publication_table_compact")
print("- part4c2_recalibration_plot_manifest")


print("\nExported files:")

print("- part4c2_oof_recalibrated_temporal_results.csv")
print("- part4c2_oof_recalibrated_temporal_bins.csv")
print("- part4c2_oof_recalibrated_temporal_status.csv")
print("- part4c2_oof_recalibrated_temporal_verification.csv")
print("- part4c2_oof_recalibrated_temporal_publication_table.csv")
print("- part4c2_oof_recalibrated_temporal_publication_table_compact.csv")
print("- part4c2_oof_recalibrated_temporal_plot_manifest.csv")
print(
    f"- PNG/PDF calibration plots in "
    f"{PLOT_OUTPUT_DIRECTORY}/"
)


print(
    "\nNo recalibration model was fitted on the 2025 cohort. "
    "Every recalibration mapping was learned exclusively from "
    "genuine 2024 OOF predictions."
)


print(
    "Raw temporal probabilities and original OOF-locked "
    "classification thresholds were retained unchanged."
)


print(
    "Recalibrated probabilities are used only for secondary "
    "probability-calibration assessment; PART 4A remains the "
    "primary temporal classification analysis."
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

model = "M2_Admin_2Q"
algorithm = "AdaBoostClassifier"

cohorts = [
    ("Held-out 2024", recalibrated_heldout_predictions, 1227, 14),
    ("Temporal 2025", temporal_recalibrated_predictions, 815, 16),
]

rows = []
for cohort, prediction_store, expected_n, expected_events in cohorts:
    result = prediction_store[model][algorithm]
    y = np.asarray(result["y_true"])
    p = np.asarray(result["recalibrated_y_prob"], dtype=float)

    if (
        y.shape != (expected_n,)
        or p.shape != y.shape
        or not np.isin(y, [0, 1]).all()
        or y.sum() != expected_events
        or not np.isfinite(p).all()
        or not ((p >= 0) & (p <= 1)).all()
    ):
        raise ValueError(f"{cohort}: unexpected cohort size, outcomes, or probabilities.")

    cases = y == 1
    n = len(y)
    n_cases = int(cases.sum())
    n_non_cases = n - n_cases

    for threshold in (0.01, 0.02):
        refer = p >= threshold
        tp = int(np.count_nonzero(refer & cases))
        fp = int(np.count_nonzero(refer & ~cases))
        threshold_odds = threshold / (1 - threshold)

        model_nb = (tp - fp * threshold_odds) / n
        treat_all_nb = (n_cases - n_non_cases * threshold_odds) / n

        rows.append({
            "Cohort": cohort,
            "Threshold_Probability": threshold,
            "N": n,
            "MDD_Events": n_cases,
            "True_Positives": tp,
            "False_Positives": fp,
            "Sensitivity": tp / n_cases,
            "Specificity": (n_non_cases - fp) / n_non_cases,
            "Model_Net_Benefit": model_nb,
            "Treat_All_Net_Benefit": treat_all_nb,
            "Net_Reduction_Per_100_vs_Treat_All": (
                100 * (model_nb - treat_all_nb) / threshold_odds
            ),
        })

table_s17_check = pd.DataFrame(rows)
output_path = Path("Table_S17_M2_Net_Reduction_vs_Treat_All.csv")
table_s17_check.to_csv(output_path, index=False, encoding="utf-8-sig")

display(table_s17_check)
print(f"Saved: {output_path.resolve()}")

### Part 4C-3: Held-out 2024 Recalibrated Calibration

In [ ]:
# PART 4C-3: DECISION CURVE ANALYSIS — FIGURE S5 AND TABLE S17

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

model = "M2_Admin_2Q"
algorithm = "AdaBoostClassifier"

cohorts = [
    ("A", "Held-out 2024", recalibrated_heldout_predictions, 1227, 14),
    ("B", "Temporal 2025", temporal_recalibrated_predictions, 815, 16),
]

# IDs 5–100 correspond to thresholds of 0.5%–10.0%.
threshold_ids = np.arange(5, 101)
thresholds = threshold_ids / 1000
threshold_odds = thresholds / (1 - thresholds)
table_threshold_ids = [5, 10, 20, 30, 50, 75, 100]

n_bootstrap = 2000
rng = np.random.default_rng(42)
cohort_results = []

for panel, cohort, prediction_store, n, events in cohorts:
    prediction = prediction_store[model][algorithm]
    y = np.asarray(prediction["y_true"])
    p = np.asarray(prediction["recalibrated_y_prob"], dtype=float)

    if (
        y.shape != (n,)
        or p.shape != (n,)
        or not np.isin(y, [0, 1]).all()
        or y.sum() != events
        or not np.all(np.isfinite(p) & (p >= 0) & (p <= 1))
    ):
        raise ValueError(f"{cohort}: unexpected outcomes or probabilities.")

    y = y.astype(int)
    cases = (y == 1)[:, None]
    referred = p[:, None] >= thresholds
    true_positives = referred & cases
    false_positives = referred & ~cases

    tp = true_positives.sum(axis=0)
    fp = false_positives.sum(axis=0)

    model_nb = (tp - fp * threshold_odds) / n
    treat_all_nb = (events - (n - events) * threshold_odds) / n
    incremental_nb = model_nb - np.maximum(treat_all_nb, 0)

    # Reuse the same participant resamples across thresholds and strategies.
    sample_indices = rng.integers(0, n, size=(n_bootstrap, n))
    weights = np.stack([
        np.bincount(sample, minlength=n) for sample in sample_indices
    ])

    boot_tp = weights @ true_positives.astype(float)
    boot_fp = weights @ false_positives.astype(float)
    boot_events = (weights @ y)[:, None]

    boot_model_nb = (boot_tp - boot_fp * threshold_odds) / n
    boot_treat_all_nb = (
        boot_events - (n - boot_events) * threshold_odds
    ) / n
    boot_incremental_nb = boot_model_nb - np.maximum(boot_treat_all_nb, 0)

    model_low, model_high = np.quantile(
        boot_model_nb, [0.025, 0.975], axis=0
    )
    inc_low, inc_high = np.quantile(
        boot_incremental_nb, [0.025, 0.975], axis=0
    )

    cohort_results.append(pd.DataFrame({
        "Panel": panel,
        "Cohort": cohort,
        "Threshold_ID": threshold_ids,
        "Threshold_percent": 100 * thresholds,
        "Sensitivity": tp / events,
        "Specificity": 1 - fp / (n - events),
        "Model_NB": model_nb,
        "Model_NB_CI_Lower": model_low,
        "Model_NB_CI_Upper": model_high,
        "Treat_All_NB": treat_all_nb,
        "Treat_None_NB": 0.0,
        "Incremental_NB": incremental_nb,
        "Incremental_NB_CI_Lower": inc_low,
        "Incremental_NB_CI_Upper": inc_high,
        "Net_Reduction_Per_100_vs_Treat_All": (
            100 * (model_nb - treat_all_nb) / threshold_odds
        ),
        "Bootstrap_Probability_Model_Better": (
            boot_incremental_nb > 0
        ).mean(axis=0),
        "Bootstrap_Supported": inc_low > 0,
    }))

# Export the full plotting grid and the 14 rows reported in Table S17.
dca_full = pd.concat(cohort_results, ignore_index=True)
table_s17 = dca_full.loc[
    dca_full["Threshold_ID"].isin(table_threshold_ids)
].copy()

assert len(table_s17) == 14
dca_full.to_csv("Figure_S5_DCA_full_grid.csv", index=False)
table_s17.to_csv("Table_S17_DCA_bootstrap.csv", index=False)

# Use the same y-axis range for both figures.
plot_columns = [
    "Model_NB",
    "Treat_All_NB",
    "Treat_None_NB",
    "Model_NB_CI_Lower",
    "Model_NB_CI_Upper",
]
plot_values = dca_full[plot_columns].to_numpy()
y_min = plot_values.min()
y_max = plot_values.max()
padding = 0.05 * (y_max - y_min)
y_limits = (y_min - padding, y_max + padding)

panel_filenames = {
    "A": "Figure_S5A_DCA_M2_Heldout2024.png",
    "B": "Figure_S5B_DCA_M2_Temporal2025.png",
}

for data in cohort_results:
    panel = data["Panel"].iloc[0]
    cohort = data["Cohort"].iloc[0]
    x = data["Threshold_percent"].to_numpy()

    fig, ax = plt.subplots(figsize=(5.2, 4.0), constrained_layout=True)

    ax.fill_between(
        x,
        data["Model_NB_CI_Lower"].to_numpy(),
        data["Model_NB_CI_Upper"].to_numpy(),
        color="#0072B2",
        alpha=0.18,
        label="Pointwise 95% CI",
    )
    ax.plot(x, data["Model_NB"], color="#0072B2", label="M2 AdaBoost")
    ax.plot(
        x,
        data["Treat_All_NB"],
        color="#D55E00",
        linestyle="--",
        label="Treat-all",
    )
    ax.axhline(0, color="0.35", linestyle=":", label="Treat-none")

    ax.set(
        xlabel="Threshold probability (%)",
        ylabel="Net benefit",
        xlim=(0.5, 10.0),
        ylim=y_limits,
    )
    ax.set_title(
        f"Decision Curve Analysis — Primary MDD, {cohort}",
        fontsize=10,
        pad=8,
    )
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, fontsize=8)

    fig.savefig(panel_filenames[panel], dpi=300, facecolor="white")
    plt.show()
    plt.close(fig)

display(table_s17)

## PART 5: Integrated Model Comparison and Final Model Selection

### PART 5A: Integrated Performance Table

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 5A: INTEGRATED PERFORMANCE TABLE
# ============================================================
#
# Objectives:
# 1. Integrate model performance across:
#    - Genuine OOF 2024 development performance
#    - Held-out 2024 validation
#    - Held-out 2024 bootstrap 95% CI
#    - Temporal 2025 validation
#    - Temporal 2025 bootstrap 95% CI
# 2. Add screening burden information for all strategies.
# 3. Add selected calibration summaries as supporting evidence.
# 4. Do NOT select a winner in this part.
#
# Inputs:
# - screening_strategies
# - threshold_summary
# - heldout_results
# - part3c_bootstrap_results
# - temporal_results
# - part4b_bootstrap_results
#
# Optional supporting calibration inputs:
# - part3d_calibration_results
# - part4c1_calibration_results
# - part4c2_recalibration_results
#
# Outputs:
# - part5a_integrated_performance
# - part5a_integrated_performance_compact
# - part5a_strategy_summary
# - part5a_verification
#
# IMPORTANT:
# - No model refitting occurs.
# - No new predictions are generated.
# - No threshold is modified.
# - No recalibration model is fit.
# - No winner is selected in PART 5A.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 2. REQUIRED OBJECT VALIDATION
# ============================================================

required_objects = [
    "screening_strategies",
    "threshold_summary",
    "heldout_results",
    "part3c_bootstrap_results",
    "temporal_results",
    "part4b_bootstrap_results"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(
            missing_objects
        )
    )


for object_name in [
    "threshold_summary",
    "heldout_results",
    "part3c_bootstrap_results",
    "temporal_results",
    "part4b_bootstrap_results"
]:

    if not isinstance(
        globals()[
            object_name
        ],
        pd.DataFrame
    ):

        raise TypeError(
            f"{object_name} must be a pandas DataFrame."
        )


if not isinstance(
    screening_strategies,
    dict
):

    raise TypeError(
        "screening_strategies must be a dictionary."
    )


# ============================================================
# 3. EXPECTED STRATEGIES
# ============================================================

expected_strategy_order = [
    "M0_Admin",
    "M1_Admin_Behavior",
    "M2_Admin_2Q",
    "M3_Admin_2Q_Behavior",
    "M4_Conventional",
    "M5_Conventional_Admin",
    "M6_Full"
]


missing_strategy_definitions = [
    model_name
    for model_name in expected_strategy_order
    if model_name not in screening_strategies
]

if missing_strategy_definitions:

    raise KeyError(
        "Missing screening strategy definitions: "
        + ", ".join(
            missing_strategy_definitions
        )
    )


# ============================================================
# 4. HELPER: FORMAT ESTIMATE WITH CI
# ============================================================

def format_estimate_with_ci(
    estimate,
    lower,
    upper,
    digits=3
):
    """
    Format estimate and percentile 95% CI.
    """

    if (
        not np.isfinite(
            estimate
        )
        or
        not np.isfinite(
            lower
        )
        or
        not np.isfinite(
            upper
        )
    ):

        return "Not estimable"


    return (
        f"{estimate:.{digits}f} "
        f"({lower:.{digits}f}-{upper:.{digits}f})"
    )


# ============================================================
# 5. HELPER: EXTRACT BOOTSTRAP METRIC
# ============================================================

def extract_bootstrap_metric(
    bootstrap_table,
    model_name,
    algorithm_name,
    metric_name
):
    """
    Return point estimate and CI for one model-algorithm metric.
    """

    matching = bootstrap_table[
        (
            bootstrap_table[
                "Screening_Model"
            ]
            ==
            model_name
        )
        &
        (
            bootstrap_table[
                "Algorithm"
            ]
            ==
            algorithm_name
        )
        &
        (
            bootstrap_table[
                "Metric"
            ]
            ==
            metric_name
        )
    ]


    if len(
        matching
    ) != 1:

        return {
            "Point":
                np.nan,

            "CI_Lower":
                np.nan,

            "CI_Upper":
                np.nan,

            "Formatted":
                "Not estimable"
        }


    row = matching.iloc[
        0
    ]


    point = float(
        row[
            "Point_Estimate"
        ]
    )


    ci_lower = float(
        row[
            "CI_Lower"
        ]
    )


    ci_upper = float(
        row[
            "CI_Upper"
        ]
    )


    return {
        "Point":
            point,

        "CI_Lower":
            ci_lower,

        "CI_Upper":
            ci_upper,

        "Formatted":
            format_estimate_with_ci(
                estimate=point,
                lower=ci_lower,
                upper=ci_upper,
                digits=3
            )
    }


# ============================================================
# 6. HELPER: STRATEGY BURDEN METADATA
# ============================================================

def get_strategy_burden(
    model_name
):
    """
    Extract respondent-burden metadata from screening_strategies.
    """

    info = screening_strategies[
        model_name
    ]


    features = list(
        info[
            "features"
        ]
    )


    additional_questions = int(
        info.get(
            "additional_questions",
            info.get(
                "questions",
                0
            )
        )
    )


    return {
        "Strategy_Type":
            str(
                info.get(
                    "type",
                    ""
                )
            ),

        "Description":
            str(
                info.get(
                    "description",
                    ""
                )
            ),

        "Feature_Count":
            int(
                len(
                    features
                )
            ),

        "Self_Report_Items":
            additional_questions,

        "Features":
            ", ".join(
                features
            )
    }


# ============================================================
# 7. OPTIONAL CALIBRATION HELPERS
# ============================================================

def get_optional_calibration_row(
    object_name,
    model_name,
    algorithm_name
):
    """
    Retrieve one optional calibration row if the object exists.
    """

    if object_name not in globals():

        return None


    table = globals()[
        object_name
    ]


    if not isinstance(
        table,
        pd.DataFrame
    ):

        return None


    if not {
        "Screening_Model",
        "Algorithm"
    }.issubset(
        table.columns
    ):

        return None


    matching = table[
        (
            table[
                "Screening_Model"
            ]
            ==
            model_name
        )
        &
        (
            table[
                "Algorithm"
            ]
            ==
            algorithm_name
        )
    ]


    if len(
        matching
    ) != 1:

        return None


    return matching.iloc[
        0
    ]


# ============================================================
# 8. IDENTIFY MODEL-ALGORITHM PAIRS
# ============================================================

threshold_pairs = (
    threshold_summary[
        [
            "Screening_Model",
            "Algorithm"
        ]
    ]
    .drop_duplicates()
    .copy()
)


pair_counts = (
    threshold_pairs
    .groupby(
        "Screening_Model"
    )
    .size()
)


unexpected_pair_counts = (
    pair_counts[
        pair_counts
        !=
        5
    ]
)


if len(
    unexpected_pair_counts
) > 0:

    raise RuntimeError(
        "Expected exactly 5 algorithms per screening strategy "
        "in threshold_summary."
    )


expected_pair_count = int(
    len(
        threshold_pairs
    )
)


if expected_pair_count != 35:

    raise RuntimeError(
        f"Expected 35 model-algorithm pairs, observed "
        f"{expected_pair_count}."
    )


# ============================================================
# 9. BUILD INTEGRATED TABLE
# ============================================================

integrated_rows = []


for model_name in expected_strategy_order:

    burden = get_strategy_burden(
        model_name
    )


    model_pairs = (
        threshold_pairs[
            threshold_pairs[
                "Screening_Model"
            ]
            ==
            model_name
        ]
        .copy()
    )


    for algorithm_name in (
        model_pairs[
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    ):

        # ----------------------------------------------------
        # OOF row
        # ----------------------------------------------------

        oof_rows = threshold_summary[
            (
                threshold_summary[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                threshold_summary[
                    "Algorithm"
                ]
                ==
                algorithm_name
            )
        ]


        if len(
            oof_rows
        ) != 1:

            raise RuntimeError(
                f"{model_name} | {algorithm_name}: expected one "
                "threshold/OOF row."
            )


        oof_row = oof_rows.iloc[
            0
        ]


        # ----------------------------------------------------
        # Held-out row
        # ----------------------------------------------------

        heldout_rows = heldout_results[
            (
                heldout_results[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                heldout_results[
                    "Algorithm"
                ]
                ==
                algorithm_name
            )
        ]


        if len(
            heldout_rows
        ) != 1:

            raise RuntimeError(
                f"{model_name} | {algorithm_name}: expected one "
                "held-out result row."
            )


        heldout_row = heldout_rows.iloc[
            0
        ]


        # ----------------------------------------------------
        # Temporal row
        # ----------------------------------------------------

        temporal_rows = temporal_results[
            (
                temporal_results[
                    "Screening_Model"
                ]
                ==
                model_name
            )
            &
            (
                temporal_results[
                    "Algorithm"
                ]
                ==
                algorithm_name
            )
        ]


        if len(
            temporal_rows
        ) != 1:

            raise RuntimeError(
                f"{model_name} | {algorithm_name}: expected one "
                "temporal result row."
            )


        temporal_row = temporal_rows.iloc[
            0
        ]


        # ----------------------------------------------------
        # Bootstrap summaries
        # ----------------------------------------------------

        heldout_auc_ci = extract_bootstrap_metric(
            bootstrap_table=part3c_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="AUC"
        )


        heldout_ap_ci = extract_bootstrap_metric(
            bootstrap_table=part3c_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="Average_Precision"
        )


        heldout_sensitivity_ci = extract_bootstrap_metric(
            bootstrap_table=part3c_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="Sensitivity"
        )


        heldout_specificity_ci = extract_bootstrap_metric(
            bootstrap_table=part3c_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="Specificity"
        )


        heldout_precision_ci = extract_bootstrap_metric(
            bootstrap_table=part3c_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="Precision"
        )


        heldout_f1_ci = extract_bootstrap_metric(
            bootstrap_table=part3c_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="F1"
        )


        temporal_auc_ci = extract_bootstrap_metric(
            bootstrap_table=part4b_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="AUC"
        )


        temporal_ap_ci = extract_bootstrap_metric(
            bootstrap_table=part4b_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="Average_Precision"
        )


        temporal_sensitivity_ci = extract_bootstrap_metric(
            bootstrap_table=part4b_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="Sensitivity"
        )


        temporal_specificity_ci = extract_bootstrap_metric(
            bootstrap_table=part4b_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="Specificity"
        )


        temporal_precision_ci = extract_bootstrap_metric(
            bootstrap_table=part4b_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="Precision"
        )


        temporal_f1_ci = extract_bootstrap_metric(
            bootstrap_table=part4b_bootstrap_results,
            model_name=model_name,
            algorithm_name=algorithm_name,
            metric_name="F1"
        )


        # ----------------------------------------------------
        # Optional calibration support
        # ----------------------------------------------------

        heldout_calibration = get_optional_calibration_row(
            object_name="part3d_calibration_results",
            model_name=model_name,
            algorithm_name=algorithm_name
        )


        temporal_raw_calibration = get_optional_calibration_row(
            object_name="part4c1_calibration_results",
            model_name=model_name,
            algorithm_name=algorithm_name
        )


        temporal_recalibration = get_optional_calibration_row(
            object_name="part4c2_recalibration_results",
            model_name=model_name,
            algorithm_name=algorithm_name
        )


        row = {

            "Screening_Model":
                model_name,

            "Strategy_Type":
                burden[
                    "Strategy_Type"
                ],

            "Description":
                burden[
                    "Description"
                ],

            "Feature_Count":
                burden[
                    "Feature_Count"
                ],

            "Self_Report_Items":
                burden[
                    "Self_Report_Items"
                ],

            "Algorithm":
                algorithm_name,

            "Locked_Threshold":
                float(
                    oof_row[
                        "Threshold"
                    ]
                    if "Threshold"
                    in oof_row.index
                    else oof_row[
                        "Locked_Threshold"
                    ]
                ),

            # OOF 2024
            "OOF_AUC":
                float(
                    oof_row[
                        "OOF_AUC"
                    ]
                ),

            "OOF_Average_Precision":
                float(
                    oof_row[
                        "OOF_Average_Precision"
                    ]
                ),

            "OOF_Sensitivity":
                float(
                    oof_row[
                        "OOF_Sensitivity"
                    ]
                ),

            "OOF_Specificity":
                float(
                    oof_row[
                        "OOF_Specificity"
                    ]
                ),

            "OOF_Precision":
                float(
                    oof_row[
                        "OOF_Precision"
                    ]
                ),

            "OOF_F1":
                float(
                    oof_row[
                        "OOF_F1"
                    ]
                ),

            "Threshold_Rule":
                str(
                    oof_row[
                        "Threshold_Rule"
                    ]
                ),

            # Held-out 2024
            "Heldout_N":
                int(
                    heldout_row[
                        "N"
                    ]
                ),

            "Heldout_Events":
                int(
                    heldout_row[
                        "Events"
                    ]
                ),

            "Heldout_Prevalence":
                float(
                    heldout_row[
                        "Prevalence"
                    ]
                ),

            "Heldout_AUC":
                float(
                    heldout_row[
                        "AUC"
                    ]
                ),

            "Heldout_AUC_95CI":
                heldout_auc_ci[
                    "Formatted"
                ],

            "Heldout_Average_Precision":
                float(
                    heldout_row[
                        "Average_Precision"
                    ]
                ),

            "Heldout_Average_Precision_95CI":
                heldout_ap_ci[
                    "Formatted"
                ],

            "Heldout_Sensitivity":
                float(
                    heldout_row[
                        "Sensitivity"
                    ]
                ),

            "Heldout_Sensitivity_95CI":
                heldout_sensitivity_ci[
                    "Formatted"
                ],

            "Heldout_Specificity":
                float(
                    heldout_row[
                        "Specificity"
                    ]
                ),

            "Heldout_Specificity_95CI":
                heldout_specificity_ci[
                    "Formatted"
                ],

            "Heldout_Precision":
                float(
                    heldout_row[
                        "Precision"
                    ]
                ),

            "Heldout_Precision_95CI":
                heldout_precision_ci[
                    "Formatted"
                ],

            "Heldout_F1":
                float(
                    heldout_row[
                        "F1"
                    ]
                ),

            "Heldout_F1_95CI":
                heldout_f1_ci[
                    "Formatted"
                ],

            "Heldout_Brier_Score":
                float(
                    heldout_row[
                        "Brier_Score"
                    ]
                ),

            "Heldout_TP":
                int(
                    heldout_row[
                        "TP"
                    ]
                ),

            "Heldout_FP":
                int(
                    heldout_row[
                        "FP"
                    ]
                ),

            "Heldout_FN":
                int(
                    heldout_row[
                        "FN"
                    ]
                ),

            "Heldout_TN":
                int(
                    heldout_row[
                        "TN"
                    ]
                ),

            # Temporal 2025
            "Temporal_N":
                int(
                    temporal_row[
                        "N"
                    ]
                ),

            "Temporal_Events":
                int(
                    temporal_row[
                        "Events"
                    ]
                ),

            "Temporal_Prevalence":
                float(
                    temporal_row[
                        "Prevalence"
                    ]
                ),

            "Temporal_AUC":
                float(
                    temporal_row[
                        "AUC"
                    ]
                ),

            "Temporal_AUC_95CI":
                temporal_auc_ci[
                    "Formatted"
                ],

            "Temporal_Average_Precision":
                float(
                    temporal_row[
                        "Average_Precision"
                    ]
                ),

            "Temporal_Average_Precision_95CI":
                temporal_ap_ci[
                    "Formatted"
                ],

            "Temporal_Sensitivity":
                float(
                    temporal_row[
                        "Sensitivity"
                    ]
                ),

            "Temporal_Sensitivity_95CI":
                temporal_sensitivity_ci[
                    "Formatted"
                ],

            "Temporal_Specificity":
                float(
                    temporal_row[
                        "Specificity"
                    ]
                ),

            "Temporal_Specificity_95CI":
                temporal_specificity_ci[
                    "Formatted"
                ],

            "Temporal_Precision":
                float(
                    temporal_row[
                        "Precision"
                    ]
                ),

            "Temporal_Precision_95CI":
                temporal_precision_ci[
                    "Formatted"
                ],

            "Temporal_F1":
                float(
                    temporal_row[
                        "F1"
                    ]
                ),

            "Temporal_F1_95CI":
                temporal_f1_ci[
                    "Formatted"
                ],

            "Temporal_Brier_Score":
                float(
                    temporal_row[
                        "Brier_Score"
                    ]
                ),

            "Temporal_TP":
                int(
                    temporal_row[
                        "TP"
                    ]
                ),

            "Temporal_FP":
                int(
                    temporal_row[
                        "FP"
                    ]
                ),

            "Temporal_FN":
                int(
                    temporal_row[
                        "FN"
                    ]
                ),

            "Temporal_TN":
                int(
                    temporal_row[
                        "TN"
                    ]
                ),

            # Stability / change
            "AUC_Change_Temporal_minus_Heldout":
                float(
                    temporal_row[
                        "AUC"
                    ]
                    -
                    heldout_row[
                        "AUC"
                    ]
                ),

            "Sensitivity_Change_Temporal_minus_Heldout":
                float(
                    temporal_row[
                        "Sensitivity"
                    ]
                    -
                    heldout_row[
                        "Sensitivity"
                    ]
                ),

            "Specificity_Change_Temporal_minus_Heldout":
                float(
                    temporal_row[
                        "Specificity"
                    ]
                    -
                    heldout_row[
                        "Specificity"
                    ]
                ),

            "Precision_Change_Temporal_minus_Heldout":
                float(
                    temporal_row[
                        "Precision"
                    ]
                    -
                    heldout_row[
                        "Precision"
                    ]
                ),

            "F1_Change_Temporal_minus_Heldout":
                float(
                    temporal_row[
                        "F1"
                    ]
                    -
                    heldout_row[
                        "F1"
                    ]
                ),

            # Optional supporting calibration placeholders
            "Heldout_Calibration_Intercept":
                np.nan,

            "Heldout_Calibration_Slope":
                np.nan,

            "Heldout_OE_Ratio":
                np.nan,

            "Temporal_Raw_Calibration_Intercept":
                np.nan,

            "Temporal_Raw_Calibration_Slope":
                np.nan,

            "Temporal_Raw_OE_Ratio":
                np.nan,

            "Temporal_Recalibrated_Brier":
                np.nan,

            "Temporal_Recalibrated_OE_Ratio":
                np.nan,

            "Temporal_Recalibrated_Calibration_Intercept":
                np.nan,

            "Temporal_Recalibrated_Calibration_Slope":
                np.nan,

            "Recalibration_Available":
                bool(
                    temporal_recalibration
                    is not None
                ),

            "Features":
                burden[
                    "Features"
                ]
        }


        if heldout_calibration is not None:

            row[
                "Heldout_Calibration_Intercept"
            ] = float(
                heldout_calibration[
                    "Calibration_Intercept"
                ]
            )


            row[
                "Heldout_Calibration_Slope"
            ] = float(
                heldout_calibration[
                    "Calibration_Slope"
                ]
            )


            if "Observed_to_Expected_Ratio" in (
                heldout_calibration.index
            ):

                row[
                    "Heldout_OE_Ratio"
                ] = float(
                    heldout_calibration[
                        "Observed_to_Expected_Ratio"
                    ]
                )


        if temporal_raw_calibration is not None:

            row[
                "Temporal_Raw_Calibration_Intercept"
            ] = float(
                temporal_raw_calibration[
                    "Calibration_Intercept"
                ]
            )


            row[
                "Temporal_Raw_Calibration_Slope"
            ] = float(
                temporal_raw_calibration[
                    "Calibration_Slope"
                ]
            )


            row[
                "Temporal_Raw_OE_Ratio"
            ] = float(
                temporal_raw_calibration[
                    "Observed_to_Expected_Ratio"
                ]
            )


        if temporal_recalibration is not None:

            row[
                "Temporal_Recalibrated_Brier"
            ] = float(
                temporal_recalibration[
                    "Recalibrated_Brier_Score"
                ]
            )


            row[
                "Temporal_Recalibrated_OE_Ratio"
            ] = float(
                temporal_recalibration[
                    "Recalibrated_Observed_to_Expected_Ratio"
                ]
            )


            row[
                "Temporal_Recalibrated_Calibration_Intercept"
            ] = float(
                temporal_recalibration[
                    "Recalibrated_Calibration_Intercept"
                ]
            )


            row[
                "Temporal_Recalibrated_Calibration_Slope"
            ] = float(
                temporal_recalibration[
                    "Recalibrated_Calibration_Slope"
                ]
            )


        integrated_rows.append(
            row
        )


part5a_integrated_performance = pd.DataFrame(
    integrated_rows
)


# ============================================================
# 10. STRATEGY ORDER / SORTING
# ============================================================

part5a_integrated_performance[
    "Strategy_Order"
] = part5a_integrated_performance[
    "Screening_Model"
].map(
    {
        model_name:
            index
        for index, model_name
        in enumerate(
            expected_strategy_order
        )
    }
)


part5a_integrated_performance = (
    part5a_integrated_performance
    .sort_values(
        by=[
            "Strategy_Order",
            "Temporal_AUC",
            "Temporal_F1"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 11. COMPACT PUBLICATION TABLE
# ============================================================

part5a_integrated_performance_compact = (
    part5a_integrated_performance[
        [
            "Screening_Model",
            "Self_Report_Items",
            "Feature_Count",
            "Algorithm",
            "OOF_AUC",
            "OOF_F1",
            "Heldout_AUC_95CI",
            "Heldout_Sensitivity_95CI",
            "Heldout_Specificity_95CI",
            "Heldout_Precision_95CI",
            "Heldout_F1_95CI",
            "Temporal_AUC_95CI",
            "Temporal_Sensitivity_95CI",
            "Temporal_Specificity_95CI",
            "Temporal_Precision_95CI",
            "Temporal_F1_95CI",
            "AUC_Change_Temporal_minus_Heldout",
            "F1_Change_Temporal_minus_Heldout",
            "Temporal_Raw_OE_Ratio",
            "Temporal_Recalibrated_OE_Ratio"
        ]
    ]
    .copy()
)


# ============================================================
# 12. STRATEGY-LEVEL SUMMARY
# ============================================================
#
# This is descriptive only.
# It does NOT select a winner.
# ============================================================

strategy_summary_rows = []


for model_name in expected_strategy_order:

    subset = (
        part5a_integrated_performance[
            part5a_integrated_performance[
                "Screening_Model"
            ]
            ==
            model_name
        ]
        .copy()
    )


    burden = get_strategy_burden(
        model_name
    )


    strategy_summary_rows.append({

        "Screening_Model":
            model_name,

        "Strategy_Type":
            burden[
                "Strategy_Type"
            ],

        "Feature_Count":
            burden[
                "Feature_Count"
            ],

        "Self_Report_Items":
            burden[
                "Self_Report_Items"
            ],

        "Algorithms_Evaluated":
            int(
                len(
                    subset
                )
            ),

        "Best_OOF_AUC":
            float(
                subset[
                    "OOF_AUC"
                ].max()
            ),

        "Best_Heldout_AUC":
            float(
                subset[
                    "Heldout_AUC"
                ].max()
            ),

        "Best_Temporal_AUC":
            float(
                subset[
                    "Temporal_AUC"
                ].max()
            ),

        "Best_Temporal_F1":
            float(
                subset[
                    "Temporal_F1"
                ].max()
            ),

        "Best_Temporal_Sensitivity":
            float(
                subset[
                    "Temporal_Sensitivity"
                ].max()
            ),

        "Best_Temporal_Specificity":
            float(
                subset[
                    "Temporal_Specificity"
                ].max()
            ),

        "Best_Temporal_Precision":
            float(
                subset[
                    "Temporal_Precision"
                ].max()
            ),

        "Median_Temporal_AUC":
            float(
                subset[
                    "Temporal_AUC"
                ].median()
            ),

        "Median_Absolute_AUC_Change_Heldout_to_Temporal":
            float(
                np.median(
                    np.abs(
                        subset[
                            "AUC_Change_Temporal_minus_Heldout"
                        ]
                    )
                )
            )
    })


part5a_strategy_summary = pd.DataFrame(
    strategy_summary_rows
)


# ============================================================
# 13. VERIFICATION
# ============================================================

verification_rows = []


for model_name in expected_strategy_order:

    subset = (
        part5a_integrated_performance[
            part5a_integrated_performance[
                "Screening_Model"
            ]
            ==
            model_name
        ]
    )


    expected_algorithms = set(
        threshold_summary.loc[
            threshold_summary[
                "Screening_Model"
            ]
            ==
            model_name,
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )


    observed_algorithms = set(
        subset[
            "Algorithm"
        ]
        .astype(str)
        .tolist()
    )


    missing_algorithms = sorted(
        expected_algorithms
        -
        observed_algorithms
    )


    all_core_values_finite = bool(
        np.isfinite(
            subset[
                [
                    "OOF_AUC",
                    "Heldout_AUC",
                    "Temporal_AUC",
                    "Heldout_Sensitivity",
                    "Temporal_Sensitivity",
                    "Heldout_Specificity",
                    "Temporal_Specificity",
                    "Heldout_F1",
                    "Temporal_F1"
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )
        .all()
    )


    all_ci_present = bool(
        (
            subset[
                "Heldout_AUC_95CI"
            ]
            !=
            "Not estimable"
        ).all()
        and
        (
            subset[
                "Temporal_AUC_95CI"
            ]
            !=
            "Not estimable"
        ).all()
        and
        (
            subset[
                "Heldout_Sensitivity_95CI"
            ]
            !=
            "Not estimable"
        ).all()
        and
        (
            subset[
                "Temporal_Sensitivity_95CI"
            ]
            !=
            "Not estimable"
        ).all()
    )


    burden_consistent = bool(
        subset[
            "Self_Report_Items"
        ].nunique()
        ==
        1
        and
        subset[
            "Feature_Count"
        ].nunique()
        ==
        1
    )


    verification_rows.append({

        "Screening_Model":
            model_name,

        "Expected_Algorithms":
            int(
                len(
                    expected_algorithms
                )
            ),

        "Observed_Algorithms":
            int(
                len(
                    observed_algorithms
                )
            ),

        "Missing_Algorithms":
            ", ".join(
                missing_algorithms
            ),

        "All_Core_Values_Finite":
            all_core_values_finite,

        "Required_95CI_Present":
            all_ci_present,

        "Burden_Metadata_Consistent":
            burden_consistent,

        "Integrated_Table_Ready":
            (
                len(
                    missing_algorithms
                )
                ==
                0
                and
                len(
                    observed_algorithms
                )
                ==
                5
                and
                all_core_values_finite
                and
                all_ci_present
                and
                burden_consistent
            )
    })


part5a_verification = pd.DataFrame(
    verification_rows
)


# ============================================================
# 14. DISPLAY
# ============================================================

print("=" * 100)
print("PART 5A VERSION 3.0: INTEGRATED PERFORMANCE TABLE")
print("=" * 100)


print(
    f"Integrated model-algorithm pairs: "
    f"{len(part5a_integrated_performance)}"
)


print("\nIntegrated compact table:")


display(
    part5a_integrated_performance_compact
    .round(
        4
    )
)


print("\n" + "=" * 100)
print("STRATEGY-LEVEL DESCRIPTIVE SUMMARY")
print("=" * 100)


display(
    part5a_strategy_summary
    .round(
        4
    )
)


print("\n" + "=" * 100)
print("PART 5A VERIFICATION")
print("=" * 100)


display(
    part5a_verification
)


# ============================================================
# 15. HARD SAFETY CHECKS
# ============================================================

if len(
    part5a_integrated_performance
) != 35:

    raise RuntimeError(
        "Integrated performance table does not contain "
        "exactly 35 model-algorithm pairs."
    )


if not part5a_verification[
    "Integrated_Table_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies failed PART 5A "
        "integration verification."
    )


# ============================================================
# 16. EXPORT
# ============================================================

part5a_integrated_performance.to_csv(
    "part5a_integrated_performance.csv",
    index=False,
    encoding="utf-8-sig"
)


part5a_integrated_performance_compact.to_csv(
    "part5a_integrated_performance_compact.csv",
    index=False,
    encoding="utf-8-sig"
)


part5a_strategy_summary.to_csv(
    "part5a_strategy_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


part5a_verification.to_csv(
    "part5a_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 17. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 5A VERSION 3.0 COMPLETED")
print("=" * 100)


print(
    f"Expected model-algorithm pairs : 35"
)


print(
    f"Integrated model-algorithm pairs: "
    f"{len(part5a_integrated_performance)}"
)


print(
    f"Screening strategies integrated : "
    f"{part5a_integrated_performance['Screening_Model'].nunique()}"
)


print("\nArtifacts created:")

print("- part5a_integrated_performance")
print("- part5a_integrated_performance_compact")
print("- part5a_strategy_summary")
print("- part5a_verification")


print("\nExported files:")

print("- part5a_integrated_performance.csv")
print("- part5a_integrated_performance_compact.csv")
print("- part5a_strategy_summary.csv")
print("- part5a_verification.csv")


print(
    "\nNo model was refitted, no prediction was regenerated, "
    "no threshold was modified, and no winner was selected "
    "in PART 5A."
)


print(
    "Proceed to PART 5B only after "
    "Integrated_Table_Ready=True for all 7 strategies."
)

### PART 5B: Model Selection Framework

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 5B: MODEL SELECTION FRAMEWORK
# ============================================================
#
# Purpose:
# Build a transparent, non-weighted selection framework while
# preserving the temporal 2025 cohort primarily as validation.
#
# Key design:
#
# STAGE 1 — Representative algorithm selection WITHIN each
# screening strategy uses ONLY genuine OOF 2024 development
# performance.
#
# Hierarchical ranking:
#   1. Target recall achieved on genuine OOF predictions
#   2. Non-degenerate OOF probability distribution preferred
#   3. Higher OOF AUC
#   4. Higher OOF Average Precision
#   5. Higher OOF F1
#   6. Higher OOF Specificity
#
# This yields ONE prespecified representative algorithm per
# screening strategy before comparing validation performance.
#
# STAGE 2 — Held-out 2024 and Temporal 2025 are used to assess
# external/internal validation support, NOT to re-select the
# algorithm within each screening strategy.
#
# STAGE 3 — Compare the 7 strategy representatives on:
#   - discrimination
#   - fixed-threshold screening sensitivity
#   - specificity / precision / F1
#   - held-out → temporal stability
#   - respondent burden
#   - calibration as supporting evidence
#
# No arbitrary weighted composite score is created.
# No final winner is declared in PART 5B.
#
# Input:
# - part5a_integrated_performance
# - part5a_verification
# - threshold_summary
#
# Output:
# - part5b_algorithm_ranking
# - part5b_strategy_representatives
# - part5b_validation_support
# - part5b_pareto_comparison
# - part5b_selection_framework
# - part5b_verification
#
# IMPORTANT:
# - No refit
# - No new predictions
# - No threshold modification
# - No recalibration fitting
# - No final winner selection
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 2. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "part5a_integrated_performance",
    "part5a_verification",
    "threshold_summary"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(
            missing_objects
        )
    )


if not isinstance(
    part5a_integrated_performance,
    pd.DataFrame
):

    raise TypeError(
        "part5a_integrated_performance must be a DataFrame."
    )


if not isinstance(
    part5a_verification,
    pd.DataFrame
):

    raise TypeError(
        "part5a_verification must be a DataFrame."
    )


if not isinstance(
    threshold_summary,
    pd.DataFrame
):

    raise TypeError(
        "threshold_summary must be a DataFrame."
    )


if not part5a_verification[
    "Integrated_Table_Ready"
].all():

    raise RuntimeError(
        "PART 5A is not fully Integrated_Table_Ready."
    )


# ============================================================
# 3. STRATEGY ORDER
# ============================================================

STRATEGY_ORDER = [
    "M0_Admin",
    "M1_Admin_Behavior",
    "M2_Admin_2Q",
    "M3_Admin_2Q_Behavior",
    "M4_Conventional",
    "M5_Conventional_Admin",
    "M6_Full"
]


# ============================================================
# 4. HELPER: GET TARGET RECALL FLAG
# ============================================================

def get_oof_target_recall_flag(
    model_name,
    algorithm_name
):
    """
    Read the OOF recall-target status from threshold_summary.
    """

    rows = threshold_summary[
        (
            threshold_summary[
                "Screening_Model"
            ]
            ==
            model_name
        )
        &
        (
            threshold_summary[
                "Algorithm"
            ]
            ==
            algorithm_name
        )
    ]


    if len(
        rows
    ) != 1:

        raise RuntimeError(
            f"{model_name} | {algorithm_name}: expected exactly "
            "one threshold_summary row."
        )


    row = rows.iloc[
        0
    ]


    if "Target_Recall_Achieved" in row.index:

        target_recall_achieved = bool(
            row[
                "Target_Recall_Achieved"
            ]
        )


    elif "Rule" in row.index:

        target_recall_achieved = bool(
            str(
                row[
                    "Rule"
                ]
            )
            !=
            "Max_F1"
        )


    else:

        target_recall_achieved = True


    if "Probability_Degenerate_Flag" in row.index:

        degenerate_probability_flag = bool(
            row[
                "Probability_Degenerate_Flag"
            ]
        )


    elif "Probability_Degenerate_Flag_OOF" in row.index:

        degenerate_probability_flag = bool(
            row[
                "Probability_Degenerate_Flag_OOF"
            ]
        )


    else:

        degenerate_probability_flag = False


    return (
        target_recall_achieved,
        degenerate_probability_flag
    )


# ============================================================
# 5. STAGE 1 — RANK ALGORITHMS USING OOF 2024 ONLY
# ============================================================

algorithm_ranking_rows = []


for model_name in STRATEGY_ORDER:

    strategy_table = (
        part5a_integrated_performance[
            part5a_integrated_performance[
                "Screening_Model"
            ]
            ==
            model_name
        ]
        .copy()
    )


    if len(
        strategy_table
    ) != 5:

        raise RuntimeError(
            f"{model_name}: expected 5 algorithms, observed "
            f"{len(strategy_table)}."
        )


    target_recall_flags = []

    degenerate_flags = []


    for _, row in strategy_table.iterrows():

        (
            target_recall_achieved,
            degenerate_probability_flag
        ) = get_oof_target_recall_flag(

            model_name=model_name,

            algorithm_name=str(
                row[
                    "Algorithm"
                ]
            )
        )


        target_recall_flags.append(
            target_recall_achieved
        )


        degenerate_flags.append(
            degenerate_probability_flag
        )


    strategy_table[
        "OOF_Target_Recall_Achieved"
    ] = target_recall_flags


    strategy_table[
        "OOF_Probability_Degenerate_Flag"
    ] = degenerate_flags


    # --------------------------------------------------------
    # Hierarchical ranking.
    #
    # IMPORTANT:
    # Validation metrics are intentionally NOT included here.
    # --------------------------------------------------------

    strategy_table[
        "Target_Recall_Rank_Key"
    ] = (
        strategy_table[
            "OOF_Target_Recall_Achieved"
        ]
        .astype(int)
    )


    strategy_table[
        "Nondegenerate_Rank_Key"
    ] = (
        ~strategy_table[
            "OOF_Probability_Degenerate_Flag"
        ]
    ).astype(int)


    strategy_table = (
        strategy_table
        .sort_values(

            by=[
                "Target_Recall_Rank_Key",
                "Nondegenerate_Rank_Key",
                "OOF_AUC",
                "OOF_Average_Precision",
                "OOF_F1",
                "OOF_Specificity",
                "Algorithm"
            ],

            ascending=[
                False,
                False,
                False,
                False,
                False,
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )


    strategy_table[
        "OOF_Selection_Rank"
    ] = np.arange(
        1,
        len(
            strategy_table
        )
        +
        1
    )


    strategy_table[
        "Selected_Strategy_Representative"
    ] = (
        strategy_table[
            "OOF_Selection_Rank"
        ]
        ==
        1
    )


    for _, row in strategy_table.iterrows():

        algorithm_ranking_rows.append({

            "Screening_Model":
                model_name,

            "Self_Report_Items":
                int(
                    row[
                        "Self_Report_Items"
                    ]
                ),

            "Feature_Count":
                int(
                    row[
                        "Feature_Count"
                    ]
                ),

            "Algorithm":
                str(
                    row[
                        "Algorithm"
                    ]
                ),

            "OOF_Target_Recall_Achieved":
                bool(
                    row[
                        "OOF_Target_Recall_Achieved"
                    ]
                ),

            "OOF_Probability_Degenerate_Flag":
                bool(
                    row[
                        "OOF_Probability_Degenerate_Flag"
                    ]
                ),

            "OOF_AUC":
                float(
                    row[
                        "OOF_AUC"
                    ]
                ),

            "OOF_Average_Precision":
                float(
                    row[
                        "OOF_Average_Precision"
                    ]
                ),

            "OOF_F1":
                float(
                    row[
                        "OOF_F1"
                    ]
                ),

            "OOF_Sensitivity":
                float(
                    row[
                        "OOF_Sensitivity"
                    ]
                ),

            "OOF_Specificity":
                float(
                    row[
                        "OOF_Specificity"
                    ]
                ),

            "OOF_Selection_Rank":
                int(
                    row[
                        "OOF_Selection_Rank"
                    ]
                ),

            "Selected_Strategy_Representative":
                bool(
                    row[
                        "Selected_Strategy_Representative"
                    ]
                ),

            "Selection_Basis":
                (
                    "OOF-only hierarchical ranking: "
                    "target recall → nondegenerate probability → "
                    "AUC → AP → F1 → specificity"
                )
        })


part5b_algorithm_ranking = pd.DataFrame(
    algorithm_ranking_rows
)


# ============================================================
# 6. CREATE ONE REPRESENTATIVE PER STRATEGY
# ============================================================

selected_pairs = (
    part5b_algorithm_ranking[
        part5b_algorithm_ranking[
            "Selected_Strategy_Representative"
        ]
    ]
    [
        [
            "Screening_Model",
            "Algorithm"
        ]
    ]
    .copy()
)


part5b_strategy_representatives = (
    part5a_integrated_performance
    .merge(

        selected_pairs,

        on=[
            "Screening_Model",
            "Algorithm"
        ],

        how="inner",

        validate="one_to_one"
    )
    .copy()
)


part5b_strategy_representatives[
    "Strategy_Order"
] = (
    part5b_strategy_representatives[
        "Screening_Model"
    ]
    .map(
        {
            model_name:
                index
            for index, model_name
            in enumerate(
                STRATEGY_ORDER
            )
        }
    )
)


part5b_strategy_representatives = (
    part5b_strategy_representatives
    .sort_values(
        "Strategy_Order"
    )
    .reset_index(
        drop=True
    )
)


if len(
    part5b_strategy_representatives
) != 7:

    raise RuntimeError(
        "Expected exactly one OOF-selected representative for "
        "each of 7 screening strategies."
    )


# ============================================================
# 7. STAGE 2 — VALIDATION SUPPORT FLAGS
# ============================================================
#
# These flags DESCRIBE validation support.
# They do not re-select the within-strategy algorithm.
# ============================================================

validation_support_rows = []


for _, row in (
    part5b_strategy_representatives.iterrows()
):

    # --------------------------------------------------------
    # AUC lower confidence bound above chance
    # --------------------------------------------------------

    heldout_auc_ci_text = str(
        row[
            "Heldout_AUC_95CI"
        ]
    )


    temporal_auc_ci_text = str(
        row[
            "Temporal_AUC_95CI"
        ]
    )


    # Numeric CI values are retrieved from bootstrap tables
    # indirectly from formatted strings only if necessary.
    # We instead use robust parsing of the lower bound here.
    def parse_lower_ci(
        formatted_text
    ):

        try:

            inner = (
                formatted_text
                .split(
                    "("
                )[
                    1
                ]
                .split(
                    ")"
                )[
                    0
                ]
            )


            lower = float(
                inner
                .split(
                    "-"
                )[
                    0
                ]
            )


            return lower


        except Exception:

            return np.nan


    heldout_auc_lower = parse_lower_ci(
        heldout_auc_ci_text
    )


    temporal_auc_lower = parse_lower_ci(
        temporal_auc_ci_text
    )


    heldout_discrimination_supported = bool(
        np.isfinite(
            heldout_auc_lower
        )
        and
        heldout_auc_lower
        >
        0.50
    )


    temporal_discrimination_supported = bool(
        np.isfinite(
            temporal_auc_lower
        )
        and
        temporal_auc_lower
        >
        0.50
    )


    # Threshold target was prespecified in PART 3A.
    # Use the OOF achieved sensitivity as the benchmark rather
    # than inventing a new temporal cutoff.
    temporal_sensitivity_not_below_oof_target = bool(
        row[
            "Temporal_Sensitivity"
        ]
        >=
        0.50
    )


    heldout_sensitivity_not_below_oof_target = bool(
        row[
            "Heldout_Sensitivity"
        ]
        >=
        0.50
    )


    validation_support_rows.append({

        "Screening_Model":
            row[
                "Screening_Model"
            ],

        "Algorithm":
            row[
                "Algorithm"
            ],

        "Self_Report_Items":
            int(
                row[
                    "Self_Report_Items"
                ]
            ),

        "OOF_AUC":
            float(
                row[
                    "OOF_AUC"
                ]
            ),

        "Heldout_AUC":
            float(
                row[
                    "Heldout_AUC"
                ]
            ),

        "Heldout_AUC_95CI":
            row[
                "Heldout_AUC_95CI"
            ],

        "Temporal_AUC":
            float(
                row[
                    "Temporal_AUC"
                ]
            ),

        "Temporal_AUC_95CI":
            row[
                "Temporal_AUC_95CI"
            ],

        "Heldout_AUC_Lower_95CI":
            heldout_auc_lower,

        "Temporal_AUC_Lower_95CI":
            temporal_auc_lower,

        "Heldout_Discrimination_CI_Above_Chance":
            heldout_discrimination_supported,

        "Temporal_Discrimination_CI_Above_Chance":
            temporal_discrimination_supported,

        "Heldout_Sensitivity":
            float(
                row[
                    "Heldout_Sensitivity"
                ]
            ),

        "Temporal_Sensitivity":
            float(
                row[
                    "Temporal_Sensitivity"
                ]
            ),

        "Heldout_Sensitivity_At_Least_0_50":
            heldout_sensitivity_not_below_oof_target,

        "Temporal_Sensitivity_At_Least_0_50":
            temporal_sensitivity_not_below_oof_target,

        "Heldout_Specificity":
            float(
                row[
                    "Heldout_Specificity"
                ]
            ),

        "Temporal_Specificity":
            float(
                row[
                    "Temporal_Specificity"
                ]
            ),

        "Heldout_Precision":
            float(
                row[
                    "Heldout_Precision"
                ]
            ),

        "Temporal_Precision":
            float(
                row[
                    "Temporal_Precision"
                ]
            ),

        "Heldout_F1":
            float(
                row[
                    "Heldout_F1"
                ]
            ),

        "Temporal_F1":
            float(
                row[
                    "Temporal_F1"
                ]
            ),

        "AUC_Change_Temporal_minus_Heldout":
            float(
                row[
                    "AUC_Change_Temporal_minus_Heldout"
                ]
            ),

        "Sensitivity_Change_Temporal_minus_Heldout":
            float(
                row[
                    "Sensitivity_Change_Temporal_minus_Heldout"
                ]
            ),

        "Specificity_Change_Temporal_minus_Heldout":
            float(
                row[
                    "Specificity_Change_Temporal_minus_Heldout"
                ]
            ),

        "F1_Change_Temporal_minus_Heldout":
            float(
                row[
                    "F1_Change_Temporal_minus_Heldout"
                ]
            ),

        "Temporal_Raw_OE_Ratio":
            float(
                row[
                    "Temporal_Raw_OE_Ratio"
                ]
            )
            if np.isfinite(
                row[
                    "Temporal_Raw_OE_Ratio"
                ]
            )
            else np.nan,

        "Temporal_Recalibrated_OE_Ratio":
            float(
                row[
                    "Temporal_Recalibrated_OE_Ratio"
                ]
            )
            if np.isfinite(
                row[
                    "Temporal_Recalibrated_OE_Ratio"
                ]
            )
            else np.nan,

        "Recalibration_Available":
            bool(
                row[
                    "Recalibration_Available"
                ]
            ),

        "Validation_Support_Summary":
            (
                "Supported"
                if (
                    heldout_discrimination_supported
                    and
                    temporal_discrimination_supported
                    and
                    heldout_sensitivity_not_below_oof_target
                    and
                    temporal_sensitivity_not_below_oof_target
                )
                else "Mixed"
            )
    })


part5b_validation_support = pd.DataFrame(
    validation_support_rows
)


# ============================================================
# 8. STAGE 3 — BURDEN-AWARE PARETO COMPARISON
# ============================================================
#
# Strategy A dominates Strategy B if:
# - A uses no more self-report items than B
# AND
# - A has temporal AUC >= B
# - A has temporal sensitivity >= B
# - A has temporal specificity >= B
# - A has temporal F1 >= B
# AND
# - at least one of these dimensions is strictly better.
#
# No weights are assigned.
# ============================================================

pareto_rows = []


representative_table = (
    part5b_strategy_representatives
    .copy()
)


for _, target_row in (
    representative_table.iterrows()
):

    target_model = str(
        target_row[
            "Screening_Model"
        ]
    )


    dominators = []


    for _, challenger_row in (
        representative_table.iterrows()
    ):

        challenger_model = str(
            challenger_row[
                "Screening_Model"
            ]
        )


        if challenger_model == target_model:

            continue


        no_more_burden = bool(
            challenger_row[
                "Self_Report_Items"
            ]
            <=
            target_row[
                "Self_Report_Items"
            ]
        )


        no_worse_auc = bool(
            challenger_row[
                "Temporal_AUC"
            ]
            >=
            target_row[
                "Temporal_AUC"
            ]
        )


        no_worse_sensitivity = bool(
            challenger_row[
                "Temporal_Sensitivity"
            ]
            >=
            target_row[
                "Temporal_Sensitivity"
            ]
        )


        no_worse_specificity = bool(
            challenger_row[
                "Temporal_Specificity"
            ]
            >=
            target_row[
                "Temporal_Specificity"
            ]
        )


        no_worse_f1 = bool(
            challenger_row[
                "Temporal_F1"
            ]
            >=
            target_row[
                "Temporal_F1"
            ]
        )


        strictly_better_somewhere = bool(

            challenger_row[
                "Self_Report_Items"
            ]
            <
            target_row[
                "Self_Report_Items"
            ]

            or

            challenger_row[
                "Temporal_AUC"
            ]
            >
            target_row[
                "Temporal_AUC"
            ]

            or

            challenger_row[
                "Temporal_Sensitivity"
            ]
            >
            target_row[
                "Temporal_Sensitivity"
            ]

            or

            challenger_row[
                "Temporal_Specificity"
            ]
            >
            target_row[
                "Temporal_Specificity"
            ]

            or

            challenger_row[
                "Temporal_F1"
            ]
            >
            target_row[
                "Temporal_F1"
            ]
        )


        dominates = bool(
            no_more_burden
            and
            no_worse_auc
            and
            no_worse_sensitivity
            and
            no_worse_specificity
            and
            no_worse_f1
            and
            strictly_better_somewhere
        )


        if dominates:

            dominators.append(
                challenger_model
            )


    pareto_rows.append({

        "Screening_Model":
            target_model,

        "Algorithm":
            str(
                target_row[
                    "Algorithm"
                ]
            ),

        "Self_Report_Items":
            int(
                target_row[
                    "Self_Report_Items"
                ]
            ),

        "Temporal_AUC":
            float(
                target_row[
                    "Temporal_AUC"
                ]
            ),

        "Temporal_Sensitivity":
            float(
                target_row[
                    "Temporal_Sensitivity"
                ]
            ),

        "Temporal_Specificity":
            float(
                target_row[
                    "Temporal_Specificity"
                ]
            ),

        "Temporal_Precision":
            float(
                target_row[
                    "Temporal_Precision"
                ]
            ),

        "Temporal_F1":
            float(
                target_row[
                    "Temporal_F1"
                ]
            ),

        "Dominated":
            bool(
                len(
                    dominators
                )
                >
                0
            ),

        "Dominated_By":
            ", ".join(
                dominators
            ),

        "Pareto_Efficient":
            bool(
                len(
                    dominators
                )
                ==
                0
            )
    })


part5b_pareto_comparison = pd.DataFrame(
    pareto_rows
)


# ============================================================
# 9. INTEGRATED SELECTION FRAMEWORK TABLE
# ============================================================

part5b_selection_framework = (
    part5b_strategy_representatives[
        [
            "Screening_Model",
            "Self_Report_Items",
            "Feature_Count",
            "Algorithm",
            "OOF_AUC",
            "OOF_Average_Precision",
            "OOF_Sensitivity",
            "OOF_Specificity",
            "OOF_F1",
            "Heldout_AUC_95CI",
            "Heldout_Sensitivity_95CI",
            "Heldout_Specificity_95CI",
            "Heldout_Precision_95CI",
            "Heldout_F1_95CI",
            "Temporal_AUC_95CI",
            "Temporal_Sensitivity_95CI",
            "Temporal_Specificity_95CI",
            "Temporal_Precision_95CI",
            "Temporal_F1_95CI",
            "AUC_Change_Temporal_minus_Heldout",
            "Sensitivity_Change_Temporal_minus_Heldout",
            "Specificity_Change_Temporal_minus_Heldout",
            "F1_Change_Temporal_minus_Heldout",
            "Temporal_Raw_OE_Ratio",
            "Temporal_Recalibrated_OE_Ratio",
            "Recalibration_Available"
        ]
    ]
    .copy()
)


part5b_selection_framework = (
    part5b_selection_framework
    .merge(

        part5b_validation_support[
            [
                "Screening_Model",
                "Algorithm",
                "Heldout_Discrimination_CI_Above_Chance",
                "Temporal_Discrimination_CI_Above_Chance",
                "Heldout_Sensitivity_At_Least_0_50",
                "Temporal_Sensitivity_At_Least_0_50",
                "Validation_Support_Summary"
            ]
        ],

        on=[
            "Screening_Model",
            "Algorithm"
        ],

        how="left",

        validate="one_to_one"
    )
)


part5b_selection_framework = (
    part5b_selection_framework
    .merge(

        part5b_pareto_comparison[
            [
                "Screening_Model",
                "Algorithm",
                "Dominated",
                "Dominated_By",
                "Pareto_Efficient"
            ]
        ],

        on=[
            "Screening_Model",
            "Algorithm"
        ],

        how="left",

        validate="one_to_one"
    )
)


part5b_selection_framework[
    "Framework_Interpretation"
] = np.select(

    [
        (
            part5b_selection_framework[
                "Validation_Support_Summary"
            ]
            ==
            "Supported"
        )
        &
        (
            part5b_selection_framework[
                "Pareto_Efficient"
            ]
        ),

        (
            part5b_selection_framework[
                "Validation_Support_Summary"
            ]
            ==
            "Supported"
        )
        &
        (
            ~part5b_selection_framework[
                "Pareto_Efficient"
            ]
        )
    ],

    [
        (
            "Externally supported and burden-performance "
            "Pareto-efficient candidate"
        ),

        (
            "Externally supported but burden-performance "
            "dominated by another strategy"
        )
    ],

    default=(
        "Mixed validation support; retain for descriptive "
        "comparison only"
    )
)


# ============================================================
# 10. VERIFICATION
# ============================================================

verification_rows = []


for model_name in STRATEGY_ORDER:

    ranking_subset = (
        part5b_algorithm_ranking[
            part5b_algorithm_ranking[
                "Screening_Model"
            ]
            ==
            model_name
        ]
    )


    representative_subset = (
        part5b_strategy_representatives[
            part5b_strategy_representatives[
                "Screening_Model"
            ]
            ==
            model_name
        ]
    )


    framework_subset = (
        part5b_selection_framework[
            part5b_selection_framework[
                "Screening_Model"
            ]
            ==
            model_name
        ]
    )


    exactly_five_ranked_algorithms = bool(
        len(
            ranking_subset
        )
        ==
        5
    )


    exactly_one_representative = bool(
        (
            ranking_subset[
                "Selected_Strategy_Representative"
            ]
        ).sum()
        ==
        1
        and
        len(
            representative_subset
        )
        ==
        1
    )


    ranking_is_complete = bool(
        set(
            ranking_subset[
                "OOF_Selection_Rank"
            ]
            .astype(int)
            .tolist()
        )
        ==
        {
            1,
            2,
            3,
            4,
            5
        }
    )


    no_validation_metric_used_in_algorithm_rank = True


    framework_row_present = bool(
        len(
            framework_subset
        )
        ==
        1
    )


    verification_rows.append({

        "Screening_Model":
            model_name,

        "Exactly_5_Algorithms_Ranked":
            exactly_five_ranked_algorithms,

        "Exactly_1_OOF_Selected_Representative":
            exactly_one_representative,

        "OOF_Ranking_Complete_1_to_5":
            ranking_is_complete,

        "Within_Strategy_Ranking_Uses_OOF_Only":
            no_validation_metric_used_in_algorithm_rank,

        "Framework_Row_Present":
            framework_row_present,

        "Final_Winner_Selected":
            False,

        "PART5B_Ready":
            (
                exactly_five_ranked_algorithms
                and
                exactly_one_representative
                and
                ranking_is_complete
                and
                no_validation_metric_used_in_algorithm_rank
                and
                framework_row_present
            )
    })


part5b_verification = pd.DataFrame(
    verification_rows
)


# ============================================================
# 11. DISPLAY
# ============================================================

print("=" * 100)
print("PART 5B VERSION 3.0: MODEL SELECTION FRAMEWORK")
print("=" * 100)


print(
    "\nSTAGE 1: OOF-ONLY REPRESENTATIVE ALGORITHM "
    "SELECTION"
)


display(
    part5b_algorithm_ranking[
        [
            "Screening_Model",
            "Algorithm",
            "OOF_Target_Recall_Achieved",
            "OOF_Probability_Degenerate_Flag",
            "OOF_AUC",
            "OOF_Average_Precision",
            "OOF_F1",
            "OOF_Specificity",
            "OOF_Selection_Rank",
            "Selected_Strategy_Representative"
        ]
    ]
    .sort_values(
        by=[
            "Screening_Model",
            "OOF_Selection_Rank"
        ]
    )
    .round(
        4
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 100)
print("ONE REPRESENTATIVE ALGORITHM PER SCREENING STRATEGY")
print("=" * 100)


display(
    part5b_strategy_representatives[
        [
            "Screening_Model",
            "Self_Report_Items",
            "Feature_Count",
            "Algorithm",
            "OOF_AUC",
            "OOF_F1",
            "Heldout_AUC_95CI",
            "Heldout_Sensitivity_95CI",
            "Heldout_Specificity_95CI",
            "Heldout_F1_95CI",
            "Temporal_AUC_95CI",
            "Temporal_Sensitivity_95CI",
            "Temporal_Specificity_95CI",
            "Temporal_F1_95CI"
        ]
    ]
    .round(
        4
    )
)


print("\n" + "=" * 100)
print("BURDEN-AWARE PARETO COMPARISON")
print("=" * 100)


display(
    part5b_pareto_comparison
    .round(
        4
    )
)


print("\n" + "=" * 100)
print("INTEGRATED MODEL SELECTION FRAMEWORK")
print("=" * 100)


display(
    part5b_selection_framework
    .round(
        4
    )
)


print("\n" + "=" * 100)
print("PART 5B VERIFICATION")
print("=" * 100)


display(
    part5b_verification
)


# ============================================================
# 12. HARD SAFETY CHECKS
# ============================================================

if len(
    part5b_strategy_representatives
) != 7:

    raise RuntimeError(
        "PART 5B did not produce exactly 7 strategy "
        "representatives."
    )


if (
    part5b_strategy_representatives[
        "Screening_Model"
    ].nunique()
    !=
    7
):

    raise RuntimeError(
        "PART 5B strategy representatives are not unique."
    )


if not part5b_verification[
    "PART5B_Ready"
].all():

    raise RuntimeError(
        "One or more screening strategies failed PART 5B "
        "verification."
    )


# ============================================================
# 13. EXPORT
# ============================================================

part5b_algorithm_ranking.to_csv(
    "part5b_algorithm_ranking_oof_only.csv",
    index=False,
    encoding="utf-8-sig"
)


part5b_strategy_representatives.to_csv(
    "part5b_strategy_representatives.csv",
    index=False,
    encoding="utf-8-sig"
)


part5b_validation_support.to_csv(
    "part5b_validation_support.csv",
    index=False,
    encoding="utf-8-sig"
)


part5b_pareto_comparison.to_csv(
    "part5b_pareto_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)


part5b_selection_framework.to_csv(
    "part5b_selection_framework.csv",
    index=False,
    encoding="utf-8-sig"
)


part5b_verification.to_csv(
    "part5b_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 5B VERSION 3.0 COMPLETED")
print("=" * 100)


print(
    f"Algorithms ranked             : "
    f"{len(part5b_algorithm_ranking)}"
)


print(
    f"Strategy representatives      : "
    f"{len(part5b_strategy_representatives)}"
)


print(
    f"Pareto-efficient strategies   : "
    f"{int(part5b_pareto_comparison['Pareto_Efficient'].sum())}"
)


print(
    "Final winner selected         : No"
)


print("\nArtifacts created:")

print("- part5b_algorithm_ranking")
print("- part5b_strategy_representatives")
print("- part5b_validation_support")
print("- part5b_pareto_comparison")
print("- part5b_selection_framework")
print("- part5b_verification")


print("\nExported files:")

print("- part5b_algorithm_ranking_oof_only.csv")
print("- part5b_strategy_representatives.csv")
print("- part5b_validation_support.csv")
print("- part5b_pareto_comparison.csv")
print("- part5b_selection_framework.csv")
print("- part5b_verification.csv")


print(
    "\nWithin each screening strategy, the representative "
    "algorithm was selected using genuine OOF 2024 performance "
    "only."
)


print(
    "Held-out 2024 and temporal 2025 results were used for "
    "validation support and burden-performance comparison, "
    "not for within-strategy algorithm re-selection."
)


print(
    "No weighted composite score was used and no final winner "
    "was declared in PART 5B."
)


print(
    "\nProceed to PART 5C only after PART5B_Ready=True for all "
    "7 screening strategies."
)

In [ ]:
display(
    part5b_strategy_representatives[
        [
            "Screening_Model",
            "Self_Report_Items",
            "Algorithm",
            "OOF_AUC",
            "OOF_F1",
            "Heldout_AUC_95CI",
            "Heldout_Sensitivity_95CI",
            "Heldout_Specificity_95CI",
            "Heldout_Precision_95CI",
            "Heldout_F1_95CI",
            "Temporal_AUC_95CI",
            "Temporal_Sensitivity_95CI",
            "Temporal_Specificity_95CI",
            "Temporal_Precision_95CI",
            "Temporal_F1_95CI"
        ]
    ]
)

In [ ]:
display(
    part5b_pareto_comparison
)

In [ ]:
display(
    part5b_selection_framework[
        [
            "Screening_Model",
            "Self_Report_Items",
            "Algorithm",
            "Validation_Support_Summary",
            "Dominated",
            "Dominated_By",
            "Pareto_Efficient",
            "Framework_Interpretation"
        ]
    ]
)

### PART 5C: Final Strategy Recommendation & Publication Table

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 5C: FINAL STRATEGY RECOMMENDATION & PUBLICATION TABLE
# ============================================================
#
# Purpose:
# Finalize the strategy-level recommendation using the
# transparent framework developed in PART 5B.
#
# IMPORTANT PRINCIPLES:
# - No model refitting
# - No new prediction generation
# - No threshold modification
# - No recalibration fitting
# - No algorithm re-selection from temporal 2025
# - No weighted composite score
#
# Primary recommendation:
# M2_Admin_2Q + its OOF-selected representative algorithm
#
# Rationale:
# - Selected representative algorithm within M2 using OOF 2024
#   development performance only
# - Validation support in held-out 2024 and temporal 2025
# - Pareto-efficient burden-performance profile
# - Only 2 self-report items
#
# Outputs:
# - part5c_final_strategy_table
# - part5c_strategy_roles
# - part5c_recommendation_summary
# - part5c_publication_table
# - part5c_verification
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 2. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "part5b_strategy_representatives",
    "part5b_selection_framework",
    "part5b_pareto_comparison",
    "part5b_verification"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


for object_name in [
    "part5b_strategy_representatives",
    "part5b_selection_framework",
    "part5b_pareto_comparison",
    "part5b_verification"
]:
    if not isinstance(
        globals()[object_name],
        pd.DataFrame
    ):
        raise TypeError(
            f"{object_name} must be a pandas DataFrame."
        )


if not part5b_verification[
    "PART5B_Ready"
].all():
    raise RuntimeError(
        "PART 5B is not fully ready."
    )


# ============================================================
# 3. PRESPECIFIED FINAL ROLES
# ============================================================

PRIMARY_MODEL = "M2_Admin_2Q"

# Retrieve the representative algorithm selected for M2 in
# PART 5B. Do not hard-code the algorithm because changing the
# resampling method can change the OOF-only ranking.
primary_algorithm_rows = (
    part5b_strategy_representatives.loc[
        part5b_strategy_representatives["Screening_Model"]
        == PRIMARY_MODEL,
        "Algorithm"
    ]
)

if len(primary_algorithm_rows) != 1:
    raise RuntimeError(
        "Expected exactly one OOF-selected representative "
        f"algorithm for {PRIMARY_MODEL} in PART 5B."
    )

PRIMARY_ALGORITHM = str(primary_algorithm_rows.iloc[0])

STRATEGY_ROLES = {
    "M0_Admin":
        "Zero-burden administrative reference",

    "M1_Admin_Behavior":
        "Behavior-enhanced administrative comparator",

    "M2_Admin_2Q":
        "Primary recommended low-burden strategy",

    "M3_Admin_2Q_Behavior":
        "Expanded low-burden screening comparator",

    "M4_Conventional":
        "Conventional screening comparator",

    "M5_Conventional_Admin":
        "Administrative-enhanced conventional comparator",

    "M6_Full":
        "Maximum-information full-assessment comparator"
}


# ============================================================
# 4. VALIDATE PRIMARY RECOMMENDATION EXISTS
# ============================================================

primary_rows = (
    part5b_strategy_representatives[
        (
            part5b_strategy_representatives[
                "Screening_Model"
            ] == PRIMARY_MODEL
        )
        &
        (
            part5b_strategy_representatives[
                "Algorithm"
            ] == PRIMARY_ALGORITHM
        )
    ]
)


if len(primary_rows) != 1:
    raise RuntimeError(
        "Primary recommended strategy/algorithm pair was not "
        "found exactly once in PART 5B representatives."
    )


primary_framework_row = (
    part5b_selection_framework[
        (
            part5b_selection_framework[
                "Screening_Model"
            ] == PRIMARY_MODEL
        )
        &
        (
            part5b_selection_framework[
                "Algorithm"
            ] == PRIMARY_ALGORITHM
        )
    ]
)


if len(primary_framework_row) != 1:
    raise RuntimeError(
        "Primary recommended strategy/algorithm pair was not "
        "found exactly once in PART 5B selection framework."
    )


primary_pareto_row = (
    part5b_pareto_comparison[
        (
            part5b_pareto_comparison[
                "Screening_Model"
            ] == PRIMARY_MODEL
        )
        &
        (
            part5b_pareto_comparison[
                "Algorithm"
            ] == PRIMARY_ALGORITHM
        )
    ]
)


if len(primary_pareto_row) != 1:
    raise RuntimeError(
        "Primary recommended strategy/algorithm pair was not "
        "found exactly once in PART 5B Pareto table."
    )


# ============================================================
# 5. FINAL STRATEGY TABLE
# ============================================================

part5c_final_strategy_table = (
    part5b_strategy_representatives
    .copy()
)


part5c_final_strategy_table[
    "Strategy_Role"
] = (
    part5c_final_strategy_table[
        "Screening_Model"
    ]
    .map(
        STRATEGY_ROLES
    )
)


part5c_final_strategy_table[
    "Primary_Recommended"
] = (
    (
        part5c_final_strategy_table[
            "Screening_Model"
        ] == PRIMARY_MODEL
    )
    &
    (
        part5c_final_strategy_table[
            "Algorithm"
        ] == PRIMARY_ALGORITHM
    )
)


part5c_final_strategy_table = (
    part5c_final_strategy_table
    .merge(
        part5b_selection_framework[
            [
                "Screening_Model",
                "Algorithm",
                "Validation_Support_Summary",
                "Dominated",
                "Dominated_By",
                "Pareto_Efficient",
                "Framework_Interpretation"
            ]
        ],
        on=[
            "Screening_Model",
            "Algorithm"
        ],
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 6. STRATEGY ROLES TABLE
# ============================================================

role_rows = []


for _, row in (
    part5c_final_strategy_table
    .iterrows()
):

    role_rows.append({

        "Screening_Model":
            row[
                "Screening_Model"
            ],

        "Algorithm":
            row[
                "Algorithm"
            ],

        "Self_Report_Items":
            int(
                row[
                    "Self_Report_Items"
                ]
            ),

        "Feature_Count":
            int(
                row[
                    "Feature_Count"
                ]
            ),

        "Strategy_Role":
            row[
                "Strategy_Role"
            ],

        "Primary_Recommended":
            bool(
                row[
                    "Primary_Recommended"
                ]
            ),

        "Validation_Support_Summary":
            row[
                "Validation_Support_Summary"
            ],

        "Pareto_Efficient":
            bool(
                row[
                    "Pareto_Efficient"
                ]
            ),

        "Dominated":
            bool(
                row[
                    "Dominated"
                ]
            ),

        "Dominated_By":
            row[
                "Dominated_By"
            ]
    })


part5c_strategy_roles = pd.DataFrame(
    role_rows
)


# ============================================================
# 7. RECOMMENDATION SUMMARY
# ============================================================

primary = primary_rows.iloc[0]

primary_framework = (
    primary_framework_row.iloc[0]
)

primary_pareto = (
    primary_pareto_row.iloc[0]
)


recommendation_rows = [

    {
        "Criterion":
            "Representative algorithm selection",

        "Evidence":
            (
                f"{PRIMARY_ALGORITHM} was selected within "
                f"{PRIMARY_MODEL} using genuine OOF 2024 "
                "performance only."
            )
    },

    {
        "Criterion":
            "Respondent burden",

        "Evidence":
            (
                f"{PRIMARY_MODEL} requires only "
                f"{int(primary['Self_Report_Items'])} "
                "self-report items."
            )
    },

    {
        "Criterion":
            "Held-out validation",

        "Evidence":
            (
                f"Held-out AUC: "
                f"{primary['Heldout_AUC_95CI']}; "
                f"sensitivity: "
                f"{primary['Heldout_Sensitivity_95CI']}; "
                f"specificity: "
                f"{primary['Heldout_Specificity_95CI']}."
            )
    },

    {
        "Criterion":
            "Temporal validation",

        "Evidence":
            (
                f"Temporal AUC: "
                f"{primary['Temporal_AUC_95CI']}; "
                f"sensitivity: "
                f"{primary['Temporal_Sensitivity_95CI']}; "
                f"specificity: "
                f"{primary['Temporal_Specificity_95CI']}."
            )
    },

    {
        "Criterion":
            "Burden-performance efficiency",

        "Evidence":
            (
                "Strategy is Pareto-efficient."
                if bool(
                    primary_pareto[
                        "Pareto_Efficient"
                    ]
                )
                else (
                    "Strategy is not Pareto-efficient."
                )
            )
    },

    {
        "Criterion":
            "Validation interpretation",

        "Evidence":
            str(
                primary_framework[
                    "Framework_Interpretation"
                ]
            )
    },

    {
        "Criterion":
            "Final recommendation",

        "Evidence":
            (
                f"{PRIMARY_MODEL} with {PRIMARY_ALGORITHM} "
                "is designated as the primary recommended "
                "low-burden screening strategy."
            )
    }
]


part5c_recommendation_summary = pd.DataFrame(
    recommendation_rows
)


# ============================================================
# 8. PUBLICATION TABLE
# ============================================================

part5c_publication_table = (
    part5c_final_strategy_table[
        [
            "Screening_Model",
            "Strategy_Role",
            "Self_Report_Items",
            "Feature_Count",
            "Algorithm",
            "OOF_AUC",
            "OOF_F1",
            "Heldout_AUC_95CI",
            "Heldout_Sensitivity_95CI",
            "Heldout_Specificity_95CI",
            "Heldout_Precision_95CI",
            "Heldout_F1_95CI",
            "Temporal_AUC_95CI",
            "Temporal_Sensitivity_95CI",
            "Temporal_Specificity_95CI",
            "Temporal_Precision_95CI",
            "Temporal_F1_95CI",
            "AUC_Change_Temporal_minus_Heldout",
            "F1_Change_Temporal_minus_Heldout",
            "Validation_Support_Summary",
            "Pareto_Efficient",
            "Primary_Recommended"
        ]
    ]
    .copy()
)


# ============================================================
# 9. COMPACT MANUSCRIPT TABLE
# ============================================================

part5c_publication_table_compact = (
    part5c_publication_table[
        [
            "Screening_Model",
            "Self_Report_Items",
            "Algorithm",
            "Heldout_AUC_95CI",
            "Heldout_Sensitivity_95CI",
            "Heldout_Specificity_95CI",
            "Heldout_F1_95CI",
            "Temporal_AUC_95CI",
            "Temporal_Sensitivity_95CI",
            "Temporal_Specificity_95CI",
            "Temporal_F1_95CI",
            "Validation_Support_Summary",
            "Pareto_Efficient",
            "Primary_Recommended"
        ]
    ]
    .copy()
)


# ============================================================
# 10. FINAL RECOMMENDATION RECORD
# ============================================================

part5c_final_recommendation = {

    "Primary_Recommended_Strategy":
        PRIMARY_MODEL,

    "Primary_Recommended_Algorithm":
        PRIMARY_ALGORITHM,

    "Self_Report_Items":
        int(
            primary[
                "Self_Report_Items"
            ]
        ),

    "Feature_Count":
        int(
            primary[
                "Feature_Count"
            ]
        ),

    "OOF_AUC":
        float(
            primary[
                "OOF_AUC"
            ]
        ),

    "OOF_F1":
        float(
            primary[
                "OOF_F1"
            ]
        ),

    "Heldout_AUC_95CI":
        primary[
            "Heldout_AUC_95CI"
        ],

    "Heldout_Sensitivity_95CI":
        primary[
            "Heldout_Sensitivity_95CI"
        ],

    "Heldout_Specificity_95CI":
        primary[
            "Heldout_Specificity_95CI"
        ],

    "Heldout_F1_95CI":
        primary[
            "Heldout_F1_95CI"
        ],

    "Temporal_AUC_95CI":
        primary[
            "Temporal_AUC_95CI"
        ],

    "Temporal_Sensitivity_95CI":
        primary[
            "Temporal_Sensitivity_95CI"
        ],

    "Temporal_Specificity_95CI":
        primary[
            "Temporal_Specificity_95CI"
        ],

    "Temporal_F1_95CI":
        primary[
            "Temporal_F1_95CI"
        ],

    "Pareto_Efficient":
        bool(
            primary_pareto[
                "Pareto_Efficient"
            ]
        ),

    "Validation_Support_Summary":
        primary_framework[
            "Validation_Support_Summary"
        ],

    "Recommendation_Label":
        "Primary recommended low-burden strategy"
}


# ============================================================
# 11. VERIFICATION
# ============================================================

primary_recommendation_count = int(
    part5c_final_strategy_table[
        "Primary_Recommended"
    ].sum()
)


all_strategy_roles_present = bool(
    part5c_final_strategy_table[
        "Strategy_Role"
    ].notna().all()
)


all_7_strategies_present = bool(
    part5c_final_strategy_table[
        "Screening_Model"
    ].nunique()
    ==
    7
)


primary_is_pareto_efficient = bool(
    primary_pareto[
        "Pareto_Efficient"
    ]
)


primary_has_supported_validation = bool(
    primary_framework[
        "Validation_Support_Summary"
    ]
    ==
    "Supported"
)


primary_uses_two_items = bool(
    int(
        primary[
            "Self_Report_Items"
        ]
    )
    ==
    2
)


primary_algorithm_matches_part5b = bool(
    (
        primary[
            "Algorithm"
        ]
        ==
        PRIMARY_ALGORITHM
    )
)


no_model_refit = True
no_new_prediction_generation = True
no_threshold_modification = True
no_recalibration_fit = True
no_weighted_score_used = True


part5c_verification = pd.DataFrame(
    [
        {

            "All_7_Strategies_Present":
                all_7_strategies_present,

            "All_Strategy_Roles_Present":
                all_strategy_roles_present,

            "Exactly_1_Primary_Recommendation":
                (
                    primary_recommendation_count
                    ==
                    1
                ),

            "Primary_Is_M2_Admin_2Q":
                bool(
                    part5c_final_recommendation[
                        "Primary_Recommended_Strategy"
                    ]
                    ==
                    "M2_Admin_2Q"
                ),

            "Primary_Algorithm_Is_OOF_Selected_Representative":
                primary_algorithm_matches_part5b,

            "Primary_Algorithm_Matches_PART5B":
                primary_algorithm_matches_part5b,

            "Primary_Uses_2_Self_Report_Items":
                primary_uses_two_items,

            "Primary_Is_Pareto_Efficient":
                primary_is_pareto_efficient,

            "Primary_Has_Supported_Validation":
                primary_has_supported_validation,

            "No_Model_Refit":
                no_model_refit,

            "No_New_Prediction_Generation":
                no_new_prediction_generation,

            "No_Threshold_Modification":
                no_threshold_modification,

            "No_Recalibration_Fit":
                no_recalibration_fit,

            "No_Weighted_Composite_Score":
                no_weighted_score_used,

            "PART5C_Ready":
                bool(
                    all_7_strategies_present
                    and
                    all_strategy_roles_present
                    and
                    primary_recommendation_count
                    ==
                    1
                    and
                    primary_algorithm_matches_part5b
                    and
                    primary_uses_two_items
                    and
                    primary_is_pareto_efficient
                    and
                    primary_has_supported_validation
                    and
                    no_model_refit
                    and
                    no_new_prediction_generation
                    and
                    no_threshold_modification
                    and
                    no_recalibration_fit
                    and
                    no_weighted_score_used
                )
        }
    ]
)


# ============================================================
# 12. DISPLAY
# ============================================================

print("=" * 100)
print(
    "PART 5C VERSION 3.0: FINAL STRATEGY "
    "RECOMMENDATION"
)
print("=" * 100)


print(
    "\nPrimary recommended strategy:"
)


print(
    f"- Strategy  : "
    f"{part5c_final_recommendation['Primary_Recommended_Strategy']}"
)


print(
    f"- Algorithm : "
    f"{part5c_final_recommendation['Primary_Recommended_Algorithm']}"
)


print(
    f"- Self-report items : "
    f"{part5c_final_recommendation['Self_Report_Items']}"
)


print(
    "\nFinal strategy comparison:"
)


display(
    part5c_publication_table
    .round(
        4
    )
)


print(
    "\nRecommendation rationale:"
)


display(
    part5c_recommendation_summary
)


print(
    "\nPART 5C verification:"
)


display(
    part5c_verification
)


# ============================================================
# 13. HARD SAFETY CHECK
# ============================================================

if not part5c_verification[
    "PART5C_Ready"
].all():

    raise RuntimeError(
        "PART 5C verification failed."
    )


# ============================================================
# 14. EXPORT
# ============================================================

part5c_final_strategy_table.to_csv(
    "part5c_final_strategy_table.csv",
    index=False,
    encoding="utf-8-sig"
)


part5c_strategy_roles.to_csv(
    "part5c_strategy_roles.csv",
    index=False,
    encoding="utf-8-sig"
)


part5c_recommendation_summary.to_csv(
    "part5c_recommendation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


part5c_publication_table.to_csv(
    "part5c_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


part5c_publication_table_compact.to_csv(
    "part5c_publication_table_compact.csv",
    index=False,
    encoding="utf-8-sig"
)


part5c_verification.to_csv(
    "part5c_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


pd.DataFrame(
    [
        part5c_final_recommendation
    ]
).to_csv(
    "part5c_final_recommendation.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 15. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 5C VERSION 3.0 COMPLETED")
print("=" * 100)


print(
    f"Primary recommended strategy : "
    f"{PRIMARY_MODEL}"
)


print(
    f"Primary algorithm            : "
    f"{PRIMARY_ALGORITHM}"
)


print(
    f"Self-report items            : "
    f"{int(primary['Self_Report_Items'])}"
)


print(
    f"Pareto-efficient             : "
    f"{bool(primary_pareto['Pareto_Efficient'])}"
)


print(
    f"Validation support           : "
    f"{primary_framework['Validation_Support_Summary']}"
)


print("\nArtifacts created:")

print("- part5c_final_strategy_table")
print("- part5c_strategy_roles")
print("- part5c_recommendation_summary")
print("- part5c_publication_table")
print("- part5c_publication_table_compact")
print("- part5c_final_recommendation")
print("- part5c_verification")


print("\nExported files:")

print("- part5c_final_strategy_table.csv")
print("- part5c_strategy_roles.csv")
print("- part5c_recommendation_summary.csv")
print("- part5c_publication_table.csv")
print("- part5c_publication_table_compact.csv")
print("- part5c_final_recommendation.csv")
print("- part5c_verification.csv")


print(
    f"\n{PRIMARY_MODEL} + {PRIMARY_ALGORITHM} is designated "
    "as the primary recommended low-burden strategy based "
    "on the prespecified hierarchical selection framework."
)


print(
    "This recommendation reflects burden-performance balance "
    "and validation support, not the highest single validation "
    "metric."
)


print(
    "No model was refitted, no new prediction was generated, "
    "no threshold was modified, and no weighted composite "
    "score was used."
)

## PART 6: Final Model Interpretation and Final Model Reporting Package

### PART 6A: Final Model Interpretation — M2_Admin_2Q + AdaBoostClassifier

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 6A: FINAL MODEL INTERPRETATION
# M2_Admin_2Q + AdaBoostClassifier
# ============================================================
#
# Purpose:
# Interpret the FINAL LOCKED model selected in PART 5C.
#
# Final model:
# - Screening strategy: M2_Admin_2Q
# - Algorithm: AdaBoostClassifier
#
# Interpretation methods:
# 1. AdaBoost intrinsic feature importance
# 2. Model-agnostic permutation importance on:
#    - Held-out 2024 cohort
#    - Temporal 2025 cohort
#
# Primary interpretation:
# - Held-out 2024 permutation importance
#
# Temporal interpretation:
# - Used only as a transportability / consistency check
#
# IMPORTANT:
# - No model refitting
# - No model tuning
# - No threshold modification
# - No recalibration
# - No feature selection
# - No winner re-selection
# - Feature importance is predictive, NOT causal
#
# Outputs:
# - part6a_intrinsic_importance
# - part6a_heldout_permutation_importance
# - part6a_temporal_permutation_importance
# - part6a_integrated_feature_importance
# - part6a_importance_consistency
# - part6a_verification
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr


# ============================================================
# 2. SETTINGS
# ============================================================

FINAL_SCREENING_MODEL = None

FINAL_ALGORITHM = None

N_PERMUTATION_REPEATS = 200

RANDOM_STATE = 42

PERMUTATION_SCORING_PRIMARY = "roc_auc"

PERMUTATION_SCORING_SECONDARY = "average_precision"

PLOT_OUTPUT_DIRECTORY = Path(
    "part6a_final_model_interpretation_plots"
)

PLOT_DPI = 300


# ============================================================
# 3. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "part5c_final_recommendation",
    "evaluation_models",
    "prepared_data",
    "temporal_datasets",
    "screening_strategies",
    "OUTCOME"
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(
            missing_objects
        )
    )


# Read the final locked recommendation only after confirming
# that the PART 5C object exists.
FINAL_SCREENING_MODEL = str(
    part5c_final_recommendation[
        "Primary_Recommended_Strategy"
    ]
)

FINAL_ALGORITHM = str(
    part5c_final_recommendation[
        "Primary_Recommended_Algorithm"
    ]
)


# ============================================================
# 4. VERIFY FINAL MODEL IS STILL LOCKED
# ============================================================

if (
    part5c_final_recommendation[
        "Primary_Recommended_Strategy"
    ]
    !=
    FINAL_SCREENING_MODEL
):

    raise RuntimeError(
        "PART 5C final strategy does not match M2_Admin_2Q."
    )


if (
    part5c_final_recommendation[
        "Primary_Recommended_Algorithm"
    ]
    !=
    FINAL_ALGORITHM
):

    raise RuntimeError(
        "PART 5C final algorithm does not match "
        "AdaBoostClassifier."
    )


# ============================================================
# 5. GET FINAL FITTED PIPELINE
# ============================================================

if FINAL_SCREENING_MODEL not in evaluation_models:

    raise KeyError(
        f"{FINAL_SCREENING_MODEL} not found in evaluation_models."
    )


if (
    FINAL_ALGORITHM
    not in
    evaluation_models[
        FINAL_SCREENING_MODEL
    ]
):

    raise KeyError(
        f"{FINAL_ALGORITHM} not found for "
        f"{FINAL_SCREENING_MODEL}."
    )


final_pipeline = (
    evaluation_models[
        FINAL_SCREENING_MODEL
    ][
        FINAL_ALGORITHM
    ]
)


if not hasattr(
    final_pipeline,
    "named_steps"
):

    raise TypeError(
        "Final evaluation model is not the expected pipeline."
    )


required_pipeline_steps = [
    "over_sampler",
    "under_sampler",
    "estimator"
]


for step_name in required_pipeline_steps:

    if step_name not in final_pipeline.named_steps:

        raise KeyError(
            f"Final pipeline is missing step: {step_name}"
        )


final_estimator = (
    final_pipeline.named_steps[
        "estimator"
    ]
)


if type(
    final_estimator
).__name__ != FINAL_ALGORITHM:

    raise RuntimeError(
        "Underlying estimator does not match final algorithm."
    )


final_over_sampler = final_pipeline.named_steps[
    "over_sampler"
]

expected_sampler_type = str(
    screening_strategies[
        FINAL_SCREENING_MODEL
    ]["sampler_type"]
)

observed_sampler_type = type(
    final_over_sampler
).__name__

if observed_sampler_type != expected_sampler_type:
    raise RuntimeError(
        "Final over-sampler does not match the strategy "
        f"definition: expected {expected_sampler_type}, "
        f"observed {observed_sampler_type}."
    )


# ============================================================
# 6. FEATURE DEFINITIONS
# ============================================================

final_features = list(
    screening_strategies[
        FINAL_SCREENING_MODEL
    ][
        "features"
    ]
)


if len(
    final_features
) != 10:

    print(
        f"Warning: M2_Admin_2Q currently contains "
        f"{len(final_features)} features."
    )


print("=" * 100)
print("PART 6A VERSION 3.0: FINAL MODEL INTERPRETATION")
print("=" * 100)

print(
    f"Final screening strategy : "
    f"{FINAL_SCREENING_MODEL}"
)

print(
    f"Final algorithm          : "
    f"{FINAL_ALGORITHM}"
)

print(
    f"Number of features       : "
    f"{len(final_features)}"
)

print(
    f"Self-report items        : "
    f"{part5c_final_recommendation['Self_Report_Items']}"
)

print(
    "\nFeatures:"
)

for index, feature in enumerate(
    final_features,
    start=1
):

    print(
        f"{index:>2}. {feature}"
    )


# ============================================================
# 7. GET HELD-OUT 2024 DATA
# ============================================================

heldout_info = (
    prepared_data[
        FINAL_SCREENING_MODEL
    ]
)


X_heldout = (
    heldout_info[
        "X_test"
    ][
        final_features
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


y_heldout = (
    heldout_info[
        "y_test"
    ]
    .astype(int)
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. GET TEMPORAL 2025 DATA
# ============================================================

temporal_data = (
    temporal_datasets[
        FINAL_SCREENING_MODEL
    ]
    .copy()
)


X_temporal = (
    temporal_data[
        final_features
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


y_temporal = (
    temporal_data[
        OUTCOME
    ]
    .astype(int)
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 9. DATA VERIFICATION
# ============================================================

if list(
    X_heldout.columns
) != final_features:

    raise RuntimeError(
        "Held-out feature order does not match final strategy."
    )


if list(
    X_temporal.columns
) != final_features:

    raise RuntimeError(
        "Temporal feature order does not match final strategy."
    )


if set(
    y_heldout.unique()
) != {
    0,
    1
}:

    raise ValueError(
        "Held-out outcome must contain both classes."
    )


if set(
    y_temporal.unique()
) != {
    0,
    1
}:

    raise ValueError(
        "Temporal outcome must contain both classes."
    )


# ============================================================
# 10. INTRINSIC ADABOOST FEATURE IMPORTANCE
# ============================================================

if not hasattr(
    final_estimator,
    "feature_importances_"
):

    raise AttributeError(
        "AdaBoost estimator does not expose feature_importances_."
    )


intrinsic_importance = np.asarray(
    final_estimator.feature_importances_,
    dtype=float
)


if len(
    intrinsic_importance
) != len(
    final_features
):

    raise RuntimeError(
        "Intrinsic importance length does not match feature count."
    )


part6a_intrinsic_importance = pd.DataFrame({

    "Feature":
        final_features,

    "Intrinsic_Importance":
        intrinsic_importance
})


part6a_intrinsic_importance[
    "Intrinsic_Rank"
] = (
    part6a_intrinsic_importance[
        "Intrinsic_Importance"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


part6a_intrinsic_importance = (
    part6a_intrinsic_importance
    .sort_values(
        [
            "Intrinsic_Rank",
            "Feature"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 11. HELPER: PERMUTATION IMPORTANCE
# ============================================================

def calculate_permutation_importance_table(
    estimator,
    X,
    y,
    cohort_name,
    scoring,
    repeats,
    random_state
):
    """
    Model-agnostic permutation importance.

    Importance = decrease in scoring metric after permuting
    one feature.

    Positive values indicate that disrupting the feature
    reduces predictive performance.
    """

    result = permutation_importance(

        estimator=estimator,

        X=X,

        y=y,

        scoring=scoring,

        n_repeats=repeats,

        random_state=random_state,

        n_jobs=-1
    )


    table = pd.DataFrame({

        "Feature":
            X.columns,

        "Importance_Mean":
            result.importances_mean,

        "Importance_SD":
            result.importances_std,

        "Importance_Min":
            np.min(
                result.importances,
                axis=1
            ),

        "Importance_Max":
            np.max(
                result.importances,
                axis=1
            ),

        "Positive_Importance_Proportion":
            np.mean(
                result.importances > 0,
                axis=1
            )
    })


    table[
        "Importance_SE"
    ] = (
        table[
            "Importance_SD"
        ]
        /
        np.sqrt(
            repeats
        )
    )


    table[
        "Approx_95CI_Lower"
    ] = (
        table[
            "Importance_Mean"
        ]
        -
        1.96
        *
        table[
            "Importance_SE"
        ]
    )


    table[
        "Approx_95CI_Upper"
    ] = (
        table[
            "Importance_Mean"
        ]
        +
        1.96
        *
        table[
            "Importance_SE"
        ]
    )


    table[
        "Permutation_Rank"
    ] = (
        table[
            "Importance_Mean"
        ]
        .rank(
            method="min",
            ascending=False
        )
        .astype(int)
    )


    table[
        "Cohort"
    ] = cohort_name


    table[
        "Scoring"
    ] = scoring


    table[
        "Permutation_Repeats"
    ] = repeats


    return (
        table
        .sort_values(
            [
                "Permutation_Rank",
                "Feature"
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 12. HELD-OUT 2024 PERMUTATION IMPORTANCE
# ============================================================

print("\n" + "=" * 100)
print("HELD-OUT 2024 PERMUTATION IMPORTANCE")
print("=" * 100)


part6a_heldout_permutation_auc = (
    calculate_permutation_importance_table(

        estimator=final_pipeline,

        X=X_heldout,

        y=y_heldout,

        cohort_name="Held-out 2024",

        scoring=PERMUTATION_SCORING_PRIMARY,

        repeats=N_PERMUTATION_REPEATS,

        random_state=RANDOM_STATE
    )
)


part6a_heldout_permutation_ap = (
    calculate_permutation_importance_table(

        estimator=final_pipeline,

        X=X_heldout,

        y=y_heldout,

        cohort_name="Held-out 2024",

        scoring=PERMUTATION_SCORING_SECONDARY,

        repeats=N_PERMUTATION_REPEATS,

        random_state=RANDOM_STATE + 1
    )
)


part6a_heldout_permutation_importance = (
    part6a_heldout_permutation_auc[
        [
            "Feature",
            "Importance_Mean",
            "Importance_SD",
            "Approx_95CI_Lower",
            "Approx_95CI_Upper",
            "Positive_Importance_Proportion",
            "Permutation_Rank"
        ]
    ]
    .rename(
        columns={
            "Importance_Mean":
                "Heldout_AUC_Importance_Mean",

            "Importance_SD":
                "Heldout_AUC_Importance_SD",

            "Approx_95CI_Lower":
                "Heldout_AUC_Approx_95CI_Lower",

            "Approx_95CI_Upper":
                "Heldout_AUC_Approx_95CI_Upper",

            "Positive_Importance_Proportion":
                "Heldout_AUC_Positive_Proportion",

            "Permutation_Rank":
                "Heldout_AUC_Rank"
        }
    )
    .merge(

        part6a_heldout_permutation_ap[
            [
                "Feature",
                "Importance_Mean",
                "Permutation_Rank"
            ]
        ]
        .rename(
            columns={
                "Importance_Mean":
                    "Heldout_AP_Importance_Mean",

                "Permutation_Rank":
                    "Heldout_AP_Rank"
            }
        ),

        on="Feature",

        how="left",

        validate="one_to_one"
    )
)


# ============================================================
# 13. TEMPORAL 2025 PERMUTATION IMPORTANCE
# ============================================================

print("\n" + "=" * 100)
print("TEMPORAL 2025 PERMUTATION IMPORTANCE")
print("=" * 100)


part6a_temporal_permutation_auc = (
    calculate_permutation_importance_table(

        estimator=final_pipeline,

        X=X_temporal,

        y=y_temporal,

        cohort_name="Temporal 2025",

        scoring=PERMUTATION_SCORING_PRIMARY,

        repeats=N_PERMUTATION_REPEATS,

        random_state=RANDOM_STATE + 100
    )
)


part6a_temporal_permutation_ap = (
    calculate_permutation_importance_table(

        estimator=final_pipeline,

        X=X_temporal,

        y=y_temporal,

        cohort_name="Temporal 2025",

        scoring=PERMUTATION_SCORING_SECONDARY,

        repeats=N_PERMUTATION_REPEATS,

        random_state=RANDOM_STATE + 101
    )
)


part6a_temporal_permutation_importance = (
    part6a_temporal_permutation_auc[
        [
            "Feature",
            "Importance_Mean",
            "Importance_SD",
            "Approx_95CI_Lower",
            "Approx_95CI_Upper",
            "Positive_Importance_Proportion",
            "Permutation_Rank"
        ]
    ]
    .rename(
        columns={
            "Importance_Mean":
                "Temporal_AUC_Importance_Mean",

            "Importance_SD":
                "Temporal_AUC_Importance_SD",

            "Approx_95CI_Lower":
                "Temporal_AUC_Approx_95CI_Lower",

            "Approx_95CI_Upper":
                "Temporal_AUC_Approx_95CI_Upper",

            "Positive_Importance_Proportion":
                "Temporal_AUC_Positive_Proportion",

            "Permutation_Rank":
                "Temporal_AUC_Rank"
        }
    )
    .merge(

        part6a_temporal_permutation_ap[
            [
                "Feature",
                "Importance_Mean",
                "Permutation_Rank"
            ]
        ]
        .rename(
            columns={
                "Importance_Mean":
                    "Temporal_AP_Importance_Mean",

                "Permutation_Rank":
                    "Temporal_AP_Rank"
            }
        ),

        on="Feature",

        how="left",

        validate="one_to_one"
    )
)


# ============================================================
# 14. INTEGRATED FEATURE IMPORTANCE TABLE
# ============================================================

part6a_integrated_feature_importance = (
    part6a_intrinsic_importance
    .merge(

        part6a_heldout_permutation_importance,

        on="Feature",

        how="left",

        validate="one_to_one"
    )
    .merge(

        part6a_temporal_permutation_importance,

        on="Feature",

        how="left",

        validate="one_to_one"
    )
)


part6a_integrated_feature_importance[
    "Mean_Validation_AUC_Importance"
] = (
    part6a_integrated_feature_importance[
        [
            "Heldout_AUC_Importance_Mean",
            "Temporal_AUC_Importance_Mean"
        ]
    ]
    .mean(
        axis=1
    )
)


part6a_integrated_feature_importance[
    "Validation_AUC_Importance_Rank"
] = (
    part6a_integrated_feature_importance[
        "Mean_Validation_AUC_Importance"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


part6a_integrated_feature_importance[
    "Importance_Consistent_Positive"
] = (
    (
        part6a_integrated_feature_importance[
            "Heldout_AUC_Importance_Mean"
        ]
        >
        0
    )
    &
    (
        part6a_integrated_feature_importance[
            "Temporal_AUC_Importance_Mean"
        ]
        >
        0
    )
)


part6a_integrated_feature_importance = (
    part6a_integrated_feature_importance
    .sort_values(
        [
            "Validation_AUC_Importance_Rank",
            "Intrinsic_Rank"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 15. IMPORTANCE CONSISTENCY
# ============================================================

heldout_rank_map = (
    part6a_integrated_feature_importance
    .set_index(
        "Feature"
    )[
        "Heldout_AUC_Rank"
    ]
)


temporal_rank_map = (
    part6a_integrated_feature_importance
    .set_index(
        "Feature"
    )[
        "Temporal_AUC_Rank"
    ]
)


intrinsic_rank_map = (
    part6a_integrated_feature_importance
    .set_index(
        "Feature"
    )[
        "Intrinsic_Rank"
    ]
)


common_features = final_features


heldout_vs_temporal_rho, heldout_vs_temporal_p = (
    spearmanr(
        [
            heldout_rank_map[
                feature
            ]
            for feature in common_features
        ],
        [
            temporal_rank_map[
                feature
            ]
            for feature in common_features
        ]
    )
)


intrinsic_vs_heldout_rho, intrinsic_vs_heldout_p = (
    spearmanr(
        [
            intrinsic_rank_map[
                feature
            ]
            for feature in common_features
        ],
        [
            heldout_rank_map[
                feature
            ]
            for feature in common_features
        ]
    )
)


intrinsic_vs_temporal_rho, intrinsic_vs_temporal_p = (
    spearmanr(
        [
            intrinsic_rank_map[
                feature
            ]
            for feature in common_features
        ],
        [
            temporal_rank_map[
                feature
            ]
            for feature in common_features
        ]
    )
)


part6a_importance_consistency = pd.DataFrame(
    [
        {
            "Comparison":
                "Held-out AUC permutation vs Temporal AUC permutation",

            "Spearman_Rho":
                heldout_vs_temporal_rho,

            "P_Value":
                heldout_vs_temporal_p
        },

        {
            "Comparison":
                "Intrinsic importance vs Held-out AUC permutation",

            "Spearman_Rho":
                intrinsic_vs_heldout_rho,

            "P_Value":
                intrinsic_vs_heldout_p
        },

        {
            "Comparison":
                "Intrinsic importance vs Temporal AUC permutation",

            "Spearman_Rho":
                intrinsic_vs_temporal_rho,

            "P_Value":
                intrinsic_vs_temporal_p
        }
    ]
)


# ============================================================
# 16. PRIMARY INTERPRETATION TABLE
# ============================================================
#
# Held-out permutation AUC importance is primary.
# Temporal permutation importance is a consistency check.
# ============================================================

part6a_primary_interpretation_table = (
    part6a_integrated_feature_importance[
        [
            "Feature",
            "Intrinsic_Importance",
            "Intrinsic_Rank",
            "Heldout_AUC_Importance_Mean",
            "Heldout_AUC_Importance_SD",
            "Heldout_AUC_Approx_95CI_Lower",
            "Heldout_AUC_Approx_95CI_Upper",
            "Heldout_AUC_Rank",
            "Heldout_AP_Importance_Mean",
            "Temporal_AUC_Importance_Mean",
            "Temporal_AUC_Rank",
            "Temporal_AP_Importance_Mean",
            "Importance_Consistent_Positive"
        ]
    ]
    .copy()
)


# ============================================================
# 17. CREATE PLOTS
# ============================================================

PLOT_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Plot 1: Intrinsic feature importance
# ------------------------------------------------------------

intrinsic_plot_data = (
    part6a_intrinsic_importance
    .sort_values(
        "Intrinsic_Importance",
        ascending=True
    )
)


figure, axis = plt.subplots(
    figsize=(
        8,
        6
    )
)


axis.barh(
    intrinsic_plot_data[
        "Feature"
    ],
    intrinsic_plot_data[
        "Intrinsic_Importance"
    ]
)


axis.set_xlabel(
    "AdaBoost intrinsic feature importance"
)


axis.set_ylabel(
    "Feature"
)


axis.set_title(
    "Final Model: Intrinsic Feature Importance\n"
    "M2_Admin_2Q + AdaBoostClassifier"
)


figure.tight_layout()


intrinsic_plot_path = (
    PLOT_OUTPUT_DIRECTORY
    /
    "part6a_intrinsic_feature_importance.png"
)


figure.savefig(
    intrinsic_plot_path,
    dpi=PLOT_DPI,
    bbox_inches="tight"
)


plt.close(
    figure
)


# ------------------------------------------------------------
# Plot 2: Held-out permutation importance
# ------------------------------------------------------------

heldout_plot_data = (
    part6a_primary_interpretation_table
    .sort_values(
        "Heldout_AUC_Importance_Mean",
        ascending=True
    )
)


figure, axis = plt.subplots(
    figsize=(
        8,
        6
    )
)


axis.barh(
    heldout_plot_data[
        "Feature"
    ],
    heldout_plot_data[
        "Heldout_AUC_Importance_Mean"
    ],
    xerr=heldout_plot_data[
        "Heldout_AUC_Importance_SD"
    ]
    if "Heldout_AUC_Importance_SD"
    in heldout_plot_data.columns
    else None
)


axis.axvline(
    0,
    linewidth=1
)


axis.set_xlabel(
    "Decrease in held-out ROC AUC after permutation"
)


axis.set_ylabel(
    "Feature"
)


axis.set_title(
    "Primary Model-Agnostic Feature Importance\n"
    "Held-out 2024 Cohort"
)


figure.tight_layout()


heldout_plot_path = (
    PLOT_OUTPUT_DIRECTORY
    /
    "part6a_heldout_permutation_importance_auc.png"
)


figure.savefig(
    heldout_plot_path,
    dpi=PLOT_DPI,
    bbox_inches="tight"
)


plt.close(
    figure
)


# ------------------------------------------------------------
# Plot 3: Temporal permutation importance
# ------------------------------------------------------------

temporal_plot_data = (
    part6a_integrated_feature_importance
    .sort_values(
        "Temporal_AUC_Importance_Mean",
        ascending=True
    )
)


figure, axis = plt.subplots(
    figsize=(
        8,
        6
    )
)


axis.barh(
    temporal_plot_data[
        "Feature"
    ],
    temporal_plot_data[
        "Temporal_AUC_Importance_Mean"
    ]
)


axis.axvline(
    0,
    linewidth=1
)


axis.set_xlabel(
    "Decrease in temporal ROC AUC after permutation"
)


axis.set_ylabel(
    "Feature"
)


axis.set_title(
    "Temporal Transportability of Feature Importance\n"
    "2025 Cohort"
)


figure.tight_layout()


temporal_plot_path = (
    PLOT_OUTPUT_DIRECTORY
    /
    "part6a_temporal_permutation_importance_auc.png"
)


figure.savefig(
    temporal_plot_path,
    dpi=PLOT_DPI,
    bbox_inches="tight"
)


plt.close(
    figure
)


# ============================================================
# 18. VERIFICATION
# ============================================================

all_features_present = bool(
    set(
        part6a_integrated_feature_importance[
            "Feature"
        ]
    )
    ==
    set(
        final_features
    )
)


feature_count_correct = bool(
    len(
        part6a_integrated_feature_importance
    )
    ==
    len(
        final_features
    )
)


intrinsic_importance_finite = bool(
    np.isfinite(
        part6a_integrated_feature_importance[
            "Intrinsic_Importance"
        ]
    ).all()
)


heldout_importance_finite = bool(
    np.isfinite(
        part6a_integrated_feature_importance[
            "Heldout_AUC_Importance_Mean"
        ]
    ).all()
)


temporal_importance_finite = bool(
    np.isfinite(
        part6a_integrated_feature_importance[
            "Temporal_AUC_Importance_Mean"
        ]
    ).all()
)


heldout_data_unchanged = bool(
    len(
        X_heldout
    )
    ==
    len(
        y_heldout
    )
)


temporal_data_unchanged = bool(
    len(
        X_temporal
    )
    ==
    len(
        y_temporal
    )
)


final_model_matches_part5c = bool(
    FINAL_SCREENING_MODEL
    ==
    part5c_final_recommendation[
        "Primary_Recommended_Strategy"
    ]
    and
    FINAL_ALGORITHM
    ==
    part5c_final_recommendation[
        "Primary_Recommended_Algorithm"
    ]
)


part6a_verification = pd.DataFrame(
    [
        {
            "Final_Model_Matches_PART5C":
                final_model_matches_part5c,

            "All_Features_Present":
                all_features_present,

            "Feature_Count_Correct":
                feature_count_correct,

            "Intrinsic_Importance_Finite":
                intrinsic_importance_finite,

            "Heldout_Permutation_Importance_Finite":
                heldout_importance_finite,

            "Temporal_Permutation_Importance_Finite":
                temporal_importance_finite,

            "Heldout_Data_Aligned":
                heldout_data_unchanged,

            "Temporal_Data_Aligned":
                temporal_data_unchanged,

            "No_Model_Refit":
                True,

            "No_Threshold_Modification":
                True,

            "No_Recalibration":
                True,

            "No_Feature_Selection":
                True,

            "Interpretation_Is_Predictive_Not_Causal":
                True,

            "PART6A_Ready":
                bool(
                    final_model_matches_part5c
                    and
                    all_features_present
                    and
                    feature_count_correct
                    and
                    intrinsic_importance_finite
                    and
                    heldout_importance_finite
                    and
                    temporal_importance_finite
                    and
                    heldout_data_unchanged
                    and
                    temporal_data_unchanged
                )
        }
    ]
)


# ============================================================
# 19. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 100)
print("PRIMARY FINAL-MODEL INTERPRETATION")
print("=" * 100)


display(
    part6a_primary_interpretation_table
    .round(
        5
    )
)


print("\n" + "=" * 100)
print("IMPORTANCE CONSISTENCY")
print("=" * 100)


display(
    part6a_importance_consistency
    .round(
        4
    )
)


print("\n" + "=" * 100)
print("PART 6A VERIFICATION")
print("=" * 100)


display(
    part6a_verification
)


# ============================================================
# 20. HARD SAFETY CHECK
# ============================================================

if not part6a_verification[
    "PART6A_Ready"
].all():

    raise RuntimeError(
        "PART 6A verification failed."
    )


# ============================================================
# 21. EXPORT
# ============================================================

part6a_intrinsic_importance.to_csv(
    "part6a_intrinsic_feature_importance.csv",
    index=False,
    encoding="utf-8-sig"
)


part6a_heldout_permutation_auc.to_csv(
    "part6a_heldout_permutation_importance_auc.csv",
    index=False,
    encoding="utf-8-sig"
)


part6a_heldout_permutation_ap.to_csv(
    "part6a_heldout_permutation_importance_ap.csv",
    index=False,
    encoding="utf-8-sig"
)


part6a_temporal_permutation_auc.to_csv(
    "part6a_temporal_permutation_importance_auc.csv",
    index=False,
    encoding="utf-8-sig"
)


part6a_temporal_permutation_ap.to_csv(
    "part6a_temporal_permutation_importance_ap.csv",
    index=False,
    encoding="utf-8-sig"
)


part6a_integrated_feature_importance.to_csv(
    "part6a_integrated_feature_importance.csv",
    index=False,
    encoding="utf-8-sig"
)


part6a_primary_interpretation_table.to_csv(
    "part6a_primary_interpretation_table.csv",
    index=False,
    encoding="utf-8-sig"
)


part6a_importance_consistency.to_csv(
    "part6a_importance_consistency.csv",
    index=False,
    encoding="utf-8-sig"
)


part6a_verification.to_csv(
    "part6a_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 22. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 6A VERSION 3.0 COMPLETED")
print("=" * 100)


print(
    f"Final model              : "
    f"{FINAL_SCREENING_MODEL} + {FINAL_ALGORITHM}"
)


print(
    f"Features interpreted     : "
    f"{len(final_features)}"
)


print(
    f"Permutation repeats      : "
    f"{N_PERMUTATION_REPEATS}"
)


print(
    "Primary interpretation   : "
    "Held-out 2024 permutation importance"
)


print(
    "Temporal role            : "
    "Feature-importance transportability check"
)


print("\nArtifacts created:")

print("- part6a_intrinsic_importance")
print("- part6a_heldout_permutation_importance")
print("- part6a_temporal_permutation_importance")
print("- part6a_integrated_feature_importance")
print("- part6a_primary_interpretation_table")
print("- part6a_importance_consistency")
print("- part6a_verification")


print("\nExported files:")

print("- part6a_intrinsic_feature_importance.csv")
print("- part6a_heldout_permutation_importance_auc.csv")
print("- part6a_heldout_permutation_importance_ap.csv")
print("- part6a_temporal_permutation_importance_auc.csv")
print("- part6a_temporal_permutation_importance_ap.csv")
print("- part6a_integrated_feature_importance.csv")
print("- part6a_primary_interpretation_table.csv")
print("- part6a_importance_consistency.csv")
print("- part6a_verification.csv")


print(
    f"- PNG plots in "
    f"{PLOT_OUTPUT_DIRECTORY}/"
)


print(
    "\nNo model was refitted, no threshold was modified, "
    "and no feature was selected or removed."
)


print(
    "Feature importance describes predictive contribution "
    "within the fitted model and must not be interpreted "
    "as a causal effect or etiologic risk estimate."
)

In [ ]:
display(
    part6a_primary_interpretation_table.round(5)
)

display(
    part6a_importance_consistency.round(4)
)

### PART 6B: Clinical and Screening Interpretation of the Final Low-Burden Strategy

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 6B: CLINICAL AND SCREENING INTERPRETATION
# OF THE FINAL LOW-BURDEN STRATEGY
# ============================================================
#
# Final strategy:
# - M2_Admin_2Q
# - AdaBoostClassifier
#
# Purpose:
# Convert the locked model-selection and interpretation results
# from PART 5C and PART 6A into publication-ready clinical and
# screening interpretation outputs.
#
# Inputs:
# - part5c_final_recommendation
# - part5c_strategy_roles
# - part5c_recommendation_summary
# - part6a_primary_interpretation_table
# - part6a_importance_consistency
# - part6a_verification
#
# Outputs:
# - part6b_final_strategy_evidence
# - part6b_feature_interpretation
# - part6b_temporal_consistency
# - part6b_clinical_positioning
# - part6b_publication_table
# - part6b_key_findings
# - part6b_manuscript_text
# - part6b_verification
#
# IMPORTANT:
# - No model fitting
# - No prediction generation
# - No threshold modification
# - No recalibration
# - No new feature selection
# - Predictive importance must NOT be interpreted causally
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 2. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "part5c_final_recommendation",
    "part5c_strategy_roles",
    "part5c_recommendation_summary",
    "part6a_primary_interpretation_table",
    "part6a_importance_consistency",
    "part6a_verification"
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(
            missing_objects
        )
    )


for object_name in [
    "part5c_strategy_roles",
    "part5c_recommendation_summary",
    "part6a_primary_interpretation_table",
    "part6a_importance_consistency",
    "part6a_verification"
]:

    if not isinstance(
        globals()[object_name],
        pd.DataFrame
    ):

        raise TypeError(
            f"{object_name} must be a pandas DataFrame."
        )


if not part6a_verification[
    "PART6A_Ready"
].all():

    raise RuntimeError(
        "PART 6A is not fully ready."
    )


# ============================================================
# 3. FINAL MODEL CONSTANTS
# ============================================================

FINAL_SCREENING_MODEL = (
    part5c_final_recommendation[
        "Primary_Recommended_Strategy"
    ]
)


FINAL_ALGORITHM = (
    part5c_final_recommendation[
        "Primary_Recommended_Algorithm"
    ]
)


SELF_REPORT_ITEMS = int(
    part5c_final_recommendation[
        "Self_Report_Items"
    ]
)


FEATURE_COUNT = int(
    part5c_final_recommendation[
        "Feature_Count"
    ]
)


if FINAL_SCREENING_MODEL != "M2_Admin_2Q":

    raise RuntimeError(
        "Expected final screening model M2_Admin_2Q."
    )


if FINAL_ALGORITHM != "AdaBoostClassifier":

    raise RuntimeError(
        "Expected final algorithm AdaBoostClassifier."
    )


# ============================================================
# 4. FEATURE GROUP DEFINITIONS
# ============================================================

PHQ2_FEATURES = [
    "2Q1",
    "2Q2"
]


ADMINISTRATIVE_FEATURES = [
    feature
    for feature in (
        part6a_primary_interpretation_table[
            "Feature"
        ]
        .astype(str)
        .tolist()
    )
    if feature not in PHQ2_FEATURES
]


# ============================================================
# 5. VALIDATE PHQ-2 FEATURES
# ============================================================

missing_phq2 = [
    feature
    for feature in PHQ2_FEATURES
    if feature not in (
        part6a_primary_interpretation_table[
            "Feature"
        ]
        .astype(str)
        .tolist()
    )
]


if missing_phq2:

    raise RuntimeError(
        "Missing PHQ-2 interpretation features: "
        + ", ".join(
            missing_phq2
        )
    )


# ============================================================
# 6. PRIMARY HELD-OUT PERMUTATION RANKING
# ============================================================

feature_table = (
    part6a_primary_interpretation_table
    .copy()
)


feature_table = (
    feature_table
    .sort_values(
        by=[
            "Heldout_AUC_Rank",
            "Feature"
        ]
    )
    .reset_index(
        drop=True
    )
)


top_feature = feature_table.iloc[
    0
]


second_feature = feature_table.iloc[
    1
]


phq2_table = (
    feature_table[
        feature_table[
            "Feature"
        ].isin(
            PHQ2_FEATURES
        )
    ]
    .copy()
)


administrative_table = (
    feature_table[
        ~feature_table[
            "Feature"
        ].isin(
            PHQ2_FEATURES
        )
    ]
    .copy()
)


# ============================================================
# 7. FINAL STRATEGY EVIDENCE TABLE
# ============================================================

part6b_final_strategy_evidence = pd.DataFrame(
    [
        {
            "Domain":
                "Final strategy",

            "Evidence":
                (
                    f"{FINAL_SCREENING_MODEL} + "
                    f"{FINAL_ALGORITHM}"
                ),

            "Interpretation":
                (
                    "Primary recommended low-burden "
                    "screening strategy."
                )
        },

        {
            "Domain":
                "Respondent burden",

            "Evidence":
                (
                    f"{SELF_REPORT_ITEMS} self-report items "
                    f"with {FEATURE_COUNT} total predictors."
                ),

            "Interpretation":
                (
                    "Adds only the two PHQ-2 symptom questions "
                    "to routinely available administrative data."
                )
        },

        {
            "Domain":
                "Held-out discrimination",

            "Evidence":
                str(
                    part5c_final_recommendation[
                        "Heldout_AUC_95CI"
                    ]
                ),

            "Interpretation":
                (
                    "Demonstrated discrimination in the "
                    "independent 2024 held-out cohort."
                )
        },

        {
            "Domain":
                "Held-out sensitivity",

            "Evidence":
                str(
                    part5c_final_recommendation[
                        "Heldout_Sensitivity_95CI"
                    ]
                ),

            "Interpretation":
                (
                    "Sensitivity was evaluated using the "
                    "prespecified OOF-locked threshold."
                )
        },

        {
            "Domain":
                "Temporal discrimination",

            "Evidence":
                str(
                    part5c_final_recommendation[
                        "Temporal_AUC_95CI"
                    ]
                ),

            "Interpretation":
                (
                    "Performance was supported in the "
                    "independent 2025 temporal cohort."
                )
        },

        {
            "Domain":
                "Temporal sensitivity",

            "Evidence":
                str(
                    part5c_final_recommendation[
                        "Temporal_Sensitivity_95CI"
                    ]
                ),

            "Interpretation":
                (
                    "Temporal sensitivity was assessed without "
                    "retuning the classification threshold."
                )
        },

        {
            "Domain":
                "Burden-performance framework",

            "Evidence":
                (
                    f"Pareto-efficient = "
                    f"{part5c_final_recommendation['Pareto_Efficient']}; "
                    f"validation support = "
                    f"{part5c_final_recommendation['Validation_Support_Summary']}"
                ),

            "Interpretation":
                (
                    "The final strategy was selected for "
                    "burden-performance balance rather than "
                    "for the highest single metric."
                )
        }
    ]
)


# ============================================================
# 8. FEATURE INTERPRETATION TABLE
# ============================================================

feature_interpretation_rows = []


for _, row in feature_table.iterrows():

    feature = str(
        row[
            "Feature"
        ]
    )


    heldout_importance = float(
        row[
            "Heldout_AUC_Importance_Mean"
        ]
    )


    heldout_rank = int(
        row[
            "Heldout_AUC_Rank"
        ]
    )


    temporal_importance = float(
        row[
            "Temporal_AUC_Importance_Mean"
        ]
    )


    temporal_rank = int(
        row[
            "Temporal_AUC_Rank"
        ]
    )


    intrinsic_importance = float(
        row[
            "Intrinsic_Importance"
        ]
    )


    intrinsic_rank = int(
        row[
            "Intrinsic_Rank"
        ]
    )


    if feature in PHQ2_FEATURES:

        feature_group = "PHQ-2"

    else:

        feature_group = "Administrative"


    if heldout_importance > 0:

        heldout_signal = (
            "Positive out-of-sample contribution"
        )

    elif np.isclose(
        heldout_importance,
        0.0,
        atol=1e-12
    ):

        heldout_signal = (
            "No detectable incremental contribution"
        )

    else:

        heldout_signal = (
            "No positive incremental contribution"
        )


    feature_interpretation_rows.append({

        "Feature":
            feature,

        "Feature_Group":
            feature_group,

        "Intrinsic_Importance":
            intrinsic_importance,

        "Intrinsic_Rank":
            intrinsic_rank,

        "Heldout_AUC_Importance":
            heldout_importance,

        "Heldout_AUC_Rank":
            heldout_rank,

        "Temporal_AUC_Importance":
            temporal_importance,

        "Temporal_AUC_Rank":
            temporal_rank,

        "Consistent_Positive_Importance":
            bool(
                row[
                    "Importance_Consistent_Positive"
                ]
            ),

        "Primary_Interpretation":
            heldout_signal,

        "Causal_Interpretation_Allowed":
            False
    })


part6b_feature_interpretation = pd.DataFrame(
    feature_interpretation_rows
)


# ============================================================
# 9. PHQ-2 INCREMENTAL VALUE SUMMARY
# ============================================================

phq2_heldout_importance_sum = float(
    phq2_table[
        "Heldout_AUC_Importance_Mean"
    ].sum()
)


admin_positive_heldout_importance_sum = float(
    administrative_table.loc[
        administrative_table[
            "Heldout_AUC_Importance_Mean"
        ]
        >
        0,
        "Heldout_AUC_Importance_Mean"
    ].sum()
)


top_two_are_phq2 = bool(
    set(
        feature_table.iloc[
            :2
        ][
            "Feature"
        ]
        .astype(str)
        .tolist()
    )
    ==
    set(
        PHQ2_FEATURES
    )
)


part6b_phq2_incremental_value = pd.DataFrame(
    [
        {
            "Indicator":
                "PHQ-2 features",

            "Value":
                ", ".join(
                    PHQ2_FEATURES
                ),

            "Interpretation":
                (
                    "The only additional self-report variables "
                    "in the final low-burden strategy."
                )
        },

        {
            "Indicator":
                "Top two held-out permutation features are PHQ-2",

            "Value":
                top_two_are_phq2,

            "Interpretation":
                (
                    "The two PHQ-2 items occupied the top two "
                    "held-out AUC permutation-importance ranks."
                )
        },

        {
            "Indicator":
                "Combined PHQ-2 held-out AUC importance",

            "Value":
                phq2_heldout_importance_sum,

            "Interpretation":
                (
                    "Descriptive sum of held-out AUC decreases "
                    "after separately permuting 2Q1 and 2Q2; "
                    "not an additive causal effect."
                )
        },

        {
            "Indicator":
                "Combined positive administrative held-out importance",

            "Value":
                admin_positive_heldout_importance_sum,

            "Interpretation":
                (
                    "Administrative variables provided smaller "
                    "supplementary predictive information."
                )
        }
    ]
)


# ============================================================
# 10. TEMPORAL IMPORTANCE CONSISTENCY
# ============================================================

heldout_temporal_rows = (
    part6a_importance_consistency[
        part6a_importance_consistency[
            "Comparison"
        ]
        .astype(str)
        .str.contains(
            "Held-out AUC permutation vs Temporal AUC",
            regex=False
        )
    ]
)


if len(
    heldout_temporal_rows
) != 1:

    raise RuntimeError(
        "Could not uniquely identify held-out vs temporal "
        "importance consistency result."
    )


heldout_temporal_row = (
    heldout_temporal_rows.iloc[
        0
    ]
)


importance_rho = float(
    heldout_temporal_row[
        "Spearman_Rho"
    ]
)


importance_p = float(
    heldout_temporal_row[
        "P_Value"
    ]
)


if importance_rho >= 0.70:

    consistency_label = (
        "Strong positive rank consistency"
    )

elif importance_rho >= 0.50:

    consistency_label = (
        "Moderate positive rank consistency"
    )

elif importance_rho > 0:

    consistency_label = (
        "Weak positive rank consistency"
    )

else:

    consistency_label = (
        "No positive rank consistency"
    )


part6b_temporal_consistency = pd.DataFrame(
    [
        {
            "Comparison":
                (
                    "Held-out 2024 vs Temporal 2025 "
                    "AUC permutation-importance ranks"
                ),

            "Spearman_Rho":
                importance_rho,

            "P_Value":
                importance_p,

            "Interpretation":
                consistency_label,

            "Role":
                (
                    "Temporal consistency / transportability "
                    "check; not algorithm selection."
                )
        }
    ]
)


# ============================================================
# 11. CLINICAL / SCREENING POSITIONING
# ============================================================

part6b_clinical_positioning = pd.DataFrame(
    [
        {
            "Domain":
                "Intended use",

            "Position":
                (
                    "Low-burden screening and triage for "
                    "identifying students who may warrant "
                    "further clinical assessment."
                )
        },

        {
            "Domain":
                "Not intended for",

            "Position":
                (
                    "Replacing a clinical interview, psychiatric "
                    "assessment, or formal diagnosis."
                )
        },

        {
            "Domain":
                "Threshold use",

            "Position":
                (
                    "Use the classification threshold locked "
                    "from genuine 2024 OOF predictions; do not "
                    "re-optimize the threshold in deployment "
                    "without a new prespecified validation study."
                )
        },

        {
            "Domain":
                "Feature interpretation",

            "Position":
                (
                    "Feature importance indicates predictive "
                    "contribution within the fitted model and "
                    "must not be interpreted as causation or "
                    "etiologic risk."
                )
        },

        {
            "Domain":
                "PHQ-2 interpretation",

            "Position":
                (
                    "The two PHQ-2 symptom items supplied the "
                    "strongest held-out predictive signal and "
                    "appear to provide substantial incremental "
                    "information beyond administrative data."
                )
        },

        {
            "Domain":
                "Administrative variables",

            "Position":
                (
                    "Administrative variables provided smaller "
                    "supplementary predictive information and "
                    "should not be overinterpreted individually."
                )
        },

        {
            "Domain":
                "Temporal generalization",

            "Position":
                (
                    "Temporal 2025 results support transportability "
                    "within the study setting, but do not constitute "
                    "validation across unrelated institutions or "
                    "health systems."
                )
        }
    ]
)


# ============================================================
# 12. PUBLICATION-READY INTERPRETATION TABLE
# ============================================================

part6b_publication_table = pd.DataFrame(
    [
        {
            "Component":
                "Selected screening strategy",

            "Evidence":
                (
                    f"{FINAL_SCREENING_MODEL} + "
                    f"{FINAL_ALGORITHM}"
                ),

            "Interpretation":
                (
                    "Primary recommended low-burden screening "
                    "strategy."
                )
        },

        {
            "Component":
                "Additional respondent burden",

            "Evidence":
                f"{SELF_REPORT_ITEMS} self-report items",

            "Interpretation":
                (
                    "Only the two PHQ-2 symptom questions are "
                    "added to routinely available administrative "
                    "information."
                )
        },

        {
            "Component":
                "Strongest predictive contributor",

            "Evidence":
                (
                    f"{top_feature['Feature']}: held-out "
                    f"ΔAUC={top_feature['Heldout_AUC_Importance_Mean']:.3f}, "
                    f"rank {int(top_feature['Heldout_AUC_Rank'])}"
                ),

            "Interpretation":
                (
                    "Largest decrease in held-out discrimination "
                    "when permuted."
                )
        },

        {
            "Component":
                "Second strongest predictive contributor",

            "Evidence":
                (
                    f"{second_feature['Feature']}: held-out "
                    f"ΔAUC={second_feature['Heldout_AUC_Importance_Mean']:.3f}, "
                    f"rank {int(second_feature['Heldout_AUC_Rank'])}"
                ),

            "Interpretation":
                (
                    "Second-largest held-out permutation "
                    "contribution."
                )
        },

        {
            "Component":
                "Administrative contribution",

            "Evidence":
                (
                    "Smaller or null held-out permutation "
                    "importance for most administrative variables."
                ),

            "Interpretation":
                (
                    "Administrative information appears "
                    "supplementary rather than the dominant "
                    "predictive signal."
                )
        },

        {
            "Component":
                "Temporal feature-importance consistency",

            "Evidence":
                (
                    f"Spearman rho={importance_rho:.3f}, "
                    f"p={importance_p:.4f}"
                ),

            "Interpretation":
                (
                    "Feature-importance ranking showed positive "
                    "consistency between held-out 2024 and "
                    "temporal 2025 cohorts."
                )
        },

        {
            "Component":
                "Clinical role",

            "Evidence":
                "Screening/triage",

            "Interpretation":
                (
                    "Model output should trigger further "
                    "assessment rather than serve as a diagnosis."
                )
        }
    ]
)


# ============================================================
# 13. MANUSCRIPT-READY KEY FINDINGS
# ============================================================

part6b_key_findings = pd.DataFrame(
    [
        {
            "Key_Finding":
                (
                    "The final burden-efficient strategy combined "
                    "routinely available administrative variables "
                    "with only two PHQ-2 symptom items."
                )
        },

        {
            "Key_Finding":
                (
                    f"{top_feature['Feature']} and "
                    f"{second_feature['Feature']} were the two "
                    "highest-ranked features by held-out AUC "
                    "permutation importance."
                )
        },

        {
            "Key_Finding":
                (
                    "Most administrative variables showed smaller "
                    "or negligible independent permutation "
                    "contributions in the held-out cohort."
                )
        },

        {
            "Key_Finding":
                (
                    f"Feature-importance ranks were positively "
                    f"consistent across held-out 2024 and temporal "
                    f"2025 cohorts (Spearman rho={importance_rho:.3f}, "
                    f"p={importance_p:.4f})."
                )
        },

        {
            "Key_Finding":
                (
                    "The model should be interpreted as a "
                    "screening and triage aid rather than a "
                    "diagnostic substitute."
                )
        }
    ]
)


# ============================================================
# 14. MANUSCRIPT-READY TEXT
# ============================================================

part6b_manuscript_text = {

    "Results_Interpretation":
        (
            f"The final recommended low-burden strategy was "
            f"{FINAL_SCREENING_MODEL} using "
            f"{FINAL_ALGORITHM}. This strategy added only "
            f"{SELF_REPORT_ITEMS} self-report items to routinely "
            "available administrative information. In the "
            "held-out 2024 cohort, the two PHQ-2 items "
            f"({top_feature['Feature']} and "
            f"{second_feature['Feature']}) had the highest "
            "permutation importance for ROC AUC, indicating that "
            "they supplied the strongest incremental predictive "
            "information within the final model. Most "
            "administrative variables showed smaller or negligible "
            "held-out permutation contributions. Feature-importance "
            "rankings were positively consistent between the "
            f"held-out 2024 and temporal 2025 cohorts "
            f"(Spearman rho={importance_rho:.3f}, "
            f"p={importance_p:.4f}), supporting temporal "
            "consistency of the model's predictive structure."
        ),

    "Clinical_Interpretation":
        (
            "These findings support the use of a two-item symptom "
            "assessment combined with routinely available "
            "administrative information as a low-burden strategy "
            "for identifying students who may warrant further "
            "clinical assessment. The model is intended for "
            "screening and triage and should not replace clinical "
            "interview, psychiatric assessment, or formal diagnosis. "
            "Feature-importance estimates represent predictive "
            "contribution within the fitted model and should not be "
            "interpreted as causal or etiologic effects."
        )
}


# ============================================================
# 15. VERIFICATION
# ============================================================

top_two_are_phq2_verified = bool(
    set(
        feature_table.iloc[
            :2
        ][
            "Feature"
        ]
        .astype(str)
        .tolist()
    )
    ==
    set(
        PHQ2_FEATURES
    )
)


importance_consistency_finite = bool(
    np.isfinite(
        importance_rho
    )
    and
    np.isfinite(
        importance_p
    )
)


all_feature_rows_present = bool(
    len(
        part6b_feature_interpretation
    )
    ==
    FEATURE_COUNT
)


all_causal_flags_false = bool(
    (
        part6b_feature_interpretation[
            "Causal_Interpretation_Allowed"
        ]
        ==
        False
    ).all()
)


part6b_verification = pd.DataFrame(
    [
        {
            "Final_Strategy_Is_M2_Admin_2Q":
                FINAL_SCREENING_MODEL
                ==
                "M2_Admin_2Q",

            "Final_Algorithm_Is_AdaBoost":
                FINAL_ALGORITHM
                ==
                "AdaBoostClassifier",

            "Self_Report_Items_Is_2":
                SELF_REPORT_ITEMS
                ==
                2,

            "All_Feature_Rows_Present":
                all_feature_rows_present,

            "Top_Two_Heldout_Features_Are_PHQ2":
                top_two_are_phq2_verified,

            "Temporal_Importance_Consistency_Estimable":
                importance_consistency_finite,

            "Predictive_Not_Causal_Interpretation_Enforced":
                all_causal_flags_false,

            "Clinical_Role_Is_Screening_Not_Diagnosis":
                True,

            "No_Model_Refit":
                True,

            "No_New_Predictions":
                True,

            "No_Threshold_Modification":
                True,

            "No_Recalibration":
                True,

            "No_Feature_Selection":
                True,

            "PART6B_Ready":
                bool(
                    FINAL_SCREENING_MODEL
                    ==
                    "M2_Admin_2Q"
                    and
                    FINAL_ALGORITHM
                    ==
                    "AdaBoostClassifier"
                    and
                    SELF_REPORT_ITEMS
                    ==
                    2
                    and
                    all_feature_rows_present
                    and
                    top_two_are_phq2_verified
                    and
                    importance_consistency_finite
                    and
                    all_causal_flags_false
                )
        }
    ]
)


# ============================================================
# 16. DISPLAY
# ============================================================

print("=" * 100)
print(
    "PART 6B VERSION 3.0: CLINICAL AND SCREENING "
    "INTERPRETATION"
)
print("=" * 100)


print(
    f"Final strategy : "
    f"{FINAL_SCREENING_MODEL}"
)


print(
    f"Algorithm      : "
    f"{FINAL_ALGORITHM}"
)


print(
    f"Self-report items : "
    f"{SELF_REPORT_ITEMS}"
)


print("\n" + "=" * 100)
print("FINAL STRATEGY EVIDENCE")
print("=" * 100)


display(
    part6b_final_strategy_evidence
)


print("\n" + "=" * 100)
print("FEATURE INTERPRETATION")
print("=" * 100)


display(
    part6b_feature_interpretation
    .round(
        5
    )
)


print("\n" + "=" * 100)
print("PHQ-2 INCREMENTAL VALUE")
print("=" * 100)


display(
    part6b_phq2_incremental_value
    .round(
        5
    )
)


print("\n" + "=" * 100)
print("TEMPORAL IMPORTANCE CONSISTENCY")
print("=" * 100)


display(
    part6b_temporal_consistency
    .round(
        4
    )
)


print("\n" + "=" * 100)
print("CLINICAL / SCREENING POSITIONING")
print("=" * 100)


display(
    part6b_clinical_positioning
)


print("\n" + "=" * 100)
print("PUBLICATION-READY INTERPRETATION TABLE")
print("=" * 100)


display(
    part6b_publication_table
)


print("\n" + "=" * 100)
print("KEY FINDINGS")
print("=" * 100)


display(
    part6b_key_findings
)


print("\n" + "=" * 100)
print("PART 6B VERIFICATION")
print("=" * 100)


display(
    part6b_verification
)


# ============================================================
# 17. HARD SAFETY CHECK
# ============================================================

if not part6b_verification[
    "PART6B_Ready"
].all():

    raise RuntimeError(
        "PART 6B verification failed."
    )


# ============================================================
# 18. EXPORT
# ============================================================

part6b_final_strategy_evidence.to_csv(
    "part6b_final_strategy_evidence.csv",
    index=False,
    encoding="utf-8-sig"
)


part6b_feature_interpretation.to_csv(
    "part6b_feature_interpretation.csv",
    index=False,
    encoding="utf-8-sig"
)


part6b_phq2_incremental_value.to_csv(
    "part6b_phq2_incremental_value.csv",
    index=False,
    encoding="utf-8-sig"
)


part6b_temporal_consistency.to_csv(
    "part6b_temporal_consistency.csv",
    index=False,
    encoding="utf-8-sig"
)


part6b_clinical_positioning.to_csv(
    "part6b_clinical_positioning.csv",
    index=False,
    encoding="utf-8-sig"
)


part6b_publication_table.to_csv(
    "part6b_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


part6b_key_findings.to_csv(
    "part6b_key_findings.csv",
    index=False,
    encoding="utf-8-sig"
)


pd.DataFrame(
    [
        part6b_manuscript_text
    ]
).to_csv(
    "part6b_manuscript_text.csv",
    index=False,
    encoding="utf-8-sig"
)


part6b_verification.to_csv(
    "part6b_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("PART 6B VERSION 3.0 COMPLETED")
print("=" * 100)


print(
    f"Final screening strategy : "
    f"{FINAL_SCREENING_MODEL}"
)


print(
    f"Final algorithm          : "
    f"{FINAL_ALGORITHM}"
)


print(
    f"Self-report items        : "
    f"{SELF_REPORT_ITEMS}"
)


print(
    f"Top held-out features    : "
    f"{top_feature['Feature']}, "
    f"{second_feature['Feature']}"
)


print(
    f"Temporal rank consistency: "
    f"rho={importance_rho:.3f}, "
    f"p={importance_p:.4f}"
)


print("\nArtifacts created:")

print("- part6b_final_strategy_evidence")
print("- part6b_feature_interpretation")
print("- part6b_phq2_incremental_value")
print("- part6b_temporal_consistency")
print("- part6b_clinical_positioning")
print("- part6b_publication_table")
print("- part6b_key_findings")
print("- part6b_manuscript_text")
print("- part6b_verification")


print("\nExported files:")

print("- part6b_final_strategy_evidence.csv")
print("- part6b_feature_interpretation.csv")
print("- part6b_phq2_incremental_value.csv")
print("- part6b_temporal_consistency.csv")
print("- part6b_clinical_positioning.csv")
print("- part6b_publication_table.csv")
print("- part6b_key_findings.csv")
print("- part6b_manuscript_text.csv")
print("- part6b_verification.csv")


print(
    "\nNo model was refitted, no new predictions were generated, "
    "no threshold was modified, and no recalibration or feature "
    "selection was performed."
)


print(
    "PART 6B is an interpretation and reporting layer only."
)

## PART 7: Fairness / Subgroup Evaluation of the Final Recommended Model

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 7A: FAIRNESS / SUBGROUP EVALUATION
# FINAL RECOMMENDED MODEL ONLY
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

FINAL_SCREENING_MODEL = None
FINAL_ALGORITHM = None
MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS = 5

PROTECTED_VARIABLES = [
    "gender",
    "institute",
    "parent",
    "momeducation",
    "dadeducation",
    "momstatus",
    "dadstatus"
]

VALIDATION_SETS = [
    "Held-out 2024",
    "Temporal 2025"
]

required_objects = [
    "part5c_final_recommendation",
    "heldout_predictions",
    "temporal_predictions",
    "prepared_data",
    "temporal_datasets",
    "screening_strategies",
    "OUTCOME"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )

FINAL_SCREENING_MODEL = str(
    part5c_final_recommendation[
        "Primary_Recommended_Strategy"
    ]
)

FINAL_ALGORITHM = str(
    part5c_final_recommendation[
        "Primary_Recommended_Algorithm"
    ]
)

if (
    part5c_final_recommendation["Primary_Recommended_Strategy"]
    != FINAL_SCREENING_MODEL
):
    raise RuntimeError(
        "Final strategy no longer matches M2_Admin_2Q."
    )

if (
    part5c_final_recommendation["Primary_Recommended_Algorithm"]
    != FINAL_ALGORITHM
):
    raise RuntimeError(
        "Final algorithm no longer matches AdaBoostClassifier."
    )

# ------------------------------------------------------------
# Fixed held-out predictions
# ------------------------------------------------------------

heldout_record = heldout_predictions[
    FINAL_SCREENING_MODEL
][
    FINAL_ALGORITHM
]

heldout_y_true = np.asarray(
    heldout_record["y_true"],
    dtype=int
)

heldout_y_prob = np.asarray(
    heldout_record["y_prob"],
    dtype=float
)

heldout_y_pred = np.asarray(
    heldout_record["y_pred"],
    dtype=int
)

heldout_X = (
    prepared_data[
        FINAL_SCREENING_MODEL
    ][
        "X_test"
    ]
    .copy()
    .reset_index(drop=True)
)

heldout_outcome_check = (
    prepared_data[FINAL_SCREENING_MODEL]["y_test"]
    .astype(int)
    .copy()
    .reset_index(drop=True)
    .to_numpy(dtype=int)
)

if len(heldout_X) != len(heldout_y_true):
    raise RuntimeError(
        "Held-out subgroup data and prediction lengths differ."
    )

if not (
    len(heldout_y_prob)
    == len(heldout_y_pred)
    == len(heldout_y_true)
):
    raise RuntimeError(
        "Held-out probability, prediction, and outcome lengths differ."
    )

if not np.array_equal(heldout_outcome_check, heldout_y_true):
    raise RuntimeError(
        "Held-out subgroup rows are not aligned with fixed outcomes."
    )

# ------------------------------------------------------------
# Fixed temporal predictions
# ------------------------------------------------------------

temporal_record = temporal_predictions[
    FINAL_SCREENING_MODEL
][
    FINAL_ALGORITHM
]

temporal_y_true = np.asarray(
    temporal_record["y_true"],
    dtype=int
)

temporal_y_prob = np.asarray(
    temporal_record["y_prob"],
    dtype=float
)

temporal_y_pred = np.asarray(
    temporal_record["y_pred"],
    dtype=int
)

temporal_X = (
    temporal_datasets[
        FINAL_SCREENING_MODEL
    ]
    .drop(columns=[OUTCOME])
    .copy()
    .reset_index(drop=True)
)

temporal_outcome_check = (
    temporal_datasets[FINAL_SCREENING_MODEL][OUTCOME]
    .astype(int)
    .copy()
    .reset_index(drop=True)
    .to_numpy(dtype=int)
)

if len(temporal_X) != len(temporal_y_true):
    raise RuntimeError(
        "Temporal subgroup data and prediction lengths differ."
    )

if not (
    len(temporal_y_prob)
    == len(temporal_y_pred)
    == len(temporal_y_true)
):
    raise RuntimeError(
        "Temporal probability, prediction, and outcome lengths differ."
    )

if not np.array_equal(temporal_outcome_check, temporal_y_true):
    raise RuntimeError(
        "Temporal subgroup rows are not aligned with fixed outcomes."
    )

# Verify reuse of one development-derived locked threshold.
heldout_locked_threshold = float(
    heldout_record.get(
        "locked_threshold",
        heldout_record.get("threshold", np.nan)
    )
)

temporal_locked_threshold = float(
    temporal_record.get(
        "locked_threshold",
        temporal_record.get("threshold", np.nan)
    )
)

if not np.isfinite(heldout_locked_threshold):
    raise RuntimeError("Held-out locked threshold is invalid.")

if not np.isfinite(temporal_locked_threshold):
    raise RuntimeError("Temporal locked threshold is invalid.")

if not np.isclose(
    heldout_locked_threshold,
    temporal_locked_threshold,
    rtol=0,
    atol=1e-15
):
    raise RuntimeError(
        "Held-out and temporal locked thresholds do not match."
    )

if not np.array_equal(
    heldout_y_pred,
    (heldout_y_prob >= heldout_locked_threshold).astype(int)
):
    raise RuntimeError(
        "Held-out classifications do not match the locked threshold."
    )

if not np.array_equal(
    temporal_y_pred,
    (temporal_y_prob >= temporal_locked_threshold).astype(int)
):
    raise RuntimeError(
        "Temporal classifications do not match the locked threshold."
    )

for variable in PROTECTED_VARIABLES:
    if variable not in heldout_X.columns:
        raise KeyError(
            f"Held-out data missing subgroup variable '{variable}'."
        )
    if variable not in temporal_X.columns:
        raise KeyError(
            f"Temporal data missing subgroup variable '{variable}'."
        )

def safe_ratio(numerator, denominator):
    if denominator <= 0:
        return np.nan
    return float(numerator / denominator)

def calculate_subgroup_metrics(
    y_true,
    y_prob,
    y_pred,
    minimum_positive_cases=5
):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = np.asarray(y_pred, dtype=int)

    n = int(len(y_true))
    positive_cases = int(np.sum(y_true == 1))
    negative_cases = int(np.sum(y_true == 0))

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = safe_ratio(
        tn,
        tn + fp
    )

    false_positive_rate = safe_ratio(
        fp,
        fp + tn
    )

    negative_predictive_value = safe_ratio(
        tn,
        tn + fn
    )

    predicted_positive_rate = float(
        np.mean(y_pred == 1)
    )

    stable_event_metrics = bool(
        positive_cases >= minimum_positive_cases
    )

    if stable_event_metrics:
        sensitivity = recall_score(
            y_true,
            y_pred,
            zero_division=0
        )
        precision = precision_score(
            y_true,
            y_pred,
            zero_division=0
        )
        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

        if len(np.unique(y_true)) == 2:
            auc = roc_auc_score(
                y_true,
                y_prob
            )
            average_precision = average_precision_score(
                y_true,
                y_prob
            )
        else:
            auc = np.nan
            average_precision = np.nan
    else:
        sensitivity = np.nan
        precision = np.nan
        f1 = np.nan
        auc = np.nan
        average_precision = np.nan

    return {
        "N": n,
        "Positive_Cases": positive_cases,
        "Negative_Cases": negative_cases,
        "Prevalence": safe_ratio(positive_cases, n),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Sensitivity": (
            float(sensitivity)
            if np.isfinite(sensitivity)
            else np.nan
        ),
        "Specificity": (
            float(specificity)
            if np.isfinite(specificity)
            else np.nan
        ),
        "Precision": (
            float(precision)
            if np.isfinite(precision)
            else np.nan
        ),
        "Negative_Predictive_Value": (
            float(negative_predictive_value)
            if np.isfinite(negative_predictive_value)
            else np.nan
        ),
        "F1": (
            float(f1)
            if np.isfinite(f1)
            else np.nan
        ),
        "False_Positive_Rate": (
            float(false_positive_rate)
            if np.isfinite(false_positive_rate)
            else np.nan
        ),
        "Predicted_Positive_Rate": predicted_positive_rate,
        "AUC": (
            float(auc)
            if np.isfinite(auc)
            else np.nan
        ),
        "Average_Precision": (
            float(average_precision)
            if np.isfinite(average_precision)
            else np.nan
        ),
        "Stable_Event_Metrics": stable_event_metrics
    }

cohort_data = {
    "Held-out 2024": {
        "X": heldout_X,
        "y_true": heldout_y_true,
        "y_prob": heldout_y_prob,
        "y_pred": heldout_y_pred
    },
    "Temporal 2025": {
        "X": temporal_X,
        "y_true": temporal_y_true,
        "y_prob": temporal_y_prob,
        "y_pred": temporal_y_pred
    }
}

subgroup_rows = []
status_rows = []

print("=" * 110)
print(
    "PART 7A VERSION 3.0: FAIRNESS / SUBGROUP EVALUATION "
    "OF FINAL MODEL"
)
print("=" * 110)
print(f"Final strategy : {FINAL_SCREENING_MODEL}")
print(f"Algorithm      : {FINAL_ALGORITHM}")
print(
    "Minimum positive cases for stable event metrics: "
    f"{MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS}"
)

for validation_set, cohort in cohort_data.items():
    print("\n" + "=" * 110)
    print(validation_set.upper())
    print("=" * 110)

    X = cohort["X"]
    y_true = cohort["y_true"]
    y_prob = cohort["y_prob"]
    y_pred = cohort["y_pred"]

    for protected_variable in PROTECTED_VARIABLES:
        print(f"\nSubgroup variable: {protected_variable}")

        values = X[
            protected_variable
        ].dropna().unique().tolist()

        try:
            values = sorted(values)
        except Exception:
            values = sorted(
                values,
                key=lambda value: str(value)
            )

        for group_value in values:
            mask = (
                X[protected_variable] == group_value
            ).to_numpy()

            metrics = calculate_subgroup_metrics(
                y_true=y_true[mask],
                y_prob=y_prob[mask],
                y_pred=y_pred[mask],
                minimum_positive_cases=(
                    MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS
                )
            )

            subgroup_rows.append({
                "Validation_Set": validation_set,
                "Screening_Model": FINAL_SCREENING_MODEL,
                "Algorithm": FINAL_ALGORITHM,
                "Protected_Group": protected_variable,
                "Group": group_value,
                **metrics
            })

            stability_text = (
                "Stable"
                if metrics["Stable_Event_Metrics"]
                else (
                    "Limited: fewer than "
                    f"{MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS} "
                    "positive cases"
                )
            )

            print(
                f"  Group={group_value} | "
                f"N={metrics['N']} | "
                f"Events={metrics['Positive_Cases']} | "
                f"{stability_text}"
            )

        missing_mask = (
            X[protected_variable]
            .isna()
            .to_numpy()
        )

        if np.any(missing_mask):
            metrics = calculate_subgroup_metrics(
                y_true=y_true[missing_mask],
                y_prob=y_prob[missing_mask],
                y_pred=y_pred[missing_mask],
                minimum_positive_cases=(
                    MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS
                )
            )

            subgroup_rows.append({
                "Validation_Set": validation_set,
                "Screening_Model": FINAL_SCREENING_MODEL,
                "Algorithm": FINAL_ALGORITHM,
                "Protected_Group": protected_variable,
                "Group": "Missing",
                **metrics
            })

        status_rows.append({
            "Validation_Set": validation_set,
            "Protected_Group": protected_variable,
            "Status": "Success",
            "Observed_Groups": int(
                X[protected_variable].nunique(dropna=False)
            ),
            "Missing_N": int(
                X[protected_variable].isna().sum()
            ),
            "Error": ""
        })

part7a_subgroup_results = pd.DataFrame(
    subgroup_rows
)

part7a_status = pd.DataFrame(
    status_rows
)

gap_metrics = [
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "False_Positive_Rate",
    "Predicted_Positive_Rate",
    "AUC",
    "Average_Precision"
]

disparity_rows = []

for validation_set in VALIDATION_SETS:
    for protected_variable in PROTECTED_VARIABLES:
        subset = (
            part7a_subgroup_results[
                (
                    part7a_subgroup_results["Validation_Set"]
                    == validation_set
                )
                &
                (
                    part7a_subgroup_results["Protected_Group"]
                    == protected_variable
                )
            ]
            .copy()
        )

        for metric_name in gap_metrics:
            estimable = (
                subset[
                    ["Group", metric_name]
                ]
                .dropna(subset=[metric_name])
                .copy()
            )

            if len(estimable) >= 2:
                max_index = estimable[
                    metric_name
                ].idxmax()

                min_index = estimable[
                    metric_name
                ].idxmin()

                max_value = float(
                    estimable.loc[
                        max_index,
                        metric_name
                    ]
                )

                min_value = float(
                    estimable.loc[
                        min_index,
                        metric_name
                    ]
                )

                max_group = estimable.loc[
                    max_index,
                    "Group"
                ]

                min_group = estimable.loc[
                    min_index,
                    "Group"
                ]

                disparity_gap = float(
                    max_value - min_value
                )
            else:
                max_value = np.nan
                min_value = np.nan
                max_group = ""
                min_group = ""
                disparity_gap = np.nan

            disparity_rows.append({
                "Validation_Set": validation_set,
                "Protected_Group": protected_variable,
                "Metric": metric_name,
                "Estimable_Groups": int(len(estimable)),
                "Maximum_Value": max_value,
                "Maximum_Group": max_group,
                "Minimum_Value": min_value,
                "Minimum_Group": min_group,
                "Max_Min_Gap": disparity_gap,
                "Interpretation": (
                    "Descriptive subgroup gap only; "
                    "no inferential fairness conclusion."
                )
            })

part7a_disparity_summary = pd.DataFrame(
    disparity_rows
)

part7a_publication_table = (
    part7a_subgroup_results[
        [
            "Validation_Set",
            "Protected_Group",
            "Group",
            "N",
            "Positive_Cases",
            "Prevalence",
            "Sensitivity",
            "Specificity",
            "Precision",
            "Negative_Predictive_Value",
            "F1",
            "False_Positive_Rate",
            "Predicted_Positive_Rate",
            "AUC",
            "Average_Precision",
            "Stable_Event_Metrics"
        ]
    ]
    .copy()
)

part7a_small_group_flags = (
    part7a_subgroup_results[
        ~part7a_subgroup_results[
            "Stable_Event_Metrics"
        ]
    ]
    [
        [
            "Validation_Set",
            "Protected_Group",
            "Group",
            "N",
            "Positive_Cases",
            "Negative_Cases"
        ]
    ]
    .copy()
)

part7a_small_group_flags[
    "Reason"
] = (
    "Positive cases < "
    + str(MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS)
    + "; event-dependent metrics suppressed."
)

verification_rows = []

for validation_set in VALIDATION_SETS:
    cohort_result = (
        part7a_subgroup_results[
            part7a_subgroup_results[
                "Validation_Set"
            ]
            == validation_set
        ]
    )

    analyzed_variables = set(
        cohort_result[
            "Protected_Group"
        ].astype(str).unique().tolist()
    )

    all_variables_analyzed = bool(
        analyzed_variables
        ==
        set(PROTECTED_VARIABLES)
    )

    all_group_counts_positive = bool(
        (
            cohort_result["N"] > 0
        ).all()
    )

    all_confusion_counts_match_n = bool(
        (
            cohort_result["TP"]
            + cohort_result["FP"]
            + cohort_result["FN"]
            + cohort_result["TN"]
            ==
            cohort_result["N"]
        ).all()
    )

    unstable_rows = (
        cohort_result[
            ~cohort_result[
                "Stable_Event_Metrics"
            ]
        ]
    )

    unstable_metrics_suppressed = bool(
        unstable_rows[
            [
                "Sensitivity",
                "Precision",
                "F1",
                "AUC",
                "Average_Precision"
            ]
        ]
        .isna()
        .all()
        .all()
    )

    specificity_retained = bool(
        unstable_rows[
            "Specificity"
        ]
        .notna()
        .all()
    ) if len(unstable_rows) > 0 else True

    verification_rows.append({
        "Validation_Set": validation_set,
        "All_Protected_Variables_Analyzed": all_variables_analyzed,
        "All_Group_N_Positive": all_group_counts_positive,
        "All_Confusion_Counts_Match_N": all_confusion_counts_match_n,
        "Unstable_Event_Metrics_Suppressed": unstable_metrics_suppressed,
        "Specificity_Retained_When_Estimable": specificity_retained,
        "No_Model_Refit": True,
        "No_New_Threshold_Optimization": True,
        "No_Recalibration": True,
        "Fairness_Analysis_Is_Descriptive": True,
        "PART7A_Ready": bool(
            all_variables_analyzed
            and all_group_counts_positive
            and all_confusion_counts_match_n
            and unstable_metrics_suppressed
            and specificity_retained
        )
    })

part7a_verification = pd.DataFrame(
    verification_rows
)

print("\n" + "=" * 110)
print("SUBGROUP PERFORMANCE SUMMARY")
print("=" * 110)

display(
    part7a_publication_table.round(4)
)

print("\n" + "=" * 110)
print("SMALL / LOW-EVENT SUBGROUP FLAGS")
print("=" * 110)

display(
    part7a_small_group_flags
)

print("\n" + "=" * 110)
print("DESCRIPTIVE DISPARITY SUMMARY")
print("=" * 110)

display(
    part7a_disparity_summary.round(4)
)

print("\n" + "=" * 110)
print("PART 7A VERIFICATION")
print("=" * 110)

display(
    part7a_verification
)

if not part7a_verification[
    "PART7A_Ready"
].all():

    raise RuntimeError(
        "PART 7A verification failed."
    )

part7a_subgroup_results.to_csv(
    "part7a_subgroup_results.csv",
    index=False,
    encoding="utf-8-sig"
)

part7a_disparity_summary.to_csv(
    "part7a_disparity_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

part7a_status.to_csv(
    "part7a_status.csv",
    index=False,
    encoding="utf-8-sig"
)

part7a_small_group_flags.to_csv(
    "part7a_small_group_flags.csv",
    index=False,
    encoding="utf-8-sig"
)

part7a_publication_table.to_csv(
    "part7a_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)

part7a_verification.to_csv(
    "part7a_verification.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 110)
print("PART 7A VERSION 3.0 COMPLETED")
print("=" * 110)

print(
    f"Final model                 : "
    f"{FINAL_SCREENING_MODEL} + {FINAL_ALGORITHM}"
)

print(
    f"Validation cohorts analyzed : "
    f"{len(VALIDATION_SETS)}"
)

print(
    f"Protected variables analyzed: "
    f"{len(PROTECTED_VARIABLES)}"
)

print(
    f"Subgroup rows generated     : "
    f"{len(part7a_subgroup_results)}"
)

print(
    f"Low-event subgroup rows     : "
    f"{len(part7a_small_group_flags)}"
)

print("\nArtifacts created:")

print("- part7a_subgroup_results")
print("- part7a_disparity_summary")
print("- part7a_status")
print("- part7a_small_group_flags")
print("- part7a_publication_table")
print("- part7a_verification")

print("\nExported files:")

print("- part7a_subgroup_results.csv")
print("- part7a_disparity_summary.csv")
print("- part7a_status.csv")
print("- part7a_small_group_flags.csv")
print("- part7a_publication_table.csv")
print("- part7a_verification.csv")

print(
    "\nSubgroup metrics were calculated from fixed held-out "
    "2024 and temporal 2025 predictions only."
)

print(
    "No model was refitted, no threshold was re-optimized, "
    "and no probability was recalibrated."
)

print(
    "Groups with fewer than "
    f"{MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS} positive "
    "cases were retained descriptively, but sensitivity, "
    "precision, F1, AUC, and average precision were suppressed "
    "as unstable."
)

print(
    "Fairness results should be interpreted descriptively and "
    "not as causal evidence or proof of absence/presence of "
    "algorithmic discrimination."
)

### PART 7B: Fairness Interpretation and Publication Summary

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 7B: FAIRNESS INTERPRETATION AND PUBLICATION SUMMARY
# FINAL RECOMMENDED MODEL ONLY
# ============================================================
#
# Final model:
# - Screening strategy: M2_Admin_2Q
# - Algorithm: AdaBoostClassifier
#
# Purpose:
# Convert PART 7A subgroup results into a conservative,
# publication-ready fairness interpretation.
#
# Core principles:
# - Descriptive fairness assessment only
# - No hypothesis testing
# - No subgroup bootstrap CI
# - No new predictions
# - No model refitting
# - No threshold modification
# - No probability recalibration
#
# Sparse subgroup policy:
# - Positive cases >= 5  -> event-dependent metrics estimable
# - Positive cases < 5   -> event-dependent metrics considered
#                           insufficiently stable
#
# Outputs:
# - part7b_estimability_summary
# - part7b_interpretable_subgroups
# - part7b_disparity_interpretation
# - part7b_cross_cohort_summary
# - part7b_publication_table
# - part7b_methods_text
# - part7b_results_text
# - part7b_limitations_text
# - part7b_key_findings
# - part7b_verification
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 2. SETTINGS
# ============================================================

FINAL_SCREENING_MODEL = "M2_Admin_2Q"
FINAL_ALGORITHM = "AdaBoostClassifier"

MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS = 5

VALIDATION_SETS = [
    "Held-out 2024",
    "Temporal 2025"
]

PROTECTED_VARIABLES = [
    "gender",
    "institute",
    "parent",
    "momeducation",
    "dadeducation",
    "momstatus",
    "dadstatus"
]


# ============================================================
# 3. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "part5c_final_recommendation",
    "part7a_subgroup_results",
    "part7a_disparity_summary",
    "part7a_small_group_flags",
    "part7a_verification"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:

    raise NameError(
        "Missing required objects: "
        + ", ".join(
            missing_objects
        )
    )


for object_name in [
    "part7a_subgroup_results",
    "part7a_disparity_summary",
    "part7a_small_group_flags",
    "part7a_verification"
]:

    if not isinstance(
        globals()[object_name],
        pd.DataFrame
    ):

        raise TypeError(
            f"{object_name} must be a pandas DataFrame."
        )


if not part7a_verification[
    "PART7A_Ready"
].all():

    raise RuntimeError(
        "PART 7A is not fully ready."
    )


# ============================================================
# 4. VERIFY FINAL MODEL
# ============================================================

if (
    part5c_final_recommendation[
        "Primary_Recommended_Strategy"
    ]
    !=
    FINAL_SCREENING_MODEL
):

    raise RuntimeError(
        "Final strategy no longer matches M2_Admin_2Q."
    )


if (
    part5c_final_recommendation[
        "Primary_Recommended_Algorithm"
    ]
    !=
    FINAL_ALGORITHM
):

    raise RuntimeError(
        "Final algorithm no longer matches AdaBoostClassifier."
    )


# ============================================================
# 5. BASIC COUNTS
# ============================================================

total_subgroup_rows = int(
    len(
        part7a_subgroup_results
    )
)


low_event_subgroup_rows = int(
    len(
        part7a_small_group_flags
    )
)


adequate_event_subgroup_rows = int(
    total_subgroup_rows
    -
    low_event_subgroup_rows
)


low_event_percentage = float(
    100
    *
    low_event_subgroup_rows
    /
    total_subgroup_rows
)


# ============================================================
# 6. HELPER: CLASSIFY VARIABLE-LEVEL ESTIMABILITY
# ============================================================

def classify_estimability(
    total_groups,
    stable_groups
):
    """
    Variable-level descriptive estimability classification.
    """

    if total_groups <= 0:

        return "No groups"


    if stable_groups == total_groups:

        return "Adequate"


    if stable_groups > 0:

        return "Partially estimable"


    return "Insufficient events"


# ============================================================
# 7. ESTIMABILITY SUMMARY
# ============================================================

estimability_rows = []


for validation_set in VALIDATION_SETS:

    for protected_variable in PROTECTED_VARIABLES:

        subset = (
            part7a_subgroup_results[
                (
                    part7a_subgroup_results[
                        "Validation_Set"
                    ]
                    ==
                    validation_set
                )
                &
                (
                    part7a_subgroup_results[
                        "Protected_Group"
                    ]
                    ==
                    protected_variable
                )
            ]
            .copy()
        )


        total_groups = int(
            len(
                subset
            )
        )


        stable_groups = int(
            subset[
                "Stable_Event_Metrics"
            ].sum()
        )


        unstable_groups = int(
            total_groups
            -
            stable_groups
        )


        total_n = int(
            subset[
                "N"
            ].sum()
        )


        total_events = int(
            subset[
                "Positive_Cases"
            ].sum()
        )


        estimability_class = (
            classify_estimability(
                total_groups=total_groups,
                stable_groups=stable_groups
            )
        )


        estimability_rows.append({

            "Validation_Set":
                validation_set,

            "Protected_Group":
                protected_variable,

            "Total_Groups":
                total_groups,

            "Stable_Groups":
                stable_groups,

            "Low_Event_Groups":
                unstable_groups,

            "Stable_Group_Percentage":
                (
                    100
                    *
                    stable_groups
                    /
                    total_groups
                    if total_groups > 0
                    else np.nan
                ),

            "Combined_N_Across_Groups":
                total_n,

            "Combined_Positive_Cases":
                total_events,

            "Estimability_Status":
                estimability_class,

            "Interpretation":
                (
                    "Event-dependent subgroup metrics may be "
                    "interpreted descriptively."
                    if estimability_class == "Adequate"
                    else (
                        "Only stable subgroup strata should be "
                        "interpreted; sparse strata remain descriptive."
                        if estimability_class == "Partially estimable"
                        else (
                            "Event counts are insufficient for "
                            "meaningful event-dependent subgroup "
                            "comparison."
                        )
                    )
                )
        })


part7b_estimability_summary = pd.DataFrame(
    estimability_rows
)


# ============================================================
# 8. INTERPRETABLE SUBGROUPS ONLY
# ============================================================

part7b_interpretable_subgroups = (
    part7a_subgroup_results[
        part7a_subgroup_results[
            "Stable_Event_Metrics"
        ]
    ]
    [
        [
            "Validation_Set",
            "Protected_Group",
            "Group",
            "N",
            "Positive_Cases",
            "Prevalence",
            "Sensitivity",
            "Specificity",
            "Precision",
            "Negative_Predictive_Value",
            "F1",
            "False_Positive_Rate",
            "Predicted_Positive_Rate",
            "AUC",
            "Average_Precision"
        ]
    ]
    .copy()
)


part7b_interpretable_subgroups[
    "Interpretation_Status"
] = (
    "Descriptively interpretable; not confirmatory"
)


# ============================================================
# 9. DISPARITY INTERPRETATION
# ============================================================
#
# Keep only disparity rows where at least 2 groups were
# estimable for the relevant metric.
# ============================================================

part7b_disparity_interpretation = (
    part7a_disparity_summary[
        part7a_disparity_summary[
            "Estimable_Groups"
        ]
        >=
        2
    ]
    .copy()
)


part7b_disparity_interpretation[
    "Fairness_Conclusion"
] = (
    "Descriptive difference only"
)


part7b_disparity_interpretation[
    "Hypothesis_Test_Performed"
] = False


part7b_disparity_interpretation[
    "Confirmatory_Fairness_Conclusion_Allowed"
] = False


# ============================================================
# 10. CROSS-COHORT INTERPRETABILITY
# ============================================================

cross_cohort_rows = []


for protected_variable in PROTECTED_VARIABLES:

    heldout_row = (
        part7b_estimability_summary[
            (
                part7b_estimability_summary[
                    "Validation_Set"
                ]
                ==
                "Held-out 2024"
            )
            &
            (
                part7b_estimability_summary[
                    "Protected_Group"
                ]
                ==
                protected_variable
            )
        ]
    )


    temporal_row = (
        part7b_estimability_summary[
            (
                part7b_estimability_summary[
                    "Validation_Set"
                ]
                ==
                "Temporal 2025"
            )
            &
            (
                part7b_estimability_summary[
                    "Protected_Group"
                ]
                ==
                protected_variable
            )
        ]
    )


    if (
        len(
            heldout_row
        )
        !=
        1
        or
        len(
            temporal_row
        )
        !=
        1
    ):

        raise RuntimeError(
            f"Could not uniquely identify estimability rows for "
            f"{protected_variable}."
        )


    heldout_row = heldout_row.iloc[
        0
    ]


    temporal_row = temporal_row.iloc[
        0
    ]


    heldout_stable = int(
        heldout_row[
            "Stable_Groups"
        ]
    )


    temporal_stable = int(
        temporal_row[
            "Stable_Groups"
        ]
    )


    if (
        heldout_stable >= 2
        and
        temporal_stable >= 2
    ):

        cross_cohort_status = (
            "Cross-cohort descriptive comparison possible"
        )


    elif (
        heldout_stable >= 1
        and
        temporal_stable >= 1
    ):

        cross_cohort_status = (
            "Limited cross-cohort comparison"
        )


    else:

        cross_cohort_status = (
            "Cross-cohort fairness comparison not supported "
            "by event counts"
        )


    cross_cohort_rows.append({

        "Protected_Group":
            protected_variable,

        "Heldout_Stable_Groups":
            heldout_stable,

        "Temporal_Stable_Groups":
            temporal_stable,

        "Heldout_Estimability_Status":
            heldout_row[
                "Estimability_Status"
            ],

        "Temporal_Estimability_Status":
            temporal_row[
                "Estimability_Status"
            ],

        "Cross_Cohort_Interpretability":
            cross_cohort_status,

        "Confirmatory_Consistency_Conclusion_Allowed":
            False
    })


part7b_cross_cohort_summary = pd.DataFrame(
    cross_cohort_rows
)


# ============================================================
# 11. PUBLICATION TABLE
# ============================================================

part7b_publication_table = (
    part7b_estimability_summary[
        [
            "Validation_Set",
            "Protected_Group",
            "Total_Groups",
            "Stable_Groups",
            "Low_Event_Groups",
            "Stable_Group_Percentage",
            "Combined_N_Across_Groups",
            "Combined_Positive_Cases",
            "Estimability_Status",
            "Interpretation"
        ]
    ]
    .copy()
)


# ============================================================
# 12. METHODS TEXT
# ============================================================

part7b_methods_text = {

    "Methods":
        (
            "Fairness was evaluated descriptively for the final "
            "recommended model across gender, institute, parental "
            "status, and parental education subgroups in both the "
            "held-out 2024 and temporal 2025 cohorts. Performance "
            "metrics were calculated using the fixed predictions "
            "and the prespecified threshold locked from genuine "
            "out-of-fold development predictions. Because the "
            "outcome was rare, subgroup strata with fewer than "
            f"{MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS} "
            "positive cases were retained for descriptive counts "
            "but event-dependent metrics, including sensitivity, "
            "precision, F1 score, ROC AUC, and average precision, "
            "were considered insufficiently stable for "
            "interpretation. No subgroup hypothesis testing or "
            "bootstrap confidence intervals were performed."
        )
}


# ============================================================
# 13. RESULTS TEXT
# ============================================================

part7b_results_text = {

    "Results":
        (
            f"A total of {total_subgroup_rows} subgroup strata "
            "were evaluated across the held-out and temporal "
            f"cohorts. Of these, {low_event_subgroup_rows} "
            f"({low_event_percentage:.1f}%) contained fewer than "
            f"{MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS} "
            "positive cases and were therefore considered "
            "insufficiently stable for interpretation of "
            "event-dependent performance metrics. Consequently, "
            "fairness findings were interpreted descriptively, "
            "with emphasis on subgroup strata meeting the "
            "minimum event-count criterion. No confirmatory "
            "evidence of performance equivalence or disparity "
            "across all demographic subgroups could be established "
            "from these data."
        )
}


# ============================================================
# 14. LIMITATIONS TEXT
# ============================================================

part7b_limitations_text = {

    "Limitations":
        (
            "The fairness analysis was limited by the low number "
            "of MDD events and substantial sparsity after "
            "stratification by demographic variables. Most "
            "subgroup strata contained fewer than five positive "
            "cases, preventing stable estimation of several "
            "event-dependent metrics. Accordingly, observed "
            "max-min subgroup gaps should be regarded as "
            "exploratory descriptive signals rather than formal "
            "evidence of algorithmic bias or fairness. Larger "
            "multi-institutional validation cohorts with greater "
            "event counts would be required for confirmatory "
            "subgroup-performance assessment."
        )
}


# ============================================================
# 15. KEY FINDINGS
# ============================================================

key_finding_rows = [

    {
        "Key_Finding":
            (
                f"{low_event_subgroup_rows} of "
                f"{total_subgroup_rows} subgroup strata "
                f"({low_event_percentage:.1f}%) had fewer than "
                f"{MIN_POSITIVE_CASES_FOR_STABLE_EVENT_METRICS} "
                "positive cases."
            )
    },

    {
        "Key_Finding":
            (
                "Event-dependent fairness metrics were suppressed "
                "for sparse subgroups rather than reported as "
                "apparently precise estimates."
            )
    },

    {
        "Key_Finding":
            (
                "Subgroup differences that remained estimable were "
                "treated as descriptive signals only."
            )
    },

    {
        "Key_Finding":
            (
                "No hypothesis testing, subgroup bootstrap "
                "confidence intervals, threshold retuning, or "
                "model refitting was performed."
            )
    },

    {
        "Key_Finding":
            (
                "The available data do not support a definitive "
                "claim that the model is fair or unfair across all "
                "subgroups."
            )
    }
]


part7b_key_findings = pd.DataFrame(
    key_finding_rows
)


# ============================================================
# 16. OVERALL FAIRNESS INTERPRETATION
# ============================================================

part7b_overall_interpretation = pd.DataFrame(
    [
        {
            "Final_Model":
                (
                    f"{FINAL_SCREENING_MODEL} + "
                    f"{FINAL_ALGORITHM}"
                ),

            "Total_Subgroup_Strata":
                total_subgroup_rows,

            "Stable_Event_Subgroup_Strata":
                adequate_event_subgroup_rows,

            "Low_Event_Subgroup_Strata":
                low_event_subgroup_rows,

            "Low_Event_Percentage":
                low_event_percentage,

            "Overall_Fairness_Interpretation":
                (
                    "Exploratory/descriptive only; sparse outcome "
                    "events preclude definitive subgroup fairness "
                    "conclusions."
                ),

            "Fairness_Claim_Allowed":
                False,

            "Bias_Claim_Allowed":
                False
        }
    ]
)


# ============================================================
# 17. VERIFICATION
# ============================================================

all_validation_sets_present = bool(
    set(
        part7b_estimability_summary[
            "Validation_Set"
        ]
        .astype(str)
        .unique()
        .tolist()
    )
    ==
    set(
        VALIDATION_SETS
    )
)


all_protected_variables_present = bool(
    set(
        part7b_estimability_summary[
            "Protected_Group"
        ]
        .astype(str)
        .unique()
        .tolist()
    )
    ==
    set(
        PROTECTED_VARIABLES
    )
)


estimability_row_count_correct = bool(
    len(
        part7b_estimability_summary
    )
    ==
    (
        len(
            VALIDATION_SETS
        )
        *
        len(
            PROTECTED_VARIABLES
        )
    )
)


sparse_event_limitations_retained = bool(
    low_event_subgroup_rows
    ==
    len(
        part7a_small_group_flags
    )
)


confirmatory_fairness_claims_disabled = bool(
    (
        part7b_disparity_interpretation[
            "Confirmatory_Fairness_Conclusion_Allowed"
        ]
        ==
        False
    ).all()
)


part7b_verification = pd.DataFrame(
    [
        {
            "Final_Model_Is_M2_Admin_2Q_AdaBoost":
                (
                    FINAL_SCREENING_MODEL
                    ==
                    "M2_Admin_2Q"
                    and
                    FINAL_ALGORITHM
                    ==
                    "AdaBoostClassifier"
                ),

            "All_Validation_Sets_Present":
                all_validation_sets_present,

            "All_Protected_Variables_Present":
                all_protected_variables_present,

            "Estimability_Row_Count_Correct":
                estimability_row_count_correct,

            "Sparse_Event_Limitations_Retained":
                sparse_event_limitations_retained,

            "Confirmatory_Fairness_Claims_Disabled":
                confirmatory_fairness_claims_disabled,

            "No_New_Predictions_Generated":
                True,

            "No_Model_Refit":
                True,

            "No_Threshold_Modification":
                True,

            "No_Recalibration_Performed":
                True,

            "No_Subgroup_Hypothesis_Testing":
                True,

            "No_Subgroup_Bootstrap_CI":
                True,

            "Fairness_Conclusions_Descriptive":
                True,

            "PART7B_Ready":
                bool(
                    FINAL_SCREENING_MODEL
                    ==
                    "M2_Admin_2Q"
                    and
                    FINAL_ALGORITHM
                    ==
                    "AdaBoostClassifier"
                    and
                    all_validation_sets_present
                    and
                    all_protected_variables_present
                    and
                    estimability_row_count_correct
                    and
                    sparse_event_limitations_retained
                    and
                    confirmatory_fairness_claims_disabled
                )
        }
    ]
)


# ============================================================
# 18. DISPLAY
# ============================================================

print("=" * 110)
print(
    "PART 7B VERSION 3.0: FAIRNESS INTERPRETATION AND "
    "PUBLICATION SUMMARY"
)
print("=" * 110)


print(
    f"Final model                 : "
    f"{FINAL_SCREENING_MODEL} + {FINAL_ALGORITHM}"
)


print(
    f"Total subgroup strata       : "
    f"{total_subgroup_rows}"
)


print(
    f"Low-event subgroup strata   : "
    f"{low_event_subgroup_rows}"
)


print(
    f"Low-event percentage        : "
    f"{low_event_percentage:.1f}%"
)


print("\n" + "=" * 110)
print("SUBGROUP ESTIMABILITY SUMMARY")
print("=" * 110)


display(
    part7b_estimability_summary
    .round(
        2
    )
)


print("\n" + "=" * 110)
print("INTERPRETABLE SUBGROUP PERFORMANCE")
print("=" * 110)


display(
    part7b_interpretable_subgroups
    .round(
        4
    )
)


print("\n" + "=" * 110)
print("DESCRIPTIVE DISPARITY INTERPRETATION")
print("=" * 110)


display(
    part7b_disparity_interpretation
    .round(
        4
    )
)


print("\n" + "=" * 110)
print("CROSS-COHORT INTERPRETABILITY")
print("=" * 110)


display(
    part7b_cross_cohort_summary
)


print("\n" + "=" * 110)
print("PUBLICATION TABLE")
print("=" * 110)


display(
    part7b_publication_table
    .round(
        2
    )
)


print("\n" + "=" * 110)
print("KEY FINDINGS")
print("=" * 110)


display(
    part7b_key_findings
)


print("\n" + "=" * 110)
print("PART 7B VERIFICATION")
print("=" * 110)


display(
    part7b_verification
)


# ============================================================
# 19. HARD SAFETY CHECK
# ============================================================

if not part7b_verification[
    "PART7B_Ready"
].all():

    raise RuntimeError(
        "PART 7B verification failed."
    )


# ============================================================
# 20. EXPORT
# ============================================================

part7b_estimability_summary.to_csv(
    "part7b_estimability_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


part7b_interpretable_subgroups.to_csv(
    "part7b_interpretable_subgroups.csv",
    index=False,
    encoding="utf-8-sig"
)


part7b_disparity_interpretation.to_csv(
    "part7b_disparity_interpretation.csv",
    index=False,
    encoding="utf-8-sig"
)


part7b_cross_cohort_summary.to_csv(
    "part7b_cross_cohort_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


part7b_publication_table.to_csv(
    "part7b_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)


pd.DataFrame(
    [
        part7b_methods_text
    ]
).to_csv(
    "part7b_methods_text.csv",
    index=False,
    encoding="utf-8-sig"
)


pd.DataFrame(
    [
        part7b_results_text
    ]
).to_csv(
    "part7b_results_text.csv",
    index=False,
    encoding="utf-8-sig"
)


pd.DataFrame(
    [
        part7b_limitations_text
    ]
).to_csv(
    "part7b_limitations_text.csv",
    index=False,
    encoding="utf-8-sig"
)


part7b_key_findings.to_csv(
    "part7b_key_findings.csv",
    index=False,
    encoding="utf-8-sig"
)


part7b_overall_interpretation.to_csv(
    "part7b_overall_fairness_interpretation.csv",
    index=False,
    encoding="utf-8-sig"
)


part7b_verification.to_csv(
    "part7b_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 21. FINAL REPORT
# ============================================================

print("\n" + "=" * 110)
print("PART 7B VERSION 3.0 COMPLETED")
print("=" * 110)


print(
    f"Final model                 : "
    f"{FINAL_SCREENING_MODEL} + {FINAL_ALGORITHM}"
)


print(
    f"Total subgroup strata       : "
    f"{total_subgroup_rows}"
)


print(
    f"Stable subgroup strata      : "
    f"{adequate_event_subgroup_rows}"
)


print(
    f"Low-event subgroup strata   : "
    f"{low_event_subgroup_rows}"
)


print(
    f"Low-event percentage        : "
    f"{low_event_percentage:.1f}%"
)


print(
    "Fairness interpretation    : "
    "Exploratory / descriptive only"
)


print("\nArtifacts created:")

print("- part7b_estimability_summary")
print("- part7b_interpretable_subgroups")
print("- part7b_disparity_interpretation")
print("- part7b_cross_cohort_summary")
print("- part7b_publication_table")
print("- part7b_methods_text")
print("- part7b_results_text")
print("- part7b_limitations_text")
print("- part7b_key_findings")
print("- part7b_overall_interpretation")
print("- part7b_verification")


print("\nExported files:")

print("- part7b_estimability_summary.csv")
print("- part7b_interpretable_subgroups.csv")
print("- part7b_disparity_interpretation.csv")
print("- part7b_cross_cohort_summary.csv")
print("- part7b_publication_table.csv")
print("- part7b_methods_text.csv")
print("- part7b_results_text.csv")
print("- part7b_limitations_text.csv")
print("- part7b_key_findings.csv")
print("- part7b_overall_fairness_interpretation.csv")
print("- part7b_verification.csv")


print(
    "\nNo new predictions were generated, no model was refitted, "
    "no threshold was modified, no recalibration was performed, "
    "and no subgroup hypothesis testing or bootstrap confidence "
    "interval analysis was added."
)


print(
    "Sparse-event limitations were preserved explicitly. "
    "Fairness results remain exploratory and descriptive."
)


print(
    "\nProceed to PART 8 only after PART7B_Ready=True."
)

## PART 8: Final Publication Package

### PART 8A: Final Main Results Table

In [ ]:
# ============================================================
# PRE-FIX FOR PART 8A
# Normalize bootstrap reporting column names
# ============================================================

import pandas as pd

def normalize_bootstrap_schema(table):
    table = table.copy()

    # PART 4B v3.1 uses Valid_Replicates
    if (
        "Valid_Bootstrap_Replicates" not in table.columns
        and "Valid_Replicates" in table.columns
    ):
        table["Valid_Bootstrap_Replicates"] = (
            table["Valid_Replicates"]
        )

    # Add reporting metadata if not already present
    if "CI_Level" not in table.columns:
        table["CI_Level"] = 0.95

    if "CI_Method" not in table.columns:
        table["CI_Method"] = (
            "Stratified percentile bootstrap "
            "(2.5th-97.5th percentiles)"
        )

    return table


part3c_bootstrap_results = normalize_bootstrap_schema(
    part3c_bootstrap_results
)

part4b_bootstrap_results = normalize_bootstrap_schema(
    part4b_bootstrap_results
)


required_columns = [
    "CI_Level",
    "CI_Method",
    "Valid_Bootstrap_Replicates"
]

print("PART 8A BOOTSTRAP SCHEMA PRE-CHECK")
print("=" * 60)

for table_name, table in [
    ("part3c_bootstrap_results", part3c_bootstrap_results),
    ("part4b_bootstrap_results", part4b_bootstrap_results)
]:
    missing = [
        column
        for column in required_columns
        if column not in table.columns
    ]

    print(
        f"{table_name}: "
        f"{'Ready' if not missing else 'Missing: ' + ', '.join(missing)}"
    )

    if missing:
        raise KeyError(
            f"{table_name} is still missing: "
            + ", ".join(missing)
        )

print("=" * 60)
print("Bootstrap tables are ready for PART 8A.")

In [ ]:
# PART 8A: FINAL MAIN RESULTS TABLE

# PUBLICATION REPORTING LAYER

# ============================================================

#

# Objectives:

# • Create the final main manuscript comparison table

# • Include all 7 prespecified screening strategies

# • Use representative algorithms selected in PART 5B

# • Preserve the final recommendation from PART 5C

# • Retrieve bootstrap estimates and 95% CIs from:

#       PART 3C = Held-out 2024

#       PART 4B = Temporal 2025

#

# IMPORTANT:

# • No model refitting

# • No new prediction

# • No threshold optimization

# • No recalibration

# • No algorithm re-selection

# • No performance metric recalculation

#

# Primary main-table metrics:

# • AUC

# • Sensitivity

# • Specificity

# • Precision

# • F1

#

# Outputs:

# • part8a_final_main_results

# • part8a_final_main_results_compact

# • part8a_metric_source_manifest

# • part8a_verification

#

# ============================================================





# ============================================================

# 1. IMPORTS

# ============================================================



import numpy as np

import pandas as pd





# ============================================================

# 2. SETTINGS

# ============================================================



FINAL_SCREENING_MODEL = "M2_Admin_2Q"

FINAL_ALGORITHM = "AdaBoostClassifier"



EXPECTED_STRATEGIES = [



    "M0_Admin",

    "M1_Admin_Behavior",

    "M2_Admin_2Q",

    "M3_Admin_2Q_Behavior",

    "M4_Conventional",

    "M5_Conventional_Admin",

    "M6_Full"



]





MAIN_METRICS = [



    "AUC",

    "Sensitivity",

    "Specificity",

    "Precision",

    "F1"



]





# ============================================================

# 3. REQUIRED OBJECTS

# ============================================================



required_objects = [



    "part5b_strategy_representatives",

    "part5c_final_strategy_table",

    "part5c_strategy_roles",

    "part5c_final_recommendation",

    "part3c_bootstrap_results",

    "part4b_bootstrap_results"



]





missing_objects = [



    object_name



    for object_name in required_objects



    if object_name not in globals()



]





if missing_objects:



    raise NameError(



        "Missing required objects: "



        + ", ".join(

            missing_objects

        )



    )





# ============================================================

# 4. TYPE CHECK

# ============================================================



dataframe_objects = [



    "part5b_strategy_representatives",

    "part5c_final_strategy_table",

    "part5c_strategy_roles",

    "part3c_bootstrap_results",

    "part4b_bootstrap_results"



]





for object_name in dataframe_objects:



    if not isinstance(

        globals()[object_name],

        pd.DataFrame

    ):



        raise TypeError(

            f"{object_name} must be a pandas DataFrame."

        )





# ============================================================

# 5. VERIFY FINAL RECOMMENDATION

# ============================================================



if (



    part5c_final_recommendation[

        "Primary_Recommended_Strategy"

    ]



    != FINAL_SCREENING_MODEL



):



    raise RuntimeError(

        "Final recommended strategy does not match M2_Admin_2Q."

    )





if (



    part5c_final_recommendation[

        "Primary_Recommended_Algorithm"

    ]



    != FINAL_ALGORITHM



):



    raise RuntimeError(

        "Final recommended algorithm does not match AdaBoostClassifier."

    )





# ============================================================

# 6. VERIFY STRATEGY REPRESENTATIVES

# ============================================================



representatives = (



    part5b_strategy_representatives

    .copy()



)





if representatives[

    "Screening_Model"

].duplicated().any():



    duplicated_models = (



        representatives.loc[

            representatives[

                "Screening_Model"

            ].duplicated(

                keep=False

            ),

            "Screening_Model"

        ]

        .tolist()



    )



    raise RuntimeError(



        "Duplicate strategy representatives detected: "



        + ", ".join(

            map(

                str,

                duplicated_models

            )

        )



    )





observed_strategies = set(



    representatives[

        "Screening_Model"

    ]

    .astype(str)

    .tolist()



)





if observed_strategies != set(

    EXPECTED_STRATEGIES

):



    raise RuntimeError(



        "PART 5B strategy representatives do not contain "

        "exactly the expected 7 strategies."



    )





# ============================================================

# 7. RETAIN REQUIRED STRATEGY INFORMATION

# ============================================================



strategy_columns = [



    "Screening_Model",

    "Strategy_Type",

    "Description",

    "Feature_Count",

    "Self_Report_Items",

    "Algorithm",

    "Locked_Threshold",

    "OOF_AUC",

    "OOF_Average_Precision",

    "OOF_Sensitivity",

    "OOF_Specificity"



]





missing_strategy_columns = [



    column



    for column in strategy_columns



    if column not in representatives.columns



]





if missing_strategy_columns:



    raise KeyError(



        "Missing columns in part5b_strategy_representatives: "



        + ", ".join(

            missing_strategy_columns

        )



    )





strategy_table = (



    representatives[

        strategy_columns

    ]

    .copy()



)





# ============================================================

# 8. ADD STRATEGY ROLES FROM PART 5C

# ============================================================



role_columns = [



    "Screening_Model",

    "Algorithm",

    "Strategy_Role",

    "Primary_Recommended",

    "Validation_Support_Summary",

    "Pareto_Efficient",

    "Dominated",

    "Dominated_By"



]





missing_role_columns = [



    column



    for column in role_columns



    if column not in part5c_strategy_roles.columns



]





if missing_role_columns:



    raise KeyError(



        "Missing columns in part5c_strategy_roles: "



        + ", ".join(

            missing_role_columns

        )



    )





strategy_roles = (



    part5c_strategy_roles[

        role_columns

    ]

    .copy()



)





strategy_table = strategy_table.merge(



    strategy_roles,



    on=[

        "Screening_Model",

        "Algorithm"

    ],



    how="left",



    validate="one_to_one"



)





if strategy_table[

    "Strategy_Role"

].isna().any():



    raise RuntimeError(



        "Some strategy representatives could not be matched "

        "to PART 5C strategy roles."



    )





# ============================================================

# 9. BOOTSTRAP TABLE SCHEMA CHECK

# ============================================================



required_bootstrap_columns = [



    "Metric",

    "Point_Estimate",

    "CI_Lower",

    "CI_Upper",

    "Validation_Set",

    "Screening_Model",

    "Algorithm",

    "N",

    "Events",

    "Prevalence",

    "Locked_Threshold",

    "CI_Level",

    "CI_Method",

    "Valid_Bootstrap_Replicates",

    "CI_Computed"



]





for table_name in [



    "part3c_bootstrap_results",

    "part4b_bootstrap_results"



]:



    table = globals()[

        table_name

    ]



    missing_columns = [



        column



        for column in required_bootstrap_columns



        if column not in table.columns



    ]



    if missing_columns:



        raise KeyError(



            f"{table_name} missing required columns: "



            + ", ".join(

                missing_columns

            )



        )





# ============================================================

# 10. HELPER FUNCTION:

#     EXTRACT ONE BOOTSTRAP METRIC

# ============================================================



def extract_bootstrap_metric(

    bootstrap_table,

    screening_model,

    algorithm,

    metric

):



    subset = (



        bootstrap_table[

            (

                bootstrap_table[

                    "Screening_Model"

                ]

                ==

                screening_model

            )

            &

            (

                bootstrap_table[

                    "Algorithm"

                ]

                ==

                algorithm

            )

            &

            (

                bootstrap_table[

                    "Metric"

                ]

                ==

                metric

            )

        ]

        .copy()



    )





    if len(

        subset

    ) != 1:



        raise RuntimeError(



            f"Expected exactly one bootstrap result for "

            f"{screening_model} + {algorithm} + {metric}; "

            f"found {len(subset)}."



        )





    row = subset.iloc[

        0

    ]





    if not bool(

        row[

            "CI_Computed"

        ]

    ):



        raise RuntimeError(



            f"Bootstrap CI not available for "

            f"{screening_model} + {algorithm} + {metric}."



        )





    return row





# ============================================================

# 11. FORMAT ESTIMATE + 95% CI

# ============================================================



def format_ci(

    point_estimate,

    ci_lower,

    ci_upper,

    digits=3

):



    if (



        pd.isna(

            point_estimate

        )

        or

        pd.isna(

            ci_lower

        )

        or

        pd.isna(

            ci_upper

        )



    ):



        return "NE"





    return (



        f"{point_estimate:.{digits}f} "

        f"({ci_lower:.{digits}f}–"

        f"{ci_upper:.{digits}f})"



    )





# ============================================================

# 12. BUILD FINAL MAIN RESULTS

# ============================================================



final_rows = []



source_manifest_rows = []





for _, strategy_row in strategy_table.iterrows():



    screening_model = strategy_row[

        "Screening_Model"

    ]



    algorithm = strategy_row[

        "Algorithm"

    ]





    # --------------------------------------------------------

    # Base strategy information

    # --------------------------------------------------------



    result_row = {



        "Screening_Model":

            screening_model,



        "Strategy_Type":

            strategy_row[

                "Strategy_Type"

            ],



        "Description":

            strategy_row[

                "Description"

            ],



        "Feature_Count":

            strategy_row[

                "Feature_Count"

            ],



        "Self_Report_Items":

            strategy_row[

                "Self_Report_Items"

            ],



        "Algorithm":

            algorithm,



        "Locked_Threshold":

            strategy_row[

                "Locked_Threshold"

            ],



        "OOF_AUC":

            strategy_row[

                "OOF_AUC"

            ],



        "OOF_Average_Precision":

            strategy_row[

                "OOF_Average_Precision"

            ],



        "OOF_Sensitivity":

            strategy_row[

                "OOF_Sensitivity"

            ],



        "OOF_Specificity":

            strategy_row[

                "OOF_Specificity"

            ],



        "Validation_Support":

            strategy_row[

                "Validation_Support_Summary"

            ],



        "Pareto_Efficient":

            strategy_row[

                "Pareto_Efficient"

            ],



        "Strategy_Role":

            strategy_row[

                "Strategy_Role"

            ],



        "Primary_Recommended":

            strategy_row[

                "Primary_Recommended"

            ]



    }





    # --------------------------------------------------------

    # Extract Held-out 2024 metrics

    # --------------------------------------------------------



    heldout_metric_rows = {}





    for metric in MAIN_METRICS:



        metric_row = extract_bootstrap_metric(



            bootstrap_table=

                part3c_bootstrap_results,



            screening_model=

                screening_model,



            algorithm=

                algorithm,



            metric=

                metric



        )





        heldout_metric_rows[

            metric

        ] = metric_row





        result_row[

            f"Heldout_{metric}"

        ] = float(

            metric_row[

                "Point_Estimate"

            ]

        )





        result_row[

            f"Heldout_{metric}_CI_Lower"

        ] = float(

            metric_row[

                "CI_Lower"

            ]

        )





        result_row[

            f"Heldout_{metric}_CI_Upper"

        ] = float(

            metric_row[

                "CI_Upper"

            ]

        )





        result_row[

            f"Heldout_{metric}_95CI"

        ] = format_ci(



            metric_row[

                "Point_Estimate"

            ],



            metric_row[

                "CI_Lower"

            ],



            metric_row[

                "CI_Upper"

            ]



        )





        source_manifest_rows.append({



            "Screening_Model":

                screening_model,



            "Algorithm":

                algorithm,



            "Validation_Set":

                "Held-out 2024",



            "Metric":

                metric,



            "Source_Object":

                "part3c_bootstrap_results",



            "Point_Estimate":

                metric_row[

                    "Point_Estimate"

                ],



            "CI_Lower":

                metric_row[

                    "CI_Lower"

                ],



            "CI_Upper":

                metric_row[

                    "CI_Upper"

                ],



            "Valid_Bootstrap_Replicates":

                metric_row[

                    "Valid_Bootstrap_Replicates"

                ],



            "CI_Method":

                metric_row[

                    "CI_Method"

                ]



        })





    # --------------------------------------------------------

    # Held-out sample information

    # --------------------------------------------------------



    heldout_reference = heldout_metric_rows[

        "AUC"

    ]





    result_row[

        "Heldout_N"

    ] = int(

        heldout_reference[

            "N"

        ]

    )





    result_row[

        "Heldout_Events"

    ] = int(

        heldout_reference[

            "Events"

        ]

    )





    result_row[

        "Heldout_Prevalence"

    ] = float(

        heldout_reference[

            "Prevalence"

        ]

    )





    # --------------------------------------------------------

    # Extract Temporal 2025 metrics

    # --------------------------------------------------------



    temporal_metric_rows = {}





    for metric in MAIN_METRICS:



        metric_row = extract_bootstrap_metric(



            bootstrap_table=

                part4b_bootstrap_results,



            screening_model=

                screening_model,



            algorithm=

                algorithm,



            metric=

                metric



        )





        temporal_metric_rows[

            metric

        ] = metric_row





        result_row[

            f"Temporal_{metric}"

        ] = float(

            metric_row[

                "Point_Estimate"

            ]

        )





        result_row[

            f"Temporal_{metric}_CI_Lower"

        ] = float(

            metric_row[

                "CI_Lower"

            ]

        )





        result_row[

            f"Temporal_{metric}_CI_Upper"

        ] = float(

            metric_row[

                "CI_Upper"

            ]

        )





        result_row[

            f"Temporal_{metric}_95CI"

        ] = format_ci(



            metric_row[

                "Point_Estimate"

            ],



            metric_row[

                "CI_Lower"

            ],



            metric_row[

                "CI_Upper"

            ]



        )





        source_manifest_rows.append({



            "Screening_Model":

                screening_model,



            "Algorithm":

                algorithm,



            "Validation_Set":

                "Temporal 2025",



            "Metric":

                metric,



            "Source_Object":

                "part4b_bootstrap_results",



            "Point_Estimate":

                metric_row[

                    "Point_Estimate"

                ],



            "CI_Lower":

                metric_row[

                    "CI_Lower"

                ],



            "CI_Upper":

                metric_row[

                    "CI_Upper"

                ],



            "Valid_Bootstrap_Replicates":

                metric_row[

                    "Valid_Bootstrap_Replicates"

                ],



            "CI_Method":

                metric_row[

                    "CI_Method"

                ]



        })





    # --------------------------------------------------------

    # Temporal sample information

    # --------------------------------------------------------



    temporal_reference = temporal_metric_rows[

        "AUC"

    ]





    result_row[

        "Temporal_N"

    ] = int(

        temporal_reference[

            "N"

        ]

    )





    result_row[

        "Temporal_Events"

    ] = int(

        temporal_reference[

            "Events"

        ]

    )





    result_row[

        "Temporal_Prevalence"

    ] = float(

        temporal_reference[

            "Prevalence"

        ]

    )





    # --------------------------------------------------------

    # Threshold consistency check

    # --------------------------------------------------------



    heldout_threshold = float(

        heldout_reference[

            "Locked_Threshold"

        ]

    )





    temporal_threshold = float(

        temporal_reference[

            "Locked_Threshold"

        ]

    )





    strategy_threshold = float(

        strategy_row[

            "Locked_Threshold"

        ]

    )





    result_row[

        "Threshold_Matches_Heldout"

    ] = bool(



        np.isclose(



            strategy_threshold,



            heldout_threshold,



            rtol=0,



            atol=1e-10



        )



    )





    result_row[

        "Threshold_Matches_Temporal"

    ] = bool(



        np.isclose(



            strategy_threshold,



            temporal_threshold,



            rtol=0,



            atol=1e-10



        )



    )





    final_rows.append(

        result_row

    )





# ============================================================

# 13. FINAL FULL TABLE

# ============================================================



part8a_final_main_results = pd.DataFrame(

    final_rows

)





# ============================================================

# 14. PRESPECIFIED STRATEGY ORDER

# ============================================================



strategy_order_map = {



    strategy:

        order



    for order, strategy in enumerate(

        EXPECTED_STRATEGIES

    )



}





part8a_final_main_results[

    "_Strategy_Order"

] = (



    part8a_final_main_results[

        "Screening_Model"

    ]

    .map(

        strategy_order_map

    )



)





part8a_final_main_results = (



    part8a_final_main_results

    .sort_values(

        "_Strategy_Order"

    )

    .drop(

        columns=[

            "_Strategy_Order"

        ]

    )

    .reset_index(

        drop=True

    )



)





# ============================================================

# 15. METRIC SOURCE MANIFEST

# ============================================================



part8a_metric_source_manifest = pd.DataFrame(

    source_manifest_rows

)





# ============================================================

# 16. COMPACT MAIN MANUSCRIPT TABLE

# ============================================================



part8a_final_main_results_compact = (



    part8a_final_main_results[

        [

            "Screening_Model",

            "Self_Report_Items",

            "Algorithm",

            "Locked_Threshold",



            "OOF_AUC",



            "Heldout_AUC_95CI",

            "Heldout_Sensitivity_95CI",

            "Heldout_Specificity_95CI",

            "Heldout_Precision_95CI",

            "Heldout_F1_95CI",



            "Temporal_AUC_95CI",

            "Temporal_Sensitivity_95CI",

            "Temporal_Specificity_95CI",

            "Temporal_Precision_95CI",

            "Temporal_F1_95CI",



            "Validation_Support",

            "Pareto_Efficient",

            "Strategy_Role",

            "Primary_Recommended"

        ]

    ]

    .copy()



)





# ============================================================

# 17. ADD PUBLICATION LABEL

# ============================================================



part8a_final_main_results_compact[

    "Publication_Label"

] = np.where(



    part8a_final_main_results_compact[

        "Primary_Recommended"

    ],



    "Primary recommended",



    part8a_final_main_results_compact[

        "Strategy_Role"

    ]



)





# ============================================================

# 18. VERIFICATION

# ============================================================



verification_rows = []





for _, row in part8a_final_main_results.iterrows():



    screening_model = row[

        "Screening_Model"

    ]



    algorithm = row[

        "Algorithm"

    ]





    representative_match = (



        part5b_strategy_representatives[

            (

                part5b_strategy_representatives[

                    "Screening_Model"

                ]

                ==

                screening_model

            )

            &

            (

                part5b_strategy_representatives[

                    "Algorithm"

                ]

                ==

                algorithm

            )

        ]

    )





    representative_algorithm_matches = bool(



        len(

            representative_match

        )

        ==

        1



    )





    all_heldout_metrics_present = bool(



        all(



            pd.notna(

                row[

                    f"Heldout_{metric}"

                ]

            )



            for metric in MAIN_METRICS



        )



    )





    all_temporal_metrics_present = bool(



        all(



            pd.notna(

                row[

                    f"Temporal_{metric}"

                ]

            )



            for metric in MAIN_METRICS



        )



    )





    all_heldout_ci_present = bool(



        all(



            pd.notna(

                row[

                    f"Heldout_{metric}_CI_Lower"

                ]

            )

            and

            pd.notna(

                row[

                    f"Heldout_{metric}_CI_Upper"

                ]

            )



            for metric in MAIN_METRICS



        )



    )





    all_temporal_ci_present = bool(



        all(



            pd.notna(

                row[

                    f"Temporal_{metric}_CI_Lower"

                ]

            )

            and

            pd.notna(

                row[

                    f"Temporal_{metric}_CI_Upper"

                ]

            )



            for metric in MAIN_METRICS



        )



    )





    verification_rows.append({



        "Screening_Model":

            screening_model,



        "Algorithm":

            algorithm,



        "Representative_Algorithm_Matches_PART5B":

            representative_algorithm_matches,



        "All_Heldout_Metrics_Present":

            all_heldout_metrics_present,



        "All_Heldout_CI_Present":

            all_heldout_ci_present,



        "All_Temporal_Metrics_Present":

            all_temporal_metrics_present,



        "All_Temporal_CI_Present":

            all_temporal_ci_present,



        "Threshold_Matches_Heldout":

            row[

                "Threshold_Matches_Heldout"

            ],



        "Threshold_Matches_Temporal":

            row[

                "Threshold_Matches_Temporal"

            ],



        "Primary_Recommended":

            row[

                "Primary_Recommended"

            ],



        "PART8A_Row_Ready":

            bool(



                representative_algorithm_matches

                and

                all_heldout_metrics_present

                and

                all_heldout_ci_present

                and

                all_temporal_metrics_present

                and

                all_temporal_ci_present

                and

                row[

                    "Threshold_Matches_Heldout"

                ]

                and

                row[

                    "Threshold_Matches_Temporal"

                ]



            )



    })





part8a_verification = pd.DataFrame(

    verification_rows

)





# ============================================================

# 19. GLOBAL VERIFICATION

# ============================================================



strategy_count_correct = bool(



    len(

        part8a_final_main_results

    )

    ==

    7



)





all_expected_strategies_present = bool(



    set(

        part8a_final_main_results[

            "Screening_Model"

        ]

    )

    ==

    set(

        EXPECTED_STRATEGIES

    )



)





primary_rows = (



    part8a_final_main_results[

        part8a_final_main_results[

            "Primary_Recommended"

        ]

    ]



)





one_primary_recommendation = bool(



    len(

        primary_rows

    )

    ==

    1



)





if one_primary_recommendation:



    primary_model_correct = bool(



        primary_rows.iloc[

            0

        ][

            "Screening_Model"

        ]

        ==

        FINAL_SCREENING_MODEL



        and



        primary_rows.iloc[

            0

        ][

            "Algorithm"

        ]

        ==

        FINAL_ALGORITHM



    )



else:



    primary_model_correct = False





manifest_expected_rows = (



    len(

        EXPECTED_STRATEGIES

    )

    *

    len(

        MAIN_METRICS

    )

    *

    2



)





manifest_row_count_correct = bool(



    len(

        part8a_metric_source_manifest

    )

    ==

    manifest_expected_rows



)





part8a_global_verification = pd.DataFrame(



    [

        {



            "Expected_Strategies":

                7,



            "Observed_Strategies":

                len(

                    part8a_final_main_results

                ),



            "Strategy_Count_Correct":

                strategy_count_correct,



            "All_Expected_Strategies_Present":

                all_expected_strategies_present,



            "One_Primary_Recommendation":

                one_primary_recommendation,



            "Primary_Model_Correct":

                primary_model_correct,



            "Expected_Source_Manifest_Rows":

                manifest_expected_rows,



            "Observed_Source_Manifest_Rows":

                len(

                    part8a_metric_source_manifest

                ),



            "Source_Manifest_Row_Count_Correct":

                manifest_row_count_correct,



            "All_Strategy_Rows_Ready":

                bool(

                    part8a_verification[

                        "PART8A_Row_Ready"

                    ].all()

                ),



            "No_Model_Refit":

                True,



            "No_New_Predictions":

                True,



            "No_Threshold_Optimization":

                True,



            "No_Recalibration":

                True,



            "No_Algorithm_Reselection":

                True,



            "No_Performance_Recalculation":

                True,



            "PART8A_Ready":

                bool(



                    strategy_count_correct

                    and

                    all_expected_strategies_present

                    and

                    one_primary_recommendation

                    and

                    primary_model_correct

                    and

                    manifest_row_count_correct

                    and

                    part8a_verification[

                        "PART8A_Row_Ready"

                    ].all()



                )



        }



    ]



)





# ============================================================

# 20. DISPLAY

# ============================================================



print(

    "=" * 120

)



print(

    "PART 8A VERSION 3.0: FINAL MAIN RESULTS TABLE"

)



print(

    "=" * 120

)





print(

    "\nPrimary recommended strategy:"

)



print(

    f"{FINAL_SCREENING_MODEL} + {FINAL_ALGORITHM}"

)





print(

    "\n" + "=" * 120

)



print(

    "FINAL MAIN RESULTS — COMPACT PUBLICATION TABLE"

)



print(

    "=" * 120

)





display(



    part8a_final_main_results_compact



)





print(

    "\n" + "=" * 120

)



print(

    "TRACEABILITY VERIFICATION"

)



print(

    "=" * 120

)





display(

    part8a_verification

)





print(

    "\n" + "=" * 120

)



print(

    "GLOBAL VERIFICATION"

)



print(

    "=" * 120

)





display(

    part8a_global_verification

)





# ============================================================

# 21. HARD SAFETY CHECK

# ============================================================



if not part8a_global_verification[

    "PART8A_Ready"

].all():



    failed_rows = (



        part8a_verification[

            ~part8a_verification[

                "PART8A_Row_Ready"

            ]

        ]



    )





    if len(

        failed_rows

    ) > 0:



        print(

            "\nRows failing PART 8A verification:"

        )



        display(

            failed_rows

        )





    raise RuntimeError(



        "PART 8A verification failed. "

        "Do not use the final publication table."



    )





# ============================================================

# 22. EXPORT

# ============================================================



part8a_final_main_results.to_csv(



    "part8a_final_main_results.csv",



    index=False,



    encoding="utf-8-sig"



)





part8a_final_main_results_compact.to_csv(



    "part8a_final_main_results_compact.csv",



    index=False,



    encoding="utf-8-sig"



)





part8a_metric_source_manifest.to_csv(



    "part8a_metric_source_manifest.csv",



    index=False,



    encoding="utf-8-sig"



)





part8a_verification.to_csv(



    "part8a_verification.csv",



    index=False,



    encoding="utf-8-sig"



)





part8a_global_verification.to_csv(



    "part8a_global_verification.csv",



    index=False,



    encoding="utf-8-sig"



)





# ============================================================

# 23. FINAL REPORT

# ============================================================



print(

    "\n" + "=" * 120

)



print(

    "PART 8A VERSION 3.0 COMPLETED"

)



print(

    "=" * 120

)





print(

    f"\nStrategies reported          : "

    f"{len(part8a_final_main_results)}"

)





print(

    f"Primary recommended strategy : "

    f"{FINAL_SCREENING_MODEL}"

)





print(

    f"Primary algorithm            : "

    f"{FINAL_ALGORITHM}"

)





print(

    f"Main performance metrics     : "

    f"{len(MAIN_METRICS)}"

)





print(

    f"Metric source records        : "

    f"{len(part8a_metric_source_manifest)}"

)





print(

    "\nArtifacts created:"

)





print(

    "- part8a_final_main_results"

)



print(

    "- part8a_final_main_results_compact"

)



print(

    "- part8a_metric_source_manifest"

)



print(

    "- part8a_verification"

)



print(

    "- part8a_global_verification"

)





print(

    "\nExported files:"

)





print(

    "- part8a_final_main_results.csv"

)



print(

    "- part8a_final_main_results_compact.csv"

)



print(

    "- part8a_metric_source_manifest.csv"

)



print(

    "- part8a_verification.csv"

)



print(

    "- part8a_global_verification.csv"

)





print(

    "\nAll held-out 2024 confidence intervals were "

    "retrieved directly from PART 3C."

)





print(

    "All temporal 2025 confidence intervals were "

    "retrieved directly from PART 4B."

)





print(

    "Representative algorithms were preserved from PART 5B, "

    "and the primary recommendation was preserved from PART 5C."

)





print(

    "No model was refitted, no new prediction was generated, "

    "no threshold was modified, no recalibration was performed, "

    "and no performance metric was recalculated."

)





print(

    "\nProceed to PART 8B only after PART8A_Ready=True."

)
# ============================================================

# MDD DX ENGINE VERSION 3.0



### PART 8B: Final Supplementary Tables Package

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 8B: FINAL SUPPLEMENTARY TABLES PACKAGE
# PUBLICATION REPORTING LAYER
# ============================================================
#
# Objectives:
# • Assemble publication-ready supplementary tables
# • Preserve previously computed results exactly
# • Provide source traceability for every supplementary table
# • Verify completeness before manuscript reporting
#
# Supplementary Tables:
#
# S1  Screening strategy definitions / feature composition
# S2  OOF model performance and locked thresholds
# S3  Held-out 2024 bootstrap performance
# S4  Temporal 2025 bootstrap performance
# S5  Held-out 2024 raw calibration
# S6  Temporal 2025 raw calibration
# S7  OOF-derived recalibration / temporal transport
# S8  Final-model feature importance
# S9  Exploratory subgroup / fairness analysis
# S10 Strategy selection / Pareto framework
#
# IMPORTANT:
# • No model refitting
# • No prediction regeneration
# • No threshold modification
# • No probability recalibration fitting
# • No bootstrap rerun
# • No metric recalculation
# • No algorithm re-selection
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd


# ============================================================
# 2. SETTINGS
# ============================================================

FINAL_SCREENING_MODEL = "M2_Admin_2Q"
FINAL_ALGORITHM = "AdaBoostClassifier"

EXPECTED_STRATEGIES = [
    "M0_Admin",
    "M1_Admin_Behavior",
    "M2_Admin_2Q",
    "M3_Admin_2Q_Behavior",
    "M4_Conventional",
    "M5_Conventional_Admin",
    "M6_Full"
]

SUPPLEMENTARY_OUTPUT_DIR = "part8b_supplementary_tables"

os.makedirs(
    SUPPLEMENTARY_OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. REQUIRED CORE OBJECTS
# ============================================================

required_core_objects = [
    "part5b_strategy_representatives",
    "part5c_final_strategy_table",
    "part5c_strategy_roles",
    "part3c_bootstrap_results",
    "part4b_bootstrap_results",
    "part3d_calibration_results",
    "part4c1_calibration_results",
    "part6a_integrated_feature_importance",
    "part7a_subgroup_results",
    "part5b_pareto_comparison",
    "part8a_global_verification"
]


missing_core_objects = [
    obj
    for obj in required_core_objects
    if obj not in globals()
]


if missing_core_objects:
    raise NameError(
        "Missing required PART 8B objects:\n- "
        + "\n- ".join(missing_core_objects)
    )


# ============================================================
# 4. VERIFY PART 8A
# ============================================================

if (
    "PART8A_Ready"
    not in part8a_global_verification.columns
):
    raise KeyError(
        "PART8A_Ready is missing from "
        "part8a_global_verification."
    )


if not bool(
    part8a_global_verification[
        "PART8A_Ready"
    ].all()
):
    raise RuntimeError(
        "PART 8A is not ready. "
        "Do not construct PART 8B."
    )


# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def safe_copy(object_name):
    """
    Return a defensive copy of an existing DataFrame.
    """

    if object_name not in globals():
        return None

    obj = globals()[object_name]

    if not isinstance(obj, pd.DataFrame):
        return None

    return obj.copy()


def first_existing_object(candidate_names):
    """
    Return the first existing DataFrame from a list
    of possible artifact names.
    """

    for name in candidate_names:

        if (
            name in globals()
            and isinstance(
                globals()[name],
                pd.DataFrame
            )
        ):
            return name, globals()[name].copy()

    return None, None


def export_supplementary_table(
    df,
    table_number,
    short_name
):
    """
    Export one supplementary table without changing values.
    """

    filename = (
        f"Table_S{table_number}_"
        f"{short_name}.csv"
    )

    filepath = os.path.join(
        SUPPLEMENTARY_OUTPUT_DIR,
        filename
    )

    df.to_csv(
        filepath,
        index=False,
        encoding="utf-8-sig"
    )

    return filepath


# ============================================================
# 6. TABLE S1
#    SCREENING STRATEGY DEFINITIONS
# ============================================================

s1_columns = [
    "Screening_Model",
    "Strategy_Type",
    "Description",
    "Feature_Count",
    "Self_Report_Items",
    "Algorithm",
    "Locked_Threshold"
]


missing_s1_columns = [
    col
    for col in s1_columns
    if col not in
    part5b_strategy_representatives.columns
]


if missing_s1_columns:
    raise KeyError(
        "Missing columns required for Table S1: "
        + ", ".join(missing_s1_columns)
    )


part8b_table_s1 = (
    part5b_strategy_representatives[
        s1_columns
    ]
    .copy()
)


# Add role information without changing source metrics.

role_cols = [
    "Screening_Model",
    "Algorithm",
    "Strategy_Role",
    "Primary_Recommended",
    "Validation_Support_Summary",
    "Pareto_Efficient"
]


available_role_cols = [
    col
    for col in role_cols
    if col in part5c_strategy_roles.columns
]


part8b_table_s1 = (
    part8b_table_s1
    .merge(
        part5c_strategy_roles[
            available_role_cols
        ],
        on=[
            "Screening_Model",
            "Algorithm"
        ],
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 7. TABLE S2
#    OOF PERFORMANCE + LOCKED THRESHOLDS
# ============================================================

# Prefer the comprehensive PART 3A threshold artifact.

s2_source_name, s2_source = first_existing_object(
    [
        "threshold_summary",
        "part3a_threshold_verification",
        "part5a_integrated_performance"
    ]
)


if s2_source is None:
    raise NameError(
        "Could not find an appropriate source "
        "for Table S2."
    )


part8b_table_s2 = s2_source.copy()


# ============================================================
# 8. TABLE S3
#    HELD-OUT 2024 BOOTSTRAP PERFORMANCE
# ============================================================

part8b_table_s3 = (
    part3c_bootstrap_results
    .copy()
)


# Preserve only successfully computed CI rows if
# CI_Computed exists.

if "CI_Computed" in part8b_table_s3.columns:

    part8b_table_s3[
        "Required_CI_Available"
    ] = (
        part8b_table_s3[
            "CI_Computed"
        ]
    )


# ============================================================
# 9. TABLE S4
#    TEMPORAL 2025 BOOTSTRAP PERFORMANCE
# ============================================================

part8b_table_s4 = (
    part4b_bootstrap_results
    .copy()
)


if "CI_Computed" in part8b_table_s4.columns:

    part8b_table_s4[
        "Required_CI_Available"
    ] = (
        part8b_table_s4[
            "CI_Computed"
        ]
    )


# ============================================================
# 10. TABLE S5
#     HELD-OUT 2024 RAW CALIBRATION
# ============================================================

part8b_table_s5 = (
    part3d_calibration_results
    .copy()
)


# ============================================================
# 11. TABLE S6
#     TEMPORAL 2025 RAW CALIBRATION
# ============================================================

part8b_table_s6 = (
    part4c1_calibration_results
    .copy()
)


# ============================================================
# 12. TABLE S7
#     OOF-DERIVED RECALIBRATION / TEMPORAL TRANSPORT
# ============================================================

heldout_recal_source_name, heldout_recal = (
    first_existing_object(
        [
            "recalibrated_heldout_results",
            "part3d2_recalibration_publication_table",
            "oof_recalibration_results"
        ]
    )
)


temporal_recal_source_name, temporal_recal = (
    first_existing_object(
        [
            "part4c2_recalibration_results",
            "part4c2_recalibration_publication_table"
        ]
    )
)


if (
    heldout_recal is None
    and temporal_recal is None
):
    raise NameError(
        "No recalibration artifacts available "
        "for Table S7."
    )


s7_frames = []


if heldout_recal is not None:

    heldout_recal = heldout_recal.copy()

    heldout_recal[
        "Recalibration_Evaluation"
    ] = "Held-out 2024"

    heldout_recal[
        "Source_Artifact"
    ] = heldout_recal_source_name

    s7_frames.append(
        heldout_recal
    )


if temporal_recal is not None:

    temporal_recal = temporal_recal.copy()

    temporal_recal[
        "Recalibration_Evaluation"
    ] = "Temporal 2025"

    temporal_recal[
        "Source_Artifact"
    ] = temporal_recal_source_name

    s7_frames.append(
        temporal_recal
    )


part8b_table_s7 = pd.concat(
    s7_frames,
    ignore_index=True,
    sort=False
)


# ============================================================
# 13. TABLE S8
#     FINAL MODEL FEATURE IMPORTANCE
# ============================================================

part8b_table_s8 = (
    part6a_integrated_feature_importance
    .copy()
)


# Add explicit final-model identifiers for publication.

part8b_table_s8.insert(
    0,
    "Final_Screening_Model",
    FINAL_SCREENING_MODEL
)

part8b_table_s8.insert(
    1,
    "Final_Algorithm",
    FINAL_ALGORITHM
)


# ============================================================
# 14. TABLE S9
#     EXPLORATORY SUBGROUP / FAIRNESS ANALYSIS
# ============================================================

part8b_table_s9 = (
    part7a_subgroup_results
    .copy()
)


# Preserve sparse-event status from PART 7.

if (
    "Low_Event_Group"
    not in part8b_table_s9.columns
):

    # Do NOT derive or overwrite the status here.
    # It must remain traceable to PART 7 artifacts.

    small_group_source_name, small_group_source = (
        first_existing_object(
            [
                "part7a_small_group_flags",
                "part7b_estimability_summary"
            ]
        )
    )

else:

    small_group_source_name = (
        "part7a_subgroup_results"
    )


# ============================================================
# 15. TABLE S10
#     STRATEGY SELECTION / PARETO FRAMEWORK
# ============================================================

part8b_table_s10 = (
    part5b_pareto_comparison
    .copy()
)


# Merge final roles when possible.

if (
    "Screening_Model"
    in part8b_table_s10.columns
):

    role_merge_columns = [
        col
        for col in [
            "Screening_Model",
            "Algorithm",
            "Strategy_Role",
            "Primary_Recommended",
            "Validation_Support_Summary"
        ]
        if col in part5c_strategy_roles.columns
    ]


    merge_keys = [
        key
        for key in [
            "Screening_Model",
            "Algorithm"
        ]
        if (
            key in part8b_table_s10.columns
            and
            key in role_merge_columns
        )
    ]


    if merge_keys:

        role_add_columns = list(
            dict.fromkeys(
                merge_keys
                +
                [
                    col
                    for col in role_merge_columns
                    if col not in merge_keys
                ]
            )
        )


        part8b_table_s10 = (
            part8b_table_s10
            .merge(
                part5c_strategy_roles[
                    role_add_columns
                ],
                on=merge_keys,
                how="left",
                suffixes=(
                    "",
                    "_PART5C"
                )
            )
        )


# ============================================================
# 16. DEFINE SUPPLEMENTARY PACKAGE
# ============================================================

supplementary_tables = {

    "S1": {
        "title":
            "Screening strategy definitions and "
            "feature composition",
        "object":
            "part8b_table_s1",
        "df":
            part8b_table_s1,
        "source":
            "PART 5B + PART 5C"
    },

    "S2": {
        "title":
            "OOF performance and locked thresholds",
        "object":
            "part8b_table_s2",
        "df":
            part8b_table_s2,
        "source":
            s2_source_name
    },

    "S3": {
        "title":
            "Held-out 2024 performance with "
            "bootstrap 95% confidence intervals",
        "object":
            "part8b_table_s3",
        "df":
            part8b_table_s3,
        "source":
            "part3c_bootstrap_results"
    },

    "S4": {
        "title":
            "Temporal 2025 performance with "
            "bootstrap 95% confidence intervals",
        "object":
            "part8b_table_s4",
        "df":
            part8b_table_s4,
        "source":
            "part4b_bootstrap_results"
    },

    "S5": {
        "title":
            "Held-out 2024 raw probability "
            "calibration",
        "object":
            "part8b_table_s5",
        "df":
            part8b_table_s5,
        "source":
            "part3d_calibration_results"
    },

    "S6": {
        "title":
            "Temporal 2025 raw probability "
            "calibration",
        "object":
            "part8b_table_s6",
        "df":
            part8b_table_s6,
        "source":
            "part4c1_calibration_results"
    },

    "S7": {
        "title":
            "OOF-derived probability recalibration "
            "and temporal transport",
        "object":
            "part8b_table_s7",
        "df":
            part8b_table_s7,
        "source":
            (
                f"{heldout_recal_source_name}; "
                f"{temporal_recal_source_name}"
            )
    },

    "S8": {
        "title":
            "Feature importance for the final "
            "M2_Admin_2Q AdaBoost model",
        "object":
            "part8b_table_s8",
        "df":
            part8b_table_s8,
        "source":
            "part6a_integrated_feature_importance"
    },

    "S9": {
        "title":
            "Exploratory subgroup and fairness "
            "analysis",
        "object":
            "part8b_table_s9",
        "df":
            part8b_table_s9,
        "source":
            "part7a_subgroup_results"
    },

    "S10": {
        "title":
            "Burden-performance and Pareto "
            "strategy-selection framework",
        "object":
            "part8b_table_s10",
        "df":
            part8b_table_s10,
        "source":
            "part5b_pareto_comparison + PART 5C"
    }

}


# ============================================================
# 17. EXPORT ALL SUPPLEMENTARY TABLES
# ============================================================

exported_files = {}


for table_id, info in supplementary_tables.items():

    number = table_id.replace(
        "S",
        ""
    )

    short_name = {
        "S1":
            "screening_strategies",
        "S2":
            "oof_performance_thresholds",
        "S3":
            "heldout_2024_bootstrap",
        "S4":
            "temporal_2025_bootstrap",
        "S5":
            "heldout_2024_calibration",
        "S6":
            "temporal_2025_calibration",
        "S7":
            "oof_recalibration_transport",
        "S8":
            "final_model_feature_importance",
        "S9":
            "subgroup_fairness",
        "S10":
            "strategy_selection_pareto"
    }[
        table_id
    ]


    filepath = export_supplementary_table(
        info["df"],
        number,
        short_name
    )


    exported_files[
        table_id
    ] = filepath


# ============================================================
# 18. SUPPLEMENTARY MANIFEST
# ============================================================

manifest_rows = []


for table_id, info in supplementary_tables.items():

    df = info[
        "df"
    ]

    manifest_rows.append(
        {
            "Supplementary_Table":
                f"Table {table_id}",

            "Title":
                info[
                    "title"
                ],

            "Artifact":
                info[
                    "object"
                ],

            "Source":
                info[
                    "source"
                ],

            "Rows":
                len(df),

            "Columns":
                len(df.columns),

            "Exported_File":
                exported_files[
                    table_id
                ],

            "Nonempty":
                bool(
                    len(df) > 0
                )
        }
    )


part8b_supplementary_manifest = (
    pd.DataFrame(
        manifest_rows
    )
)


# ============================================================
# 19. TABLE-LEVEL VERIFICATION
# ============================================================

verification_rows = []


for table_id, info in supplementary_tables.items():

    df = info[
        "df"
    ]


    nonempty = bool(
        len(df) > 0
    )


    no_duplicate_columns = bool(
        not df.columns.duplicated().any()
    )


    file_exists = bool(
        os.path.exists(
            exported_files[
                table_id
            ]
        )
    )


    verification_rows.append(
        {
            "Supplementary_Table":
                f"Table {table_id}",

            "Rows":
                len(df),

            "Columns":
                len(df.columns),

            "Nonempty":
                nonempty,

            "No_Duplicate_Columns":
                no_duplicate_columns,

            "Exported_File_Exists":
                file_exists,

            "Table_Ready":
                bool(
                    nonempty
                    and
                    no_duplicate_columns
                    and
                    file_exists
                )
        }
    )


part8b_verification = pd.DataFrame(
    verification_rows
)


# ============================================================
# 20. SCIENTIFIC VERIFICATION
# ============================================================

# ------------------------------------------------------------
# S1: exactly seven screening strategies
# ------------------------------------------------------------

s1_strategy_count_ok = bool(
    part8b_table_s1[
        "Screening_Model"
    ].nunique()
    ==
    7
)


s1_strategy_set_ok = bool(
    set(
        part8b_table_s1[
            "Screening_Model"
        ]
    )
    ==
    set(
        EXPECTED_STRATEGIES
    )
)


# ------------------------------------------------------------
# S3: 35 held-out model-algorithm pairs expected
# ------------------------------------------------------------

s3_pairs = (
    part8b_table_s3[
        [
            "Screening_Model",
            "Algorithm"
        ]
    ]
    .drop_duplicates()
)


s3_pair_count_ok = bool(
    len(s3_pairs)
    ==
    35
)


# ------------------------------------------------------------
# S4: 35 temporal model-algorithm pairs expected
# ------------------------------------------------------------

s4_pairs = (
    part8b_table_s4[
        [
            "Screening_Model",
            "Algorithm"
        ]
    ]
    .drop_duplicates()
)


s4_pair_count_ok = bool(
    len(s4_pairs)
    ==
    35
)


# ------------------------------------------------------------
# S5 / S6 calibration completeness
# ------------------------------------------------------------

s5_pair_count_ok = bool(
    part8b_table_s5[
        [
            "Screening_Model",
            "Algorithm"
        ]
    ]
    .drop_duplicates()
    .shape[0]
    ==
    35
)


s6_pair_count_ok = bool(
    part8b_table_s6[
        [
            "Screening_Model",
            "Algorithm"
        ]
    ]
    .drop_duplicates()
    .shape[0]
    ==
    35
)


# ------------------------------------------------------------
# S8 final model identity
# ------------------------------------------------------------

s8_final_model_ok = bool(
    (
        part8b_table_s8[
            "Final_Screening_Model"
        ]
        ==
        FINAL_SCREENING_MODEL
    ).all()
    and
    (
        part8b_table_s8[
            "Final_Algorithm"
        ]
        ==
        FINAL_ALGORITHM
    ).all()
)


# ------------------------------------------------------------
# S9 subgroup analysis available
# ------------------------------------------------------------

s9_available = bool(
    len(
        part8b_table_s9
    )
    >
    0
)


# ------------------------------------------------------------
# S10 selection framework available
# ------------------------------------------------------------

s10_available = bool(
    len(
        part8b_table_s10
    )
    >
    0
)


# ============================================================
# 21. GLOBAL VERIFICATION
# ============================================================

all_tables_ready = bool(
    part8b_verification[
        "Table_Ready"
    ].all()
)


part8b_global_verification = pd.DataFrame(
    [
        {
            "Expected_Supplementary_Tables":
                10,

            "Observed_Supplementary_Tables":
                len(
                    supplementary_tables
                ),

            "All_Tables_Ready":
                all_tables_ready,

            "S1_Seven_Strategies":
                s1_strategy_count_ok,

            "S1_Strategy_Set_Correct":
                s1_strategy_set_ok,

            "S3_ThirtyFive_Model_Algorithm_Pairs":
                s3_pair_count_ok,

            "S4_ThirtyFive_Model_Algorithm_Pairs":
                s4_pair_count_ok,

            "S5_ThirtyFive_Calibration_Pairs":
                s5_pair_count_ok,

            "S6_ThirtyFive_Calibration_Pairs":
                s6_pair_count_ok,

            "S8_Final_Model_Correct":
                s8_final_model_ok,

            "S9_Fairness_Table_Available":
                s9_available,

            "S10_Selection_Table_Available":
                s10_available,

            "PART8A_Ready":
                bool(
                    part8a_global_verification[
                        "PART8A_Ready"
                    ].all()
                ),

            "No_Model_Refit":
                True,

            "No_New_Predictions":
                True,

            "No_Threshold_Modification":
                True,

            "No_New_Recalibration":
                True,

            "No_Bootstrap_Rerun":
                True,

            "No_Metric_Recalculation":
                True,

            "No_Algorithm_Reselection":
                True
        }
    ]
)


# ============================================================
# 22. FINAL READY FLAG
# ============================================================

part8b_ready = bool(
    all_tables_ready
    and
    s1_strategy_count_ok
    and
    s1_strategy_set_ok
    and
    s3_pair_count_ok
    and
    s4_pair_count_ok
    and
    s5_pair_count_ok
    and
    s6_pair_count_ok
    and
    s8_final_model_ok
    and
    s9_available
    and
    s10_available
    and
    part8a_global_verification[
        "PART8A_Ready"
    ].all()
)


part8b_global_verification[
    "PART8B_Ready"
] = part8b_ready


# ============================================================
# 23. EXPORT MANIFEST + VERIFICATION
# ============================================================

part8b_supplementary_manifest.to_csv(
    os.path.join(
        SUPPLEMENTARY_OUTPUT_DIR,
        "part8b_supplementary_manifest.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


part8b_verification.to_csv(
    os.path.join(
        SUPPLEMENTARY_OUTPUT_DIR,
        "part8b_verification.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


part8b_global_verification.to_csv(
    os.path.join(
        SUPPLEMENTARY_OUTPUT_DIR,
        "part8b_global_verification.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 24. DISPLAY MANIFEST
# ============================================================

print(
    "=" * 120
)

print(
    "PART 8B VERSION 3.0: "
    "FINAL SUPPLEMENTARY TABLES PACKAGE"
)

print(
    "=" * 120
)


print(
    "\nSUPPLEMENTARY TABLE MANIFEST"
)

display(
    part8b_supplementary_manifest
)


print(
    "\n" + "=" * 120
)

print(
    "TABLE-LEVEL VERIFICATION"
)

print(
    "=" * 120
)

display(
    part8b_verification
)


print(
    "\n" + "=" * 120
)

print(
    "GLOBAL VERIFICATION"
)

print(
    "=" * 120
)

display(
    part8b_global_verification
)


# ============================================================
# 25. HARD SAFETY CHECK
# ============================================================

if not part8b_ready:

    failed_tables = (
        part8b_verification[
            ~part8b_verification[
                "Table_Ready"
            ]
        ]
    )


    if len(
        failed_tables
    ) > 0:

        print(
            "\nSupplementary tables failing "
            "verification:"
        )

        display(
            failed_tables
        )


    raise RuntimeError(
        "PART 8B verification failed. "
        "Do not use the supplementary package."
    )


# ============================================================
# 26. FINAL REPORT
# ============================================================

print(
    "\n" + "=" * 120
)

print(
    "PART 8B VERSION 3.0 COMPLETED"
)

print(
    "=" * 120
)


print(
    f"\nSupplementary tables created : "
    f"{len(supplementary_tables)}"
)

print(
    f"Final model                  : "
    f"{FINAL_SCREENING_MODEL} + "
    f"{FINAL_ALGORITHM}"
)

print(
    f"Output directory             : "
    f"{SUPPLEMENTARY_OUTPUT_DIR}"
)


print(
    "\nSupplementary tables:"
)

for table_id, info in supplementary_tables.items():

    print(
        f"- Table {table_id}: "
        f"{info['title']}"
    )


print(
    "\nArtifacts created:"
)

for i in range(
    1,
    11
):

    print(
        f"- part8b_table_s{i}"
    )


print(
    "- part8b_supplementary_manifest"
)

print(
    "- part8b_verification"
)

print(
    "- part8b_global_verification"
)


print(
    "\nNo model was refitted, no new prediction was "
    "generated, no threshold was modified, no new "
    "recalibration mapping was fitted, no bootstrap "
    "analysis was rerun, and no performance metric "
    "was recalculated."
)


print(
    "\nPART 8B is a publication-reporting layer only."
)


print(
    "\nProceed only after PART8B_Ready=True."
)

### PART 8C: Final Figures and Publication Package

In [ ]:
# ============================================================
# PRE-CHECK FOR PART 8C VERSION 3.0
# ============================================================

objects_to_check = {

    "part5b_pareto_comparison":
        part5b_pareto_comparison,

    "part5c_final_strategy_table":
        part5c_final_strategy_table,

    "part6a_integrated_feature_importance":
        part6a_integrated_feature_importance,

    "part6a_primary_interpretation_table":
        part6a_primary_interpretation_table,

    "part8a_final_main_results":
        part8a_final_main_results,

    "part8a_final_main_results_compact":
        part8a_final_main_results_compact
}


for name, df in objects_to_check.items():

    print("\n" + "=" * 110)
    print(name)
    print("=" * 110)

    print("Shape:", df.shape)

    print("\nColumns:")

    for i, col in enumerate(df.columns):
        print(f"{i:02d}: {col}")

    print("\nFirst 3 rows:")

    display(
        df.head(3)
    )

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 8C: FINAL FIGURES AND PUBLICATION PACKAGE
# PUBLICATION REPORTING LAYER
# ============================================================
#
# Purpose:
# Create publication-ready figures from EXISTING artifacts only.
#
# Main figures generated here:
# Figure 2. Burden-performance comparison across 7 strategies
# Figure 3. Final-model validation performance with 95% CI
# Figure 4. Final-model feature importance
#
# Figure 1 (study/model-development framework) is intentionally
# NOT generated here because it is a conceptual diagram rather
# than an analytical figure.
#
# IMPORTANT:
# - No model refitting
# - No new predictions
# - No threshold optimization
# - No recalibration
# - No bootstrap rerun
# - No model/algorithm re-selection
# - No new performance metric calculation
#
# Inputs:
# - part8a_final_main_results
# - part8a_final_main_results_compact
# - part5c_final_recommendation
# - part6a_primary_interpretation_table
# - part8a_global_verification
#
# Optional existing calibration plot manifests:
# - part3d_calibration_plot_manifest
# - part4c1_calibration_plot_manifest
# - part4c2_recalibration_plot_manifest
#
# Outputs:
# - part8c_figure_manifest
# - part8c_supplementary_figure_manifest
# - part8c_verification
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 2. SETTINGS
# ============================================================

FINAL_SCREENING_MODEL = "M2_Admin_2Q"
FINAL_ALGORITHM = "AdaBoostClassifier"

EXPECTED_STRATEGIES = [
    "M0_Admin",
    "M1_Admin_Behavior",
    "M2_Admin_2Q",
    "M3_Admin_2Q_Behavior",
    "M4_Conventional",
    "M5_Conventional_Admin",
    "M6_Full"
]

OUTPUT_DIRECTORY = Path(
    "part8c_final_figures"
)

SUPPLEMENTARY_DIRECTORY = Path(
    "part8c_supplementary_figures"
)

OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

SUPPLEMENTARY_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_DPI = 300

# Slightly shorter manuscript labels
STRATEGY_LABELS = {
    "M0_Admin":
        "M0 Admin",
    "M1_Admin_Behavior":
        "M1 Admin + Behavior",
    "M2_Admin_2Q":
        "M2 Admin + Thai 2Q",
    "M3_Admin_2Q_Behavior":
        "M3 Admin + Thai 2Q + Behavior",
    "M4_Conventional":
        "M4 Conventional",
    "M5_Conventional_Admin":
        "M5 Conventional + Admin",
    "M6_Full":
        "M6 Full"
}


# ============================================================
# 3. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "part8a_final_main_results",
    "part8a_final_main_results_compact",
    "part5c_final_recommendation",
    "part6a_primary_interpretation_table",
    "part8a_global_verification"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise NameError(
        "Missing required PART 8C objects: "
        + ", ".join(
            missing_objects
        )
    )


for object_name in [
    "part8a_final_main_results",
    "part8a_final_main_results_compact",
    "part6a_primary_interpretation_table",
    "part8a_global_verification"
]:

    if not isinstance(
        globals()[object_name],
        pd.DataFrame
    ):

        raise TypeError(
            f"{object_name} must be a pandas DataFrame."
        )


if not bool(
    part8a_global_verification[
        "PART8A_Ready"
    ].all()
):

    raise RuntimeError(
        "PART 8A is not ready."
    )


# ============================================================
# 4. VERIFY FINAL RECOMMENDATION
# ============================================================

if (
    part5c_final_recommendation[
        "Primary_Recommended_Strategy"
    ]
    !=
    FINAL_SCREENING_MODEL
):

    raise RuntimeError(
        "Final strategy does not match M2_Admin_2Q."
    )


if (
    part5c_final_recommendation[
        "Primary_Recommended_Algorithm"
    ]
    !=
    FINAL_ALGORITHM
):

    raise RuntimeError(
        "Final algorithm does not match AdaBoostClassifier."
    )


# ============================================================
# 5. SCHEMA CHECKS
# ============================================================

required_main_columns = [
    "Screening_Model",
    "Self_Report_Items",
    "Algorithm",
    "Primary_Recommended",

    "Temporal_AUC",
    "Temporal_AUC_CI_Lower",
    "Temporal_AUC_CI_Upper",

    "Heldout_AUC",
    "Heldout_AUC_CI_Lower",
    "Heldout_AUC_CI_Upper",

    "Heldout_Sensitivity",
    "Heldout_Sensitivity_CI_Lower",
    "Heldout_Sensitivity_CI_Upper",

    "Heldout_Specificity",
    "Heldout_Specificity_CI_Lower",
    "Heldout_Specificity_CI_Upper",

    "Heldout_Precision",
    "Heldout_Precision_CI_Lower",
    "Heldout_Precision_CI_Upper",

    "Heldout_F1",
    "Heldout_F1_CI_Lower",
    "Heldout_F1_CI_Upper",

    "Temporal_Sensitivity",
    "Temporal_Sensitivity_CI_Lower",
    "Temporal_Sensitivity_CI_Upper",

    "Temporal_Specificity",
    "Temporal_Specificity_CI_Lower",
    "Temporal_Specificity_CI_Upper",

    "Temporal_Precision",
    "Temporal_Precision_CI_Lower",
    "Temporal_Precision_CI_Upper",

    "Temporal_F1",
    "Temporal_F1_CI_Lower",
    "Temporal_F1_CI_Upper"
]


missing_main_columns = [
    col
    for col in required_main_columns
    if col not in part8a_final_main_results.columns
]


if missing_main_columns:

    raise KeyError(
        "part8a_final_main_results is missing required "
        "columns: "
        + ", ".join(
            missing_main_columns
        )
    )


required_importance_columns = [
    "Feature",
    "Heldout_AUC_Importance_Mean",
    "Heldout_AUC_Importance_SD",
    "Heldout_AUC_Rank",
    "Temporal_AUC_Importance_Mean",
    "Temporal_AUC_Rank"
]


missing_importance_columns = [
    col
    for col in required_importance_columns
    if col not in part6a_primary_interpretation_table.columns
]


if missing_importance_columns:

    raise KeyError(
        "part6a_primary_interpretation_table is missing "
        "required columns: "
        + ", ".join(
            missing_importance_columns
        )
    )


# ============================================================
# 6. VERIFY 7 STRATEGIES
# ============================================================

observed_strategies = set(
    part8a_final_main_results[
        "Screening_Model"
    ]
    .astype(str)
    .tolist()
)


if observed_strategies != set(
    EXPECTED_STRATEGIES
):

    raise RuntimeError(
        "PART 8A final results do not contain the expected "
        "7 strategies."
    )


# ============================================================
# 7. FIGURE MANIFEST CONTAINERS
# ============================================================

figure_manifest_rows = []

supplementary_manifest_rows = []


# ============================================================
# 8. FIGURE 2
# BURDEN–PERFORMANCE COMPARISON
# ============================================================
#
# X = number of self-report items
# Y = temporal 2025 AUC
# Error bars = temporal bootstrap 95% CI
#
# This figure visually communicates the central research
# question: predictive performance versus respondent burden.
# ============================================================

figure2_data = (
    part8a_final_main_results[
        [
            "Screening_Model",
            "Self_Report_Items",
            "Algorithm",
            "Temporal_AUC",
            "Temporal_AUC_CI_Lower",
            "Temporal_AUC_CI_Upper",
            "Primary_Recommended"
        ]
    ]
    .copy()
)


figure2_data[
    "Strategy_Label"
] = (
    figure2_data[
        "Screening_Model"
    ]
    .map(
        STRATEGY_LABELS
    )
)


figure2_data[
    "AUC_Error_Lower"
] = (
    figure2_data[
        "Temporal_AUC"
    ]
    -
    figure2_data[
        "Temporal_AUC_CI_Lower"
    ]
)


figure2_data[
    "AUC_Error_Upper"
] = (
    figure2_data[
        "Temporal_AUC_CI_Upper"
    ]
    -
    figure2_data[
        "Temporal_AUC"
    ]
)


figure, axis = plt.subplots(
    figsize=(
        9,
        6
    )
)


for _, row in figure2_data.iterrows():

    axis.errorbar(
        x=row[
            "Self_Report_Items"
        ],
        y=row[
            "Temporal_AUC"
        ],
        yerr=np.array(
            [
                [
                    row[
                        "AUC_Error_Lower"
                    ]
                ],
                [
                    row[
                        "AUC_Error_Upper"
                    ]
                ]
            ]
        ),
        fmt="o",
        capsize=4
    )


    axis.annotate(
        row[
            "Strategy_Label"
        ],
        (
            row[
                "Self_Report_Items"
            ],
            row[
                "Temporal_AUC"
            ]
        ),
        xytext=(
            5,
            5
        ),
        textcoords="offset points",
        fontsize=8
    )


axis.set_xlabel(
    "Additional self-report items"
)


axis.set_ylabel(
    "Temporal 2025 ROC AUC"
)


axis.set_title(
    "Burden–Performance Comparison Across Screening Strategies"
)


axis.set_ylim(
    max(
        0.45,
        float(
            figure2_data[
                "Temporal_AUC_CI_Lower"
            ].min()
        )
        -
        0.05
    ),
    min(
        1.0,
        float(
            figure2_data[
                "Temporal_AUC_CI_Upper"
            ].max()
        )
        +
        0.03
    )
)


axis.grid(
    alpha=0.25
)


figure.tight_layout()


figure2_png = (
    OUTPUT_DIRECTORY
    /
    "Figure_2_Burden_Performance_Comparison.png"
)


figure2_pdf = (
    OUTPUT_DIRECTORY
    /
    "Figure_2_Burden_Performance_Comparison.pdf"
)


figure.savefig(
    figure2_png,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)


figure.savefig(
    figure2_pdf,
    bbox_inches="tight"
)


plt.close(
    figure
)


figure_manifest_rows.append({

    "Figure":
        "Figure 2",

    "Title":
        "Burden–performance comparison across screening strategies",

    "Primary_Source":
        "part8a_final_main_results",

    "PNG_File":
        str(
            figure2_png
        ),

    "PDF_File":
        str(
            figure2_pdf
        ),

    "Analysis_Newly_Performed":
        False,

    "Purpose":
        (
            "Visualize temporal discrimination against "
            "additional respondent burden."
        )
})


# ============================================================
# 9. FIGURE 3
# FINAL MODEL VALIDATION PERFORMANCE
# ============================================================
#
# Compare held-out 2024 vs temporal 2025 point estimates and
# bootstrap 95% CIs for the final model.
# ============================================================

final_model_rows = (
    part8a_final_main_results[
        (
            part8a_final_main_results[
                "Screening_Model"
            ]
            ==
            FINAL_SCREENING_MODEL
        )
        &
        (
            part8a_final_main_results[
                "Algorithm"
            ]
            ==
            FINAL_ALGORITHM
        )
    ]
)


if len(
    final_model_rows
) != 1:

    raise RuntimeError(
        "Could not uniquely identify final model in PART 8A."
    )


final_model_row = final_model_rows.iloc[
    0
]


figure3_metrics = [
    "AUC",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1"
]


figure3_rows = []


for metric in figure3_metrics:

    figure3_rows.append({

        "Metric":
            metric,

        "Validation_Set":
            "Held-out 2024",

        "Estimate":
            float(
                final_model_row[
                    f"Heldout_{metric}"
                ]
            ),

        "CI_Lower":
            float(
                final_model_row[
                    f"Heldout_{metric}_CI_Lower"
                ]
            ),

        "CI_Upper":
            float(
                final_model_row[
                    f"Heldout_{metric}_CI_Upper"
                ]
            )
    })


    figure3_rows.append({

        "Metric":
            metric,

        "Validation_Set":
            "Temporal 2025",

        "Estimate":
            float(
                final_model_row[
                    f"Temporal_{metric}"
                ]
            ),

        "CI_Lower":
            float(
                final_model_row[
                    f"Temporal_{metric}_CI_Lower"
                ]
            ),

        "CI_Upper":
            float(
                final_model_row[
                    f"Temporal_{metric}_CI_Upper"
                ]
            )
    })


figure3_data = pd.DataFrame(
    figure3_rows
)


metric_positions = {
    metric:
        index
    for index, metric
    in enumerate(
        figure3_metrics
    )
}


figure, axis = plt.subplots(
    figsize=(
        9,
        6
    )
)


offset_map = {
    "Held-out 2024":
        -0.10,
    "Temporal 2025":
        0.10
}


for validation_set in [
    "Held-out 2024",
    "Temporal 2025"
]:

    subset = (
        figure3_data[
            figure3_data[
                "Validation_Set"
            ]
            ==
            validation_set
        ]
    )


    x_positions = np.array(
        [
            metric_positions[
                metric
            ]
            +
            offset_map[
                validation_set
            ]
            for metric in subset[
                "Metric"
            ]
        ]
    )


    estimates = subset[
        "Estimate"
    ].to_numpy(
        dtype=float
    )


    lower_errors = (
        estimates
        -
        subset[
            "CI_Lower"
        ].to_numpy(
            dtype=float
        )
    )


    upper_errors = (
        subset[
            "CI_Upper"
        ].to_numpy(
            dtype=float
        )
        -
        estimates
    )


    axis.errorbar(
        x=x_positions,
        y=estimates,
        yerr=np.vstack(
            [
                lower_errors,
                upper_errors
            ]
        ),
        fmt="o",
        capsize=4,
        label=validation_set
    )


axis.set_xticks(
    list(
        metric_positions.values()
    )
)


axis.set_xticklabels(
    list(
        metric_positions.keys()
    )
)


axis.set_ylabel(
    "Performance estimate"
)


axis.set_xlabel(
    "Metric"
)


axis.set_title(
    "Validation Performance of the Final Low-Burden Model\n"
    "M2 Admin + Thai 2Q with AdaBoost"
)


axis.set_ylim(
    0,
    1.03
)


axis.legend()


axis.grid(
    alpha=0.25
)


figure.tight_layout()


figure3_png = (
    OUTPUT_DIRECTORY
    /
    "Figure_3_Final_Model_Validation_Performance.png"
)


figure3_pdf = (
    OUTPUT_DIRECTORY
    /
    "Figure_3_Final_Model_Validation_Performance.pdf"
)


figure.savefig(
    figure3_png,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)


figure.savefig(
    figure3_pdf,
    bbox_inches="tight"
)


plt.close(
    figure
)


figure_manifest_rows.append({

    "Figure":
        "Figure 3",

    "Title":
        "Validation performance of the final low-burden model",

    "Primary_Source":
        "part8a_final_main_results",

    "PNG_File":
        str(
            figure3_png
        ),

    "PDF_File":
        str(
            figure3_pdf
        ),

    "Analysis_Newly_Performed":
        False,

    "Purpose":
        (
            "Compare held-out 2024 and temporal 2025 "
            "performance with bootstrap 95% confidence intervals."
        )
})


# ============================================================
# 10. FIGURE 4
# FINAL MODEL FEATURE IMPORTANCE
# ============================================================
#
# Primary interpretation = held-out AUC permutation importance.
# Temporal importance is included as a transportability check.
# ============================================================

figure4_data = (
    part6a_primary_interpretation_table[
        [
            "Feature",
            "Heldout_AUC_Importance_Mean",
            "Heldout_AUC_Importance_SD",
            "Heldout_AUC_Rank",
            "Temporal_AUC_Importance_Mean",
            "Temporal_AUC_Rank"
        ]
    ]
    .copy()
)


figure4_data = (
    figure4_data
    .sort_values(
        "Heldout_AUC_Importance_Mean",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


figure, axis = plt.subplots(
    figsize=(
        8,
        6
    )
)


axis.barh(
    figure4_data[
        "Feature"
    ],
    figure4_data[
        "Heldout_AUC_Importance_Mean"
    ],
    xerr=figure4_data[
        "Heldout_AUC_Importance_SD"
    ]
)


axis.axvline(
    0,
    linewidth=1
)


axis.set_xlabel(
    "Decrease in held-out ROC AUC after feature permutation"
)


axis.set_ylabel(
    "Feature"
)


axis.set_title(
    "Feature Importance of the Final Low-Burden Model\n"
    "Held-out 2024 Permutation Importance"
)


axis.grid(
    alpha=0.25
)


figure.tight_layout()


figure4_png = (
    OUTPUT_DIRECTORY
    /
    "Figure_4_Final_Model_Feature_Importance.png"
)


figure4_pdf = (
    OUTPUT_DIRECTORY
    /
    "Figure_4_Final_Model_Feature_Importance.pdf"
)


figure.savefig(
    figure4_png,
    dpi=FIGURE_DPI,
    bbox_inches="tight"
)


figure.savefig(
    figure4_pdf,
    bbox_inches="tight"
)


plt.close(
    figure
)


figure_manifest_rows.append({

    "Figure":
        "Figure 4",

    "Title":
        "Feature importance of the final low-burden model",

    "Primary_Source":
        "part6a_primary_interpretation_table",

    "PNG_File":
        str(
            figure4_png
        ),

    "PDF_File":
        str(
            figure4_pdf
        ),

    "Analysis_Newly_Performed":
        False,

    "Purpose":
        (
            "Display primary held-out 2024 model-agnostic "
            "permutation importance."
        )
})


# ============================================================
# 11. MAIN FIGURE MANIFEST
# ============================================================

part8c_figure_manifest = pd.DataFrame(
    figure_manifest_rows
)


# ============================================================
# 12. SUPPLEMENTARY FIGURE MANIFEST
# ============================================================
#
# Existing calibration plots are indexed here.
# The plots themselves are NOT regenerated.
# ============================================================

optional_plot_manifests = [

    (
        "Held-out 2024 raw calibration",
        "part3d_calibration_plot_manifest"
    ),

    (
        "Temporal 2025 raw calibration",
        "part4c1_calibration_plot_manifest"
    ),

    (
        "Temporal 2025 transported recalibration",
        "part4c2_recalibration_plot_manifest"
    )
]


for figure_group, object_name in optional_plot_manifests:

    if (
        object_name
        in globals()
        and
        isinstance(
            globals()[
                object_name
            ],
            pd.DataFrame
        )
    ):

        manifest = globals()[
            object_name
        ].copy()


        for _, row in manifest.iterrows():

            supplementary_manifest_rows.append({

                "Figure_Group":
                    figure_group,

                "Source_Manifest":
                    object_name,

                "Screening_Model":
                    row.get(
                        "Screening_Model",
                        ""
                    ),

                "Algorithm":
                    row.get(
                        "Algorithm",
                        ""
                    ),

                "PNG_File":
                    row.get(
                        "PNG_File",
                        row.get(
                            "PNG_Path",
                            ""
                        )
                    ),

                "PDF_File":
                    row.get(
                        "PDF_File",
                        row.get(
                            "PDF_Path",
                            ""
                        )
                    ),

                "Plot_Created":
                    row.get(
                        "Plot_Created",
                        True
                    )
            })


part8c_supplementary_figure_manifest = pd.DataFrame(
    supplementary_manifest_rows
)


# ============================================================
# 13. FIGURE LEGENDS
# ============================================================

part8c_figure_legends = pd.DataFrame(
    [
        {
            "Figure":
                "Figure 1",

            "Legend":
                (
                    "Study and model-development framework. "
                    "The 2024 cohort was divided into development "
                    "and held-out validation partitions. "
                    "Representative algorithms and classification "
                    "thresholds were derived using development "
                    "data only, followed by fixed held-out 2024 "
                    "and temporal 2025 validation. Screening "
                    "strategies were subsequently compared using "
                    "a burden-performance framework."
                )
        },

        {
            "Figure":
                "Figure 2",

            "Legend":
                (
                    "Burden-performance comparison across the "
                    "seven screening strategies. The x-axis "
                    "represents the number of additional "
                    "self-report items, and the y-axis represents "
                    "temporal 2025 ROC AUC. Error bars indicate "
                    "95% confidence intervals from stratified "
                    "participant-level bootstrap resampling. "
                    "The figure illustrates the trade-off between "
                    "respondent burden and temporal discrimination."
                )
        },

        {
            "Figure":
                "Figure 3",

            "Legend":
                (
                    "Performance of the final recommended "
                    "M2_Admin_2Q AdaBoost model in the held-out "
                    "2024 and temporal 2025 cohorts. Point estimates "
                    "are shown for ROC AUC, sensitivity, specificity, "
                    "precision, and F1 score; error bars indicate "
                    "bootstrap 95% confidence intervals. The "
                    "classification threshold was locked using "
                    "genuine 2024 out-of-fold predictions and was "
                    "not re-optimized in either validation cohort."
                )
        },

        {
            "Figure":
                "Figure 4",

            "Legend":
                (
                    "Model-agnostic permutation feature importance "
                    "for the final M2_Admin_2Q AdaBoost model in the "
                    "held-out 2024 cohort. Importance represents the "
                    "decrease in ROC AUC after permutation of each "
                    "feature. Positive values indicate reduced "
                    "discrimination after feature disruption. "
                    "Feature importance reflects predictive "
                    "contribution within the fitted model and should "
                    "not be interpreted causally."
                )
        }
    ]
)


# ============================================================
# 14. VERIFICATION
# ============================================================

main_files_exist = bool(

    all(
        Path(
            path
        ).exists()

        for path in (
            part8c_figure_manifest[
                "PNG_File"
            ].tolist()
            +
            part8c_figure_manifest[
                "PDF_File"
            ].tolist()
        )
    )
)


three_main_figures_created = bool(
    len(
        part8c_figure_manifest
    )
    ==
    3
)


final_model_unique = bool(
    len(
        final_model_rows
    )
    ==
    1
)


figure2_all_strategies_present = bool(
    set(
        figure2_data[
            "Screening_Model"
        ]
    )
    ==
    set(
        EXPECTED_STRATEGIES
    )
)


figure3_all_metrics_present = bool(
    set(
        figure3_data[
            "Metric"
        ]
    )
    ==
    set(
        figure3_metrics
    )
    and
    figure3_data[
        "Validation_Set"
    ].nunique()
    ==
    2
)


figure4_all_features_present = bool(
    len(
        figure4_data
    )
    ==
    len(
        part6a_primary_interpretation_table
    )
)


part8c_verification = pd.DataFrame(
    [
        {
            "Main_Figures_Created":
                three_main_figures_created,

            "All_Main_Figure_Files_Exist":
                main_files_exist,

            "Figure2_All_7_Strategies_Present":
                figure2_all_strategies_present,

            "Figure3_Final_Model_Unique":
                final_model_unique,

            "Figure3_All_5_Metrics_Present":
                figure3_all_metrics_present,

            "Figure4_All_Features_Present":
                figure4_all_features_present,

            "Final_Model_Is_M2_Admin_2Q_AdaBoost":
                bool(
                    FINAL_SCREENING_MODEL
                    ==
                    "M2_Admin_2Q"
                    and
                    FINAL_ALGORITHM
                    ==
                    "AdaBoostClassifier"
                ),

            "No_Model_Refit":
                True,

            "No_New_Predictions":
                True,

            "No_Threshold_Optimization":
                True,

            "No_Recalibration":
                True,

            "No_Bootstrap_Rerun":
                True,

            "No_Model_Reselection":
                True,

            "No_New_Performance_Metrics":
                True,

            "PART8C_Ready":
                bool(
                    three_main_figures_created
                    and
                    main_files_exist
                    and
                    figure2_all_strategies_present
                    and
                    final_model_unique
                    and
                    figure3_all_metrics_present
                    and
                    figure4_all_features_present
                )
        }
    ]
)


# ============================================================
# 15. EXPORT SUPPORTING TABLES
# ============================================================

figure2_data.to_csv(
    OUTPUT_DIRECTORY
    /
    "Figure_2_source_data.csv",
    index=False,
    encoding="utf-8-sig"
)


figure3_data.to_csv(
    OUTPUT_DIRECTORY
    /
    "Figure_3_source_data.csv",
    index=False,
    encoding="utf-8-sig"
)


figure4_data.to_csv(
    OUTPUT_DIRECTORY
    /
    "Figure_4_source_data.csv",
    index=False,
    encoding="utf-8-sig"
)


part8c_figure_manifest.to_csv(
    OUTPUT_DIRECTORY
    /
    "part8c_figure_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


part8c_supplementary_figure_manifest.to_csv(
    SUPPLEMENTARY_DIRECTORY
    /
    "part8c_supplementary_figure_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


part8c_figure_legends.to_csv(
    OUTPUT_DIRECTORY
    /
    "part8c_figure_legends.csv",
    index=False,
    encoding="utf-8-sig"
)


part8c_verification.to_csv(
    OUTPUT_DIRECTORY
    /
    "part8c_verification.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 16. DISPLAY
# ============================================================

print("=" * 120)
print(
    "PART 8C VERSION 3.0: FINAL FIGURES AND "
    "PUBLICATION PACKAGE"
)
print("=" * 120)


print("\nMAIN FIGURE MANIFEST")

display(
    part8c_figure_manifest
)


print("\nFIGURE LEGENDS")

display(
    part8c_figure_legends
)


print("\nPART 8C VERIFICATION")

display(
    part8c_verification
)


# ============================================================
# 17. HARD SAFETY CHECK
# ============================================================

if not part8c_verification[
    "PART8C_Ready"
].all():

    raise RuntimeError(
        "PART 8C verification failed."
    )


# ============================================================
# 18. FINAL REPORT
# ============================================================

print("\n" + "=" * 120)
print("PART 8C VERSION 3.0 COMPLETED")
print("=" * 120)


print(
    f"Main analytical figures created : "
    f"{len(part8c_figure_manifest)}"
)


print(
    f"Final model                     : "
    f"{FINAL_SCREENING_MODEL} + "
    f"{FINAL_ALGORITHM}"
)


print(
    f"Main figure directory           : "
    f"{OUTPUT_DIRECTORY}"
)


print(
    f"Supplementary figure directory  : "
    f"{SUPPLEMENTARY_DIRECTORY}"
)


print("\nMain manuscript figures:")

print(
    "- Figure 1: Study/model-development framework "
    "(conceptual; prepare separately)"
)

print(
    "- Figure 2: Burden-performance comparison"
)

print(
    "- Figure 3: Final-model validation performance"
)

print(
    "- Figure 4: Final-model feature importance"
)


print(
    "\nExisting calibration plots were indexed for "
    "supplementary reporting when their manifests were "
    "available."
)


print(
    "\nNo model was refitted, no new predictions were "
    "generated, no threshold was optimized, no recalibration "
    "was performed, no bootstrap analysis was rerun, and no "
    "model was re-selected."
)


print(
    "\nProceed to PART 8D only after PART8C_Ready=True."
)

### PART 8D: Manuscript-Ready Reporting Package

In [ ]:
# ============================================================
# MDD DX ENGINE VERSION 3.0
# PART 8D: MANUSCRIPT-READY REPORTING PACKAGE
# ============================================================
#
# Objectives:
# 1. Freeze the final analytical identity of the study
# 2. Assemble manuscript-ready reporting components
# 3. Generate Methods and Results drafts from existing artifacts
# 4. Generate model-selection rationale
# 5. Generate table footnotes and figure legends
# 6. Preserve complete traceability to PARTS 5–8
#
# IMPORTANT:
# - No model refitting
# - No new predictions
# - No threshold optimization
# - No recalibration
# - No bootstrap rerun
# - No feature selection
# - No algorithm re-selection
# - No new statistical testing
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd


# ============================================================
# 2. SETTINGS
# ============================================================

FINAL_SCREENING_MODEL = "M2_Admin_2Q"
FINAL_ALGORITHM = "AdaBoostClassifier"

OUTPUT_DIR = "part8d_manuscript_reporting"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. REQUIRED OBJECTS
# ============================================================

required_objects = [

    "part5c_final_recommendation",
    "part5c_final_strategy_table",

    "part6b_final_strategy_evidence",
    "part6b_feature_interpretation",
    "part6b_temporal_consistency",

    "part7b_overall_interpretation",

    "part8a_final_main_results",
    "part8a_final_main_results_compact",
    "part8a_global_verification",

    "part8b_global_verification",

    "part8c_figure_legends",
    "part8c_verification"
]


missing_objects = [

    object_name

    for object_name in required_objects

    if object_name not in globals()

]


if missing_objects:

    raise NameError(

        "Missing required PART 8D objects:\n- "

        + "\n- ".join(
            missing_objects
        )

    )


# ============================================================
# 4. VERIFY PREVIOUS REPORTING PARTS
# ============================================================

if not bool(
    part8a_global_verification[
        "PART8A_Ready"
    ].all()
):

    raise RuntimeError(
        "PART 8A is not ready."
    )


if not bool(
    part8b_global_verification[
        "PART8B_Ready"
    ].all()
):

    raise RuntimeError(
        "PART 8B is not ready."
    )


if not bool(
    part8c_verification[
        "PART8C_Ready"
    ].all()
):

    raise RuntimeError(
        "PART 8C is not ready."
    )


# ============================================================
# 5. VERIFY FINAL MODEL
# ============================================================

if (

    part5c_final_recommendation[
        "Primary_Recommended_Strategy"
    ]

    != FINAL_SCREENING_MODEL

):

    raise RuntimeError(
        "Final strategy mismatch."
    )


if (

    part5c_final_recommendation[
        "Primary_Recommended_Algorithm"
    ]

    != FINAL_ALGORITHM

):

    raise RuntimeError(
        "Final algorithm mismatch."
    )


# ============================================================
# 6. IDENTIFY FINAL MODEL IN PART 8A
# ============================================================

final_model_rows = (

    part8a_final_main_results[
        (
            part8a_final_main_results[
                "Screening_Model"
            ]
            ==
            FINAL_SCREENING_MODEL
        )
        &
        (
            part8a_final_main_results[
                "Algorithm"
            ]
            ==
            FINAL_ALGORITHM
        )
    ]
)


if len(
    final_model_rows
) != 1:

    raise RuntimeError(

        "Expected exactly one final model row "
        "in PART 8A."

    )


final_model = final_model_rows.iloc[
    0
]


# ============================================================
# 7. BASIC FINAL MODEL INFORMATION
# ============================================================

SELF_REPORT_ITEMS = int(
    final_model[
        "Self_Report_Items"
    ]
)


LOCKED_THRESHOLD = float(
    final_model[
        "Locked_Threshold"
    ]
)


HELDOUT_N = int(
    final_model[
        "Heldout_N"
    ]
)


HELDOUT_EVENTS = int(
    final_model[
        "Heldout_Events"
    ]
)


TEMPORAL_N = int(
    final_model[
        "Temporal_N"
    ]
)


TEMPORAL_EVENTS = int(
    final_model[
        "Temporal_Events"
    ]
)


# Identify the two strongest held-out permutation-importance
# contributors from the completed PART 6B results. This avoids
# hard-coding either their names or their ordering.
top_feature_rows = (
    part6b_feature_interpretation
    .sort_values(
        by=["Heldout_AUC_Rank", "Feature"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

if len(top_feature_rows) < 2:
    raise RuntimeError(
        "At least two feature-importance rows are required."
    )

TOP_FEATURE_1 = str(top_feature_rows.loc[0, "Feature"])
TOP_FEATURE_2 = str(top_feature_rows.loc[1, "Feature"])


# ============================================================
# 8. HELPER FUNCTIONS
# ============================================================

def fmt_number(
    value,
    digits=3
):

    if pd.isna(
        value
    ):

        return "not estimable"

    return f"{float(value):.{digits}f}"


def fmt_ci_from_row(
    row,
    prefix,
    metric,
    digits=3
):

    point = row[
        f"{prefix}_{metric}"
    ]

    lower = row[
        f"{prefix}_{metric}_CI_Lower"
    ]

    upper = row[
        f"{prefix}_{metric}_CI_Upper"
    ]


    if (
        pd.isna(point)
        or
        pd.isna(lower)
        or
        pd.isna(upper)
    ):

        return "not estimable"


    return (

        f"{float(point):.{digits}f} "

        f"(95% CI "

        f"{float(lower):.{digits}f}–"

        f"{float(upper):.{digits}f})"

    )


# ============================================================
# 9. EXTRACT FINAL PERFORMANCE
# ============================================================

heldout_auc = fmt_ci_from_row(
    final_model,
    "Heldout",
    "AUC"
)

heldout_sensitivity = fmt_ci_from_row(
    final_model,
    "Heldout",
    "Sensitivity"
)

heldout_specificity = fmt_ci_from_row(
    final_model,
    "Heldout",
    "Specificity"
)

heldout_precision = fmt_ci_from_row(
    final_model,
    "Heldout",
    "Precision"
)

heldout_f1 = fmt_ci_from_row(
    final_model,
    "Heldout",
    "F1"
)


temporal_auc = fmt_ci_from_row(
    final_model,
    "Temporal",
    "AUC"
)

temporal_sensitivity = fmt_ci_from_row(
    final_model,
    "Temporal",
    "Sensitivity"
)

temporal_specificity = fmt_ci_from_row(
    final_model,
    "Temporal",
    "Specificity"
)

temporal_precision = fmt_ci_from_row(
    final_model,
    "Temporal",
    "Precision"
)

temporal_f1 = fmt_ci_from_row(
    final_model,
    "Temporal",
    "F1"
)


# ============================================================
# 10. FINAL MODEL SUMMARY
# ============================================================

part8d_final_model_summary = pd.DataFrame(

    [
        {

            "Final_Screening_Strategy":
                FINAL_SCREENING_MODEL,

            "Final_Algorithm":
                FINAL_ALGORITHM,

            "Self_Report_Items":
                SELF_REPORT_ITEMS,

            "Locked_Threshold":
                LOCKED_THRESHOLD,

            "Heldout_N":
                HELDOUT_N,

            "Heldout_Events":
                HELDOUT_EVENTS,

            "Heldout_AUC_95CI":
                heldout_auc,

            "Heldout_Sensitivity_95CI":
                heldout_sensitivity,

            "Heldout_Specificity_95CI":
                heldout_specificity,

            "Heldout_Precision_95CI":
                heldout_precision,

            "Heldout_F1_95CI":
                heldout_f1,

            "Temporal_N":
                TEMPORAL_N,

            "Temporal_Events":
                TEMPORAL_EVENTS,

            "Temporal_AUC_95CI":
                temporal_auc,

            "Temporal_Sensitivity_95CI":
                temporal_sensitivity,

            "Temporal_Specificity_95CI":
                temporal_specificity,

            "Temporal_Precision_95CI":
                temporal_precision,

            "Temporal_F1_95CI":
                temporal_f1

        }

    ]

)


# ============================================================
# 11. METHODS TEXT
# ============================================================

methods_text = f"""
Seven prespecified screening strategies were evaluated for
prediction of major depressive disorder diagnosis. The
strategies differed in the amount of additional self-reported
information required, ranging from routinely available
administrative information to more comprehensive screening
information.

Candidate machine-learning algorithms were evaluated within
each screening strategy using genuine out-of-fold predictions
from the 2024 development partition. Representative algorithms
were selected within each strategy using development data only.
Classification thresholds were also determined exclusively
from the out-of-fold predictions and were subsequently locked.

GPAX was treated as continuous, whereas the remaining
administrative, behavioral, and questionnaire predictors were
treated as categorical or ordinal for resampling. Within every
development and out-of-fold training fold, class imbalance was
addressed using SMOTENC for mixed continuous-categorical
strategies and SMOTEN for the categorical-only conventional
strategy, followed by random undersampling. Validation-fold,
held-out 2024, and temporal 2025 observations were never
resampled.

The locked models and thresholds were evaluated without
modification in an independent held-out 2024 cohort and in a
temporally distinct 2025 cohort. Discrimination and
classification performance were summarized using the area
under the receiver operating characteristic curve (ROC AUC),
sensitivity, specificity, precision, and F1 score.

Uncertainty in validation performance was quantified using
2,000 stratified participant-level bootstrap replicates.
Percentile 95% confidence intervals were calculated from the
bootstrap distributions. Model calibration was evaluated
separately using calibration-in-the-large, calibration slope,
Brier score, expected calibration error, and integrated
calibration index.

Because the original model probabilities showed evidence of
miscalibration, probability recalibration was examined as a
secondary analysis. Logistic recalibration mappings were
estimated exclusively from genuine 2024 out-of-fold
predictions and transported without refitting to the
validation cohorts. Recalibrated probabilities were not used
to redefine the primary classification analysis.

Screening strategies were compared using a hierarchical
burden-performance framework rather than a weighted composite
score. Within-strategy algorithm selection was based on
development performance, whereas held-out and temporal
validation results were used to assess validation support and
the trade-off between predictive performance and respondent
burden.

The final model was interpreted using model-agnostic
permutation importance in the held-out cohort, with temporal
feature-importance analysis used as a transportability check.
Subgroup performance analyses were conducted descriptively.
Metrics requiring adequate positive events were suppressed in
subgroups with fewer than five positive cases.
""".strip()


# ============================================================
# 12. MODEL-SELECTION RATIONALE
# ============================================================

selection_rationale = f"""
The {FINAL_SCREENING_MODEL} strategy with
{FINAL_ALGORITHM} was designated as the primary recommended
low-burden screening strategy. This strategy requires only
{SELF_REPORT_ITEMS} additional self-report items beyond
routinely available administrative information.

The recommendation was based on the prespecified hierarchical
selection framework and reflects the balance between
respondent burden, out-of-fold model performance, independent
held-out validation, temporal validation, and Pareto
efficiency. The final recommendation was therefore not based
on the highest value of any single validation metric and did
not use a post hoc weighted composite performance score.
""".strip()


# ============================================================
# 13. RESULTS TEXT
# ============================================================

results_text = f"""
Among the seven prespecified screening strategies, the
{FINAL_SCREENING_MODEL} strategy using
{FINAL_ALGORITHM} was selected as the primary recommended
low-burden approach. The strategy required
{SELF_REPORT_ITEMS} additional self-report items.

In the independent held-out 2024 cohort
(n={HELDOUT_N:,}; {HELDOUT_EVENTS} positive cases), the final
model achieved an ROC AUC of {heldout_auc}. At the threshold
locked from the 2024 out-of-fold development predictions,
sensitivity was {heldout_sensitivity}, specificity was
{heldout_specificity}, precision was {heldout_precision}, and
the F1 score was {heldout_f1}.

Temporal validation was subsequently performed in the 2025
cohort (n={TEMPORAL_N:,}; {TEMPORAL_EVENTS} positive cases)
without model refitting or threshold re-optimization. The ROC
AUC was {temporal_auc}, with sensitivity of
{temporal_sensitivity}, specificity of
{temporal_specificity}, precision of {temporal_precision}, and
an F1 score of {temporal_f1}.

The burden-performance comparison supported the final
strategy as a low-burden option rather than simply the
strategy with the highest individual performance estimate.
In the primary held-out permutation-importance analysis, the
two highest-ranked predictive contributors were
{TOP_FEATURE_1} and {TOP_FEATURE_2}. Feature importance was
interpreted as predictive rather than causal, and the temporal
analysis was used only as an exploratory consistency check.

Subgroup analyses were exploratory because positive outcomes
were sparse within many strata. Accordingly, subgroup
fairness results were interpreted descriptively and were not
used as evidence of causal discrimination or proof of the
absence of algorithmic disparities.
""".strip()


# ============================================================
# 14. INTERPRETATION / REPORTING STATEMENTS
# ============================================================

interpretation_text = """
The final model should be interpreted as a low-burden
screening and risk-stratification approach rather than a
stand-alone diagnostic instrument. Its intended role is to
identify individuals who may warrant further clinical
assessment while minimizing additional respondent burden.

The observed validation results support temporal
transportability of discrimination and threshold-based
classification performance; however, probability calibration
should be considered separately from discrimination.
OOF-derived recalibration analyses were therefore treated as
secondary analyses and did not alter the primary locked
classification model.

Feature importance reflects predictive contribution within
the fitted model and does not establish causal or etiologic
effects. Similarly, subgroup analyses remain exploratory
because sparse outcome counts substantially limited stable
estimation in many protected-group strata.
""".strip()


# ============================================================
# 15. TABLE FOOTNOTES
# ============================================================

table_footnotes = [

    {
        "Table":
            "Main Table",

        "Footnote":
            (
                "Representative algorithms were selected "
                "within each screening strategy using genuine "
                "2024 out-of-fold predictions only."
            )
    },

    {
        "Table":
            "Main Table",

        "Footnote":
            (
                "Classification thresholds were determined "
                "from 2024 out-of-fold predictions and were "
                "locked before held-out 2024 and temporal "
                "2025 validation."
            )
    },

    {
        "Table":
            "Main Table",

        "Footnote":
            (
                "Values for held-out and temporal validation "
                "are point estimates with stratified "
                "participant-level bootstrap 95% confidence "
                "intervals based on 2,000 replicates."
            )
    },

    {
        "Table":
            "Main Table",

        "Footnote":
            (
                "The primary recommended strategy was selected "
                "using a prespecified hierarchical "
                "burden-performance framework rather than a "
                "weighted composite score."
            )
    },

    {
        "Table":
            "Supplementary calibration tables",

        "Footnote":
            (
                "Calibration regression and LOWESS smoothing "
                "were used for diagnostic assessment only and "
                "did not modify the raw validation "
                "probabilities."
            )
    },

    {
        "Table":
            "Supplementary recalibration table",

        "Footnote":
            (
                "Logistic recalibration mappings were fitted "
                "exclusively using genuine 2024 out-of-fold "
                "predictions and were transported without "
                "refitting to the validation cohorts."
            )
    },

    {
        "Table":
            "Supplementary feature-importance table",

        "Footnote":
            (
                "Permutation importance quantifies predictive "
                "contribution within the fitted model and "
                "should not be interpreted as a causal effect."
            )
    },

    {
        "Table":
            "Supplementary fairness table",

        "Footnote":
            (
                "Subgroup analyses were exploratory. "
                "Sensitivity, precision, F1 score, ROC AUC, "
                "and average precision were suppressed for "
                "groups with fewer than five positive cases "
                "because of unstable estimation."
            )
    }

]


part8d_table_footnotes = pd.DataFrame(
    table_footnotes
)


# ============================================================
# 16. FIGURE LEGENDS
# ============================================================

part8d_figure_legends = (
    part8c_figure_legends
    .copy()
)


# ============================================================
# 17. KEY REPORTING PRINCIPLES
# ============================================================

part8d_reporting_principles = pd.DataFrame(

    [

        {
            "Domain":
                "Final model",

            "Reporting_Principle":
                (
                    "M2_Admin_2Q + AdaBoostClassifier is the "
                    "primary recommended low-burden strategy."
                )
        },

        {
            "Domain":
                "Model selection",

            "Reporting_Principle":
                (
                    "Do not describe the final model as the "
                    "best model solely on the basis of one "
                    "performance metric."
                )
        },

        {
            "Domain":
                "Threshold",

            "Reporting_Principle":
                (
                    "State explicitly that the classification "
                    "threshold was locked using genuine 2024 "
                    "OOF predictions."
                )
        },

        {
            "Domain":
                "Temporal validation",

            "Reporting_Principle":
                (
                    "State explicitly that the 2025 cohort "
                    "was evaluated without model refitting or "
                    "threshold re-optimization."
                )
        },

        {
            "Domain":
                "Calibration",

            "Reporting_Principle":
                (
                    "Distinguish discrimination and "
                    "classification performance from "
                    "probability calibration."
                )
        },

        {
            "Domain":
                "Recalibration",

            "Reporting_Principle":
                (
                    "Describe OOF-derived recalibration as a "
                    "secondary probability-calibration "
                    "analysis."
                )
        },

        {
            "Domain":
                "Feature importance",

            "Reporting_Principle":
                (
                    "Do not interpret feature importance as "
                    "causal or etiologic evidence."
                )
        },

        {
            "Domain":
                "Fairness",

            "Reporting_Principle":
                (
                    "Describe subgroup fairness analyses as "
                    "exploratory and descriptive because of "
                    "sparse positive outcomes."
                )
        },

        {
            "Domain":
                "Clinical role",

            "Reporting_Principle":
                (
                    "Describe the model as a screening or "
                    "risk-stratification aid rather than a "
                    "replacement for clinical diagnosis."
                )
        }

    ]

)


# ============================================================
# 18. MANUSCRIPT TEXT TABLE
# ============================================================

part8d_manuscript_text = pd.DataFrame(

    [

        {
            "Section":
                "Methods",

            "Text":
                methods_text
        },

        {
            "Section":
                "Results",

            "Text":
                results_text
        },

        {
            "Section":
                "Model selection rationale",

            "Text":
                selection_rationale
        },

        {
            "Section":
                "Interpretation",

            "Text":
                interpretation_text
        }

    ]

)


# ============================================================
# 19. SOURCE MANIFEST
# ============================================================

part8d_source_manifest = pd.DataFrame(

    [

        {
            "Reporting_Component":
                "Final model identity",

            "Source":
                "part5c_final_recommendation"
        },

        {
            "Reporting_Component":
                "Final model performance",

            "Source":
                "part8a_final_main_results"
        },

        {
            "Reporting_Component":
                "Feature interpretation",

            "Source":
                "PART 6A / PART 6B artifacts"
        },

        {
            "Reporting_Component":
                "Fairness interpretation",

            "Source":
                "PART 7A / PART 7B artifacts"
        },

        {
            "Reporting_Component":
                "Supplementary tables",

            "Source":
                "PART 8B"
        },

        {
            "Reporting_Component":
                "Figure legends",

            "Source":
                "part8c_figure_legends"
        }

    ]

)


# ============================================================
# 20. VERIFICATION
# ============================================================

final_model_correct = bool(

    final_model[
        "Screening_Model"
    ]
    ==
    FINAL_SCREENING_MODEL

    and

    final_model[
        "Algorithm"
    ]
    ==
    FINAL_ALGORITHM

)


self_report_items_correct = bool(
    SELF_REPORT_ITEMS
    ==
    2
)


threshold_consistency = bool(

    final_model[
        "Threshold_Matches_Heldout"
    ]

    and

    final_model[
        "Threshold_Matches_Temporal"
    ]

)


performance_complete = bool(

    all(

        pd.notna(
            final_model[
                column
            ]
        )

        for column in [

            "Heldout_AUC",
            "Heldout_Sensitivity",
            "Heldout_Specificity",
            "Heldout_Precision",
            "Heldout_F1",

            "Temporal_AUC",
            "Temporal_Sensitivity",
            "Temporal_Specificity",
            "Temporal_Precision",
            "Temporal_F1"

        ]

    )

)


reporting_sections_complete = bool(

    set(
        part8d_manuscript_text[
            "Section"
        ]
    )

    ==

    {
        "Methods",
        "Results",
        "Model selection rationale",
        "Interpretation"
    }

)


figure_legends_available = bool(
    len(
        part8d_figure_legends
    )
    >=
    4
)


footnotes_available = bool(
    len(
        part8d_table_footnotes
    )
    >=
    1
)


part8d_verification = pd.DataFrame(

    [

        {

            "PART8A_Ready":
                bool(
                    part8a_global_verification[
                        "PART8A_Ready"
                    ].all()
                ),

            "PART8B_Ready":
                bool(
                    part8b_global_verification[
                        "PART8B_Ready"
                    ].all()
                ),

            "PART8C_Ready":
                bool(
                    part8c_verification[
                        "PART8C_Ready"
                    ].all()
                ),

            "Final_Model_Correct":
                final_model_correct,

            "Self_Report_Items_Equals_2":
                self_report_items_correct,

            "Locked_Threshold_Consistent":
                threshold_consistency,

            "Final_Performance_Complete":
                performance_complete,

            "Reporting_Sections_Complete":
                reporting_sections_complete,

            "Figure_Legends_Available":
                figure_legends_available,

            "Table_Footnotes_Available":
                footnotes_available,

            "No_Model_Refit":
                True,

            "No_New_Predictions":
                True,

            "No_Threshold_Optimization":
                True,

            "No_Recalibration":
                True,

            "No_Bootstrap_Rerun":
                True,

            "No_Feature_Selection":
                True,

            "No_Algorithm_Reselection":
                True,

            "No_New_Statistical_Test":
                True

        }

    ]

)


# ============================================================
# 21. FINAL READY FLAG
# ============================================================

part8d_ready = bool(

    part8d_verification[
        [
            "PART8A_Ready",
            "PART8B_Ready",
            "PART8C_Ready",
            "Final_Model_Correct",
            "Self_Report_Items_Equals_2",
            "Locked_Threshold_Consistent",
            "Final_Performance_Complete",
            "Reporting_Sections_Complete",
            "Figure_Legends_Available",
            "Table_Footnotes_Available"
        ]
    ]
    .all(
        axis=None
    )

)


part8d_verification[
    "PART8D_Ready"
] = part8d_ready


# ============================================================
# 22. EXPORT CSV FILES
# ============================================================

part8d_final_model_summary.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "part8d_final_model_summary.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


part8d_manuscript_text.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "part8d_manuscript_text.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


part8d_table_footnotes.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "part8d_table_footnotes.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


part8d_figure_legends.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "part8d_figure_legends.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


part8d_reporting_principles.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "part8d_reporting_principles.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


part8d_source_manifest.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "part8d_source_manifest.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


part8d_verification.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "part8d_verification.csv"
    ),

    index=False,

    encoding="utf-8-sig"

)


# ============================================================
# 23. EXPORT PLAIN-TEXT MANUSCRIPT DRAFT
# ============================================================

manuscript_text_file = os.path.join(

    OUTPUT_DIR,

    "part8d_manuscript_ready_text.txt"

)


with open(

    manuscript_text_file,

    "w",

    encoding="utf-8"

) as file:

    file.write(
        "METHODS\n"
    )

    file.write(
        "=" * 80
        + "\n\n"
    )

    file.write(
        methods_text
        + "\n\n\n"
    )


    file.write(
        "RESULTS\n"
    )

    file.write(
        "=" * 80
        + "\n\n"
    )

    file.write(
        results_text
        + "\n\n\n"
    )


    file.write(
        "MODEL SELECTION RATIONALE\n"
    )

    file.write(
        "=" * 80
        + "\n\n"
    )

    file.write(
        selection_rationale
        + "\n\n\n"
    )


    file.write(
        "INTERPRETATION\n"
    )

    file.write(
        "=" * 80
        + "\n\n"
    )

    file.write(
        interpretation_text
        + "\n"
    )


# ============================================================
# 24. DISPLAY
# ============================================================

print(
    "=" * 120
)

print(
    "PART 8D VERSION 3.0: "
    "MANUSCRIPT-READY REPORTING PACKAGE"
)

print(
    "=" * 120
)


print(
    "\nFINAL MODEL SUMMARY"
)

display(
    part8d_final_model_summary
)


print(
    "\n" + "=" * 120
)

print(
    "MANUSCRIPT TEXT"
)

print(
    "=" * 120
)

display(
    part8d_manuscript_text
)


print(
    "\n" + "=" * 120
)

print(
    "TABLE FOOTNOTES"
)

print(
    "=" * 120
)

display(
    part8d_table_footnotes
)


print(
    "\n" + "=" * 120
)

print(
    "REPORTING PRINCIPLES"
)

print(
    "=" * 120
)

display(
    part8d_reporting_principles
)


print(
    "\n" + "=" * 120
)

print(
    "PART 8D VERIFICATION"
)

print(
    "=" * 120
)

display(
    part8d_verification
)


# ============================================================
# 25. HARD SAFETY CHECK
# ============================================================

if not part8d_ready:

    raise RuntimeError(

        "PART 8D verification failed. "
        "Do not freeze manuscript reporting."

    )


# ============================================================
# 26. FINAL REPORT
# ============================================================

print(
    "\n" + "=" * 120
)

print(
    "PART 8D VERSION 3.0 COMPLETED"
)

print(
    "=" * 120
)


print(
    f"\nFinal model                  : "
    f"{FINAL_SCREENING_MODEL} + "
    f"{FINAL_ALGORITHM}"
)


print(
    f"Self-report items            : "
    f"{SELF_REPORT_ITEMS}"
)


print(
    f"Held-out cohort              : "
    f"N={HELDOUT_N:,}, "
    f"events={HELDOUT_EVENTS}"
)


print(
    f"Temporal cohort              : "
    f"N={TEMPORAL_N:,}, "
    f"events={TEMPORAL_EVENTS}"
)


print(
    "\nReporting components created:"
)


print(
    "- Final model summary"
)

print(
    "- Methods draft"
)

print(
    "- Results draft"
)

print(
    "- Model-selection rationale"
)

print(
    "- Interpretation statements"
)

print(
    "- Table footnotes"
)

print(
    "- Figure legends"
)

print(
    "- Reporting principles"
)

print(
    "- Source manifest"
)

print(
    "- Verification"
)


print(
    f"\nOutput directory             : "
    f"{OUTPUT_DIR}"
)


print(
    "\nNo model was refitted, no new prediction was "
    "generated, no threshold was modified, no recalibration "
    "was performed, no bootstrap analysis was rerun, no "
    "feature was selected, no algorithm was re-selected, "
    "and no new statistical test was performed."
)


print(
    "\nPART 8D is the final manuscript-reporting layer."
)


print(
    "\nAnalytical pipeline may be frozen only after "
    "PART8D_Ready=True."
)

### PART 9A: Reproducibility and Feature-Definition Audit

In [ ]:
# ============================================================
# PART 9A: REPRODUCIBILITY AND FEATURE-DEFINITION AUDIT
# VERIFICATION OF THE LOW-BURDEN M2 SCREENING STRATEGY
# ============================================================
#
# Purpose:
# - Verify the exact predictors used by M2_Admin_2Q.
# - Confirm agreement with the manuscript and Appendix.
# - Confirm respondent burden and categorical-aware sampler.
# - Export a traceable feature manifest.
#
# This is a quality-control audit only.
# No model fitting, resampling, prediction, threshold selection,
# recalibration, or performance calculation is performed.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================

TARGET_STRATEGY = "M2_Admin_2Q"
EXPECTED_FEATURE_COUNT = 10
EXPECTED_SELF_REPORT_ITEMS = 2
EXPECTED_SAMPLER_TYPE = "SMOTENC"

EXPECTED_ADMINISTRATIVE_FEATURES = [
    "institute",
    "GPAX",
    "dadstatus",
    "momstatus",
    "dadeducation",
    "momeducation",
    "parent",
    "gender",
]

EXPECTED_THAI_2Q_FEATURES = [
    "2Q1",
    "2Q2",
]

EXPECTED_M2_FEATURES = (
    EXPECTED_ADMINISTRATIVE_FEATURES
    + EXPECTED_THAI_2Q_FEATURES
)


# ============================================================
# 2. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "screening_strategies",
    "OUTCOME",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )

if not isinstance(screening_strategies, dict):
    raise TypeError(
        "screening_strategies must be a dictionary."
    )

if TARGET_STRATEGY not in screening_strategies:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from screening_strategies."
    )


# ============================================================
# 3. EXTRACT CANONICAL M2 DEFINITION
# ============================================================

m2_definition = screening_strategies[TARGET_STRATEGY]

if "features" not in m2_definition:
    raise KeyError(
        f"{TARGET_STRATEGY} does not contain a features field."
    )

observed_features = [
    str(feature)
    for feature in m2_definition["features"]
]

observed_self_report_items = int(
    m2_definition.get("additional_questions", -1)
)

observed_sampler_type = str(
    m2_definition.get("sampler_type", "")
)


# ============================================================
# 4. VALIDATION CHECKS
# ============================================================

feature_count_correct = (
    len(observed_features) == EXPECTED_FEATURE_COUNT
)

feature_order_correct = (
    observed_features == EXPECTED_M2_FEATURES
)

feature_set_correct = (
    set(observed_features) == set(EXPECTED_M2_FEATURES)
)

no_duplicate_features = (
    len(observed_features) == len(set(observed_features))
)

administrative_features_correct = (
    observed_features[:8]
    == EXPECTED_ADMINISTRATIVE_FEATURES
)

thai_2q_features_correct = (
    observed_features[-2:]
    == EXPECTED_THAI_2Q_FEATURES
)

self_report_burden_correct = (
    observed_self_report_items
    == EXPECTED_SELF_REPORT_ITEMS
)

sampler_type_correct = (
    observed_sampler_type == EXPECTED_SAMPLER_TYPE
)

outcome_excluded = (
    str(OUTCOME) not in observed_features
)

gpax_present_once = (
    observed_features.count("GPAX") == 1
)

all_checks_passed = all([
    feature_count_correct,
    feature_order_correct,
    feature_set_correct,
    no_duplicate_features,
    administrative_features_correct,
    thai_2q_features_correct,
    self_report_burden_correct,
    sampler_type_correct,
    outcome_excluded,
    gpax_present_once,
])


# ============================================================
# 5. FEATURE MANIFEST
# ============================================================

feature_rows = []

for position, feature in enumerate(observed_features, start=1):
    feature_rows.append({
        "Screening_Strategy": TARGET_STRATEGY,
        "Feature_Position": position,
        "Feature": feature,
        "Feature_Group": (
            "Thai 2Q"
            if feature in EXPECTED_THAI_2Q_FEATURES
            else "Administrative"
        ),
        "Data_Type_For_Resampling": (
            "Continuous" if feature == "GPAX" else "Categorical_or_Ordinal"
        ),
        "Additional_Self_Report_Item": (
            feature in EXPECTED_THAI_2Q_FEATURES
        ),
        "Expected_Feature": feature in EXPECTED_M2_FEATURES,
    })

part9a_m2_feature_manifest = pd.DataFrame(feature_rows)


# ============================================================
# 6. AUDIT SUMMARY
# ============================================================

part9a_m2_feature_audit = pd.DataFrame([
    {
        "Screening_Strategy": TARGET_STRATEGY,
        "Observed_Feature_Count": len(observed_features),
        "Expected_Feature_Count": EXPECTED_FEATURE_COUNT,
        "Observed_Self_Report_Items": observed_self_report_items,
        "Expected_Self_Report_Items": EXPECTED_SELF_REPORT_ITEMS,
        "Observed_Sampler_Type": observed_sampler_type,
        "Expected_Sampler_Type": EXPECTED_SAMPLER_TYPE,
        "Feature_Count_Correct": feature_count_correct,
        "Feature_Order_Correct": feature_order_correct,
        "Feature_Set_Correct": feature_set_correct,
        "No_Duplicate_Features": no_duplicate_features,
        "Administrative_Features_Correct": administrative_features_correct,
        "Thai_2Q_Features_Correct": thai_2q_features_correct,
        "Self_Report_Burden_Correct": self_report_burden_correct,
        "Sampler_Type_Correct": sampler_type_correct,
        "Outcome_Excluded": outcome_excluded,
        "GPAX_Present_Once": gpax_present_once,
        "M2_Feature_Definition_Ready": all_checks_passed,
    }
])


# ============================================================
# 7. HARD SAFETY CHECK
# ============================================================

if not all_checks_passed:
    print("Observed M2 features:")
    print(observed_features)
    print("\nExpected M2 features:")
    print(EXPECTED_M2_FEATURES)
    display(part9a_m2_feature_audit)
    raise RuntimeError(
        "M2 feature-definition audit failed. Do not use the "
        "M2 manuscript or Appendix tables until resolved."
    )


# ============================================================
# 8. EXPORT AND REPORT
# ============================================================

part9a_m2_feature_manifest.to_csv(
    "part9a_m2_feature_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

part9a_m2_feature_audit.to_csv(
    "part9a_m2_feature_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("=" * 90)
print("PART 9A COMPLETED: M2 FEATURE-DEFINITION AUDIT")
print("=" * 90)
print(f"Strategy                 : {TARGET_STRATEGY}")
print(f"Feature count            : {len(observed_features)}")
print(f"Administrative predictors: {len(EXPECTED_ADMINISTRATIVE_FEATURES)}")
print(f"Thai 2Q items            : {len(EXPECTED_THAI_2Q_FEATURES)}")
print(f"Self-report burden       : {observed_self_report_items}")
print(f"Sampler type             : {observed_sampler_type}")
print(f"Outcome excluded         : {outcome_excluded}")
print(f"All audit checks passed  : {all_checks_passed}")
print("\nExact M2 feature order:")
for position, feature in enumerate(observed_features, start=1):
    print(f"{position:>2}. {feature}")
print("\nNo model was fitted, no data were resampled, and no prediction")
print("or threshold was generated during this quality-control audit.")

### PART 9B: Reproducibility and Cohort-Integrity Audit

In [ ]:
# ============================================================
# PART 9B: REPRODUCIBILITY AND COHORT-INTEGRITY AUDIT
# FINAL LOW-BURDEN M2 ANALYSIS COHORTS
# ============================================================
#
# Purpose:
# - Verify the frozen M2 feature set and cohort composition.
# - Confirm development, held-out, and temporal sample sizes.
# - Confirm event counts, outcome validity, and missingness.
# - Export aggregate verification metadata only.
#
# Privacy safeguard:
# - Participant-level data are NOT exported.
# - Participant identifiers or row indices are NOT exported.
#
# No model fitting, resampling, prediction, threshold selection,
# recalibration, or performance calculation is performed.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================

TARGET_STRATEGY = "M2_Admin_2Q"

EXPECTED_COHORTS = {
    "Development 2024": {"N": 2863, "Events": 32},
    "Held-out 2024": {"N": 1227, "Events": 14},
    "Temporal 2025": {"N": 815, "Events": 16},
}

EXPECTED_FEATURES = [
    "institute",
    "GPAX",
    "dadstatus",
    "momstatus",
    "dadeducation",
    "momeducation",
    "parent",
    "gender",
    "2Q1",
    "2Q2",
]


# ============================================================
# 2. REQUIRED OBJECTS
# ============================================================

required_objects = [
    "screening_strategies",
    "prepared_data",
    "temporal_datasets",
    "OUTCOME",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )

if TARGET_STRATEGY not in screening_strategies:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from screening_strategies."
    )

if TARGET_STRATEGY not in prepared_data:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from prepared_data."
    )

if TARGET_STRATEGY not in temporal_datasets:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from temporal_datasets."
    )


# ============================================================
# 3. VERIFY CANONICAL FEATURE DEFINITION
# ============================================================

observed_features = list(
    screening_strategies[TARGET_STRATEGY]["features"]
)

if observed_features != EXPECTED_FEATURES:
    raise RuntimeError(
        "The canonical M2 feature definition does not match "
        "the manuscript-specified 10-feature order."
    )


# ============================================================
# 4. EXTRACT CANONICAL COHORT OBJECTS IN MEMORY
# ============================================================

m2_prepared = prepared_data[TARGET_STRATEGY]

required_prepared_keys = [
    "X_train",
    "y_train",
    "X_test",
    "y_test",
]

missing_prepared_keys = [
    key for key in required_prepared_keys
    if key not in m2_prepared
]

if missing_prepared_keys:
    raise KeyError(
        "prepared_data is missing keys: "
        + ", ".join(missing_prepared_keys)
    )

X_development = (
    m2_prepared["X_train"][EXPECTED_FEATURES]
    .copy()
    .reset_index(drop=True)
)
y_development = (
    m2_prepared["y_train"]
    .astype(int)
    .copy()
    .reset_index(drop=True)
)

X_heldout = (
    m2_prepared["X_test"][EXPECTED_FEATURES]
    .copy()
    .reset_index(drop=True)
)
y_heldout = (
    m2_prepared["y_test"]
    .astype(int)
    .copy()
    .reset_index(drop=True)
)

temporal_data = temporal_datasets[TARGET_STRATEGY].copy()

if OUTCOME not in temporal_data.columns:
    raise KeyError(
        f"Temporal data are missing outcome column {OUTCOME}."
    )

X_temporal = (
    temporal_data[EXPECTED_FEATURES]
    .copy()
    .reset_index(drop=True)
)
y_temporal = (
    temporal_data[OUTCOME]
    .astype(int)
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 5. COHORT AUDIT
# ============================================================

cohort_objects = {
    "Development 2024": (X_development, y_development),
    "Held-out 2024": (X_heldout, y_heldout),
    "Temporal 2025": (X_temporal, y_temporal),
}

audit_rows = []

for cohort_name, (X_cohort, y_cohort) in cohort_objects.items():
    expected = EXPECTED_COHORTS[cohort_name]

    observed_n = int(len(y_cohort))
    observed_events = int(y_cohort.sum())
    observed_nonevents = int((y_cohort == 0).sum())

    row_alignment_correct = (
        len(X_cohort) == len(y_cohort)
    )
    feature_order_correct = (
        list(X_cohort.columns) == EXPECTED_FEATURES
    )
    binary_outcome = (
        set(y_cohort.unique()).issubset({0, 1})
    )
    no_predictor_missingness = (
        int(X_cohort.isna().sum().sum()) == 0
    )
    no_outcome_missingness = (
        int(y_cohort.isna().sum()) == 0
    )
    n_correct = observed_n == expected["N"]
    events_correct = observed_events == expected["Events"]

    cohort_ready = all([
        row_alignment_correct,
        feature_order_correct,
        binary_outcome,
        no_predictor_missingness,
        no_outcome_missingness,
        n_correct,
        events_correct,
    ])

    audit_rows.append({
        "Cohort": cohort_name,
        "Observed_N": observed_n,
        "Expected_N": expected["N"],
        "N_Correct": n_correct,
        "Observed_Events": observed_events,
        "Expected_Events": expected["Events"],
        "Events_Correct": events_correct,
        "Observed_Nonevents": observed_nonevents,
        "Feature_Count": len(X_cohort.columns),
        "Feature_Order_Correct": feature_order_correct,
        "Rows_Aligned": row_alignment_correct,
        "Binary_Outcome": binary_outcome,
        "Predictor_Missing_Cells": int(X_cohort.isna().sum().sum()),
        "Outcome_Missing_Cells": int(y_cohort.isna().sum()),
        "No_Missing_Values": (
            no_predictor_missingness and no_outcome_missingness
        ),
        "Cohort_Ready": cohort_ready,
    })

part9b_m2_cohort_integrity = pd.DataFrame(audit_rows)


# ============================================================
# 6. FEATURE-DEFINITION TABLE
# ============================================================

part9b_m2_feature_definition = pd.DataFrame({
    "Feature_Order": np.arange(1, len(EXPECTED_FEATURES) + 1),
    "Feature": EXPECTED_FEATURES,
    "Feature_Group": [
        "Thai 2Q" if feature in ["2Q1", "2Q2"]
        else "Administrative"
        for feature in EXPECTED_FEATURES
    ],
    "Data_Type_For_Resampling": [
        "Continuous" if feature == "GPAX"
        else "Categorical_or_Ordinal"
        for feature in EXPECTED_FEATURES
    ],
})


# ============================================================
# 7. GLOBAL VERIFICATION
# ============================================================

development_plus_heldout_n_correct = (
    len(y_development) + len(y_heldout) == 4090
)

development_plus_heldout_events_correct = (
    int(y_development.sum()) + int(y_heldout.sum()) == 46
)

PART9B_READY = bool(
    part9b_m2_cohort_integrity["Cohort_Ready"].all()
    and development_plus_heldout_n_correct
    and development_plus_heldout_events_correct
)

part9b_global_verification = pd.DataFrame([
    {
        "M2_Feature_Count": len(EXPECTED_FEATURES),
        "Development_Plus_Heldout_N": (
            len(y_development) + len(y_heldout)
        ),
        "Development_Plus_Heldout_Events": (
            int(y_development.sum()) + int(y_heldout.sum())
        ),
        "Development_Plus_Heldout_N_Correct": (
            development_plus_heldout_n_correct
        ),
        "Development_Plus_Heldout_Events_Correct": (
            development_plus_heldout_events_correct
        ),
        "Participant_Level_Data_Exported": False,
        "Model_Fitted": False,
        "Resampling_Performed": False,
        "Predictions_Generated": False,
        "Threshold_Selected": False,
        "PART9B_Ready": PART9B_READY,
    }
])


# ============================================================
# 8. HARD SAFETY CHECK
# ============================================================

if not PART9B_READY:
    display(part9b_m2_cohort_integrity)
    display(part9b_global_verification)
    raise RuntimeError(
        "M2 cohort-integrity audit failed. Participant-level "
        "data were not exported."
    )


# ============================================================
# 9. EXPORT AGGREGATE METADATA ONLY
# ============================================================

part9b_m2_cohort_integrity.to_csv(
    "part9b_m2_cohort_integrity.csv",
    index=False,
    encoding="utf-8-sig",
)

part9b_m2_feature_definition.to_csv(
    "part9b_m2_feature_definition.csv",
    index=False,
    encoding="utf-8-sig",
)

part9b_global_verification.to_csv(
    "part9b_global_verification.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 10. FINAL REPORT
# ============================================================

print("=" * 95)
print("PART 9B COMPLETED: M2 COHORT-INTEGRITY AUDIT")
print("=" * 95)
display(part9b_m2_cohort_integrity)
print(f"\nPART9B_READY: {PART9B_READY}")
print("\nExported aggregate metadata:")
print("- part9b_m2_cohort_integrity.csv")
print("- part9b_m2_feature_definition.csv")
print("- part9b_global_verification.csv")
print("\nParticipant-level data, identifiers, and row indices were NOT exported.")
print("No model was fitted, no resampling was performed, no prediction was")
print("generated, and no classification threshold was selected.")

### PART 9C: Temporal 2025 Source and Row-Alignment Integrity Audit

In [ ]:
# ============================================================
# PART 9C: TEMPORAL 2025 SOURCE AND ROW-ALIGNMENT
#          INTEGRITY AUDIT
# ============================================================
#
# Purpose:
# - Verify that the canonical temporal M2 dataset used in PART 4A
#   retains the prespecified feature definition and row order.
# - Verify alignment of the outcome, original index, and row position
#   across every stored M2 temporal prediction record.
# - Verify the primary M2 AdaBoost record selected in PART 5C.
# - Export aggregate verification metadata only.
#
# Privacy safeguard:
# - No participant-level predictors, outcomes, identifiers, row
#   indices, or predictions are exported.
#
# This audit performs no model fitting, resampling, prediction,
# recalibration, threshold selection, or performance calculation.
# ============================================================

import hashlib
import numpy as np
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================

TARGET_STRATEGY = "M2_Admin_2Q"
EXPECTED_OUTCOME = "mdddx"
EXPECTED_N = 815
EXPECTED_EVENTS = 16
THRESHOLD_ATOL = 1e-12

EXPECTED_FEATURES = [
    "institute",
    "GPAX",
    "dadstatus",
    "momstatus",
    "dadeducation",
    "momeducation",
    "parent",
    "gender",
    "2Q1",
    "2Q2",
]


# ============================================================
# 2. REQUIRED UPSTREAM OBJECTS
# ============================================================

required_objects = [
    "screening_strategies",
    "temporal_datasets",
    "temporal_predictions",
    "temporal_prediction_table",
    "locked_thresholds",
    "part5c_final_recommendation",
    "OUTCOME",
    "PART9B_READY",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required upstream objects: "
        + ", ".join(missing_objects)
    )

if not bool(PART9B_READY):
    raise RuntimeError(
        "PART 9B is not ready. Run and resolve PART 9B first."
    )

if str(OUTCOME) != EXPECTED_OUTCOME:
    raise RuntimeError(
        f"Expected OUTCOME='{EXPECTED_OUTCOME}', observed '{OUTCOME}'."
    )

if TARGET_STRATEGY not in screening_strategies:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from screening_strategies."
    )

if TARGET_STRATEGY not in temporal_datasets:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from temporal_datasets."
    )

if TARGET_STRATEGY not in temporal_predictions:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from temporal_predictions."
    )

if TARGET_STRATEGY not in locked_thresholds:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from locked_thresholds."
    )


# ============================================================
# 3. VERIFY FINAL STRATEGY AND ALGORITHM FROM PART 5C
# ============================================================

primary_strategy = str(
    part5c_final_recommendation[
        "Primary_Recommended_Strategy"
    ]
)

primary_algorithm = str(
    part5c_final_recommendation[
        "Primary_Recommended_Algorithm"
    ]
)

if primary_strategy != TARGET_STRATEGY:
    raise RuntimeError(
        "PART 5C primary strategy does not match M2_Admin_2Q."
    )

if primary_algorithm not in temporal_predictions[TARGET_STRATEGY]:
    raise KeyError(
        f"Primary algorithm '{primary_algorithm}' is missing from "
        f"temporal_predictions['{TARGET_STRATEGY}']."
    )

if primary_algorithm not in locked_thresholds[TARGET_STRATEGY]:
    raise KeyError(
        f"Primary algorithm '{primary_algorithm}' has no locked threshold."
    )


# ============================================================
# 4. VERIFY THE LOCKED M2 FEATURE DEFINITION
# ============================================================

observed_features = list(
    screening_strategies[TARGET_STRATEGY]["features"]
)

feature_definition_correct = (
    observed_features == EXPECTED_FEATURES
)

if not feature_definition_correct:
    raise RuntimeError(
        "The M2 feature names or order differ from the "
        "prespecified manuscript definition."
    )


# ============================================================
# 5. EXTRACT THE CANONICAL TEMPORAL SOURCE
# ============================================================

source_dataset = temporal_datasets[TARGET_STRATEGY]

if not isinstance(source_dataset, pd.DataFrame):
    raise TypeError(
        "The canonical M2 temporal source must be a pandas DataFrame."
    )

required_columns = EXPECTED_FEATURES + [EXPECTED_OUTCOME]
missing_columns = [
    column for column in required_columns
    if column not in source_dataset.columns
]

if missing_columns:
    raise KeyError(
        "Canonical temporal source is missing columns: "
        + ", ".join(missing_columns)
    )

X_source = source_dataset.loc[:, EXPECTED_FEATURES].copy()
y_source = source_dataset.loc[:, EXPECTED_OUTCOME].copy()
source_index = np.asarray(source_dataset.index).copy()
source_row_position = np.arange(len(source_dataset), dtype=int)

if X_source.columns.tolist() != EXPECTED_FEATURES:
    raise RuntimeError("Canonical M2 feature order is incorrect.")

if X_source.isna().any().any():
    raise ValueError("Canonical temporal M2 predictors contain missing values.")

if y_source.isna().any():
    raise ValueError("Canonical temporal outcome contains missing values.")

try:
    y_source = pd.to_numeric(y_source, errors="raise").astype(int)
except Exception as exc:
    raise TypeError("Temporal outcome cannot be converted to integer.") from exc

if set(y_source.unique()) != {0, 1}:
    raise ValueError("Temporal outcome must contain binary classes 0 and 1.")

try:
    gpax_numeric = pd.to_numeric(X_source["GPAX"], errors="raise")
except Exception as exc:
    raise TypeError("GPAX must be numeric in the temporal cohort.") from exc

if not np.isfinite(gpax_numeric.to_numpy(dtype=float)).all():
    raise ValueError("GPAX contains nonfinite values.")

observed_n = int(len(y_source))
observed_events = int(y_source.sum())

if observed_n != EXPECTED_N:
    raise ValueError(
        f"Expected temporal N={EXPECTED_N}, observed {observed_n}."
    )

if observed_events != EXPECTED_EVENTS:
    raise ValueError(
        f"Expected temporal events={EXPECTED_EVENTS}, "
        f"observed {observed_events}."
    )


# ============================================================
# 6. NONREVERSIBLE IN-MEMORY FINGERPRINTS
# ============================================================
# These fingerprints are used only to confirm that the audited
# in-memory objects remain unchanged. They do not contain rows.
# ============================================================

def dataframe_fingerprint(frame):
    hashed = pd.util.hash_pandas_object(
        frame,
        index=True,
        categorize=True,
    ).to_numpy(dtype=np.uint64)
    return hashlib.sha256(hashed.tobytes()).hexdigest()


def series_fingerprint(series):
    hashed = pd.util.hash_pandas_object(
        series,
        index=True,
        categorize=True,
    ).to_numpy(dtype=np.uint64)
    return hashlib.sha256(hashed.tobytes()).hexdigest()


source_feature_fingerprint = dataframe_fingerprint(X_source)
source_outcome_fingerprint = series_fingerprint(y_source)


# ============================================================
# 7. VERIFY EVERY STORED M2 TEMPORAL PREDICTION RECORD
# ============================================================

prediction_audit_rows = []

for algorithm_name, record in temporal_predictions[
    TARGET_STRATEGY
].items():
    required_record_keys = [
        "features",
        "y_true",
        "y_prob",
        "y_pred",
        "threshold",
        "locked_threshold",
        "row_position",
        "original_index",
        "n",
        "events",
        "model_refitted",
        "threshold_modified",
    ]

    missing_record_keys = [
        key for key in required_record_keys
        if key not in record
    ]

    if missing_record_keys:
        raise KeyError(
            f"{algorithm_name}: temporal prediction record is missing: "
            + ", ".join(missing_record_keys)
        )

    record_y = np.asarray(record["y_true"], dtype=int)
    record_prob = np.asarray(record["y_prob"], dtype=float)
    record_pred = np.asarray(record["y_pred"], dtype=int)
    record_position = np.asarray(record["row_position"], dtype=int)
    record_index = np.asarray(record["original_index"])

    expected_threshold = float(
        locked_thresholds[TARGET_STRATEGY][algorithm_name]
    )
    record_threshold = float(record["threshold"])
    record_locked_threshold = float(record["locked_threshold"])

    checks = {
        "Feature_Order_Correct": (
            list(record["features"]) == EXPECTED_FEATURES
        ),
        "N_Correct": (
            int(record["n"]) == EXPECTED_N
            and len(record_y) == EXPECTED_N
            and len(record_prob) == EXPECTED_N
            and len(record_pred) == EXPECTED_N
        ),
        "Events_Correct": (
            int(record["events"]) == EXPECTED_EVENTS
            and int(record_y.sum()) == EXPECTED_EVENTS
        ),
        "Outcome_Order_Correct": np.array_equal(
            record_y,
            y_source.to_numpy(dtype=int),
        ),
        "Row_Position_Correct": np.array_equal(
            record_position,
            source_row_position,
        ),
        "Original_Index_Order_Correct": np.array_equal(
            record_index,
            source_index,
        ),
        "Probability_Length_Correct": (
            len(record_prob) == EXPECTED_N
        ),
        "Probability_Valid": bool(
            np.isfinite(record_prob).all()
            and (record_prob >= 0).all()
            and (record_prob <= 1).all()
        ),
        "Prediction_Binary": set(
            np.unique(record_pred)
        ).issubset({0, 1}),
        "Threshold_Matches_Locked": bool(
            np.isclose(
                record_threshold,
                expected_threshold,
                rtol=0,
                atol=THRESHOLD_ATOL,
            )
            and np.isclose(
                record_locked_threshold,
                expected_threshold,
                rtol=0,
                atol=THRESHOLD_ATOL,
            )
        ),
        "Predictions_Match_Locked_Threshold": np.array_equal(
            record_pred,
            (record_prob >= expected_threshold).astype(int),
        ),
        "Model_Not_Refitted": not bool(record["model_refitted"]),
        "Threshold_Not_Modified": not bool(
            record["threshold_modified"]
        ),
    }

    prediction_audit_rows.append({
        "Screening_Model": TARGET_STRATEGY,
        "Algorithm": algorithm_name,
        **checks,
        "Record_Ready": bool(all(checks.values())),
    })

part9c_prediction_record_audit = pd.DataFrame(
    prediction_audit_rows
)

if part9c_prediction_record_audit.empty:
    raise RuntimeError("No M2 temporal prediction records were found.")


# ============================================================
# 8. VERIFY THE LONG-FORM PART 4A PREDICTION TABLE
# ============================================================

required_table_columns = [
    "Validation_Set",
    "Screening_Model",
    "Algorithm",
    "Row_Position",
    "Original_Index",
    "y_true",
    "y_prob",
    "Locked_Threshold",
    "y_pred",
]

missing_table_columns = [
    column for column in required_table_columns
    if column not in temporal_prediction_table.columns
]

if missing_table_columns:
    raise KeyError(
        "temporal_prediction_table is missing columns: "
        + ", ".join(missing_table_columns)
    )

m2_prediction_table = temporal_prediction_table.loc[
    temporal_prediction_table["Screening_Model"] == TARGET_STRATEGY,
    required_table_columns,
].copy()

table_audit_rows = []

for algorithm_name in temporal_predictions[TARGET_STRATEGY]:
    group = (
        m2_prediction_table.loc[
            m2_prediction_table["Algorithm"] == algorithm_name
        ]
        .sort_values("Row_Position", kind="stable")
        .reset_index(drop=True)
    )

    expected_threshold = float(
        locked_thresholds[TARGET_STRATEGY][algorithm_name]
    )

    table_checks = {
        "Exactly_815_Rows": len(group) == EXPECTED_N,
        "Row_Position_Correct": np.array_equal(
            group["Row_Position"].to_numpy(dtype=int),
            source_row_position,
        ) if len(group) == EXPECTED_N else False,
        "Original_Index_Order_Correct": np.array_equal(
            group["Original_Index"].to_numpy(),
            source_index,
        ) if len(group) == EXPECTED_N else False,
        "Outcome_Order_Correct": np.array_equal(
            group["y_true"].to_numpy(dtype=int),
            y_source.to_numpy(dtype=int),
        ) if len(group) == EXPECTED_N else False,
        "Threshold_Matches_Locked": bool(
            len(group) == EXPECTED_N
            and np.isclose(
                group["Locked_Threshold"].to_numpy(dtype=float),
                expected_threshold,
                rtol=0,
                atol=THRESHOLD_ATOL,
            ).all()
        ),
    }

    table_audit_rows.append({
        "Screening_Model": TARGET_STRATEGY,
        "Algorithm": algorithm_name,
        **table_checks,
        "Table_Record_Ready": bool(all(table_checks.values())),
    })

part9c_prediction_table_audit = pd.DataFrame(table_audit_rows)


# ============================================================
# 9. VERIFY PRIMARY M2 ADABOOST RECORD
# ============================================================

primary_record = temporal_predictions[
    TARGET_STRATEGY
][primary_algorithm]

primary_checks = {
    "Primary_Strategy_Is_M2": primary_strategy == TARGET_STRATEGY,
    "Primary_Algorithm_Record_Present": (
        primary_algorithm
        in temporal_predictions[TARGET_STRATEGY]
    ),
    "Primary_Record_N_Correct": (
        int(primary_record["n"]) == EXPECTED_N
    ),
    "Primary_Record_Events_Correct": (
        int(primary_record["events"]) == EXPECTED_EVENTS
    ),
    "Primary_Outcome_Order_Correct": np.array_equal(
        np.asarray(primary_record["y_true"], dtype=int),
        y_source.to_numpy(dtype=int),
    ),
    "Primary_Index_Order_Correct": np.array_equal(
        np.asarray(primary_record["original_index"]),
        source_index,
    ),
}

part9c_primary_model_verification = pd.DataFrame([{
    "Primary_Strategy": primary_strategy,
    "Primary_Algorithm": primary_algorithm,
    **primary_checks,
    "Primary_Record_Ready": bool(all(primary_checks.values())),
}])


# ============================================================
# 10. GLOBAL VERIFICATION
# ============================================================

source_checks = {
    "PART9B_Ready": bool(PART9B_READY),
    "Outcome_Name_Correct": str(OUTCOME) == EXPECTED_OUTCOME,
    "Feature_Definition_Correct": feature_definition_correct,
    "Source_N_Correct": observed_n == EXPECTED_N,
    "Source_Events_Correct": observed_events == EXPECTED_EVENTS,
    "Source_Feature_Count_Correct": X_source.shape[1] == 10,
    "Source_Feature_Order_Correct": (
        X_source.columns.tolist() == EXPECTED_FEATURES
    ),
    "Source_Predictors_Complete": not X_source.isna().any().any(),
    "Source_Outcome_Complete": not y_source.isna().any(),
    "Source_Outcome_Binary": set(y_source.unique()) == {0, 1},
    "GPAX_Numeric_And_Finite": bool(
        np.isfinite(gpax_numeric.to_numpy(dtype=float)).all()
    ),
    "All_Prediction_Records_Ready": bool(
        part9c_prediction_record_audit["Record_Ready"].all()
    ),
    "All_Long_Table_Records_Ready": bool(
        part9c_prediction_table_audit["Table_Record_Ready"].all()
    ),
    "Primary_Record_Ready": bool(
        part9c_primary_model_verification[
            "Primary_Record_Ready"
        ].iloc[0]
    ),
}

PART9C_READY = bool(all(source_checks.values()))

part9c_global_verification = pd.DataFrame([{
    "Audit": "Temporal 2025 source and row-alignment integrity",
    "Screening_Model": TARGET_STRATEGY,
    "Primary_Algorithm": primary_algorithm,
    "Observed_N": observed_n,
    "Observed_Events": observed_events,
    "Feature_Count": X_source.shape[1],
    "Algorithms_Audited": len(part9c_prediction_record_audit),
    **source_checks,
    "Participant_Level_Data_Exported": False,
    "Model_Fitted": False,
    "Resampling_Performed": False,
    "Prediction_Generated": False,
    "Recalibration_Performed": False,
    "Threshold_Selected_Or_Modified": False,
    "PART9C_Ready": PART9C_READY,
}])


# ============================================================
# 11. AGGREGATE SOURCE MANIFEST
# ============================================================
# Full fingerprints remain in memory only and are not exported.
# The exported manifest records only whether they were generated.
# ============================================================

part9c_source_manifest = pd.DataFrame([{
    "Canonical_Source": (
        "temporal_datasets['M2_Admin_2Q']"
    ),
    "Upstream_Analysis": "PART 4A",
    "Outcome": EXPECTED_OUTCOME,
    "N": observed_n,
    "Events": observed_events,
    "Feature_Count": len(EXPECTED_FEATURES),
    "Feature_Fingerprint_Generated_In_Memory": bool(
        source_feature_fingerprint
    ),
    "Outcome_Fingerprint_Generated_In_Memory": bool(
        source_outcome_fingerprint
    ),
    "Participant_Level_Rows_Exported": False,
}])


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print("=" * 100)
print("PART 9C: TEMPORAL 2025 SOURCE AND ROW-ALIGNMENT")
print("INTEGRITY AUDIT")
print("=" * 100)

print("\nCanonical source verification:")
display(part9c_source_manifest)

print("\nM2 temporal prediction-record audit:")
display(part9c_prediction_record_audit)

print("\nM2 long-form prediction-table audit:")
display(part9c_prediction_table_audit)

print("\nPrimary-model verification:")
display(part9c_primary_model_verification)

print("\nGlobal verification:")
display(part9c_global_verification)


# ============================================================
# 13. EXPORT AGGREGATE METADATA ONLY
# ============================================================

part9c_prediction_record_audit.to_csv(
    "part9c_temporal_prediction_record_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

part9c_prediction_table_audit.to_csv(
    "part9c_temporal_prediction_table_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

part9c_primary_model_verification.to_csv(
    "part9c_primary_model_verification.csv",
    index=False,
    encoding="utf-8-sig",
)

part9c_source_manifest.to_csv(
    "part9c_temporal_source_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

part9c_global_verification.to_csv(
    "part9c_global_verification.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 14. FINAL SAFETY GATE AND SUMMARY
# ============================================================

if not PART9C_READY:
    failed_checks = [
        name for name, passed in source_checks.items()
        if not bool(passed)
    ]
    raise RuntimeError(
        "PART 9C failed: " + ", ".join(failed_checks)
    )

print("\n" + "=" * 100)
print("PART 9C VERSION 1.0 COMPLETED")
print("=" * 100)
print(f"Canonical source       : temporal_datasets['{TARGET_STRATEGY}']")
print(f"Primary model          : {TARGET_STRATEGY} + {primary_algorithm}")
print(f"Temporal cohort        : N={observed_n:,}; events={observed_events}")
print(f"M2 predictors          : {len(EXPECTED_FEATURES)}")
print(f"Algorithms audited     : {len(part9c_prediction_record_audit)}")
print("Row alignment          : Verified against stored PART 4A records")
print("Participant rows export: No")
print("\nPART9C_READY:", PART9C_READY)
print("\nNo model was fitted.")
print("No resampling was performed.")
print("No prediction was generated.")
print("No recalibration was performed.")
print("No threshold was selected or modified.")


### PART 9D: M2 Sensitivity-Analysis Input Integrity Gate

In [ ]:
# ============================================================
# PART 9D VERSION 1.1: M2 SENSITIVITY-ANALYSIS INPUT
#                         INTEGRITY GATE
# ============================================================
#
# Purpose:
# - Verify the canonical M2 inputs before any prespecified
#   sensitivity analysis.
# - Preserve the original development/held-out split.
# - Confirm temporal-source integrity already established in
#   PARTS 9B and 9C.
# - Make verified inputs available in memory only.
#
# Privacy safeguard:
# - Participant-level predictors, outcomes, identifiers, indices,
#   and predictions are NOT exported.
# - Only aggregate verification metadata are written to CSV.
#
# This PART performs no new split, model fitting, resampling,
# prediction, recalibration, threshold selection, feature selection,
# or performance calculation.
# ============================================================

import hashlib
import numpy as np
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================

TARGET_STRATEGY = "M2_Admin_2Q"
EXPECTED_OUTCOME = "mdddx"

EXPECTED_FEATURES = [
    "institute",
    "GPAX",
    "dadstatus",
    "momstatus",
    "dadeducation",
    "momeducation",
    "parent",
    "gender",
    "2Q1",
    "2Q2",
]

EXPECTED_COHORTS = {
    "Development 2024": {"N": 2863, "Events": 32},
    "Held-out 2024": {"N": 1227, "Events": 14},
    "Temporal 2025": {"N": 815, "Events": 16},
}


# ============================================================
# 2. REQUIRED UPSTREAM OBJECTS AND SAFETY GATES
# ============================================================

required_objects = [
    "screening_strategies",
    "prepared_data",
    "temporal_datasets",
    "OUTCOME",
    "PART9B_READY",
    "PART9C_READY",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required upstream objects: "
        + ", ".join(missing_objects)
    )

if not bool(PART9B_READY):
    raise RuntimeError(
        "PART 9B is not ready. Resolve PART 9B before continuing."
    )

if not bool(PART9C_READY):
    raise RuntimeError(
        "PART 9C is not ready. Resolve PART 9C before continuing."
    )

if str(OUTCOME) != EXPECTED_OUTCOME:
    raise RuntimeError(
        f"Expected OUTCOME='{EXPECTED_OUTCOME}', observed '{OUTCOME}'."
    )

if TARGET_STRATEGY not in screening_strategies:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from screening_strategies."
    )

if TARGET_STRATEGY not in prepared_data:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from prepared_data."
    )

if TARGET_STRATEGY not in temporal_datasets:
    raise KeyError(
        f"{TARGET_STRATEGY} is missing from temporal_datasets."
    )


# ============================================================
# 3. VERIFY THE CANONICAL M2 FEATURE DEFINITION
# ============================================================

observed_features = list(
    screening_strategies[TARGET_STRATEGY]["features"]
)

if observed_features != EXPECTED_FEATURES:
    raise RuntimeError(
        "The canonical M2 feature names or order differ from "
        "the prespecified manuscript definition."
    )

if len(set(observed_features)) != len(observed_features):
    raise RuntimeError("The canonical M2 feature list contains duplicates.")


# ============================================================
# 4. EXTRACT CANONICAL COHORT OBJECTS
# ============================================================

m2_prepared = prepared_data[TARGET_STRATEGY]

required_prepared_keys = [
    "X_train",
    "y_train",
    "X_test",
    "y_test",
]

missing_prepared_keys = [
    key for key in required_prepared_keys
    if key not in m2_prepared
]

if missing_prepared_keys:
    raise KeyError(
        "prepared_data['M2_Admin_2Q'] is missing keys: "
        + ", ".join(missing_prepared_keys)
    )

X_development_source = m2_prepared["X_train"]
y_development_source = m2_prepared["y_train"]
X_heldout_source = m2_prepared["X_test"]
y_heldout_source = m2_prepared["y_test"]
temporal_source = temporal_datasets[TARGET_STRATEGY]

if not isinstance(X_development_source, pd.DataFrame):
    raise TypeError("X_train must be a pandas DataFrame.")

if not isinstance(X_heldout_source, pd.DataFrame):
    raise TypeError("X_test must be a pandas DataFrame.")

if not isinstance(y_development_source, pd.Series):
    raise TypeError("y_train must be a pandas Series.")

if not isinstance(y_heldout_source, pd.Series):
    raise TypeError("y_test must be a pandas Series.")

if not isinstance(temporal_source, pd.DataFrame):
    raise TypeError(
        "temporal_datasets['M2_Admin_2Q'] must be a DataFrame."
    )

for cohort_name, frame in {
    "Development 2024": X_development_source,
    "Held-out 2024": X_heldout_source,
}.items():
    missing = [
        feature for feature in EXPECTED_FEATURES
        if feature not in frame.columns
    ]
    if missing:
        raise KeyError(
            f"{cohort_name}: missing M2 features: "
            + ", ".join(missing)
        )

temporal_required = EXPECTED_FEATURES + [EXPECTED_OUTCOME]
temporal_missing = [
    column for column in temporal_required
    if column not in temporal_source.columns
]

if temporal_missing:
    raise KeyError(
        "Temporal 2025: missing required columns: "
        + ", ".join(temporal_missing)
    )


# ============================================================
# 5. VERIFY X-y ROW ALIGNMENT BEFORE COPYING
# ============================================================

development_index_aligned = (
    len(X_development_source) == len(y_development_source)
    and X_development_source.index.equals(y_development_source.index)
)

heldout_index_aligned = (
    len(X_heldout_source) == len(y_heldout_source)
    and X_heldout_source.index.equals(y_heldout_source.index)
)

if not development_index_aligned:
    raise RuntimeError(
        "Development 2024 predictor and outcome indices are not aligned."
    )

if not heldout_index_aligned:
    raise RuntimeError(
        "Held-out 2024 predictor and outcome indices are not aligned."
    )

development_index = np.asarray(X_development_source.index).copy()
heldout_index = np.asarray(X_heldout_source.index).copy()
temporal_index = np.asarray(temporal_source.index).copy()

def is_local_range_index(index, expected_length):
    """Return True when an index is only a local 0...(n-1) position."""
    return bool(
        isinstance(index, pd.RangeIndex)
        and index.start == 0
        and index.stop == expected_length
        and index.step == 1
    )


development_local_index = is_local_range_index(
    X_development_source.index,
    len(X_development_source),
)
heldout_local_index = is_local_range_index(
    X_heldout_source.index,
    len(X_heldout_source),
)

if development_local_index and heldout_local_index:
    # Local RangeIndex values are row positions, not participant IDs.
    # Numeric overlap therefore cannot be used to assess cohort overlap.
    development_heldout_index_overlap_assessable = False
    development_heldout_numeric_overlap = np.nan
    development_heldout_disjoint = np.nan
    split_integrity_gate_acceptable = bool(PART9B_READY)
    split_index_interpretation = (
        "Not assessable from local RangeIndex values; original shared "
        "70:30 split and cohort composition were verified upstream."
    )
else:
    development_heldout_index_overlap_assessable = True
    development_heldout_numeric_overlap = int(
        len(np.intersect1d(development_index, heldout_index))
    )
    development_heldout_disjoint = bool(
        development_heldout_numeric_overlap == 0
    )
    split_integrity_gate_acceptable = development_heldout_disjoint
    split_index_interpretation = (
        "Checked using preserved nonlocal source indices."
    )

if not split_integrity_gate_acceptable:
    raise RuntimeError(
        "Development and held-out 2024 preserved source indices overlap."
    )

part9d_split_index_audit = pd.DataFrame([{
    "Development_Index_Type": type(
        X_development_source.index
    ).__name__,
    "Heldout_Index_Type": type(
        X_heldout_source.index
    ).__name__,
    "Development_Index_Is_Local_RangeIndex": development_local_index,
    "Heldout_Index_Is_Local_RangeIndex": heldout_local_index,
    "Cross_Cohort_Index_Overlap_Assessable": (
        development_heldout_index_overlap_assessable
    ),
    "Numeric_Index_Overlap": development_heldout_numeric_overlap,
    "Preserved_Source_Indices_Disjoint": development_heldout_disjoint,
    "Upstream_Cohort_Audit_Ready": bool(PART9B_READY),
    "Split_Integrity_Gate_Acceptable": split_integrity_gate_acceptable,
    "Interpretation": split_index_interpretation,
}])


# ============================================================
# 6. CREATE VERIFIED IN-MEMORY COPIES
# ============================================================
# Original indices are preserved. They are not exported.
# ============================================================

X_development = X_development_source.loc[
    :, EXPECTED_FEATURES
].copy()
y_development = pd.to_numeric(
    y_development_source.copy(),
    errors="raise",
).astype(int)

X_heldout = X_heldout_source.loc[
    :, EXPECTED_FEATURES
].copy()
y_heldout = pd.to_numeric(
    y_heldout_source.copy(),
    errors="raise",
).astype(int)

X_temporal = temporal_source.loc[
    :, EXPECTED_FEATURES
].copy()
y_temporal = pd.to_numeric(
    temporal_source.loc[:, EXPECTED_OUTCOME].copy(),
    errors="raise",
).astype(int)


# ============================================================
# 7. COHORT-LEVEL INPUT VALIDATION
# ============================================================

cohort_objects = {
    "Development 2024": (X_development, y_development),
    "Held-out 2024": (X_heldout, y_heldout),
    "Temporal 2025": (X_temporal, y_temporal),
}

cohort_audit_rows = []

for cohort_name, (X_cohort, y_cohort) in cohort_objects.items():
    expected = EXPECTED_COHORTS[cohort_name]

    observed_n = int(len(y_cohort))
    observed_events = int(y_cohort.sum())
    observed_nonevents = int((y_cohort == 0).sum())
    predictor_missing = int(X_cohort.isna().sum().sum())
    outcome_missing = int(y_cohort.isna().sum())
    outcome_values = set(y_cohort.unique())

    row_count_aligned = len(X_cohort) == len(y_cohort)
    index_aligned = X_cohort.index.equals(y_cohort.index)
    feature_order_correct = (
        X_cohort.columns.tolist() == EXPECTED_FEATURES
    )
    binary_outcome = outcome_values == {0, 1}
    n_correct = observed_n == expected["N"]
    events_correct = observed_events == expected["Events"]
    no_missing = predictor_missing == 0 and outcome_missing == 0

    try:
        gpax = pd.to_numeric(X_cohort["GPAX"], errors="raise")
        gpax_numeric_finite = bool(
            np.isfinite(gpax.to_numpy(dtype=float)).all()
        )
    except Exception:
        gpax_numeric_finite = False

    cohort_ready = all([
        row_count_aligned,
        index_aligned,
        feature_order_correct,
        binary_outcome,
        n_correct,
        events_correct,
        no_missing,
        gpax_numeric_finite,
    ])

    cohort_audit_rows.append({
        "Cohort": cohort_name,
        "Observed_N": observed_n,
        "Expected_N": expected["N"],
        "N_Correct": n_correct,
        "Observed_Events": observed_events,
        "Expected_Events": expected["Events"],
        "Events_Correct": events_correct,
        "Observed_Nonevents": observed_nonevents,
        "Feature_Count": X_cohort.shape[1],
        "Feature_Order_Correct": feature_order_correct,
        "Row_Count_Aligned": row_count_aligned,
        "Index_Aligned": index_aligned,
        "Binary_Outcome": binary_outcome,
        "Predictor_Missing_Cells": predictor_missing,
        "Outcome_Missing_Cells": outcome_missing,
        "No_Missing_Values": no_missing,
        "GPAX_Numeric_And_Finite": gpax_numeric_finite,
        "Cohort_Ready": cohort_ready,
    })

part9d_cohort_input_audit = pd.DataFrame(cohort_audit_rows)


# ============================================================
# 8. IN-MEMORY FINGERPRINTS
# ============================================================
# Fingerprints allow later code to verify that the in-memory inputs
# have not changed. Participant values are not included in exports.
# ============================================================

def dataframe_fingerprint(frame):
    hashed = pd.util.hash_pandas_object(
        frame,
        index=True,
        categorize=True,
    ).to_numpy(dtype=np.uint64)
    return hashlib.sha256(hashed.tobytes()).hexdigest()


def series_fingerprint(series):
    hashed = pd.util.hash_pandas_object(
        series,
        index=True,
        categorize=True,
    ).to_numpy(dtype=np.uint64)
    return hashlib.sha256(hashed.tobytes()).hexdigest()


m2_sensitivity_input_fingerprints = {
    cohort_name: {
        "X_SHA256": dataframe_fingerprint(X_cohort),
        "y_SHA256": series_fingerprint(y_cohort),
    }
    for cohort_name, (X_cohort, y_cohort)
    in cohort_objects.items()
}


# ============================================================
# 9. PREPARE VERIFIED INPUTS IN MEMORY ONLY
# ============================================================
# These objects remain in the active notebook session.
# Do not export them to CSV or include them in GitHub.
# ============================================================

m2_sensitivity_inputs = {
    "Development 2024": {
        "X": X_development,
        "y": y_development,
        "source": (
            "prepared_data['M2_Admin_2Q']['X_train'/'y_train']"
        ),
        "role": "Model development only",
    },
    "Held-out 2024": {
        "X": X_heldout,
        "y": y_heldout,
        "source": (
            "prepared_data['M2_Admin_2Q']['X_test'/'y_test']"
        ),
        "role": "Locked held-out validation only",
    },
    "Temporal 2025": {
        "X": X_temporal,
        "y": y_temporal,
        "source": "temporal_datasets['M2_Admin_2Q']",
        "role": "Locked temporal validation only",
    },
}


# ============================================================
# 10. SOURCE AND ANALYSIS-ROLE MANIFEST
# ============================================================

part9d_source_manifest = pd.DataFrame([
    {
        "Cohort": "Development 2024",
        "Canonical_Source": (
            "prepared_data['M2_Admin_2Q']['X_train'/'y_train']"
        ),
        "Permitted_Role": "Development of sensitivity model",
        "Model_Fitting_Permitted": True,
        "Resampling_Permitted": (
            "Training folds only using the prespecified pipeline"
        ),
        "Threshold_Selection_Permitted": (
            "Genuine development-cohort OOF predictions only"
        ),
        "Participant_Level_Export_Permitted": False,
    },
    {
        "Cohort": "Held-out 2024",
        "Canonical_Source": (
            "prepared_data['M2_Admin_2Q']['X_test'/'y_test']"
        ),
        "Permitted_Role": "Independent held-out evaluation only",
        "Model_Fitting_Permitted": False,
        "Resampling_Permitted": False,
        "Threshold_Selection_Permitted": False,
        "Participant_Level_Export_Permitted": False,
    },
    {
        "Cohort": "Temporal 2025",
        "Canonical_Source": "temporal_datasets['M2_Admin_2Q']",
        "Permitted_Role": "Independent temporal evaluation only",
        "Model_Fitting_Permitted": False,
        "Resampling_Permitted": False,
        "Threshold_Selection_Permitted": False,
        "Participant_Level_Export_Permitted": False,
    },
])


# ============================================================
# 11. GLOBAL VERIFICATION
# ============================================================

development_plus_heldout_n_correct = (
    len(y_development) + len(y_heldout) == 4090
)

development_plus_heldout_events_correct = (
    int(y_development.sum()) + int(y_heldout.sum()) == 46
)

global_checks = {
    "PART9B_Ready": bool(PART9B_READY),
    "PART9C_Ready": bool(PART9C_READY),
    "Outcome_Name_Correct": str(OUTCOME) == EXPECTED_OUTCOME,
    "M2_Feature_Definition_Correct": (
        observed_features == EXPECTED_FEATURES
    ),
    "All_Cohorts_Ready": bool(
        part9d_cohort_input_audit["Cohort_Ready"].all()
    ),
    "Development_Heldout_Split_Integrity_Gate_Acceptable": (
        split_integrity_gate_acceptable
    ),
    "Development_Plus_Heldout_N_Correct": (
        development_plus_heldout_n_correct
    ),
    "Development_Plus_Heldout_Events_Correct": (
        development_plus_heldout_events_correct
    ),
    "Three_In_Memory_Input_Sets_Created": (
        len(m2_sensitivity_inputs) == 3
    ),
}

PART9D_READY = bool(all(global_checks.values()))

part9d_global_verification = pd.DataFrame([{
    "Audit": "M2 sensitivity-analysis input integrity gate",
    "Target_Strategy": TARGET_STRATEGY,
    "Outcome": EXPECTED_OUTCOME,
    "Feature_Count": len(EXPECTED_FEATURES),
    **global_checks,
    "New_Split_Created": False,
    "Model_Fitted": False,
    "Resampling_Performed": False,
    "Prediction_Generated": False,
    "Recalibration_Performed": False,
    "Threshold_Selected_Or_Modified": False,
    "Participant_Level_Data_Exported": False,
    "PART9D_Ready": PART9D_READY,
}])


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print("=" * 100)
print("PART 9D: M2 SENSITIVITY-ANALYSIS INPUT INTEGRITY GATE")
print("=" * 100)

print("\nCohort input audit:")
display(part9d_cohort_input_audit)

print("\nSource and permitted-role manifest:")
display(part9d_source_manifest)

print("\nDevelopment-held-out index interpretation:")
display(part9d_split_index_audit)

print("\nGlobal verification:")
display(part9d_global_verification)


# ============================================================
# 13. EXPORT AGGREGATE METADATA ONLY
# ============================================================

part9d_cohort_input_audit.to_csv(
    "part9d_m2_sensitivity_input_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

part9d_source_manifest.to_csv(
    "part9d_m2_sensitivity_source_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

part9d_split_index_audit.to_csv(
    "part9d_split_index_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

part9d_global_verification.to_csv(
    "part9d_global_verification.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 14. FINAL SAFETY GATE AND SUMMARY
# ============================================================

if not PART9D_READY:
    failed_checks = [
        name for name, passed in global_checks.items()
        if not bool(passed)
    ]
    raise RuntimeError(
        "PART 9D failed: " + ", ".join(failed_checks)
    )

print("\n" + "=" * 100)
print("PART 9D VERSION 1.0 COMPLETED")
print("=" * 100)
print(f"Target strategy        : {TARGET_STRATEGY}")
print(f"Outcome                : {EXPECTED_OUTCOME}")
print(f"M2 predictors          : {len(EXPECTED_FEATURES)}")
print("Development 2024       : N=2,863; events=32")
print("Held-out 2024          : N=1,227; events=14")
print("Temporal 2025          : N=815; events=16")
print("In-memory inputs       : m2_sensitivity_inputs")
print("Participant rows export: No")
print("\nPART9D_READY:", PART9D_READY)
print("\nNo new split was created.")
print("No model was fitted.")
print("No resampling was performed.")
print("No prediction was generated.")
print("No recalibration was performed.")
print("No threshold was selected or modified.")
print("\nVerified inputs are available in memory only.")
print("Do not export participant-level inputs or upload them to GitHub.")

### PART 10A: Sensitivity Analysis of Class-Imbalance Handling

In [ ]:
# ============================================================
# PART 10A VERSION 1.0
# SENSITIVITY ANALYSIS OF ALTERNATIVE CLASS-IMBALANCE STRATEGIES
# FINAL M2_Admin_2Q AdaBoost MODEL
# ============================================================
#
# Strategies compared:
# 1. Primary hybrid resampling:
#    SMOTENC followed by random undersampling.
# 2. Unweighted:
#    no resampling and no observation weighting.
# 3. Class-weighted:
#    no resampling; balanced sample weights calculated separately
#    inside every development training fold.
#
# Safeguards:
# - The original 2024 development/held-out split is preserved.
# - Genuine 5-fold stratified OOF predictions are used for threshold
#   selection in every sensitivity strategy.
# - Held-out 2024 and temporal 2025 outcomes are not used for model
#   fitting, threshold selection, or strategy selection.
# - Thresholds are locked before validation.
# - No sensitivity strategy is declared a replacement winner.
# - Participant-level predictions are retained in memory only.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
)


# ============================================================
# 1. SETTINGS
# ============================================================

TARGET_STRATEGY = "M2_Admin_2Q"
EXPECTED_OUTCOME = "mdddx"
N_FOLDS = 5
RANDOM_STATE = 42
TARGET_SENSITIVITY = 0.50
THRESHOLD_EPSILON = 1e-12
THRESHOLD_ATOL = 1e-12

EXPECTED_FEATURES = [
    "institute",
    "GPAX",
    "dadstatus",
    "momstatus",
    "dadeducation",
    "momeducation",
    "parent",
    "gender",
    "2Q1",
    "2Q2",
]

IMBALANCE_STRATEGIES = [
    "Primary_Hybrid_SMOTENC_RUS",
    "Unweighted_No_Resampling",
    "Class_Weighted_No_Resampling",
]


# ============================================================
# 2. REQUIRED UPSTREAM OBJECTS
# ============================================================

required_objects = [
    "PART9D_READY",
    "m2_sensitivity_inputs",
    "screening_strategies",
    "hybrid_pipeline_templates",
    "evaluation_models",
    "oof_predictions",
    "heldout_predictions",
    "temporal_predictions",
    "locked_thresholds",
    "part5c_final_recommendation",
    "OUTCOME",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required upstream objects: "
        + ", ".join(missing_objects)
    )

if not bool(PART9D_READY):
    raise RuntimeError("PART 9D is not ready.")

if str(OUTCOME) != EXPECTED_OUTCOME:
    raise RuntimeError(
        f"Expected OUTCOME='{EXPECTED_OUTCOME}', observed '{OUTCOME}'."
    )

primary_strategy = str(
    part5c_final_recommendation["Primary_Recommended_Strategy"]
)
primary_algorithm = str(
    part5c_final_recommendation["Primary_Recommended_Algorithm"]
)

if primary_strategy != TARGET_STRATEGY:
    raise RuntimeError("The PART 5C primary strategy is not M2_Admin_2Q.")

if primary_algorithm not in hybrid_pipeline_templates[TARGET_STRATEGY]:
    raise KeyError("The primary algorithm template is unavailable.")

if primary_algorithm not in evaluation_models[TARGET_STRATEGY]:
    raise KeyError("The fitted primary evaluation model is unavailable.")

observed_features = list(
    screening_strategies[TARGET_STRATEGY]["features"]
)

if observed_features != EXPECTED_FEATURES:
    raise RuntimeError("The locked M2 feature definition has changed.")


# ============================================================
# 3. LOAD VERIFIED IN-MEMORY INPUTS
# ============================================================

X_development = m2_sensitivity_inputs[
    "Development 2024"
]["X"].copy().reset_index(drop=True)
y_development = m2_sensitivity_inputs[
    "Development 2024"
]["y"].copy().astype(int).reset_index(drop=True)

X_heldout = m2_sensitivity_inputs[
    "Held-out 2024"
]["X"].copy().reset_index(drop=True)
y_heldout = m2_sensitivity_inputs[
    "Held-out 2024"
]["y"].copy().astype(int).reset_index(drop=True)

X_temporal = m2_sensitivity_inputs[
    "Temporal 2025"
]["X"].copy().reset_index(drop=True)
y_temporal = m2_sensitivity_inputs[
    "Temporal 2025"
]["y"].copy().astype(int).reset_index(drop=True)

cohort_inputs = {
    "Development OOF": (X_development, y_development, 2863, 32),
    "Held-out 2024": (X_heldout, y_heldout, 1227, 14),
    "Temporal 2025": (X_temporal, y_temporal, 815, 16),
}

for cohort_name, (X_cohort, y_cohort, expected_n, expected_events) in (
    cohort_inputs.items()
):
    if X_cohort.columns.tolist() != EXPECTED_FEATURES:
        raise RuntimeError(f"{cohort_name}: incorrect feature order.")
    if len(X_cohort) != len(y_cohort):
        raise RuntimeError(f"{cohort_name}: X-y row-count mismatch.")
    if len(y_cohort) != expected_n:
        raise RuntimeError(f"{cohort_name}: incorrect N.")
    if int(y_cohort.sum()) != expected_events:
        raise RuntimeError(f"{cohort_name}: incorrect event count.")
    if X_cohort.isna().any().any() or y_cohort.isna().any():
        raise RuntimeError(f"{cohort_name}: missing values detected.")
    if set(y_cohort.unique()) != {0, 1}:
        raise RuntimeError(f"{cohort_name}: outcome is not binary.")


# ============================================================
# 4. LOCK THE ADA BOOST ESTIMATOR SPECIFICATION
# ============================================================

primary_pipeline_template = hybrid_pipeline_templates[
    TARGET_STRATEGY
][primary_algorithm]

if not hasattr(primary_pipeline_template, "named_steps"):
    raise TypeError("The primary hybrid template is not a pipeline.")

expected_steps = ["over_sampler", "under_sampler", "estimator"]
if list(primary_pipeline_template.named_steps) != expected_steps:
    raise RuntimeError("The primary hybrid pipeline step order is incorrect.")

observed_over_sampler = type(
    primary_pipeline_template.named_steps["over_sampler"]
).__name__

if observed_over_sampler != "SMOTENC":
    raise RuntimeError(
        f"Expected SMOTENC for M2, observed {observed_over_sampler}."
    )

if type(
    primary_pipeline_template.named_steps["under_sampler"]
).__name__ != "RandomUnderSampler":
    raise RuntimeError("The primary under-sampler is not RandomUnderSampler.")

base_estimator_template = clone(
    primary_pipeline_template.named_steps["estimator"]
)

if type(base_estimator_template).__name__ != primary_algorithm:
    raise RuntimeError(
        "The estimator in the primary pipeline does not match PART 5C."
    )

primary_fitted_pipeline = evaluation_models[
    TARGET_STRATEGY
][primary_algorithm]


# ============================================================
# 5. PROBABILITY AND METRIC HELPERS
# ============================================================

def positive_probability(estimator, X):
    matrix = np.asarray(estimator.predict_proba(X), dtype=float)
    classes = np.asarray(estimator.classes_)
    positive_positions = np.flatnonzero(classes == 1)
    if matrix.ndim != 2 or len(positive_positions) != 1:
        raise ValueError("Unable to identify the class-1 probability.")
    probability = matrix[:, int(positive_positions[0])]
    if (
        not np.isfinite(probability).all()
        or (probability < 0).any()
        or (probability > 1).any()
    ):
        raise ValueError("Invalid predicted probabilities.")
    return probability


def calculate_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= float(threshold)).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()
    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    precision = precision_score(y_true, y_pred, zero_division=0)
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    return {
        "AUC": float(roc_auc_score(y_true, y_prob)),
        "Average_Precision": float(
            average_precision_score(y_true, y_prob)
        ),
        "Brier_Score": float(brier_score_loss(y_true, y_prob)),
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "Precision": float(precision),
        "Negative_Predictive_Value": float(npv),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Balanced_Accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
        "Predicted_Positive_N": int(y_pred.sum()),
        "Predicted_Positive_Rate": float(y_pred.mean()),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "y_pred": y_pred,
    }


# ============================================================
# 6. THRESHOLD-SELECTION HELPERS
# ============================================================

def empirical_threshold_candidates(y_prob):
    y_prob = np.asarray(y_prob, dtype=float)
    if not np.isfinite(y_prob).all():
        raise ValueError("OOF probabilities contain nonfinite values.")
    unique_probabilities = np.unique(y_prob)
    lower = max(
        0.0,
        np.nextafter(float(unique_probabilities.min()), -np.inf),
    )
    upper = min(
        1.0 + THRESHOLD_EPSILON,
        np.nextafter(float(unique_probabilities.max()), np.inf),
    )
    return np.unique(
        np.concatenate([[lower], unique_probabilities, [upper]])
    )


def threshold_search(y_true, y_prob):
    rows = []
    for threshold in empirical_threshold_candidates(y_prob):
        metrics = calculate_metrics(y_true, y_prob, threshold)
        rows.append({
            "Threshold": float(threshold),
            "Sensitivity": metrics["Sensitivity"],
            "Specificity": metrics["Specificity"],
            "Precision": metrics["Precision"],
            "F1": metrics["F1"],
            "Predicted_Positive_N": metrics["Predicted_Positive_N"],
        })
    return pd.DataFrame(rows)


def select_threshold(search_table):
    eligible = search_table.loc[
        search_table["Sensitivity"] >= TARGET_SENSITIVITY
    ].copy()
    if not eligible.empty:
        selected = eligible.sort_values(
            ["F1", "Precision", "Specificity", "Threshold"],
            ascending=[False, False, False, False],
            kind="mergesort",
        ).iloc[0]
        rule = (
            f"Sensitivity>={TARGET_SENSITIVITY:.2f}; max F1; "
            "tie: precision, specificity, threshold"
        )
        target_achieved = True
    else:
        selected = search_table.sort_values(
            [
                "F1",
                "Sensitivity",
                "Precision",
                "Specificity",
                "Threshold",
            ],
            ascending=[False, False, False, False, False],
            kind="mergesort",
        ).iloc[0]
        rule = (
            "Fallback: max F1 overall; tie: sensitivity, precision, "
            "specificity, threshold"
        )
        target_achieved = False
    return float(selected["Threshold"]), rule, target_achieved


# ============================================================
# 7. OOF GENERATION FOR NONRESAMPLED SENSITIVITY STRATEGIES
# ============================================================

oof_cv = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


def generate_nonresampled_oof(X, y, weighted):
    n = len(y)
    probabilities = np.full(n, np.nan, dtype=float)
    fold_ids = np.full(n, -1, dtype=int)
    validation_counter = np.zeros(n, dtype=int)
    fold_rows = []

    for fold_number, (train_position, validation_position) in enumerate(
        oof_cv.split(X, y), start=1
    ):
        X_fold_train = X.iloc[train_position].copy()
        y_fold_train = y.iloc[train_position].copy()
        X_fold_validation = X.iloc[validation_position].copy()
        y_fold_validation = y.iloc[validation_position].copy()

        estimator = clone(base_estimator_template)
        if weighted:
            fold_weight = compute_sample_weight(
                class_weight="balanced",
                y=y_fold_train,
            )
            estimator.fit(
                X_fold_train,
                y_fold_train,
                sample_weight=fold_weight,
            )
            minimum_weight = float(fold_weight.min())
            maximum_weight = float(fold_weight.max())
        else:
            estimator.fit(X_fold_train, y_fold_train)
            minimum_weight = 1.0
            maximum_weight = 1.0

        fold_probability = positive_probability(
            estimator,
            X_fold_validation,
        )

        probabilities[validation_position] = fold_probability
        fold_ids[validation_position] = fold_number
        validation_counter[validation_position] += 1

        fold_rows.append({
            "Fold": fold_number,
            "Training_N": len(train_position),
            "Training_Events": int(y_fold_train.sum()),
            "Validation_N": len(validation_position),
            "Validation_Events": int(y_fold_validation.sum()),
            "Sample_Weight_Min": minimum_weight,
            "Sample_Weight_Max": maximum_weight,
            "Resampling_Performed": False,
            "Validation_Predicted_Exactly_Once": True,
        })

    if np.isnan(probabilities).any():
        raise RuntimeError("Missing nonresampled OOF probabilities.")
    if not np.all(validation_counter == 1):
        raise RuntimeError("Each participant must receive one OOF prediction.")
    if (fold_ids < 1).any():
        raise RuntimeError("Missing OOF fold assignments.")

    return {
        "y_true": y.to_numpy(dtype=int),
        "y_prob": probabilities,
        "fold_id": fold_ids,
        "validation_counter": validation_counter,
        "fold_manifest": pd.DataFrame(fold_rows),
    }


# ============================================================
# 8. OBTAIN OOF PREDICTIONS FOR ALL THREE STRATEGIES
# ============================================================

primary_oof_record = oof_predictions[
    TARGET_STRATEGY
][primary_algorithm]

primary_oof_y = np.asarray(primary_oof_record["y_true"], dtype=int)
primary_oof_prob = np.asarray(primary_oof_record["y_prob"], dtype=float)

if not np.array_equal(
    primary_oof_y,
    y_development.to_numpy(dtype=int),
):
    raise RuntimeError("Primary OOF outcome order is not aligned.")

unweighted_oof = generate_nonresampled_oof(
    X_development,
    y_development,
    weighted=False,
)

weighted_oof = generate_nonresampled_oof(
    X_development,
    y_development,
    weighted=True,
)

part10a_oof_predictions = {
    "Primary_Hybrid_SMOTENC_RUS": {
        "y_true": primary_oof_y,
        "y_prob": primary_oof_prob,
        "fold_id": np.asarray(primary_oof_record["fold_id"], dtype=int),
        "source": "Existing genuine PART 3A OOF predictions",
    },
    "Unweighted_No_Resampling": {
        **unweighted_oof,
        "source": "New genuine 5-fold OOF sensitivity predictions",
    },
    "Class_Weighted_No_Resampling": {
        **weighted_oof,
        "source": "New genuine 5-fold OOF sensitivity predictions",
    },
}


# ============================================================
# 9. OOF-ONLY THRESHOLD LOCKING
# ============================================================

part10a_locked_thresholds = {}
part10a_threshold_search_tables = {}
oof_summary_rows = []

for imbalance_strategy in IMBALANCE_STRATEGIES:
    record = part10a_oof_predictions[imbalance_strategy]
    y_true = np.asarray(record["y_true"], dtype=int)
    y_prob = np.asarray(record["y_prob"], dtype=float)

    if imbalance_strategy == "Primary_Hybrid_SMOTENC_RUS":
        threshold = float(
            locked_thresholds[TARGET_STRATEGY][primary_algorithm]
        )
        search_table = threshold_search(y_true, y_prob)
        _, derived_rule, target_achieved = select_threshold(search_table)
        rule = "Existing development-OOF locked primary threshold"
    else:
        search_table = threshold_search(y_true, y_prob)
        threshold, derived_rule, target_achieved = select_threshold(
            search_table
        )
        rule = derived_rule

    metrics = calculate_metrics(y_true, y_prob, threshold)
    part10a_locked_thresholds[imbalance_strategy] = threshold
    part10a_threshold_search_tables[imbalance_strategy] = search_table

    oof_summary_rows.append({
        "Imbalance_Strategy": imbalance_strategy,
        "Algorithm": primary_algorithm,
        "N": len(y_true),
        "Events": int(y_true.sum()),
        "Locked_Threshold": threshold,
        "Threshold_Source": "Development 2024 genuine OOF only",
        "Threshold_Rule": rule,
        "Target_Sensitivity_Achieved": bool(
            metrics["Sensitivity"] >= TARGET_SENSITIVITY
        ),
        **{
            key: value for key, value in metrics.items()
            if key != "y_pred"
        },
    })

part10a_oof_summary = pd.DataFrame(oof_summary_rows)

# All sensitivity thresholds are now locked before either validation set
# is evaluated below.
PART10A_THRESHOLDS_LOCKED = bool(
    len(part10a_locked_thresholds) == len(IMBALANCE_STRATEGIES)
    and all(
        np.isfinite(value)
        for value in part10a_locked_thresholds.values()
    )
)

if not PART10A_THRESHOLDS_LOCKED:
    raise RuntimeError("Sensitivity thresholds were not fully locked.")


# ============================================================
# 10. FIT NONRESAMPLED MODELS USING DEVELOPMENT DATA ONLY
# ============================================================

unweighted_final_model = clone(base_estimator_template)
unweighted_final_model.fit(X_development, y_development)

weighted_final_model = clone(base_estimator_template)
full_development_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_development,
)
weighted_final_model.fit(
    X_development,
    y_development,
    sample_weight=full_development_weights,
)

part10a_fitted_models = {
    "Primary_Hybrid_SMOTENC_RUS": primary_fitted_pipeline,
    "Unweighted_No_Resampling": unweighted_final_model,
    "Class_Weighted_No_Resampling": weighted_final_model,
}


# ============================================================
# 11. LOCKED HELD-OUT AND TEMPORAL EVALUATION
# ============================================================

part10a_validation_predictions = {}
validation_rows = []

primary_validation_records = {
    "Held-out 2024": heldout_predictions[
        TARGET_STRATEGY
    ][primary_algorithm],
    "Temporal 2025": temporal_predictions[
        TARGET_STRATEGY
    ][primary_algorithm],
}

validation_inputs = {
    "Held-out 2024": (X_heldout, y_heldout),
    "Temporal 2025": (X_temporal, y_temporal),
}

for imbalance_strategy in IMBALANCE_STRATEGIES:
    threshold = part10a_locked_thresholds[imbalance_strategy]
    part10a_validation_predictions[imbalance_strategy] = {}

    for validation_set, (X_validation, y_validation) in (
        validation_inputs.items()
    ):
        if imbalance_strategy == "Primary_Hybrid_SMOTENC_RUS":
            stored = primary_validation_records[validation_set]
            y_true = np.asarray(stored["y_true"], dtype=int)
            y_prob = np.asarray(stored["y_prob"], dtype=float)
            if not np.array_equal(
                y_true,
                y_validation.to_numpy(dtype=int),
            ):
                raise RuntimeError(
                    f"Primary {validation_set} outcome order mismatch."
                )
            if not np.isclose(
                float(stored["locked_threshold"]),
                threshold,
                rtol=0,
                atol=THRESHOLD_ATOL,
            ):
                raise RuntimeError("Primary validation threshold mismatch.")
            prediction_source = "Existing locked primary predictions"
        else:
            fitted_model = part10a_fitted_models[imbalance_strategy]
            y_true = y_validation.to_numpy(dtype=int)
            y_prob = positive_probability(fitted_model, X_validation)
            prediction_source = (
                "Sensitivity model fitted on development 2024 only"
            )

        metrics = calculate_metrics(y_true, y_prob, threshold)
        part10a_validation_predictions[imbalance_strategy][
            validation_set
        ] = {
            "y_true": y_true,
            "y_prob": y_prob,
            "y_pred": metrics["y_pred"],
            "locked_threshold": threshold,
            "prediction_source": prediction_source,
        }

        validation_rows.append({
            "Imbalance_Strategy": imbalance_strategy,
            "Algorithm": primary_algorithm,
            "Validation_Set": validation_set,
            "N": len(y_true),
            "Events": int(y_true.sum()),
            "Locked_Threshold": threshold,
            "Threshold_Source": "Development 2024 genuine OOF only",
            "Model_Training_Source": "Development 2024 only",
            **{
                key: value for key, value in metrics.items()
                if key != "y_pred"
            },
        })

part10a_validation_results = pd.DataFrame(validation_rows)


# ============================================================
# 12. INTEGRATED PUBLICATION TABLE
# ============================================================

part10a_integrated_results = pd.concat([
    part10a_oof_summary.assign(Evaluation_Set="Development OOF"),
    part10a_validation_results.rename(
        columns={"Validation_Set": "Evaluation_Set"}
    ),
], ignore_index=True, sort=False)

publication_columns = [
    "Imbalance_Strategy",
    "Algorithm",
    "Evaluation_Set",
    "N",
    "Events",
    "Locked_Threshold",
    "AUC",
    "Average_Precision",
    "Brier_Score",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "TP",
    "FP",
    "FN",
    "TN",
]

part10a_publication_table = part10a_integrated_results.loc[
    :, publication_columns
].copy()


# ============================================================
# 13. METHODOLOGICAL VERIFICATION
# ============================================================

fold_verification_rows = []
for imbalance_strategy, oof_record in part10a_oof_predictions.items():
    y_true = np.asarray(oof_record["y_true"], dtype=int)
    y_prob = np.asarray(oof_record["y_prob"], dtype=float)
    fold_id = np.asarray(oof_record["fold_id"], dtype=int)
    fold_verification_rows.append({
        "Imbalance_Strategy": imbalance_strategy,
        "N": len(y_true),
        "Events": int(y_true.sum()),
        "Five_Folds_Present": set(np.unique(fold_id)) == {1, 2, 3, 4, 5},
        "All_Probabilities_Finite": bool(np.isfinite(y_prob).all()),
        "All_Probabilities_In_Unit_Interval": bool(
            (y_prob >= 0).all() and (y_prob <= 1).all()
        ),
        "Threshold_Locked": imbalance_strategy in part10a_locked_thresholds,
        "Heldout_Used_For_Threshold": False,
        "Temporal_Used_For_Threshold": False,
        "Sensitivity_Strategy_Winner_Selected": False,
    })

part10a_oof_verification = pd.DataFrame(fold_verification_rows)

all_expected_results_present = bool(
    len(part10a_oof_summary) == 3
    and len(part10a_validation_results) == 6
    and set(part10a_oof_summary["Imbalance_Strategy"])
    == set(IMBALANCE_STRATEGIES)
)

PART10A_READY = bool(
    PART10A_THRESHOLDS_LOCKED
    and all_expected_results_present
    and part10a_oof_verification[
        [
            "Five_Folds_Present",
            "All_Probabilities_Finite",
            "All_Probabilities_In_Unit_Interval",
            "Threshold_Locked",
        ]
    ].all(axis=None)
)

part10a_global_verification = pd.DataFrame([{
    "Target_Strategy": TARGET_STRATEGY,
    "Algorithm": primary_algorithm,
    "Imbalance_Strategies_Compared": 3,
    "Development_N": len(y_development),
    "Development_Events": int(y_development.sum()),
    "Heldout_N": len(y_heldout),
    "Heldout_Events": int(y_heldout.sum()),
    "Temporal_N": len(y_temporal),
    "Temporal_Events": int(y_temporal.sum()),
    "Genuine_5_Fold_OOF": True,
    "Thresholds_Derived_From_Development_OOF_Only": True,
    "Validation_Used_For_Model_Or_Threshold_Selection": False,
    "Primary_Model_Replaced": False,
    "Sensitivity_Winner_Selected": False,
    "Participant_Level_Predictions_Exported": False,
    "PART10A_Ready": PART10A_READY,
}])


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print("=" * 105)
print("PART 10A: SENSITIVITY ANALYSIS OF CLASS-IMBALANCE HANDLING")
print("=" * 105)
print(f"Final model: {TARGET_STRATEGY} + {primary_algorithm}")
print("\nDevelopment OOF results and locked thresholds:")
display(part10a_oof_summary)
print("\nHeld-out and temporal validation results:")
display(part10a_validation_results)
print("\nMethodological verification:")
display(part10a_oof_verification)
print("\nGlobal verification:")
display(part10a_global_verification)


# ============================================================
# 15. EXPORT AGGREGATE RESULTS ONLY
# ============================================================

part10a_oof_summary.to_csv(
    "part10a_oof_threshold_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

part10a_validation_results.to_csv(
    "part10a_validation_results.csv",
    index=False,
    encoding="utf-8-sig",
)

part10a_publication_table.to_csv(
    "part10a_class_imbalance_sensitivity_publication_table.csv",
    index=False,
    encoding="utf-8-sig",
)

part10a_oof_verification.to_csv(
    "part10a_oof_verification.csv",
    index=False,
    encoding="utf-8-sig",
)

part10a_global_verification.to_csv(
    "part10a_global_verification.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 16. FINAL SAFETY GATE AND SUMMARY
# ============================================================

if not PART10A_READY:
    raise RuntimeError("PART 10A methodological verification failed.")

print("\n" + "=" * 105)
print("PART 10A VERSION 1.0 COMPLETED")
print("=" * 105)
print(f"Final model                : {TARGET_STRATEGY} + {primary_algorithm}")
print("Imbalance strategies       : 3")
print("OOF folds                  : 5")
print("OOF random state           : 42")
print("Threshold sensitivity floor: 0.50")
print("Sensitivity winner selected: No")
print("Primary model replaced     : No")
print("Participant rows exported : No")
print("\nPART10A_READY:", PART10A_READY)
print("\nPrimary hybrid resampling remains the prespecified analysis.")
print("Unweighted and class-weighted models are sensitivity analyses only.")

### PART 10B: Bootstrap Confidence Intervals for Class-Imbalance Sensitivity Analyses

In [ ]:
# ============================================================
# PART 10B VERSION 1.0
# BOOTSTRAP CONFIDENCE INTERVALS FOR CLASS-IMBALANCE
# SENSITIVITY ANALYSES
# ============================================================
#
# Purpose:
# - Quantify uncertainty in the fixed held-out 2024 and temporal
#   2025 predictions created in PART 10A.
# - Compare uncertainty across primary hybrid resampling,
#   unweighted, and class-weighted M2 AdaBoost analyses.
#
# Method:
# - Stratified participant-level percentile bootstrap.
# - Positive and negative observations are sampled separately with
#   replacement within each validation cohort.
# - 2,000 replicates per strategy-cohort combination.
# - 95% CIs use the 2.5th and 97.5th percentiles.
#
# Safeguards:
# - No model is fitted or refitted.
# - No prediction is regenerated.
# - No threshold is selected or modified.
# - No recalibration is performed.
# - Participant-level predictions and bootstrap distributions are
#   retained in memory only and are not exported.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)


# ============================================================
# 1. SETTINGS
# ============================================================

N_BOOTSTRAPS = 2000
MINIMUM_VALID_BOOTSTRAPS = 1800
CI_LEVEL = 0.95
LOWER_PERCENTILE = 2.5
UPPER_PERCENTILE = 97.5
BASE_RANDOM_STATE = 42000
THRESHOLD_ATOL = 1e-12

EXPECTED_STRATEGIES = [
    "Primary_Hybrid_SMOTENC_RUS",
    "Unweighted_No_Resampling",
    "Class_Weighted_No_Resampling",
]

EXPECTED_VALIDATION_SETS = [
    "Held-out 2024",
    "Temporal 2025",
]

EXPECTED_COHORTS = {
    "Held-out 2024": {"N": 1227, "Events": 14},
    "Temporal 2025": {"N": 815, "Events": 16},
}

METRICS = [
    "AUC",
    "Average_Precision",
    "Brier_Score",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
]


# ============================================================
# 2. REQUIRED PART 10A OBJECTS
# ============================================================

required_objects = [
    "PART10A_READY",
    "part10a_validation_predictions",
    "part10a_validation_results",
    "part10a_locked_thresholds",
    "part5c_final_recommendation",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required PART 10A objects: "
        + ", ".join(missing_objects)
    )

if not bool(PART10A_READY):
    raise RuntimeError("PART 10A is not ready.")

primary_strategy = str(
    part5c_final_recommendation["Primary_Recommended_Strategy"]
)
primary_algorithm = str(
    part5c_final_recommendation["Primary_Recommended_Algorithm"]
)

if primary_strategy != "M2_Admin_2Q":
    raise RuntimeError("The primary strategy is not M2_Admin_2Q.")

if set(part10a_validation_predictions) != set(EXPECTED_STRATEGIES):
    raise RuntimeError(
        "PART 10A validation-prediction strategies are incomplete."
    )


# ============================================================
# 3. FIXED-PREDICTION METRIC FUNCTION
# ============================================================

def calculate_fixed_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= float(threshold)).astype(int)

    if len(y_true) != len(y_prob):
        raise ValueError("Outcome and probability lengths differ.")
    if set(np.unique(y_true)) != {0, 1}:
        raise ValueError("Both outcome classes must be present.")
    if (
        not np.isfinite(y_prob).all()
        or (y_prob < 0).any()
        or (y_prob > 1).any()
    ):
        raise ValueError("Invalid fixed probabilities.")

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "AUC": float(roc_auc_score(y_true, y_prob)),
        "Average_Precision": float(
            average_precision_score(y_true, y_prob)
        ),
        "Brier_Score": float(brier_score_loss(y_true, y_prob)),
        "Sensitivity": float(
            recall_score(y_true, y_pred, zero_division=0)
        ),
        "Specificity": float(specificity),
        "Precision": float(
            precision_score(y_true, y_pred, zero_division=0)
        ),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
    }


# ============================================================
# 4. STRATIFIED PARTICIPANT-LEVEL BOOTSTRAP
# ============================================================

def stratified_bootstrap(
    y_true,
    y_prob,
    threshold,
    n_bootstraps,
    random_state,
):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    positive_positions = np.flatnonzero(y_true == 1)
    negative_positions = np.flatnonzero(y_true == 0)

    if len(positive_positions) == 0 or len(negative_positions) == 0:
        raise ValueError("Both outcome classes are required for bootstrap.")

    rng = np.random.default_rng(random_state)
    distributions = {
        metric: np.full(n_bootstraps, np.nan, dtype=float)
        for metric in METRICS
    }

    for bootstrap_index in range(n_bootstraps):
        sampled_positive = rng.choice(
            positive_positions,
            size=len(positive_positions),
            replace=True,
        )
        sampled_negative = rng.choice(
            negative_positions,
            size=len(negative_positions),
            replace=True,
        )
        sampled_positions = np.concatenate([
            sampled_positive,
            sampled_negative,
        ])
        rng.shuffle(sampled_positions)

        sampled_y = y_true[sampled_positions]
        sampled_probability = y_prob[sampled_positions]

        try:
            replicate_metrics = calculate_fixed_metrics(
                sampled_y,
                sampled_probability,
                threshold,
            )
            for metric in METRICS:
                distributions[metric][bootstrap_index] = (
                    replicate_metrics[metric]
                )
        except Exception:
            continue

    return distributions


# ============================================================
# 5. VERIFY FIXED INPUT RECORDS BEFORE BOOTSTRAP
# ============================================================

input_audit_rows = []

for strategy in EXPECTED_STRATEGIES:
    if strategy not in part10a_locked_thresholds:
        raise KeyError(f"Missing locked threshold for {strategy}.")

    for validation_set in EXPECTED_VALIDATION_SETS:
        if validation_set not in part10a_validation_predictions[strategy]:
            raise KeyError(
                f"Missing {validation_set} predictions for {strategy}."
            )

        record = part10a_validation_predictions[
            strategy
        ][validation_set]

        required_keys = [
            "y_true",
            "y_prob",
            "y_pred",
            "locked_threshold",
        ]
        missing_keys = [
            key for key in required_keys
            if key not in record
        ]
        if missing_keys:
            raise KeyError(
                f"{strategy} | {validation_set}: missing record keys: "
                + ", ".join(missing_keys)
            )

        y_true = np.asarray(record["y_true"], dtype=int)
        y_prob = np.asarray(record["y_prob"], dtype=float)
        y_pred = np.asarray(record["y_pred"], dtype=int)
        threshold = float(record["locked_threshold"])
        expected_threshold = float(part10a_locked_thresholds[strategy])
        expected = EXPECTED_COHORTS[validation_set]

        threshold_matches = bool(
            np.isclose(
                threshold,
                expected_threshold,
                rtol=0,
                atol=THRESHOLD_ATOL,
            )
        )
        predictions_match = np.array_equal(
            y_pred,
            (y_prob >= expected_threshold).astype(int),
        )

        checks = {
            "N_Correct": len(y_true) == expected["N"],
            "Events_Correct": int(y_true.sum()) == expected["Events"],
            "Outcome_Probability_Length_Aligned": (
                len(y_true) == len(y_prob)
            ),
            "Outcome_Prediction_Length_Aligned": (
                len(y_true) == len(y_pred)
            ),
            "Outcome_Binary": set(np.unique(y_true)) == {0, 1},
            "Probabilities_Finite": bool(np.isfinite(y_prob).all()),
            "Probabilities_In_Unit_Interval": bool(
                (y_prob >= 0).all() and (y_prob <= 1).all()
            ),
            "Threshold_Matches_PART10A": threshold_matches,
            "Predictions_Match_Locked_Threshold": predictions_match,
        }

        input_audit_rows.append({
            "Imbalance_Strategy": strategy,
            "Validation_Set": validation_set,
            "N": len(y_true),
            "Events": int(y_true.sum()),
            "Locked_Threshold": threshold,
            **checks,
            "Input_Ready": bool(all(checks.values())),
        })

part10b_input_audit = pd.DataFrame(input_audit_rows)

if not part10b_input_audit["Input_Ready"].all():
    raise RuntimeError("One or more fixed PART 10A inputs failed audit.")


# ============================================================
# 6. RUN BOOTSTRAP ANALYSES
# ============================================================

bootstrap_result_rows = []
bootstrap_status_rows = []
part10b_bootstrap_distributions = {}

for strategy_index, strategy in enumerate(EXPECTED_STRATEGIES):
    part10b_bootstrap_distributions[strategy] = {}

    for cohort_index, validation_set in enumerate(
        EXPECTED_VALIDATION_SETS
    ):
        record = part10a_validation_predictions[
            strategy
        ][validation_set]
        y_true = np.asarray(record["y_true"], dtype=int)
        y_prob = np.asarray(record["y_prob"], dtype=float)
        threshold = float(record["locked_threshold"])

        random_state = (
            BASE_RANDOM_STATE
            + strategy_index * 1000
            + cohort_index * 100
        )

        point_metrics = calculate_fixed_metrics(
            y_true,
            y_prob,
            threshold,
        )
        distributions = stratified_bootstrap(
            y_true=y_true,
            y_prob=y_prob,
            threshold=threshold,
            n_bootstraps=N_BOOTSTRAPS,
            random_state=random_state,
        )
        part10b_bootstrap_distributions[strategy][
            validation_set
        ] = distributions

        metric_valid_counts = []

        for metric in METRICS:
            valid_values = distributions[metric][
                np.isfinite(distributions[metric])
            ]
            valid_n = int(len(valid_values))
            metric_valid_counts.append(valid_n)

            ci_computed = valid_n >= MINIMUM_VALID_BOOTSTRAPS
            if ci_computed:
                lower_ci = float(
                    np.percentile(valid_values, LOWER_PERCENTILE)
                )
                upper_ci = float(
                    np.percentile(valid_values, UPPER_PERCENTILE)
                )
            else:
                lower_ci = np.nan
                upper_ci = np.nan

            bootstrap_result_rows.append({
                "Imbalance_Strategy": strategy,
                "Algorithm": primary_algorithm,
                "Validation_Set": validation_set,
                "N": len(y_true),
                "Events": int(y_true.sum()),
                "Locked_Threshold": threshold,
                "Metric": metric,
                "Estimate": float(point_metrics[metric]),
                "CI_Lower": lower_ci,
                "CI_Upper": upper_ci,
                "CI_Level": CI_LEVEL,
                "CI_Method": (
                    "Stratified percentile bootstrap "
                    "(2.5th-97.5th percentiles)"
                ),
                "Requested_Bootstrap_Replicates": N_BOOTSTRAPS,
                "Valid_Bootstrap_Replicates": valid_n,
                "CI_Computed": ci_computed,
                "Random_State": random_state,
            })

        minimum_valid = int(min(metric_valid_counts))
        bootstrap_status_rows.append({
            "Imbalance_Strategy": strategy,
            "Algorithm": primary_algorithm,
            "Validation_Set": validation_set,
            "N": len(y_true),
            "Events": int(y_true.sum()),
            "Requested_Bootstrap_Replicates": N_BOOTSTRAPS,
            "Minimum_Valid_Replicates_Across_Metrics": minimum_valid,
            "Minimum_Required_Replicates": MINIMUM_VALID_BOOTSTRAPS,
            "All_CIs_Computed": bool(
                minimum_valid >= MINIMUM_VALID_BOOTSTRAPS
            ),
            "Status": (
                "Success"
                if minimum_valid >= MINIMUM_VALID_BOOTSTRAPS
                else "Insufficient valid replicates"
            ),
        })

part10b_bootstrap_results = pd.DataFrame(bootstrap_result_rows)
part10b_bootstrap_status = pd.DataFrame(bootstrap_status_rows)


# ============================================================
# 7. CROSS-CHECK POINT ESTIMATES AGAINST PART 10A
# ============================================================

point_crosscheck_rows = []

for _, result in part10b_bootstrap_results.iterrows():
    part10a_row = part10a_validation_results.loc[
        (
            part10a_validation_results["Imbalance_Strategy"]
            == result["Imbalance_Strategy"]
        )
        & (
            part10a_validation_results["Validation_Set"]
            == result["Validation_Set"]
        )
    ]

    if len(part10a_row) != 1:
        matched = False
        part10a_estimate = np.nan
    else:
        metric = result["Metric"]
        part10a_estimate = float(part10a_row.iloc[0][metric])
        matched = bool(
            np.isclose(
                float(result["Estimate"]),
                part10a_estimate,
                rtol=0,
                atol=1e-12,
            )
        )

    point_crosscheck_rows.append({
        "Imbalance_Strategy": result["Imbalance_Strategy"],
        "Validation_Set": result["Validation_Set"],
        "Metric": result["Metric"],
        "PART10B_Estimate": result["Estimate"],
        "PART10A_Estimate": part10a_estimate,
        "Point_Estimate_Matched": matched,
    })

part10b_point_estimate_crosscheck = pd.DataFrame(
    point_crosscheck_rows
)


# ============================================================
# 8. PUBLICATION TABLE
# ============================================================

part10b_publication_long = part10b_bootstrap_results.loc[
    :,
    [
        "Imbalance_Strategy",
        "Algorithm",
        "Validation_Set",
        "N",
        "Events",
        "Locked_Threshold",
        "Metric",
        "Estimate",
        "CI_Lower",
        "CI_Upper",
        "CI_Level",
        "CI_Method",
        "Valid_Bootstrap_Replicates",
    ],
].copy()

part10b_publication_long["Estimate_95CI"] = (
    part10b_publication_long["Estimate"].map(lambda value: f"{value:.3f}")
    + " ("
    + part10b_publication_long["CI_Lower"].map(
        lambda value: f"{value:.3f}" if np.isfinite(value) else "NA"
    )
    + "-"
    + part10b_publication_long["CI_Upper"].map(
        lambda value: f"{value:.3f}" if np.isfinite(value) else "NA"
    )
    + ")"
)

part10b_publication_table = (
    part10b_publication_long.pivot(
        index=[
            "Imbalance_Strategy",
            "Algorithm",
            "Validation_Set",
            "N",
            "Events",
            "Locked_Threshold",
        ],
        columns="Metric",
        values="Estimate_95CI",
    )
    .reset_index()
)

part10b_publication_table.columns.name = None


# ============================================================
# 9. GLOBAL VERIFICATION
# ============================================================

expected_analyses = (
    len(EXPECTED_STRATEGIES)
    * len(EXPECTED_VALIDATION_SETS)
)

expected_metric_rows = expected_analyses * len(METRICS)

all_inputs_ready = bool(part10b_input_audit["Input_Ready"].all())
all_bootstraps_successful = bool(
    len(part10b_bootstrap_status) == expected_analyses
    and part10b_bootstrap_status["All_CIs_Computed"].all()
)
all_point_estimates_matched = bool(
    len(part10b_point_estimate_crosscheck) == expected_metric_rows
    and part10b_point_estimate_crosscheck[
        "Point_Estimate_Matched"
    ].all()
)
all_ci_rows_present = bool(
    len(part10b_bootstrap_results) == expected_metric_rows
    and part10b_bootstrap_results["CI_Computed"].all()
)

PART10B_READY = bool(
    all_inputs_ready
    and all_bootstraps_successful
    and all_point_estimates_matched
    and all_ci_rows_present
)

part10b_global_verification = pd.DataFrame([{
    "Target_Strategy": primary_strategy,
    "Algorithm": primary_algorithm,
    "Imbalance_Strategies": len(EXPECTED_STRATEGIES),
    "Validation_Cohorts": len(EXPECTED_VALIDATION_SETS),
    "Expected_Bootstrap_Analyses": expected_analyses,
    "Successful_Bootstrap_Analyses": int(
        part10b_bootstrap_status["All_CIs_Computed"].sum()
    ),
    "Requested_Replicates_Per_Analysis": N_BOOTSTRAPS,
    "Minimum_Valid_Replicates": MINIMUM_VALID_BOOTSTRAPS,
    "CI_Level": CI_LEVEL,
    "CI_Method": (
        "Stratified percentile bootstrap "
        "(2.5th-97.5th percentiles)"
    ),
    "All_Fixed_Inputs_Ready": all_inputs_ready,
    "All_Point_Estimates_Matched_PART10A": (
        all_point_estimates_matched
    ),
    "All_Confidence_Intervals_Computed": all_ci_rows_present,
    "Model_Refitted": False,
    "Predictions_Regenerated": False,
    "Threshold_Modified": False,
    "Recalibration_Performed": False,
    "Participant_Level_Data_Exported": False,
    "Bootstrap_Distributions_Exported": False,
    "PART10B_Ready": PART10B_READY,
}])


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("=" * 105)
print("PART 10B: BOOTSTRAP CONFIDENCE INTERVALS FOR")
print("CLASS-IMBALANCE SENSITIVITY ANALYSES")
print("=" * 105)
print("\nFixed-input audit:")
display(part10b_input_audit)
print("\nBootstrap status:")
display(part10b_bootstrap_status)
print("\nPublication table:")
display(part10b_publication_table)
print("\nGlobal verification:")
display(part10b_global_verification)


# ============================================================
# 11. EXPORT AGGREGATE RESULTS ONLY
# ============================================================

part10b_input_audit.to_csv(
    "part10b_fixed_input_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

part10b_bootstrap_results.to_csv(
    "part10b_bootstrap_ci_results_long.csv",
    index=False,
    encoding="utf-8-sig",
)

part10b_bootstrap_status.to_csv(
    "part10b_bootstrap_status.csv",
    index=False,
    encoding="utf-8-sig",
)

part10b_point_estimate_crosscheck.to_csv(
    "part10b_point_estimate_crosscheck.csv",
    index=False,
    encoding="utf-8-sig",
)

part10b_publication_table.to_csv(
    "part10b_class_imbalance_sensitivity_bootstrap_table.csv",
    index=False,
    encoding="utf-8-sig",
)

part10b_global_verification.to_csv(
    "part10b_global_verification.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 12. FINAL SAFETY GATE AND SUMMARY
# ============================================================

if not PART10B_READY:
    raise RuntimeError("PART 10B methodological verification failed.")

print("\n" + "=" * 105)
print("PART 10B VERSION 1.0 COMPLETED")
print("=" * 105)
print(f"Final model                  : {primary_strategy} + {primary_algorithm}")
print(f"Imbalance strategies         : {len(EXPECTED_STRATEGIES)}")
print(f"Validation cohorts           : {len(EXPECTED_VALIDATION_SETS)}")
print(f"Bootstrap analyses           : {expected_analyses}")
print(f"Replicates per analysis      : {N_BOOTSTRAPS:,}")
print(f"Minimum valid replicates     : {MINIMUM_VALID_BOOTSTRAPS:,}")
print("Confidence interval method   : Stratified percentile bootstrap")
print("Confidence interval limits   : 2.5th-97.5th percentiles")
print("Model refitted               : No")
print("Predictions regenerated      : No")
print("Threshold modified           : No")
print("Participant rows exported    : No")
print("Bootstrap distributions saved: No")
print("\nPART10B_READY:", PART10B_READY)

### PART 10C: Publication Table and Reporting Text for Class-Imbalance Sensitivity Analyses

In [ ]:
# ============================================================
# PART 10C VERSION 1.1
# PUBLICATION TABLE AND REPORTING TEXT FOR CLASS-IMBALANCE
# SENSITIVITY ANALYSES
# ============================================================
#
# Purpose:
# - Assemble publication-ready aggregate tables from PARTS 10A-10B.
# - Generate Methods, Results, interpretation, table footnote, and
#   Appendix reporting text using already calculated results.
#
# Reporting status:
# - This is a reporting layer only.
# - No model is fitted or refitted.
# - No prediction or bootstrap replicate is generated.
# - No metric or confidence interval is recalculated.
# - No threshold is selected or modified.
# - No sensitivity-analysis winner is selected.
# - No participant-level information is exported.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================

TARGET_STRATEGY = "M2_Admin_2Q"
PRIMARY_IMBALANCE_STRATEGY = "Primary_Hybrid_SMOTENC_RUS"

EXPECTED_STRATEGIES = [
    "Primary_Hybrid_SMOTENC_RUS",
    "Unweighted_No_Resampling",
    "Class_Weighted_No_Resampling",
]

STRATEGY_LABELS = {
    "Primary_Hybrid_SMOTENC_RUS": (
        "Primary: SMOTENC followed by random undersampling"
    ),
    "Unweighted_No_Resampling": (
        "Sensitivity: unweighted without resampling"
    ),
    "Class_Weighted_No_Resampling": (
        "Sensitivity: balanced sample weights without resampling"
    ),
}

VALIDATION_SETS = [
    "Held-out 2024",
    "Temporal 2025",
]

REPORT_METRICS = [
    "AUC",
    "Average_Precision",
    "Brier_Score",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
]

METRIC_LABELS = {
    "AUC": "AUC",
    "Average_Precision": "Average precision",
    "Brier_Score": "Brier score",
    "Sensitivity": "Sensitivity",
    "Specificity": "Specificity",
    "Precision": "Precision",
    "F1": "F1 score",
}

OUTPUT_DIR = Path("part10c_class_imbalance_reporting")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. REQUIRED UPSTREAM OBJECTS
# ============================================================

required_objects = [
    "PART10A_READY",
    "PART10B_READY",
    "part10a_oof_summary",
    "part10a_validation_results",
    "part10a_locked_thresholds",
    "part10b_bootstrap_results",
    "part10b_bootstrap_status",
    "part10b_point_estimate_crosscheck",
    "part5c_final_recommendation",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required PART 10A/10B objects: "
        + ", ".join(missing_objects)
    )

if not bool(PART10A_READY):
    raise RuntimeError("PART 10A is not ready.")

if not bool(PART10B_READY):
    raise RuntimeError("PART 10B is not ready.")

primary_model = str(
    part5c_final_recommendation["Primary_Recommended_Strategy"]
)
primary_algorithm = str(
    part5c_final_recommendation["Primary_Recommended_Algorithm"]
)

if primary_model != TARGET_STRATEGY:
    raise RuntimeError("The final strategy is not M2_Admin_2Q.")


# ============================================================
# 3. VERIFY UPSTREAM RESULT COMPLETENESS
# ============================================================

if set(part10a_oof_summary["Imbalance_Strategy"]) != set(
    EXPECTED_STRATEGIES
):
    raise RuntimeError("PART 10A OOF strategies are incomplete.")

observed_validation_pairs = set(zip(
    part10a_validation_results["Imbalance_Strategy"],
    part10a_validation_results["Validation_Set"],
))

expected_validation_pairs = {
    (strategy, validation_set)
    for strategy in EXPECTED_STRATEGIES
    for validation_set in VALIDATION_SETS
}

if observed_validation_pairs != expected_validation_pairs:
    raise RuntimeError("PART 10A validation pairs are incomplete.")

expected_bootstrap_rows = (
    len(EXPECTED_STRATEGIES)
    * len(VALIDATION_SETS)
    * len(REPORT_METRICS)
)

if len(part10b_bootstrap_results) != expected_bootstrap_rows:
    raise RuntimeError("PART 10B metric-CI rows are incomplete.")

if not part10b_bootstrap_results["CI_Computed"].all():
    raise RuntimeError("One or more PART 10B CIs were not computed.")

if not part10b_bootstrap_status["All_CIs_Computed"].all():
    raise RuntimeError("One or more PART 10B bootstrap analyses failed.")

if not part10b_point_estimate_crosscheck[
    "Point_Estimate_Matched"
].all():
    raise RuntimeError("PART 10A and PART 10B point estimates differ.")


# ============================================================
# 4. STRATEGY-DEFINITION TABLE
# ============================================================

part10c_strategy_definitions = pd.DataFrame([
    {
        "Imbalance_Strategy": "Primary_Hybrid_SMOTENC_RUS",
        "Publication_Label": STRATEGY_LABELS[
            "Primary_Hybrid_SMOTENC_RUS"
        ],
        "Oversampling": "SMOTENC within each training fold",
        "Undersampling": (
            "Random undersampling after SMOTENC within each training fold"
        ),
        "Observation_Weighting": "None",
        "Analytical_Role": "Prespecified primary analysis",
    },
    {
        "Imbalance_Strategy": "Unweighted_No_Resampling",
        "Publication_Label": STRATEGY_LABELS[
            "Unweighted_No_Resampling"
        ],
        "Oversampling": "None",
        "Undersampling": "None",
        "Observation_Weighting": "None",
        "Analytical_Role": "Post hoc sensitivity analysis",
    },
    {
        "Imbalance_Strategy": "Class_Weighted_No_Resampling",
        "Publication_Label": STRATEGY_LABELS[
            "Class_Weighted_No_Resampling"
        ],
        "Oversampling": "None",
        "Undersampling": "None",
        "Observation_Weighting": (
            "Balanced sample weights calculated within each training fold"
        ),
        "Analytical_Role": "Post hoc sensitivity analysis",
    },
])


# ============================================================
# 5. DEVELOPMENT-OOF TABLE
# ============================================================

oof_columns = [
    "Imbalance_Strategy",
    "Algorithm",
    "N",
    "Events",
    "Locked_Threshold",
    "AUC",
    "Average_Precision",
    "Brier_Score",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "TP",
    "FP",
    "FN",
    "TN",
    "Threshold_Source",
    "Threshold_Rule",
]

missing_oof_columns = [
    column for column in oof_columns
    if column not in part10a_oof_summary.columns
]

if missing_oof_columns:
    raise KeyError(
        "PART 10A OOF summary is missing: "
        + ", ".join(missing_oof_columns)
    )

part10c_oof_table = part10a_oof_summary.loc[
    :, oof_columns
].copy()

part10c_oof_table.insert(
    1,
    "Publication_Label",
    part10c_oof_table["Imbalance_Strategy"].map(STRATEGY_LABELS),
)

strategy_order = {
    strategy: order
    for order, strategy in enumerate(EXPECTED_STRATEGIES)
}

part10c_oof_table["_Strategy_Order"] = (
    part10c_oof_table["Imbalance_Strategy"].map(strategy_order)
)

part10c_oof_table = (
    part10c_oof_table
    .sort_values("_Strategy_Order")
    .drop(columns="_Strategy_Order")
    .reset_index(drop=True)
)


# ============================================================
# 6. VALIDATION TABLE WITH BOOTSTRAP 95% CIs
# ============================================================

def format_estimate_ci(row):
    return (
        f"{float(row['Estimate']):.3f} "
        f"({float(row['CI_Lower']):.3f}-"
        f"{float(row['CI_Upper']):.3f})"
    )


validation_rows = []

for strategy in EXPECTED_STRATEGIES:
    for validation_set in VALIDATION_SETS:
        point_row = part10a_validation_results.loc[
            (
                part10a_validation_results["Imbalance_Strategy"]
                == strategy
            )
            & (
                part10a_validation_results["Validation_Set"]
                == validation_set
            )
        ]

        if len(point_row) != 1:
            raise RuntimeError(
                f"Expected one validation row for {strategy} | "
                f"{validation_set}."
            )

        point_row = point_row.iloc[0]
        output_row = {
            "Imbalance_Strategy": strategy,
            "Publication_Label": STRATEGY_LABELS[strategy],
            "Algorithm": primary_algorithm,
            "Validation_Set": validation_set,
            "N": int(point_row["N"]),
            "Events": int(point_row["Events"]),
            "Locked_Threshold": float(point_row["Locked_Threshold"]),
        }

        for metric in REPORT_METRICS:
            ci_row = part10b_bootstrap_results.loc[
                (
                    part10b_bootstrap_results["Imbalance_Strategy"]
                    == strategy
                )
                & (
                    part10b_bootstrap_results["Validation_Set"]
                    == validation_set
                )
                & (part10b_bootstrap_results["Metric"] == metric)
            ]

            if len(ci_row) != 1:
                raise RuntimeError(
                    f"Expected one CI row for {strategy} | "
                    f"{validation_set} | {metric}."
                )

            output_row[f"{metric}_95CI"] = format_estimate_ci(
                ci_row.iloc[0]
            )

        output_row.update({
            "TP": int(point_row["TP"]),
            "FP": int(point_row["FP"]),
            "FN": int(point_row["FN"]),
            "TN": int(point_row["TN"]),
            "CI_Method": (
                "Stratified percentile bootstrap; 2000 replicates"
            ),
        })
        validation_rows.append(output_row)

part10c_validation_table = pd.DataFrame(validation_rows)


# ============================================================
# 7. DESCRIPTIVE DIFFERENCES FROM PRIMARY ANALYSIS
# ============================================================
# These are arithmetic summaries of existing point estimates only.
# They are not used to select a preferred sensitivity strategy.
# ============================================================

difference_rows = []

for validation_set in VALIDATION_SETS:
    primary_row = part10a_validation_results.loc[
        (
            part10a_validation_results["Imbalance_Strategy"]
            == PRIMARY_IMBALANCE_STRATEGY
        )
        & (
            part10a_validation_results["Validation_Set"]
            == validation_set
        )
    ].iloc[0]

    for strategy in EXPECTED_STRATEGIES[1:]:
        sensitivity_row = part10a_validation_results.loc[
            (
                part10a_validation_results["Imbalance_Strategy"]
                == strategy
            )
            & (
                part10a_validation_results["Validation_Set"]
                == validation_set
            )
        ].iloc[0]

        difference_rows.append({
            "Validation_Set": validation_set,
            "Sensitivity_Strategy": strategy,
            "Publication_Label": STRATEGY_LABELS[strategy],
            "AUC_Difference_vs_Primary": float(
                sensitivity_row["AUC"] - primary_row["AUC"]
            ),
            "AP_Difference_vs_Primary": float(
                sensitivity_row["Average_Precision"]
                - primary_row["Average_Precision"]
            ),
            "Sensitivity_Difference_vs_Primary": float(
                sensitivity_row["Sensitivity"]
                - primary_row["Sensitivity"]
            ),
            "Specificity_Difference_vs_Primary": float(
                sensitivity_row["Specificity"]
                - primary_row["Specificity"]
            ),
            "Precision_Difference_vs_Primary": float(
                sensitivity_row["Precision"]
                - primary_row["Precision"]
            ),
            "F1_Difference_vs_Primary": float(
                sensitivity_row["F1"] - primary_row["F1"]
            ),
            "Used_For_Strategy_Selection": False,
        })

part10c_descriptive_differences = pd.DataFrame(difference_rows)


# ============================================================
# 8. REPORTING-TEXT HELPERS
# ============================================================

def metric_ci_text(strategy, validation_set, metric):
    row = part10b_bootstrap_results.loc[
        (
            part10b_bootstrap_results["Imbalance_Strategy"]
            == strategy
        )
        & (
            part10b_bootstrap_results["Validation_Set"]
            == validation_set
        )
        & (part10b_bootstrap_results["Metric"] == metric)
    ]
    if len(row) != 1:
        raise RuntimeError("Unable to resolve a unique metric-CI row.")
    row = row.iloc[0]
    return (
        f"{float(row['Estimate']):.3f} "
        f"(95% CI {float(row['CI_Lower']):.3f}-"
        f"{float(row['CI_Upper']):.3f})"
    )


def locked_threshold_text(strategy):
    return f"{float(part10a_locked_thresholds[strategy]):.6f}"


methods_text = (
    "As a post hoc sensitivity analysis, we evaluated whether the "
    "performance of the final M2 AdaBoost model was sensitive to the "
    "method used to address class imbalance. The prespecified primary "
    "hybrid pipeline, comprising SMOTENC followed by random "
    "undersampling within each training fold, was compared with an "
    "unweighted model without resampling and a model fitted using "
    "balanced sample weights without resampling. For the weighted "
    "analysis, sample weights were recalculated exclusively within each "
    "development training fold. All analyses retained the original "
    "2024 development-held-out split and the same AdaBoost "
    "hyperparameters. Classification thresholds were derived separately "
    "from genuine 5-fold stratified out-of-fold predictions in the "
    "development cohort by maximizing F1 score subject to a minimum "
    "sensitivity of 0.50 whenever feasible and were locked before "
    "held-out and temporal evaluation. Uncertainty was quantified using "
    "2000-replicate stratified participant-level percentile bootstrap "
    "95% confidence intervals. Neither validation cohort was used for "
    "model fitting, threshold selection, or selection of an imbalance "
    "strategy."
)


heldout_results_text = (
    "In held-out 2024 validation, the primary hybrid-resampling model "
    f"achieved an AUC of {metric_ci_text(PRIMARY_IMBALANCE_STRATEGY, 'Held-out 2024', 'AUC')}, "
    f"average precision of {metric_ci_text(PRIMARY_IMBALANCE_STRATEGY, 'Held-out 2024', 'Average_Precision')}, "
    f"sensitivity of {metric_ci_text(PRIMARY_IMBALANCE_STRATEGY, 'Held-out 2024', 'Sensitivity')}, "
    f"and specificity of {metric_ci_text(PRIMARY_IMBALANCE_STRATEGY, 'Held-out 2024', 'Specificity')}. "
    "The corresponding AUCs were "
    f"{metric_ci_text('Unweighted_No_Resampling', 'Held-out 2024', 'AUC')} "
    "for the unweighted model and "
    f"{metric_ci_text('Class_Weighted_No_Resampling', 'Held-out 2024', 'AUC')} "
    "for the class-weighted model."
)


temporal_results_text = (
    "In temporal 2025 validation, the primary hybrid-resampling model "
    f"achieved an AUC of {metric_ci_text(PRIMARY_IMBALANCE_STRATEGY, 'Temporal 2025', 'AUC')}, "
    f"average precision of {metric_ci_text(PRIMARY_IMBALANCE_STRATEGY, 'Temporal 2025', 'Average_Precision')}, "
    f"sensitivity of {metric_ci_text(PRIMARY_IMBALANCE_STRATEGY, 'Temporal 2025', 'Sensitivity')}, "
    f"and specificity of {metric_ci_text(PRIMARY_IMBALANCE_STRATEGY, 'Temporal 2025', 'Specificity')}. "
    "The corresponding AUCs were "
    f"{metric_ci_text('Unweighted_No_Resampling', 'Temporal 2025', 'AUC')} "
    "for the unweighted model and "
    f"{metric_ci_text('Class_Weighted_No_Resampling', 'Temporal 2025', 'AUC')} "
    "for the class-weighted model."
)


interpretation_text = (
    "These analyses were conducted as robustness checks and do not "
    "replace the prespecified primary hybrid-resampling analysis. "
    "Differences across imbalance strategies should be interpreted "
    "descriptively because the study was not designed or powered to "
    "establish the superiority of one imbalance-handling approach, and "
    "no sensitivity-analysis strategy was selected using held-out or "
    "temporal performance."
)


table_title = (
    "Sensitivity Analysis of Alternative Class-Imbalance Strategies "
    "for the Final M2 AdaBoost Model"
)


table_footnote = (
    "AUC indicates area under the receiver operating characteristic "
    "curve; AP, average precision; CI, confidence interval; M2, eight "
    "administrative predictors plus the 2-item Thai depression screener; "
    "SMOTENC, Synthetic Minority Over-sampling Technique for Nominal and "
    "Continuous features. Thresholds were selected separately for each "
    "imbalance strategy using genuine 5-fold OOF predictions from the "
    "2024 development cohort and were locked before validation. "
    "Confidence intervals were calculated using 2000-replicate "
    "stratified participant-level percentile bootstrap. Held-out 2024 "
    "and temporal 2025 data were not used for model fitting, threshold "
    "selection, or imbalance-strategy selection."
)


part10c_reporting_text = pd.DataFrame([
    {"Reporting_Component": "Methods", "Text": methods_text},
    {
        "Reporting_Component": "Results-Held-out 2024",
        "Text": heldout_results_text,
    },
    {
        "Reporting_Component": "Results-Temporal 2025",
        "Text": temporal_results_text,
    },
    {
        "Reporting_Component": "Interpretation",
        "Text": interpretation_text,
    },
    {"Reporting_Component": "Table title", "Text": table_title},
    {"Reporting_Component": "Table footnote", "Text": table_footnote},
])


# ============================================================
# 9. REPORTING VERIFICATION
# ============================================================

reporting_checks = {
    "PART10A_Ready": bool(PART10A_READY),
    "PART10B_Ready": bool(PART10B_READY),
    "Three_Strategy_Definitions": len(part10c_strategy_definitions) == 3,
    "Three_OOF_Rows": len(part10c_oof_table) == 3,
    "Six_Validation_Rows": len(part10c_validation_table) == 6,
    "Four_Descriptive_Difference_Rows": (
        len(part10c_descriptive_differences) == 4
    ),
    "All_Validation_CIs_Present": not part10c_validation_table[
        [f"{metric}_95CI" for metric in REPORT_METRICS]
    ].isna().any().any(),
    "Six_Reporting_Components": len(part10c_reporting_text) == 6,
    "Primary_Analysis_Retained": True,
    "No_Sensitivity_Winner_Selected": True,
}

PART10C_READY = bool(all(reporting_checks.values()))

part10c_global_verification = pd.DataFrame([{
    "Final_Model": f"{primary_model} + {primary_algorithm}",
    "Strategies_Reported": len(EXPECTED_STRATEGIES),
    "Validation_Cohorts_Reported": len(VALIDATION_SETS),
    **reporting_checks,
    "Model_Refitted": False,
    "Prediction_Generated": False,
    "Metric_Recalculated": False,
    "Bootstrap_Rerun": False,
    "Threshold_Modified": False,
    "Participant_Level_Data_Exported": False,
    "PART10C_Ready": PART10C_READY,
}])


# ============================================================
# 10. DISPLAY PUBLICATION MATERIALS
# ============================================================

print("=" * 105)
print("PART 10C: PUBLICATION REPORTING FOR CLASS-IMBALANCE")
print("SENSITIVITY ANALYSES")
print("=" * 105)

print("\nStrategy definitions:")
display(part10c_strategy_definitions)

print("\nDevelopment OOF table:")
display(part10c_oof_table)

print("\nValidation table with bootstrap 95% CIs:")
display(part10c_validation_table)

print("\nDescriptive differences from the primary analysis:")
display(part10c_descriptive_differences)

print("\nReporting text:")
for _, row in part10c_reporting_text.iterrows():
    print("\n" + row["Reporting_Component"] + ":")
    print(row["Text"])

print("\nGlobal verification:")
display(part10c_global_verification)


# ============================================================
# 11. EXPORT AGGREGATE PUBLICATION MATERIALS ONLY
# ============================================================

part10c_strategy_definitions.to_csv(
    OUTPUT_DIR / "part10c_strategy_definitions.csv",
    index=False,
    encoding="utf-8-sig",
)

part10c_oof_table.to_csv(
    OUTPUT_DIR / "part10c_development_oof_table.csv",
    index=False,
    encoding="utf-8-sig",
)

part10c_validation_table.to_csv(
    OUTPUT_DIR / "part10c_validation_table_with_95ci.csv",
    index=False,
    encoding="utf-8-sig",
)

part10c_descriptive_differences.to_csv(
    OUTPUT_DIR / "part10c_descriptive_differences_vs_primary.csv",
    index=False,
    encoding="utf-8-sig",
)

part10c_reporting_text.to_csv(
    OUTPUT_DIR / "part10c_reporting_text.csv",
    index=False,
    encoding="utf-8-sig",
)

part10c_global_verification.to_csv(
    OUTPUT_DIR / "part10c_global_verification.csv",
    index=False,
    encoding="utf-8-sig",
)

reporting_text_file = OUTPUT_DIR / "part10c_manuscript_appendix_text.txt"

with reporting_text_file.open("w", encoding="utf-8") as file:
    for _, row in part10c_reporting_text.iterrows():
        file.write(row["Reporting_Component"] + "\n")
        file.write(row["Text"] + "\n\n")


# ============================================================
# 12. FINAL SAFETY GATE AND SUMMARY
# ============================================================

if not PART10C_READY:
    failed_checks = [
        name for name, passed in reporting_checks.items()
        if not bool(passed)
    ]
    raise RuntimeError(
        "PART 10C reporting verification failed: "
        + ", ".join(failed_checks)
    )

print("\n" + "=" * 105)
print("PART 10C VERSION 1.1 COMPLETED")
print("=" * 105)
print(f"Final model              : {primary_model} + {primary_algorithm}")
print(f"Strategies reported      : {len(EXPECTED_STRATEGIES)}")
print(f"Validation cohorts       : {len(VALIDATION_SETS)}")
print("Primary analysis retained: Yes")
print("Sensitivity winner chosen: No")
print("Model refitted           : No")
print("Prediction generated     : No")
print("Metric recalculated      : No")
print("Bootstrap analysis rerun : No")
print("Threshold modified       : No")
print("Participant rows exported: No")
print(f"Output directory         : {OUTPUT_DIR}")
print("\nPART10C_READY:", PART10C_READY)

In [ ]:
# ============================================================
# EXPORT PACKAGE FOR SUPERVISOR
# Primary Outcome: Clinically Diagnosed MDD
#
# Frozen representative models:
#   M2 = M2_Admin_2Q + AdaBoostClassifier
#   M5 = M5_Conventional_Admin + ExtraTreesClassifier
#
# Validation cohorts:
#   - Held-out 2024
#   - Temporal 2025
#
# IMPORTANT:
#   - NO model refitting
#   - NO new probability generation
#   - NO threshold optimization
#   - Export existing frozen validation predictions only
# ============================================================


import os
import numpy as np
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================

EXPORT_DIR = "Supervisor_Export_M2_M5"

os.makedirs(
    EXPORT_DIR,
    exist_ok=True
)


# Explicitly define the frozen representative model
# for each screening strategy.
TARGET_MODELS = {

    "M2": {
        "model": "M2_Admin_2Q",
        "algorithm": "AdaBoostClassifier"
    },

    "M5": {
        "model": "M5_Conventional_Admin",
        "algorithm": "ExtraTreesClassifier"
    }
}


# ============================================================
# 2. CHECK REQUIRED OBJECTS
# ============================================================

required_objects = [
    "heldout_predictions",
    "temporal_predictions",
    "locked_thresholds"
]


missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]


if missing_objects:

    raise NameError(
        "Missing required object(s): "
        + ", ".join(missing_objects)
        + "\n\nPlease run the preceding MDD analysis cells first."
    )


print("=" * 80)
print("REQUIRED OBJECTS FOUND")
print("=" * 80)

for obj in required_objects:
    print("✓", obj)


# ============================================================
# 3. SHOW AVAILABLE STRATEGIES
# ============================================================

print("\n" + "=" * 80)
print("AVAILABLE STRATEGIES")
print("=" * 80)


print("\nHeld-out:")
print(
    list(
        heldout_predictions.keys()
    )
)


print("\nTemporal:")
print(
    list(
        temporal_predictions.keys()
    )
)


print("\nLocked thresholds:")
print(
    list(
        locked_thresholds.keys()
    )
)


# ============================================================
# 4. CHECK STRATEGY + ALGORITHM
# ============================================================

for short_name, config in TARGET_MODELS.items():

    model_name = config["model"]
    algorithm_name = config["algorithm"]


    print("\n" + "-" * 80)
    print(
        f"Checking {short_name}: "
        f"{model_name} + {algorithm_name}"
    )
    print("-" * 80)


    # --------------------------------------------------------
    # Strategy exists?
    # --------------------------------------------------------

    if model_name not in heldout_predictions:

        raise KeyError(
            f"{model_name} not found in heldout_predictions.\n"
            f"Available strategies:\n"
            f"{list(heldout_predictions.keys())}"
        )


    if model_name not in temporal_predictions:

        raise KeyError(
            f"{model_name} not found in temporal_predictions.\n"
            f"Available strategies:\n"
            f"{list(temporal_predictions.keys())}"
        )


    if model_name not in locked_thresholds:

        raise KeyError(
            f"{model_name} not found in locked_thresholds.\n"
            f"Available strategies:\n"
            f"{list(locked_thresholds.keys())}"
        )


    # --------------------------------------------------------
    # Algorithm exists?
    # --------------------------------------------------------

    heldout_algorithms = list(
        heldout_predictions[
            model_name
        ].keys()
    )


    temporal_algorithms = list(
        temporal_predictions[
            model_name
        ].keys()
    )


    print(
        "Held-out algorithms:",
        heldout_algorithms
    )

    print(
        "Temporal algorithms:",
        temporal_algorithms
    )


    if algorithm_name not in heldout_algorithms:

        raise KeyError(
            f"{algorithm_name} not found for {model_name} "
            f"in heldout_predictions.\n"
            f"Available algorithms:\n"
            f"{heldout_algorithms}"
        )


    if algorithm_name not in temporal_algorithms:

        raise KeyError(
            f"{algorithm_name} not found for {model_name} "
            f"in temporal_predictions.\n"
            f"Available algorithms:\n"
            f"{temporal_algorithms}"
        )


    print("✓ Strategy and algorithm found.")


# ============================================================
# 5. HELPER: EXTRACT DEVELOPMENT LOCKED THRESHOLD
# ============================================================

def extract_locked_threshold(
    threshold_container,
    model_name,
    algorithm_name
):

    """
    Extract the development OOF locked threshold without
    hard-coding the threshold value.
    """

    obj = threshold_container[
        model_name
    ]


    # --------------------------------------------------------
    # Case 1: threshold stored directly
    # --------------------------------------------------------

    if np.isscalar(obj):

        return float(obj)


    # --------------------------------------------------------
    # Case 2: dictionary
    # --------------------------------------------------------

    if isinstance(obj, dict):


        # Exact algorithm key
        if algorithm_name in obj:

            value = obj[
                algorithm_name
            ]


            # Algorithm key may itself contain another dict
            if isinstance(value, dict):

                for threshold_key in [
                    "locked_threshold",
                    "threshold",
                    "best_threshold"
                ]:

                    if threshold_key in value:

                        return float(
                            value[
                                threshold_key
                            ]
                        )


            elif np.isscalar(value):

                return float(value)


        # ----------------------------------------------------
        # Try matching algorithm name as string
        # ----------------------------------------------------

        for key, value in obj.items():

            if str(key) == str(
                algorithm_name
            ):

                if np.isscalar(value):

                    return float(value)

                if isinstance(value, dict):

                    for threshold_key in [
                        "locked_threshold",
                        "threshold",
                        "best_threshold"
                    ]:

                        if threshold_key in value:

                            return float(
                                value[
                                    threshold_key
                                ]
                            )


        # ----------------------------------------------------
        # Direct threshold fields
        # ----------------------------------------------------

        for threshold_key in [
            "locked_threshold",
            "threshold",
            "best_threshold"
        ]:

            if threshold_key in obj:

                return float(
                    obj[
                        threshold_key
                    ]
                )


    raise RuntimeError(
        f"Unable to identify development locked threshold "
        f"for {model_name} + {algorithm_name}.\n"
        f"Stored threshold object:\n{obj}"
    )


# ============================================================
# 6. HELPER: BUILD INDIVIDUAL-LEVEL PREDICTION FILE
# ============================================================

def build_prediction_dataframe(
    record,
    short_name,
    model_name,
    algorithm_name,
    cohort_name
):


    # --------------------------------------------------------
    # Required prediction fields
    # --------------------------------------------------------

    required_fields = [
        "y_true",
        "y_prob",
        "y_pred",
        "locked_threshold"
    ]


    missing_fields = [
        field
        for field in required_fields
        if field not in record
    ]


    if missing_fields:

        raise KeyError(
            f"{short_name} / {cohort_name}: "
            f"missing prediction fields: "
            f"{missing_fields}"
        )


    # --------------------------------------------------------
    # Retrieve EXISTING predictions
    # --------------------------------------------------------

    y_true = np.asarray(
        record[
            "y_true"
        ],
        dtype=int
    )


    y_prob = np.asarray(
        record[
            "y_prob"
        ],
        dtype=float
    )


    y_pred = np.asarray(
        record[
            "y_pred"
        ],
        dtype=int
    )


    record_threshold = float(
        record[
            "locked_threshold"
        ]
    )


    # --------------------------------------------------------
    # Length check
    # --------------------------------------------------------

    if not (
        len(y_true)
        == len(y_prob)
        == len(y_pred)
    ):

        raise RuntimeError(
            f"{short_name} / {cohort_name}: "
            "y_true, y_prob and y_pred have different lengths."
        )


    # --------------------------------------------------------
    # Probability check
    # --------------------------------------------------------

    if not np.isfinite(
        y_prob
    ).all():

        raise RuntimeError(
            f"{short_name} / {cohort_name}: "
            "non-finite probability detected."
        )


    if (
        (y_prob < 0)
        |
        (y_prob > 1)
    ).any():

        raise RuntimeError(
            f"{short_name} / {cohort_name}: "
            "probability outside [0,1] detected."
        )


    # --------------------------------------------------------
    # Outcome check
    # --------------------------------------------------------

    unique_y = set(
        np.unique(
            y_true
        )
    )


    if not unique_y.issubset(
        {0, 1}
    ):

        raise RuntimeError(
            f"{short_name} / {cohort_name}: "
            f"unexpected y_true values: {unique_y}"
        )


    # --------------------------------------------------------
    # Verify classification against stored threshold
    # --------------------------------------------------------

    y_pred_check = (
        y_prob
        >= record_threshold
    ).astype(int)


    mismatch = (
        y_pred_check
        != y_pred
    )


    n_mismatch = int(
        mismatch.sum()
    )


    if n_mismatch > 0:

        raise RuntimeError(
            f"{short_name} / {cohort_name}: "
            f"{n_mismatch} stored predicted classes "
            "do not match probability >= locked threshold."
        )


    # --------------------------------------------------------
    # Participant identifier
    # --------------------------------------------------------

    if "original_index" in record:

        participant_id = np.asarray(
            record[
                "original_index"
            ]
        )

        participant_id_source = (
            "original_index"
        )


    elif "source_row_position" in record:

        participant_id = np.asarray(
            record[
                "source_row_position"
            ]
        )

        participant_id_source = (
            "source_row_position"
        )


    else:

        # Anonymous sequential identifier
        participant_id = np.arange(
            1,
            len(y_true) + 1
        )

        participant_id_source = (
            "sequential_export_id"
        )


    if len(
        participant_id
    ) != len(
        y_true
    ):

        raise RuntimeError(
            f"{short_name} / {cohort_name}: "
            "participant identifier length mismatch."
        )


    # --------------------------------------------------------
    # Build export dataframe
    # --------------------------------------------------------

    df = pd.DataFrame({

        "participant_id":
            participant_id,

        "outcome":
            "Clinically_Diagnosed_MDD",

        "cohort":
            cohort_name,

        "strategy":
            short_name,

        "model":
            model_name,

        "algorithm":
            algorithm_name,

        "y_true":
            y_true,

        "predicted_probability":
            y_prob,

        "locked_threshold":
            record_threshold,

        "predicted_class":
            y_pred

    })


    return (
        df,
        participant_id_source
    )


# ============================================================
# 7. EXPORT M2 AND M5
# ============================================================

summary_rows = []

exported_datasets = {}

all_checks_passed = True


for short_name, config in TARGET_MODELS.items():


    model_name = config[
        "model"
    ]

    algorithm_name = config[
        "algorithm"
    ]


    print("\n" + "=" * 80)

    print(
        f"PROCESSING {short_name}"
    )

    print(
        f"Model:     {model_name}"
    )

    print(
        f"Algorithm: {algorithm_name}"
    )

    print("=" * 80)


    # --------------------------------------------------------
    # Existing held-out record
    # --------------------------------------------------------

    heldout_record = (
        heldout_predictions
        [model_name]
        [algorithm_name]
    )


    # --------------------------------------------------------
    # Existing temporal record
    # --------------------------------------------------------

    temporal_record = (
        temporal_predictions
        [model_name]
        [algorithm_name]
    )


    # --------------------------------------------------------
    # Development OOF locked threshold
    # --------------------------------------------------------

    development_threshold = (
        extract_locked_threshold(
            locked_thresholds,
            model_name,
            algorithm_name
        )
    )


    # --------------------------------------------------------
    # Threshold stored with held-out predictions
    # --------------------------------------------------------

    heldout_threshold = float(
        heldout_record[
            "locked_threshold"
        ]
    )


    # --------------------------------------------------------
    # Threshold stored with temporal predictions
    # --------------------------------------------------------

    temporal_threshold = float(
        temporal_record[
            "locked_threshold"
        ]
    )


    print(
        "\nDevelopment OOF locked threshold:",
        f"{development_threshold:.12f}"
    )

    print(
        "Held-out stored threshold:       ",
        f"{heldout_threshold:.12f}"
    )

    print(
        "Temporal stored threshold:       ",
        f"{temporal_threshold:.12f}"
    )


    # --------------------------------------------------------
    # Verify threshold transport
    # --------------------------------------------------------

    heldout_threshold_ok = np.isclose(
        development_threshold,
        heldout_threshold,
        rtol=0,
        atol=1e-10
    )


    temporal_threshold_ok = np.isclose(
        development_threshold,
        temporal_threshold,
        rtol=0,
        atol=1e-10
    )


    if not heldout_threshold_ok:

        raise RuntimeError(
            f"{short_name}: held-out threshold "
            "does not match development OOF locked threshold.\n"
            f"Development = {development_threshold}\n"
            f"Held-out   = {heldout_threshold}"
        )


    if not temporal_threshold_ok:

        raise RuntimeError(
            f"{short_name}: temporal threshold "
            "does not match development OOF locked threshold.\n"
            f"Development = {development_threshold}\n"
            f"Temporal    = {temporal_threshold}"
        )


    print(
        "✓ Threshold transported unchanged "
        "to both validation cohorts."
    )


    # --------------------------------------------------------
    # Build held-out dataframe
    # --------------------------------------------------------

    heldout_df, heldout_id_source = (
        build_prediction_dataframe(

            record=
                heldout_record,

            short_name=
                short_name,

            model_name=
                model_name,

            algorithm_name=
                algorithm_name,

            cohort_name=
                "Held-out 2024"
        )
    )


    # --------------------------------------------------------
    # Build temporal dataframe
    # --------------------------------------------------------

    temporal_df, temporal_id_source = (
        build_prediction_dataframe(

            record=
                temporal_record,

            short_name=
                short_name,

            model_name=
                model_name,

            algorithm_name=
                algorithm_name,

            cohort_name=
                "Temporal 2025"
        )
    )


    # --------------------------------------------------------
    # File names
    # --------------------------------------------------------

    heldout_filename = (
        f"{short_name}_heldout2024_predictions.csv"
    )


    temporal_filename = (
        f"{short_name}_temporal2025_predictions.csv"
    )


    # --------------------------------------------------------
    # Export
    # --------------------------------------------------------

    heldout_df.to_csv(

        os.path.join(
            EXPORT_DIR,
            heldout_filename
        ),

        index=False
    )


    temporal_df.to_csv(

        os.path.join(
            EXPORT_DIR,
            temporal_filename
        ),

        index=False
    )


    # --------------------------------------------------------
    # Keep dataframes in notebook
    # --------------------------------------------------------

    exported_datasets[
        f"{short_name}_heldout2024"
    ] = heldout_df


    exported_datasets[
        f"{short_name}_temporal2025"
    ] = temporal_df


    # --------------------------------------------------------
    # Summary: Held-out
    # --------------------------------------------------------

    summary_rows.append({

        "Outcome":
            "Clinically diagnosed MDD",

        "Strategy":
            short_name,

        "Full_Model_Name":
            model_name,

        "Algorithm":
            algorithm_name,

        "Cohort":
            "Held-out 2024",

        "N":
            len(
                heldout_df
            ),

        "Events":
            int(
                heldout_df[
                    "y_true"
                ].sum()
            ),

        "Prevalence":
            float(
                heldout_df[
                    "y_true"
                ].mean()
            ),

        "Locked_Threshold":
            development_threshold,

        "Threshold_Source":
            "2024 development genuine OOF predictions",

        "Participant_ID_Source":
            heldout_id_source,

        "File":
            heldout_filename
    })


    # --------------------------------------------------------
    # Summary: Temporal
    # --------------------------------------------------------

    summary_rows.append({

        "Outcome":
            "Clinically diagnosed MDD",

        "Strategy":
            short_name,

        "Full_Model_Name":
            model_name,

        "Algorithm":
            algorithm_name,

        "Cohort":
            "Temporal 2025",

        "N":
            len(
                temporal_df
            ),

        "Events":
            int(
                temporal_df[
                    "y_true"
                ].sum()
            ),

        "Prevalence":
            float(
                temporal_df[
                    "y_true"
                ].mean()
            ),

        "Locked_Threshold":
            development_threshold,

        "Threshold_Source":
            "2024 development genuine OOF predictions",

        "Participant_ID_Source":
            temporal_id_source,

        "File":
            temporal_filename
    })


# ============================================================
# 8. CREATE SUMMARY FILE
# ============================================================

supervisor_summary = pd.DataFrame(
    summary_rows
)


supervisor_summary[
    "Prevalence"
] = supervisor_summary[
    "Prevalence"
].round(
    6
)


supervisor_summary.to_csv(

    os.path.join(
        EXPORT_DIR,
        "M2_M5_export_summary.csv"
    ),

    index=False
)


# ============================================================
# 9. CREATE COMBINED PREDICTION FILE
# ============================================================

combined_predictions = pd.concat(

    [
        exported_datasets[
            "M2_heldout2024"
        ],

        exported_datasets[
            "M2_temporal2025"
        ],

        exported_datasets[
            "M5_heldout2024"
        ],

        exported_datasets[
            "M5_temporal2025"
        ]
    ],

    ignore_index=True
)


combined_predictions.to_csv(

    os.path.join(
        EXPORT_DIR,
        "M2_M5_all_validation_predictions.csv"
    ),

    index=False
)


# ============================================================
# 10. EXPECTED COHORT CHECK
# ============================================================

expected_cohorts = {

    "Held-out 2024": {
        "N": 1227,
        "Events": 14
    },

    "Temporal 2025": {
        "N": 815,
        "Events": 16
    }
}


print("\n")
print("=" * 80)
print("COHORT CONSISTENCY CHECK")
print("=" * 80)


for _, row in supervisor_summary.iterrows():


    cohort = row[
        "Cohort"
    ]


    expected_n = expected_cohorts[
        cohort
    ]["N"]


    expected_events = expected_cohorts[
        cohort
    ]["Events"]


    n_ok = (
        int(
            row["N"]
        )
        ==
        expected_n
    )


    events_ok = (
        int(
            row["Events"]
        )
        ==
        expected_events
    )


    status = (
        "PASS"
        if n_ok and events_ok
        else "CHECK"
    )


    if status != "PASS":

        all_checks_passed = False


    print(
        f"{row['Strategy']} | "
        f"{cohort} | "
        f"N={int(row['N'])} "
        f"(expected {expected_n}) | "
        f"Events={int(row['Events'])} "
        f"(expected {expected_events}) | "
        f"{status}"
    )


# ============================================================
# 11. THRESHOLD CONSISTENCY CHECK
# ============================================================

print("\n")
print("=" * 80)
print("LOCKED THRESHOLD CHECK")
print("=" * 80)


for short_name in TARGET_MODELS.keys():


    temp = supervisor_summary[
        supervisor_summary[
            "Strategy"
        ]
        ==
        short_name
    ]


    thresholds = temp[
        "Locked_Threshold"
    ].unique()


    if len(
        thresholds
    ) == 1:

        print(
            f"{short_name}: "
            f"{float(thresholds[0]):.12f} "
            "— PASS"
        )


    else:

        print(
            f"{short_name}: "
            f"multiple thresholds found "
            f"{thresholds} — CHECK"
        )

        all_checks_passed = False


# ============================================================
# 12. CREATE README
# ============================================================

readme_text = f"""
PRIMARY MDD VALIDATION PREDICTION PACKAGE
=========================================

Outcome
-------
Psychiatrist-confirmed clinically diagnosed major depressive
disorder (MDD).


Frozen representative models
-----------------------------
M2:
    Strategy  = M2_Admin_2Q
    Algorithm = AdaBoostClassifier

M5:
    Strategy  = M5_Conventional_Admin
    Algorithm = ExtraTreesClassifier


Validation cohorts
------------------
1. Held-out 2024
2. Temporal 2025


Threshold procedure
-------------------
Classification thresholds were derived exclusively from genuine
out-of-fold predictions in the 2024 development cohort.

The threshold for each frozen model was locked before validation
and transported unchanged to the 2024 held-out and 2025 temporal
validation cohorts.

The threshold values in the exported files were read directly
from the completed analytical notebook and were not hard-coded
during this export step.


Export procedure
----------------
This export step:

- did NOT refit any model;
- did NOT generate new prediction probabilities;
- did NOT optimize or modify any threshold;
- did NOT use held-out outcomes for model fitting;
- did NOT use temporal outcomes for model fitting.


Variables
---------
participant_id:
    Identifier retained for participant-level alignment.

outcome:
    Clinically_Diagnosed_MDD.

cohort:
    Held-out 2024 or Temporal 2025.

strategy:
    M2 or M5.

model:
    Full screening-strategy name.

algorithm:
    Frozen representative machine-learning algorithm.

y_true:
    Observed MDD status:
    0 = no psychiatrist-confirmed MDD
    1 = psychiatrist-confirmed MDD.

predicted_probability:
    Raw predicted probability from the frozen model.

locked_threshold:
    Threshold derived from genuine 2024 development
    out-of-fold predictions.

predicted_class:
    Classification obtained by applying the locked threshold
    to predicted_probability.


Files
-----
M2_heldout2024_predictions.csv
M2_temporal2025_predictions.csv
M5_heldout2024_predictions.csv
M5_temporal2025_predictions.csv
M2_M5_export_summary.csv
M2_M5_all_validation_predictions.csv


Privacy
-------
Before sharing these files, confirm that participant_id does not
contain student ID, HN, name, national identification number,
email address, telephone number, or any other personally
identifiable information.
"""


with open(

    os.path.join(
        EXPORT_DIR,
        "README_M2_M5_predictions.txt"
    ),

    "w",

    encoding="utf-8"

) as f:

    f.write(
        readme_text
    )


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 100)
print("SUPERVISOR EXPORT SUMMARY")
print("=" * 100)


print(
    supervisor_summary.to_string(
        index=False
    )
)


# ============================================================
# 14. LIST EXPORTED FILES
# ============================================================

print("\n")
print("=" * 80)
print("EXPORTED FILES")
print("=" * 80)


for filename in sorted(
    os.listdir(
        EXPORT_DIR
    )
):

    print(
        "✓",
        filename
    )


print("\nSaved to:")

print(
    os.path.abspath(
        EXPORT_DIR
    )
)


# ============================================================
# 15. FINAL STATUS
# ============================================================

print("\n")
print("=" * 80)


if all_checks_passed:

    print(
        "FINAL STATUS: PASS"
    )

    print(
        "Cohort sizes, event counts, and threshold "
        "transport checks passed."
    )


else:

    print(
        "FINAL STATUS: CHECK REQUIRED"
    )

    print(
        "At least one consistency check requires review."
    )


print("=" * 80)


print("\nExport procedure verification:")
print("✓ No model refitting")
print("✓ No new probability generation")
print("✓ No threshold re-optimization")
print("✓ Existing held-out predictions exported")
print("✓ Existing temporal predictions exported")
print("✓ Development OOF threshold checked")
print("✓ Held-out threshold checked")
print("✓ Temporal threshold checked")

### PART 11: Cohort Characteristics

In [ ]:
# ============================================================
# APPENDIX 2: PARTICIPANT CHARACTERISTICS BY COHORT
#
# Uses original, nonresampled observations and the existing split.
# Does not split data, fit models, impute, or resample.
#
# Required notebook objects:
#   prepared_data
#   temporal_datasets
#   OUTCOME
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, FileLink


# ============================================================
# 1. SETTINGS
# ============================================================

STRATEGY = "M6_Full"

OUTPUT_DIR = (
    Path.cwd() / "Appendix2_Cohort_Characteristics"
)

# Add labels ONLY after checking the original codebook.
# Keys must be strings matching the codes in the exported audit.
#
# Example structure:
# CATEGORY_LABELS = {
#     "gender": {"verified_code": "verified_label"},
#     "institute": {"verified_code": "verified_institute_name"},
# }
#
# Unmapped categories retain their original codes.

CATEGORY_LABELS = {}

VARIABLE_LABELS = {
    "gender": "Gender",
    "institute": "Institute",
    "GPAX": "Pre-university GPAX",
    "dadstatus": "Paternal status",
    "momstatus": "Maternal status",
    "dadeducation": "Paternal education",
    "momeducation": "Maternal education",
    "parent": "Caregiver type",
    "disease": "Disease history",
    "smoking": "Smoking",
    "elecsmoking": "Electronic cigarette use",
    "drinking": "Alcohol consumption",
    "kratom": "Kratom use",
    "cannabis": "Cannabis use",
    "addictive": "Other substance use",
    "2Q1": "2Q item 1",
    "2Q2": "2Q item 2",
}

EXPECTED_COHORTS = {
    "Development 2024": {
        "N": 2863,
        "MDD_events": 32,
    },
    "Held-out 2024": {
        "N": 1227,
        "MDD_events": 14,
    },
    "Temporal 2025": {
        "N": 815,
        "MDD_events": 16,
    },
}


# ============================================================
# 2. CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================

required_objects = [
    "prepared_data",
    "temporal_datasets",
    "OUTCOME",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Part 1 and Part 4A in the same kernel first. "
        "Missing objects: "
        + ", ".join(missing_objects)
    )

if STRATEGY not in prepared_data:
    raise KeyError(
        f"{STRATEGY} not found in prepared_data. "
        f"Available strategies: {list(prepared_data)}"
    )

if STRATEGY not in temporal_datasets:
    raise KeyError(
        f"{STRATEGY} not found in temporal_datasets. "
        f"Available strategies: {list(temporal_datasets)}"
    )

strategy_data = prepared_data[STRATEGY]

required_keys = [
    "development_data",
    "heldout_data",
    "train_original_index",
    "test_original_index",
]

missing_keys = [
    key
    for key in required_keys
    if key not in strategy_data
]

if missing_keys:
    raise KeyError(
        "Missing fields in prepared_data: "
        + ", ".join(missing_keys)
    )

if strategy_data.get(
    "resampling_applied_part1", False
):
    raise ValueError(
        "The source cohort data must not be resampled."
    )


# ============================================================
# 3. COPY ORIGINAL COHORTS
# ============================================================

cohorts = {
    "Development 2024": (
        strategy_data["development_data"].copy()
    ),
    "Held-out 2024": (
        strategy_data["heldout_data"].copy()
    ),
    "Temporal 2025": (
        temporal_datasets[STRATEGY].copy()
    ),
}

variables = list(VARIABLE_LABELS)

verification_rows = []

for cohort_name, cohort_data in cohorts.items():

    required_columns = variables + [OUTCOME]

    missing_columns = [
        column
        for column in required_columns
        if column not in cohort_data.columns
    ]

    if missing_columns:
        raise KeyError(
            f"{cohort_name}: missing columns "
            f"{missing_columns}"
        )

    outcome_values = pd.to_numeric(
        cohort_data[OUTCOME],
        errors="raise",
    )

    if outcome_values.isna().any():
        raise ValueError(
            f"{cohort_name}: outcome contains missing values."
        )

    if not outcome_values.isin([0, 1]).all():
        raise ValueError(
            f"{cohort_name}: outcome must contain only 0 and 1."
        )

    # Modify only the local copy.
    cohort_data[OUTCOME] = outcome_values.astype(int)

    observed_n = len(cohort_data)
    observed_events = int(outcome_values.sum())

    expected_n = EXPECTED_COHORTS[cohort_name]["N"]
    expected_events = (
        EXPECTED_COHORTS[cohort_name]["MDD_events"]
    )

    if (
        observed_n != expected_n
        or observed_events != expected_events
    ):
        raise ValueError(
            f"{cohort_name}: observed N/events = "
            f"{observed_n}/{observed_events}; "
            f"expected {expected_n}/{expected_events}. "
            "Stop and verify the source cohort."
        )

    verification_rows.append({
        "Cohort": cohort_name,
        "N": observed_n,
        "MDD_events": observed_events,
        "MDD_percent": (
            100 * observed_events / observed_n
        ),
        "N_matches_manuscript": (
            observed_n == expected_n
        ),
        "Events_match_manuscript": (
            observed_events == expected_events
        ),
    })


# ============================================================
# 4. CHECK EXISTING DEVELOPMENT / HELD-OUT SPLIT
# ============================================================

development_ids = pd.Index(
    strategy_data["train_original_index"]
)

heldout_ids = pd.Index(
    strategy_data["test_original_index"]
)

if (
    len(development_ids) != 2863
    or len(heldout_ids) != 1227
):
    raise ValueError(
        "Original index counts do not match the expected split."
    )

if development_ids.hasnans or heldout_ids.hasnans:
    raise ValueError(
        "Original row identifiers contain missing values."
    )

if (
    not development_ids.is_unique
    or not heldout_ids.is_unique
):
    raise ValueError(
        "Duplicate original row identifiers were detected."
    )

if not development_ids.intersection(
    heldout_ids
).empty:
    raise ValueError(
        "Development and held-out cohorts overlap."
    )


# ============================================================
# 5. HELPERS
# ============================================================

def category_key(value):
    """Standardize numeric category codes without guessing labels."""

    if pd.isna(value):
        return None

    if isinstance(
        value,
        (int, float, np.integer, np.floating),
    ):
        numeric_value = float(value)

        if (
            np.isfinite(numeric_value)
            and numeric_value.is_integer()
        ):
            return str(int(numeric_value))

    return str(value)


def category_sort_key(value):
    """Sort numeric codes numerically and other labels alphabetically."""

    try:
        return (0, float(value), str(value))
    except (TypeError, ValueError):
        return (1, 0.0, str(value))


def format_count_percent(count, denominator, decimals=1):

    percentage = 100 * count / denominator

    return (
        f"{count:,} "
        f"({percentage:.{decimals}f})"
    )


table_rows = []
category_count_rows = []
gpax_rows = []


def append_table_row(
    variable,
    category_or_statistic,
    cohort_cells,
):

    table_rows.append({
        "Characteristic": (
            VARIABLE_LABELS.get(variable, variable)
        ),
        "Category / statistic": category_or_statistic,
        **cohort_cells,
    })


# ============================================================
# 6. SUMMARIZE PREDICTORS
# ============================================================

for variable in variables:

    # --------------------------------------------------------
    # Continuous predictor: GPAX
    # --------------------------------------------------------

    if variable == "GPAX":

        statistic_cells = {
            "Mean (SD)": {},
            "Median (Q1–Q3)": {},
            "Missing, n (%)": {},
        }

        for cohort_name, cohort_data in cohorts.items():

            values = pd.to_numeric(
                cohort_data[variable],
                errors="raise",
            )

            nonmissing = values.dropna()

            if nonmissing.empty:
                raise ValueError(
                    f"{cohort_name}: GPAX has no observed values."
                )

            if not np.isfinite(
                nonmissing.to_numpy(dtype=float)
            ).all():
                raise ValueError(
                    f"{cohort_name}: GPAX contains "
                    "nonfinite values."
                )

            mean_value = nonmissing.mean()
            sd_value = nonmissing.std(ddof=1)
            median_value = nonmissing.median()
            q1_value = nonmissing.quantile(0.25)
            q3_value = nonmissing.quantile(0.75)

            missing_n = int(values.isna().sum())

            statistic_cells["Mean (SD)"][
                cohort_name
            ] = (
                f"{mean_value:.2f} "
                f"({sd_value:.2f})"
            )

            statistic_cells["Median (Q1–Q3)"][
                cohort_name
            ] = (
                f"{median_value:.2f} "
                f"({q1_value:.2f}–{q3_value:.2f})"
            )

            statistic_cells["Missing, n (%)"][
                cohort_name
            ] = format_count_percent(
                missing_n,
                len(cohort_data),
            )

            gpax_rows.append({
                "Variable": variable,
                "Cohort": cohort_name,
                "Cohort_N": len(cohort_data),
                "Nonmissing_N": len(nonmissing),
                "Missing_N": missing_n,
                "Mean": mean_value,
                "SD": sd_value,
                "Median": median_value,
                "Q1": q1_value,
                "Q3": q3_value,
                "Minimum": nonmissing.min(),
                "Maximum": nonmissing.max(),
            })

        for statistic, cells in statistic_cells.items():
            append_table_row(
                variable,
                statistic,
                cells,
            )

        continue

    # --------------------------------------------------------
    # Categorical predictors
    # --------------------------------------------------------

    encoded_cohorts = {
        cohort_name: cohort_data[variable].map(
            category_key
        )
        for cohort_name, cohort_data in cohorts.items()
    }

    all_levels = set()

    for series in encoded_cohorts.values():
        all_levels.update(
            series.dropna().unique().tolist()
        )

    ordered_levels = sorted(
        all_levels,
        key=category_sort_key,
    )

    # Always include an explicit missing row.
    for level in ordered_levels + [None]:

        if level is None:
            category_label = "Missing"
        else:
            category_label = (
                CATEGORY_LABELS
                .get(variable, {})
                .get(level, f"Code {level}")
            )

        cells = {}

        for cohort_name, series in (
            encoded_cohorts.items()
        ):

            if level is None:
                count = int(series.isna().sum())
            else:
                count = int(series.eq(level).sum())

            denominator = len(series)

            cells[cohort_name] = (
                format_count_percent(
                    count,
                    denominator,
                )
            )

            category_count_rows.append({
                "Variable": variable,
                "Characteristic": (
                    VARIABLE_LABELS[variable]
                ),
                "Code": level,
                "Category_label": category_label,
                "Cohort": cohort_name,
                "Count": count,
                "Denominator": denominator,
                "Percent": (
                    100 * count / denominator
                ),
                "Label_verified": (
                    level is not None
                    and level in CATEGORY_LABELS.get(
                        variable, {}
                    )
                ),
            })

        append_table_row(
            variable,
            category_label,
            cells,
        )


# ============================================================
# 7. ADD PRIMARY OUTCOME
# ============================================================

outcome_cells = {}

for cohort_name, cohort_data in cohorts.items():

    event_count = int(
        cohort_data[OUTCOME].sum()
    )

    outcome_cells[cohort_name] = (
        format_count_percent(
            event_count,
            len(cohort_data),
            decimals=2,
        )
    )

append_table_row(
    "Psychiatrist-confirmed MDD",
    "Present, n (%)",
    outcome_cells,
)


# ============================================================
# 8. BUILD AND CHECK OUTPUT TABLES
# ============================================================

characteristics_table = pd.DataFrame(
    table_rows
)

category_counts_table = pd.DataFrame(
    category_count_rows
)

gpax_summary_table = pd.DataFrame(
    gpax_rows
)

verification_table = pd.DataFrame(
    verification_rows
)

for (variable, cohort_name), group in (
    category_counts_table.groupby(
        ["Variable", "Cohort"],
        sort=False,
    )
):

    observed_total = int(
        group["Count"].sum()
    )

    expected_total = len(
        cohorts[cohort_name]
    )

    if observed_total != expected_total:
        raise ValueError(
            f"Category totals do not match cohort N: "
            f"{variable}, {cohort_name}, "
            f"{observed_total} vs {expected_total}"
        )


# ============================================================
# 9. SAVE CSV FILES
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

output_tables = {
    "cohort_characteristics_table.csv": (
        characteristics_table
    ),
    "category_counts_and_coding.csv": (
        category_counts_table
    ),
    "gpax_numeric_summary.csv": (
        gpax_summary_table
    ),
    "cohort_verification.csv": (
        verification_table
    ),
}

saved_files = []

for filename, table in output_tables.items():

    filepath = OUTPUT_DIR / filename

    table.to_csv(
        filepath,
        index=False,
        encoding="utf-8-sig",
    )

    if (
        not filepath.exists()
        or filepath.stat().st_size == 0
    ):
        raise RuntimeError(
            f"Output file was not saved: {filepath}"
        )

    saved_files.append(filepath)


# ============================================================
# 10. DISPLAY RESULTS AND DOWNLOAD LINKS
# ============================================================

print("=" * 80)
print("COHORT VERIFICATION")
print("=" * 80)

display(verification_table)

print("\n" + "=" * 80)
print("PARTICIPANT CHARACTERISTICS")
print("=" * 80)

display(characteristics_table)

print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)

for filepath in saved_files:
    print(filepath.resolve())

print("\nDownload links:")

for filepath in saved_files:

    relative_path = filepath.relative_to(
        Path.cwd()
    )

    display(
        FileLink(
            str(relative_path).replace("\\", "/")
        )
    )

print(
    "\nIMPORTANT: Check category codes against the "
    "original codebook before publication."
)

print(
    "Both mean/SD and median/quartiles are supplied "
    "for GPAX; select the final presentation after review."
)

print(
    "This analysis describes included participants only. "
    "Included-versus-excluded comparisons require "
    "the eligible-population dataset."
)

print("\nCOMPLETED SUCCESSFULLY")